<center> <img src = "Images/logo.png" alt="drawing" style="width:400px;">

<center>  

<span style="background-size: 600px;background:White;color:REd;font-size: 60px;font-family: Comic Sans MS">Кредитный скоринг Альфа банка</span>

# <span style="color:DeepSkyBlue">Задача</span>

**Задача**

Кредитный скоринг – важнейшая банковская задача. Стандартным подходом к ее решению   
является построение классических моделей машинного обучения, таких как логистическая   
регрессия и градиентный бустинг, на табличных данных, в том числе используя агрегации  
от каких-нибудь последовательных данных, например, транзакционных историй клиентов.   
Альтернативный подход заключается в использовании последовательных данных “как есть”,   
подавая их на вход рекуррентной нейронной сети.

В этом соревновании участникам предлагается решить задачу кредитного скоринга клиентов   
Альфа-Банка, используя только данные кредитных историй. [Источник](https://www.kaggle.com/competitions/alfa-bank-pd-credit-history)

**Данные**

Датасет соревнования устроен таким образом, что кредиты для тренировочной выборки взяты   
за период в М месяцев, а кредиты для тестовой выборки взяты за последующие K месяцев.

Каждая запись кредитной истории содержит самую разнообразную информацию о прошлом кредите   
клиента, например, сумму, отношение клиента к кредиту, дату открытия и закрытия, информацию   
о просрочках по платежам и др. Все публикуемые данные тщательно анонимизированы.

Целевая переменная – бинарная величина, принимающая значения 0 и 1, где 1 соответствует   
дефолту клиента по кредиту.


**Проверка решений**

Метрика соревнования – ROC AUC. Подробнее про метрику можно почитать, например, [здесь](https://dyakonov.org/2017/07/28/auc-roc-площадь-под-кривой-ошибок/).

# <span style="color:DeepSkyBlue">Используемые библиотеки</span>

In [ ]:
import os

# работа с регулярными выражениями
import re

# библиотеки для работы с табличными данными
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import fastparquet as fp

# генерация случайных чисел
import random
from random import randint
from sklearn.utils import shuffle

# библиотеки для построения графики
import seaborn as sns
import matplotlib.pyplot as plt #для визуализации
import plotly.graph_objects as go
import plotly.figure_factory as ff
import plotly.express as px
from plotly.subplots import make_subplots
import nbformat

# библиотеки для математических преобразований с массивами данных
import numpy as np
import mlx.core as mx
from sklearn import model_selection
from sklearn.model_selection import train_test_split

# библиотеки для работы с функциями(частичная передача аргументов в функцию)
from functools import partial

# библиотеки для работы со статистическими характеристиками
from scipy import stats
import statistics
from collections import Counter

# библиотеки для работы с pipeline
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer

# проверка временного ряда на статичность
from statsmodels.tsa.stattools import adfuller

# Импортируем DBSCAN-кластеризацию
from sklearn.cluster import DBSCAN

# вставить картинку в Jupiter Notebook
from IPython.display import Image

# линейные модели машинного обучения
from sklearn import linear_model

# ансамбли моделей машинного обучения
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import HistGradientBoostingClassifier

# поиск гиперпараметров модели
from sklearn.model_selection import RandomizedSearchCV
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.feature_selection import RFE
import optuna

 # метрики
from sklearn import metrics

# библиотека для стандартизации данных
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import RobustScaler

# сохранить полученные модели
import joblib
from joblib import dump, load

# сборщик мусора
import gc

# для ограничения времени выполнения функции
import signal
import func_timeout

# для отслеживания времени выполнения функции
import time

# очистить output
from IPython.display import clear_output

# <span style="color:DeepSkyBlue">Разведывательный анализ данных (EDA-Exploratory Data Analysis)</span>

## <span style="color:DodgerBlue">Исходные данные</span>

In [ ]:
# Подгрузим данные о кредитной истории клиентов
td0 = pd.read_parquet('data_source/train_data/train_data_0.pq')
td1 = pd.read_parquet('data_source/train_data/train_data_1.pq')
td2 = pd.read_parquet('data_source/train_data/train_data_2.pq')
td3 = pd.read_parquet('data_source/train_data/train_data_3.pq')
td4 = pd.read_parquet('data_source/train_data/train_data_4.pq')
td5 = pd.read_parquet('data_source/train_data/train_data_5.pq')
td6 = pd.read_parquet('data_source/train_data/train_data_6.pq')
td7 = pd.read_parquet('data_source/train_data/train_data_7.pq')
td8 = pd.read_parquet('data_source/train_data/train_data_8.pq')
td9 = pd.read_parquet('data_source/train_data/train_data_9.pq')
td10 = pd.read_parquet('data_source/train_data/train_data_10.pq')
td11 = pd.read_parquet('data_source/train_data/train_data_11.pq')
# Обьединим данные в один датасет
train_data = pd.concat([td0,td1,td2,td3,td4,td5,td6,td7,td8,td9,td10,td11],ignore_index=True)

# Подгрузим данные о дефлоте клиента
target = pd.read_csv('data_source/train_data/train_target.csv')

In [ ]:
# очистим память от ненужных данных
del td0,td1,td2,td3,td4,td5,td6,td7,td8,td9,td10,td11
gc.collect()

In [ ]:
# посмотрим совпадают ли размерности датасетов
print(train_data.shape)
print(target.shape)

In [ ]:
# Посмотрм на данные о кредитной истории клиентов
train_data.head(11)

In [ ]:
print(train_data.columns)

id	-	Идентификатор заявки(id -клиента).  
rn	-	Порядковый номер кредитного продукта в кредитной истории. Большему номеру соответствует продукт с более поздней датой открытия.	  

**date features:**  
1. pre_since_opened	-	Дней с даты открытия кредита до даты сбора данных (бинаризовано**)	  
2. pre_since_confirmed	-	Дней с даты подтверждения информации по кредиту до даты сбора данных (бинаризовано**)	  
3. pre_pterm	-	Плановое количество дней с даты открытия кредита до даты закрытия (бинаризовано**)	  
4. pre_fterm	-	Фактическое количество дней с даты открытия кредита до даты закрытия (бинаризовано**)	  
5. pre_till_pclose	-	Плановое количество дней с даты сбора данных до даты закрытия кредита (бинаризовано**)	  
6. pre_till_fclose	-	Фактическое количество дней с даты сбора данных до даты закрытия кредита (бинаризовано**)
7. pclose_flag	-	Флаг: плановое количество дней с даты открытия кредита до даты закрытия не определено 	  
8. fclose_flag	-	Флаг: фактическое количество дней с даты открытия кредита до даты закрытия не определено

**late payments features:**  
1. pre_loans5	-	Число просрочек до 5 дней (бинаризовано**)	  
2. pre_loans530	-	Число просрочек от 5 до 30 дней (бинаризовано**)	  
3. pre_loans3060	-	Число просрочек от 30 до 60 дней (бинаризовано**)	  
4. pre_loans6090	-	Число просрочек от 60 до 90 дней (бинаризовано**)	  
5. pre_loans90	-	Число просрочек более, чем на 90 дней (бинаризовано**)	  
6. is_zero_loans5	-	Флаг: нет просрочек до 5 дней	  
7. is_zero_loans530	-	Флаг: нет просрочек от 5 до 30 дней	  
8. is_zero_loans3060	-	Флаг: нет просрочек от 30 до 60 дней	  
9. is_zero_loans6090	-	Флаг: нет просрочек от 60 до 90 дней	  
10. is_zero_loans90	-	Флаг: нет просрочек более, чем на 90 дней	  
11. pre_loans_total_overdue	-	Текущая просроченная задолженность (бинаризовано**)	  
12. pre_loans_max_overdue_sum	-	Максимальная просроченная задолженность (бинаризовано**)	


**credit features:** 
1. pre_loans_credit_limit	-	Кредитный лимит (бинаризовано**)	  
2. pre_loans_next_pay_summ	-	Сумма следующего платежа по кредиту (бинаризовано**)	  
3. pre_loans_outstanding	-	Оставшаяся невыплаченная сумма кредита (бинаризовано**)	  
4. pre_loans_credit_cost_rate	-	Полная стоимость кредита (бинаризовано**)	  
  

**relative features:**  
1. pre_util	-	Отношение оставшейся невыплаченной суммы кредита к кредитному лимиту (бинаризовано**)	  
2. pre_over2limit	-	Отношение текущей просроченной задолженности к кредитному лимиту (бинаризовано**)	  
3. pre_maxover2limit	-	Отношенение максимальной просроченной задолженности к кредитному лимиту (бинаризовано**)	  
4. is_zero_util	-	Флаг: отношение оставшейся невыплаченной суммы кредита к кредитному лимиту равняется 0	  
5. is_zero_over2limit	-	Флаг: отношение текущей просроченной задолженности к кредитному лимиту равняется 0	  
6. is_zero_maxover2limit	-	Флаг: отношение максимальной просроченной задолженности к кредитному лимиту равняется 0	  

**payments features:**   
1. enc_paym_{0..N}	-	Статусы ежемесячных платежей за последние N месяцев (закодировано***)	  

**service features:**  
1. enc_loans_account_holder_type	-	Тип отношения к кредиту (закодировано***)	  
2. enc_loans_credit_status	-	Статус кредита (закодировано***)	  
3. enc_loans_account_cur	-	Валюта кредита (закодировано***)	  
4. enc_loans_credit_type	-	Тип кредита (закодировано***)	  

  ** область значений поля разбивается на N непересекающихся промежутков, каждому промежутку случайным образом ставится в соответствие уникальный номер от 0 до N-1, значение поля заменяется номером промежутка, которому оно принадлежит  
  *** каждому уникальному значению поля случайным образом ставится в соответствие уникальный номер от 0 до K, значение поля заменяется номером этого значения		

In [ ]:
# посмотрим на данные о дефолте клиента
target

id	-	Идентификатор заявки(id -клиента).   
flag – Целевая переменная, 1 – факт ухода в дефолт.

In [ ]:
# проверим целевую переменную на сбалансированность
target['flag'].value_counts(normalize=True)

In [ ]:
# тоже в абсолютных значениях
target['flag'].value_counts()

## <span style="color:DodgerBlue">Анализ данных</span>

Обратим внимание, что данные представлены в виде кредитной истории клиента.  
У каждого клиента своя "продолжительность" кредитной истории.  
Посмотрим "каких" историй больше для каждого типа клиентов (дефолт/не дефолт)  

In [ ]:
# Создадим рабочую плоскость на котором можно разместить две диаграммы одну под другой с общей осью абсцисс
fig_1 = make_subplots(
    rows=2, # количество диаграмм по вертикали
    cols=1, # количество диаграмм по горизонтали
    shared_xaxes=True, # использовать одну ось х для двух диаграмм
    vertical_spacing = 0.05, # уменьшим расстояние между диаграммами
    x_title= 'Количество клиетских операций',
)
# Настроем оформление рабочей поверхности
fig_1.update_layout(
    title ={
        'text':'Распределение количества клиентских операций (клиент без дефолта)', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 800,# Высота рабочей плоскости
    width = 1400, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    template='simple_white', # зададим тему оформления для рабочей поверхности 
    )

# Добавим на рабочую плоскость коробчатую диаграмму
fig_1.add_box(
    x=train_data.groupby('id')['rn'].count().iloc[target[target['flag']==0].index], # Выбираем данные по оси х
    y0=0, # Устанавливаем значение в нуле по оси y
    row=1, col=1, # Выбираем в какой координатной плоскости будет строиться диаграмма
    showlegend=False, # убираем легенду (обе диаграммы представляют распределение одной величины)
    boxmean=True, # добавим на коробчатую диаграмму отметку среднего значения заработной платы
    name= '', # при наведение на значение не будет отображаться имя диаграммы
    hoverinfo='x' # при наведение отражается только значение заработной платы
) 

# Настроим отображение оси y для коробчатой диаграммы
fig_1.update_yaxes(
    title={'text':'Коробчатая диаграмма', # добавим название оси "y" для коробчатой диаграммы
            'font':{'size':20,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
            },
        ticklabelposition = 'inside top', # "спрячем" значения по оси "y"
    row=1,col=1) 

# Добавим на рабочую плоскость гистограмму c графиком оценки плотности ядра KDE
# Воспользуемся библиотекой plotly.figure_factory
# Формируем графику
dist_fig = ff.create_distplot(hist_data=[train_data.groupby('id')['rn'].count().iloc[target[target['flag']==0].index]], 
                              group_labels=[''],
                              bin_size=1,
                              show_rug=False,
                              show_hist=True,
                              histnorm='probability',
                              colors=['red'],
                              )
# Настроим отображение графики
dist_fig.update_traces(hovertemplate='Количество: %{x}<br>Плотность вероятности: %{y}', # добавить надписи при наведение
                       showlegend=False # уберем значок из легенды
                       )
# я не нашел как другим способом объединить subplots и figure_factory
# на сколько я разобрался, суть метода построения заключается в извлечение данных из figure_factory и добавление фрагментов в subplots
for trace in dist_fig.select_traces():
    fig_1.add_trace(trace, row=2, col=1)

# # Настроим отображение оси y для гистограммы
fig_1.update_yaxes(
     title ={
        'text':'Гистограмма распределения<br>(KDE-плотность вероятности)', # добавим название оси "y"
        'font':{'size':20,'family':"Times New Roman"}, # размер и стиль написания имени
        },
    ticklabelposition = 'inside top', # "спрячем" значения по оси "y"
    row=2,col=1) 

fig_1.show()

<span style="color:Blue"> 

Вывод:  

Распределение количества клиентских операций отлично от нормального.  
Из коробчатой диаграммы мы наблюдаем наличие выбросов - клиенты с количеством опреций больше 24.  
Так же коробчатая диаграмма показывает, что среднее значение больше медианы распределения. В этом случае, график    
распределения "скошен" вправо относительно нормального, что и наблюдается на графике распределения плотности вероятности.  
В рамках разброса количество клиетских операций принимает значения от 1 до 24, при этом основная часть приходится на   
знаечния от 4 до 12. 
Медиальное значение равно 7, а среднее значение приближается к 9.


In [ ]:
# Создадим рабочую плоскость на котором можно разместить две диаграммы одну под другой с общей осью абсцисс
fig_1 = make_subplots(
    rows=2, # количество диаграмм по вертикали
    cols=1, # количество диаграмм по горизонтали
    shared_xaxes=True, # использовать одну ось х для двух диаграмм
    vertical_spacing = 0.05, # уменьшим расстояние между диаграммами
    x_title= 'Количество клиетских операций',
)
# Настроем оформление рабочей поверхности
fig_1.update_layout(
    title ={
        'text':'Распределение количества клиентских операций (клиент с дефолтом)', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 800,# Высота рабочей плоскости
    width = 1400, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    template='simple_white', # зададим тему оформления для рабочей поверхности 
    )

# Добавим на рабочую плоскость коробчатую диаграмму
fig_1.add_box(
    x=train_data.groupby('id')['rn'].count().iloc[target[target['flag']==1].index], # Выбираем данные по оси х
    y0=0, # Устанавливаем значение в нуле по оси y
    row=1, col=1, # Выбираем в какой координатной плоскости будет строиться диаграмма
    showlegend=False, # убираем легенду (обе диаграммы представляют распределение одной величины)
    boxmean=True, # добавим на коробчатую диаграмму отметку среднего значения заработной платы
    name= '', # при наведение на значение не будет отображаться имя диаграммы
    hoverinfo='x' # при наведение отражается только значение заработной платы
) 

# Настроим отображение оси y для коробчатой диаграммы
fig_1.update_yaxes(
    title={'text':'Коробчатая диаграмма', # добавим название оси "y" для коробчатой диаграммы
            'font':{'size':20,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
            },
        ticklabelposition = 'inside top', # "спрячем" значения по оси "y"
    row=1,col=1) 

# Добавим на рабочую плоскость гистограмму c графиком оценки плотности ядра KDE
# Воспользуемся библиотекой plotly.figure_factory
# Формируем графику
dist_fig = ff.create_distplot(hist_data=[train_data.groupby('id')['rn'].count().iloc[target[target['flag']==0].index]], 
                              group_labels=[''],
                              bin_size=1,
                              show_rug=False,
                              show_hist=True,
                              histnorm='probability',
                              colors=['red'],
                              )
# Настроим отображение графики
dist_fig.update_traces(hovertemplate='Количество: %{x}<br>Плотность вероятности: %{y}', # добавить надписи при наведение
                       showlegend=False # уберем значок из легенды
                       )
# я не нашел как другим способом объединить subplots и figure_factory
# на сколько я разобрался, суть метода построения заключается в извлечение данных из figure_factory и добавление фрагментов в subplots
for trace in dist_fig.select_traces():
    fig_1.add_trace(trace, row=2, col=1)

# # Настроим отображение оси y для гистограммы
fig_1.update_yaxes(
     title ={
        'text':'Гистограмма распределения<br>(KDE-плотность вероятности)', # добавим название оси "y"
        'font':{'size':20,'family':"Times New Roman"}, # размер и стиль написания имени
        },
    ticklabelposition = 'inside top', # "спрячем" значения по оси "y"
    row=2,col=1) 

fig_1.show()

<span style="color:Blue"> 

Вывод для клиентов с дефолтом:  

Распределение количества клиентских операций отлично от нормального.  
Из коробчатой диаграммы мы наблюдаем наличие выбросов - клиенты с количеством опреций больше 25.  
Так же коробчатая диаграмма показывает, что среднее значение больше медианы распределения. В этом случае, график    
распределения "скошен" вправо относительно нормального, что и наблюдается на графике распределения плотности вероятности.  
В рамках разброса количество клиетских операций принимает значения от 1 до 25, при этом основная часть приходится на   
знаечния от 3 до 12. 
Медиальное значение равно 7, а среднее значение приближается к 8.


## <span style="color:DodgerBlue">Анализ признаков</span>

Выводы по внешнему предварительному анализу данных:

Все признаки в датасете либо бинаризированы, либо закодированы - соответственно являются категориальными.  
Данных в датасете содержащием кредитные истории клиентов больше, чем в датасете содержащем информацию о дефолте клиента.  
Это связано с тем, что на каждого клиента существует кредитная история из нескольких кредитов.  
Для того, чтобы подать эти данные в модель необходимо агрегировать информацию по каждой клиенту.   
В качестве агрегирующих функций примем статистические характеристики случайных величин.   
Проаналировав поле признаков - его можно разделить на 6 групп.  
Помимо всего небходимо учитывать, что данные кредитных историй имееют хронологическую последовательность.  
Необходимо, не потерять информацию об этом при агригации данных.  
Так же необходимо не потерять информацию, о количестве  кредитов в истории.  
В связи с выше изложенным, примем следующую концепцию решения задачи:

Подготовка данных для обучения модели:
1. Выделетить часть данных для проверки модели (80/20 %).
2. Разбить признаки на 6 групп.
3. Внутри каждой группы в границах одного клиента выполниить преобразование признаков:  
    3.1. Заменить значения признаков на их статистичекие характеристики без учета хронологии кредитсной истории;  
    3.2. Добавить признак количества кредитных историй.  
    3.3. Создать признаки учитывающие историю клинетских операций.

## <span style="color:DodgerBlue">Постановка задачи в рамках EDA</span>

Данные о клиентах представлены в виде заявок, номера которых соотностятся с датами подачи заявки.  
Большему номеру соответствует более поздняя дата заявки.

Разделим процесс преобразования признаков на три части: 

1. Разделим признаки на под пространства.

2. Преобразуем кредитные операции клиента в его признаки, сохранив последовательность операций.  
Такие методы кодирования как Ordinal Encoding, OneHot Encoding не подойдут, так как уничтожится   
информация о последовательности операций.

3. Преобразование признаков как дискренного ряда.  
Для каждого признака в "границах" одного "id" рассчитать статистические показатели ряда, такие как:  
   - среднее значение ряда (математическое ожидание);  
   - средне гармоническое значение ряда;   
   - стандартное отклонение;  
   - минимальное значение;  
   - 25% квантиль;  
   - 50% квантиль;  
   - 75% квантиль;  
   - максимальное значение;  
   - мода ряда (среднее значение мод);  
   - частота появления моды (средненего значения мод);
   - и т.д. 


## <span style="color:DodgerBlue">Разработка инструментов преобразования данных</span>

### <span style="color:RoyalBlue">функция torow_transformer</span>

In [ ]:
# функция torow_transformer

# Назначение: Преобразование признака столбца в признаки строки
#               с сохранением обратной последовательности в признаке.
#               (извлечение последних операций клиента)

# Внешние переменные функции: DataFrame, n_last
#   DataFrame - исоходый DataFrame
#   n_last - необходимое счисло послдених операций клиента
#   Структура DataFrame:
#       1. id
#       2. feature1
#       3. feature2
#       4. feature3
#       ...

# Результат работы функции: New DataFrame
#   Признаки New DataFrame:
#       1. id
#       2. feature1.1
#       3. feature1.2
#       4. feature1.3
#       ...
#       5. feature1.N
#       6. feature2.1
#       7. feature2.2
#       ...
#       8. feature2.N
#       ...
#   где featureX.1 - соотвествует последней операции клиента,
#       featureX.2 - соотвествует предпоследней операции клиента,
#       .....

# 2. dict_features - словарь (карта) признаков,
#    в котором отображается сокращения признаков и их расщифровка

# Алгоритм работы функции:
# 1. извлекаем признаки из данных DataFrame
# 3. формируем карту признаков dict_features:
#    3.1. каждый признак кодируется следующим образом: 'fn',
#         где n - порядковый номер признака
#    3.2. полные имена признаков задаются следующим образом: 'feature_N',
#         где N - порядок клиентской операции (большему N соотвествует, более ранняя операция)

# 4. Преобразуем данные к массиву
# 5. Группируем массив для каждого клиента
# 6. К групированному массиву прменяем следующие преобразования:
#    6.1. Обращаем порядок клиенских операций
#    6.2. выбираем посление n_last операций
#    6.3. Если число клиенски операций меньше чем n_last, дополняем их нулями
# 7. Преобразуем полученные данные к DataFrame

# Описание локальных переменных функции:
# 1. pd_data - исходный DataFrame
# 2. n_last - необходимое число последних операций клиента
# 3. list_id - список для хранения id клиентов внутри функции
# 4. list_features - список для хранения признаков исхдного dataframe внутри функции
# 5. dict_features - локальная карта признаков
# 6. rn_id - список количества операций для каждого клиента
# 6. array_data - данные преобразоанные к numpy-массиву
# 6. split_array - сгрупированные по клиентам данные преобразованные

# обьявлем функцию
def torow_func(dict_params):
    pd_data = dict_params['data']
    n_last = dict_params['n_last']

    # извлекаем список "id" клиентов
    list_id = pd_data['id'].unique().tolist()
    
    # извлекаем список признаков из данных        
    list_features = pd_data.columns.drop(['id','rn'])

    # формируем словарь для зашифрованных признаков
    dict_features = {}
    
    # заполним словарь dict_features
    num_f=0
    for feature in list_features:
        # шифруем признак: fk = "feature_agg_function"
        for num_feature in range(1,n_last+1):
            dict_features['f'+str(num_feature+num_f)] = feature+'_'+str(num_feature)
        num_f+=n_last
    

    # формируем словарь rn_id
    rn_id = pd_data.groupby('id')['id'].count().to_list()

    # для улучшения производительности преобразуем DataFrame в array-массив
    array_data = np.array(pd_data.iloc[:,2:]).transpose()

    # "порежем массив" по длине кредитной истории клиента
    split_array = np.array_split(array_data, np.cumsum(rn_id),axis=1)
    
    # определим порядок последующих преобразований в функции
    def transform_array(array_id):
        # обратим порядок клиентских операций 
        reverse_array_id = array_id[::,::-1]
        # выбрем после n операций клиента
        list_n_last = reverse_array_id[::,:n_last]
        # если клиенских операций было меньше чем n_last
        # дополним недастающие нулями и преобразуем данные к строке
        if len(list_n_last[0])<n_last:
            full_list_n_last = np.hstack((list_n_last,np.zeros((list_n_last.shape[0],n_last-len(list_n_last[0])),dtype='int64')))
            # преобразуем список к строке
            full_list_n_last = full_list_n_last.reshape(-1)
        else:
            full_list_n_last = list_n_last.reshape(-1)
        return full_list_n_last

    # применим transform_array преобразование к списку split_array
    list_data = np.array(list(map(transform_array,split_array)))[:-1]
    
    # преобразуем полученные данные к dataframe
    dataframe = pd.DataFrame(data=list_data, columns=dict_features.keys())

    # добавим столбец id
    dataframe.insert(0,'id',list_id)
    
    return dataframe, dict_features,rn_id


# преобразуем функции в инструмент для преобразования данных (Transformer)
torow_transformer = FunctionTransformer(torow_func)

In [ ]:
# функция features_from_transform_data_torow

# Назначение: Извлечение из данных, над которомы совершено 
#             row_fich_transformer() преобразование, признаков  
#             соотвествующих заданному числу последних  
#             опреаций клиента n_last

# Внешние переменные функции: DataFrame
#   Признаки DataFrame:
#       1. n_last - необходимое число последних операций клиента
#       2. n_groups - число групп признаков в transform_data_torow
#       3. N_last - число последних операций клиента показанных в transform_data_torow

# Результат работы функции: 
# 1. list_n_last_features - список признаков в transform_data_torow
#    соотвествующий заданному числу n_last. 
    

# обьявлем функцию
def features_from_transform_data_torow(n_last,n_groups,N_last):
    # создадим список под необходимые признаки
    list_n_last_features = []
    
    # обьявим начальное значение в группе признаков
    n_start = 0
    
    for i in range(n_groups):
        for n in range(n_last):
            list_n_last_features.append(n+n_start)
        n_start+=N_last
    
    return list_n_last_features   

### <span style="color:RoyalBlue">функция diff_feature</span>

In [ ]:
# функция diff_feature

# Назначение: Определение дифференциальных характеристик ряда 

# Внешние переменные функции: 
#           1.Series/np.array/list


# Результат работы функции: 
# 1. diff_list - Список из значений:
#                   1.1. speed - скорость изменения ряда;
#                   1.2. accel - ускорение изменения ряда;
#                   1.3. bias - смещение ряда;
#                   1.4. pulse - импульс ряда;

# обьявлем функцию
def diff_feature(data):
    # преобразуем данные к numpy массиву
    data = np.array(data)
    # расчитаем необходимые характеристики
    speed = round(float(np.diff(data,1).mean()),2)
    accel = round(float(np.diff(data,2).mean()),2)
    bias = round(float(np.diff(data,1).sum()),2)
    pulse = round(float(np.diff(data,2).sum()),2)
    # сформируем из найденных значений в список
    diff_list = [speed,accel,bias,pulse]
    return diff_list

### <span style="color:RoyalBlue">функция statistic_features</span>

In [ ]:
# функция statistic_features

# Назначение: Извлечение основных статистических характеристик
#             из признаков в исходном DataFrame.

# Внешние переменные функции: DataFrame
#   Признаки DataFrame:
#       1. id
#       2. feature1
#       3. feature2
#       4. feature3
#       ...

# Результат работы функции: 
# 1. dataframe - таблица с данными. 
#    Признаки dataframe:
#       1. id
#       2. feature1_mean
#       3. fearture1_hmean
#       4. feature1_std
#       5. feature1_min
#       6. feature1_25%
#       7. feature1_50%
#       8. feature1_75%
#       9. feature1_max
#       10. feature1_mode
#       11. feature1_frequency_mode
#       12. feature2_mean
#       ...
    
# 2. dict_features - словарь (карта) признаков,
#    в котором отображается сокращения признаков и их расщифровка

# Алгоритм работы функции:
# 1. извлекаем признаки из данных
# 2. формируем карту признаков:
#    2.1. каждый признак кодируется следующим образом: 'fn' где n - порядковый номер признака
#    2.2. полные имена признаков задаются следующим образом:
#         2.2.1 если в исходном dataframe признак бинарный, то: "Исходное имя признака"+"binary"
#         2.2.2 если в исходном dataframe признак не бинарный, то: "Исходное имя признака"+"Статистическая характеристика"
# 3. для каждого клиента по каждому признаку из исходного dataframe расчитываем статистические характеристики
# 4. записываем полученные значение в новый dataframe

# Описание локальных переменных функции:
# 1. dict_agg_function - словарь из агригирующих функций
#       keys: имена для обращения к функциям:
#       values: lamda-функция, соотвествующей статистической характристики
# 2. list_features - список для хранения признаков исхдного dataframe внутри функции
# 3. list_id - список для хранения id клиентов внутри функции
# 4. dict_features - локальная карта признаков
# 5. k - номер признака в dict_features на текущей итерации
# 6. dataframe - результирующий dataframe

# обьявлем функцию
def statistic_features(pd_data):
    # формируем список из функций для статистических преобразований
    # предусмотрим работу функций на случай, если в массиве данных всего 1 строка
    dict_agg_function = {
    'ptp' : lambda x: 0 if len(x) <= 3 else np.ptp(x),
    'mean': lambda x: 0 if len(x) <= 3 else x.mean(), 
    'gmean' : lambda x: stats.gmean(x),   
    'hmean': lambda x: stats.gmean(x),
    'pmean25': lambda x: stats.pmean(x,25),
    'pmean50': lambda x: stats.pmean(x,50),
    'pmean75': lambda x: stats.pmean(x,75),
    'expectile25': lambda x: stats.expectile(x,0.25),
    'expectile50': lambda x: stats.expectile(x),
    'expectile75': lambda x: stats.expectile(x,0.75),
    'moment': lambda x: stats.moment(x),
    'std': lambda x: 0 if len(x) <= 3 else np.std(x),
    'min': lambda x: min(x),
    '20%': lambda x: x.mean() if len(x) <= 3 else np.percentile(x,q=20),
    '30%': lambda x: x.mean() if len(x) <= 3 else np.percentile(x,q=30),
    '40%': lambda x: x.mean() if len(x) <= 3 else np.percentile(x,q=40),
    '50%': lambda x: x.mean() if len(x) <= 3 else np.percentile(x,q=50),
    '60%': lambda x: x.mean() if len(x) <= 3 else np.percentile(x,q=60),
    '70%': lambda x: x.mean() if len(x) <= 3 else np.percentile(x,q=70),
    'max': lambda x: max(x),
    'mode': lambda x: statistics.mean(statistics.multimode(x)),
    'frequency_mode': lambda x: round(list(x).count(statistics.multimode(x)[0])*len(statistics.multimode(x))/len(x),2),
    'cov' : lambda x: 0 if len(x) <= 3 else np.cov(x),
    'histogram' : lambda x: 0 if len(x) <= 3 else np.histogram(x)[1].mean(), 
    'speed': lambda x: 0 if len(x) <= 3 else diff_feature(x)[0],
    'accel': lambda x: 0 if len(x) <= 3 else diff_feature(x)[1],
    'bias': lambda x: 0 if len(x) <= 3 else diff_feature(x)[2],
    'pulse': lambda x: 0 if len(x) <= 3  else diff_feature(x)[3]
    } 

    # напишем функцию для преобразования массива до статистических характеристик
    def stat_func(array_data): 
        # сформируем лист под результаты преобразования
        list_for_result = []
        # запишем все статистические харкетристики из словаря dict_agg_function
        for func in dict_agg_function.values():
            list_for_result.append(func(array_data))
        return np.array(list_for_result).round(3)

    # напишем функцию для применения функции stat_func к списку
    def submap(list_data):
        # расчитаем количество операция клиента
        max_rn = len(list_data[0])
        # получим статистические характристики массива
        list_stat_features = np.array(list(map(stat_func,list_data))).reshape(-1)
        return np.hstack((max_rn,list_stat_features))
    
    # извлекаем список "id" клиентов
    list_id = pd_data['id'].unique().tolist()

    # извлекаем список признаков из данных        
    list_features = pd_data.columns.drop(['id','rn'])
    
    # формируем словарь для зашифрованных признаков
    dict_features = {'f1':'count'}
    k=1 # порядковый номер защифрованного признака

    # заполним словарь dict_features
    for feature in list_features:
        # шифруем признак: fk = "feature_agg_function"
        for key_function in dict_agg_function.keys():
            k+=1
            dict_features['f'+str(k)] = feature+'_'+key_function

    # формируем список rn_id
    rn_id = pd_data.groupby('id')['id'].count().to_list()

    # для улучшения производительности преобразуем DataFrame в array-массив
    array_data = np.array(pd_data.iloc[:,2:]).transpose()

    # "порежем массив" по длине кредитной истории клиента
    split_array = np.array_split(array_data, np.cumsum(rn_id),axis=1)[:-1]

    # получем статические характеристики признаков
    stat_features = np.array(list(map(submap,split_array)))
    
    # Сформируем dataframe из полученных данных
    dataframe = pd.DataFrame(data=stat_features, columns=dict_features.keys())

    # добавим столбец id
    dataframe.insert(0,'id',list_id)

    return dataframe, dict_features

# преобразуем функции в инструмент для преобразования данных (Transformer)
stat_transformer = FunctionTransformer(statistic_features)

### <span style="color:RoyalBlue">функция corr_transform_to_force</span>

In [ ]:
# функция corr_transform_to_force

# Назначение: из матрицы взаимных корреляций
#             выделить не корелирующие признаки

# Внешние переменные функции: 
#           1. df.corr() - матрица корреляций
#           2. threshold - порог значимости корреляции:
#               значение коэффициента корреляции, больше которого
#               признаки считаются скоррелированными.

# Пояснение: 
# Под силой корреляции будем понимать следующее: если коэффициент 
# коррелиции между признаками больше значения threshold, то принимаем,
# что между признаками сильная корреляционная связь значение коэффициента 
# коррелияции заменяем на 1, иначе корреляционная связь слабая и значение 
# коээфициента корреляции заменяем на 0

# Результат работы функции: 
# 1. corr_matrix - матрица корреляций(отражает силу корреляции)
# 2. list_ncorr_features - список не скореллированных признаков
# 3. corr_force - сила корреляции всей матрицы: отношение числа скоррелированных 
# признаков к числу всех признаков в матрице

# Описание локальных переменных функции:
# 1. coor_force - функция преобазующая значение
#        коэффициента корряляции в силу корреляции
# 2. corr_matrix - матрица отражающая силу корряляции между признаками
# 3. max_corr - максимальное число взаимных корреляций между признаками
# 4. list_ncorr_features - список не коррелируемых признаков


# обьявлем функцию
def corr_transform_to_force(matrix,threshold=0.7):
    list_features = matrix.index.tolist()
    
    
    # создадим функцию для разметки матрицы корреляции
    # 1 - корреляция признаков выше порога значимости threshold
    # 0 - корреляция признаков ниже порога значимости threshold
    corr_force = lambda x: 1 if x >threshold else 0
    # выполним разметку матрицы корреляции
    corr_matrix = matrix.map(lambda x: corr_force(x))
    
    # алгоритм отбора не коррелиарных признаков:
    #   1. Найдем признак с наибольшим числом взаимных корреляций
    #   2. удалим найденный признак
    #   3. составим матрицу корреляций из отсавшися признаков
    #   4. повторяем пункты 1-3 до тех пор пока в матрице не останутся 
    #       не коррелированные признаки

    # ищем наибольшее число взаимных корреляций среди признаков
    max_corr = corr_matrix.sum().max()

    while max_corr > 1:
        # определяем признак с наибольшим числом взаимных корреляций
        max_corr_feature = corr_matrix.sum()[corr_matrix.sum()==corr_matrix.sum().max()].index[0]
        # удалем признак из матрицы корреляций
        corr_matrix = corr_matrix.drop(max_corr_feature).drop(max_corr_feature,axis=1)
        max_corr = corr_matrix.sum().max()
    # запишем не скоррелированные признаки в список
    list_ncorr_features = corr_matrix.index.tolist()
    # найдем силу корреляции всей матрицы как отношение
    # количества скоррелированных признаков к всмеу количеству признаков
    corr_force = round(1-len(list_ncorr_features)/len(list_features),3)
    return corr_matrix, list_ncorr_features, corr_force

### <span style="color:RoyalBlue">функция search_DBSCAN_parameters</span>

In [ ]:
# функция search_DBSCAN_parameters

# Назначение: Для подбора eps и min_samples параметров,
#               функция "прогоняет" DBSCAN кластеризацию 
#               с параметрами eps и min_samples
#               примающими значения из заданного диапазона.

# Внешние переменные функции: 
#           1. data - dataframe для кластеризации
#           2. r1 - начало диапазона
#           3. r2 - конец диапазона  
#           4. n - предпалагамое число кластеров      

# Результат работы функции: 
# 1. data_cluster - кластеризация данных при различных 
#       значениях параметров eps и min_samples

# Описание локальных переменных функции:
# 1. parametr_range - диапазон изменения параметров
# 2. dataframe_columns - колонки в результирующем dataframe
# 3. data_cluster - результрующий dataframe
# 4. index_cluster - текущая позиция в data_cluster
# 5. clustering - кластеризатор
# 6. list_cluster_values - список для заполнения текущими 
#                          значениями data_cluster

# обьявлем функцию
def search_DBSCAN_parameters(dataframe,r1,r2,n=3):
    # задаем диапозон измениния параметров
    parameter_range = range(r1,r2)
    # формируем заготовку для результирующего dataframe
    dataframe_columns = ['eps','min_samples',-1,0,1]
    # проверим что задано не меньше минимального количества кластеров
    if n<=3: 
        data_cluster = pd.DataFrame(columns=dataframe_columns)
    else: 
        for claster in range(4,n+1):
            dataframe_columns.append(claster-2)
        data_cluster = pd.DataFrame(columns=dataframe_columns)
    # задаем начально значение индекса в data_cluster
    index_cluster = 0

    # для подсчета обьектов в кластерах создадим dataframe
    dataframe_count = pd.DataFrame()
    
    # "прогоняем" DBSCAN кластеризациию по диапазону параметров
    for eps in parameter_range:
        
        for min_samples in parameter_range:
            print('current eps:',eps,'  current min_samples:', min_samples, end='\r')
            # запускаем кластеризацию с текущими параметрами
            clustering = DBSCAN(eps=eps, min_samples=min_samples).fit(dataframe)
            # добавлем к данным столбец с разметкой
            dataframe_count['clater'] = clustering.labels_
            # формируем пустой список для заполнения
            list_cluster_values = []
            # добавлеям в список текущие параметры
            list_cluster_values.append(eps)
            list_cluster_values.append(min_samples)
            # добавлем в список количество обьктов в каждом кластере
            for column in dataframe_columns[2:]:
                list_cluster_values.append(len(dataframe_count['clater'][dataframe_count['clater']==column]))
                
            # заполняем dataframe  текущими данными
            data_cluster.loc[index_cluster] = list_cluster_values
            index_cluster +=1
            # сбрасываем dataframe_count
            dataframe_count = pd.DataFrame()
    return data_cluster

### <span style="color:RoyalBlue">функция generate_samples</span>

In [ ]:
# функция generate_samples

# Назначение: для генерации индексов выборок данных

# Внешние переменные функции: 
#           1. max - определяет максимальное значение множества
#               из которого формируются выборки
#           2. n - количество выборок
#           3. k - мощность одной выборки
     

# Результат работы функции: 
# 1. samples_list - список с выбороками

# алгоритм работы:
# 1. задаем отрезок натурального ряда N мощностью max и добавлем в него 0.
#       In = N U {0}, I = {0,1,2,3,4,5,..,max}
# 2. если мощность множества In больше, необходимого количества элементов
#       cardo(In) > n x k , то из множества In формируем n случайных выборок
# размера k без повторения.
# 3. если мощность множества In меньше, необходимого количества элементов
#       cardo(In) < n x k , то из множества In формируем случайные выборки
# размера k без повторения, до тех пор пока не закончится множество In.
# После, добираем недостающее количество выборок случайными выборками 
# размера k из множества In с повторением (bootstrap метод).


# обьявлем функцию
def generate_samples(max,n,k,random_state = None):
    # создадим список под результат
    samples_list = []
    # формируем множество натуральных числе от 0 до max
    In = list(range(max+1))
    # Будем выполнять код пока не наберем необходимого количества выборок
    # нарушим порядок в множестве
    In = shuffle(In,random_state=random_state)
    # random.shuffle(In)

    # задаим границы извлечения данных из In
    In_start = 0
    In_end = k
    while len(samples_list) < n:
        # сформируем список под одну выборку
        sample = []
        # первые списки будем наполнять значениеми из множества In
        # без повторения, до тех пор пока все значения из множества In
        # не распределяться по выборкам
        if len(In)-In_end >= 0:
            sample.extend(In[In_start:In_end])
        else:                    
            # если элементов во множестве In недостаточно,
            # запоняем выборку "остатками" 
            sample.extend(In[In_start:])

            # остальные данные заполняем методом bootstrap
            # выполнем код пока не заполним выборку k значениями
            while len(sample) < k:
                # генерируем случайное число из диапазона от 0 до len(In)-1
                random_index = randint(0,len(In)-1)
                # добавляем значение из множества In с индексом random_index
                # в список index_list
                sample.append(In[random_index])

        # после того как мы набрали значения в выборку отправлем ее в samples_list
        samples_list.append(sample)
        # переходим к следующим данным в множестве In
        In_start+=k
        In_end+=k

    return samples_list

### <span style="color:RoyalBlue">функция my_train_test_split</span>

In [ ]:
def my_train_test_split(X,y,random_state=42,train_size=0.8,):
    # если разбиение без стратификации

    # зададим число элементов в выборке train
    len_train = round(len(y)*train_size)
    # формируем множество натуральных чисел от 0 до max
    list_random_index = list(range(len(y)))
    # нарушим порядок в множестве
    list_random_index = shuffle(list_random_index,random_state=random_state)
    # формируем список индексов под train выборку
    train_samples = list_random_index[:len_train]
    # формируем список индексов под test выборку
    test_samples = list_random_index[len_train:]
    # выполнем код пока не заполним выборку k значениями
   
    X_train = X.iloc[train_samples]
    y_train = y.iloc[train_samples]
    X_test = X.iloc[test_samples]
    y_test = y.iloc[test_samples]

    return X_train, y_train, X_test, y_test

### <span style="color:RoyalBlue">функция class_1_percent_samples</span>

In [ ]:
# функция class_1_percent_samples

# Назначение: для генерации индексов сбалансированных выборок

# Внешние переменные функции: 
#           1. data_target - массив из id и значений класса
#           2. class_1_percent - процент класса 1 в результирующей выборке
#           3. random_state - параметр для обеспечения воспроизваодимости функции
     

# Результат работы функции: 
# 1. samples_list - список со сблансированными выбороками

# обьявлем функцию
def class_1_percent_samples(data_target,class_1_percent,random_state = None):
    # приведем данные к нужной форме
    data_target = pd.DataFrame(data=np.array(data_target),columns =['id','flag'])
    
    # разделим клиентов  по признаку flag
    flag_0 = data_target[data_target['flag']==0].reset_index(drop=True)
    flag_1 = data_target[data_target['flag']==1].reset_index(drop=True)

    # определим класс большинства
    if flag_1.shape[0] > flag_0.shape[0]:
        majority_class = flag_1
        minority_class = flag_0
        # расчитаем необходимую величину выборки majority_class
        majority_class_size = round(minority_class.shape[0]*(class_1_percent)/(1-class_1_percent))
        # с помощью функции generate_samples сформируем выборку для majority_class
        samples_majority_class= generate_samples(majority_class.shape[0]-1,1,majority_class_size,random_state=random_state)
    else:
        majority_class = flag_0
        minority_class = flag_1
        # расчитаем необходимую величину выборки majority_class
        majority_class_size = round(minority_class.shape[0]*(1-class_1_percent)/(class_1_percent))
        # с помощью функции generate_samples сформируем выборку для majority_class
        samples_majority_class= generate_samples(majority_class.shape[0]-1,1,majority_class_size,random_state=random_state)
    
    # сформируем список выбороки с заданным процентом класс 1
    samples_list_id = minority_class['id'].values.tolist()+majority_class['id'].iloc[samples_majority_class[0]].tolist()

    return samples_list_id

## <span style="color:DodgerBlue">Преобразование признаков и анализ признаков</span>

### <span style="color:RoyalBlue">Разделим признаки на 6 под пространств</span>

**date features:**  
1. pre_since_opened	-	Дней с даты открытия кредита до даты сбора данных (бинаризовано**)	  
2. pre_since_confirmed	-	Дней с даты подтверждения информации по кредиту до даты сбора данных (бинаризовано**)	  
3. pre_pterm	-	Плановое количество дней с даты открытия кредита до даты закрытия (бинаризовано**)	  
4. pre_fterm	-	Фактическое количество дней с даты открытия кредита до даты закрытия (бинаризовано**)	  
5. pre_till_pclose	-	Плановое количество дней с даты сбора данных до даты закрытия кредита (бинаризовано**)	  
6. pre_till_fclose	-	Фактическое количество дней с даты сбора данных до даты закрытия кредита (бинаризовано**)
7. pclose_flag	-	Флаг: плановое количество дней с даты открытия кредита до даты закрытия не определено 	  
8. fclose_flag	-	Флаг: фактическое количество дней с даты открытия кредита до даты закрытия не определено

**late payments features:**  
1. pre_loans5	-	Число просрочек до 5 дней (бинаризовано**)	  
2. pre_loans530	-	Число просрочек от 5 до 30 дней (бинаризовано**)	  
3. pre_loans3060	-	Число просрочек от 30 до 60 дней (бинаризовано**)	  
4. pre_loans6090	-	Число просрочек от 60 до 90 дней (бинаризовано**)	  
5. pre_loans90	-	Число просрочек более, чем на 90 дней (бинаризовано**)	  
6. is_zero_loans5	-	Флаг: нет просрочек до 5 дней	  
7. is_zero_loans530	-	Флаг: нет просрочек от 5 до 30 дней	  
8. is_zero_loans3060	-	Флаг: нет просрочек от 30 до 60 дней	  
9. is_zero_loans6090	-	Флаг: нет просрочек от 60 до 90 дней	  
10. is_zero_loans90	-	Флаг: нет просрочек более, чем на 90 дней	  
11. pre_loans_total_overdue	-	Текущая просроченная задолженность (бинаризовано**)	  
12. pre_loans_max_overdue_sum	-	Максимальная просроченная задолженность (бинаризовано**)	


**credit features:** 
1. pre_loans_credit_limit	-	Кредитный лимит (бинаризовано**)	  
2. pre_loans_next_pay_summ	-	Сумма следующего платежа по кредиту (бинаризовано**)	  
3. pre_loans_outstanding	-	Оставшаяся невыплаченная сумма кредита (бинаризовано**)	  
4. pre_loans_credit_cost_rate	-	Полная стоимость кредита (бинаризовано**)	  
  

**relative features:**  
1. pre_util	-	Отношение оставшейся невыплаченной суммы кредита к кредитному лимиту (бинаризовано**)	  
2. pre_over2limit	-	Отношение текущей просроченной задолженности к кредитному лимиту (бинаризовано**)	  
3. pre_maxover2limit	-	Отношенение максимальной просроченной задолженности к кредитному лимиту (бинаризовано**)	  
4. is_zero_util	-	Флаг: отношение оставшейся невыплаченной суммы кредита к кредитному лимиту равняется 0	  
5. is_zero_over2limit	-	Флаг: отношение текущей просроченной задолженности к кредитному лимиту равняется 0	  
6. is_zero_maxover2limit	-	Флаг: отношение максимальной просроченной задолженности к кредитному лимиту равняется 0	  

**payments features:**   
1. enc_paym_{0..N}	-	Статусы ежемесячных платежей за последние N месяцев (закодировано***)	  

**service features:**  
1. enc_loans_account_holder_type	-	Тип отношения к кредиту (закодировано***)	  
2. enc_loans_credit_status	-	Статус кредита (закодировано***)	  
3. enc_loans_account_cur	-	Валюта кредита (закодировано***)	  
4. enc_loans_credit_type	-	Тип кредита (закодировано***)	  

  ** область значений поля разбивается на N непересекающихся промежутков, каждому промежутку случайным образом ставится в соответствие уникальный номер от 0 до N-1, значение поля заменяется номером промежутка, которому оно принадлежит  
  *** каждому уникальному значению поля случайным образом ставится в соответствие уникальный номер от 0 до K, значение поля заменяется номером этого значения		

In [ ]:
# сформируем списки признаков каждого подпространства
date_features = ['id','rn','pre_since_opened','pre_since_confirmed','pre_pterm','pre_fterm',
                 'pre_till_pclose','pre_till_fclose','pclose_flag','fclose_flag']
late_features = ['id','rn','pre_loans5','pre_loans530','pre_loans3060','pre_loans6090',
                 'pre_loans90','is_zero_loans5','is_zero_loans530','is_zero_loans3060',
                 'is_zero_loans6090','is_zero_loans90','pre_loans_total_overdue','pre_loans_max_overdue_sum']
credit_features = ['id','rn','pre_loans_credit_limit','pre_loans_next_pay_summ','pre_loans_outstanding','pre_loans_credit_cost_rate']

relative_features = ['id','rn','pre_util','pre_over2limit','pre_maxover2limit','is_zero_util',
                 'is_zero_over2limit','is_zero_maxover2limit']

payments_features = ['id','rn'] + train_data.columns[30:55].to_list()
service_features = ['id','rn'] + train_data.columns[55:59].to_list()

### <span style="color:RoyalBlue">Выполним torow-transform преобразование</span>

In [ ]:
# составим словарь небходимых данных и их признаков
dict_torow_data = {
    'date_torow' : date_features,
    'late_torow' : late_features,
    'credit_torow' : credit_features,
    'relative_torow' : relative_features,
    'payments_torow': payments_features,
    'service_torow': service_features}
# сформируем данные за последние 25 операций клиентов
for space_features in dict_torow_data.keys():
    clear_output()
    # добавим индекацию процесса
    print('Current space features:',space_features)
    # сформируем данные для преобразования
    data_to_transform = train_data[dict_torow_data[space_features]]
    print('Start transform')
    data_torow = torow_transformer.transform({'data':data_to_transform,'n_last':25})[0]
    # сохраним преобразованные данные на диск для быстрого воспроизведения
    print('Start save')
    fp.write('base_models/data/'+space_features,data_torow)
    # удалим использованные данные
    del data_to_transform
    gc.collect()

### <span style="color:RoyalBlue">Выполним stat-transform преобразование</span>

In [ ]:
# составим словарь небходимых данных и их признаков
dict_stat_data = {
    'date_stat' : date_features,
    'late_stat' : late_features,
    'credit_stat' : credit_features,
    'relative_stat' : relative_features,
    'payments_stat': payments_features,
    'service_stat': service_features}
# сформируем данные за последние 25 операций клиентов
for space_features in dict_stat_data.keys():
    clear_output()
    # добавим индекацию процесса
    print('Current space features:',space_features)
    # сформируем данные для преобразования
    data_to_transform = train_data[dict_stat_data[space_features]]
    print('Start transform')
    data_stat = stat_transformer.transform(data_to_transform)[0]
    # сохраним преобразованные данные на диск для быстрого воспроизведения
    print('Start save')
    fp.write('base_models/data/'+space_features,data_stat)
    # удалим использованные данные
    del data_to_transform, data_stat
    gc.collect()

### <span style="color:RoyalBlue">Корреляция признаков</span>

In [ ]:
# составим словарь небходимых данных и их признаков
list_space_features = ['date','late','credit','relative','payments','service']

# сформируем данные за последние 25 операций клиентов
for space_features in list_space_features:
    clear_output()
    # добавим индекацию процесса
    print('Current space features:',space_features)
    # подгружаем данные
    drop_id_stat_data = fp.ParquetFile('base_models/data/'+space_features+'_stat').to_pandas().drop('id',axis=1)
    # сформируем матрицу взаимных корреляций
    print('Found corr matrix')
    corr_matrix = drop_id_stat_data.corr()
    # сохраним матрицу взаимных корреляций на диск для быстрого воспроизведения
    print('Start save matrix')
    fp.write('base_models/data/corr_matrix_'+space_features,corr_matrix)
    # удалим использованные данные
    del drop_id_stat_data, corr_matrix
    gc.collect()

## <span style="color:DodgerBlue">Формирование данных для обучения модели</span>

### <span style="color:RoyalBlue">Формирование тренировочного, валидационного, тестового и отложенного набора данных</span>

In [ ]:
# для формирования наборов данных выберем dataset service_stat
transform_data_stat = fp.ParquetFile('base_models/data/stat/service_stat').to_pandas()

In [ ]:
# выделим отложенный набор данных для проверки результирующей модели
X_atrain, y_atrain, X_hold, y_hold = my_train_test_split(transform_data_stat.drop('id',axis=1),target['flag'], train_size=0.95)

In [ ]:
# Оставшиеся данные раздедлим на тренировочный и тестовый наборы
X_train, y_train, X_test, y_test = my_train_test_split(X_atrain, y_atrain, train_size=0.25)

In [ ]:
# Для валидации модели выделим данные из тренировочного набора
X_train, y_train, X_valid, y_valid = my_train_test_split(X_train,y_train, train_size=0.9)

In [ ]:
# проверим сбалансированность данных по признаку flag
print('target', target['flag'].value_counts(normalize=True))
print('train', y_train.value_counts(normalize=True))
print('valid', y_valid.value_counts(normalize=True))
print('test',y_test.value_counts(normalize=True))
print('hold',y_hold.value_counts(normalize=True))

In [ ]:
# проверим размерности полученных наборов
print(X_train.shape,y_train.shape)
print(X_valid.shape,y_valid.shape)
print(X_test.shape,y_test.shape)
print(X_hold.shape,y_hold.shape)

In [ ]:
X_train.shape[0]+X_valid.shape[0]

Для воспроизводимости кода и корректного сравнения различных моделей  
завиксируем id для тренировочного, валидационного и тестового наборов данных.

In [ ]:
# Спрячем данные в DataDfame для последующего сохраниения в csv
train_id = pd.DataFrame(y_train.index)
valid_id = pd.DataFrame(y_valid.index)
test_id = pd.DataFrame(y_test.index)
hold_id = pd.DataFrame(y_hold.index)

In [ ]:
# Сохраняем полученные DataFrame
train_id.to_csv('base_models/data/reproduce/train_id.csv',index=False)
valid_id.to_csv('base_models/data/reproduce/valid_id.csv',index=False)
test_id.to_csv('base_models/data/reproduce/test_id.csv',index=False)
hold_id.to_csv('base_models/data/reproduce/hold_id.csv',index=False)

In [ ]:
# подгрузим DataFrame с id для тренировочного/валидационного/тестового набора данных
train_id = pd.read_csv('base_models/data/reproduce/train_id.csv')['0'].to_list()
valid_id = pd.read_csv('base_models/data/reproduce/valid_id.csv')['0'].to_list()
test_id = pd.read_csv('base_models/data/reproduce/test_id.csv')['0'].to_list()
hold_id = pd.read_csv('base_models/data/reproduce/hold_id.csv')['0'].to_list()

In [ ]:
# подгрузим данные целевой переменной
target = pd.read_csv('data_source/train_data/train_target.csv')

In [ ]:
# сохраним тренировочный валидационный и тестовый целевой переменной
# чтобы иметь возможность к ними обратиться
target.loc[train_id].to_csv('target/target_train.csv',index=False)
target.loc[valid_id].to_csv('target/target_valid.csv',index=False)
target.loc[test_id].to_csv('target/target_test.csv',index=False)
target.loc[hold_id].to_csv('target/target_hold.csv',index=False)

### <span style="color:RoyalBlue">Формирование тренировочного, валидационного, тестового и отложенного набора на transform_data_torow_25 данных</span>

In [ ]:
# составим список пространств признаков
list_spaces = ['date_torow','late_torow','credit_torow','relative_torow','payments_torow','service_torow']
# составим словарь небходимых наборов данных
dict_datasets = {
    'train' : pd.read_csv('base_models/data/reproduce/train_id.csv')['0'].to_list(),
    'valid' : pd.read_csv('base_models/data/reproduce/valid_id.csv')['0'].to_list(),
    'test' : pd.read_csv('base_models/data/reproduce/test_id.csv')['0'].to_list(),
    'hold' : pd.read_csv('base_models/data/reproduce/hold_id.csv')['0'].to_list()
}
# разобьем torow данные на обучающий/валидационный/тестовый/отложенный наборы
for space_features in list_spaces:
    for name_dataset in dict_datasets.keys():
        clear_output()
        # добавим индекацию процесса
        print('Current space features:',space_features)
        print('Current name_dataset:',name_dataset)
        # загрузим необходимые данные
        dataset_id = dict_datasets[name_dataset]
        dataset = fp.ParquetFile('base_models/data/torow/'+space_features).to_pandas().set_index('id').iloc[dataset_id]
        # сохраним отборанные данные
        print('Start save '+space_features+'_'+name_dataset)
        fp.write('base_models/data/torow/'+space_features+'_'+name_dataset,dataset)
        # удалим использованные данные
        del dataset, dataset_id
        gc.collect()

### <span style="color:RoyalBlue">Формирование тренировочного, валидационного, тестового и отложенного набора на transform_data_stat данных</span>

In [ ]:
# составим список простраств признаков
list_spaces = ['date_stat','late_stat','credit_stat','relative_stat','payments_stat','service_stat']
# составим словарь небходимых наборов данных
dict_datasets = {
    'train' : pd.read_csv('base_models/data/reproduce/train_id.csv')['0'].to_list(),
    'valid' : pd.read_csv('base_models/data/reproduce/valid_id.csv')['0'].to_list(),
    'test' : pd.read_csv('base_models/data/reproduce/test_id.csv')['0'].to_list(),
    'hold' : pd.read_csv('base_models/data/reproduce/hold_id.csv')['0'].to_list()
}
# разобьем stat данные на обучающий/валидационный/тестовый/отложенный наборы
for space_features in list_spaces:
    for name_dataset in dict_datasets.keys():
        clear_output()
        # добавим индекацию процесса
        print('Current space features:',space_features)
        print('Current name_dataset:',name_dataset)
        # загрузим необходимые данные
        dataset_id = dict_datasets[name_dataset]
        dataset = fp.ParquetFile('base_models/data/stat/'+space_features).to_pandas().set_index('id').iloc[dataset_id]
        # сохраним отборанные данные
        print('Start save '+space_features+'_'+name_dataset)
        fp.write('base_models/data/stat/'+space_features+'_'+name_dataset,dataset)
        # удалим использованные данные
        del dataset, dataset_id
        gc.collect()

# <span style="color:DeepSkyBlue">Процесс машинного обучения (ML-Machine Learning)</span>

Постановка задачи в рамках Machine Learning:
1. Для решения задачи построем блендинг моделей. 

2. В качестве базовых и метамоделей рассмотрим следующие классические модели классификации:
    - linear_model.LogisticRegression (Логистическая регрессия);
    - RandomForestClassifier (Деревья решений);
    - HistGradientBoostingClassifier (Градиентный бустинг).

3. В результате, преобразования данных было получено два пространства признаков (torow и stat признаки),  
состоящих из 6 подпространств:
    - date features;
    - late payments features; 
    - credit features;
    - relative features;
    - payments features;
    - service features.

4. На первом этапе построения потроения блендинга, сфокусируем обучение базовых моделей,   
на каждом подпространстве в отдельности друго от друга.  

5. На втором этапе построения блендинга, обучим несколько групп метамоделей.   
Первая группа метамоделей в качестве метапризнаков использует предсказания базовых моделей,    
обученных на пространстве признаков torow.  
Вторая группа метамоделей в качестве метапризнаков использует предсказания базовых моделей,    
обученных на пространстве признаков stat.

6. На третьем этапе построения блендинга метамодель обучится на метапризнаках пространства 
torow и stat.

<center> <img src = "Images/Blending.jpg" alt="drawing" style="width:1400px;">

## <span style="color:DodgerBlue">Первый этап построения блендинга моделей</span>

### <span style="color:RoyalBlue">Построение модели на сбалансированных данных подпространства date torow</span>

#### <span style="color:MediumBlue">Формирование данных для обучения</span>

Анализ исходных данных, в частности target, показал что представленные данные   
имееют перекос: 96% клиентов у которых не случился дефолт (flag = 0),    
против 4 % - для которых произошел дефолт (flag = 1).  
С помощью функции class_1_percent_samples, сформируем сбаланисрованную выборку  
для обучения моделей.
За основу сбалансировных данных возьмем данные клиентов с дефолтом  
и к ним будем добирать необходимое количество клиентов до сбалансированности. 


Функция class_1_percent_samples принемает две переменные:
- target_data - массив с id и целевой переменной
- class_1_percent - процент класса 1 в результирующем массиве

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'date'
count_features = dict_spaces[feature_space]

In [ ]:
# сформируем данные для анализа сбалансированности обучающих данных
X_train_pd = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas()
y_train_pd = pd.read_csv('target/target_train.csv')

In [ ]:
# поготовим данные y_train_pd к работе с функцией class_1_percent_samples
y_train_pd.set_index('id',drop=False,inplace=True)
y_train_pd.head(4)

In [ ]:
# взглянем на сблансиорованность данных в y_train
y_train_pd['flag'].value_counts(normalize=True)

In [ ]:
# сбалансируем данные в y_train_pd
list_c1_percent_id = class_1_percent_samples(y_train_pd,0.5)
# list_c1_percent_features - список 'id' из y_train_pd соотвествующих  
# заданному проценту класса 1
len(list_c1_percent_id)

In [ ]:
# проверим сбалансированность полученного y_train
y_train_pd.loc[list_c1_percent_id]['flag'].value_counts(normalize=True)

*transform_data_torow*, показываеются последние N операций клиента.  
В данных в *date_torow*, собраны последние 25 операций клиентов.

Предварительный анализ показывает, что медиальное значение количества клиентских  
операций равно 7. Поэтому, для начала, будем показывать моделе $n_{last} = 7$   
последних клиентских операций.

Для извлечения из *transform_data_torow_25* необходимого количества последних клиенских  
операций ($n_{last}$) используем функцию features_from_transform_data_torow.

Функция features_from_transform_data_torow примает следующие аргументы:
- $n_{last} = 7$ - необходимое количество последних клиентских операций; 
- $n_{groups} = 8$ - количество признаков в $date$ $features$;
- $N_{last} = 25$ - количество $n_{last}$ операций, показанных в transform_data_torow_25


In [ ]:
# с помощью функции features_from_transform_data_torow извлечем  
# из данных transform_data_torow_25
# признаки сооствествующие 7 последним клиенским операциям
list_n_last_features = features_from_transform_data_torow(7,count_features,25)

In [ ]:
# подготовим данные для обучения
X_train_balanced = X_train_pd.loc[list_c1_percent_id]
y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag']
# сформируем данные для проверики модели
X_train = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas().to_numpy()
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
X_valid = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_valid').to_pandas().to_numpy()
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()

In [ ]:
# посмотрим на структуру данных в X_train_balanced
X_train_balanced.head(2)

In [ ]:
# посмотрим на структуру данных в y_train_balanced
y_train_balanced.head(2)

In [ ]:
# проверим что выборки действительно совпадают по id
X_train_balanced.index.to_list() == y_train_balanced.index.to_list()

#### <span style="color:MediumBlue">Построение модели Logistic Regression</span>

##### <span style="color:MediumSlateBlue">Baseline обучение модели LogisticRegression</span>

In [ ]:
# обучаем модель логистической регрессии
logistic_regression = linear_model.LogisticRegression(random_state=42, max_iter=10000)
logistic_regression.fit(np.array(X_train_balanced)[:,list_n_last_features],np.array(y_train_balanced))

# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_LR_pred_proba = logistic_regression.predict_proba(X_train[:,list_n_last_features])[:,1]
y_valid_LR_pred_proba = logistic_regression.predict_proba(X_valid[:,list_n_last_features])[:,1]

# Делаем предсказание для валидационной выборки
y_valid_LR_pred = logistic_regression.predict(X_valid[:,list_n_last_features])

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_LR_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_LR_pred_proba),3))

print('Основные метрики на тестовом наборе:')
print(metrics.classification_report(y_valid,y_valid_LR_pred))

# time: 1m 12s

<span style="color:Blue">

Выводы:

В сравнение с моделью $LogisticRegression$ построенной   
на несбалансированных данных $transform$ $data$ $torow$:
1. Качество модели по метрики $ROC AUC$ не изменилось.
2. Качество модели по метрики $precision_1$ значительно  
улучшилось. Хотя, способность модели отличать класс 1 от  
класса 0, практически равна 0.
3. На порядок уменьшилось время обучения модели. Это логично,  
велична сбалансированной выборки всего $107000$ $samples$. 

##### <span style="color:MediumSlateBlue">Анализ влияния scaler преобразования на качество и скорость схождения модели</span>

In [ ]:
# формируем список из необходимых scaler
scalers = [MinMaxScaler(),RobustScaler() ,StandardScaler()]

# сформируем списко для заполнения данными результатов анализа
scalers_data = []

# установим ограничение на время обучения модели в 600 секунд (10 минут)
time_out = 600

for scaler in scalers:
        # для того чтобы следить за выполнением цикла введем индикатор
        print('current scaler: ', scaler)

         # сформируем список для заполнения данными текушего scaler
        current_scaler_data = [scaler]
        
        # выполним scaler преобразование
        X_train_s_balanced = scaler.fit_transform(np.array(X_train_balanced)[:,list_n_last_features])
        X_train_s = scaler.transform(X_train[:,list_n_last_features])
        X_valid_s = scaler.transform(X_valid[:,list_n_last_features])

        try:
                # для модуля func_time_out упакуем обучение модели в функцию try_func
                def try_func():
                        logistic_regression = linear_model.LogisticRegression(random_state=42, max_iter=10000)
                        return logistic_regression.fit(X_train_s_balanced,y_train_balanced)
                    
                # фиксируем время начала
                start_time = time.time()
                
                # запускаем обучение модели с ограничением по времени
                logistic_regression = func_timeout.func_timeout(time_out, try_func)
                
                # фиксируем время окончания обучения
                end_time = time.time()
                
                # запишем время обучения модели
                current_scaler_data.append(round(end_time-start_time))

                # Делаем предсказание для обучающей и валидационной выборки
                y_valid_lr_pred = logistic_regression.predict(X_valid_s)
                
                # для метрик ROC AUC делаем предсказание модели для обучающей и валидационной выборки  
                # в виде вероятности принадлежности к классу 1 
                y_train_lr_pred_proba = logistic_regression.predict_proba(X_train_s)[:,1]
                y_valid_lr_pred_proba = logistic_regression.predict_proba(X_valid_s)[:,1]

                #Считаем метрики ROC AUC yна обуающем и валидационном наборе и добавляем их в спискок
                current_scaler_data.append(round(metrics.roc_auc_score(y_train, y_train_lr_pred_proba),3))
                current_scaler_data.append(round(metrics.roc_auc_score(y_valid, y_valid_lr_pred_proba),3))

                #Считаем метрики для класса 0 на валидационном наборе и добавляем их в список
                current_scaler_data.append(round(metrics.recall_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))
                current_scaler_data.append(round(metrics.precision_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))


                #Считаем метрики для класса 1 на валидационном набопе и добавляем их в список
                current_scaler_data.append(round(metrics.recall_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))
                current_scaler_data.append(round(metrics.precision_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))

                # добавляем полученные данные в список scalers_data
                scalers_data.append(current_scaler_data)

                # чтобы подстраховаться на случай ошибки : "The Kernel crashed while executing code"
                # сохраним в локальную папку удачную итерацию
                # для этого из полученных данных сформируем dataframe
                # из полученных данных сформируем dataframe
                scaler_pd =pd.DataFrame(
                        columns=['scaler','time_fit',
                                 'roc_train','roc_valid', 
                                 'recall_0','precision_0',
                                 'recall_1','precision_1'],
                        data = scalers_data)
                # сохраняем dataframe
                scaler_pd.to_csv('data_recovery/scaler_pd.csv',index=False)
               
                # очистим output от прошедшей итерации
                clear_output()

        except func_timeout.FunctionTimedOut:
                # если время обучения привышает лимит
                # запишем в current_scaler_data соотвествующую данные
                current_scaler_data = [scaler,'>30m',0,0,0,0,0,0]
                # добавляем полученные данные в список scalers_data
                scalers_data.append(current_scaler_data)

                # чтобы подстраховаться на случай ошибки : "The Kernel crashed while executing code"
                # сохраним в локальную папку удачную итерацию
                # для этого из полученных данных сформируем dataframe
                # из полученных данных сформируем dataframe
                scaler_pd =pd.DataFrame(
                        columns=['scaler','time_fit',
                                 'roc_train','roc_valid', 
                                 'recall_0','precision_0',
                                 'recall_1','precision_1'],
                        data = scalers_data)
                # сохраняем dataframe
                scaler_pd.to_csv('data_recovery/scaler_pd.csv',index=False)
                
                # очистим output от прошедшей итерации
                clear_output()
                # переходим к следующему scaler
                pass
# очистим output
clear_output()

# сформируем данные соотвествующие лучщему ROC AUC на валидацинном наборе
best_roc_valid_data = scaler_pd[scaler_pd['roc_valid']==scaler_pd['roc_valid'].max()]

# выведем лучший результат на валидационном наборе
print('result:')
print('best scaler on valid:',best_roc_valid_data.iloc[0]['scaler'])
print('ROC AUC on train:', best_roc_valid_data.iloc[0]['roc_train'])
print('ROC AUC on valid:', best_roc_valid_data.iloc[0]['roc_valid'])
print('Time fit for best scaler:', best_roc_valid_data.iloc[0]['time_fit'],' seconds')

# выведем полученный dataframe
scaler_pd

# time: 

<span style="color:Blue">

Выводы:
1. $sceler$ преобразование ни как не повлияло на качество модели.
2. Модель, обученная на данных обработанных $StandardScaler()$ преобразованием,   
самую высокую скорость скохждения.



##### <span style="color:MediumSlateBlue"> Анализ влияния solver оптимизатора на качество и скорость обучения модели</span>

Проверим качество и скорость сходимости модели при разных solver  
Проверку осуществим через цикл  
Чтобы модель не сходилась бесконечно с помощью модуля func_timeout  
поставим ограничение времени схождения в 10 минут

In [ ]:
# подготовим данные для обучения модели

# обьявим scaler
scaler = StandardScaler()
# выполним scaler преобразование
X_train_s_balanced = scaler.fit_transform(np.array(X_train_balanced)[:,list_n_last_features])
X_train_s = scaler.transform(X_train[:,list_n_last_features])
X_valid_s = scaler.transform(X_valid[:,list_n_last_features])

# time: 10s

In [ ]:
# Зададим список solvers
solvers_list = ['lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga']

# сформируем список для заполнения
solvers_data = []

# установим ограничение на время обучения модели в 300 секунд (5 минут)
time_out = 300

for solver in solvers_list:
        # для того чтобы следить за выполнением цикла введем индикатор
        print('current solver: ', solver)

         # сформируем список для заполнения данными текушего solver
        current_solver_data = [solver]
        
        try:
                # для модуля func_time_out упакуем обучение модели в функцию try_func
                def try_func():
                        logistic_regression = linear_model.LogisticRegression(solver= solver,
                                                                  random_state=42, 
                                                                  max_iter=10000)
                        return logistic_regression.fit(X_train_s_balanced,y_train_balanced)
                    
                # фиксируем время начала
                start_time = time.time()
                
                # запускаем обучение модели с ограничением по времени
                logistic_regression = func_timeout.func_timeout(time_out, try_func)
                
                # фиксируем время окончания обучения
                end_time = time.time()
                
                # запишем время обучения модели
                current_solver_data.append(round(end_time-start_time))

                # Делаем предсказание для обучающей и валидационной выборки
                y_valid_lr_pred = logistic_regression.predict(X_valid_s)
                
                # для метрик ROC AUC делаем предсказание модели для обучающей и валидационной выборки  
                # в виде вероятности принадлежности к классу 1 
                y_train_lr_pred_proba = logistic_regression.predict_proba(X_train_s)[:,1]
                y_valid_lr_pred_proba = logistic_regression.predict_proba(X_valid_s)[:,1]

                #Считаем метрики ROC AUC yна обуающем и валидационном наборе и добавляем их в спискок
                current_solver_data.append(round(metrics.roc_auc_score(y_train, y_train_lr_pred_proba),3))
                current_solver_data.append(round(metrics.roc_auc_score(y_valid, y_valid_lr_pred_proba),3))

                #Считаем метрики для класса 0 на валидационном наборе и добавляем их в список
                current_solver_data.append(round(metrics.recall_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))
                current_solver_data.append(round(metrics.precision_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))


                #Считаем метрики для класса 1 на валидационном набопе и добавляем их в список
                current_solver_data.append(round(metrics.recall_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))
                current_solver_data.append(round(metrics.precision_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))

                # запишем данные полученные на одной итерации
                solvers_data.append(current_solver_data)

                # чтобы подстраховаться на случай ошибки : "The Kernel crashed while executing code"
                # сохраним в локальную папку удачную итерацию
                # для этого из полученных данных сформируем dataframe
                # из полученных данных сформируем dataframe
                solvers_pd =pd.DataFrame(
                        columns=['solver','time_fit',
                                 'roc_train','roc_valid', 
                                 'recall_0','precision_0',
                                 'recall_1','precision_1'],
                        data = solvers_data)
                # сохраняем dataframe
                solvers_pd.to_csv('data_recovery/solvers_pd.csv',index=False)
               
                # очистим output от прошедшей итерации
                clear_output()

        except func_timeout.FunctionTimedOut:
                # если время обучения привышает лимит
                # запишем в current_solver_data соотвествующую данные
                current_solver_data = [solver,'>10m',0,0,0,0,0,0]
                # добавляем полученные данные в список solvers_data
                solvers_data.append(current_solver_data)

                # чтобы подстраховаться на случай ошибки : "The Kernel crashed while executing code"
                # сохраним в локальную папку удачную итерацию
                # для этого из полученных данных сформируем dataframe
                # из полученных данных сформируем dataframe
                solvers_pd =pd.DataFrame(
                        columns=['solver','time_fit',
                                 'roc_train','roc_valid', 
                                 'recall_0','precision_0',
                                 'recall_1','precision_1'],
                        data = solvers_data)
                # сохраняем dataframe
                solvers_pd.to_csv('data_recovery/solvers_pd.csv',index=False)
                
                # очистим output от прошедшей итерации
                clear_output()
                # переходим к следующему solver
                pass
# очистим output
clear_output()

# сформируем данные соотвествующие лучщему ROC AUC на валидацинном наборе
best_roc_valid_data = solvers_pd[solvers_pd['roc_valid']==solvers_pd['roc_valid'].max()]

# выведем лучший результат на валидационном наборе
print('result:')
print('best solver on valid:',best_roc_valid_data.iloc[0]['solver'])
print('ROC AUC on train:', best_roc_valid_data.iloc[0]['roc_train'])
print('ROC AUC on valid:', best_roc_valid_data.iloc[0]['roc_valid'])
print('Time fit for best solver:', best_roc_valid_data.iloc[0]['time_fit'],' seconds')

# выведем полученный dataframe
solvers_pd

# time: 11m 35s

<span style="color:Blue">

Вывод:

1. Выбор $solver$ оптимизатора не влияет на качество модели;
2. $sag$ $saga$ не смогли обучиться за 10 минут.

##### <span style="color:MediumSlateBlue">Подбор гиперпараметров модели</span>

In [ ]:
# для выбора sceler преобразвания на понадобится словарь
dict_scalers ={
    'MinMaxScaler':MinMaxScaler(),
    'RobustScaler':RobustScaler(),
    'StandardScaler':StandardScaler(),
}
# определим пространство признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'date'
count_features = dict_spaces[feature_space]

# настроим оптимизацию гипер параметров
def optuna_lg(trial):
  # задаем пространства поиска гиперпараметров
  C = trial.suggest_float('C',0.01,1)
  solver = trial.suggest_categorical('solver', ['lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky'])
  class_0_weight = trial.suggest_float('class_0_weight',0.4,1)
  class_1_weight = trial.suggest_float('class_1_weight',0.01,0.5)
  n_last = trial.suggest_int('n_last', 10, 25,step = 1)
  class_1_percent = trial.suggest_float('class_1_percent',0.01,1)
  scaler = trial.suggest_categorical('scaler', ['MinMaxScaler', 'RobustScaler','StandardScaler'])
  random_state = trial.suggest_int('random_state', 1, 1000000)

  # создаем модель
  optuna_log_reg = linear_model.LogisticRegression(
      solver=solver,
      C=C,
      class_weight={0:class_0_weight,1:class_1_weight},
      random_state=random_state,
      max_iter=10000)

  # с помощью функции features_from_transform_data_torow извлечем  
  # из данных transform_data_torow
  # признаки сооствествующие n_last последним клиенским операциям
  list_n_last_features = features_from_transform_data_torow(n_last,count_features,25)

  # обьявим scaler
  scaler = dict_scalers[scaler]

  # загружаем обучующие наборы
  X_train_pd = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas()
  y_train_pd = pd.read_csv('target/target_train.csv')

  # поготовим данные y_train_pd к работе с функцией class_1_percent_samples
  y_train_pd.set_index('id',drop=False,inplace=True)

  # подготовим данные для обучения модели
  # с помощью функции class_1_percent_samples зададим долю 
  # класса 1
  list_c1_percent_id = class_1_percent_samples(y_train_pd,class_1_percent,random_state=random_state)[:1000000]

  # сформируем данные для обучения базовой модели
  X_train_s_balanced = scaler.fit_transform(np.array(X_train_pd.loc[list_c1_percent_id])[:,list_n_last_features])
  y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

  # освободим память от "тяжелых" и ненужных файлов
  del X_train_pd, y_train_pd
  gc.collect()

  # я не воспользовался параметром timeout метода optimize потому, что  
  # optimize отставливает поиск параметров, если время обучения 
  # превышает timeout. Мне нужно чтобы поиск параметров продолжился.
  try:
    # для модуля func_time_out упакуем обучение модели в функцию try_func
    def try_func():
            return optuna_log_reg.fit(X_train_s_balanced, y_train_balanced)
    # обучаем модель с ограничением по времени 10 минут (600 секунд)
    optuna_log_reg = func_timeout.func_timeout(600, try_func)

    # удаляем крупные файлы чтобы высвободить память 
    del X_train_s_balanced,y_train_balanced
    gc.collect()

    # подгружаем данные для тестирования
    X_train = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas().to_numpy()[:,list_n_last_features]
    y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
    X_valid = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_valid').to_pandas().to_numpy()[:,list_n_last_features]
    y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()

    # сформируем данные для проверки модели
    X_train_s = scaler.transform(X_train)
    X_valid_s = scaler.transform(X_valid)

    # удаляем крупные файлы чтобы высвободить память 
    del X_train, X_valid
    gc.collect()

    # делаем предсказание на обучающем и валидационном наборе и считаем метрики
    roc_train = metrics.roc_auc_score(y_train, optuna_log_reg.predict_proba(X_train_s)[:,1])
    roc_valid = metrics.roc_auc_score(y_valid, optuna_log_reg.predict_proba(X_valid_s)[:,1])
    f1_score = metrics.f1_score(y_valid, optuna_log_reg.predict(X_valid_s))
    recall_1 = metrics.recall_score(y_valid, optuna_log_reg.predict(X_valid_s),average=None,zero_division=0)[1]
    precision_1 = metrics.precision_score(y_valid, optuna_log_reg.predict(X_valid_s),average=None,zero_division=0)[1]

    del X_train_s, X_valid_s, y_train, y_valid
    gc.collect()

  except func_timeout.FunctionTimedOut:
    # освободим память от "тяжелых" и ненужных файлов
    del X_train_s_balanced, y_train_balanced
    gc.collect()

    # обьявим метрики пустыми значениями
    roc_train = 0
    roc_valid = 0
    f1_score = 0
    recall_1 = 0
    precision_1 = 0
    pass

  return roc_train, roc_valid, f1_score, recall_1, precision_1

In [ ]:
# cоздаем объект исследования
# чтобы модель не переобучалась минимизируем roc_train 
# и максимизируем roc_valid
optuna_study_lg = optuna.create_study(study_name='LogisticRegression_'+feature_space+'_torow', 
                               directions=['maximize','maximize','maximize','maximize','maximize'], 
                               sampler=optuna.samplers.TPESampler(),
                               pruner='Hyperband',
                               storage='sqlite:///optuna_studies.db',
                               load_if_exists=True)
# на случай удаления 
# optuna.delete_study(study_name='LogisticRegression_'+feature_space+'_torow', storage='sqlite:///optuna_studies.db')

In [ ]:
# ищем лучшую комбинацию гиперпараметров n_trials раз
optuna_study_lg.optimize(optuna_lg, n_trials=300)

In [ ]:
# из полученного результат соврмируем Data Frame
optuna_study_lg_pd = optuna_study_lg.trials_dataframe().dropna()
# переименуем столбы
optuna_study_lg_pd.rename(columns={
    'values_0': 'roc_train',
    'values_1': 'roc_valid',
    'values_2': 'f1_score',
    'values_3': 'recall_1',
    'values_4': 'precision_1'
},inplace=True)
# Выделим время потраченное на обучение модели
optuna_study_lg_pd.head(5)

In [ ]:
# покаждем статистику обучения
print('Максимальное значение метрики ROC AUC на валидационном наборе:',round(optuna_study_lg_pd['roc_valid'].max(),3))
print('Среднее значение метрики ROC AUC на валидационном наборе:',round(optuna_study_lg_pd['roc_valid'].mean(),3))
print('Максимальное значение метрики f1_score на валидационном наборе:',round(optuna_study_lg_pd['f1_score'].max(),3))
print('Среднее значение метрики f1_score на валидационном наборе:',round(optuna_study_lg_pd['f1_score'].mean(),3))
print('Максимальное значение метрики recall_1 на валидационном наборе:',round(optuna_study_lg_pd['recall_1'].max(),3))
print('Среднее значение метрики recall_1 на валидационном наборе:',round(optuna_study_lg_pd['recall_1'].mean(),3))
print('Максимальное значение метрики precision_1 на валидационном наборе:',round(optuna_study_lg_pd['precision_1'].max(),3))
print('Среднее значение метрики precision_1 на валидационном наборе:',round(optuna_study_lg_pd['precision_1'].mean(),3))

<span style="color:Blue">
Вывод:

Модель плохо отличает класс 1 от класса 0.  
Попробуем определить от чего зависит велечина метрики precision_1

In [ ]:
# построим график важности гиперпарметров
optuna.visualization.plot_param_importances(optuna_study_lg, target_name='Score')

<span style="color:Blue">
Вывод:

Судя по $Hyperparameter$ $Importances$, на качество модели влияет $class$ $1$ $percent$.  
Попробуем выявить характер влияния.

In [ ]:
# Построим зависимость метрики precision_1 от гипер параметров

fig= px.scatter(
    data_frame=optuna_study_lg_pd,
    x='precision_1', #ось абсцисс
    y=['recall_1','f1_score', 
        'params_C','params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight', 'params_n_last',
    ], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision_1',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

<span style="color:Blue">

Вывод:  

1. Прослеживается зависимость между метриками $precision_1$ и $recall_1$.  
Очевидно, что модель не сможет достичь знаечний $precision_1>0.08$,   
при этом не опустив значения метрики $recall_1$ в 0.

2. Прослеживается зависимость между метриками $precision_1$ и $f_1$ - $score$.  
Максимальное значение метрики $f_1$ - $score$, наблюдается при значении  
$precision_1 \approx 0.06$

3. Обратим внимание на параметр $С$.  
Ярко выраженного влияния на метрику $precision_1$, не наблюдается.  

4. Обратим внимание на параметр $class$ $0$ $weigth$,  
Для более высокого значения метрики $precision_1$, характерны  
следующие значения гиперпараметра:
- $class$ $0$ $weigth \in [0.5;1]$.

5. Обратим внимание на параметр $class$ $1$ $weigth$,  
Для более высокого значения метрики $precision_1$, характерны  
следующие значения гиперпараметра:
- $class$ $0$ $weigth \in [0.2;0.6]$.

6. Обратим внимание на параметр $class$ $1$ $percent$,  
Для более высокого значения метрики $precision_1$, характерны  
следующие значения гиперпараметра:
- $class$ $0$ $weigth \in [0.2;0.7]$.

7. Обратим внимание на параметр $n$ $last$.  
Ярко выраженного влияния на метрику $precision_1$, не наблюдается.

In [ ]:
# Построим зависимость метрики precision_1 от гипер параметров

fig= px.scatter(
    data_frame=optuna_study_lg_pd,
    x='f1_score', #ось абсцисс
    y=['params_C','params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight', 'params_n_last',
    ], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision_1',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

<span style="color:Blue">

Вывод:  

1. Обратим внимание на параметр $С$.  
Ярко выраженного влияния на метрику $f_1$ - $score$, не наблюдается.  

2. Обратим внимание на параметр $class$ $0$ $weigth$,  
Для более высокого значения метрики $f_1$ - $score$, характерны  
следующие значения гиперпараметра:
- $class$ $0$ $weigth \in [0.5;1]$.

3. Обратим внимание на параметр $class$ $1$ $weigth$,  
Для более высокого значения метрики $f_1$ - $score$, характерны  
следующие значения гиперпараметра:
- $class$ $1$ $weigth \in [0.2;0.6]$.

4. Обратим внимание на параметр $class$ $1$ $percent$,  
Для более высокого значения метрики $f_1$ - $score$, характерны  
следующие значения гиперпараметра:
- $class$ $1$ $percent \in [0.2;0.7]$.

5. Обратим внимание на параметр $n$ $last$.  
Ярко выраженного влияния на метрику $f_1$ - $score$, не наблюдается.


In [ ]:
# поготовим данные для визуализации
# выбираем те точки с оптимальными значениями метрик
fig_data_frame=optuna_study_lg_pd[(optuna_study_lg_pd['precision_1']>0.4*optuna_study_lg_pd['precision_1'].max())&
                                  (optuna_study_lg_pd['recall_1']>0.3*optuna_study_lg_pd['recall_1'].max())&
                                  (optuna_study_lg_pd['roc_valid']>0.95*optuna_study_lg_pd['roc_valid'].max())
                                  ]
fig = go.Figure()
fig.add_trace(go.Histogram(x=fig_data_frame['params_scaler'].round(1),name='scaler'))
fig.add_trace(go.Histogram(x=fig_data_frame['params_solver'].round(1),name='solver'))


fig.update_traces(opacity=0.75)
fig.update_layout(
    title ={
        'text':'ТОП "лучших" значений параметров по версии метрик precision и recall класса 1', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =1000,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    yaxis_title='Количество ')

fig.show()

<span style="color:Blue">
Вывод:

Можно сделать вывод, что для получения приемлемых значений метрик,  
оптимизотор использует всю сетку пространства параметров $scaler$ и $solver$.  
Чаще всего использовался $MinMaxScaler$ и $newton-cg$.

In [ ]:
# Построим зависимость метрики precision_1 от гипер параметров

fig= px.scatter(
    data_frame=optuna_study_lg_pd,
    x='roc_valid', #ось абсцисс
    y=['params_C','params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight', 'params_n_last',
    ], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision_1',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

<span style="color:Blue">

Вывод:  

1. Обратим внимание на параметр $С$.  
Для более высокого значения метрики $roc$ $valid$, характерны  
следующие значения гиперпараметра:
- $C \in [0;0.2]$.

2. Обратим внимание на параметр $class$ $0$ $weigth$,  
Для более высокого значения метрики $roc$ $valid$, характерны  
следующие значения гиперпараметра:
- $class$ $0$ $weigth \in [0.5;1]$.

3. Обратим внимание на параметр $class$ $1$ $weigth$,  
Для более высокого значения метрики $roc$ $valid$, характерны  
следующие значения гиперпараметра:
- $class$ $1$ $weigth \in [0.2;0.6]$.

4. Обратим внимание на параметр $class$ $1$ $percent$,  
Для более высокого значения метрики $roc$ $valid$, характерны  
следующие значения гиперпараметра:
- $class$ $1$ $percent \in [0.2;0.7]$.

5. Обратим внимание на параметр $n$ $last$.  
Ярко выраженного влияния на метрику $roc$ $valid$, не наблюдается.


In [ ]:
# Визуализируем метрики полученные при optuna оптимизации
fig = px.scatter(
    data_frame=optuna_study_lg_pd[
                                (optuna_study_lg_pd['roc_valid']>0.994*optuna_study_lg_pd['roc_valid'].max())& 
                                (optuna_study_lg_pd['precision_1']>0.44*optuna_study_lg_pd['precision_1'].max())&
                                (optuna_study_lg_pd['recall_1']>0.7*optuna_study_lg_pd['recall_1'].max())],
    x='number', #ось абсцисс
    y=['roc_train','roc_valid','recall_1','precision_1'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость качества модели от trials optuna', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='trials optuna',
    yaxis_title='Величина метрики')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

<span style="color:Blue">

Вывод:  

Их всех выбираем точку $number =63$.   
В этой точке одно из самых больших значений $ROC AUC$.  
Посмотрим каким значениям гиперпараметров соотвествует выбранная точка.

In [ ]:
# определим номер лучшге варианта
best_optuna_number = 63

# сформируем словарь лучших гипирпарметров RandomForestClassifier
best_param_lr_torow = {
    'C' : round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_C'].iloc[0],3),
    'class_weight' : {0:round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_class_0_weight'].iloc[0],3),
                      1:round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_class_1_weight'].iloc[0],3)},
    }
# создадим перменные
best_n_last = int(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_n_last'].iloc[0])
best_scaler = optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_scaler'].iloc[0]
best_class_1_percent = optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_class_1_percent'].iloc[0]
best_random_state = int(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_random_state'].iloc[0])
# Выведем принятые наилучшие праметры
print('best C:',best_param_lr_torow['C'])
print('best class 0 weight:',best_param_lr_torow['class_weight'][0])
print('best class 1 weight:',best_param_lr_torow['class_weight'][1])

print('best class 1 percent:',round(best_class_1_percent,3))
print('best scaler:',best_scaler)
print('best n_last:',round(best_n_last,3))
print('best best random state:',best_random_state)
print('time for best train:',round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['duration'].iloc[0].seconds/60),'minutes')

print()
print('ROC AUC на обучающем наборе:', round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['roc_train'].iloc[0],3))
print('ROC AUC на валидационном наборе:', round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['roc_valid'].iloc[0],3))
print('precision класса 1:', round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['precision_1'].iloc[0],3))
print('recall класса 1:', round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['recall_1'].iloc[0],3))

##### <span style="color:MediumSlateBlue">Обучение модели с лучшими параметрами</span>

In [ ]:
# для выбора sceler преобразвания на понадобится словарь
dict_scalers ={
    'MinMaxScaler':MinMaxScaler(),
    'RobustScaler':RobustScaler(),
    'StandardScaler':StandardScaler(),
}

# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'date'
count_features = dict_spaces[feature_space]

# с помощью функции features_from_transform_data_torow извлечем  
# из данных transform_data_torow_25
# признаки сооствествующие n_last последним клиенским операциям
list_n_last_features = features_from_transform_data_torow(best_n_last,count_features,25)

# обьявим scaler
scaler = dict_scalers[best_scaler]

# загружаем обучующие наборы
X_train_pd = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas()
y_train_pd = pd.read_csv('target/target_train.csv')

# поготовим данные y_train_pd к работе с функцией class_1_percent_samples
y_train_pd.set_index('id',drop=False,inplace=True)

# подготовим данные для обучения модели
# с помощью функции class_1_percent_samples зададим долю 
# класса 1
list_c1_percent_id = class_1_percent_samples(y_train_pd,best_class_1_percent,random_state=best_random_state)[:1000000]

# сформируем данные для обучения базовой модели
X_train_s_balanced = scaler.fit_transform(np.array(X_train_pd.loc[list_c1_percent_id])[:,list_n_last_features])
y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

# освободим память от "тяжелых" и ненужных файлов
del X_train_pd, y_train_pd
gc.collect()


# обучаем модель LogisticRegression с наилучшеми параметрами
logistic_regression = linear_model.LogisticRegression(
        **best_param_lr_torow,
        random_state=best_random_state,
        max_iter=10000)
logistic_regression.fit(X_train_s_balanced,y_train_balanced)

# удаляем крупные файлы чтобы высвободить память 
del X_train_s_balanced,y_train_balanced
gc.collect()

# подгружаем данные для тестирования
X_train = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas().to_numpy()[:,list_n_last_features]
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
X_valid = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_valid').to_pandas().to_numpy()[:,list_n_last_features]
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()
    
# сформируем данные для проверки модели
X_train_s = scaler.transform(X_train)
X_valid_s = scaler.transform(X_valid)

# удаляем крупные файлы чтобы высвободить память 
del X_train, X_valid
gc.collect()

# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_lr_pred_proba = logistic_regression.predict_proba(X_train_s)[:,1]
y_valid_lr_pred_proba = logistic_regression.predict_proba(X_valid_s)[:,1]

# Делаем предсказание для валидационной выборки
y_valid_lr_pred = logistic_regression.predict(X_valid_s)

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_lr_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_lr_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_valid,y_valid_lr_pred,zero_division=0))

# time: 2m 40s

<span style="color:Blue">
Вывод:  

Модель $LogisticRegression$, обученная на сбалансированных данных   
$transform$ $data$ $torow$ имеет такое же качество как и модель  
модель $LogisticRegression$, обученная на не тех же не сбалансированных  
данных.

Подбор гиперпараметров модели показал, что модель $LogisticRegression$   
имеет низкую пресказательную способность на данных $transform$ $data$ $torow$.

##### <span style="color:MediumSlateBlue">Анализ влияния порога классификации на качество модели</span>

In [ ]:
# чтобы проше обращаться к каждому обьекту в данных
# преобразуем y_valid_LR_pred к Series типу
y_valid_lr_pred_proba = pd.Series(y_valid_lr_pred_proba)

# формируем список из необходимых значений thresholds
thresholds = np.arange(0.01, 1, 0.01)

# сформируем списко для заполнения данными результатов анализа
threshold_data = []


for threshold in thresholds:
        # сформируем список для заполнения данными текушего threshold
        threshold_current_data = [threshold]

        #клиентов, для которых вероятность дефолта > threshold, относим к классу 1
        #в противном случае — к классу 0
        y_valid_lr_pred = y_valid_lr_pred_proba.apply(lambda x: 1 if x>threshold else 0)


        #Считаем метрики для класса 0 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.precision_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.f1_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))

        #Считаем метрики для класса 1 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.precision_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.f1_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))

        # добавляем полученные данные в список threshold_data
        threshold_data.append(threshold_current_data)

        # из полученных данных сформируем dataframe
        thresholds_pd =pd.DataFrame(
        columns=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'],
        data = threshold_data)

# time: 17s

In [ ]:
# Визуализируем метрики на валидационном наборе при различных threshold
fig_threshold = px.line(
    data_frame=thresholds_pd,
    x='threshold', #ось абсцисс
    y=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'], #ось ординат
)
fig_threshold.update_layout(
    title ={
        'text':'Зависимость качества модели от threshold', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Порог классификации threshold',
    yaxis_title='Величина метрики')
fig_threshold.update_xaxes(showspikes=True)
fig_threshold.update_yaxes(showspikes=True)

fig_threshold.show()

Метрика $precision$ показывает на сколько сильно ошиблась модель при определиние класса.  
Чем выше метрика, тем меньше ошиблась модель. 

$$precision = \frac{TP}{TP+FP}$$

Метрика $recall$ показывает способность модели найти класс.  
Чем выше метрика, тем выше способность модели находить класс.  

$$recall = \frac{TP}{TP+FN}$$

Метрика $f1-score$ показывает баланс между $precision$ и $recall$ 

$threshold$- порог класификации модели.  
Все объекты для которых $у_{\text{proba pred}} < threshold$, модель относит к класс 0.  
Соотвественно, если $у_{\text{proba pred}}  > threshold$ модель относит обьект к классу 1.  
Для идельаной модели $threshold$, является границей между областями класса 0 и класса 1.  

<span style="color:Blue">

Выводы по результам анализа:  

Для улучшения качества модели по метрике $ROC AUC$, нет смысла сдвигать порог   
классификации. Но зависимость метрик качества модели от $threshold$, может рассказать о  
спосбоности модели искать класс 1 и класс 0.

При фокусировке модели на классе 1 (более низкий $threshold$):  
- на классе 1 метрика $precision$ уменьшается, метрика $recall$ растет;  
- на классе 0 метрика $precision$ растет, метрика $recall$ уменьшается.  

При фокусировке модели на классе 0 (более высокий $threshold$):  
- на классе 1 метрика $precision$ увеличивается, метрика $recall$ уменьшается;  
- на классе 0 метрика $precision$ уменьшается, метрика $recall$ растет.  

Обьсняется тем, что в условия дисбаланса модель научилась хорошо определять класс 0 и плохо класс 1.  

При фокусировке модели на классе 1 (более низкий $threshold$), все больше обьектов отнесены к классу 1.   
Так как $precision$ класса 1 уменьшается, значит в области класса 1 стало больше обектов класса 0,  
что явлется закономерным для "идеальной" модели.  
Однако при этом, $recall$ класса 1 растет, значит в добавленных обьектах также присутвует класс 1.  
Это как раз указывает на то, что модель не нучилась находить среди класса 0 класс 1. В области с     
высокой вероятностью класса 0 (более низкий $threshold$), находятся обьекты класса 1.  

Фокусировка на классе 0 (более высокий $threshold$), заставляет модель все больше обьектов из области класса 1  
относить к классу 0. При этом увеличение метрики $precision$ класса 1 говорит о том, что в области класса 1     
становится меньше 0, а значит модель умеет среди класса 1, находить класс 0.  


Обратим внимание на $threshold=0.66$, в этой точке находится баланс межу метриками качества модели:
$$precision_1 = recall_1 = \text{f1-score}_1 = 0.062$$
$$precision_0 = recall_0 = \text{f1-score}_0 = 0.995$$

Для модели с лучшим качеством эти две точки должны находится максимально близко к друг другу и иметь  
высокое значение метрик $precision_1$ и $recall_1$.  
Для класса 0 это условие выполнется, что также говорит о том, что модель научилась искать этот класс.   
Для класса 1 это условие не выполнется, что также говорит о том, что модель плохо ищет класс 1.

##### <span style="color:MediumSlateBlue">Построение ROC-кривой выбранной модели на тестовом наборе</span>

In [ ]:
# определим пространство признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'date'
count_features = dict_spaces[feature_space]

In [ ]:
# подгружаем данные для тестирования
X_train = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas().to_numpy()[:,list_n_last_features]
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
X_valid = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_valid').to_pandas().to_numpy()[:,list_n_last_features]
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()
X_test = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_test').to_pandas().to_numpy()[:,list_n_last_features]
y_test = pd.read_csv('target/target_test.csv')['flag'].to_numpy()

In [ ]:
# выполним scaler преобразование
X_train_s = scaler.transform(X_train)
X_valid_s = scaler.transform(X_valid)
X_test_s = scaler.transform(X_test)

print("Размеры обучающего набора",X_train_s.shape)
print("Размеры валидационного набора",X_valid_s.shape)
print("Размеры тестового набора",X_test_s.shape)
# time: 1m 43s

In [ ]:
# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_lr_pred_proba = logistic_regression.predict_proba(X_train_s)[:,1]
y_valid_lr_pred_proba = logistic_regression.predict_proba(X_valid_s)[:,1]
y_test_lr_pred_proba = logistic_regression.predict_proba(X_test_s)[:,1]

# Делаем предсказание для валидационной выборки
y_test_lr_pred = logistic_regression.predict(X_test_s)

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_lr_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_lr_pred_proba),3))
print('ROC AUC на тестовом наборе', round(metrics.roc_auc_score(y_test, y_test_lr_pred_proba),3))

print('Основные метрики на тестовом наборе:')
print(metrics.classification_report(y_test,y_test_lr_pred,zero_division=0))

In [ ]:
# Для расчета значений TPR и FPR используем функцию roc_curve
fpr_train, tpr_train, _ = metrics.roc_curve(y_train, y_train_lr_pred_proba)
fpr_valid, tpr_valid, _ = metrics.roc_curve(y_valid, y_valid_lr_pred_proba)
fpr_test, tpr_test, _ = metrics.roc_curve(y_test, y_test_lr_pred_proba)

# рассчитаем площадь под кривой
auc_train = round(metrics.roc_auc_score(y_train, y_train_lr_pred_proba),3)
auc_valid = round(metrics.roc_auc_score(y_valid, y_valid_lr_pred_proba),3)
auc_test = round(metrics.roc_auc_score(y_test, y_test_lr_pred_proba),3)

trace_train = go.Scatter(x=fpr_train, y=tpr_train, mode='lines', name='AUC_train = %0.2f' % auc_train,
                   line=dict(color='red', width=2),
                   hovertemplate='train<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_train])

trace_valid = go.Scatter(x=fpr_valid, y=tpr_valid, mode='lines', name='AUC_valid = %0.2f' % auc_valid,
                   line=dict(color='green', width=2),
                   hovertemplate='valid<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_valid])

trace_test = go.Scatter(x=fpr_test, y=tpr_test, mode='lines', name='AUC_test = %0.2f' % auc_test,
                   line=dict(color='darkorange', width=2),
                   hovertemplate='test<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_test])


reference_line = go.Scatter(x=[0,1], y=[0,1], mode='lines', name='Reference Line',
                            line=dict(color='navy', width=2, dash='dash'))

fig_roc = go.Figure(data=[trace_train,trace_valid,trace_test,reference_line])

fig_roc.update_layout(
    title ={
        'text':'ROC-кривая', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 1350,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate')

fig_roc.update_xaxes(showspikes=True)
fig_roc.update_yaxes(showspikes=True)


fig_roc.show()

<span style="color:Blue">

Вывод по ROC-кривой:
1. Модель выдает сталибильное качество на всех наборах, а значит  
не переобучена - разброс ($variance$) не большой.  
2. Смещение ($bias$) модели относительно большое. Качество чуть лучше, чем  
случайное угадываение.
На тестовом наборе полученно качество $ROC AUC = 0.58$.

#### <span style="color:MediumBlue">Построение модели Random Forest Classifier</span>

##### <span style="color:MediumSlateBlue">Baseline обучение модели RandomForestClassifier</span>

In [ ]:
# обучаем модель логистической регрессии
# чтобы модель пыталась найти связь во всех признаках
random_forest = RandomForestClassifier(random_state=42, n_jobs=-1)
random_forest.fit(np.array(X_train_balanced)[:,list_n_last_features],np.array(y_train_balanced))

# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_rf__pred_proba = random_forest.predict_proba(X_train[:,list_n_last_features])[:,1]
y_valid_rf_pred_proba = random_forest.predict_proba(X_valid[:,list_n_last_features])[:,1]

# Делаем предсказание для валидационной выборки
y_valid_rf_pred = random_forest.predict(X_valid[:,list_n_last_features])

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_rf__pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_rf_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_valid,y_valid_rf_pred))

# time: 2m 5s

<span style="color:Blue">

Выводы:

1. Модель подает признаки переобучения.
2. Качество модели по метрики $roc$ $valid$ чуть лучше, чем у модели   
$Logistic$ $Regression$.


##### <span style="color:MediumSlateBlue">Подбор гиперпараметров модели</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'date'
count_features = dict_spaces[feature_space]

# настроим оптимизацию гипер параметров
def optuna_rf(trial):
  # задаем пространства поиска гиперпараметров
  # задаем пространства поиска гиперпараметров
  n_estimators = trial.suggest_int('n_estimators', 50, 150,step = 5)
  criterion = trial.suggest_categorical('criterion',['gini', 'entropy', 'log_loss'])
  max_depth = trial.suggest_int('max_depth', 9, 20,step = 1)
  min_samples_split = trial.suggest_int('min_samples_split', 5, 15,step = 1)
  min_samples_leaf = trial.suggest_int('min_samples_leaf', 5, 25,step = 1)
  max_features = trial.suggest_float('max_features',0.1,1)
  max_samples = trial.suggest_float('max_samples',0.1,1)
  class_0_weight = trial.suggest_float('class_0_weight',0.2,0.6)
  class_1_weight = trial.suggest_float('class_1_weight',0.4,1)
  n_last = trial.suggest_int('n_last', 5, 25,step = 1)
  class_1_percent = trial.suggest_float('class_1_percent',0.2,0.6)
  random_state = trial.suggest_int('random_state', 1, 1000000)


  # создаем модель
  optuna_random_forest = RandomForestClassifier(
      n_estimators = n_estimators, 
      criterion = criterion,
      max_depth = max_depth,
      min_samples_split = min_samples_split,  
      min_samples_leaf = min_samples_leaf,
      max_features=max_features,
      max_samples=max_samples,
      class_weight={0:class_0_weight,1:class_1_weight},
      n_jobs=-1,
      random_state=random_state)

  # с помощью функции features_from_transform_data_torow извлечем  
  # из данных transform_data_torow
  # признаки сооствествующие n_last последним клиенским операциям
  list_n_last_features = features_from_transform_data_torow(n_last,count_features,25)

  # загружаем обучующие наборы
  X_train_pd = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas()
  y_train_pd = pd.read_csv('target/target_train.csv')

  # поготовим данные y_train_pd к работе с функцией class_1_percent_samples
  y_train_pd.set_index('id',drop=False,inplace=True)

  # подготовим данные для обучения модели
  # с помощью функции class_1_percent_samples зададим долю 
  # класса 1
  list_c1_percent_id = class_1_percent_samples(y_train_pd,class_1_percent,random_state=random_state)[:1000000]

  # сформируем данные для обучения базовой модели
  X_train_balanced = np.array(X_train_pd.loc[list_c1_percent_id])[:,list_n_last_features]
  y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

  # освободим память от "тяжелых" и ненужных файлов
  del X_train_pd, y_train_pd
  gc.collect()

  # я не воспользовался параметром timeout метода optimize потому, что  
  # optimize отставливает поиск параметров, если время обучения 
  # превышает timeout. Мне нужно чтобы поиск параметров продолжился.
  try:
    # для модуля func_time_out упакуем обучение модели в функцию try_func
    def try_func():
            return optuna_random_forest.fit(X_train_balanced,y_train_balanced)
    # обучаем модель с ограничением по времени 10 минут (600 секунд)
    optuna_random_forest = func_timeout.func_timeout(600, try_func)

    # удаляем крупные файлы чтобы высвободить память 
    del X_train_balanced, y_train_balanced
    gc.collect()

    # подгружаем данные для тестирования
    X_train = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas().to_numpy()[:,list_n_last_features]
    y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
    X_valid = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_valid').to_pandas().to_numpy()[:,list_n_last_features]
    y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()

    # делаем предсказание на обучающем и валидационном наборе и считаем метрики
    roc_train = metrics.roc_auc_score(y_train, optuna_random_forest.predict_proba(X_train)[:,1])
    roc_valid = metrics.roc_auc_score(y_valid, optuna_random_forest.predict_proba(X_valid)[:,1])
    f1_score = metrics.f1_score(y_valid, optuna_random_forest.predict(X_valid))
    recall_1 = metrics.recall_score(y_valid, optuna_random_forest.predict(X_valid),average=None,zero_division=0)[1]
    precision_1 = metrics.precision_score(y_valid, optuna_random_forest.predict(X_valid),average=None,zero_division=0)[1]

    # удаляем крупные файлы чтобы высвободить память 
    del X_train, X_valid, y_train, y_valid 
    gc.collect()

  except func_timeout.FunctionTimedOut:
    # освободим память от "тяжелых" и ненужных файлов
    del X_train_balanced, y_train_balanced
    gc.collect()

    # обьявим метрики пустыми значениями
    roc_train = 0
    roc_valid = 0
    f1_score = 0
    recall_1 = 0
    precision_1 = 0
    pass

  return roc_train, roc_valid, f1_score, recall_1, precision_1

In [ ]:
# так как Random Forest склонен к переобучению 
# попробуем направить optimize в сторону уменьшения roc_train
# cоздаем объект исследования
optuna_study_rf = optuna.create_study(study_name='RandomForestClassifier_'+feature_space+'_torow', 
                               directions=['minimize','maximize','maximize','maximize','maximize'], 
                               sampler=optuna.samplers.TPESampler(),
                               pruner='Hyperband',
                               storage='sqlite:///optuna_studies.db',
                               load_if_exists=True)
# на случай удаления 
# optuna.delete_study(study_name='RandomForestClassifier_'+feature_space+'_torow', storage='sqlite:///optuna_studies.db')

In [ ]:
# ищем лучшую комбинацию гиперпараметров n_trials раз
optuna_study_rf.optimize(optuna_rf, n_trials=300)

In [ ]:
# из полученного результат соврмируем Data Frame
optuna_study_rf_pd = optuna_study_rf.trials_dataframe().dropna()
# переименуем столбы
optuna_study_rf_pd.rename(columns={
    'values_0': 'roc_train',
    'values_1': 'roc_valid',
    'values_2': 'f1_score',
    'values_3': 'recall_1',
    'values_4': 'precision_1'
},inplace=True)
optuna_study_rf_pd.sort_values(by='precision_1').drop(['datetime_start','datetime_complete','duration'],axis=1)

In [ ]:
# покаждем статистику обучения
print('Максимальное значение метрики ROC AUC на валидационном наборе:',round(optuna_study_rf_pd['roc_valid'].max(),3))
print('Среднее значение метрики ROC AUC на валидационном наборе:',round(optuna_study_rf_pd['roc_valid'].mean(),3))
print('Максимальное значение метрики f1_score на валидационном наборе:',round(optuna_study_rf_pd['f1_score'].max(),3))
print('Среднее значение метрики f1_score на валидационном наборе:',round(optuna_study_rf_pd['f1_score'].mean(),3))
print('Максимальное значение метрики recall_1 на валидационном наборе:',round(optuna_study_rf_pd['recall_1'].max(),3))
print('Среднее значение метрики recall_1 на валидационном наборе:',round(optuna_study_rf_pd['recall_1'].mean(),3))
print('Максимальное значение метрики precision_1 на валидационном наборе:',round(optuna_study_rf_pd['precision_1'].max(),3))
print('Среднее значение метрики precision_1 на валидационном наборе:',round(optuna_study_rf_pd['precision_1'].mean(),3))

In [ ]:
# построим график важности гиперпарметров
optuna.visualization.plot_param_importances(optuna_study_rf, target_name='Score')

In [ ]:
# Построим зависимость метрики precision_1 от гиперпараметров

fig = px.scatter(
    data_frame=optuna_study_rf_pd,
    x='precision_1', #ось абсцисс
    y=['roc_train', 'roc_valid', 'recall_1',
       'params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight','params_max_depth','params_max_features',
        'params_max_samples', 'params_min_samples_leaf', 'params_min_samples_split', 
        'params_n_estimators', 'params_n_last'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision класса 1 от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision на классе 1',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

<span style="color:Blue">

Вывод:  

1. Прослеживается зависимость между метриками $precision_1$ и $recall_1$.  
Очевидно, что модель не сможет достичь знаечний $precision_1>0.2$,   
при этом не опустив значения метрики $recall_1$ в 0.

2. Обратим внимание на параметры $class$ $0$ $weigth$, $class$ $1$ $weigth$,   
$class$ $1$ $percent$.   
Для более высокого значения метрики $precision_1$, характерны значения:
- $class$ $0$ $weigth \in [0.4;1]$
- $class$ $1$ $weigth \in [0.01;0.6]$
- $class$ $1$ $percent \in [0.01;0.3]$

3. Обратим внимание на параметры $max$ $depth$.  
Прослеживается влияние гиперпарметра на метрику $precision_1$.  
Для более высокого значения метрики $precision_1$, характерны значения:
- $max$ $depth \in [20;30]$
  
4. Обратим внимание на параметры $max$ $features$.  
Прослеживается влияние гиперпарметра на метрику $precision_1$.  
Для более высокого значения метрики $precision_1$, характерны значения:
- $max$ $features \in [0.55;0.7]$

5. Обратим внимание на параметры $max$ $samples$.  
Прослеживается влияние гиперпарметра на метрику $precision_1$.  
Для более высокого значения метрики $precision_1$, характерны значения:
- $max$ $samples \in [0.55;0.7]$

6. Обратим внимание на параметры $min$ $samples$ $leaf$.  
Прослеживается влияние гиперпарметра на метрику $precision_1$.  
Для более высокого значения метрики $precision_1$, характерны значения:
- $min$ $samples$ $leaf \in [12;14]$

7. Обратим внимание на параметры $min$ $samples$ $split$.  
Прослеживается влияние гиперпарметра на метрику $precision_1$.  
Для более высокого значения метрики $precision_1$, характерны значения:
- $min$ $samples$ $split \in [3;10]$

8. Обратим внимание на параметры $n$ $estimators$.  
Явного влияния на на метрику $precision_1$, не наблюдается. 

9. Обратим внимание на параметры $n$ $last$.  
Прослеживается влияние гиперпарметра на метрику $precision_1$.  
Для более высокого значения метрики $precision_1$, характерны значения:
- $n$ $last \in [14;20]$


In [ ]:
# Построим зависимость метрики roc_valid от гиперпараметров

fig = px.scatter(
    data_frame=optuna_study_rf_pd,
    x='roc_valid', #ось абсцисс
    y=['params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight','params_max_depth','params_max_features',
        'params_max_samples', 'params_min_samples_leaf', 'params_min_samples_split', 
        'params_n_estimators', 'params_n_last'],
)
fig.update_layout(
    title ={
        'text':'Зависимость ROC AUC на валидационном наборе от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики ROC AUC',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

<span style="color:Blue">

Вывод:  

1. Обратим внимание на параметры $class$ $0$ $weigth$, $class$ $1$ $weigth$,   
$class$ $1$ $percent$.   
Я думаю, правильно рассматривать их совместное влияние на метрику.  
Для более высокого значения метрики $ROC AUC$, характерны значения:
- $class$ $0$ $weigth \in [0.2;0.6]$
- $class$ $1$ $weigth \in [0.6;1]$
- $class$ $1$ $percent \in [0.15;0.4]$

3. Обратим внимание на параметры $max$ $depth$.  
Прослеживается влияние гиперпарметра на метрику $ROC AUC$.  
Для более высокого значения метрики $ROC AUC$, характерны значения:
- $max$ $depth \in [10;15]$
  
4. Обратим внимание на параметры $max$ $features$.  
Прослеживается влияние гиперпарметра на метрику $ROC AUC$.  
Для более высокого значения метрики $ROC AUC$, характерны значения:
- $max$ $features \in [0.4;0.8]$

5. Обратим внимание на параметры $max$ $samples$.  
Прослеживается влияние гиперпарметра на метрику $ROC AUC$.  
Для более высокого значения метрики $ROC AUC$, характерны значения:
- $max$ $samples \in [0.4;0.7]$

6. Обратим внимание на параметры $min$ $samples$ $leaf$.  
Прослеживается влияние гиперпарметра на метрику $ROC AUC$.  
Для более высокого значения метрики $ROC AUC$, характерны значения:
- $min$ $samples$ $leaf \in [3;9]$

7. Обратим внимание на параметры $min$ $samples$ $split$.  
Прослеживается влияние гиперпарметра на метрику $ROC AUC$.  
Для более высокого значения метрики $ROC AUC$, характерны значения:
- $min$ $samples$ $split \in [3;10]$

8. Обратим внимание на параметры $n$ $estimators$.  
Прослеживается влияние гиперпарметра на метрику $ROC AUC$.  
Для более высокого значения метрики $ROC AUC$, характерны значения:
- $n$ $estimators \in [70;100]$

9. Обратим внимание на параметры $n$ $last$.  
Прослеживается влияние гиперпарметра на метрику $ROC AUC$.    
Для более высокого значения метрики $ROC AUC$, характерны значения:  
- $n$ $last \in [20;22]$


In [ ]:
# Построим зависимость метрики roc_valid от гиперпараметров

fig = px.scatter(
    data_frame=optuna_study_rf_pd,
    x='f1_score', #ось абсцисс
    y=['params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight','params_max_depth','params_max_features',
        'params_max_samples', 'params_min_samples_leaf', 'params_min_samples_split', 
        'params_n_estimators', 'params_n_last'],
)
fig.update_layout(
    title ={
        'text':'Зависимость f1 score на валидационном наборе от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики f1 - score',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# поготовим данные для визуализации
# выбираем те точки в которых относительно высокие  
# значения precision_1 и recall_1
fig_data_frame=optuna_study_rf_pd[(optuna_study_rf_pd['precision_1']>0.1*optuna_study_rf_pd['precision_1'].max())&
                                  (optuna_study_rf_pd['recall_1']>0.05*optuna_study_rf_pd['recall_1'].max())&
                                  (optuna_study_rf_pd['roc_valid']>0.9*optuna_study_rf_pd['roc_valid'].max())
                                  ]
fig = go.Figure()
fig.add_trace(go.Histogram(x=fig_data_frame['params_criterion'],name='criterion'))


fig.update_traces(opacity=0.75)
fig.update_layout(
    title ={
        'text':'ТОП "лучших" значений параметров по версии метрик precision и recall класса 1', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =500,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    yaxis_title='Количество ')

fig.show()

<span style="color:Blue">
Вывод:

Можно сделать вывод, что для получения приемлемых значений метрик,  
оптимизотор испоользует всю сетку пространства параметров  $criterion$.

In [ ]:
# Построим зависимость метрик качества модели от number

fig = px.scatter(
    data_frame=optuna_study_rf_pd[(optuna_study_rf_pd['roc_valid']>=0.62) & (optuna_study_rf_pd['precision_1']>0.1)],
    x='number', #ось абсцисс
    y=['roc_valid','roc_train','precision_1','recall_1'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость качества модели от trials optuna', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='trials optuna',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

<span style="color:Blue">

Вывод:  

Их всех выбираем точку $number =50$.   
В этой точке оптимальные значения метрик $recall_1$ и $precision_1$, а  
также относительно высокая метрика $ROC AUC$.   
Посмотрим каким значениям гиперпараметров соотвествует выбраннаяточка.

In [ ]:
# определим номер лучшге варианта
best_optuna_number = 50

# сформируем словарь лучших гипирпарметров RandomForestClassifier
best_param_rf = {
    'n_estimators' : int(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_n_estimators'].iloc[0]),
    'criterion': optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_criterion'].iloc[0],
    'max_depth' : optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_max_depth'].iloc[0],
    'min_samples_split': optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_min_samples_split'].iloc[0],
    'min_samples_leaf' : optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_min_samples_leaf'].iloc[0],
    'max_features': optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_max_features'].iloc[0],
    'max_samples' : optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_max_samples'].iloc[0],
    'class_weight' : {0:optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_class_0_weight'].iloc[0],
                      1:optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_class_1_weight'].iloc[0]}
    }
# определим перменные для лучших значений параметров
best_n_last = optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_n_last'].iloc[0]
best_class_1_percent = optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_class_1_percent'].iloc[0]
best_random_state = optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_random_state'].iloc[0]


# Выведем принятые наилучшие праметры
print('best n estimators:',best_param_rf['n_estimators'])
print('best criterion:',best_param_rf['criterion'])
print('best max depth:',best_param_rf['max_depth'])
print('best min samples split:',best_param_rf['min_samples_split'])
print('best min samples leaf:',best_param_rf['min_samples_leaf'])
print('best max features(%):',round(best_param_rf['max_features'],3))
print('best max samples(%):',round(best_param_rf['max_samples'],3))
print('best class 0 weight:',round(best_param_rf['class_weight'][0],3))
print('best class 1 weight:',round(best_param_rf['class_weight'][1],3))

print('best class 1 percent:',round(best_class_1_percent,3))
print('best n last:',best_n_last)
print('best random state:',best_random_state)
print('time for best train:',round(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['duration'].iloc[0].seconds/60),'minutes')

print()
print('ROC AUC на обучающем наборе:', round(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['roc_train'].iloc[0],3))
print('ROC AUC на валидационном наборе:', round(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['roc_valid'].iloc[0],3))
print('precision класса 1:', round(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['precision_1'].iloc[0],3))
print('recall класса 1:', round(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['recall_1'].iloc[0],3))

##### <span style="color:MediumSlateBlue">Обучение модели с лучшими параметрами</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'date'
count_features = dict_spaces[feature_space]

# с помощью функции features_from_transform_data_torow извлечем  
# из данных transform_data_torow
# признаки сооствествующие n_last последним клиенским операциям
list_n_last_features = features_from_transform_data_torow(best_n_last,count_features,25)

# загружаем обучующие наборы
X_train_pd = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas()
y_train_pd = pd.read_csv('target/target_train.csv')

# поготовим данные y_train_pd к работе с функцией class_1_percent_samples
y_train_pd.set_index('id',drop=False,inplace=True)

# подготовим данные для обучения модели
# с помощью функции class_1_percent_samples зададим долю 
# класса 1
list_c1_percent_id = class_1_percent_samples(y_train_pd,best_class_1_percent,random_state=best_random_state)[:1000000]

# сформируем данные для обучения базовой модели
X_train_balanced = np.array(X_train_pd.loc[list_c1_percent_id])[:,list_n_last_features]
y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

# освободим память от "тяжелых" и ненужных файлов
del X_train_pd, y_train_pd
gc.collect()

# обучаем модель RandomForestClassifier с наилучшеми параметрами
random_forest = RandomForestClassifier(
        **best_param_rf,
        n_jobs=-1,
        random_state=best_random_state)
random_forest.fit(X_train_balanced, y_train_balanced)

# освободим память от "тяжелых" и ненужных файлов
del X_train_balanced, y_train_balanced
gc.collect()


# подгружаем данные для тестирования
X_train = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas().to_numpy()[:,list_n_last_features]
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
X_valid = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_valid').to_pandas().to_numpy()[:,list_n_last_features]
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()

# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_rf__pred_proba = random_forest.predict_proba(X_train)[:,1]
y_valid_rf_pred_proba = random_forest.predict_proba(X_valid)[:,1]

# Делаем предсказание для валидационной выборки
y_valid_rf_pred = random_forest.predict(X_valid)

# удаляем крупные файлы чтобы высвободить память 
del X_train, X_valid
gc.collect()

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_rf__pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_rf_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_valid,y_valid_rf_pred,zero_division=0))

# time: 2m 10s

<span style="color:Blue">
Вывод:  

Модель $Random$ $Forest$ $Classifier$, обученная на сбалансированных данных   
$date$ $torow$, показывает чуть лучшее качество по метрике $ROC AUC$,  
чем модель $Logistic$ $Regression$ $Classifier$, обученная на тех же данных.  
В остальном модель имеют похожую способность отделять класс 1 от класс 0.

##### <span style="color:MediumSlateBlue">Анализ влияния порога классификации на качество модели</span>

In [ ]:
# чтобы проше обращаться к каждому обьекту в данных
# преобразуем y_valid_LR_pred к Series типу
y_valid_rf_pred_proba = pd.Series(y_valid_rf_pred_proba)

# формируем список из необходимых значений thresholds
thresholds = np.arange(0.01, 1, 0.01)

# сформируем списко для заполнения данными результатов анализа
threshold_data = []


for threshold in thresholds:
        # сформируем список для заполнения данными текушего threshold
        threshold_current_data = [threshold]

        #клиентов, для которых вероятность дефолта > threshold, относим к классу 1
        #в противном случае — к классу 0
        y_valid_rf_pred = y_valid_rf_pred_proba.apply(lambda x: 1 if x>threshold else 0)


        #Считаем метрики для класса 0 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(y_valid, y_valid_rf_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.precision_score(y_valid, y_valid_rf_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.f1_score(y_valid, y_valid_rf_pred,average=None,zero_division=0)[0],3))

        #Считаем метрики для класса 1 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(y_valid, y_valid_rf_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.precision_score(y_valid, y_valid_rf_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.f1_score(y_valid, y_valid_rf_pred,average=None,zero_division=0)[1],3))

        # добавляем полученные данные в список threshold_data
        threshold_data.append(threshold_current_data)

        # из полученных данных сформируем dataframe
        thresholds_pd =pd.DataFrame(
        columns=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'],
        data = threshold_data)

# time: 17s

In [ ]:
# Визуализируем метрики на валидационном наборе при различных threshold
fig_threshold = px.line(
    data_frame=thresholds_pd,
    x='threshold', #ось абсцисс
    y=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'], #ось ординат
)
fig_threshold.update_layout(
    title ={
        'text':'Зависимость качества модели от threshold', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Порог классификации threshold',
    yaxis_title='Величина метрики')
fig_threshold.update_xaxes(showspikes=True)
fig_threshold.update_yaxes(showspikes=True)

fig_threshold.show()

Метрика $precision$ показывает на сколько сильно ошиблась модель при определиние класса.  
Чем выше метрика, тем меньше ошиблась модель. 

$$precision = \frac{TP}{TP+FP}$$

Метрика $recall$ показывает способность модели найти класс.  
Чем выше метрика, тем выше способность модели находить класс.  

$$recall = \frac{TP}{TP+FN}$$

Метрика $f1-score$ показывает баланс между $precision$ и $recall$ 

$threshold$- порог класификации модели.  
Все объекты для которых $у_{\text{proba pred}} < threshold$, модель относит к класс 0.  
Соотвественно, если $у_{\text{proba pred}}  > threshold$ модель относит обьект к классу 1.  
Для идельаной модели $threshold$, является границей между областями класса 0 и класса 1.  

<span style="color:Blue">

Выводы по результам анализа:  

Для улучшения качества модели по метрике $ROC AUC$, нет смысла сдвигать порог   
классификации. Но зависимость метрик качества модели от $threshold$, может рассказать о  
способности модели искать класс 1 и класс 0.

При фокусировке модели на классе 1 (более низкий $threshold$):  
- на классе 1 метрика $precision$ уменьшается, метрика $recall$ растет;  
- на классе 0 метрика $precision$ растет, метрика $recall$ уменьшается.  

При фокусировке модели на классе 0 (более высокий $threshold$):  
- на классе 1 метрика $precision$ увеличивается, метрика $recall$ уменьшается;  
- на классе 0 метрика $precision$ уменьшается, метрика $recall$ растет.  

Обьсняется тем, что в условия дисбаланса модель научилась хорошо определять класс 0 и плохо класс 1.  

При фокусировке модели на классе 1 (более низкий $threshold$), все больше обьектов отнесены к классу 1.   
Так как $precision$ класса 1 уменьшается, значит в области класса 1 стало больше обектов класса 0,  
что явлется закономерным для "идеальной" модели.  
Однако при этом, $recall$ класса 1 растет, значит в добавленных обьектах также присутвует класс 1.  
Это как раз указывает на то, что модель не нучилась находить среди класса 0 класс 1. В области с     
высокой вероятностью класса 0 (более низкий $threshold$), находятся обьекты класса 1.  

Фокусировка на классе 0 (более высокий $threshold$), заставляет модель все больше обьектов из области класса 1  
относить к классу 0. При этом увеличение метрики $precision$ класса 1 говорит о том, что в области класса 1     
становится меньше 0, а значит модель умеет среди класса 1, находить класс 0.  
После  $threshold=0.65$ модель впринципе теряет способность отличать класс 1. У модели нет обьектов класса 1   
в которых она "уверена" на 70+%.

Обратим внимание на $threshold=0.38$, в этой точке находится баланс межу метриками качества модели:
$$precision_1 = recall_1 = \text{f1-score}_1 = 0.096$$
$$precision_0 = recall_0 = \text{f1-score}_0 = 0.966$$

Для модели с лучшим качеством эти две точки должны находится максимально близко к друг другу и иметь   
высокое значение метрик $precision_1$ и $recall_1$.  
Для класса 0 это условие выполнется, что также говорит о том, что модель научилась искать этот класс.    
Для класса 1 это условие не выполнется, что также говорит о том, что модель плохо ищет класс 1.

##### <span style="color:MediumSlateBlue">Построение ROC-кривой выбранной модели на тестовом наборе</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'date'
count_features = dict_spaces[feature_space]

In [ ]:
# подгружаем данные для тестирования
X_train = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas().to_numpy()[:,list_n_last_features]
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
X_valid = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_valid').to_pandas().to_numpy()[:,list_n_last_features]
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()
X_test = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_test').to_pandas().to_numpy()[:,list_n_last_features]
y_test = pd.read_csv('target/target_test.csv')['flag'].to_numpy()

In [ ]:
print("Размеры обучающего набора",X_train.shape,y_train.shape)
print("Размеры валидационного набора",X_valid.shape,y_valid.shape)
print("Размеры тестового набора",X_test.shape,y_test.shape)
# time: 1m 43s

In [ ]:
# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_rf_pred_proba = random_forest.predict_proba(X_train)[:,1]
y_valid_rf_pred_proba = random_forest.predict_proba(X_valid)[:,1]
y_test_rf_pred_proba = random_forest.predict_proba(X_test)[:,1]

# Делаем предсказание для валидационной выборки
y_test_lr_pred = random_forest.predict(X_test)

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_rf_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_rf_pred_proba),3))
print('ROC AUC на тестовом наборе', round(metrics.roc_auc_score(y_test, y_test_rf_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_test,y_test_lr_pred,zero_division=0))

In [ ]:
# Для расчета значений TPR и FPR используем функцию roc_curve
fpr_train, tpr_train, _ = metrics.roc_curve(y_train, y_train_rf_pred_proba)
fpr_valid, tpr_valid, _ = metrics.roc_curve(y_valid, y_valid_rf_pred_proba)
fpr_test, tpr_test, _ = metrics.roc_curve(y_test, y_test_rf_pred_proba)

# рассчитаем площадь под кривой
auc_train = round(metrics.roc_auc_score(y_train, y_train_rf_pred_proba),3)
auc_valid = round(metrics.roc_auc_score(y_valid, y_valid_rf_pred_proba),3)
auc_test = round(metrics.roc_auc_score(y_test, y_test_rf_pred_proba),3)

trace_train = go.Scatter(x=fpr_train, y=tpr_train, mode='lines', name='AUC_train = %0.2f' % auc_train,
                   line=dict(color='red', width=2),
                   hovertemplate='train<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_train])

trace_valid = go.Scatter(x=fpr_valid, y=tpr_valid, mode='lines', name='AUC_valid = %0.2f' % auc_valid,
                   line=dict(color='green', width=2),
                   hovertemplate='valid<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_valid])

trace_test = go.Scatter(x=fpr_test, y=tpr_test, mode='lines', name='AUC_test = %0.2f' % auc_test,
                   line=dict(color='darkorange', width=2),
                   hovertemplate='test<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_test])


reference_line = go.Scatter(x=[0,1], y=[0,1], mode='lines', name='Reference Line',
                            line=dict(color='navy', width=2, dash='dash'))

fig_roc = go.Figure(data=[trace_train,trace_valid,trace_test,reference_line])

fig_roc.update_layout(
    title ={
        'text':'ROC-кривая', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 1350,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate')

fig_roc.update_xaxes(showspikes=True)
fig_roc.update_yaxes(showspikes=True)


fig_roc.show()

<span style="color:Blue">

Вывод по ROC-кривой:
1. Модель выдает сталибильное качество на валидационном и тестовом наборах, а значит  
не переобучена - разброс ($variance$) не большой. Хотя, качество модели на обучающем наборе,  
заметно лучше.
2. Смещение ($bias$) модели чуть лучше, чем у модели $Logicstic$ $Regression$ $lassifier$, обученной   
на тех же данных.  
На тестовом наборе полученно качество $ROC AUC = 0.62$, против $ROC AUC = 0.58$

#### <span style="color:MediumBlue">Построение модели Gradient Boosting Classifier</span>

##### <span style="color:MediumSlateBlue">Baseline обучение модели Hist Gradient Boosting Classifier</span>

In [ ]:
# обучаем модель Gradient Boosting Classifier
gradient_boosting = HistGradientBoostingClassifier(random_state=42)
gradient_boosting.fit(np.array(X_train_balanced)[:,list_n_last_features],np.array(y_train_balanced))

# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_rf__pred_proba = gradient_boosting.predict_proba(X_train[:,list_n_last_features])[:,1]
y_valid_rf_pred_proba = gradient_boosting.predict_proba(X_valid[:,list_n_last_features])[:,1]

# Делаем предсказание для валидационной выборки
y_valid_rf_pred = gradient_boosting.predict(X_valid[:,list_n_last_features])

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_rf__pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_rf_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_valid,y_valid_rf_pred))

# time: 

<span style="color:Blue">

Выводы:

1. Качество модели по метрики $ROC AUC$ лучше, чем у предыдущих моделей.
2. В остальном, качество модели не изменилось.

##### <span style="color:MediumSlateBlue">Подбор гиперпараметров модели</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'date'
count_features = dict_spaces[feature_space]

# настроим оптимизацию гипер параметров
def optuna_hgbc(trial):
  # задаем пространства поиска гиперпараметров
  learning_rate = trial.suggest_float('learning_rate',0.2,0.6)
  max_iter = trial.suggest_int('max_iter', 50, 300,step = 1)
  max_leaf_nodes = trial.suggest_int('max_leaf_nodes', 2, 50,step = 1)
  max_depth = trial.suggest_int('max_depth', 1, 15,step = 1)
  min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 50,step = 1)
  max_features = trial.suggest_float('max_features',0.1,1)
  l2_regularization = trial.suggest_float('l2_regularization',0.01,1)
  class_0_weight = trial.suggest_float('class_0_weight',0.01,1)
  class_1_weight = trial.suggest_float('class_1_weight',0.01,1)
  n_last = trial.suggest_int('n_last', 5, 25,step = 1)
  class_1_percent = trial.suggest_float('class_1_percent',0.01,0.5)
  random_state = trial.suggest_int('random_state', 1, 1000000)


  # создаем модель
  optuna_gradient_boosting = HistGradientBoostingClassifier(
      learning_rate= learning_rate,
      max_iter = max_iter,
      max_leaf_nodes =max_leaf_nodes,
      max_depth = max_depth,
      min_samples_leaf = min_samples_leaf,
      max_features=max_features,
      l2_regularization = l2_regularization,
      class_weight={0:class_0_weight,1:class_1_weight},
      random_state=42)

  # с помощью функции features_from_transform_data_torow извлечем  
  # из данных transform_data_torow
  # признаки сооствествующие n_last последним клиенским операциям
  list_n_last_features = features_from_transform_data_torow(n_last,count_features,25)

  # загружаем обучующие наборы
  X_train_pd = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas()
  y_train_pd = pd.read_csv('target/target_train.csv')

  # поготовим данные y_train_pd к работе с функцией class_1_percent_samples
  y_train_pd.set_index('id',drop=False,inplace=True)

  # подготовим данные для обучения модели
  # с помощью функции class_1_percent_samples зададим долю класса 1
  list_c1_percent_id = class_1_percent_samples(y_train_pd,class_1_percent,random_state=random_state)[:1000000]

  # сформируем данные для обучения модели
  X_train_balanced = np.array(X_train_pd.loc[list_c1_percent_id])[:,list_n_last_features]
  y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

  # освободим память от "тяжелых" и ненужных файлов
  del X_train_pd, y_train_pd
  gc.collect()

  # я не воспользовался параметром timeout метода optimize потому, что  
  # optimize отставливает поиск параметров, если время обучения 
  # превышает timeout. Мне нужно чтобы поиск параметров продолжился.
  try:
    # для модуля func_time_out упакуем обучение модели в функцию try_func
    def try_func():
            return optuna_gradient_boosting.fit(X_train_balanced,y_train_balanced)
    # обучаем модель с ограничением по времени 10 минут (600 секунд)
    optuna_gradient_boosting = func_timeout.func_timeout(600, try_func)

    # удаляем крупные файлы чтобы высвободить память 
    del X_train_balanced, y_train_balanced
    gc.collect()

    # подгружаем данные для тестирования
    X_train = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas().to_numpy()[:,list_n_last_features]
    y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
    X_valid = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_valid').to_pandas().to_numpy()[:,list_n_last_features]
    y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()

    # делаем предсказание на обучающем и валидационном наборе и считаем метрики
    roc_train = metrics.roc_auc_score(y_train, optuna_gradient_boosting.predict_proba(X_train)[:,1])
    roc_valid = metrics.roc_auc_score(y_valid, optuna_gradient_boosting.predict_proba(X_valid)[:,1])
    f1_score = metrics.f1_score(y_valid, optuna_gradient_boosting.predict(X_valid))
    recall_1 = metrics.recall_score(y_valid, optuna_gradient_boosting.predict(X_valid),average=None,zero_division=0)[1]
    precision_1 = metrics.precision_score(y_valid, optuna_gradient_boosting.predict(X_valid),average=None,zero_division=0)[1]

    # удаляем крупные файлы чтобы высвободить память 
    del X_train, X_valid, y_train, y_valid 
    gc.collect()

  except func_timeout.FunctionTimedOut:
    # освободим память от "тяжелых" и ненужных файлов
    del X_train_balanced, y_train_balanced
    gc.collect()

    # обьявим метрики пустыми значениями
    roc_train = 0
    roc_valid = 0
    f1_score = 0
    recall_1 = 0
    precision_1 = 0
    pass

  return roc_train, roc_valid, f1_score, recall_1, precision_1

In [ ]:
# cоздаем объект исследования
# чтобы модель не переобучалась минимизируем roc_train 
# и максимизируем roc_valid
optuna_study_hsbs = optuna.create_study(study_name='HistGradientBoostingClassifier_'+feature_space+'_torow', 
                               directions=['maximize','maximize','maximize','maximize','maximize'], 
                               sampler=optuna.samplers.TPESampler(),
                               pruner='Hyperband',
                               storage='sqlite:///optuna_studies.db',
                               load_if_exists=True)

# ну случай удаления обучения
# optuna.delete_study(study_name='HistGradientBoostingClassifier_'+feature_space+'_torow', storage='sqlite:///optuna_studies.db')

In [ ]:
# ищем лучшую комбинацию гиперпараметров n_trials раз
optuna_study_hsbs.optimize(optuna_hgbc, n_trials=300)

In [ ]:
# из полученного результат соврмируем Data Frame
optuna_study_hsbs_pd = optuna_study_hsbs.trials_dataframe()
# переименуем столбы
optuna_study_hsbs_pd.rename(columns={
    'values_0': 'roc_train',
    'values_1': 'roc_valid',
    'values_2': 'f1_score',
    'values_3': 'recall_1',
    'values_4': 'precision_1'
},inplace=True)
# Выделим время потраченное на обучение модели
last_trail = optuna_study_hsbs_pd['number'].max()
optuna_study_hsbs_pd.tail(5)


In [ ]:
# покаждем статистику обучения
print('Максимальное значение метрики ROC AUC на валидационном наборе:',round(optuna_study_hsbs_pd['roc_valid'].max(),3))
print('Среднее значение метрики ROC AUC на валидационном наборе:',round(optuna_study_hsbs_pd['roc_valid'].mean(),3))
print('Максимальное значение метрики f1_score на валидационном наборе:',round(optuna_study_hsbs_pd['f1_score'].max(),3))
print('Среднее значение метрики f1_score на валидационном наборе:',round(optuna_study_hsbs_pd['f1_score'].mean(),3))
print('Максимальное значение метрики recall_1 на валидационном наборе:',round(optuna_study_hsbs_pd['recall_1'].max(),3))
print('Среднее значение метрики recall_1 на валидационном наборе:',round(optuna_study_hsbs_pd['recall_1'].mean(),3))
print('Максимальное значение метрики precision_1 на валидационном наборе:',round(optuna_study_hsbs_pd['precision_1'].max(),3))
print('Среднее значение метрики precision_1 на валидационном наборе:',round(optuna_study_hsbs_pd['precision_1'].mean(),3))

In [ ]:
# построим график важности гиперпарметров
optuna.visualization.plot_param_importances(optuna_study_hsbs, target_name='Score')

In [ ]:
# Построим зависимость метрики precision_1 от гиперпараметров
# max_iter

fig = px.scatter(
    data_frame=optuna_study_hsbs_pd,
    x='precision_1', #ось абсцисс
    y=['roc_train', 'roc_valid', 'recall_1',
       'params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight', 'params_l2_regularization',
       'params_learning_rate', 'params_max_depth', 'params_max_features',
       'params_max_iter', 'params_max_leaf_nodes', 'params_min_samples_leaf',
       'params_n_last'
    ], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость  метрик качества модели от params max iter', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина params max iter',
    yaxis_title='Величина метрики')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики precision_1 от гиперпараметров
fig = px.scatter(
    data_frame=optuna_study_hsbs_pd,
    x='roc_valid', #ось абсцисс
    y=['roc_train', 'recall_1', 'precision_1',
       'params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight', 'params_l2_regularization',
       'params_learning_rate', 'params_max_depth', 'params_max_features',
       'params_max_iter', 'params_max_leaf_nodes', 'params_min_samples_leaf',
       'params_n_last'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision класса 1 от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision класса 1',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики precision_1 от гиперпараметров
fig = px.scatter(
    data_frame=optuna_study_hsbs_pd,
    x='f1_score', #ось абсцисс
    y=['params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight', 'params_l2_regularization',
       'params_learning_rate', 'params_max_depth', 'params_max_features',
       'params_max_iter', 'params_max_leaf_nodes', 'params_min_samples_leaf',
       'params_n_last'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision класса 1 от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision класса 1',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрик качества модели от number

fig = px.scatter(
    data_frame=optuna_study_hsbs_pd[(optuna_study_hsbs_pd['recall_1']>0.05)&
                                    (optuna_study_hsbs_pd['precision_1']>0.1)&
                                    (optuna_study_hsbs_pd['roc_valid']>0.63)
                                    ],
    x='number', #ось абсцисс
    y=['roc_train','roc_valid','recall_1','precision_1'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision_1',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

<span style="color:Blue">

Вывод:  

Их всех выбираем точку $number =225$.   
В этой точке оптимальные значения метрик $recall_1$ и $precision_1$, а  
также относительно высокая метрика $ROC AUC$.   
Посмотрим каким значениям гиперпараметров соотвествует выбранная точка.

In [ ]:
# определим номер лучшге варианта
best_optuna_number = 225

# сформируем словарь лучших гипирпарметров RandomForestClassifier
best_param_rf = {
    'learning_rate' : optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['params_learning_rate'].iloc[0],
    'max_iter' : optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['params_max_iter'].iloc[0],
    'max_leaf_nodes' : optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['params_max_leaf_nodes'].iloc[0],
    'max_depth' : optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['params_max_depth'].iloc[0],
    'min_samples_leaf' : optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['params_min_samples_leaf'].iloc[0],
    'max_features' : optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['params_max_features'].iloc[0],
    'l2_regularization' : optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['params_l2_regularization'].iloc[0],
    'class_weight' : {0: optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['params_class_0_weight'].iloc[0],
                      1: optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['params_class_1_weight'].iloc[0],}
    }

# определим перменные для лучших значений параметров
best_n_last = optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['params_n_last'].iloc[0]
best_class_1_percent = optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['params_class_1_percent'].iloc[0]
best_random_state = optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['params_random_state'].iloc[0]


# Выведем принятые наилучшие праметры
print('best learning rate:',round(best_param_rf['learning_rate'],3))
print('best max iter:',best_param_rf['max_iter'])
print('best max leaf nodes:',best_param_rf['max_leaf_nodes'])
print('best max depth:',best_param_rf['max_depth'])
print('best min samples leaf:',best_param_rf['min_samples_leaf'])
print('best max features:',round(best_param_rf['max_features'],3))
print('best l2 regularization:',round(best_param_rf['l2_regularization'],3))
print('best class 0 weight:',round(best_param_rf['class_weight'][0],3))
print('best class 1 weight:',round(best_param_rf['class_weight'][1],3))

print('best class 1 percent:',round(best_class_1_percent,3))
print('best n last:',best_n_last)
print('best random state:',best_random_state)
print('time for best train:',round(optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['duration'].iloc[0].seconds/60),'minutes')

print()
print('ROC AUC на обучающем наборе:', round(optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['roc_train'].iloc[0],3))
print('ROC AUC на валидационном наборе:', round(optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['roc_valid'].iloc[0],3))
print('precision класса 1:', round(optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['precision_1'].iloc[0],3))
print('recall класса 1:', round(optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['recall_1'].iloc[0],3))

##### <span style="color:MediumSlateBlue">Обучение модели с лучшими параметрами</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'date'
count_features = dict_spaces[feature_space]

# с помощью функции features_from_transform_data_torow извлечем  
# из данных transform_data_torow
# признаки сооствествующие n_last последним клиенским операциям
list_n_last_features = features_from_transform_data_torow(best_n_last,count_features,25)

# загружаем обучующие наборы
X_train_pd = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas()
y_train_pd = pd.read_csv('target/target_train.csv')

# поготовим данные y_train_pd к работе с функцией class_1_percent_samples
y_train_pd.set_index('id',drop=False,inplace=True)

# подготовим данные для обучения модели
# с помощью функции class_1_percent_samples зададим долю класса 1
list_c1_percent_id = class_1_percent_samples(y_train_pd,best_class_1_percent,random_state=best_random_state)[:1000000]

# сформируем данные для обучения модели
X_train_balanced = np.array(X_train_pd.loc[list_c1_percent_id])[:,list_n_last_features]
y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

# освободим память от "тяжелых" и ненужных файлов
del X_train_pd, y_train_pd
gc.collect()


# обучаем модель HistGradientBoostingClassifier с наилучшеми параметрами
gradient_boosting = HistGradientBoostingClassifier(
        **best_param_rf,
        random_state=42)
gradient_boosting.fit(X_train_balanced,y_train_balanced)

# освободим память от "тяжелых" и ненужных файлов
del X_train_balanced, y_train_balanced
gc.collect()

# подгружаем данные для тестирования
X_train = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas().to_numpy()[:,list_n_last_features]
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
X_valid = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_valid').to_pandas().to_numpy()[:,list_n_last_features]
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()

# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_gb__pred_proba = gradient_boosting.predict_proba(X_train)[:,1]
y_valid_gb_pred_proba = gradient_boosting.predict_proba(X_valid)[:,1]

# Делаем предсказание для валидационной выборки
y_valid_gb_pred = gradient_boosting.predict(X_valid)

# освободим память от "тяжелых" и ненужных файлов
del X_train, X_valid
gc.collect()

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_gb__pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_gb_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_valid,y_valid_gb_pred,zero_division=0))

# time: 2m 30s

<span style="color:Blue">
Вывод:  

Модель $Hist$ $Gradient$ $Boosting$ $Classifier$, обученная на сбалансированных данных   
$transform$ $data$ $torow$, показывает лучшее качество по метрике $ROC AUC$,  
чем предыдущие модели, обученные на тех же данных.  
В остальном модель имеют похожую способность отделять класс 1 от класс 0.

##### <span style="color:MediumSlateBlue">Анализ влияния порога классификации на качество модели</span>

In [ ]:
# чтобы проше обращаться к каждому обьекту в данных
# преобразуем y_valid_LR_pred к Series типу
y_valid_gb_pred_proba = pd.Series(y_valid_gb_pred_proba)

# формируем список из необходимых значений thresholds
thresholds = np.arange(0.01, 1, 0.01)

# сформируем списко для заполнения данными результатов анализа
threshold_data = []


for threshold in thresholds:
        # сформируем список для заполнения данными текушего threshold
        threshold_current_data = [threshold]

        #клиентов, для которых вероятность дефолта > threshold, относим к классу 1
        #в противном случае — к классу 0
        y_valid_gb_pred = y_valid_gb_pred_proba.apply(lambda x: 1 if x>threshold else 0)


        #Считаем метрики для класса 0 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(y_valid, y_valid_gb_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.precision_score(y_valid, y_valid_gb_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.f1_score(y_valid, y_valid_gb_pred,average=None,zero_division=0)[0],3))

        #Считаем метрики для класса 1 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(y_valid, y_valid_gb_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.precision_score(y_valid, y_valid_gb_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.f1_score(y_valid, y_valid_gb_pred,average=None,zero_division=0)[1],3))

        # добавляем полученные данные в список threshold_data
        threshold_data.append(threshold_current_data)

        # из полученных данных сформируем dataframe
        thresholds_pd =pd.DataFrame(
        columns=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'],
        data = threshold_data)

# time: 17s

In [ ]:
# Визуализируем метрики на валидационном наборе при различных threshold
fig_threshold = px.line(
    data_frame=thresholds_pd,
    x='threshold', #ось абсцисс
    y=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'], #ось ординат
)
fig_threshold.update_layout(
    title ={
        'text':'Зависимость качества модели от threshold', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Порог классификации threshold',
    yaxis_title='Величина метрики')
fig_threshold.update_xaxes(showspikes=True)
fig_threshold.update_yaxes(showspikes=True)

fig_threshold.show()

Метрика $precision$ показывает на сколько сильно ошиблась модель при определиние класса.  
Чем выше метрика, тем меньше ошиблась модель. 

$$precision = \frac{TP}{TP+FP}$$

Метрика $recall$ показывает способность модели найти класс.  
Чем выше метрика, тем выше способность модели находить класс.  

$$recall = \frac{TP}{TP+FN}$$

Метрика $f1-score$ показывает баланс между $precision$ и $recall$ 

$threshold$- порог класификации модели.  
Все объекты для которых $у_{\text{proba pred}} < threshold$, модель относит к класс 0.  
Соотвественно, если $у_{\text{proba pred}}  > threshold$ модель относит обьект к классу 1.  
Для идельаной модели $threshold$, является границей между областями класса 0 и класса 1.  

<span style="color:Blue">

Выводы по результам анализа:  

Для улучшения качества модели по метрике $ROC AUC$, нет смысла сдвигать порог   
классификации. Но зависимость метрик качества модели от $threshold$, может рассказать о  
способности модели искать класс 1 и класс 0.

При фокусировке модели на классе 1 (более низкий $threshold$):  
- на классе 1 метрика $precision$ уменьшается, метрика $recall$ растет;  
- на классе 0 метрика $precision$ растет, метрика $recall$ уменьшается.  

При фокусировке модели на классе 0 (более высокий $threshold$):  
- на классе 1 метрика $precision$ увеличивается, метрика $recall$ уменьшается;  
- на классе 0 метрика $precision$ уменьшается, метрика $recall$ растет.  

Обьсняется тем, что в условия дисбаланса модель научилась хорошо определять класс 0 и плохо класс 1.  

При фокусировке модели на классе 1 (более низкий $threshold$), все больше обьектов отнесены к классу 1.   
Так как $precision$ класса 1 уменьшается, значит в области класса 1 стало больше обектов класса 0,  
что явлется закономерным для "идеальной" модели.  
Однако при этом, $recall$ класса 1 растет, значит в добавленных обьектах также присутвует класс 1.  
Это как раз указывает на то, что модель не нучилась находить среди класса 0 класс 1. В области с     
высокой вероятностью класса 0 (более низкий $threshold$), находятся обьекты класса 1.  

Фокусировка на классе 0 (более высокий $threshold$), заставляет модель все больше обьектов из области класса 1  
относить к классу 0. При этом увеличение метрики $precision$ класса 1 говорит о том, что в области класса 1     
становится меньше 0, а значит модель умеет среди класса 1, находить класс 0.  
После  $threshold=0.83$ модель впринципе теряет способность отличать класс 1. У модели нет обьектов класса 1   
в которых она "уверена" на 95+%.

Обратим внимание на $threshold=0.51$, в этой точке находится баланс межу метриками качества модели:
$$precision_1 = recall_1 = \text{f1-score}_1 = 0.101$$
$$precision_0 = recall_0 = \text{f1-score}_0 = 0.966$$

Для модели с лучшим качеством эти две точки должны находится максимально близко к друг другу и иметь   
высокое значение метрик $precision_1$ и $recall_1$.  
Для класса 0 это условие выполнется, что также говорит о том, что модель научилась искать этот класс.    
Для класса 1 это условие не выполнется, что также говорит о том, что модель плохо ищет класс 1.

##### <span style="color:MediumSlateBlue">Построение ROC-кривой выбранной модели на тестовом наборе</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'date'
count_features = dict_spaces[feature_space]

In [ ]:
# подгружаем данные для тестирования
X_train = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas().to_numpy()[:,list_n_last_features]
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
X_valid = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_valid').to_pandas().to_numpy()[:,list_n_last_features]
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()
X_test = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_test').to_pandas().to_numpy()[:,list_n_last_features]
y_test = pd.read_csv('target/target_test.csv')['flag'].to_numpy()

In [ ]:
print("Размеры обучающего набора",X_train.shape,y_train.shape)
print("Размеры валидационного набора",X_valid.shape,y_valid.shape)
print("Размеры тестового набора",X_test.shape,y_test.shape)
# time: 1m 43s

In [ ]:
# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_gb_pred_proba = gradient_boosting.predict_proba(X_train)[:,1]
y_valid_gb_pred_proba = gradient_boosting.predict_proba(X_valid)[:,1]
y_test_gb_pred_proba = gradient_boosting.predict_proba(X_test)[:,1]

# Делаем предсказание для валидационной выборки
y_test_gb_pred = gradient_boosting.predict(X_test)

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_gb_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_gb_pred_proba),3))
print('ROC AUC на тестовом наборе', round(metrics.roc_auc_score(y_test, y_test_gb_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_test,y_test_gb_pred,zero_division=0))

In [ ]:
# Для расчета значений TPR и FPR используем функцию roc_curve
fpr_train, tpr_train, _ = metrics.roc_curve(y_train, y_train_gb_pred_proba)
fpr_valid, tpr_valid, _ = metrics.roc_curve(y_valid, y_valid_gb_pred_proba)
fpr_test, tpr_test, _ = metrics.roc_curve(y_test, y_test_gb_pred_proba)

# рассчитаем площадь под кривой
auc_train = round(metrics.roc_auc_score(y_train, y_train_gb_pred_proba),3)
auc_valid = round(metrics.roc_auc_score(y_valid, y_valid_gb_pred_proba),3)
auc_test = round(metrics.roc_auc_score(y_test, y_test_gb_pred_proba),3)

trace_train = go.Scatter(x=fpr_train, y=tpr_train, mode='lines', name='AUC_train = %0.2f' % auc_train,
                   line=dict(color='red', width=2),
                   hovertemplate='train<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_train])

trace_valid = go.Scatter(x=fpr_valid, y=tpr_valid, mode='lines', name='AUC_valid = %0.2f' % auc_valid,
                   line=dict(color='green', width=2),
                   hovertemplate='valid<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_valid])

trace_test = go.Scatter(x=fpr_test, y=tpr_test, mode='lines', name='AUC_test = %0.2f' % auc_test,
                   line=dict(color='darkorange', width=2),
                   hovertemplate='test<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_test])


reference_line = go.Scatter(x=[0,1], y=[0,1], mode='lines', name='Reference Line',
                            line=dict(color='navy', width=2, dash='dash'))

fig_roc = go.Figure(data=[trace_train,trace_valid,trace_test,reference_line])

fig_roc.update_layout(
    title ={
        'text':'ROC-кривая', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 1350,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate')

fig_roc.update_xaxes(showspikes=True)
fig_roc.update_yaxes(showspikes=True)


fig_roc.show()

<span style="color:Blue">

Вывод по ROC-кривой:
1. Модель выдает сталибильное качество на валидационном и тестовом наборах, а значит  
не переобучена - разброс ($variance$) не большой. Хотя, качество модели на обучающем наборе,  
заметно лучше.
2. Смещение ($bias$) модели чуть лучше, чем у модели $Random$ $Forest$ $Classifier$,  
обученной на тех же данных.  
На тестовом наборе полученно качество $ROC AUC = 0.64$, против $ROC AUC = 0.62$

### <span style="color:RoyalBlue">Построение модели на сбалансированных данных подпространства late torow</span>

#### <span style="color:MediumBlue">Формирование данных для обучения</span>

Анализ исходных данных, в частности target, показал что представленные данные   
имееют перекос: 96% клиентов у которых не случился дефолт (flag = 0),    
против 4 % - для которых произошел дефолт (flag = 1).  
С помощью функции class_1_percent_samples, сформируем сбаланисрованную выборку  
для обучения моделей.
За основу сбалансировных данных возьмем данные клиентов с дефолтом  
и к ним будем добирать необходимое количество клиентов до сбалансированности. 


Функция class_1_percent_samples принемает две переменные:
- target_data - массив с id и целевой переменной
- class_1_percent - процент класса 1 в результирующем массиве

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'late'
count_features = dict_spaces[feature_space]

In [ ]:
# сформируем данные для анализа сбалансированности обучающих данных
X_train_pd = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas()
y_train_pd = pd.read_csv('target/target_train.csv')

In [ ]:
# поготовим данные y_train_pd к работе с функцией class_1_percent_samples
y_train_pd.set_index('id',drop=False,inplace=True)
y_train_pd.head(4)

In [ ]:
# взглянем на сблансиорованность данных в y_train
y_train_pd['flag'].value_counts(normalize=True)

In [ ]:
# сбалансируем данные в y_train_pd
list_c1_percent_id = class_1_percent_samples(y_train_pd,0.5)
# list_c1_percent_features - список 'id' из y_train_pd соотвествующих  
# заданному проценту класса 1
len(list_c1_percent_id)

In [ ]:
# проверим сбалансированность полученного y_train
y_train_pd.loc[list_c1_percent_id]['flag'].value_counts(normalize=True)

*transform_data_torow*, показываеются последние N операций клиента.  
В данных в *date_torow*, собраны последние 25 операций клиентов.

Предварительный анализ показывает, что медиальное значение количества клиентских  
операций равно 7. Поэтому, для начала, будем показывать моделе $n_{last} = 7$   
последних клиентских операций.

Для извлечения из *transform_data_torow_25* необходимого количества последних клиенских  
операций ($n_{last}$) используем функцию features_from_transform_data_torow.

Функция features_from_transform_data_torow примает следующие аргументы:
- $n_{last} = 7$ - необходимое количество последних клиентских операций; 
- $n_{groups} = 8$ - количество признаков в $date$ $features$;
- $N_{last} = 25$ - количество $n_{last}$ операций, показанных в transform_data_torow_25


In [ ]:
# с помощью функции features_from_transform_data_torow извлечем  
# из данных transform_data_torow_25
# признаки сооствествующие 7 последним клиенским операциям
list_n_last_features = features_from_transform_data_torow(7,count_features,25)

In [ ]:
# подготовим данные для обучения
X_train_balanced = X_train_pd.loc[list_c1_percent_id]
y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag']
# сформируем данные для проверики модели
X_train = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas().to_numpy()
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
X_valid = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_valid').to_pandas().to_numpy()
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()

In [ ]:
# посмотрим на структуру данных в X_train_balanced
X_train_balanced.head(2)

In [ ]:
# посмотрим на структуру данных в y_train_balanced
y_train_balanced.head(2)

In [ ]:
# проверим что выборки действительно совпадают по id
X_train_balanced.index.to_list() == y_train_balanced.index.to_list()

#### <span style="color:MediumBlue">Построение модели Logistic Regression</span>

##### <span style="color:MediumSlateBlue">Baseline обучение модели LogisticRegression</span>

In [ ]:
# обучаем модель логистической регрессии
logistic_regression = linear_model.LogisticRegression(random_state=42, max_iter=10000)
logistic_regression.fit(np.array(X_train_balanced)[:,list_n_last_features],np.array(y_train_balanced))

# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_LR_pred_proba = logistic_regression.predict_proba(X_train[:,list_n_last_features])[:,1]
y_valid_LR_pred_proba = logistic_regression.predict_proba(X_valid[:,list_n_last_features])[:,1]

# Делаем предсказание для валидационной выборки
y_valid_LR_pred = logistic_regression.predict(X_valid[:,list_n_last_features])

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_LR_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_LR_pred_proba),3))

print('Основные метрики на тестовом наборе:')
print(metrics.classification_report(y_valid,y_valid_LR_pred))

# time: 1m 12s

##### <span style="color:MediumSlateBlue">Анализ влияния scaler преобразования на качество и скорость схождения модели</span>

In [ ]:
# формируем список из необходимых scaler
scalers = [MinMaxScaler(),RobustScaler() ,StandardScaler()]

# сформируем списко для заполнения данными результатов анализа
scalers_data = []

# установим ограничение на время обучения модели в 600 секунд (10 минут)
time_out = 600

for scaler in scalers:
        # для того чтобы следить за выполнением цикла введем индикатор
        print('current scaler: ', scaler)

         # сформируем список для заполнения данными текушего scaler
        current_scaler_data = [scaler]
        
        # выполним scaler преобразование
        X_train_s_balanced = scaler.fit_transform(np.array(X_train_balanced)[:,list_n_last_features])
        X_train_s = scaler.transform(X_train[:,list_n_last_features])
        X_valid_s = scaler.transform(X_valid[:,list_n_last_features])

        try:
                # для модуля func_time_out упакуем обучение модели в функцию try_func
                def try_func():
                        logistic_regression = linear_model.LogisticRegression(random_state=42, max_iter=10000)
                        return logistic_regression.fit(X_train_s_balanced,y_train_balanced)
                    
                # фиксируем время начала
                start_time = time.time()
                
                # запускаем обучение модели с ограничением по времени
                logistic_regression = func_timeout.func_timeout(time_out, try_func)
                
                # фиксируем время окончания обучения
                end_time = time.time()
                
                # запишем время обучения модели
                current_scaler_data.append(round(end_time-start_time))

                # Делаем предсказание для обучающей и валидационной выборки
                y_valid_lr_pred = logistic_regression.predict(X_valid_s)
                
                # для метрик ROC AUC делаем предсказание модели для обучающей и валидационной выборки  
                # в виде вероятности принадлежности к классу 1 
                y_train_lr_pred_proba = logistic_regression.predict_proba(X_train_s)[:,1]
                y_valid_lr_pred_proba = logistic_regression.predict_proba(X_valid_s)[:,1]

                #Считаем метрики ROC AUC yна обуающем и валидационном наборе и добавляем их в спискок
                current_scaler_data.append(round(metrics.roc_auc_score(y_train, y_train_lr_pred_proba),3))
                current_scaler_data.append(round(metrics.roc_auc_score(y_valid, y_valid_lr_pred_proba),3))

                #Считаем метрики для класса 0 на валидационном наборе и добавляем их в список
                current_scaler_data.append(round(metrics.recall_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))
                current_scaler_data.append(round(metrics.precision_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))


                #Считаем метрики для класса 1 на валидационном набопе и добавляем их в список
                current_scaler_data.append(round(metrics.recall_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))
                current_scaler_data.append(round(metrics.precision_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))

                # добавляем полученные данные в список scalers_data
                scalers_data.append(current_scaler_data)

                # чтобы подстраховаться на случай ошибки : "The Kernel crashed while executing code"
                # сохраним в локальную папку удачную итерацию
                # для этого из полученных данных сформируем dataframe
                # из полученных данных сформируем dataframe
                scaler_pd =pd.DataFrame(
                        columns=['scaler','time_fit',
                                 'roc_train','roc_valid', 
                                 'recall_0','precision_0',
                                 'recall_1','precision_1'],
                        data = scalers_data)
                # сохраняем dataframe
                scaler_pd.to_csv('data_recovery/scaler_pd.csv',index=False)
               
                # очистим output от прошедшей итерации
                clear_output()

        except func_timeout.FunctionTimedOut:
                # если время обучения привышает лимит
                # запишем в current_scaler_data соотвествующую данные
                current_scaler_data = [scaler,'>30m',0,0,0,0,0,0]
                # добавляем полученные данные в список scalers_data
                scalers_data.append(current_scaler_data)

                # чтобы подстраховаться на случай ошибки : "The Kernel crashed while executing code"
                # сохраним в локальную папку удачную итерацию
                # для этого из полученных данных сформируем dataframe
                # из полученных данных сформируем dataframe
                scaler_pd =pd.DataFrame(
                        columns=['scaler','time_fit',
                                 'roc_train','roc_valid', 
                                 'recall_0','precision_0',
                                 'recall_1','precision_1'],
                        data = scalers_data)
                # сохраняем dataframe
                scaler_pd.to_csv('data_recovery/scaler_pd.csv',index=False)
                
                # очистим output от прошедшей итерации
                clear_output()
                # переходим к следующему scaler
                pass
# очистим output
clear_output()

# сформируем данные соотвествующие лучщему ROC AUC на валидацинном наборе
best_roc_valid_data = scaler_pd[scaler_pd['roc_valid']==scaler_pd['roc_valid'].max()]

# выведем лучший результат на валидационном наборе
print('result:')
print('best scaler on valid:',best_roc_valid_data.iloc[0]['scaler'])
print('ROC AUC on train:', best_roc_valid_data.iloc[0]['roc_train'])
print('ROC AUC on valid:', best_roc_valid_data.iloc[0]['roc_valid'])
print('Time fit for best scaler:', best_roc_valid_data.iloc[0]['time_fit'],' seconds')

# выведем полученный dataframe
scaler_pd

# time: 

##### <span style="color:MediumSlateBlue"> Анализ влияния solver оптимизатора на качество и скорость обучения модели</span>

Проверим качество и скорость сходимости модели при разных solver  
Проверку осуществим через цикл  
Чтобы модель не сходилась бесконечно с помощью модуля func_timeout  
поставим ограничение времени схождения в 10 минут

In [ ]:
# подготовим данные для обучения модели

# обьявим scaler
scaler = StandardScaler()
# выполним scaler преобразование
X_train_s_balanced = scaler.fit_transform(np.array(X_train_balanced)[:,list_n_last_features])
X_train_s = scaler.transform(X_train[:,list_n_last_features])
X_valid_s = scaler.transform(X_valid[:,list_n_last_features])

# time: 10s

In [ ]:
# Зададим список solvers
solvers_list = ['lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga']

# сформируем список для заполнения
solvers_data = []

# установим ограничение на время обучения модели в 300 секунд (5 минут)
time_out = 300

for solver in solvers_list:
        # для того чтобы следить за выполнением цикла введем индикатор
        print('current solver: ', solver)

         # сформируем список для заполнения данными текушего solver
        current_solver_data = [solver]
        
        try:
                # для модуля func_time_out упакуем обучение модели в функцию try_func
                def try_func():
                        logistic_regression = linear_model.LogisticRegression(solver= solver,
                                                                  random_state=42, 
                                                                  max_iter=10000)
                        return logistic_regression.fit(X_train_s_balanced,y_train_balanced)
                    
                # фиксируем время начала
                start_time = time.time()
                
                # запускаем обучение модели с ограничением по времени
                logistic_regression = func_timeout.func_timeout(time_out, try_func)
                
                # фиксируем время окончания обучения
                end_time = time.time()
                
                # запишем время обучения модели
                current_solver_data.append(round(end_time-start_time))

                # Делаем предсказание для обучающей и валидационной выборки
                y_valid_lr_pred = logistic_regression.predict(X_valid_s)
                
                # для метрик ROC AUC делаем предсказание модели для обучающей и валидационной выборки  
                # в виде вероятности принадлежности к классу 1 
                y_train_lr_pred_proba = logistic_regression.predict_proba(X_train_s)[:,1]
                y_valid_lr_pred_proba = logistic_regression.predict_proba(X_valid_s)[:,1]

                #Считаем метрики ROC AUC yна обуающем и валидационном наборе и добавляем их в спискок
                current_solver_data.append(round(metrics.roc_auc_score(y_train, y_train_lr_pred_proba),3))
                current_solver_data.append(round(metrics.roc_auc_score(y_valid, y_valid_lr_pred_proba),3))

                #Считаем метрики для класса 0 на валидационном наборе и добавляем их в список
                current_solver_data.append(round(metrics.recall_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))
                current_solver_data.append(round(metrics.precision_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))


                #Считаем метрики для класса 1 на валидационном набопе и добавляем их в список
                current_solver_data.append(round(metrics.recall_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))
                current_solver_data.append(round(metrics.precision_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))

                # запишем данные полученные на одной итерации
                solvers_data.append(current_solver_data)

                # чтобы подстраховаться на случай ошибки : "The Kernel crashed while executing code"
                # сохраним в локальную папку удачную итерацию
                # для этого из полученных данных сформируем dataframe
                # из полученных данных сформируем dataframe
                solvers_pd =pd.DataFrame(
                        columns=['solver','time_fit',
                                 'roc_train','roc_valid', 
                                 'recall_0','precision_0',
                                 'recall_1','precision_1'],
                        data = solvers_data)
                # сохраняем dataframe
                solvers_pd.to_csv('data_recovery/solvers_pd.csv',index=False)
               
                # очистим output от прошедшей итерации
                clear_output()

        except func_timeout.FunctionTimedOut:
                # если время обучения привышает лимит
                # запишем в current_solver_data соотвествующую данные
                current_solver_data = [solver,'>10m',0,0,0,0,0,0]
                # добавляем полученные данные в список solvers_data
                solvers_data.append(current_solver_data)

                # чтобы подстраховаться на случай ошибки : "The Kernel crashed while executing code"
                # сохраним в локальную папку удачную итерацию
                # для этого из полученных данных сформируем dataframe
                # из полученных данных сформируем dataframe
                solvers_pd =pd.DataFrame(
                        columns=['solver','time_fit',
                                 'roc_train','roc_valid', 
                                 'recall_0','precision_0',
                                 'recall_1','precision_1'],
                        data = solvers_data)
                # сохраняем dataframe
                solvers_pd.to_csv('data_recovery/solvers_pd.csv',index=False)
                
                # очистим output от прошедшей итерации
                clear_output()
                # переходим к следующему solver
                pass
# очистим output
clear_output()

# сформируем данные соотвествующие лучщему ROC AUC на валидацинном наборе
best_roc_valid_data = solvers_pd[solvers_pd['roc_valid']==solvers_pd['roc_valid'].max()]

# выведем лучший результат на валидационном наборе
print('result:')
print('best solver on valid:',best_roc_valid_data.iloc[0]['solver'])
print('ROC AUC on train:', best_roc_valid_data.iloc[0]['roc_train'])
print('ROC AUC on valid:', best_roc_valid_data.iloc[0]['roc_valid'])
print('Time fit for best solver:', best_roc_valid_data.iloc[0]['time_fit'],' seconds')

# выведем полученный dataframe
solvers_pd

# time: 11m 35s

##### <span style="color:MediumSlateBlue">Подбор гиперпараметров модели</span>

In [ ]:
# для выбора sceler преобразвания на понадобится словарь
dict_scalers ={
    'MinMaxScaler':MinMaxScaler(),
    'RobustScaler':RobustScaler(),
    'StandardScaler':StandardScaler(),
}
# определим пространство признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'late'
count_features = dict_spaces[feature_space]

# настроим оптимизацию гипер параметров
def optuna_lg(trial):
  # задаем пространства поиска гиперпараметров
  C = trial.suggest_float('C',0.01,1)
  solver = trial.suggest_categorical('solver', ['lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky'])
  class_0_weight = trial.suggest_float('class_0_weight',0.01,1)
  class_1_weight = trial.suggest_float('class_1_weight',0.01,1)
  n_last = trial.suggest_int('n_last', 10, 25,step = 1)
  class_1_percent = trial.suggest_float('class_1_percent',0.01,1)
  scaler = trial.suggest_categorical('scaler', ['MinMaxScaler', 'RobustScaler','StandardScaler'])
  random_state = trial.suggest_int('random_state', 1, 1000000)

  # создаем модель
  optuna_log_reg = linear_model.LogisticRegression(
      solver=solver,
      C=C,
      class_weight={0:class_0_weight,1:class_1_weight},
      random_state=random_state,
      max_iter=10000)

  # с помощью функции features_from_transform_data_torow извлечем  
  # из данных transform_data_torow
  # признаки сооствествующие n_last последним клиенским операциям
  list_n_last_features = features_from_transform_data_torow(n_last,count_features,25)

  # обьявим scaler
  scaler = dict_scalers[scaler]

  # загружаем обучующие наборы
  X_train_pd = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas()
  y_train_pd = pd.read_csv('target/target_train.csv')

  # поготовим данные y_train_pd к работе с функцией class_1_percent_samples
  y_train_pd.set_index('id',drop=False,inplace=True)

  # подготовим данные для обучения модели
  # с помощью функции class_1_percent_samples зададим долю 
  # класса 1
  list_c1_percent_id = class_1_percent_samples(y_train_pd,class_1_percent,random_state=random_state)[:1000000]

  # сформируем данные для обучения базовой модели
  X_train_s_balanced = scaler.fit_transform(np.array(X_train_pd.loc[list_c1_percent_id])[:,list_n_last_features])
  y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

  # освободим память от "тяжелых" и ненужных файлов
  del X_train_pd, y_train_pd
  gc.collect()

  # я не воспользовался параметром timeout метода optimize потому, что  
  # optimize отставливает поиск параметров, если время обучения 
  # превышает timeout. Мне нужно чтобы поиск параметров продолжился.
  try:
    # для модуля func_time_out упакуем обучение модели в функцию try_func
    def try_func():
            return optuna_log_reg.fit(X_train_s_balanced, y_train_balanced)
    # обучаем модель с ограничением по времени 10 минут (600 секунд)
    optuna_log_reg = func_timeout.func_timeout(600, try_func)

    # удаляем крупные файлы чтобы высвободить память 
    del X_train_s_balanced,y_train_balanced
    gc.collect()

    # подгружаем данные для тестирования
    X_train = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas().to_numpy()[:,list_n_last_features]
    y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
    X_valid = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_valid').to_pandas().to_numpy()[:,list_n_last_features]
    y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()

    # сформируем данные для проверки модели
    X_train_s = scaler.transform(X_train)
    X_valid_s = scaler.transform(X_valid)

    # удаляем крупные файлы чтобы высвободить память 
    del X_train, X_valid
    gc.collect()

    # делаем предсказание на обучающем и валидационном наборе и считаем метрики
    roc_train = metrics.roc_auc_score(y_train, optuna_log_reg.predict_proba(X_train_s)[:,1])
    roc_valid = metrics.roc_auc_score(y_valid, optuna_log_reg.predict_proba(X_valid_s)[:,1])
    f1_score = metrics.f1_score(y_valid, optuna_log_reg.predict(X_valid_s))
    recall_1 = metrics.recall_score(y_valid, optuna_log_reg.predict(X_valid_s),average=None,zero_division=0)[1]
    precision_1 = metrics.precision_score(y_valid, optuna_log_reg.predict(X_valid_s),average=None,zero_division=0)[1]

    del X_train_s, X_valid_s, y_train, y_valid
    gc.collect()

  except func_timeout.FunctionTimedOut:
    # освободим память от "тяжелых" и ненужных файлов
    del X_train_s_balanced, y_train_balanced
    gc.collect()

    # обьявим метрики пустыми значениями
    roc_train = 0
    roc_valid = 0
    f1_score = 0
    recall_1 = 0
    precision_1 = 0
    pass

  return roc_train, roc_valid, f1_score, recall_1, precision_1

In [ ]:
# cоздаем объект исследования
# чтобы модель не переобучалась минимизируем roc_train 
# и максимизируем roc_valid
optuna_study_lg = optuna.create_study(study_name='LogisticRegression_'+feature_space+'_torow', 
                               directions=['maximize','maximize','maximize','maximize','maximize'], 
                               sampler=optuna.samplers.TPESampler(),
                               pruner='Hyperband',
                               storage='sqlite:///optuna_studies.db',
                               load_if_exists=True)
# на случай удаления 
# optuna.delete_study(study_name='LogisticRegression_'+feature_space+'_torow', storage='sqlite:///optuna_studies.db')

In [ ]:
# ищем лучшую комбинацию гиперпараметров n_trials раз
optuna_study_lg.optimize(optuna_lg, n_trials=300)

In [ ]:
# из полученного результат соврмируем Data Frame
optuna_study_lg_pd = optuna_study_lg.trials_dataframe().dropna()
# переименуем столбы
optuna_study_lg_pd.rename(columns={
    'values_0': 'roc_train',
    'values_1': 'roc_valid',
    'values_2': 'f1_score',
    'values_3': 'recall_1',
    'values_4': 'precision_1'
},inplace=True)
# Выделим время потраченное на обучение модели
optuna_study_lg_pd.head(5)

In [ ]:
# покаждем статистику обучения
print('Максимальное значение метрики ROC AUC на валидационном наборе:',round(optuna_study_lg_pd['roc_valid'].max(),3))
print('Среднее значение метрики ROC AUC на валидационном наборе:',round(optuna_study_lg_pd['roc_valid'].mean(),3))
print('Максимальное значение метрики f1_score на валидационном наборе:',round(optuna_study_lg_pd['f1_score'].max(),3))
print('Среднее значение метрики f1_score на валидационном наборе:',round(optuna_study_lg_pd['f1_score'].mean(),3))
print('Максимальное значение метрики recall_1 на валидационном наборе:',round(optuna_study_lg_pd['recall_1'].max(),3))
print('Среднее значение метрики recall_1 на валидационном наборе:',round(optuna_study_lg_pd['recall_1'].mean(),3))
print('Максимальное значение метрики precision_1 на валидационном наборе:',round(optuna_study_lg_pd['precision_1'].max(),3))
print('Среднее значение метрики precision_1 на валидационном наборе:',round(optuna_study_lg_pd['precision_1'].mean(),3))

In [ ]:
# построим график важности гиперпарметров
optuna.visualization.plot_param_importances(optuna_study_lg, target_name='Score')

In [ ]:
# Построим зависимость метрики precision_1 от гипер параметров

fig= px.scatter(
    data_frame=optuna_study_lg_pd,
    x='precision_1', #ось абсцисс
    y=['recall_1','f1_score', 
        'params_C','params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight', 'params_n_last',
    ], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision_1',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики precision_1 от гипер параметров

fig= px.scatter(
    data_frame=optuna_study_lg_pd,
    x='f1_score', #ось абсцисс
    y=['params_C','params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight', 'params_n_last',
    ], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision_1',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики precision_1 от гипер параметров

fig= px.scatter(
    data_frame=optuna_study_lg_pd,
    x='roc_valid', #ось абсцисс
    y=['params_C','params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight', 'params_n_last',
    ], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость roc valid от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики roc valid',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Визуализируем метрики полученные при optuna оптимизации
fig = px.scatter(
    data_frame=optuna_study_lg_pd[
                                (optuna_study_lg_pd['roc_valid']>0.9*optuna_study_lg_pd['roc_valid'].max())& 
                                (optuna_study_lg_pd['precision_1']>0.22*optuna_study_lg_pd['precision_1'].max())&
                                (optuna_study_lg_pd['recall_1']>0.7*optuna_study_lg_pd['recall_1'].max())],
    x='number', #ось абсцисс
    y=['roc_train','roc_valid','recall_1','precision_1'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость качества модели от trials optuna', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='trials optuna',
    yaxis_title='Величина метрики')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

<span style="color:Blue">

Вывод:  

Их всех выбираем точку $number =42$.   
В этой точке одно из самых больших значений $ROC AUC$.  
Посмотрим каким значениям гиперпараметров соотвествует выбранная точка.

In [ ]:
# определим номер лучшге варианта
best_optuna_number = 42

# сформируем словарь лучших гипирпарметров RandomForestClassifier
best_param_lr_torow = {
    'C' : round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_C'].iloc[0],3),
    'class_weight' : {0:round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_class_0_weight'].iloc[0],3),
                      1:round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_class_1_weight'].iloc[0],3)},
    }
# создадим перменные
best_n_last = int(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_n_last'].iloc[0])
best_scaler = optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_scaler'].iloc[0]
best_class_1_percent = optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_class_1_percent'].iloc[0]
best_random_state = int(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_random_state'].iloc[0])
# Выведем принятые наилучшие праметры
print('best C:',best_param_lr_torow['C'])
print('best class 0 weight:',best_param_lr_torow['class_weight'][0])
print('best class 1 weight:',best_param_lr_torow['class_weight'][1])

print('best class 1 percent:',round(best_class_1_percent,3))
print('best scaler:',best_scaler)
print('best n_last:',round(best_n_last,3))
print('best best random state:',best_random_state)
print('time for best train:',round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['duration'].iloc[0].seconds/60),'minutes')

print()
print('ROC AUC на обучающем наборе:', round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['roc_train'].iloc[0],3))
print('ROC AUC на валидационном наборе:', round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['roc_valid'].iloc[0],3))
print('precision класса 1:', round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['precision_1'].iloc[0],3))
print('recall класса 1:', round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['recall_1'].iloc[0],3))

##### <span style="color:MediumSlateBlue">Обучение модели с лучшими параметрами</span>

In [ ]:
# для выбора sceler преобразвания на понадобится словарь
dict_scalers ={
    'MinMaxScaler':MinMaxScaler(),
    'RobustScaler':RobustScaler(),
    'StandardScaler':StandardScaler(),
}

# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'late'
count_features = dict_spaces[feature_space]

# с помощью функции features_from_transform_data_torow извлечем  
# из данных transform_data_torow_25
# признаки сооствествующие n_last последним клиенским операциям
list_n_last_features = features_from_transform_data_torow(best_n_last,count_features,25)

# обьявим scaler
scaler = dict_scalers[best_scaler]

# загружаем обучующие наборы
X_train_pd = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas()
y_train_pd = pd.read_csv('target/target_train.csv')

# поготовим данные y_train_pd к работе с функцией class_1_percent_samples
y_train_pd.set_index('id',drop=False,inplace=True)

# подготовим данные для обучения модели
# с помощью функции class_1_percent_samples зададим долю 
# класса 1
list_c1_percent_id = class_1_percent_samples(y_train_pd,best_class_1_percent,random_state=best_random_state)[:1000000]

# сформируем данные для обучения базовой модели
X_train_s_balanced = scaler.fit_transform(np.array(X_train_pd.loc[list_c1_percent_id])[:,list_n_last_features])
y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

# освободим память от "тяжелых" и ненужных файлов
del X_train_pd, y_train_pd
gc.collect()


# обучаем модель LogisticRegression с наилучшеми параметрами
logistic_regression = linear_model.LogisticRegression(
        **best_param_lr_torow,
        random_state=best_random_state,
        max_iter=10000)
logistic_regression.fit(X_train_s_balanced,y_train_balanced)

# удаляем крупные файлы чтобы высвободить память 
del X_train_s_balanced,y_train_balanced
gc.collect()

# подгружаем данные для тестирования
X_train = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas().to_numpy()[:,list_n_last_features]
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
X_valid = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_valid').to_pandas().to_numpy()[:,list_n_last_features]
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()
    
# сформируем данные для проверки модели
X_train_s = scaler.transform(X_train)
X_valid_s = scaler.transform(X_valid)

# удаляем крупные файлы чтобы высвободить память 
del X_train, X_valid
gc.collect()

# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_lr_pred_proba = logistic_regression.predict_proba(X_train_s)[:,1]
y_valid_lr_pred_proba = logistic_regression.predict_proba(X_valid_s)[:,1]

# Делаем предсказание для валидационной выборки
y_valid_lr_pred = logistic_regression.predict(X_valid_s)

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_lr_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_lr_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_valid,y_valid_lr_pred,zero_division=0))

# time: 2m 40s

##### <span style="color:MediumSlateBlue">Анализ влияния порога классификации на качество модели</span>

In [ ]:
# чтобы проше обращаться к каждому обьекту в данных
# преобразуем y_valid_LR_pred к Series типу
y_valid_lr_pred_proba = pd.Series(y_valid_lr_pred_proba)

# формируем список из необходимых значений thresholds
thresholds = np.arange(0.01, 1, 0.01)

# сформируем списко для заполнения данными результатов анализа
threshold_data = []


for threshold in thresholds:
        # сформируем список для заполнения данными текушего threshold
        threshold_current_data = [threshold]

        #клиентов, для которых вероятность дефолта > threshold, относим к классу 1
        #в противном случае — к классу 0
        y_valid_lr_pred = y_valid_lr_pred_proba.apply(lambda x: 1 if x>threshold else 0)


        #Считаем метрики для класса 0 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.precision_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.f1_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))

        #Считаем метрики для класса 1 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.precision_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.f1_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))

        # добавляем полученные данные в список threshold_data
        threshold_data.append(threshold_current_data)

        # из полученных данных сформируем dataframe
        thresholds_pd =pd.DataFrame(
        columns=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'],
        data = threshold_data)

# time: 17s

In [ ]:
# Визуализируем метрики на валидационном наборе при различных threshold
fig_threshold = px.line(
    data_frame=thresholds_pd,
    x='threshold', #ось абсцисс
    y=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'], #ось ординат
)
fig_threshold.update_layout(
    title ={
        'text':'Зависимость качества модели от threshold', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Порог классификации threshold',
    yaxis_title='Величина метрики')
fig_threshold.update_xaxes(showspikes=True)
fig_threshold.update_yaxes(showspikes=True)

fig_threshold.show()

Метрика $precision$ показывает на сколько сильно ошиблась модель при определиние класса.  
Чем выше метрика, тем меньше ошиблась модель. 

$$precision = \frac{TP}{TP+FP}$$

Метрика $recall$ показывает способность модели найти класс.  
Чем выше метрика, тем выше способность модели находить класс.  

$$recall = \frac{TP}{TP+FN}$$

Метрика $f1-score$ показывает баланс между $precision$ и $recall$ 

$threshold$- порог класификации модели.  
Все объекты для которых $у_{\text{proba pred}} < threshold$, модель относит к класс 0.  
Соотвественно, если $у_{\text{proba pred}}  > threshold$ модель относит обьект к классу 1.  
Для идельаной модели $threshold$, является границей между областями класса 0 и класса 1.  

<span style="color:Blue">

Выводы по результам анализа:  

Для улучшения качества модели по метрике $ROC AUC$, нет смысла сдвигать порог   
классификации. Но зависимость метрик качества модели от $threshold$, может рассказать о  
спосбоности модели искать класс 1 и класс 0.

При фокусировке модели на классе 1 (более низкий $threshold$):  
- на классе 1 метрика $precision$ уменьшается, метрика $recall$ растет;  
- на классе 0 метрика $precision$ растет, метрика $recall$ уменьшается.  

При фокусировке модели на классе 0 (более высокий $threshold$):  
- на классе 1 метрика $precision$ увеличивается, метрика $recall$ уменьшается;  
- на классе 0 метрика $precision$ уменьшается, метрика $recall$ растет.  

Обьсняется тем, что в условия дисбаланса модель научилась хорошо определять класс 0 и плохо класс 1.  

При фокусировке модели на классе 1 (более низкий $threshold$), все больше обьектов отнесены к классу 1.   
Так как $precision$ класса 1 уменьшается, значит в области класса 1 стало больше обектов класса 0,  
что явлется закономерным для "идеальной" модели.  
Однако при этом, $recall$ класса 1 растет, значит в добавленных обьектах также присутвует класс 1.  
Это как раз указывает на то, что модель не нучилась находить среди класса 0 класс 1. В области с     
высокой вероятностью класса 0 (более низкий $threshold$), находятся обьекты класса 1.  

Фокусировка на классе 0 (более высокий $threshold$), заставляет модель все больше обьектов из области класса 1  
относить к классу 0. При этом увеличение метрики $precision$ класса 1 говорит о том, что в области класса 1     
становится меньше 0, а значит модель умеет среди класса 1, находить класс 0.  


Обратим внимание на $threshold=0.72$, в этой точке находится баланс межу метриками качества модели:
$$precision_1 = recall_1 = \text{f1-score}_1 = 0.099$$
$$precision_0 = recall_0 = \text{f1-score}_0 = 0.965$$

Для модели с лучшим качеством эти две точки должны находится максимально близко к друг другу и иметь  
высокое значение метрик $precision_1$ и $recall_1$.  
Для класса 0 это условие выполнется, что также говорит о том, что модель научилась искать этот класс.   
Для класса 1 это условие не выполнется, что также говорит о том, что модель плохо ищет класс 1.

##### <span style="color:MediumSlateBlue">Построение ROC-кривой выбранной модели на тестовом наборе</span>

In [ ]:
# определим пространство признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'late'
count_features = dict_spaces[feature_space]

In [ ]:
# подгружаем данные для тестирования
X_train = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas().to_numpy()[:,list_n_last_features]
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
X_valid = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_valid').to_pandas().to_numpy()[:,list_n_last_features]
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()
X_test = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_test').to_pandas().to_numpy()[:,list_n_last_features]
y_test = pd.read_csv('target/target_test.csv')['flag'].to_numpy()

In [ ]:
# выполним scaler преобразование
X_train_s = scaler.transform(X_train)
X_valid_s = scaler.transform(X_valid)
X_test_s = scaler.transform(X_test)

print("Размеры обучающего набора",X_train_s.shape)
print("Размеры валидационного набора",X_valid_s.shape)
print("Размеры тестового набора",X_test_s.shape)
# time: 1m 43s

In [ ]:
# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_lr_pred_proba = logistic_regression.predict_proba(X_train_s)[:,1]
y_valid_lr_pred_proba = logistic_regression.predict_proba(X_valid_s)[:,1]
y_test_lr_pred_proba = logistic_regression.predict_proba(X_test_s)[:,1]

# Делаем предсказание для валидационной выборки
y_test_lr_pred = logistic_regression.predict(X_test_s)

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_lr_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_lr_pred_proba),3))
print('ROC AUC на тестовом наборе', round(metrics.roc_auc_score(y_test, y_test_lr_pred_proba),3))

print('Основные метрики на тестовом наборе:')
print(metrics.classification_report(y_test,y_test_lr_pred,zero_division=0))

In [ ]:
# Для расчета значений TPR и FPR используем функцию roc_curve
fpr_train, tpr_train, _ = metrics.roc_curve(y_train, y_train_lr_pred_proba)
fpr_valid, tpr_valid, _ = metrics.roc_curve(y_valid, y_valid_lr_pred_proba)
fpr_test, tpr_test, _ = metrics.roc_curve(y_test, y_test_lr_pred_proba)

# рассчитаем площадь под кривой
auc_train = round(metrics.roc_auc_score(y_train, y_train_lr_pred_proba),3)
auc_valid = round(metrics.roc_auc_score(y_valid, y_valid_lr_pred_proba),3)
auc_test = round(metrics.roc_auc_score(y_test, y_test_lr_pred_proba),3)

trace_train = go.Scatter(x=fpr_train, y=tpr_train, mode='lines', name='AUC_train = %0.2f' % auc_train,
                   line=dict(color='red', width=2),
                   hovertemplate='train<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_train])

trace_valid = go.Scatter(x=fpr_valid, y=tpr_valid, mode='lines', name='AUC_valid = %0.2f' % auc_valid,
                   line=dict(color='green', width=2),
                   hovertemplate='valid<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_valid])

trace_test = go.Scatter(x=fpr_test, y=tpr_test, mode='lines', name='AUC_test = %0.2f' % auc_test,
                   line=dict(color='darkorange', width=2),
                   hovertemplate='test<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_test])


reference_line = go.Scatter(x=[0,1], y=[0,1], mode='lines', name='Reference Line',
                            line=dict(color='navy', width=2, dash='dash'))

fig_roc = go.Figure(data=[trace_train,trace_valid,trace_test,reference_line])

fig_roc.update_layout(
    title ={
        'text':'ROC-кривая', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 1350,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate')

fig_roc.update_xaxes(showspikes=True)
fig_roc.update_yaxes(showspikes=True)


fig_roc.show()

<span style="color:Blue">

Вывод по ROC-кривой:
1. Модель выдает сталибильное качество на всех наборах, а значит  
не переобучена - разброс ($variance$) не большой.  
2. Смещение ($bias$) модели относительно большое.  
На тестовом наборе полученно качество $ROC AUC = 0.62$.


#### <span style="color:MediumBlue">Построение модели Random Forest Classifier</span>

##### <span style="color:MediumSlateBlue">Baseline обучение модели RandomForestClassifier</span>

In [ ]:
# обучаем модель логистической регрессии
# чтобы модель пыталась найти связь во всех признаках
random_forest = RandomForestClassifier(random_state=42, n_jobs=-1)
random_forest.fit(np.array(X_train_balanced)[:,list_n_last_features],np.array(y_train_balanced))

# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_rf__pred_proba = random_forest.predict_proba(X_train[:,list_n_last_features])[:,1]
y_valid_rf_pred_proba = random_forest.predict_proba(X_valid[:,list_n_last_features])[:,1]

# Делаем предсказание для валидационной выборки
y_valid_rf_pred = random_forest.predict(X_valid[:,list_n_last_features])

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_rf__pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_rf_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_valid,y_valid_rf_pred))

# time: 2m 5s

##### <span style="color:MediumSlateBlue">Подбор гиперпараметров модели</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'late'
count_features = dict_spaces[feature_space]

# настроим оптимизацию гипер параметров
def optuna_rf(trial):
  # задаем пространства поиска гиперпараметров
  # задаем пространства поиска гиперпараметров
  n_estimators = trial.suggest_int('n_estimators', 50, 150,step = 5)
  criterion = trial.suggest_categorical('criterion',['gini', 'entropy', 'log_loss'])
  max_depth = trial.suggest_int('max_depth', 9, 20,step = 1)
  min_samples_split = trial.suggest_int('min_samples_split', 5, 15,step = 1)
  min_samples_leaf = trial.suggest_int('min_samples_leaf', 5, 25,step = 1)
  max_features = trial.suggest_float('max_features',0.1,1)
  max_samples = trial.suggest_float('max_samples',0.1,1)
  class_0_weight = trial.suggest_float('class_0_weight',0.01,1)
  class_1_weight = trial.suggest_float('class_1_weight',0.01,1)
  n_last = trial.suggest_int('n_last', 5, 25,step = 1)
  class_1_percent = trial.suggest_float('class_1_percent',0.01,0.9)
  random_state = trial.suggest_int('random_state', 1, 1000000)


  # создаем модель
  optuna_random_forest = RandomForestClassifier(
      n_estimators = n_estimators, 
      criterion = criterion,
      max_depth = max_depth,
      min_samples_split = min_samples_split,  
      min_samples_leaf = min_samples_leaf,
      max_features=max_features,
      max_samples=max_samples,
      class_weight={0:class_0_weight,1:class_1_weight},
      n_jobs=-1,
      random_state=random_state)

  # с помощью функции features_from_transform_data_torow извлечем  
  # из данных transform_data_torow
  # признаки сооствествующие n_last последним клиенским операциям
  list_n_last_features = features_from_transform_data_torow(n_last,count_features,25)

  # загружаем обучующие наборы
  X_train_pd = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas()
  y_train_pd = pd.read_csv('target/target_train.csv')

  # поготовим данные y_train_pd к работе с функцией class_1_percent_samples
  y_train_pd.set_index('id',drop=False,inplace=True)

  # подготовим данные для обучения модели
  # с помощью функции class_1_percent_samples зададим долю 
  # класса 1
  list_c1_percent_id = class_1_percent_samples(y_train_pd,class_1_percent,random_state=random_state)[:1000000]

  # сформируем данные для обучения базовой модели
  X_train_balanced = np.array(X_train_pd.loc[list_c1_percent_id])[:,list_n_last_features]
  y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

  # освободим память от "тяжелых" и ненужных файлов
  del X_train_pd, y_train_pd
  gc.collect()

  # я не воспользовался параметром timeout метода optimize потому, что  
  # optimize отставливает поиск параметров, если время обучения 
  # превышает timeout. Мне нужно чтобы поиск параметров продолжился.
  try:
    # для модуля func_time_out упакуем обучение модели в функцию try_func
    def try_func():
            return optuna_random_forest.fit(X_train_balanced,y_train_balanced)
    # обучаем модель с ограничением по времени 10 минут (600 секунд)
    optuna_random_forest = func_timeout.func_timeout(600, try_func)

    # удаляем крупные файлы чтобы высвободить память 
    del X_train_balanced, y_train_balanced
    gc.collect()

    # подгружаем данные для тестирования
    X_train = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas().to_numpy()[:,list_n_last_features]
    y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
    X_valid = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_valid').to_pandas().to_numpy()[:,list_n_last_features]
    y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()

    # делаем предсказание на обучающем и валидационном наборе и считаем метрики
    roc_train = metrics.roc_auc_score(y_train, optuna_random_forest.predict_proba(X_train)[:,1])
    roc_valid = metrics.roc_auc_score(y_valid, optuna_random_forest.predict_proba(X_valid)[:,1])
    f1_score = metrics.f1_score(y_valid, optuna_random_forest.predict(X_valid))
    recall_1 = metrics.recall_score(y_valid, optuna_random_forest.predict(X_valid),average=None,zero_division=0)[1]
    precision_1 = metrics.precision_score(y_valid, optuna_random_forest.predict(X_valid),average=None,zero_division=0)[1]

    # удаляем крупные файлы чтобы высвободить память 
    del X_train, X_valid, y_train, y_valid 
    gc.collect()

  except func_timeout.FunctionTimedOut:
    # освободим память от "тяжелых" и ненужных файлов
    del X_train_balanced, y_train_balanced
    gc.collect()

    # обьявим метрики пустыми значениями
    roc_train = 0
    roc_valid = 0
    f1_score = 0
    recall_1 = 0
    precision_1 = 0
    pass

  return roc_train, roc_valid, f1_score, recall_1, precision_1

In [ ]:
# так как Random Forest склонен к переобучению 
# попробуем направить optimize в сторону уменьшения roc_train
# cоздаем объект исследования
optuna_study_rf = optuna.create_study(study_name='RandomForestClassifier_'+feature_space+'_torow', 
                               directions=['minimize','maximize','maximize','maximize','maximize'], 
                               sampler=optuna.samplers.TPESampler(),
                               pruner='Hyperband',
                               storage='sqlite:///optuna_studies.db',
                               load_if_exists=True)
# на случай удаления 
# optuna.delete_study(study_name='RandomForestClassifier_'+feature_space+'_torow', storage='sqlite:///optuna_studies.db')

In [ ]:
# ищем лучшую комбинацию гиперпараметров n_trials раз
optuna_study_rf.optimize(optuna_rf, n_trials=300)

In [ ]:
# из полученного результат соврмируем Data Frame
optuna_study_rf_pd = optuna_study_rf.trials_dataframe().dropna()
# переименуем столбы
optuna_study_rf_pd.rename(columns={
    'values_0': 'roc_train',
    'values_1': 'roc_valid',
    'values_2': 'f1_score',
    'values_3': 'recall_1',
    'values_4': 'precision_1'
},inplace=True)
optuna_study_rf_pd.sort_values(by='precision_1').drop(['datetime_start','datetime_complete','duration'],axis=1)

In [ ]:
# покаждем статистику обучения
print('Максимальное значение метрики ROC AUC на валидационном наборе:',round(optuna_study_rf_pd['roc_valid'].max(),3))
print('Среднее значение метрики ROC AUC на валидационном наборе:',round(optuna_study_rf_pd['roc_valid'].mean(),3))
print('Максимальное значение метрики f1_score на валидационном наборе:',round(optuna_study_rf_pd['f1_score'].max(),3))
print('Среднее значение метрики f1_score на валидационном наборе:',round(optuna_study_rf_pd['f1_score'].mean(),3))
print('Максимальное значение метрики recall_1 на валидационном наборе:',round(optuna_study_rf_pd['recall_1'].max(),3))
print('Среднее значение метрики recall_1 на валидационном наборе:',round(optuna_study_rf_pd['recall_1'].mean(),3))
print('Максимальное значение метрики precision_1 на валидационном наборе:',round(optuna_study_rf_pd['precision_1'].max(),3))
print('Среднее значение метрики precision_1 на валидационном наборе:',round(optuna_study_rf_pd['precision_1'].mean(),3))

In [ ]:
# построим график важности гиперпарметров
optuna.visualization.plot_param_importances(optuna_study_rf, target_name='Score')

In [ ]:
# Построим зависимость метрики precision_1 от гиперпараметров

fig = px.scatter(
    data_frame=optuna_study_rf_pd,
    x='precision_1', #ось абсцисс
    y=['recall_1', 'f1_score'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision класса 1 от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision на классе 1',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики precision_1 от гиперпараметров

fig = px.scatter(
    data_frame=optuna_study_rf_pd,
    x='precision_1', #ось абсцисс
    y=['params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight','params_max_depth','params_max_features',
        'params_max_samples', 'params_min_samples_leaf', 'params_min_samples_split', 
        'params_n_estimators', 'params_n_last'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision класса 1 от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision на классе 1',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики roc_valid от гиперпараметров

fig = px.scatter(
    data_frame=optuna_study_rf_pd,
    x='f1_score', #ось абсцисс
    y=['params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight','params_max_depth','params_max_features',
        'params_max_samples', 'params_min_samples_leaf', 'params_min_samples_split', 
        'params_n_estimators', 'params_n_last'],
)
fig.update_layout(
    title ={
        'text':'Зависимость ROC AUC на валидационном наборе от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики ROC AUC',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики roc_valid от гиперпараметров

fig = px.scatter(
    data_frame=optuna_study_rf_pd,
    x='roc_valid', #ось абсцисс
    y=['params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight','params_max_depth','params_max_features',
        'params_max_samples', 'params_min_samples_leaf', 'params_min_samples_split', 
        'params_n_estimators', 'params_n_last'],
)
fig.update_layout(
    title ={
        'text':'Зависимость ROC AUC на валидационном наборе от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики ROC AUC',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрик качества модели от number

fig = px.scatter(
    data_frame=optuna_study_rf_pd[(optuna_study_rf_pd['roc_valid'])>=0.999*(optuna_study_rf_pd['roc_valid'].max())],
    x='number', #ось абсцисс
    y=['roc_valid','roc_train','precision_1','recall_1'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость метрик качества модели от number', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики ROC Valid',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

<span style="color:Blue">

Вывод:  

Их всех выбираем точку $number =148$.   
В этой точке оптимальные значения метрик $recall_1$ и $precision_1$, а  
также относительно высокая метрика $ROC AUC$.   
Посмотрим каким значениям гиперпараметров соотвествует выбраннаяточка.

In [ ]:
# определим номер лучшге варианта
best_optuna_number = 148

# сформируем словарь лучших гипирпарметров RandomForestClassifier
best_param_rf = {
    'n_estimators' : int(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_n_estimators'].iloc[0]),
    'criterion': optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_criterion'].iloc[0],
    'max_depth' : optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_max_depth'].iloc[0],
    'min_samples_split': optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_min_samples_split'].iloc[0],
    'min_samples_leaf' : optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_min_samples_leaf'].iloc[0],
    'max_features': optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_max_features'].iloc[0],
    'max_samples' : optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_max_samples'].iloc[0],
    'class_weight' : {0:optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_class_0_weight'].iloc[0],
                      1:optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_class_1_weight'].iloc[0]}
    }
# определим перменные для лучших значений параметров
best_n_last = optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_n_last'].iloc[0]
best_class_1_percent = optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_class_1_percent'].iloc[0]
best_random_state = optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_random_state'].iloc[0]


# Выведем принятые наилучшие праметры
print('best n estimators:',best_param_rf['n_estimators'])
print('best criterion:',best_param_rf['criterion'])
print('best max depth:',best_param_rf['max_depth'])
print('best min samples split:',best_param_rf['min_samples_split'])
print('best min samples leaf:',best_param_rf['min_samples_leaf'])
print('best max features(%):',round(best_param_rf['max_features'],3))
print('best max samples(%):',round(best_param_rf['max_samples'],3))
print('best class 0 weight:',round(best_param_rf['class_weight'][0],3))
print('best class 1 weight:',round(best_param_rf['class_weight'][1],3))

print('best class 1 percent:',round(best_class_1_percent,3))
print('best n last:',best_n_last)
print('best random state:',best_random_state)
print('time for best train:',round(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['duration'].iloc[0].seconds/60),'minutes')

print()
print('ROC AUC на обучающем наборе:', round(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['roc_train'].iloc[0],3))
print('ROC AUC на валидационном наборе:', round(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['roc_valid'].iloc[0],3))
print('precision класса 1:', round(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['precision_1'].iloc[0],3))
print('recall класса 1:', round(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['recall_1'].iloc[0],3))

##### <span style="color:MediumSlateBlue">Обучение модели с лучшими параметрами</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'late'
count_features = dict_spaces[feature_space]

# с помощью функции features_from_transform_data_torow извлечем  
# из данных transform_data_torow
# признаки сооствествующие n_last последним клиенским операциям
list_n_last_features = features_from_transform_data_torow(best_n_last,count_features,25)

# загружаем обучующие наборы
X_train_pd = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas()
y_train_pd = pd.read_csv('target/target_train.csv')

# поготовим данные y_train_pd к работе с функцией class_1_percent_samples
y_train_pd.set_index('id',drop=False,inplace=True)

# подготовим данные для обучения модели
# с помощью функции class_1_percent_samples зададим долю 
# класса 1
list_c1_percent_id = class_1_percent_samples(y_train_pd,best_class_1_percent,random_state=best_random_state)[:1000000]

# сформируем данные для обучения базовой модели
X_train_balanced = np.array(X_train_pd.loc[list_c1_percent_id])[:,list_n_last_features]
y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

# освободим память от "тяжелых" и ненужных файлов
del X_train_pd, y_train_pd
gc.collect()

# обучаем модель RandomForestClassifier с наилучшеми параметрами
random_forest = RandomForestClassifier(
        **best_param_rf,
        n_jobs=-1,
        random_state=best_random_state)
random_forest.fit(X_train_balanced, y_train_balanced)

# освободим память от "тяжелых" и ненужных файлов
del X_train_balanced, y_train_balanced
gc.collect()


# подгружаем данные для тестирования
X_train = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas().to_numpy()[:,list_n_last_features]
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
X_valid = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_valid').to_pandas().to_numpy()[:,list_n_last_features]
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()

# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_rf__pred_proba = random_forest.predict_proba(X_train)[:,1]
y_valid_rf_pred_proba = random_forest.predict_proba(X_valid)[:,1]

# Делаем предсказание для валидационной выборки
y_valid_rf_pred = random_forest.predict(X_valid)

# удаляем крупные файлы чтобы высвободить память 
del X_train, X_valid
gc.collect()

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_rf__pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_rf_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_valid,y_valid_rf_pred,zero_division=0))

# time: 2m 10s

##### <span style="color:MediumSlateBlue">Анализ влияния порога классификации на качество модели</span>

In [ ]:
# чтобы проше обращаться к каждому обьекту в данных
# преобразуем y_valid_LR_pred к Series типу
y_valid_rf_pred_proba = pd.Series(y_valid_rf_pred_proba)

# формируем список из необходимых значений thresholds
thresholds = np.arange(0.01, 1, 0.01)

# сформируем списко для заполнения данными результатов анализа
threshold_data = []


for threshold in thresholds:
        # сформируем список для заполнения данными текушего threshold
        threshold_current_data = [threshold]

        #клиентов, для которых вероятность дефолта > threshold, относим к классу 1
        #в противном случае — к классу 0
        y_valid_rf_pred = y_valid_rf_pred_proba.apply(lambda x: 1 if x>threshold else 0)


        #Считаем метрики для класса 0 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(y_valid, y_valid_rf_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.precision_score(y_valid, y_valid_rf_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.f1_score(y_valid, y_valid_rf_pred,average=None,zero_division=0)[0],3))

        #Считаем метрики для класса 1 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(y_valid, y_valid_rf_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.precision_score(y_valid, y_valid_rf_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.f1_score(y_valid, y_valid_rf_pred,average=None,zero_division=0)[1],3))

        # добавляем полученные данные в список threshold_data
        threshold_data.append(threshold_current_data)

        # из полученных данных сформируем dataframe
        thresholds_pd =pd.DataFrame(
        columns=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'],
        data = threshold_data)

# time: 17s

In [ ]:
# Визуализируем метрики на валидационном наборе при различных threshold
fig_threshold = px.line(
    data_frame=thresholds_pd,
    x='threshold', #ось абсцисс
    y=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'], #ось ординат
)
fig_threshold.update_layout(
    title ={
        'text':'Зависимость качества модели от threshold', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Порог классификации threshold',
    yaxis_title='Величина метрики')
fig_threshold.update_xaxes(showspikes=True)
fig_threshold.update_yaxes(showspikes=True)

fig_threshold.show()

Метрика $precision$ показывает на сколько сильно ошиблась модель при определиние класса.  
Чем выше метрика, тем меньше ошиблась модель. 

$$precision = \frac{TP}{TP+FP}$$

Метрика $recall$ показывает способность модели найти класс.  
Чем выше метрика, тем выше способность модели находить класс.  

$$recall = \frac{TP}{TP+FN}$$

Метрика $f1-score$ показывает баланс между $precision$ и $recall$ 

$threshold$- порог класификации модели.  
Все объекты для которых $у_{\text{proba pred}} < threshold$, модель относит к класс 0.  
Соотвественно, если $у_{\text{proba pred}}  > threshold$ модель относит обьект к классу 1.  
Для идельаной модели $threshold$, является границей между областями класса 0 и класса 1.  

<span style="color:Blue">

Выводы по результам анализа:  

Для улучшения качества модели по метрике $ROC AUC$, нет смысла сдвигать порог   
классификации. Но зависимость метрик качества модели от $threshold$, может рассказать о  
способности модели искать класс 1 и класс 0.

При фокусировке модели на классе 1 (более низкий $threshold$):  
- на классе 1 метрика $precision$ уменьшается, метрика $recall$ растет;  
- на классе 0 метрика $precision$ растет, метрика $recall$ уменьшается.  

При фокусировке модели на классе 0 (более высокий $threshold$):  
- на классе 1 метрика $precision$ увеличивается, метрика $recall$ уменьшается;  
- на классе 0 метрика $precision$ уменьшается, метрика $recall$ растет.  

Обьсняется тем, что в условия дисбаланса модель научилась хорошо определять класс 0 и плохо класс 1.  

При фокусировке модели на классе 1 (более низкий $threshold$), все больше обьектов отнесены к классу 1.   
Так как $precision$ класса 1 уменьшается, значит в области класса 1 стало больше обектов класса 0,  
что явлется закономерным для "идеальной" модели.  
Однако при этом, $recall$ класса 1 растет, значит в добавленных обьектах также присутвует класс 1.  
Это как раз указывает на то, что модель не нучилась находить среди класса 0 класс 1. В области с     
высокой вероятностью класса 0 (более низкий $threshold$), находятся обьекты класса 1.  

Фокусировка на классе 0 (более высокий $threshold$), заставляет модель все больше обьектов из области класса 1  
относить к классу 0. При этом увеличение метрики $precision$ класса 1 говорит о том, что в области класса 1     
становится меньше 0, а значит модель умеет среди класса 1, находить класс 0.  
После  $threshold=0.75$ модель впринципе теряет способность отличать класс 1. У модели нет обьектов класса 1   
в которых она "уверена" на 80+%.

Обратим внимание на $threshold=0.51$, в этой точке находится баланс межу метриками качества модели:
$$precision_1 = recall_1 = \text{f1-score}_1 = 0.101$$
$$precision_0 = recall_0 = \text{f1-score}_0 = 0.966$$

Для модели с лучшим качеством эти две точки должны находится максимально близко к друг другу и иметь   
высокое значение метрик $precision_1$ и $recall_1$.  
Для класса 0 это условие выполнется, что также говорит о том, что модель научилась искать этот класс.    
Для класса 1 это условие не выполнется, что также говорит о том, что модель плохо ищет класс 1.

##### <span style="color:MediumSlateBlue">Построение ROC-кривой выбранной модели на тестовом наборе</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'late'
count_features = dict_spaces[feature_space]

In [ ]:
# подгружаем данные для тестирования
X_train = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas().to_numpy()[:,list_n_last_features]
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
X_valid = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_valid').to_pandas().to_numpy()[:,list_n_last_features]
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()
X_test = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_test').to_pandas().to_numpy()[:,list_n_last_features]
y_test = pd.read_csv('target/target_test.csv')['flag'].to_numpy()

In [ ]:
print("Размеры обучающего набора",X_train.shape,y_train.shape)
print("Размеры валидационного набора",X_valid.shape,y_valid.shape)
print("Размеры тестового набора",X_test.shape,y_test.shape)
# time: 1m 43s

In [ ]:
# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_rf_pred_proba = random_forest.predict_proba(X_train)[:,1]
y_valid_rf_pred_proba = random_forest.predict_proba(X_valid)[:,1]
y_test_rf_pred_proba = random_forest.predict_proba(X_test)[:,1]

# Делаем предсказание для валидационной выборки
y_test_lr_pred = random_forest.predict(X_test)

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_rf_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_rf_pred_proba),3))
print('ROC AUC на тестовом наборе', round(metrics.roc_auc_score(y_test, y_test_rf_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_test,y_test_lr_pred,zero_division=0))

In [ ]:
# Для расчета значений TPR и FPR используем функцию roc_curve
fpr_train, tpr_train, _ = metrics.roc_curve(y_train, y_train_rf_pred_proba)
fpr_valid, tpr_valid, _ = metrics.roc_curve(y_valid, y_valid_rf_pred_proba)
fpr_test, tpr_test, _ = metrics.roc_curve(y_test, y_test_rf_pred_proba)

# рассчитаем площадь под кривой
auc_train = round(metrics.roc_auc_score(y_train, y_train_rf_pred_proba),3)
auc_valid = round(metrics.roc_auc_score(y_valid, y_valid_rf_pred_proba),3)
auc_test = round(metrics.roc_auc_score(y_test, y_test_rf_pred_proba),3)

trace_train = go.Scatter(x=fpr_train, y=tpr_train, mode='lines', name='AUC_train = %0.2f' % auc_train,
                   line=dict(color='red', width=2),
                   hovertemplate='train<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_train])

trace_valid = go.Scatter(x=fpr_valid, y=tpr_valid, mode='lines', name='AUC_valid = %0.2f' % auc_valid,
                   line=dict(color='green', width=2),
                   hovertemplate='valid<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_valid])

trace_test = go.Scatter(x=fpr_test, y=tpr_test, mode='lines', name='AUC_test = %0.2f' % auc_test,
                   line=dict(color='darkorange', width=2),
                   hovertemplate='test<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_test])


reference_line = go.Scatter(x=[0,1], y=[0,1], mode='lines', name='Reference Line',
                            line=dict(color='navy', width=2, dash='dash'))

fig_roc = go.Figure(data=[trace_train,trace_valid,trace_test,reference_line])

fig_roc.update_layout(
    title ={
        'text':'ROC-кривая', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 1350,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate')

fig_roc.update_xaxes(showspikes=True)
fig_roc.update_yaxes(showspikes=True)


fig_roc.show()

<span style="color:Blue">

Вывод по ROC-кривой:
1. Модель выдает сталибильное качество на валидационном и тестовом наборах, а значит  
не переобучена - разброс ($variance$) не большой. Хотя, качество модели на обучающем наборе,  
заметно лучше.
2. Смещение ($bias$) модели такое же как и у модели $Logicstic$ $Regression$ $lassifier$, обученной   
на тех же данных.  
На тестовом наборе полученно качество $ROC AUC = 0.62$.

#### <span style="color:MediumBlue">Построение модели Gradient Boosting Classifier</span>

##### <span style="color:MediumSlateBlue">Baseline обучение модели Hist Gradient Boosting Classifier</span>

In [ ]:
# обучаем модель Gradient Boosting Classifier
gradient_boosting = HistGradientBoostingClassifier(random_state=42)
gradient_boosting.fit(np.array(X_train_balanced)[:,list_n_last_features],np.array(y_train_balanced))

# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_rf__pred_proba = gradient_boosting.predict_proba(X_train[:,list_n_last_features])[:,1]
y_valid_rf_pred_proba = gradient_boosting.predict_proba(X_valid[:,list_n_last_features])[:,1]

# Делаем предсказание для валидационной выборки
y_valid_rf_pred = gradient_boosting.predict(X_valid[:,list_n_last_features])

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_rf__pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_rf_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_valid,y_valid_rf_pred))

# time: 

##### <span style="color:MediumSlateBlue">Подбор гиперпараметров модели</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'late'
count_features = dict_spaces[feature_space]

# настроим оптимизацию гипер параметров
def optuna_hgbc(trial):
  # задаем пространства поиска гиперпараметров
  learning_rate = trial.suggest_float('learning_rate',0.2,0.6)
  max_iter = trial.suggest_int('max_iter', 50, 300,step = 1)
  max_leaf_nodes = trial.suggest_int('max_leaf_nodes', 2, 50,step = 1)
  max_depth = trial.suggest_int('max_depth', 1, 15,step = 1)
  min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 50,step = 1)
  max_features = trial.suggest_float('max_features',0.1,1)
  l2_regularization = trial.suggest_float('l2_regularization',0.01,1)
  class_0_weight = trial.suggest_float('class_0_weight',0.01,1)
  class_1_weight = trial.suggest_float('class_1_weight',0.01,1)
  n_last = trial.suggest_int('n_last', 5, 25,step = 1)
  class_1_percent = trial.suggest_float('class_1_percent',0.01,1)
  random_state = trial.suggest_int('random_state', 1, 1000000)


  # создаем модель
  optuna_gradient_boosting = HistGradientBoostingClassifier(
      learning_rate= learning_rate,
      max_iter = max_iter,
      max_leaf_nodes =max_leaf_nodes,
      max_depth = max_depth,
      min_samples_leaf = min_samples_leaf,
      max_features=max_features,
      l2_regularization = l2_regularization,
      class_weight={0:class_0_weight,1:class_1_weight},
      random_state=42)

  # с помощью функции features_from_transform_data_torow извлечем  
  # из данных transform_data_torow
  # признаки сооствествующие n_last последним клиенским операциям
  list_n_last_features = features_from_transform_data_torow(n_last,count_features,25)

  # загружаем обучующие наборы
  X_train_pd = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas()
  y_train_pd = pd.read_csv('target/target_train.csv')

  # поготовим данные y_train_pd к работе с функцией class_1_percent_samples
  y_train_pd.set_index('id',drop=False,inplace=True)

  # подготовим данные для обучения модели
  # с помощью функции class_1_percent_samples зададим долю класса 1
  list_c1_percent_id = class_1_percent_samples(y_train_pd,class_1_percent,random_state=random_state)[:1000000]

  # сформируем данные для обучения модели
  X_train_balanced = np.array(X_train_pd.loc[list_c1_percent_id])[:,list_n_last_features]
  y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

  # освободим память от "тяжелых" и ненужных файлов
  del X_train_pd, y_train_pd
  gc.collect()

  # я не воспользовался параметром timeout метода optimize потому, что  
  # optimize отставливает поиск параметров, если время обучения 
  # превышает timeout. Мне нужно чтобы поиск параметров продолжился.
  try:
    # для модуля func_time_out упакуем обучение модели в функцию try_func
    def try_func():
            return optuna_gradient_boosting.fit(X_train_balanced,y_train_balanced)
    # обучаем модель с ограничением по времени 10 минут (600 секунд)
    optuna_gradient_boosting = func_timeout.func_timeout(600, try_func)

    # удаляем крупные файлы чтобы высвободить память 
    del X_train_balanced, y_train_balanced
    gc.collect()

    # подгружаем данные для тестирования
    X_train = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas().to_numpy()[:,list_n_last_features]
    y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
    X_valid = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_valid').to_pandas().to_numpy()[:,list_n_last_features]
    y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()

    # делаем предсказание на обучающем и валидационном наборе и считаем метрики
    roc_train = metrics.roc_auc_score(y_train, optuna_gradient_boosting.predict_proba(X_train)[:,1])
    roc_valid = metrics.roc_auc_score(y_valid, optuna_gradient_boosting.predict_proba(X_valid)[:,1])
    f1_score = metrics.f1_score(y_valid, optuna_gradient_boosting.predict(X_valid))
    recall_1 = metrics.recall_score(y_valid, optuna_gradient_boosting.predict(X_valid),average=None,zero_division=0)[1]
    precision_1 = metrics.precision_score(y_valid, optuna_gradient_boosting.predict(X_valid),average=None,zero_division=0)[1]

    # удаляем крупные файлы чтобы высвободить память 
    del X_train, X_valid, y_train, y_valid 
    gc.collect()

  except func_timeout.FunctionTimedOut:
    # освободим память от "тяжелых" и ненужных файлов
    del X_train_balanced, y_train_balanced
    gc.collect()

    # обьявим метрики пустыми значениями
    roc_train = 0
    roc_valid = 0
    f1_score = 0
    recall_1 = 0
    precision_1 = 0
    pass

  return roc_train, roc_valid, f1_score, recall_1, precision_1

In [ ]:
# cоздаем объект исследования
# чтобы модель не переобучалась минимизируем roc_train 
# и максимизируем roc_valid
optuna_study_hsbs = optuna.create_study(study_name='HistGradientBoostingClassifier_'+feature_space+'_torow', 
                               directions=['maximize','maximize','maximize','maximize','maximize'], 
                               sampler=optuna.samplers.TPESampler(),
                               pruner='Hyperband',
                               storage='sqlite:///optuna_studies.db',
                               load_if_exists=True)

# ну случай удаления обучения
# optuna.delete_study(study_name='HistGradientBoostingClassifier_'+feature_space+'_torow', storage='sqlite:///optuna_studies.db')

In [ ]:
# ищем лучшую комбинацию гиперпараметров n_trials раз
optuna_study_hsbs.optimize(optuna_hgbc, n_trials=300)

In [ ]:
# из полученного результат соврмируем Data Frame
optuna_study_hsbs_pd = optuna_study_hsbs.trials_dataframe()
# переименуем столбы
optuna_study_hsbs_pd.rename(columns={
    'values_0': 'roc_train',
    'values_1': 'roc_valid',
    'values_2': 'f1_score',
    'values_3': 'recall_1',
    'values_4': 'precision_1'
},inplace=True)
# Выделим время потраченное на обучение модели
last_trail = optuna_study_hsbs_pd['number'].max()
optuna_study_hsbs_pd.tail(5)


In [ ]:
# покаждем статистику обучения
print('Максимальное значение метрики ROC AUC на валидационном наборе:',round(optuna_study_hsbs_pd['roc_valid'].max(),3))
print('Среднее значение метрики ROC AUC на валидационном наборе:',round(optuna_study_hsbs_pd['roc_valid'].mean(),3))
print('Максимальное значение метрики f1_score на валидационном наборе:',round(optuna_study_hsbs_pd['f1_score'].max(),3))
print('Среднее значение метрики f1_score на валидационном наборе:',round(optuna_study_hsbs_pd['f1_score'].mean(),3))
print('Максимальное значение метрики recall_1 на валидационном наборе:',round(optuna_study_hsbs_pd['recall_1'].max(),3))
print('Среднее значение метрики recall_1 на валидационном наборе:',round(optuna_study_hsbs_pd['recall_1'].mean(),3))
print('Максимальное значение метрики precision_1 на валидационном наборе:',round(optuna_study_hsbs_pd['precision_1'].max(),3))
print('Среднее значение метрики precision_1 на валидационном наборе:',round(optuna_study_hsbs_pd['precision_1'].mean(),3))

In [ ]:
# построим график важности гиперпарметров
optuna.visualization.plot_param_importances(optuna_study_hsbs, target_name='Score')

In [ ]:
# Построим зависимость метрики precision_1 от гиперпараметров
# max_iter

fig = px.scatter(
    data_frame=optuna_study_hsbs_pd,
    x='precision_1', #ось абсцисс
    y=['f1_score', 'recall_1'
    ], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость  метрик качества модели от params max iter', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина params max iter',
    yaxis_title='Величина метрики')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики precision_1 от гиперпараметров
# max_iter

fig = px.scatter(
    data_frame=optuna_study_hsbs_pd,
    x='precision_1', #ось абсцисс
    y=['params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight', 'params_l2_regularization',
       'params_learning_rate', 'params_max_depth', 'params_max_features',
       'params_max_iter', 'params_max_leaf_nodes', 'params_min_samples_leaf',
       'params_n_last'
    ], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость  метрик качества модели от params max iter', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина params max iter',
    yaxis_title='Величина метрики')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики precision_1 от гиперпараметров
fig = px.scatter(
    data_frame=optuna_study_hsbs_pd,
    x='f1_score', #ось абсцисс
    y=['params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight', 'params_l2_regularization',
       'params_learning_rate', 'params_max_depth', 'params_max_features',
       'params_max_iter', 'params_max_leaf_nodes', 'params_min_samples_leaf',
       'params_n_last'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision класса 1 от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision класса 1',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики precision_1 от гиперпараметров
fig = px.scatter(
    data_frame=optuna_study_hsbs_pd,
    x='roc_valid', #ось абсцисс
    y=['params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight', 'params_l2_regularization',
       'params_learning_rate', 'params_max_depth', 'params_max_features',
       'params_max_iter', 'params_max_leaf_nodes', 'params_min_samples_leaf',
       'params_n_last'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision класса 1 от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision класса 1',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрик качества модели от number

fig = px.scatter(
    data_frame=optuna_study_hsbs_pd[optuna_study_hsbs_pd['roc_valid']>0.997*(optuna_study_hsbs_pd['roc_valid'].max())],
    x='number', #ось абсцисс
    y=['roc_train','roc_valid','recall_1','precision_1'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision_1',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

<span style="color:Blue">

Вывод:  

Их всех выбираем точку $number =241$.   
В этой точке оптимальные значения метрик $recall_1$ и $precision_1$, а  
также относительно высокая метрика $ROC AUC$.   
Посмотрим каким значениям гиперпараметров соотвествует выбранная точка.

In [ ]:
# определим номер лучшге варианта
best_optuna_number = 241

# сформируем словарь лучших гипирпарметров RandomForestClassifier
best_param_rf = {
    'learning_rate' : optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['params_learning_rate'].iloc[0],
    'max_iter' : optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['params_max_iter'].iloc[0],
    'max_leaf_nodes' : optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['params_max_leaf_nodes'].iloc[0],
    'max_depth' : optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['params_max_depth'].iloc[0],
    'min_samples_leaf' : optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['params_min_samples_leaf'].iloc[0],
    'max_features' : optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['params_max_features'].iloc[0],
    'l2_regularization' : optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['params_l2_regularization'].iloc[0],
    'class_weight' : {0: optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['params_class_0_weight'].iloc[0],
                      1: optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['params_class_1_weight'].iloc[0],}
    }

# определим перменные для лучших значений параметров
best_n_last = optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['params_n_last'].iloc[0]
best_class_1_percent = optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['params_class_1_percent'].iloc[0]
best_random_state = optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['params_random_state'].iloc[0]


# Выведем принятые наилучшие праметры
print('best learning rate:',round(best_param_rf['learning_rate'],3))
print('best max iter:',best_param_rf['max_iter'])
print('best max leaf nodes:',best_param_rf['max_leaf_nodes'])
print('best max depth:',best_param_rf['max_depth'])
print('best min samples leaf:',best_param_rf['min_samples_leaf'])
print('best max features:',round(best_param_rf['max_features'],3))
print('best l2 regularization:',round(best_param_rf['l2_regularization'],3))
print('best class 0 weight:',round(best_param_rf['class_weight'][0],3))
print('best class 1 weight:',round(best_param_rf['class_weight'][1],3))

print('best class 1 percent:',round(best_class_1_percent,3))
print('best n last:',best_n_last)
print('best random state:',best_random_state)
print('time for best train:',round(optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['duration'].iloc[0].seconds/60),'minutes')

print()
print('ROC AUC на обучающем наборе:', round(optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['roc_train'].iloc[0],3))
print('ROC AUC на валидационном наборе:', round(optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['roc_valid'].iloc[0],3))
print('precision класса 1:', round(optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['precision_1'].iloc[0],3))
print('recall класса 1:', round(optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['recall_1'].iloc[0],3))

##### <span style="color:MediumSlateBlue">Обучение модели с лучшими параметрами</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'late'
count_features = dict_spaces[feature_space]

# с помощью функции features_from_transform_data_torow извлечем  
# из данных transform_data_torow
# признаки сооствествующие n_last последним клиенским операциям
list_n_last_features = features_from_transform_data_torow(best_n_last,count_features,25)

# загружаем обучующие наборы
X_train_pd = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas()
y_train_pd = pd.read_csv('target/target_train.csv')

# поготовим данные y_train_pd к работе с функцией class_1_percent_samples
y_train_pd.set_index('id',drop=False,inplace=True)

# подготовим данные для обучения модели
# с помощью функции class_1_percent_samples зададим долю класса 1
list_c1_percent_id = class_1_percent_samples(y_train_pd,best_class_1_percent,random_state=best_random_state)[:1000000]

# сформируем данные для обучения модели
X_train_balanced = np.array(X_train_pd.loc[list_c1_percent_id])[:,list_n_last_features]
y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

# освободим память от "тяжелых" и ненужных файлов
del X_train_pd, y_train_pd
gc.collect()


# обучаем модель HistGradientBoostingClassifier с наилучшеми параметрами
gradient_boosting = HistGradientBoostingClassifier(
        **best_param_rf,
        random_state=42)
gradient_boosting.fit(X_train_balanced,y_train_balanced)

# освободим память от "тяжелых" и ненужных файлов
del X_train_balanced, y_train_balanced
gc.collect()

# подгружаем данные для тестирования
X_train = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas().to_numpy()[:,list_n_last_features]
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
X_valid = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_valid').to_pandas().to_numpy()[:,list_n_last_features]
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()

# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_gb__pred_proba = gradient_boosting.predict_proba(X_train)[:,1]
y_valid_gb_pred_proba = gradient_boosting.predict_proba(X_valid)[:,1]

# Делаем предсказание для валидационной выборки
y_valid_gb_pred = gradient_boosting.predict(X_valid)

# освободим память от "тяжелых" и ненужных файлов
del X_train, X_valid
gc.collect()

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_gb__pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_gb_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_valid,y_valid_gb_pred,zero_division=0))

# time: 2m 30s

##### <span style="color:MediumSlateBlue">Анализ влияния порога классификации на качество модели</span>

In [ ]:
# чтобы проше обращаться к каждому обьекту в данных
# преобразуем y_valid_LR_pred к Series типу
y_valid_gb_pred_proba = pd.Series(y_valid_gb_pred_proba)

# формируем список из необходимых значений thresholds
thresholds = np.arange(0.01, 1, 0.01)

# сформируем списко для заполнения данными результатов анализа
threshold_data = []


for threshold in thresholds:
        # сформируем список для заполнения данными текушего threshold
        threshold_current_data = [threshold]

        #клиентов, для которых вероятность дефолта > threshold, относим к классу 1
        #в противном случае — к классу 0
        y_valid_gb_pred = y_valid_gb_pred_proba.apply(lambda x: 1 if x>threshold else 0)


        #Считаем метрики для класса 0 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(y_valid, y_valid_gb_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.precision_score(y_valid, y_valid_gb_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.f1_score(y_valid, y_valid_gb_pred,average=None,zero_division=0)[0],3))

        #Считаем метрики для класса 1 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(y_valid, y_valid_gb_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.precision_score(y_valid, y_valid_gb_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.f1_score(y_valid, y_valid_gb_pred,average=None,zero_division=0)[1],3))

        # добавляем полученные данные в список threshold_data
        threshold_data.append(threshold_current_data)

        # из полученных данных сформируем dataframe
        thresholds_pd =pd.DataFrame(
        columns=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'],
        data = threshold_data)

# time: 17s

In [ ]:
# Визуализируем метрики на валидационном наборе при различных threshold
fig_threshold = px.line(
    data_frame=thresholds_pd,
    x='threshold', #ось абсцисс
    y=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'], #ось ординат
)
fig_threshold.update_layout(
    title ={
        'text':'Зависимость качества модели от threshold', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Порог классификации threshold',
    yaxis_title='Величина метрики')
fig_threshold.update_xaxes(showspikes=True)
fig_threshold.update_yaxes(showspikes=True)

fig_threshold.show()

Метрика $precision$ показывает на сколько сильно ошиблась модель при определиние класса.  
Чем выше метрика, тем меньше ошиблась модель. 

$$precision = \frac{TP}{TP+FP}$$

Метрика $recall$ показывает способность модели найти класс.  
Чем выше метрика, тем выше способность модели находить класс.  

$$recall = \frac{TP}{TP+FN}$$

Метрика $f1-score$ показывает баланс между $precision$ и $recall$ 

$threshold$- порог класификации модели.  
Все объекты для которых $у_{\text{proba pred}} < threshold$, модель относит к класс 0.  
Соотвественно, если $у_{\text{proba pred}}  > threshold$ модель относит обьект к классу 1.  
Для идельаной модели $threshold$, является границей между областями класса 0 и класса 1.  

<span style="color:Blue">

Выводы по результам анализа:  

Для улучшения качества модели по метрике $ROC AUC$, нет смысла сдвигать порог   
классификации. Но зависимость метрик качества модели от $threshold$, может рассказать о  
способности модели искать класс 1 и класс 0.

При фокусировке модели на классе 1 (более низкий $threshold$):  
- на классе 1 метрика $precision$ уменьшается, метрика $recall$ растет;  
- на классе 0 метрика $precision$ растет, метрика $recall$ уменьшается.  

При фокусировке модели на классе 0 (более высокий $threshold$):  
- на классе 1 метрика $precision$ увеличивается, метрика $recall$ уменьшается;  
- на классе 0 метрика $precision$ уменьшается, метрика $recall$ растет.  

Обьсняется тем, что в условия дисбаланса модель научилась хорошо определять класс 0 и плохо класс 1.  

При фокусировке модели на классе 1 (более низкий $threshold$), все больше обьектов отнесены к классу 1.   
Так как $precision$ класса 1 уменьшается, значит в области класса 1 стало больше обектов класса 0,  
что явлется закономерным для "идеальной" модели.  
Однако при этом, $recall$ класса 1 растет, значит в добавленных обьектах также присутвует класс 1.  
Это как раз указывает на то, что модель не нучилась находить среди класса 0 класс 1. В области с     
высокой вероятностью класса 0 (более низкий $threshold$), находятся обьекты класса 1.  

Фокусировка на классе 0 (более высокий $threshold$), заставляет модель все больше обьектов из области класса 1  
относить к классу 0. При этом увеличение метрики $precision$ класса 1 говорит о том, что в области класса 1     
становится меньше 0, а значит модель умеет среди класса 1, находить класс 0.  
После  $threshold=0.83$ модель впринципе теряет способность отличать класс 1. У модели нет обьектов класса 1   
в которых она "уверена" на 90+%.

Обратим внимание на $threshold=0.68$, в этой точке находится баланс межу метриками качества модели:
$$precision_1 = recall_1 = \text{f1-score}_1 = 0.102$$
$$precision_0 = recall_0 = \text{f1-score}_0 = 0.968$$

Для модели с лучшим качеством эти две точки должны находится максимально близко к друг другу и иметь   
высокое значение метрик $precision_1$ и $recall_1$.  
Для класса 0 это условие выполнется, что также говорит о том, что модель научилась искать этот класс.    
Для класса 1 это условие не выполнется, что также говорит о том, что модель плохо ищет класс 1.

##### <span style="color:MediumSlateBlue">Построение ROC-кривой выбранной модели на тестовом наборе</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'late'
count_features = dict_spaces[feature_space]

In [ ]:
# подгружаем данные для тестирования
X_train = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas().to_numpy()[:,list_n_last_features]
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
X_valid = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_valid').to_pandas().to_numpy()[:,list_n_last_features]
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()
X_test = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_test').to_pandas().to_numpy()[:,list_n_last_features]
y_test = pd.read_csv('target/target_test.csv')['flag'].to_numpy()

In [ ]:
print("Размеры обучающего набора",X_train.shape,y_train.shape)
print("Размеры валидационного набора",X_valid.shape,y_valid.shape)
print("Размеры тестового набора",X_test.shape,y_test.shape)
# time: 1m 43s

In [ ]:
# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_gb_pred_proba = gradient_boosting.predict_proba(X_train)[:,1]
y_valid_gb_pred_proba = gradient_boosting.predict_proba(X_valid)[:,1]
y_test_gb_pred_proba = gradient_boosting.predict_proba(X_test)[:,1]

# Делаем предсказание для валидационной выборки
y_test_gb_pred = gradient_boosting.predict(X_test)

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_gb_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_gb_pred_proba),3))
print('ROC AUC на тестовом наборе', round(metrics.roc_auc_score(y_test, y_test_gb_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_test,y_test_gb_pred,zero_division=0))

In [ ]:
# Для расчета значений TPR и FPR используем функцию roc_curve
fpr_train, tpr_train, _ = metrics.roc_curve(y_train, y_train_gb_pred_proba)
fpr_valid, tpr_valid, _ = metrics.roc_curve(y_valid, y_valid_gb_pred_proba)
fpr_test, tpr_test, _ = metrics.roc_curve(y_test, y_test_gb_pred_proba)

# рассчитаем площадь под кривой
auc_train = round(metrics.roc_auc_score(y_train, y_train_gb_pred_proba),3)
auc_valid = round(metrics.roc_auc_score(y_valid, y_valid_gb_pred_proba),3)
auc_test = round(metrics.roc_auc_score(y_test, y_test_gb_pred_proba),3)

trace_train = go.Scatter(x=fpr_train, y=tpr_train, mode='lines', name='AUC_train = %0.2f' % auc_train,
                   line=dict(color='red', width=2),
                   hovertemplate='train<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_train])

trace_valid = go.Scatter(x=fpr_valid, y=tpr_valid, mode='lines', name='AUC_valid = %0.2f' % auc_valid,
                   line=dict(color='green', width=2),
                   hovertemplate='valid<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_valid])

trace_test = go.Scatter(x=fpr_test, y=tpr_test, mode='lines', name='AUC_test = %0.2f' % auc_test,
                   line=dict(color='darkorange', width=2),
                   hovertemplate='test<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_test])


reference_line = go.Scatter(x=[0,1], y=[0,1], mode='lines', name='Reference Line',
                            line=dict(color='navy', width=2, dash='dash'))

fig_roc = go.Figure(data=[trace_train,trace_valid,trace_test,reference_line])

fig_roc.update_layout(
    title ={
        'text':'ROC-кривая', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 1350,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate')

fig_roc.update_xaxes(showspikes=True)
fig_roc.update_yaxes(showspikes=True)


fig_roc.show()

<span style="color:Blue">

Вывод по ROC-кривой:
1. Модель выдает сталибильное качество на валидационном и тестовом наборах, а значит  
не переобучена - разброс ($variance$) не большой.
2. Смещение ($bias$) модели такое же как и у предыдущих моделей,    
обученной на тех же данных.  
На тестовом наборе полученно качество $ROC AUC = 0.62$.

### <span style="color:RoyalBlue">Построение модели на сбалансированных данных подпространства credit torow</span>

#### <span style="color:MediumBlue">Формирование данных для обучения</span>

Анализ исходных данных, в частности target, показал что представленные данные   
имееют перекос: 96% клиентов у которых не случился дефолт (flag = 0),    
против 4 % - для которых произошел дефолт (flag = 1).  
С помощью функции class_1_percent_samples, сформируем сбаланисрованную выборку  
для обучения моделей.
За основу сбалансировных данных возьмем данные клиентов с дефолтом  
и к ним будем добирать необходимое количество клиентов до сбалансированности. 


Функция class_1_percent_samples принемает две переменные:
- target_data - массив с id и целевой переменной
- class_1_percent - процент класса 1 в результирующем массиве

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'credit'
count_features = dict_spaces[feature_space]

In [ ]:
# сформируем данные для анализа сбалансированности обучающих данных
X_train_pd = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas()
y_train_pd = pd.read_csv('target/target_train.csv')

In [ ]:
# поготовим данные y_train_pd к работе с функцией class_1_percent_samples
y_train_pd.set_index('id',drop=False,inplace=True)
y_train_pd.head(4)

In [ ]:
# взглянем на сблансиорованность данных в y_train
y_train_pd['flag'].value_counts(normalize=True)

In [ ]:
# сбалансируем данные в y_train_pd
list_c1_percent_id = class_1_percent_samples(y_train_pd,0.5)
# list_c1_percent_features - список 'id' из y_train_pd соотвествующих  
# заданному проценту класса 1
len(list_c1_percent_id)

In [ ]:
# проверим сбалансированность полученного y_train
y_train_pd.loc[list_c1_percent_id]['flag'].value_counts(normalize=True)

*transform_data_torow*, показываеются последние N операций клиента.  
В данных в *date_torow*, собраны последние 25 операций клиентов.

Предварительный анализ показывает, что медиальное значение количества клиентских  
операций равно 7. Поэтому, для начала, будем показывать моделе $n_{last} = 7$   
последних клиентских операций.

Для извлечения из *transform_data_torow_25* необходимого количества последних клиенских  
операций ($n_{last}$) используем функцию features_from_transform_data_torow.

Функция features_from_transform_data_torow примает следующие аргументы:
- $n_{last} = 7$ - необходимое количество последних клиентских операций; 
- $n_{groups} = 8$ - количество признаков в $date$ $features$;
- $N_{last} = 25$ - количество $n_{last}$ операций, показанных в transform_data_torow_25


In [ ]:
# с помощью функции features_from_transform_data_torow извлечем  
# из данных transform_data_torow_25
# признаки сооствествующие 7 последним клиенским операциям
list_n_last_features = features_from_transform_data_torow(7,count_features,25)

In [ ]:
# подготовим данные для обучения
X_train_balanced = X_train_pd.loc[list_c1_percent_id]
y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag']
# сформируем данные для проверики модели
X_train = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas().to_numpy()
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
X_valid = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_valid').to_pandas().to_numpy()
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()

In [ ]:
# посмотрим на структуру данных в X_train_balanced
X_train_balanced.head(2)

In [ ]:
# посмотрим на структуру данных в y_train_balanced
y_train_balanced.head(2)

In [ ]:
# проверим что выборки действительно совпадают по id
X_train_balanced.index.to_list() == y_train_balanced.index.to_list()

#### <span style="color:MediumBlue">Построение модели Logistic Regression</span>

##### <span style="color:MediumSlateBlue">Baseline обучение модели LogisticRegression</span>

In [ ]:
# обучаем модель логистической регрессии
logistic_regression = linear_model.LogisticRegression(random_state=42, max_iter=10000)
logistic_regression.fit(np.array(X_train_balanced)[:,list_n_last_features],np.array(y_train_balanced))

# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_LR_pred_proba = logistic_regression.predict_proba(X_train[:,list_n_last_features])[:,1]
y_valid_LR_pred_proba = logistic_regression.predict_proba(X_valid[:,list_n_last_features])[:,1]

# Делаем предсказание для валидационной выборки
y_valid_LR_pred = logistic_regression.predict(X_valid[:,list_n_last_features])

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_LR_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_LR_pred_proba),3))

print('Основные метрики на тестовом наборе:')
print(metrics.classification_report(y_valid,y_valid_LR_pred))

# time: 1m 12s

##### <span style="color:MediumSlateBlue">Анализ влияния scaler преобразования на качество и скорость схождения модели</span>

In [ ]:
# формируем список из необходимых scaler
scalers = [MinMaxScaler(),RobustScaler() ,StandardScaler()]

# сформируем списко для заполнения данными результатов анализа
scalers_data = []

# установим ограничение на время обучения модели в 600 секунд (10 минут)
time_out = 600

for scaler in scalers:
        # для того чтобы следить за выполнением цикла введем индикатор
        print('current scaler: ', scaler)

         # сформируем список для заполнения данными текушего scaler
        current_scaler_data = [scaler]
        
        # выполним scaler преобразование
        X_train_s_balanced = scaler.fit_transform(np.array(X_train_balanced)[:,list_n_last_features])
        X_train_s = scaler.transform(X_train[:,list_n_last_features])
        X_valid_s = scaler.transform(X_valid[:,list_n_last_features])

        try:
                # для модуля func_time_out упакуем обучение модели в функцию try_func
                def try_func():
                        logistic_regression = linear_model.LogisticRegression(random_state=42, max_iter=10000)
                        return logistic_regression.fit(X_train_s_balanced,y_train_balanced)
                    
                # фиксируем время начала
                start_time = time.time()
                
                # запускаем обучение модели с ограничением по времени
                logistic_regression = func_timeout.func_timeout(time_out, try_func)
                
                # фиксируем время окончания обучения
                end_time = time.time()
                
                # запишем время обучения модели
                current_scaler_data.append(round(end_time-start_time))

                # Делаем предсказание для обучающей и валидационной выборки
                y_valid_lr_pred = logistic_regression.predict(X_valid_s)
                
                # для метрик ROC AUC делаем предсказание модели для обучающей и валидационной выборки  
                # в виде вероятности принадлежности к классу 1 
                y_train_lr_pred_proba = logistic_regression.predict_proba(X_train_s)[:,1]
                y_valid_lr_pred_proba = logistic_regression.predict_proba(X_valid_s)[:,1]

                #Считаем метрики ROC AUC yна обуающем и валидационном наборе и добавляем их в спискок
                current_scaler_data.append(round(metrics.roc_auc_score(y_train, y_train_lr_pred_proba),3))
                current_scaler_data.append(round(metrics.roc_auc_score(y_valid, y_valid_lr_pred_proba),3))

                #Считаем метрики для класса 0 на валидационном наборе и добавляем их в список
                current_scaler_data.append(round(metrics.recall_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))
                current_scaler_data.append(round(metrics.precision_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))


                #Считаем метрики для класса 1 на валидационном набопе и добавляем их в список
                current_scaler_data.append(round(metrics.recall_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))
                current_scaler_data.append(round(metrics.precision_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))

                # добавляем полученные данные в список scalers_data
                scalers_data.append(current_scaler_data)

                # чтобы подстраховаться на случай ошибки : "The Kernel crashed while executing code"
                # сохраним в локальную папку удачную итерацию
                # для этого из полученных данных сформируем dataframe
                # из полученных данных сформируем dataframe
                scaler_pd =pd.DataFrame(
                        columns=['scaler','time_fit',
                                 'roc_train','roc_valid', 
                                 'recall_0','precision_0',
                                 'recall_1','precision_1'],
                        data = scalers_data)
                # сохраняем dataframe
                scaler_pd.to_csv('data_recovery/scaler_pd.csv',index=False)
               
                # очистим output от прошедшей итерации
                clear_output()

        except func_timeout.FunctionTimedOut:
                # если время обучения привышает лимит
                # запишем в current_scaler_data соотвествующую данные
                current_scaler_data = [scaler,'>30m',0,0,0,0,0,0]
                # добавляем полученные данные в список scalers_data
                scalers_data.append(current_scaler_data)

                # чтобы подстраховаться на случай ошибки : "The Kernel crashed while executing code"
                # сохраним в локальную папку удачную итерацию
                # для этого из полученных данных сформируем dataframe
                # из полученных данных сформируем dataframe
                scaler_pd =pd.DataFrame(
                        columns=['scaler','time_fit',
                                 'roc_train','roc_valid', 
                                 'recall_0','precision_0',
                                 'recall_1','precision_1'],
                        data = scalers_data)
                # сохраняем dataframe
                scaler_pd.to_csv('data_recovery/scaler_pd.csv',index=False)
                
                # очистим output от прошедшей итерации
                clear_output()
                # переходим к следующему scaler
                pass
# очистим output
clear_output()

# сформируем данные соотвествующие лучщему ROC AUC на валидацинном наборе
best_roc_valid_data = scaler_pd[scaler_pd['roc_valid']==scaler_pd['roc_valid'].max()]

# выведем лучший результат на валидационном наборе
print('result:')
print('best scaler on valid:',best_roc_valid_data.iloc[0]['scaler'])
print('ROC AUC on train:', best_roc_valid_data.iloc[0]['roc_train'])
print('ROC AUC on valid:', best_roc_valid_data.iloc[0]['roc_valid'])
print('Time fit for best scaler:', best_roc_valid_data.iloc[0]['time_fit'],' seconds')

# выведем полученный dataframe
scaler_pd

# time: 

##### <span style="color:MediumSlateBlue"> Анализ влияния solver оптимизатора на качество и скорость обучения модели</span>

Проверим качество и скорость сходимости модели при разных solver  
Проверку осуществим через цикл  
Чтобы модель не сходилась бесконечно с помощью модуля func_timeout  
поставим ограничение времени схождения в 10 минут

In [ ]:
# подготовим данные для обучения модели

# обьявим scaler
scaler = StandardScaler()
# выполним scaler преобразование
X_train_s_balanced = scaler.fit_transform(np.array(X_train_balanced)[:,list_n_last_features])
X_train_s = scaler.transform(X_train[:,list_n_last_features])
X_valid_s = scaler.transform(X_valid[:,list_n_last_features])

# time: 10s

In [ ]:
# Зададим список solvers
solvers_list = ['lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga']

# сформируем список для заполнения
solvers_data = []

# установим ограничение на время обучения модели в 300 секунд (5 минут)
time_out = 300

for solver in solvers_list:
        # для того чтобы следить за выполнением цикла введем индикатор
        print('current solver: ', solver)

         # сформируем список для заполнения данными текушего solver
        current_solver_data = [solver]
        
        try:
                # для модуля func_time_out упакуем обучение модели в функцию try_func
                def try_func():
                        logistic_regression = linear_model.LogisticRegression(solver= solver,
                                                                  random_state=42, 
                                                                  max_iter=10000)
                        return logistic_regression.fit(X_train_s_balanced,y_train_balanced)
                    
                # фиксируем время начала
                start_time = time.time()
                
                # запускаем обучение модели с ограничением по времени
                logistic_regression = func_timeout.func_timeout(time_out, try_func)
                
                # фиксируем время окончания обучения
                end_time = time.time()
                
                # запишем время обучения модели
                current_solver_data.append(round(end_time-start_time))

                # Делаем предсказание для обучающей и валидационной выборки
                y_valid_lr_pred = logistic_regression.predict(X_valid_s)
                
                # для метрик ROC AUC делаем предсказание модели для обучающей и валидационной выборки  
                # в виде вероятности принадлежности к классу 1 
                y_train_lr_pred_proba = logistic_regression.predict_proba(X_train_s)[:,1]
                y_valid_lr_pred_proba = logistic_regression.predict_proba(X_valid_s)[:,1]

                #Считаем метрики ROC AUC yна обуающем и валидационном наборе и добавляем их в спискок
                current_solver_data.append(round(metrics.roc_auc_score(y_train, y_train_lr_pred_proba),3))
                current_solver_data.append(round(metrics.roc_auc_score(y_valid, y_valid_lr_pred_proba),3))

                #Считаем метрики для класса 0 на валидационном наборе и добавляем их в список
                current_solver_data.append(round(metrics.recall_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))
                current_solver_data.append(round(metrics.precision_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))


                #Считаем метрики для класса 1 на валидационном набопе и добавляем их в список
                current_solver_data.append(round(metrics.recall_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))
                current_solver_data.append(round(metrics.precision_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))

                # запишем данные полученные на одной итерации
                solvers_data.append(current_solver_data)

                # чтобы подстраховаться на случай ошибки : "The Kernel crashed while executing code"
                # сохраним в локальную папку удачную итерацию
                # для этого из полученных данных сформируем dataframe
                # из полученных данных сформируем dataframe
                solvers_pd =pd.DataFrame(
                        columns=['solver','time_fit',
                                 'roc_train','roc_valid', 
                                 'recall_0','precision_0',
                                 'recall_1','precision_1'],
                        data = solvers_data)
                # сохраняем dataframe
                solvers_pd.to_csv('data_recovery/solvers_pd.csv',index=False)
               
                # очистим output от прошедшей итерации
                clear_output()

        except func_timeout.FunctionTimedOut:
                # если время обучения привышает лимит
                # запишем в current_solver_data соотвествующую данные
                current_solver_data = [solver,'>10m',0,0,0,0,0,0]
                # добавляем полученные данные в список solvers_data
                solvers_data.append(current_solver_data)

                # чтобы подстраховаться на случай ошибки : "The Kernel crashed while executing code"
                # сохраним в локальную папку удачную итерацию
                # для этого из полученных данных сформируем dataframe
                # из полученных данных сформируем dataframe
                solvers_pd =pd.DataFrame(
                        columns=['solver','time_fit',
                                 'roc_train','roc_valid', 
                                 'recall_0','precision_0',
                                 'recall_1','precision_1'],
                        data = solvers_data)
                # сохраняем dataframe
                solvers_pd.to_csv('data_recovery/solvers_pd.csv',index=False)
                
                # очистим output от прошедшей итерации
                clear_output()
                # переходим к следующему solver
                pass
# очистим output
clear_output()

# сформируем данные соотвествующие лучщему ROC AUC на валидацинном наборе
best_roc_valid_data = solvers_pd[solvers_pd['roc_valid']==solvers_pd['roc_valid'].max()]

# выведем лучший результат на валидационном наборе
print('result:')
print('best solver on valid:',best_roc_valid_data.iloc[0]['solver'])
print('ROC AUC on train:', best_roc_valid_data.iloc[0]['roc_train'])
print('ROC AUC on valid:', best_roc_valid_data.iloc[0]['roc_valid'])
print('Time fit for best solver:', best_roc_valid_data.iloc[0]['time_fit'],' seconds')

# выведем полученный dataframe
solvers_pd

# time: 11m 35s

##### <span style="color:MediumSlateBlue">Подбор гиперпараметров модели</span>

In [ ]:
# для выбора sceler преобразвания на понадобится словарь
dict_scalers ={
    'MinMaxScaler':MinMaxScaler(),
    'RobustScaler':RobustScaler(),
    'StandardScaler':StandardScaler(),
}
# определим пространство признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'credit'
count_features = dict_spaces[feature_space]

# настроим оптимизацию гипер параметров
def optuna_lg(trial):
  # задаем пространства поиска гиперпараметров
  C = trial.suggest_float('C',0.01,1)
  solver = trial.suggest_categorical('solver', ['lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky'])
  class_0_weight = trial.suggest_float('class_0_weight',0.01,1)
  class_1_weight = trial.suggest_float('class_1_weight',0.01,1)
  n_last = trial.suggest_int('n_last', 10, 25,step = 1)
  class_1_percent = trial.suggest_float('class_1_percent',0.01,1)
  scaler = trial.suggest_categorical('scaler', ['MinMaxScaler', 'RobustScaler','StandardScaler'])
  random_state = trial.suggest_int('random_state', 1, 1000000)

  # создаем модель
  optuna_log_reg = linear_model.LogisticRegression(
      solver=solver,
      C=C,
      class_weight={0:class_0_weight,1:class_1_weight},
      random_state=random_state,
      max_iter=10000)

  # с помощью функции features_from_transform_data_torow извлечем  
  # из данных transform_data_torow
  # признаки сооствествующие n_last последним клиенским операциям
  list_n_last_features = features_from_transform_data_torow(n_last,count_features,25)

  # обьявим scaler
  scaler = dict_scalers[scaler]

  # загружаем обучующие наборы
  X_train_pd = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas()
  y_train_pd = pd.read_csv('target/target_train.csv')

  # поготовим данные y_train_pd к работе с функцией class_1_percent_samples
  y_train_pd.set_index('id',drop=False,inplace=True)

  # подготовим данные для обучения модели
  # с помощью функции class_1_percent_samples зададим долю 
  # класса 1
  list_c1_percent_id = class_1_percent_samples(y_train_pd,class_1_percent,random_state=random_state)[:1000000]

  # сформируем данные для обучения базовой модели
  X_train_s_balanced = scaler.fit_transform(np.array(X_train_pd.loc[list_c1_percent_id])[:,list_n_last_features])
  y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

  # освободим память от "тяжелых" и ненужных файлов
  del X_train_pd, y_train_pd
  gc.collect()

  # я не воспользовался параметром timeout метода optimize потому, что  
  # optimize отставливает поиск параметров, если время обучения 
  # превышает timeout. Мне нужно чтобы поиск параметров продолжился.
  try:
    # для модуля func_time_out упакуем обучение модели в функцию try_func
    def try_func():
            return optuna_log_reg.fit(X_train_s_balanced, y_train_balanced)
    # обучаем модель с ограничением по времени 10 минут (600 секунд)
    optuna_log_reg = func_timeout.func_timeout(600, try_func)

    # удаляем крупные файлы чтобы высвободить память 
    del X_train_s_balanced,y_train_balanced
    gc.collect()

    # подгружаем данные для тестирования
    X_train = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas().to_numpy()[:,list_n_last_features]
    y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
    X_valid = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_valid').to_pandas().to_numpy()[:,list_n_last_features]
    y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()

    # сформируем данные для проверки модели
    X_train_s = scaler.transform(X_train)
    X_valid_s = scaler.transform(X_valid)

    # удаляем крупные файлы чтобы высвободить память 
    del X_train, X_valid
    gc.collect()

    # делаем предсказание на обучающем и валидационном наборе и считаем метрики
    roc_train = metrics.roc_auc_score(y_train, optuna_log_reg.predict_proba(X_train_s)[:,1])
    roc_valid = metrics.roc_auc_score(y_valid, optuna_log_reg.predict_proba(X_valid_s)[:,1])
    f1_score = metrics.f1_score(y_valid, optuna_log_reg.predict(X_valid_s))
    recall_1 = metrics.recall_score(y_valid, optuna_log_reg.predict(X_valid_s),average=None,zero_division=0)[1]
    precision_1 = metrics.precision_score(y_valid, optuna_log_reg.predict(X_valid_s),average=None,zero_division=0)[1]

    del X_train_s, X_valid_s, y_train, y_valid
    gc.collect()

  except func_timeout.FunctionTimedOut:
    # освободим память от "тяжелых" и ненужных файлов
    del X_train_s_balanced, y_train_balanced
    gc.collect()

    # обьявим метрики пустыми значениями
    roc_train = 0
    roc_valid = 0
    f1_score = 0
    recall_1 = 0
    precision_1 = 0
    pass

  return roc_train, roc_valid, f1_score, recall_1, precision_1

In [ ]:
# cоздаем объект исследования
# чтобы модель не переобучалась минимизируем roc_train 
# и максимизируем roc_valid
optuna_study_lg = optuna.create_study(study_name='LogisticRegression_'+feature_space+'_torow', 
                               directions=['maximize','maximize','maximize','maximize','maximize'], 
                               sampler=optuna.samplers.TPESampler(),
                               pruner='Hyperband',
                               storage='sqlite:///optuna_studies.db',
                               load_if_exists=True)
# на случай удаления 
# optuna.delete_study(study_name='LogisticRegression_'+feature_space+'_torow', storage='sqlite:///optuna_studies.db')

In [ ]:
# ищем лучшую комбинацию гиперпараметров n_trials раз
optuna_study_lg.optimize(optuna_lg, n_trials=300)

In [ ]:
# из полученного результат соврмируем Data Frame
optuna_study_lg_pd = optuna_study_lg.trials_dataframe().dropna()
# переименуем столбы
optuna_study_lg_pd.rename(columns={
    'values_0': 'roc_train',
    'values_1': 'roc_valid',
    'values_2': 'f1_score',
    'values_3': 'recall_1',
    'values_4': 'precision_1'
},inplace=True)
# Выделим время потраченное на обучение модели
optuna_study_lg_pd.head(5)

In [ ]:
# покаждем статистику обучения
print('Максимальное значение метрики ROC AUC на валидационном наборе:',round(optuna_study_lg_pd['roc_valid'].max(),3))
print('Среднее значение метрики ROC AUC на валидационном наборе:',round(optuna_study_lg_pd['roc_valid'].mean(),3))
print('Максимальное значение метрики f1_score на валидационном наборе:',round(optuna_study_lg_pd['f1_score'].max(),3))
print('Среднее значение метрики f1_score на валидационном наборе:',round(optuna_study_lg_pd['f1_score'].mean(),3))
print('Максимальное значение метрики recall_1 на валидационном наборе:',round(optuna_study_lg_pd['recall_1'].max(),3))
print('Среднее значение метрики recall_1 на валидационном наборе:',round(optuna_study_lg_pd['recall_1'].mean(),3))
print('Максимальное значение метрики precision_1 на валидационном наборе:',round(optuna_study_lg_pd['precision_1'].max(),3))
print('Среднее значение метрики precision_1 на валидационном наборе:',round(optuna_study_lg_pd['precision_1'].mean(),3))

In [ ]:
# построим график важности гиперпарметров
optuna.visualization.plot_param_importances(optuna_study_lg, target_name='Score')

In [ ]:
# Построим зависимость метрики precision_1 от гипер параметров

fig= px.scatter(
    data_frame=optuna_study_lg_pd,
    x='precision_1', #ось абсцисс
    y=['recall_1','f1_score'
    ], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision_1',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики precision_1 от гипер параметров

fig= px.scatter(
    data_frame=optuna_study_lg_pd,
    x='precision_1', #ось абсцисс
    y=['params_C','params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight', 'params_n_last',
    ], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision_1',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики precision_1 от гипер параметров

fig= px.scatter(
    data_frame=optuna_study_lg_pd,
    x='f1_score', #ось абсцисс
    y=['params_C','params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight', 'params_n_last',
    ], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision_1',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики precision_1 от гипер параметров

fig= px.scatter(
    data_frame=optuna_study_lg_pd,
    x='roc_valid', #ось абсцисс
    y=['params_C','params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight', 'params_n_last',
    ], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision_1',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Визуализируем метрики полученные при optuna оптимизации
fig = px.scatter(
    data_frame=optuna_study_lg_pd[
                                (optuna_study_lg_pd['roc_valid']>0.997*optuna_study_lg_pd['roc_valid'].max())],
    x='number', #ось абсцисс
    y=['roc_train','roc_valid','recall_1','precision_1'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость качества модели от trials optuna', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='trials optuna',
    yaxis_title='Величина метрики')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

<span style="color:Blue">

Вывод:  

Их всех выбираем точку $number =201$.   
В этой точке одно из самых больших значений $ROC AUC$.  
Посмотрим каким значениям гиперпараметров соотвествует выбранная точка.

In [ ]:
# определим номер лучшге варианта
best_optuna_number = 201

# сформируем словарь лучших гипирпарметров RandomForestClassifier
best_param_lr_torow = {
    'C' : round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_C'].iloc[0],3),
    'class_weight' : {0:round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_class_0_weight'].iloc[0],3),
                      1:round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_class_1_weight'].iloc[0],3)},
    }
# создадим перменные
best_n_last = int(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_n_last'].iloc[0])
best_scaler = optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_scaler'].iloc[0]
best_class_1_percent = optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_class_1_percent'].iloc[0]
best_random_state = int(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_random_state'].iloc[0])
# Выведем принятые наилучшие праметры
print('best C:',best_param_lr_torow['C'])
print('best class 0 weight:',best_param_lr_torow['class_weight'][0])
print('best class 1 weight:',best_param_lr_torow['class_weight'][1])

print('best class 1 percent:',round(best_class_1_percent,3))
print('best scaler:',best_scaler)
print('best n_last:',round(best_n_last,3))
print('best best random state:',best_random_state)
print('time for best train:',round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['duration'].iloc[0].seconds/60),'minutes')

print()
print('ROC AUC на обучающем наборе:', round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['roc_train'].iloc[0],3))
print('ROC AUC на валидационном наборе:', round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['roc_valid'].iloc[0],3))
print('precision класса 1:', round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['precision_1'].iloc[0],3))
print('recall класса 1:', round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['recall_1'].iloc[0],3))

##### <span style="color:MediumSlateBlue">Обучение модели с лучшими параметрами</span>

In [ ]:
# для выбора sceler преобразвания на понадобится словарь
dict_scalers ={
    'MinMaxScaler':MinMaxScaler(),
    'RobustScaler':RobustScaler(),
    'StandardScaler':StandardScaler(),
}

# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'credit'
count_features = dict_spaces[feature_space]

# с помощью функции features_from_transform_data_torow извлечем  
# из данных transform_data_torow_25
# признаки сооствествующие n_last последним клиенским операциям
list_n_last_features = features_from_transform_data_torow(best_n_last,count_features,25)

# обьявим scaler
scaler = dict_scalers[best_scaler]

# загружаем обучующие наборы
X_train_pd = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas()
y_train_pd = pd.read_csv('target/target_train.csv')

# поготовим данные y_train_pd к работе с функцией class_1_percent_samples
y_train_pd.set_index('id',drop=False,inplace=True)

# подготовим данные для обучения модели
# с помощью функции class_1_percent_samples зададим долю 
# класса 1
list_c1_percent_id = class_1_percent_samples(y_train_pd,best_class_1_percent,random_state=best_random_state)[:1000000]

# сформируем данные для обучения базовой модели
X_train_s_balanced = scaler.fit_transform(np.array(X_train_pd.loc[list_c1_percent_id])[:,list_n_last_features])
y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

# освободим память от "тяжелых" и ненужных файлов
del X_train_pd, y_train_pd
gc.collect()


# обучаем модель LogisticRegression с наилучшеми параметрами
logistic_regression = linear_model.LogisticRegression(
        **best_param_lr_torow,
        random_state=best_random_state,
        max_iter=10000)
logistic_regression.fit(X_train_s_balanced,y_train_balanced)

# удаляем крупные файлы чтобы высвободить память 
del X_train_s_balanced,y_train_balanced
gc.collect()

# подгружаем данные для тестирования
X_train = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas().to_numpy()[:,list_n_last_features]
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
X_valid = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_valid').to_pandas().to_numpy()[:,list_n_last_features]
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()
    
# сформируем данные для проверки модели
X_train_s = scaler.transform(X_train)
X_valid_s = scaler.transform(X_valid)

# удаляем крупные файлы чтобы высвободить память 
del X_train, X_valid
gc.collect()

# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_lr_pred_proba = logistic_regression.predict_proba(X_train_s)[:,1]
y_valid_lr_pred_proba = logistic_regression.predict_proba(X_valid_s)[:,1]

# Делаем предсказание для валидационной выборки
y_valid_lr_pred = logistic_regression.predict(X_valid_s)

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_lr_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_lr_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_valid,y_valid_lr_pred,zero_division=0))

# time: 2m 40s

##### <span style="color:MediumSlateBlue">Анализ влияния порога классификации на качество модели</span>

In [ ]:
# чтобы проше обращаться к каждому обьекту в данных
# преобразуем y_valid_LR_pred к Series типу
y_valid_lr_pred_proba = pd.Series(y_valid_lr_pred_proba)

# формируем список из необходимых значений thresholds
thresholds = np.arange(0.01, 1, 0.01)

# сформируем списко для заполнения данными результатов анализа
threshold_data = []


for threshold in thresholds:
        # сформируем список для заполнения данными текушего threshold
        threshold_current_data = [threshold]

        #клиентов, для которых вероятность дефолта > threshold, относим к классу 1
        #в противном случае — к классу 0
        y_valid_lr_pred = y_valid_lr_pred_proba.apply(lambda x: 1 if x>threshold else 0)


        #Считаем метрики для класса 0 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.precision_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.f1_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))

        #Считаем метрики для класса 1 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.precision_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.f1_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))

        # добавляем полученные данные в список threshold_data
        threshold_data.append(threshold_current_data)

        # из полученных данных сформируем dataframe
        thresholds_pd =pd.DataFrame(
        columns=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'],
        data = threshold_data)

# time: 17s

In [ ]:
# Визуализируем метрики на валидационном наборе при различных threshold
fig_threshold = px.line(
    data_frame=thresholds_pd,
    x='threshold', #ось абсцисс
    y=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'], #ось ординат
)
fig_threshold.update_layout(
    title ={
        'text':'Зависимость качества модели от threshold', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Порог классификации threshold',
    yaxis_title='Величина метрики')
fig_threshold.update_xaxes(showspikes=True)
fig_threshold.update_yaxes(showspikes=True)

fig_threshold.show()

Метрика $precision$ показывает на сколько сильно ошиблась модель при определиние класса.  
Чем выше метрика, тем меньше ошиблась модель. 

$$precision = \frac{TP}{TP+FP}$$

Метрика $recall$ показывает способность модели найти класс.  
Чем выше метрика, тем выше способность модели находить класс.  

$$recall = \frac{TP}{TP+FN}$$

Метрика $f1-score$ показывает баланс между $precision$ и $recall$ 

$threshold$- порог класификации модели.  
Все объекты для которых $у_{\text{proba pred}} < threshold$, модель относит к класс 0.  
Соотвественно, если $у_{\text{proba pred}}  > threshold$ модель относит обьект к классу 1.  
Для идельаной модели $threshold$, является границей между областями класса 0 и класса 1.  

<span style="color:Blue">

Выводы по результам анализа:  

Для улучшения качества модели по метрике $ROC AUC$, нет смысла сдвигать порог   
классификации. Но зависимость метрик качества модели от $threshold$, может рассказать о  
спосбоности модели искать класс 1 и класс 0.

При фокусировке модели на классе 1 (более низкий $threshold$):  
- на классе 1 метрика $precision$ уменьшается, метрика $recall$ растет;  
- на классе 0 метрика $precision$ растет, метрика $recall$ уменьшается.  

При фокусировке модели на классе 0 (более высокий $threshold$):  
- на классе 1 метрика $precision$ увеличивается, метрика $recall$ уменьшается;  
- на классе 0 метрика $precision$ уменьшается, метрика $recall$ растет.  

Обьсняется тем, что в условия дисбаланса модель научилась хорошо определять класс 0 и плохо класс 1.  

При фокусировке модели на классе 1 (более низкий $threshold$), все больше обьектов отнесены к классу 1.   
Так как $precision$ класса 1 уменьшается, значит в области класса 1 стало больше обектов класса 0,  
что явлется закономерным для "идеальной" модели.  
Однако при этом, $recall$ класса 1 растет, значит в добавленных обьектах также присутвует класс 1.  
Это как раз указывает на то, что модель не нучилась находить среди класса 0 класс 1. В области с     
высокой вероятностью класса 0 (более низкий $threshold$), находятся обьекты класса 1.  

Фокусировка на классе 0 (более высокий $threshold$), заставляет модель все больше обьектов из области класса 1  
относить к классу 0. При этом увеличение метрики $precision$ класса 1 говорит о том, что в области класса 1     
становится меньше 0, а значит модель умеет среди класса 1, находить класс 0.  


Обратим внимание на $threshold=0.63$, в этой точке находится баланс межу метриками качества модели:
$$precision_1 = recall_1 = \text{f1-score}_1 = 0.053$$
$$precision_0 = recall_0 = \text{f1-score}_0 = 0.964$$

Для модели с лучшим качеством эти две точки должны находится максимально близко к друг другу и иметь  
высокое значение метрик $precision_1$ и $recall_1$.  
Для класса 0 это условие выполнется, что также говорит о том, что модель научилась искать этот класс.   
Для класса 1 это условие не выполнется, что также говорит о том, что модель плохо ищет класс 1.

##### <span style="color:MediumSlateBlue">Построение ROC-кривой выбранной модели на тестовом наборе</span>

In [ ]:
# определим пространство признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'credit'
count_features = dict_spaces[feature_space]

In [ ]:
# подгружаем данные для тестирования
X_train = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas().to_numpy()[:,list_n_last_features]
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
X_valid = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_valid').to_pandas().to_numpy()[:,list_n_last_features]
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()
X_test = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_test').to_pandas().to_numpy()[:,list_n_last_features]
y_test = pd.read_csv('target/target_test.csv')['flag'].to_numpy()

In [ ]:
# выполним scaler преобразование
X_train_s = scaler.transform(X_train)
X_valid_s = scaler.transform(X_valid)
X_test_s = scaler.transform(X_test)

print("Размеры обучающего набора",X_train_s.shape)
print("Размеры валидационного набора",X_valid_s.shape)
print("Размеры тестового набора",X_test_s.shape)
# time: 1m 43s

In [ ]:
# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_lr_pred_proba = logistic_regression.predict_proba(X_train_s)[:,1]
y_valid_lr_pred_proba = logistic_regression.predict_proba(X_valid_s)[:,1]
y_test_lr_pred_proba = logistic_regression.predict_proba(X_test_s)[:,1]

# Делаем предсказание для валидационной выборки
y_test_lr_pred = logistic_regression.predict(X_test_s)

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_lr_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_lr_pred_proba),3))
print('ROC AUC на тестовом наборе', round(metrics.roc_auc_score(y_test, y_test_lr_pred_proba),3))

print('Основные метрики на тестовом наборе:')
print(metrics.classification_report(y_test,y_test_lr_pred,zero_division=0))

In [ ]:
# Для расчета значений TPR и FPR используем функцию roc_curve
fpr_train, tpr_train, _ = metrics.roc_curve(y_train, y_train_lr_pred_proba)
fpr_valid, tpr_valid, _ = metrics.roc_curve(y_valid, y_valid_lr_pred_proba)
fpr_test, tpr_test, _ = metrics.roc_curve(y_test, y_test_lr_pred_proba)

# рассчитаем площадь под кривой
auc_train = round(metrics.roc_auc_score(y_train, y_train_lr_pred_proba),3)
auc_valid = round(metrics.roc_auc_score(y_valid, y_valid_lr_pred_proba),3)
auc_test = round(metrics.roc_auc_score(y_test, y_test_lr_pred_proba),3)

trace_train = go.Scatter(x=fpr_train, y=tpr_train, mode='lines', name='AUC_train = %0.2f' % auc_train,
                   line=dict(color='red', width=2),
                   hovertemplate='train<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_train])

trace_valid = go.Scatter(x=fpr_valid, y=tpr_valid, mode='lines', name='AUC_valid = %0.2f' % auc_valid,
                   line=dict(color='green', width=2),
                   hovertemplate='valid<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_valid])

trace_test = go.Scatter(x=fpr_test, y=tpr_test, mode='lines', name='AUC_test = %0.2f' % auc_test,
                   line=dict(color='darkorange', width=2),
                   hovertemplate='test<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_test])


reference_line = go.Scatter(x=[0,1], y=[0,1], mode='lines', name='Reference Line',
                            line=dict(color='navy', width=2, dash='dash'))

fig_roc = go.Figure(data=[trace_train,trace_valid,trace_test,reference_line])

fig_roc.update_layout(
    title ={
        'text':'ROC-кривая', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 1350,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate')

fig_roc.update_xaxes(showspikes=True)
fig_roc.update_yaxes(showspikes=True)


fig_roc.show()

<span style="color:Blue">

Вывод по ROC-кривой:
1. Модель выдает сталибильное качество на всех наборах, а значит  
не переобучена - разброс ($variance$) не большой.  
2. Смещение ($bias$) модели относительно большое. Качество чуть лучше, чем  
случайное угадывание.
На тестовом наборе полученно качество $ROC AUC = 0.56$.

#### <span style="color:MediumBlue">Построение модели Random Forest Classifier</span>

##### <span style="color:MediumSlateBlue">Baseline обучение модели RandomForestClassifier</span>

In [ ]:
# обучаем модель логистической регрессии
# чтобы модель пыталась найти связь во всех признаках
random_forest = RandomForestClassifier(random_state=42, n_jobs=-1)
random_forest.fit(np.array(X_train_balanced)[:,list_n_last_features],np.array(y_train_balanced))

# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_rf__pred_proba = random_forest.predict_proba(X_train[:,list_n_last_features])[:,1]
y_valid_rf_pred_proba = random_forest.predict_proba(X_valid[:,list_n_last_features])[:,1]

# Делаем предсказание для валидационной выборки
y_valid_rf_pred = random_forest.predict(X_valid[:,list_n_last_features])

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_rf__pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_rf_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_valid,y_valid_rf_pred))

# time: 2m 5s

##### <span style="color:MediumSlateBlue">Подбор гиперпараметров модели</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'credit'
count_features = dict_spaces[feature_space]

# настроим оптимизацию гипер параметров
def optuna_rf(trial):
  # задаем пространства поиска гиперпараметров
  # задаем пространства поиска гиперпараметров
  n_estimators = trial.suggest_int('n_estimators', 50, 150,step = 5)
  criterion = trial.suggest_categorical('criterion',['gini', 'entropy', 'log_loss'])
  max_depth = trial.suggest_int('max_depth', 9, 20,step = 1)
  min_samples_split = trial.suggest_int('min_samples_split', 5, 15,step = 1)
  min_samples_leaf = trial.suggest_int('min_samples_leaf', 5, 25,step = 1)
  max_features = trial.suggest_float('max_features',0.1,1)
  max_samples = trial.suggest_float('max_samples',0.1,1)
  class_0_weight = trial.suggest_float('class_0_weight',0.01,1)
  class_1_weight = trial.suggest_float('class_1_weight',0.01,1)
  n_last = trial.suggest_int('n_last', 5, 25,step = 1)
  class_1_percent = trial.suggest_float('class_1_percent',0.01,1)
  random_state = trial.suggest_int('random_state', 1, 1000000)


  # создаем модель
  optuna_random_forest = RandomForestClassifier(
      n_estimators = n_estimators, 
      criterion = criterion,
      max_depth = max_depth,
      min_samples_split = min_samples_split,  
      min_samples_leaf = min_samples_leaf,
      max_features=max_features,
      max_samples=max_samples,
      class_weight={0:class_0_weight,1:class_1_weight},
      n_jobs=-1,
      random_state=random_state)

  # с помощью функции features_from_transform_data_torow извлечем  
  # из данных transform_data_torow
  # признаки сооствествующие n_last последним клиенским операциям
  list_n_last_features = features_from_transform_data_torow(n_last,count_features,25)

  # загружаем обучующие наборы
  X_train_pd = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas()
  y_train_pd = pd.read_csv('target/target_train.csv')

  # поготовим данные y_train_pd к работе с функцией class_1_percent_samples
  y_train_pd.set_index('id',drop=False,inplace=True)

  # подготовим данные для обучения модели
  # с помощью функции class_1_percent_samples зададим долю 
  # класса 1
  list_c1_percent_id = class_1_percent_samples(y_train_pd,class_1_percent,random_state=random_state)[:1000000]

  # сформируем данные для обучения базовой модели
  X_train_balanced = np.array(X_train_pd.loc[list_c1_percent_id])[:,list_n_last_features]
  y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

  # освободим память от "тяжелых" и ненужных файлов
  del X_train_pd, y_train_pd
  gc.collect()

  # я не воспользовался параметром timeout метода optimize потому, что  
  # optimize отставливает поиск параметров, если время обучения 
  # превышает timeout. Мне нужно чтобы поиск параметров продолжился.
  try:
    # для модуля func_time_out упакуем обучение модели в функцию try_func
    def try_func():
            return optuna_random_forest.fit(X_train_balanced,y_train_balanced)
    # обучаем модель с ограничением по времени 10 минут (600 секунд)
    optuna_random_forest = func_timeout.func_timeout(600, try_func)

    # удаляем крупные файлы чтобы высвободить память 
    del X_train_balanced, y_train_balanced
    gc.collect()

    # подгружаем данные для тестирования
    X_train = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas().to_numpy()[:,list_n_last_features]
    y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
    X_valid = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_valid').to_pandas().to_numpy()[:,list_n_last_features]
    y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()

    # делаем предсказание на обучающем и валидационном наборе и считаем метрики
    roc_train = metrics.roc_auc_score(y_train, optuna_random_forest.predict_proba(X_train)[:,1])
    roc_valid = metrics.roc_auc_score(y_valid, optuna_random_forest.predict_proba(X_valid)[:,1])
    f1_score = metrics.f1_score(y_valid, optuna_random_forest.predict(X_valid))
    recall_1 = metrics.recall_score(y_valid, optuna_random_forest.predict(X_valid),average=None,zero_division=0)[1]
    precision_1 = metrics.precision_score(y_valid, optuna_random_forest.predict(X_valid),average=None,zero_division=0)[1]

    # удаляем крупные файлы чтобы высвободить память 
    del X_train, X_valid, y_train, y_valid 
    gc.collect()

  except func_timeout.FunctionTimedOut:
    # освободим память от "тяжелых" и ненужных файлов
    del X_train_balanced, y_train_balanced
    gc.collect()

    # обьявим метрики пустыми значениями
    roc_train = 0
    roc_valid = 0
    f1_score = 0
    recall_1 = 0
    precision_1 = 0
    pass

  return roc_train, roc_valid, f1_score, recall_1, precision_1

In [ ]:
# так как Random Forest склонен к переобучению 
# попробуем направить optimize в сторону уменьшения roc_train
# cоздаем объект исследования
optuna_study_rf = optuna.create_study(study_name='RandomForestClassifier_'+feature_space+'_torow', 
                               directions=['minimize','maximize','maximize','maximize','maximize'], 
                               sampler=optuna.samplers.TPESampler(),
                               pruner='Hyperband',
                               storage='sqlite:///optuna_studies.db',
                               load_if_exists=True)
# на случай удаления 
# optuna.delete_study(study_name='RandomForestClassifier_'+feature_space+'_torow', storage='sqlite:///optuna_studies.db')

In [ ]:
# ищем лучшую комбинацию гиперпараметров n_trials раз
optuna_study_rf.optimize(optuna_rf, n_trials=300)

In [ ]:
# из полученного результат соврмируем Data Frame
optuna_study_rf_pd = optuna_study_rf.trials_dataframe().dropna()
# переименуем столбы
optuna_study_rf_pd.rename(columns={
    'values_0': 'roc_train',
    'values_1': 'roc_valid',
    'values_2': 'f1_score',
    'values_3': 'recall_1',
    'values_4': 'precision_1'
},inplace=True)
optuna_study_rf_pd.sort_values(by='precision_1').drop(['datetime_start','datetime_complete','duration'],axis=1)

In [ ]:
# покаждем статистику обучения
print('Максимальное значение метрики ROC AUC на валидационном наборе:',round(optuna_study_rf_pd['roc_valid'].max(),3))
print('Среднее значение метрики ROC AUC на валидационном наборе:',round(optuna_study_rf_pd['roc_valid'].mean(),3))
print('Максимальное значение метрики f1_score на валидационном наборе:',round(optuna_study_rf_pd['f1_score'].max(),3))
print('Среднее значение метрики f1_score на валидационном наборе:',round(optuna_study_rf_pd['f1_score'].mean(),3))
print('Максимальное значение метрики recall_1 на валидационном наборе:',round(optuna_study_rf_pd['recall_1'].max(),3))
print('Среднее значение метрики recall_1 на валидационном наборе:',round(optuna_study_rf_pd['recall_1'].mean(),3))
print('Максимальное значение метрики precision_1 на валидационном наборе:',round(optuna_study_rf_pd['precision_1'].max(),3))
print('Среднее значение метрики precision_1 на валидационном наборе:',round(optuna_study_rf_pd['precision_1'].mean(),3))

In [ ]:
# построим график важности гиперпарметров
optuna.visualization.plot_param_importances(optuna_study_rf, target_name='Score')

In [ ]:
# Построим зависимость метрики precision_1 от гиперпараметров

fig = px.scatter(
    data_frame=optuna_study_rf_pd,
    x='precision_1', #ось абсцисс
    y=['f1_score', 'recall_1'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision класса 1 от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision на классе 1',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики precision_1 от гиперпараметров

fig = px.scatter(
    data_frame=optuna_study_rf_pd,
    x='precision_1', #ось абсцисс
    y=['params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight','params_max_depth','params_max_features',
        'params_max_samples', 'params_min_samples_leaf', 'params_min_samples_split', 
        'params_n_estimators', 'params_n_last'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision класса 1 от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision на классе 1',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики f1_score от гиперпараметров

fig = px.scatter(
    data_frame=optuna_study_rf_pd,
    x='f1_score', #ось абсцисс
    y=['params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight','params_max_depth','params_max_features',
        'params_max_samples', 'params_min_samples_leaf', 'params_min_samples_split', 
        'params_n_estimators', 'params_n_last'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision класса 1 от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision на классе 1',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики precision_1 от гиперпараметров

fig = px.scatter(
    data_frame=optuna_study_rf_pd,
    x='roc_valid', #ось абсцисс
    y=['params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight','params_max_depth','params_max_features',
        'params_max_samples', 'params_min_samples_leaf', 'params_min_samples_split', 
        'params_n_estimators', 'params_n_last'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость roc valid от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики roc valid',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрик качества модели от number

fig = px.scatter(
    data_frame=optuna_study_rf_pd[(optuna_study_rf_pd['roc_valid']>=0.644)],
    x='number', #ось абсцисс
    y=['roc_valid','roc_train','precision_1','recall_1'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость метрик качества модели от number', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики ROC Valid',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

<span style="color:Blue">

Вывод:  

Их всех выбираем точку $number =148$.   
В этой точке оптимальные значения метрик $recall_1$ и $precision_1$, а  
также относительно высокая метрика $ROC AUC$.   
Посмотрим каким значениям гиперпараметров соотвествует выбраннаяточка.

In [ ]:
# определим номер лучшге варианта
best_optuna_number = 148

# сформируем словарь лучших гипирпарметров RandomForestClassifier
best_param_rf = {
    'n_estimators' : int(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_n_estimators'].iloc[0]),
    'criterion': optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_criterion'].iloc[0],
    'max_depth' : optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_max_depth'].iloc[0],
    'min_samples_split': optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_min_samples_split'].iloc[0],
    'min_samples_leaf' : optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_min_samples_leaf'].iloc[0],
    'max_features': optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_max_features'].iloc[0],
    'max_samples' : optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_max_samples'].iloc[0],
    'class_weight' : {0:optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_class_0_weight'].iloc[0],
                      1:optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_class_1_weight'].iloc[0]}
    }
# определим перменные для лучших значений параметров
best_n_last = optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_n_last'].iloc[0]
best_class_1_percent = optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_class_1_percent'].iloc[0]
best_random_state = optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_random_state'].iloc[0]


# Выведем принятые наилучшие праметры
print('best n estimators:',best_param_rf['n_estimators'])
print('best criterion:',best_param_rf['criterion'])
print('best max depth:',best_param_rf['max_depth'])
print('best min samples split:',best_param_rf['min_samples_split'])
print('best min samples leaf:',best_param_rf['min_samples_leaf'])
print('best max features(%):',round(best_param_rf['max_features'],3))
print('best max samples(%):',round(best_param_rf['max_samples'],3))
print('best class 0 weight:',round(best_param_rf['class_weight'][0],3))
print('best class 1 weight:',round(best_param_rf['class_weight'][1],3))

print('best class 1 percent:',round(best_class_1_percent,3))
print('best n last:',best_n_last)
print('best random state:',best_random_state)
print('time for best train:',round(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['duration'].iloc[0].seconds/60),'minutes')

print()
print('ROC AUC на обучающем наборе:', round(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['roc_train'].iloc[0],3))
print('ROC AUC на валидационном наборе:', round(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['roc_valid'].iloc[0],3))
print('precision класса 1:', round(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['precision_1'].iloc[0],3))
print('recall класса 1:', round(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['recall_1'].iloc[0],3))

##### <span style="color:MediumSlateBlue">Обучение модели с лучшими параметрами</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'credit'
count_features = dict_spaces[feature_space]

# с помощью функции features_from_transform_data_torow извлечем  
# из данных transform_data_torow
# признаки сооствествующие n_last последним клиенским операциям
list_n_last_features = features_from_transform_data_torow(best_n_last,count_features,25)

# загружаем обучующие наборы
X_train_pd = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas()
y_train_pd = pd.read_csv('target/target_train.csv')

# поготовим данные y_train_pd к работе с функцией class_1_percent_samples
y_train_pd.set_index('id',drop=False,inplace=True)

# подготовим данные для обучения модели
# с помощью функции class_1_percent_samples зададим долю 
# класса 1
list_c1_percent_id = class_1_percent_samples(y_train_pd,best_class_1_percent,random_state=best_random_state)[:1000000]

# сформируем данные для обучения базовой модели
X_train_balanced = np.array(X_train_pd.loc[list_c1_percent_id])[:,list_n_last_features]
y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

# освободим память от "тяжелых" и ненужных файлов
del X_train_pd, y_train_pd
gc.collect()

# обучаем модель RandomForestClassifier с наилучшеми параметрами
random_forest = RandomForestClassifier(
        **best_param_rf,
        n_jobs=-1,
        random_state=best_random_state)
random_forest.fit(X_train_balanced, y_train_balanced)

# освободим память от "тяжелых" и ненужных файлов
del X_train_balanced, y_train_balanced
gc.collect()


# подгружаем данные для тестирования
X_train = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas().to_numpy()[:,list_n_last_features]
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
X_valid = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_valid').to_pandas().to_numpy()[:,list_n_last_features]
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()

# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_rf__pred_proba = random_forest.predict_proba(X_train)[:,1]
y_valid_rf_pred_proba = random_forest.predict_proba(X_valid)[:,1]

# Делаем предсказание для валидационной выборки
y_valid_rf_pred = random_forest.predict(X_valid)

# удаляем крупные файлы чтобы высвободить память 
del X_train, X_valid
gc.collect()

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_rf__pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_rf_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_valid,y_valid_rf_pred,zero_division=0))

# time: 2m 10s

##### <span style="color:MediumSlateBlue">Анализ влияния порога классификации на качество модели</span>

In [ ]:
# чтобы проше обращаться к каждому обьекту в данных
# преобразуем y_valid_LR_pred к Series типу
y_valid_rf_pred_proba = pd.Series(y_valid_rf_pred_proba)

# формируем список из необходимых значений thresholds
thresholds = np.arange(0.01, 1, 0.01)

# сформируем списко для заполнения данными результатов анализа
threshold_data = []


for threshold in thresholds:
        # сформируем список для заполнения данными текушего threshold
        threshold_current_data = [threshold]

        #клиентов, для которых вероятность дефолта > threshold, относим к классу 1
        #в противном случае — к классу 0
        y_valid_rf_pred = y_valid_rf_pred_proba.apply(lambda x: 1 if x>threshold else 0)


        #Считаем метрики для класса 0 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(y_valid, y_valid_rf_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.precision_score(y_valid, y_valid_rf_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.f1_score(y_valid, y_valid_rf_pred,average=None,zero_division=0)[0],3))

        #Считаем метрики для класса 1 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(y_valid, y_valid_rf_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.precision_score(y_valid, y_valid_rf_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.f1_score(y_valid, y_valid_rf_pred,average=None,zero_division=0)[1],3))

        # добавляем полученные данные в список threshold_data
        threshold_data.append(threshold_current_data)

        # из полученных данных сформируем dataframe
        thresholds_pd =pd.DataFrame(
        columns=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'],
        data = threshold_data)

# time: 17s

In [ ]:
# Визуализируем метрики на валидационном наборе при различных threshold
fig_threshold = px.line(
    data_frame=thresholds_pd,
    x='threshold', #ось абсцисс
    y=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'], #ось ординат
)
fig_threshold.update_layout(
    title ={
        'text':'Зависимость качества модели от threshold', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Порог классификации threshold',
    yaxis_title='Величина метрики')
fig_threshold.update_xaxes(showspikes=True)
fig_threshold.update_yaxes(showspikes=True)

fig_threshold.show()

Метрика $precision$ показывает на сколько сильно ошиблась модель при определиние класса.  
Чем выше метрика, тем меньше ошиблась модель. 

$$precision = \frac{TP}{TP+FP}$$

Метрика $recall$ показывает способность модели найти класс.  
Чем выше метрика, тем выше способность модели находить класс.  

$$recall = \frac{TP}{TP+FN}$$

Метрика $f1-score$ показывает баланс между $precision$ и $recall$ 

$threshold$- порог класификации модели.  
Все объекты для которых $у_{\text{proba pred}} < threshold$, модель относит к класс 0.  
Соотвественно, если $у_{\text{proba pred}}  > threshold$ модель относит обьект к классу 1.  
Для идельаной модели $threshold$, является границей между областями класса 0 и класса 1.  

<span style="color:Blue">

Выводы по результам анализа:  

Для улучшения качества модели по метрике $ROC AUC$, нет смысла сдвигать порог   
классификации. Но зависимость метрик качества модели от $threshold$, может рассказать о  
способности модели искать класс 1 и класс 0.

При фокусировке модели на классе 1 (более низкий $threshold$):  
- на классе 1 метрика $precision$ уменьшается, метрика $recall$ растет;  
- на классе 0 метрика $precision$ растет, метрика $recall$ уменьшается.  

При фокусировке модели на классе 0 (более высокий $threshold$):  
- на классе 1 метрика $precision$ увеличивается, метрика $recall$ уменьшается;  
- на классе 0 метрика $precision$ уменьшается, метрика $recall$ растет.  

Обьсняется тем, что в условия дисбаланса модель научилась хорошо определять класс 0 и плохо класс 1.  

При фокусировке модели на классе 1 (более низкий $threshold$), все больше обьектов отнесены к классу 1.   
Так как $precision$ класса 1 уменьшается, значит в области класса 1 стало больше обектов класса 0,  
что явлется закономерным для "идеальной" модели.  
Однако при этом, $recall$ класса 1 растет, значит в добавленных обьектах также присутвует класс 1.  
Это как раз указывает на то, что модель не нучилась находить среди класса 0 класс 1. В области с     
высокой вероятностью класса 0 (более низкий $threshold$), находятся обьекты класса 1.  

Фокусировка на классе 0 (более высокий $threshold$), заставляет модель все больше обьектов из области класса 1  
относить к классу 0. При этом увеличение метрики $precision$ класса 1 говорит о том, что в области класса 1     
становится меньше 0, а значит модель умеет среди класса 1, находить класс 0.  
После  $threshold=0.78$ модель впринципе теряет способность отличать класс 1. У модели нет обьектов класса 1   
в которых она "уверена" на 80+%.

Обратим внимание на $threshold=0.5$, в этой точке находится баланс межу метриками качества модели:
$$precision_1 = recall_1 = \text{f1-score}_1 = 0.113$$
$$precision_0 = recall_0 = \text{f1-score}_0 = 0.968$$

Для модели с лучшим качеством эти две точки должны находится максимально близко к друг другу и иметь   
высокое значение метрик $precision_1$ и $recall_1$.  
Для класса 0 это условие выполнется, что также говорит о том, что модель научилась искать этот класс.    
Для класса 1 это условие не выполнется, что также говорит о том, что модель плохо ищет класс 1.

##### <span style="color:MediumSlateBlue">Построение ROC-кривой выбранной модели на тестовом наборе</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'credit'
count_features = dict_spaces[feature_space]

In [ ]:
# подгружаем данные для тестирования
X_train = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas().to_numpy()[:,list_n_last_features]
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
X_valid = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_valid').to_pandas().to_numpy()[:,list_n_last_features]
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()
X_test = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_test').to_pandas().to_numpy()[:,list_n_last_features]
y_test = pd.read_csv('target/target_test.csv')['flag'].to_numpy()

In [ ]:
print("Размеры обучающего набора",X_train.shape,y_train.shape)
print("Размеры валидационного набора",X_valid.shape,y_valid.shape)
print("Размеры тестового набора",X_test.shape,y_test.shape)
# time: 1m 43s

In [ ]:
# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_rf_pred_proba = random_forest.predict_proba(X_train)[:,1]
y_valid_rf_pred_proba = random_forest.predict_proba(X_valid)[:,1]
y_test_rf_pred_proba = random_forest.predict_proba(X_test)[:,1]

# Делаем предсказание для валидационной выборки
y_test_lr_pred = random_forest.predict(X_test)

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_rf_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_rf_pred_proba),3))
print('ROC AUC на тестовом наборе', round(metrics.roc_auc_score(y_test, y_test_rf_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_test,y_test_lr_pred,zero_division=0))

In [ ]:
# Для расчета значений TPR и FPR используем функцию roc_curve
fpr_train, tpr_train, _ = metrics.roc_curve(y_train, y_train_rf_pred_proba)
fpr_valid, tpr_valid, _ = metrics.roc_curve(y_valid, y_valid_rf_pred_proba)
fpr_test, tpr_test, _ = metrics.roc_curve(y_test, y_test_rf_pred_proba)

# рассчитаем площадь под кривой
auc_train = round(metrics.roc_auc_score(y_train, y_train_rf_pred_proba),3)
auc_valid = round(metrics.roc_auc_score(y_valid, y_valid_rf_pred_proba),3)
auc_test = round(metrics.roc_auc_score(y_test, y_test_rf_pred_proba),3)

trace_train = go.Scatter(x=fpr_train, y=tpr_train, mode='lines', name='AUC_train = %0.2f' % auc_train,
                   line=dict(color='red', width=2),
                   hovertemplate='train<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_train])

trace_valid = go.Scatter(x=fpr_valid, y=tpr_valid, mode='lines', name='AUC_valid = %0.2f' % auc_valid,
                   line=dict(color='green', width=2),
                   hovertemplate='valid<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_valid])

trace_test = go.Scatter(x=fpr_test, y=tpr_test, mode='lines', name='AUC_test = %0.2f' % auc_test,
                   line=dict(color='darkorange', width=2),
                   hovertemplate='test<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_test])


reference_line = go.Scatter(x=[0,1], y=[0,1], mode='lines', name='Reference Line',
                            line=dict(color='navy', width=2, dash='dash'))

fig_roc = go.Figure(data=[trace_train,trace_valid,trace_test,reference_line])

fig_roc.update_layout(
    title ={
        'text':'ROC-кривая', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 1350,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate')

fig_roc.update_xaxes(showspikes=True)
fig_roc.update_yaxes(showspikes=True)


fig_roc.show()

<span style="color:Blue">

Вывод по ROC-кривой:
1. Модель выдает сталибильное качество на валидационном и тестовом наборах, а значит  
не переобучена - разброс ($variance$) не большой. Хотя, качество модели на обучающем наборе,  
заметно лучше.
2. Смещение ($bias$) модели гораздо лучше, чем у модели $Logicstic$ $Regression$ $lassifier$, обученной   
на тех же данных.  
На тестовом наборе полученно качество $ROC AUC = 0.65$, против $ROC AUC = 0.58$

#### <span style="color:MediumBlue">Построение модели Gradient Boosting Classifier</span>

##### <span style="color:MediumSlateBlue">Baseline обучение модели Hist Gradient Boosting Classifier</span>

In [ ]:
# обучаем модель Gradient Boosting Classifier
gradient_boosting = HistGradientBoostingClassifier(random_state=42)
gradient_boosting.fit(np.array(X_train_balanced)[:,list_n_last_features],np.array(y_train_balanced))

# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_rf__pred_proba = gradient_boosting.predict_proba(X_train[:,list_n_last_features])[:,1]
y_valid_rf_pred_proba = gradient_boosting.predict_proba(X_valid[:,list_n_last_features])[:,1]

# Делаем предсказание для валидационной выборки
y_valid_rf_pred = gradient_boosting.predict(X_valid[:,list_n_last_features])

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_rf__pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_rf_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_valid,y_valid_rf_pred))

# time: 

##### <span style="color:MediumSlateBlue">Подбор гиперпараметров модели</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'credit'
count_features = dict_spaces[feature_space]

# настроим оптимизацию гипер параметров
def optuna_hgbc(trial):
  # задаем пространства поиска гиперпараметров
  learning_rate = trial.suggest_float('learning_rate',0.2,0.6)
  max_iter = trial.suggest_int('max_iter', 50, 300,step = 1)
  max_leaf_nodes = trial.suggest_int('max_leaf_nodes', 2, 50,step = 1)
  max_depth = trial.suggest_int('max_depth', 1, 15,step = 1)
  min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 50,step = 1)
  max_features = trial.suggest_float('max_features',0.1,1)
  l2_regularization = trial.suggest_float('l2_regularization',0.01,1)
  class_0_weight = trial.suggest_float('class_0_weight',0.01,1)
  class_1_weight = trial.suggest_float('class_1_weight',0.01,1)
  n_last = trial.suggest_int('n_last', 5, 25,step = 1)
  class_1_percent = trial.suggest_float('class_1_percent',0.01,1)
  random_state = trial.suggest_int('random_state', 1, 1000000)


  # создаем модель
  optuna_gradient_boosting = HistGradientBoostingClassifier(
      learning_rate= learning_rate,
      max_iter = max_iter,
      max_leaf_nodes =max_leaf_nodes,
      max_depth = max_depth,
      min_samples_leaf = min_samples_leaf,
      max_features=max_features,
      l2_regularization = l2_regularization,
      class_weight={0:class_0_weight,1:class_1_weight},
      random_state=42)

  # с помощью функции features_from_transform_data_torow извлечем  
  # из данных transform_data_torow
  # признаки сооствествующие n_last последним клиенским операциям
  list_n_last_features = features_from_transform_data_torow(n_last,count_features,25)

  # загружаем обучующие наборы
  X_train_pd = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas()
  y_train_pd = pd.read_csv('target/target_train.csv')

  # поготовим данные y_train_pd к работе с функцией class_1_percent_samples
  y_train_pd.set_index('id',drop=False,inplace=True)

  # подготовим данные для обучения модели
  # с помощью функции class_1_percent_samples зададим долю класса 1
  list_c1_percent_id = class_1_percent_samples(y_train_pd,class_1_percent,random_state=random_state)[:1000000]

  # сформируем данные для обучения модели
  X_train_balanced = np.array(X_train_pd.loc[list_c1_percent_id])[:,list_n_last_features]
  y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

  # освободим память от "тяжелых" и ненужных файлов
  del X_train_pd, y_train_pd
  gc.collect()

  # я не воспользовался параметром timeout метода optimize потому, что  
  # optimize отставливает поиск параметров, если время обучения 
  # превышает timeout. Мне нужно чтобы поиск параметров продолжился.
  try:
    # для модуля func_time_out упакуем обучение модели в функцию try_func
    def try_func():
            return optuna_gradient_boosting.fit(X_train_balanced,y_train_balanced)
    # обучаем модель с ограничением по времени 10 минут (600 секунд)
    optuna_gradient_boosting = func_timeout.func_timeout(600, try_func)

    # удаляем крупные файлы чтобы высвободить память 
    del X_train_balanced, y_train_balanced
    gc.collect()

    # подгружаем данные для тестирования
    X_train = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas().to_numpy()[:,list_n_last_features]
    y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
    X_valid = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_valid').to_pandas().to_numpy()[:,list_n_last_features]
    y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()

    # делаем предсказание на обучающем и валидационном наборе и считаем метрики
    roc_train = metrics.roc_auc_score(y_train, optuna_gradient_boosting.predict_proba(X_train)[:,1])
    roc_valid = metrics.roc_auc_score(y_valid, optuna_gradient_boosting.predict_proba(X_valid)[:,1])
    f1_score = metrics.f1_score(y_valid, optuna_gradient_boosting.predict(X_valid))
    recall_1 = metrics.recall_score(y_valid, optuna_gradient_boosting.predict(X_valid),average=None,zero_division=0)[1]
    precision_1 = metrics.precision_score(y_valid, optuna_gradient_boosting.predict(X_valid),average=None,zero_division=0)[1]

    # удаляем крупные файлы чтобы высвободить память 
    del X_train, X_valid, y_train, y_valid 
    gc.collect()

  except func_timeout.FunctionTimedOut:
    # освободим память от "тяжелых" и ненужных файлов
    del X_train_balanced, y_train_balanced
    gc.collect()

    # обьявим метрики пустыми значениями
    roc_train = 0
    roc_valid = 0
    f1_score = 0
    recall_1 = 0
    precision_1 = 0
    pass

  return roc_train, roc_valid, f1_score, recall_1, precision_1

In [ ]:
# cоздаем объект исследования
# чтобы модель не переобучалась минимизируем roc_train 
# и максимизируем roc_valid
optuna_study_hsbs = optuna.create_study(study_name='HistGradientBoostingClassifier_'+feature_space+'_torow', 
                               directions=['maximize','maximize','maximize','maximize','maximize'], 
                               sampler=optuna.samplers.TPESampler(),
                               pruner='Hyperband',
                               storage='sqlite:///optuna_studies.db',
                               load_if_exists=True)

# ну случай удаления обучения
# optuna.delete_study(study_name='HistGradientBoostingClassifier_'+feature_space+'_torow', storage='sqlite:///optuna_studies.db')

In [ ]:
# ищем лучшую комбинацию гиперпараметров n_trials раз
optuna_study_hsbs.optimize(optuna_hgbc, n_trials=300)

In [ ]:
# из полученного результат соврмируем Data Frame
optuna_study_hsbs_pd = optuna_study_hsbs.trials_dataframe()
# переименуем столбы
optuna_study_hsbs_pd.rename(columns={
    'values_0': 'roc_train',
    'values_1': 'roc_valid',
    'values_2': 'f1_score',
    'values_3': 'recall_1',
    'values_4': 'precision_1'
},inplace=True)
# Выделим время потраченное на обучение модели
last_trail = optuna_study_hsbs_pd['number'].max()
optuna_study_hsbs_pd.tail(5)


In [ ]:
# покаждем статистику обучения
print('Максимальное значение метрики ROC AUC на валидационном наборе:',round(optuna_study_hsbs_pd['roc_valid'].max(),3))
print('Среднее значение метрики ROC AUC на валидационном наборе:',round(optuna_study_hsbs_pd['roc_valid'].mean(),3))
print('Максимальное значение метрики f1_score на валидационном наборе:',round(optuna_study_hsbs_pd['f1_score'].max(),3))
print('Среднее значение метрики f1_score на валидационном наборе:',round(optuna_study_hsbs_pd['f1_score'].mean(),3))
print('Максимальное значение метрики recall_1 на валидационном наборе:',round(optuna_study_hsbs_pd['recall_1'].max(),3))
print('Среднее значение метрики recall_1 на валидационном наборе:',round(optuna_study_hsbs_pd['recall_1'].mean(),3))
print('Максимальное значение метрики precision_1 на валидационном наборе:',round(optuna_study_hsbs_pd['precision_1'].max(),3))
print('Среднее значение метрики precision_1 на валидационном наборе:',round(optuna_study_hsbs_pd['precision_1'].mean(),3))

In [ ]:
# построим график важности гиперпарметров
optuna.visualization.plot_param_importances(optuna_study_hsbs, target_name='Score')

In [ ]:
# Построим зависимость метрики precision_1 от гиперпараметров
# max_iter

fig = px.scatter(
    data_frame=optuna_study_hsbs_pd,
    x='precision_1', #ось абсцисс
    y=['f1_score', 'recall_1'
    ], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость  метрик качества модели от params max iter', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина params max iter',
    yaxis_title='Величина метрики')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики precision_1 от гиперпараметров

fig = px.scatter(
    data_frame=optuna_study_hsbs_pd,
    x='precision_1', #ось абсцисс
    y=['params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight', 'params_l2_regularization',
       'params_learning_rate', 'params_max_depth', 'params_max_features',
       'params_max_iter', 'params_max_leaf_nodes', 'params_min_samples_leaf',
       'params_n_last'
    ], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость метрики precision класса 1 от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision класса 1',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики f1_score от гиперпараметров
fig = px.scatter(
    data_frame=optuna_study_hsbs_pd,
    x='f1_score', #ось абсцисс
    y=['params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight', 'params_l2_regularization',
       'params_learning_rate', 'params_max_depth', 'params_max_features',
       'params_max_iter', 'params_max_leaf_nodes', 'params_min_samples_leaf',
       'params_n_last'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость f1 score от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики f1 score',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики roc_valid от гиперпараметров
fig = px.scatter(
    data_frame=optuna_study_hsbs_pd,
    x='roc_valid', #ось абсцисс
    y=['params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight', 'params_l2_regularization',
       'params_learning_rate', 'params_max_depth', 'params_max_features',
       'params_max_iter', 'params_max_leaf_nodes', 'params_min_samples_leaf',
       'params_n_last'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость метрики roc valid от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики roc valid',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрик качества модели от number

fig = px.scatter(
    data_frame=optuna_study_hsbs_pd[(optuna_study_hsbs_pd['roc_valid']>0.67)],
    x='number', #ось абсцисс
    y=['roc_train','roc_valid','recall_1','precision_1'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision_1',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

<span style="color:Blue">

Вывод:  

Их всех выбираем точку $number =187$.   
В этой точке оптимальные значения метрик $recall_1$ и $precision_1$, а  
также относительно высокая метрика $ROC AUC$.   
Посмотрим каким значениям гиперпараметров соотвествует выбранная точка.

In [ ]:
# определим номер лучшге варианта
best_optuna_number = 187

# сформируем словарь лучших гипирпарметров RandomForestClassifier
best_param_rf = {
    'learning_rate' : optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['params_learning_rate'].iloc[0],
    'max_iter' : optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['params_max_iter'].iloc[0],
    'max_leaf_nodes' : optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['params_max_leaf_nodes'].iloc[0],
    'max_depth' : optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['params_max_depth'].iloc[0],
    'min_samples_leaf' : optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['params_min_samples_leaf'].iloc[0],
    'max_features' : optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['params_max_features'].iloc[0],
    'l2_regularization' : optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['params_l2_regularization'].iloc[0],
    'class_weight' : {0: optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['params_class_0_weight'].iloc[0],
                      1: optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['params_class_1_weight'].iloc[0],}
    }

# определим перменные для лучших значений параметров
best_n_last = optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['params_n_last'].iloc[0]
best_class_1_percent = optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['params_class_1_percent'].iloc[0]
best_random_state = optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['params_random_state'].iloc[0]


# Выведем принятые наилучшие праметры
print('best learning rate:',round(best_param_rf['learning_rate'],3))
print('best max iter:',best_param_rf['max_iter'])
print('best max leaf nodes:',best_param_rf['max_leaf_nodes'])
print('best max depth:',best_param_rf['max_depth'])
print('best min samples leaf:',best_param_rf['min_samples_leaf'])
print('best max features:',round(best_param_rf['max_features'],3))
print('best l2 regularization:',round(best_param_rf['l2_regularization'],3))
print('best class 0 weight:',round(best_param_rf['class_weight'][0],3))
print('best class 1 weight:',round(best_param_rf['class_weight'][1],3))

print('best class 1 percent:',round(best_class_1_percent,3))
print('best n last:',best_n_last)
print('best random state:',best_random_state)
print('time for best train:',round(optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['duration'].iloc[0].seconds/60),'minutes')

print()
print('ROC AUC на обучающем наборе:', round(optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['roc_train'].iloc[0],3))
print('ROC AUC на валидационном наборе:', round(optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['roc_valid'].iloc[0],3))
print('precision класса 1:', round(optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['precision_1'].iloc[0],3))
print('recall класса 1:', round(optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['recall_1'].iloc[0],3))

##### <span style="color:MediumSlateBlue">Обучение модели с лучшими параметрами</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'credit'
count_features = dict_spaces[feature_space]

# с помощью функции features_from_transform_data_torow извлечем  
# из данных transform_data_torow
# признаки сооствествующие n_last последним клиенским операциям
list_n_last_features = features_from_transform_data_torow(best_n_last,count_features,25)

# загружаем обучующие наборы
X_train_pd = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas()
y_train_pd = pd.read_csv('target/target_train.csv')

# поготовим данные y_train_pd к работе с функцией class_1_percent_samples
y_train_pd.set_index('id',drop=False,inplace=True)

# подготовим данные для обучения модели
# с помощью функции class_1_percent_samples зададим долю класса 1
list_c1_percent_id = class_1_percent_samples(y_train_pd,best_class_1_percent,random_state=best_random_state)[:1000000]

# сформируем данные для обучения модели
X_train_balanced = np.array(X_train_pd.loc[list_c1_percent_id])[:,list_n_last_features]
y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

# освободим память от "тяжелых" и ненужных файлов
del X_train_pd, y_train_pd
gc.collect()


# обучаем модель HistGradientBoostingClassifier с наилучшеми параметрами
gradient_boosting = HistGradientBoostingClassifier(
        **best_param_rf,
        random_state=42)
gradient_boosting.fit(X_train_balanced,y_train_balanced)

# освободим память от "тяжелых" и ненужных файлов
del X_train_balanced, y_train_balanced
gc.collect()

# подгружаем данные для тестирования
X_train = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas().to_numpy()[:,list_n_last_features]
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
X_valid = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_valid').to_pandas().to_numpy()[:,list_n_last_features]
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()

# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_gb__pred_proba = gradient_boosting.predict_proba(X_train)[:,1]
y_valid_gb_pred_proba = gradient_boosting.predict_proba(X_valid)[:,1]

# Делаем предсказание для валидационной выборки
y_valid_gb_pred = gradient_boosting.predict(X_valid)

# освободим память от "тяжелых" и ненужных файлов
del X_train, X_valid
gc.collect()

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_gb__pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_gb_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_valid,y_valid_gb_pred,zero_division=0))

# time: 2m 30s

##### <span style="color:MediumSlateBlue">Анализ влияния порога классификации на качество модели</span>

In [ ]:
# чтобы проше обращаться к каждому обьекту в данных
# преобразуем y_valid_LR_pred к Series типу
y_valid_gb_pred_proba = pd.Series(y_valid_gb_pred_proba)

# формируем список из необходимых значений thresholds
thresholds = np.arange(0.01, 1, 0.01)

# сформируем списко для заполнения данными результатов анализа
threshold_data = []


for threshold in thresholds:
        # сформируем список для заполнения данными текушего threshold
        threshold_current_data = [threshold]

        #клиентов, для которых вероятность дефолта > threshold, относим к классу 1
        #в противном случае — к классу 0
        y_valid_gb_pred = y_valid_gb_pred_proba.apply(lambda x: 1 if x>threshold else 0)


        #Считаем метрики для класса 0 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(y_valid, y_valid_gb_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.precision_score(y_valid, y_valid_gb_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.f1_score(y_valid, y_valid_gb_pred,average=None,zero_division=0)[0],3))

        #Считаем метрики для класса 1 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(y_valid, y_valid_gb_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.precision_score(y_valid, y_valid_gb_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.f1_score(y_valid, y_valid_gb_pred,average=None,zero_division=0)[1],3))

        # добавляем полученные данные в список threshold_data
        threshold_data.append(threshold_current_data)

        # из полученных данных сформируем dataframe
        thresholds_pd =pd.DataFrame(
        columns=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'],
        data = threshold_data)

# time: 17s

In [ ]:
# Визуализируем метрики на валидационном наборе при различных threshold
fig_threshold = px.line(
    data_frame=thresholds_pd,
    x='threshold', #ось абсцисс
    y=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'], #ось ординат
)
fig_threshold.update_layout(
    title ={
        'text':'Зависимость качества модели от threshold', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Порог классификации threshold',
    yaxis_title='Величина метрики')
fig_threshold.update_xaxes(showspikes=True)
fig_threshold.update_yaxes(showspikes=True)

fig_threshold.show()

Метрика $precision$ показывает на сколько сильно ошиблась модель при определиние класса.  
Чем выше метрика, тем меньше ошиблась модель. 

$$precision = \frac{TP}{TP+FP}$$

Метрика $recall$ показывает способность модели найти класс.  
Чем выше метрика, тем выше способность модели находить класс.  

$$recall = \frac{TP}{TP+FN}$$

Метрика $f1-score$ показывает баланс между $precision$ и $recall$ 

$threshold$- порог класификации модели.  
Все объекты для которых $у_{\text{proba pred}} < threshold$, модель относит к класс 0.  
Соотвественно, если $у_{\text{proba pred}}  > threshold$ модель относит обьект к классу 1.  
Для идельаной модели $threshold$, является границей между областями класса 0 и класса 1.  

<span style="color:Blue">

Выводы по результам анализа:  

Для улучшения качества модели по метрике $ROC AUC$, нет смысла сдвигать порог   
классификации. Но зависимость метрик качества модели от $threshold$, может рассказать о  
способности модели искать класс 1 и класс 0.

При фокусировке модели на классе 1 (более низкий $threshold$):  
- на классе 1 метрика $precision$ уменьшается, метрика $recall$ растет;  
- на классе 0 метрика $precision$ растет, метрика $recall$ уменьшается.  

При фокусировке модели на классе 0 (более высокий $threshold$):  
- на классе 1 метрика $precision$ увеличивается, метрика $recall$ уменьшается;  
- на классе 0 метрика $precision$ уменьшается, метрика $recall$ растет.  

Обьсняется тем, что в условия дисбаланса модель научилась хорошо определять класс 0 и плохо класс 1.  

При фокусировке модели на классе 1 (более низкий $threshold$), все больше обьектов отнесены к классу 1.   
Так как $precision$ класса 1 уменьшается, значит в области класса 1 стало больше обектов класса 0,  
что явлется закономерным для "идеальной" модели.  
Однако при этом, $recall$ класса 1 растет, значит в добавленных обьектах также присутвует класс 1.  
Это как раз указывает на то, что модель не нучилась находить среди класса 0 класс 1. В области с     
высокой вероятностью класса 0 (более низкий $threshold$), находятся обьекты класса 1.  

Фокусировка на классе 0 (более высокий $threshold$), заставляет модель все больше обьектов из области класса 1  
относить к классу 0. При этом увеличение метрики $precision$ класса 1 говорит о том, что в области класса 1     
становится меньше 0, а значит модель умеет среди класса 1, находить класс 0.  
После  $threshold=0.83$ модель впринципе теряет способность отличать класс 1. У модели нет обьектов класса 1   
в которых она "уверена" на 95+%.

Обратим внимание на $threshold=0.55$, в этой точке находится баланс межу метриками качества модели:
$$precision_1 = recall_1 = \text{f1-score}_1 = 0.111$$
$$precision_0 = recall_0 = \text{f1-score}_0 = 0.966$$

Для модели с лучшим качеством эти две точки должны находится максимально близко к друг другу и иметь   
высокое значение метрик $precision_1$ и $recall_1$.  
Для класса 0 это условие выполнется, что также говорит о том, что модель научилась искать этот класс.    
Для класса 1 это условие не выполнется, что также говорит о том, что модель плохо ищет класс 1.

##### <span style="color:MediumSlateBlue">Построение ROC-кривой выбранной модели на тестовом наборе</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'credit'
count_features = dict_spaces[feature_space]

In [ ]:
# подгружаем данные для тестирования
X_train = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas().to_numpy()[:,list_n_last_features]
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
X_valid = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_valid').to_pandas().to_numpy()[:,list_n_last_features]
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()
X_test = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_test').to_pandas().to_numpy()[:,list_n_last_features]
y_test = pd.read_csv('target/target_test.csv')['flag'].to_numpy()

In [ ]:
print("Размеры обучающего набора",X_train.shape,y_train.shape)
print("Размеры валидационного набора",X_valid.shape,y_valid.shape)
print("Размеры тестового набора",X_test.shape,y_test.shape)
# time: 1m 43s

In [ ]:
# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_gb_pred_proba = gradient_boosting.predict_proba(X_train)[:,1]
y_valid_gb_pred_proba = gradient_boosting.predict_proba(X_valid)[:,1]
y_test_gb_pred_proba = gradient_boosting.predict_proba(X_test)[:,1]

# Делаем предсказание для валидационной выборки
y_test_gb_pred = gradient_boosting.predict(X_test)

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_gb_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_gb_pred_proba),3))
print('ROC AUC на тестовом наборе', round(metrics.roc_auc_score(y_test, y_test_gb_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_test,y_test_gb_pred,zero_division=0))

In [ ]:
# Для расчета значений TPR и FPR используем функцию roc_curve
fpr_train, tpr_train, _ = metrics.roc_curve(y_train, y_train_gb_pred_proba)
fpr_valid, tpr_valid, _ = metrics.roc_curve(y_valid, y_valid_gb_pred_proba)
fpr_test, tpr_test, _ = metrics.roc_curve(y_test, y_test_gb_pred_proba)

# рассчитаем площадь под кривой
auc_train = round(metrics.roc_auc_score(y_train, y_train_gb_pred_proba),3)
auc_valid = round(metrics.roc_auc_score(y_valid, y_valid_gb_pred_proba),3)
auc_test = round(metrics.roc_auc_score(y_test, y_test_gb_pred_proba),3)

trace_train = go.Scatter(x=fpr_train, y=tpr_train, mode='lines', name='AUC_train = %0.2f' % auc_train,
                   line=dict(color='red', width=2),
                   hovertemplate='train<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_train])

trace_valid = go.Scatter(x=fpr_valid, y=tpr_valid, mode='lines', name='AUC_valid = %0.2f' % auc_valid,
                   line=dict(color='green', width=2),
                   hovertemplate='valid<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_valid])

trace_test = go.Scatter(x=fpr_test, y=tpr_test, mode='lines', name='AUC_test = %0.2f' % auc_test,
                   line=dict(color='darkorange', width=2),
                   hovertemplate='test<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_test])


reference_line = go.Scatter(x=[0,1], y=[0,1], mode='lines', name='Reference Line',
                            line=dict(color='navy', width=2, dash='dash'))

fig_roc = go.Figure(data=[trace_train,trace_valid,trace_test,reference_line])

fig_roc.update_layout(
    title ={
        'text':'ROC-кривая', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 1350,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate')

fig_roc.update_xaxes(showspikes=True)
fig_roc.update_yaxes(showspikes=True)


fig_roc.show()

<span style="color:Blue">

Вывод по ROC-кривой:
1. Модель выдает сталибильное качество на валидационном и тестовом наборах, а значит  
не переобучена - разброс ($variance$) не большой. Хотя, качество модели на обучающем наборе,  
заметно лучше.
2. Смещение ($bias$) модели чуть лучше, чем у модели $Random$ $Forest$ $Classifier$,  
обученной на тех же данных.  
На тестовом наборе полученно качество $ROC AUC = 0.68$, против $ROC AUC = 0.65$

### <span style="color:RoyalBlue">Построение модели на сбалансированных данных подпространства relative torow</span>

#### <span style="color:MediumBlue">Формирование данных для обучения</span>

Анализ исходных данных, в частности target, показал что представленные данные   
имееют перекос: 96% клиентов у которых не случился дефолт (flag = 0),    
против 4 % - для которых произошел дефолт (flag = 1).  
С помощью функции class_1_percent_samples, сформируем сбаланисрованную выборку  
для обучения моделей.
За основу сбалансировных данных возьмем данные клиентов с дефолтом  
и к ним будем добирать необходимое количество клиентов до сбалансированности. 


Функция class_1_percent_samples принемает две переменные:
- target_data - массив с id и целевой переменной
- class_1_percent - процент класса 1 в результирующем массиве

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'relative'
count_features = dict_spaces[feature_space]

In [ ]:
# сформируем данные для анализа сбалансированности обучающих данных
X_train_pd = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas()
y_train_pd = pd.read_csv('target/target_train.csv')

In [ ]:
# поготовим данные y_train_pd к работе с функцией class_1_percent_samples
y_train_pd.set_index('id',drop=False,inplace=True)
y_train_pd.head(4)

In [ ]:
# взглянем на сблансиорованность данных в y_train
y_train_pd['flag'].value_counts(normalize=True)

In [ ]:
# сбалансируем данные в y_train_pd
list_c1_percent_id = class_1_percent_samples(y_train_pd,0.5)
# list_c1_percent_features - список 'id' из y_train_pd соотвествующих  
# заданному проценту класса 1
len(list_c1_percent_id)

In [ ]:
# проверим сбалансированность полученного y_train
y_train_pd.loc[list_c1_percent_id]['flag'].value_counts(normalize=True)

*transform_data_torow*, показываеются последние N операций клиента.  
В данных в *date_torow*, собраны последние 25 операций клиентов.

Предварительный анализ показывает, что медиальное значение количества клиентских  
операций равно 7. Поэтому, для начала, будем показывать моделе $n_{last} = 7$   
последних клиентских операций.

Для извлечения из *transform_data_torow_25* необходимого количества последних клиенских  
операций ($n_{last}$) используем функцию features_from_transform_data_torow.

Функция features_from_transform_data_torow примает следующие аргументы:
- $n_{last} = 7$ - необходимое количество последних клиентских операций; 
- $n_{groups} = 8$ - количество признаков в $date$ $features$;
- $N_{last} = 25$ - количество $n_{last}$ операций, показанных в transform_data_torow_25


In [ ]:
# с помощью функции features_from_transform_data_torow извлечем  
# из данных transform_data_torow_25
# признаки сооствествующие 7 последним клиенским операциям
list_n_last_features = features_from_transform_data_torow(7,count_features,25)

In [ ]:
# подготовим данные для обучения
X_train_balanced = X_train_pd.loc[list_c1_percent_id]
y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag']
# сформируем данные для проверики модели
X_train = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas().to_numpy()
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
X_valid = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_valid').to_pandas().to_numpy()
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()

In [ ]:
# посмотрим на структуру данных в X_train_balanced
X_train_balanced.head(2)

In [ ]:
# посмотрим на структуру данных в y_train_balanced
y_train_balanced.head(2)

In [ ]:
# проверим что выборки действительно совпадают по id
X_train_balanced.index.to_list() == y_train_balanced.index.to_list()

#### <span style="color:MediumBlue">Построение модели Logistic Regression</span>

##### <span style="color:MediumSlateBlue">Baseline обучение модели LogisticRegression</span>

In [ ]:
# обучаем модель логистической регрессии
logistic_regression = linear_model.LogisticRegression(random_state=42, max_iter=10000)
logistic_regression.fit(np.array(X_train_balanced)[:,list_n_last_features],np.array(y_train_balanced))

# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_LR_pred_proba = logistic_regression.predict_proba(X_train[:,list_n_last_features])[:,1]
y_valid_LR_pred_proba = logistic_regression.predict_proba(X_valid[:,list_n_last_features])[:,1]

# Делаем предсказание для валидационной выборки
y_valid_LR_pred = logistic_regression.predict(X_valid[:,list_n_last_features])

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_LR_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_LR_pred_proba),3))

print('Основные метрики на тестовом наборе:')
print(metrics.classification_report(y_valid,y_valid_LR_pred))

# time: 1m 12s

##### <span style="color:MediumSlateBlue">Анализ влияния scaler преобразования на качество и скорость схождения модели</span>

In [ ]:
# формируем список из необходимых scaler
scalers = [MinMaxScaler(),RobustScaler() ,StandardScaler()]

# сформируем списко для заполнения данными результатов анализа
scalers_data = []

# установим ограничение на время обучения модели в 600 секунд (10 минут)
time_out = 600

for scaler in scalers:
        # для того чтобы следить за выполнением цикла введем индикатор
        print('current scaler: ', scaler)

         # сформируем список для заполнения данными текушего scaler
        current_scaler_data = [scaler]
        
        # выполним scaler преобразование
        X_train_s_balanced = scaler.fit_transform(np.array(X_train_balanced)[:,list_n_last_features])
        X_train_s = scaler.transform(X_train[:,list_n_last_features])
        X_valid_s = scaler.transform(X_valid[:,list_n_last_features])

        try:
                # для модуля func_time_out упакуем обучение модели в функцию try_func
                def try_func():
                        logistic_regression = linear_model.LogisticRegression(random_state=42, max_iter=10000)
                        return logistic_regression.fit(X_train_s_balanced,y_train_balanced)
                    
                # фиксируем время начала
                start_time = time.time()
                
                # запускаем обучение модели с ограничением по времени
                logistic_regression = func_timeout.func_timeout(time_out, try_func)
                
                # фиксируем время окончания обучения
                end_time = time.time()
                
                # запишем время обучения модели
                current_scaler_data.append(round(end_time-start_time))

                # Делаем предсказание для обучающей и валидационной выборки
                y_valid_lr_pred = logistic_regression.predict(X_valid_s)
                
                # для метрик ROC AUC делаем предсказание модели для обучающей и валидационной выборки  
                # в виде вероятности принадлежности к классу 1 
                y_train_lr_pred_proba = logistic_regression.predict_proba(X_train_s)[:,1]
                y_valid_lr_pred_proba = logistic_regression.predict_proba(X_valid_s)[:,1]

                #Считаем метрики ROC AUC yна обуающем и валидационном наборе и добавляем их в спискок
                current_scaler_data.append(round(metrics.roc_auc_score(y_train, y_train_lr_pred_proba),3))
                current_scaler_data.append(round(metrics.roc_auc_score(y_valid, y_valid_lr_pred_proba),3))

                #Считаем метрики для класса 0 на валидационном наборе и добавляем их в список
                current_scaler_data.append(round(metrics.recall_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))
                current_scaler_data.append(round(metrics.precision_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))


                #Считаем метрики для класса 1 на валидационном набопе и добавляем их в список
                current_scaler_data.append(round(metrics.recall_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))
                current_scaler_data.append(round(metrics.precision_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))

                # добавляем полученные данные в список scalers_data
                scalers_data.append(current_scaler_data)

                # чтобы подстраховаться на случай ошибки : "The Kernel crashed while executing code"
                # сохраним в локальную папку удачную итерацию
                # для этого из полученных данных сформируем dataframe
                # из полученных данных сформируем dataframe
                scaler_pd =pd.DataFrame(
                        columns=['scaler','time_fit',
                                 'roc_train','roc_valid', 
                                 'recall_0','precision_0',
                                 'recall_1','precision_1'],
                        data = scalers_data)
                # сохраняем dataframe
                scaler_pd.to_csv('data_recovery/scaler_pd.csv',index=False)
               
                # очистим output от прошедшей итерации
                clear_output()

        except func_timeout.FunctionTimedOut:
                # если время обучения привышает лимит
                # запишем в current_scaler_data соотвествующую данные
                current_scaler_data = [scaler,'>30m',0,0,0,0,0,0]
                # добавляем полученные данные в список scalers_data
                scalers_data.append(current_scaler_data)

                # чтобы подстраховаться на случай ошибки : "The Kernel crashed while executing code"
                # сохраним в локальную папку удачную итерацию
                # для этого из полученных данных сформируем dataframe
                # из полученных данных сформируем dataframe
                scaler_pd =pd.DataFrame(
                        columns=['scaler','time_fit',
                                 'roc_train','roc_valid', 
                                 'recall_0','precision_0',
                                 'recall_1','precision_1'],
                        data = scalers_data)
                # сохраняем dataframe
                scaler_pd.to_csv('data_recovery/scaler_pd.csv',index=False)
                
                # очистим output от прошедшей итерации
                clear_output()
                # переходим к следующему scaler
                pass
# очистим output
clear_output()

# сформируем данные соотвествующие лучщему ROC AUC на валидацинном наборе
best_roc_valid_data = scaler_pd[scaler_pd['roc_valid']==scaler_pd['roc_valid'].max()]

# выведем лучший результат на валидационном наборе
print('result:')
print('best scaler on valid:',best_roc_valid_data.iloc[0]['scaler'])
print('ROC AUC on train:', best_roc_valid_data.iloc[0]['roc_train'])
print('ROC AUC on valid:', best_roc_valid_data.iloc[0]['roc_valid'])
print('Time fit for best scaler:', best_roc_valid_data.iloc[0]['time_fit'],' seconds')

# выведем полученный dataframe
scaler_pd

# time: 

##### <span style="color:MediumSlateBlue"> Анализ влияния solver оптимизатора на качество и скорость обучения модели</span>

Проверим качество и скорость сходимости модели при разных solver  
Проверку осуществим через цикл  
Чтобы модель не сходилась бесконечно с помощью модуля func_timeout  
поставим ограничение времени схождения в 10 минут

In [ ]:
# подготовим данные для обучения модели

# обьявим scaler
scaler = StandardScaler()
# выполним scaler преобразование
X_train_s_balanced = scaler.fit_transform(np.array(X_train_balanced)[:,list_n_last_features])
X_train_s = scaler.transform(X_train[:,list_n_last_features])
X_valid_s = scaler.transform(X_valid[:,list_n_last_features])

# time: 10s

In [ ]:
# Зададим список solvers
solvers_list = ['lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga']

# сформируем список для заполнения
solvers_data = []

# установим ограничение на время обучения модели в 300 секунд (5 минут)
time_out = 300

for solver in solvers_list:
        # для того чтобы следить за выполнением цикла введем индикатор
        print('current solver: ', solver)

         # сформируем список для заполнения данными текушего solver
        current_solver_data = [solver]
        
        try:
                # для модуля func_time_out упакуем обучение модели в функцию try_func
                def try_func():
                        logistic_regression = linear_model.LogisticRegression(solver= solver,
                                                                  random_state=42, 
                                                                  max_iter=10000)
                        return logistic_regression.fit(X_train_s_balanced,y_train_balanced)
                    
                # фиксируем время начала
                start_time = time.time()
                
                # запускаем обучение модели с ограничением по времени
                logistic_regression = func_timeout.func_timeout(time_out, try_func)
                
                # фиксируем время окончания обучения
                end_time = time.time()
                
                # запишем время обучения модели
                current_solver_data.append(round(end_time-start_time))

                # Делаем предсказание для обучающей и валидационной выборки
                y_valid_lr_pred = logistic_regression.predict(X_valid_s)
                
                # для метрик ROC AUC делаем предсказание модели для обучающей и валидационной выборки  
                # в виде вероятности принадлежности к классу 1 
                y_train_lr_pred_proba = logistic_regression.predict_proba(X_train_s)[:,1]
                y_valid_lr_pred_proba = logistic_regression.predict_proba(X_valid_s)[:,1]

                #Считаем метрики ROC AUC yна обуающем и валидационном наборе и добавляем их в спискок
                current_solver_data.append(round(metrics.roc_auc_score(y_train, y_train_lr_pred_proba),3))
                current_solver_data.append(round(metrics.roc_auc_score(y_valid, y_valid_lr_pred_proba),3))

                #Считаем метрики для класса 0 на валидационном наборе и добавляем их в список
                current_solver_data.append(round(metrics.recall_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))
                current_solver_data.append(round(metrics.precision_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))


                #Считаем метрики для класса 1 на валидационном набопе и добавляем их в список
                current_solver_data.append(round(metrics.recall_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))
                current_solver_data.append(round(metrics.precision_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))

                # запишем данные полученные на одной итерации
                solvers_data.append(current_solver_data)

                # чтобы подстраховаться на случай ошибки : "The Kernel crashed while executing code"
                # сохраним в локальную папку удачную итерацию
                # для этого из полученных данных сформируем dataframe
                # из полученных данных сформируем dataframe
                solvers_pd =pd.DataFrame(
                        columns=['solver','time_fit',
                                 'roc_train','roc_valid', 
                                 'recall_0','precision_0',
                                 'recall_1','precision_1'],
                        data = solvers_data)
                # сохраняем dataframe
                solvers_pd.to_csv('data_recovery/solvers_pd.csv',index=False)
               
                # очистим output от прошедшей итерации
                clear_output()

        except func_timeout.FunctionTimedOut:
                # если время обучения привышает лимит
                # запишем в current_solver_data соотвествующую данные
                current_solver_data = [solver,'>10m',0,0,0,0,0,0]
                # добавляем полученные данные в список solvers_data
                solvers_data.append(current_solver_data)

                # чтобы подстраховаться на случай ошибки : "The Kernel crashed while executing code"
                # сохраним в локальную папку удачную итерацию
                # для этого из полученных данных сформируем dataframe
                # из полученных данных сформируем dataframe
                solvers_pd =pd.DataFrame(
                        columns=['solver','time_fit',
                                 'roc_train','roc_valid', 
                                 'recall_0','precision_0',
                                 'recall_1','precision_1'],
                        data = solvers_data)
                # сохраняем dataframe
                solvers_pd.to_csv('data_recovery/solvers_pd.csv',index=False)
                
                # очистим output от прошедшей итерации
                clear_output()
                # переходим к следующему solver
                pass
# очистим output
clear_output()

# сформируем данные соотвествующие лучщему ROC AUC на валидацинном наборе
best_roc_valid_data = solvers_pd[solvers_pd['roc_valid']==solvers_pd['roc_valid'].max()]

# выведем лучший результат на валидационном наборе
print('result:')
print('best solver on valid:',best_roc_valid_data.iloc[0]['solver'])
print('ROC AUC on train:', best_roc_valid_data.iloc[0]['roc_train'])
print('ROC AUC on valid:', best_roc_valid_data.iloc[0]['roc_valid'])
print('Time fit for best solver:', best_roc_valid_data.iloc[0]['time_fit'],' seconds')

# выведем полученный dataframe
solvers_pd

# time: 11m 35s

##### <span style="color:MediumSlateBlue">Подбор гиперпараметров модели</span>

In [ ]:
# для выбора sceler преобразвания на понадобится словарь
dict_scalers ={
    'MinMaxScaler':MinMaxScaler(),
    'RobustScaler':RobustScaler(),
    'StandardScaler':StandardScaler(),
}
# определим пространство признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'relative'
count_features = dict_spaces[feature_space]

# настроим оптимизацию гипер параметров
def optuna_lg(trial):
  # задаем пространства поиска гиперпараметров
  C = trial.suggest_float('C',0.01,1)
  solver = trial.suggest_categorical('solver', ['lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky'])
  class_0_weight = trial.suggest_float('class_0_weight',0.01,1)
  class_1_weight = trial.suggest_float('class_1_weight',0.01,1)
  n_last = trial.suggest_int('n_last', 10, 25,step = 1)
  class_1_percent = trial.suggest_float('class_1_percent',0.01,1)
  scaler = trial.suggest_categorical('scaler', ['MinMaxScaler', 'RobustScaler','StandardScaler'])
  random_state = trial.suggest_int('random_state', 1, 1000000)

  # создаем модель
  optuna_log_reg = linear_model.LogisticRegression(
      solver=solver,
      C=C,
      class_weight={0:class_0_weight,1:class_1_weight},
      random_state=random_state,
      max_iter=10000)

  # с помощью функции features_from_transform_data_torow извлечем  
  # из данных transform_data_torow
  # признаки сооствествующие n_last последним клиенским операциям
  list_n_last_features = features_from_transform_data_torow(n_last,count_features,25)

  # обьявим scaler
  scaler = dict_scalers[scaler]

  # загружаем обучующие наборы
  X_train_pd = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas()
  y_train_pd = pd.read_csv('target/target_train.csv')

  # поготовим данные y_train_pd к работе с функцией class_1_percent_samples
  y_train_pd.set_index('id',drop=False,inplace=True)

  # подготовим данные для обучения модели
  # с помощью функции class_1_percent_samples зададим долю 
  # класса 1
  list_c1_percent_id = class_1_percent_samples(y_train_pd,class_1_percent,random_state=random_state)[:1000000]

  # сформируем данные для обучения базовой модели
  X_train_s_balanced = scaler.fit_transform(np.array(X_train_pd.loc[list_c1_percent_id])[:,list_n_last_features])
  y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

  # освободим память от "тяжелых" и ненужных файлов
  del X_train_pd, y_train_pd
  gc.collect()

  # я не воспользовался параметром timeout метода optimize потому, что  
  # optimize отставливает поиск параметров, если время обучения 
  # превышает timeout. Мне нужно чтобы поиск параметров продолжился.
  try:
    # для модуля func_time_out упакуем обучение модели в функцию try_func
    def try_func():
            return optuna_log_reg.fit(X_train_s_balanced, y_train_balanced)
    # обучаем модель с ограничением по времени 10 минут (600 секунд)
    optuna_log_reg = func_timeout.func_timeout(600, try_func)

    # удаляем крупные файлы чтобы высвободить память 
    del X_train_s_balanced,y_train_balanced
    gc.collect()

    # подгружаем данные для тестирования
    X_train = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas().to_numpy()[:,list_n_last_features]
    y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
    X_valid = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_valid').to_pandas().to_numpy()[:,list_n_last_features]
    y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()

    # сформируем данные для проверки модели
    X_train_s = scaler.transform(X_train)
    X_valid_s = scaler.transform(X_valid)

    # удаляем крупные файлы чтобы высвободить память 
    del X_train, X_valid
    gc.collect()

    # делаем предсказание на обучающем и валидационном наборе и считаем метрики
    roc_train = metrics.roc_auc_score(y_train, optuna_log_reg.predict_proba(X_train_s)[:,1])
    roc_valid = metrics.roc_auc_score(y_valid, optuna_log_reg.predict_proba(X_valid_s)[:,1])
    f1_score = metrics.f1_score(y_valid, optuna_log_reg.predict(X_valid_s))
    recall_1 = metrics.recall_score(y_valid, optuna_log_reg.predict(X_valid_s),average=None,zero_division=0)[1]
    precision_1 = metrics.precision_score(y_valid, optuna_log_reg.predict(X_valid_s),average=None,zero_division=0)[1]

    del X_train_s, X_valid_s, y_train, y_valid
    gc.collect()

  except func_timeout.FunctionTimedOut:
    # освободим память от "тяжелых" и ненужных файлов
    del X_train_s_balanced, y_train_balanced
    gc.collect()

    # обьявим метрики пустыми значениями
    roc_train = 0
    roc_valid = 0
    f1_score = 0
    recall_1 = 0
    precision_1 = 0
    pass

  return roc_train, roc_valid, f1_score, recall_1, precision_1

In [ ]:
# cоздаем объект исследования
# чтобы модель не переобучалась минимизируем roc_train 
# и максимизируем roc_valid
optuna_study_lg = optuna.create_study(study_name='LogisticRegression_'+feature_space+'_torow', 
                               directions=['maximize','maximize','maximize','maximize','maximize'], 
                               sampler=optuna.samplers.TPESampler(),
                               pruner='Hyperband',
                               storage='sqlite:///optuna_studies.db',
                               load_if_exists=True)
# на случай удаления 
# optuna.delete_study(study_name='LogisticRegression_'+feature_space+'_torow', storage='sqlite:///optuna_studies.db')

In [ ]:
# ищем лучшую комбинацию гиперпараметров n_trials раз
optuna_study_lg.optimize(optuna_lg, n_trials=300)

In [ ]:
# из полученного результат соврмируем Data Frame
optuna_study_lg_pd = optuna_study_lg.trials_dataframe().dropna()
# переименуем столбы
optuna_study_lg_pd.rename(columns={
    'values_0': 'roc_train',
    'values_1': 'roc_valid',
    'values_2': 'f1_score',
    'values_3': 'recall_1',
    'values_4': 'precision_1'
},inplace=True)
# Выделим время потраченное на обучение модели
optuna_study_lg_pd.head(5)

In [ ]:
# покаждем статистику обучения
print('Максимальное значение метрики ROC AUC на валидационном наборе:',round(optuna_study_lg_pd['roc_valid'].max(),3))
print('Среднее значение метрики ROC AUC на валидационном наборе:',round(optuna_study_lg_pd['roc_valid'].mean(),3))
print('Максимальное значение метрики f1_score на валидационном наборе:',round(optuna_study_lg_pd['f1_score'].max(),3))
print('Среднее значение метрики f1_score на валидационном наборе:',round(optuna_study_lg_pd['f1_score'].mean(),3))
print('Максимальное значение метрики recall_1 на валидационном наборе:',round(optuna_study_lg_pd['recall_1'].max(),3))
print('Среднее значение метрики recall_1 на валидационном наборе:',round(optuna_study_lg_pd['recall_1'].mean(),3))
print('Максимальное значение метрики precision_1 на валидационном наборе:',round(optuna_study_lg_pd['precision_1'].max(),3))
print('Среднее значение метрики precision_1 на валидационном наборе:',round(optuna_study_lg_pd['precision_1'].mean(),3))

In [ ]:
# построим график важности гиперпарметров
optuna.visualization.plot_param_importances(optuna_study_lg, target_name='Score')

In [ ]:
# Построим зависимость метрики precision_1

fig= px.scatter(
    data_frame=optuna_study_lg_pd,
    x='precision_1', #ось абсцисс
    y=['recall_1','f1_score'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision класса 1 от recall класса 1', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision класса 1',
    yaxis_title='Величина метрики recall класса 1 и f1-score')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики precision_1 от гипер параметров

fig= px.scatter(
    data_frame=optuna_study_lg_pd,
    x='precision_1', #ось абсцисс
    y=['params_C','params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight', 'params_n_last',
    ], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision класса 1 от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision класса 1',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики f1_score от гипер параметров

fig= px.scatter(
    data_frame=optuna_study_lg_pd,
    x='f1_score', #ось абсцисс
    y=['params_C','params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight', 'params_n_last',
    ], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость f1-score от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики f1-score',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики roc_valid от гипер параметров

fig= px.scatter(
    data_frame=optuna_study_lg_pd,
    x='roc_valid', #ось абсцисс
    y=['params_C','params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight', 'params_n_last',
    ], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость roc valid от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики roc valid',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Визуализируем метрики полученные при optuna оптимизации
fig = px.scatter(
    data_frame=optuna_study_lg_pd[(optuna_study_lg_pd['roc_valid']>0.998*optuna_study_lg_pd['roc_valid'].max())],
    x='number', #ось абсцисс
    y=['roc_train','roc_valid','recall_1','precision_1'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость качества модели от trials optuna', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='trials optuna',
    yaxis_title='Величина метрики')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

<span style="color:Blue">

Вывод:  

Их всех выбираем точку $number =195$.   
В этой точке одно из самых больших значений $ROC AUC$.  
Посмотрим каким значениям гиперпараметров соотвествует выбранная точка.

In [ ]:
# определим номер лучшге варианта
best_optuna_number = 195

# сформируем словарь лучших гипирпарметров RandomForestClassifier
best_param_lr_torow = {
    'C' : round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_C'].iloc[0],3),
    'class_weight' : {0:round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_class_0_weight'].iloc[0],3),
                      1:round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_class_1_weight'].iloc[0],3)},
    }
# создадим перменные
best_n_last = int(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_n_last'].iloc[0])
best_scaler = optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_scaler'].iloc[0]
best_class_1_percent = optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_class_1_percent'].iloc[0]
best_random_state = int(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_random_state'].iloc[0])
# Выведем принятые наилучшие праметры
print('best C:',best_param_lr_torow['C'])
print('best class 0 weight:',best_param_lr_torow['class_weight'][0])
print('best class 1 weight:',best_param_lr_torow['class_weight'][1])

print('best class 1 percent:',round(best_class_1_percent,3))
print('best scaler:',best_scaler)
print('best n_last:',round(best_n_last,3))
print('best best random state:',best_random_state)
print('time for best train:',round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['duration'].iloc[0].seconds/60),'minutes')

print()
print('ROC AUC на обучающем наборе:', round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['roc_train'].iloc[0],3))
print('ROC AUC на валидационном наборе:', round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['roc_valid'].iloc[0],3))
print('precision класса 1:', round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['precision_1'].iloc[0],3))
print('recall класса 1:', round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['recall_1'].iloc[0],3))

##### <span style="color:MediumSlateBlue">Обучение модели с лучшими параметрами</span>

In [ ]:
# для выбора sceler преобразвания на понадобится словарь
dict_scalers ={
    'MinMaxScaler':MinMaxScaler(),
    'RobustScaler':RobustScaler(),
    'StandardScaler':StandardScaler(),
}

# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'relative'
count_features = dict_spaces[feature_space]

# с помощью функции features_from_transform_data_torow извлечем  
# из данных transform_data_torow_25
# признаки сооствествующие n_last последним клиенским операциям
list_n_last_features = features_from_transform_data_torow(best_n_last,count_features,25)

# обьявим scaler
scaler = dict_scalers[best_scaler]

# загружаем обучующие наборы
X_train_pd = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas()
y_train_pd = pd.read_csv('target/target_train.csv')

# поготовим данные y_train_pd к работе с функцией class_1_percent_samples
y_train_pd.set_index('id',drop=False,inplace=True)

# подготовим данные для обучения модели
# с помощью функции class_1_percent_samples зададим долю 
# класса 1
list_c1_percent_id = class_1_percent_samples(y_train_pd,best_class_1_percent,random_state=best_random_state)[:1000000]

# сформируем данные для обучения базовой модели
X_train_s_balanced = scaler.fit_transform(np.array(X_train_pd.loc[list_c1_percent_id])[:,list_n_last_features])
y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

# освободим память от "тяжелых" и ненужных файлов
del X_train_pd, y_train_pd
gc.collect()


# обучаем модель LogisticRegression с наилучшеми параметрами
logistic_regression = linear_model.LogisticRegression(
        **best_param_lr_torow,
        random_state=best_random_state,
        max_iter=10000)
logistic_regression.fit(X_train_s_balanced,y_train_balanced)

# удаляем крупные файлы чтобы высвободить память 
del X_train_s_balanced,y_train_balanced
gc.collect()

# подгружаем данные для тестирования
X_train = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas().to_numpy()[:,list_n_last_features]
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
X_valid = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_valid').to_pandas().to_numpy()[:,list_n_last_features]
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()
    
# сформируем данные для проверки модели
X_train_s = scaler.transform(X_train)
X_valid_s = scaler.transform(X_valid)

# удаляем крупные файлы чтобы высвободить память 
del X_train, X_valid
gc.collect()

# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_lr_pred_proba = logistic_regression.predict_proba(X_train_s)[:,1]
y_valid_lr_pred_proba = logistic_regression.predict_proba(X_valid_s)[:,1]

# Делаем предсказание для валидационной выборки
y_valid_lr_pred = logistic_regression.predict(X_valid_s)

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_lr_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_lr_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_valid,y_valid_lr_pred,zero_division=0))

# time: 2m 40s

##### <span style="color:MediumSlateBlue">Анализ влияния порога классификации на качество модели</span>

In [ ]:
# чтобы проше обращаться к каждому обьекту в данных
# преобразуем y_valid_LR_pred к Series типу
y_valid_lr_pred_proba = pd.Series(y_valid_lr_pred_proba)

# формируем список из необходимых значений thresholds
thresholds = np.arange(0.01, 1, 0.01)

# сформируем списко для заполнения данными результатов анализа
threshold_data = []


for threshold in thresholds:
        # сформируем список для заполнения данными текушего threshold
        threshold_current_data = [threshold]

        #клиентов, для которых вероятность дефолта > threshold, относим к классу 1
        #в противном случае — к классу 0
        y_valid_lr_pred = y_valid_lr_pred_proba.apply(lambda x: 1 if x>threshold else 0)


        #Считаем метрики для класса 0 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.precision_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.f1_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))

        #Считаем метрики для класса 1 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.precision_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.f1_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))

        # добавляем полученные данные в список threshold_data
        threshold_data.append(threshold_current_data)

        # из полученных данных сформируем dataframe
        thresholds_pd =pd.DataFrame(
        columns=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'],
        data = threshold_data)

# time: 17s

In [ ]:
# Визуализируем метрики на валидационном наборе при различных threshold
fig_threshold = px.line(
    data_frame=thresholds_pd,
    x='threshold', #ось абсцисс
    y=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'], #ось ординат
)
fig_threshold.update_layout(
    title ={
        'text':'Зависимость качества модели от threshold', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Порог классификации threshold',
    yaxis_title='Величина метрики')
fig_threshold.update_xaxes(showspikes=True)
fig_threshold.update_yaxes(showspikes=True)

fig_threshold.show()

Метрика $precision$ показывает на сколько сильно ошиблась модель при определиние класса.  
Чем выше метрика, тем меньше ошиблась модель. 

$$precision = \frac{TP}{TP+FP}$$

Метрика $recall$ показывает способность модели найти класс.  
Чем выше метрика, тем выше способность модели находить класс.  

$$recall = \frac{TP}{TP+FN}$$

Метрика $f1-score$ показывает баланс между $precision$ и $recall$ 

$threshold$- порог класификации модели.  
Все объекты для которых $у_{\text{proba pred}} < threshold$, модель относит к класс 0.  
Соотвественно, если $у_{\text{proba pred}}  > threshold$ модель относит обьект к классу 1.  
Для идельаной модели $threshold$, является границей между областями класса 0 и класса 1.  

<span style="color:Blue">

Выводы по результам анализа:  

Для улучшения качества модели по метрике $ROC AUC$, нет смысла сдвигать порог   
классификации. Но зависимость метрик качества модели от $threshold$, может рассказать о  
спосбоности модели искать класс 1 и класс 0.

При фокусировке модели на классе 1 (более низкий $threshold$):  
- на классе 1 метрика $precision$ уменьшается, метрика $recall$ растет;  
- на классе 0 метрика $precision$ растет, метрика $recall$ уменьшается.  

При фокусировке модели на классе 0 (более высокий $threshold$):  
- на классе 1 метрика $precision$ увеличивается, метрика $recall$ уменьшается;  
- на классе 0 метрика $precision$ уменьшается, метрика $recall$ растет.  

Обьсняется тем, что в условия дисбаланса модель научилась хорошо определять класс 0 и плохо класс 1.  

При фокусировке модели на классе 1 (более низкий $threshold$), все больше обьектов отнесены к классу 1.   
Так как $precision$ класса 1 уменьшается, значит в области класса 1 стало больше обектов класса 0,  
что явлется закономерным для "идеальной" модели.  
Однако при этом, $recall$ класса 1 растет, значит в добавленных обьектах также присутвует класс 1.  
Это как раз указывает на то, что модель не нучилась находить среди класса 0 класс 1. В области с     
высокой вероятностью класса 0 (более низкий $threshold$), находятся обьекты класса 1.  

Фокусировка на классе 0 (более высокий $threshold$), заставляет модель все больше обьектов из области класса 1  
относить к классу 0. При этом увеличение метрики $precision$ класса 1 говорит о том, что в области класса 1     
становится меньше 0, а значит модель умеет среди класса 1, находить класс 0.  


Обратим внимание на $threshold=0.53$, в этой точке находится баланс межу метриками качества модели:
$$precision_1 = recall_1 = \text{f1-score}_1 = 0.068$$
$$precision_0 = recall_0 = \text{f1-score}_0 = 0.965$$

Для модели с лучшим качеством эти две точки должны находится максимально близко к друг другу и иметь  
высокое значение метрик $precision_1$ и $recall_1$.  
Для класса 0 это условие выполнется, что также говорит о том, что модель научилась искать этот класс.   
Для класса 1 это условие не выполнется, что также говорит о том, что модель плохо ищет класс 1.

##### <span style="color:MediumSlateBlue">Построение ROC-кривой выбранной модели на тестовом наборе</span>

In [ ]:
# определим пространство признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'relative'
count_features = dict_spaces[feature_space]

In [ ]:
# подгружаем данные для тестирования
X_train = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas().to_numpy()[:,list_n_last_features]
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
X_valid = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_valid').to_pandas().to_numpy()[:,list_n_last_features]
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()
X_test = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_test').to_pandas().to_numpy()[:,list_n_last_features]
y_test = pd.read_csv('target/target_test.csv')['flag'].to_numpy()

In [ ]:
# выполним scaler преобразование
X_train_s = scaler.transform(X_train)
X_valid_s = scaler.transform(X_valid)
X_test_s = scaler.transform(X_test)

print("Размеры обучающего набора",X_train_s.shape)
print("Размеры валидационного набора",X_valid_s.shape)
print("Размеры тестового набора",X_test_s.shape)
# time: 1m 43s

In [ ]:
# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_lr_pred_proba = logistic_regression.predict_proba(X_train_s)[:,1]
y_valid_lr_pred_proba = logistic_regression.predict_proba(X_valid_s)[:,1]
y_test_lr_pred_proba = logistic_regression.predict_proba(X_test_s)[:,1]

# Делаем предсказание для валидационной выборки
y_test_lr_pred = logistic_regression.predict(X_test_s)

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_lr_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_lr_pred_proba),3))
print('ROC AUC на тестовом наборе', round(metrics.roc_auc_score(y_test, y_test_lr_pred_proba),3))

print('Основные метрики на тестовом наборе:')
print(metrics.classification_report(y_test,y_test_lr_pred,zero_division=0))

In [ ]:
# Для расчета значений TPR и FPR используем функцию roc_curve
fpr_train, tpr_train, _ = metrics.roc_curve(y_train, y_train_lr_pred_proba)
fpr_valid, tpr_valid, _ = metrics.roc_curve(y_valid, y_valid_lr_pred_proba)
fpr_test, tpr_test, _ = metrics.roc_curve(y_test, y_test_lr_pred_proba)

# рассчитаем площадь под кривой
auc_train = round(metrics.roc_auc_score(y_train, y_train_lr_pred_proba),3)
auc_valid = round(metrics.roc_auc_score(y_valid, y_valid_lr_pred_proba),3)
auc_test = round(metrics.roc_auc_score(y_test, y_test_lr_pred_proba),3)

trace_train = go.Scatter(x=fpr_train, y=tpr_train, mode='lines', name='AUC_train = %0.2f' % auc_train,
                   line=dict(color='red', width=2),
                   hovertemplate='train<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_train])

trace_valid = go.Scatter(x=fpr_valid, y=tpr_valid, mode='lines', name='AUC_valid = %0.2f' % auc_valid,
                   line=dict(color='green', width=2),
                   hovertemplate='valid<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_valid])

trace_test = go.Scatter(x=fpr_test, y=tpr_test, mode='lines', name='AUC_test = %0.2f' % auc_test,
                   line=dict(color='darkorange', width=2),
                   hovertemplate='test<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_test])


reference_line = go.Scatter(x=[0,1], y=[0,1], mode='lines', name='Reference Line',
                            line=dict(color='navy', width=2, dash='dash'))

fig_roc = go.Figure(data=[trace_train,trace_valid,trace_test,reference_line])

fig_roc.update_layout(
    title ={
        'text':'ROC-кривая', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 1350,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate')

fig_roc.update_xaxes(showspikes=True)
fig_roc.update_yaxes(showspikes=True)


fig_roc.show()

<span style="color:Blue">

Вывод по ROC-кривой:
1. Модель выдает сталибильное качество на всех наборах, а значит  
не переобучена - разброс ($variance$) не большой.  
2. Смещение ($bias$) модели относительно большое.  
На тестовом наборе полученно качество $ROC AUC = 0.61$.

#### <span style="color:MediumBlue">Построение модели Random Forest Classifier</span>

##### <span style="color:MediumSlateBlue">Baseline обучение модели RandomForestClassifier</span>

In [ ]:
# обучаем модель логистической регрессии
# чтобы модель пыталась найти связь во всех признаках
random_forest = RandomForestClassifier(random_state=42, n_jobs=-1)
random_forest.fit(np.array(X_train_balanced)[:,list_n_last_features],np.array(y_train_balanced))

# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_rf__pred_proba = random_forest.predict_proba(X_train[:,list_n_last_features])[:,1]
y_valid_rf_pred_proba = random_forest.predict_proba(X_valid[:,list_n_last_features])[:,1]

# Делаем предсказание для валидационной выборки
y_valid_rf_pred = random_forest.predict(X_valid[:,list_n_last_features])

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_rf__pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_rf_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_valid,y_valid_rf_pred))

# time: 2m 5s

##### <span style="color:MediumSlateBlue">Подбор гиперпараметров модели</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'relative'
count_features = dict_spaces[feature_space]

# настроим оптимизацию гипер параметров
def optuna_rf(trial):
  # задаем пространства поиска гиперпараметров
  # задаем пространства поиска гиперпараметров
  n_estimators = trial.suggest_int('n_estimators', 50, 150,step = 5)
  criterion = trial.suggest_categorical('criterion',['gini', 'entropy', 'log_loss'])
  max_depth = trial.suggest_int('max_depth', 9, 20,step = 1)
  min_samples_split = trial.suggest_int('min_samples_split', 5, 15,step = 1)
  min_samples_leaf = trial.suggest_int('min_samples_leaf', 5, 25,step = 1)
  max_features = trial.suggest_float('max_features',0.1,1)
  max_samples = trial.suggest_float('max_samples',0.1,1)
  class_0_weight = trial.suggest_float('class_0_weight',0.01,1)
  class_1_weight = trial.suggest_float('class_1_weight',0.01,1)
  n_last = trial.suggest_int('n_last', 5, 25,step = 1)
  class_1_percent = trial.suggest_float('class_1_percent',0.01,1)
  random_state = trial.suggest_int('random_state', 1, 1000000)


  # создаем модель
  optuna_random_forest = RandomForestClassifier(
      n_estimators = n_estimators, 
      criterion = criterion,
      max_depth = max_depth,
      min_samples_split = min_samples_split,  
      min_samples_leaf = min_samples_leaf,
      max_features=max_features,
      max_samples=max_samples,
      class_weight={0:class_0_weight,1:class_1_weight},
      n_jobs=-1,
      random_state=random_state)

  # с помощью функции features_from_transform_data_torow извлечем  
  # из данных transform_data_torow
  # признаки сооствествующие n_last последним клиенским операциям
  list_n_last_features = features_from_transform_data_torow(n_last,count_features,25)

  # загружаем обучующие наборы
  X_train_pd = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas()
  y_train_pd = pd.read_csv('target/target_train.csv')

  # поготовим данные y_train_pd к работе с функцией class_1_percent_samples
  y_train_pd.set_index('id',drop=False,inplace=True)

  # подготовим данные для обучения модели
  # с помощью функции class_1_percent_samples зададим долю 
  # класса 1
  list_c1_percent_id = class_1_percent_samples(y_train_pd,class_1_percent,random_state=random_state)[:1000000]

  # сформируем данные для обучения базовой модели
  X_train_balanced = np.array(X_train_pd.loc[list_c1_percent_id])[:,list_n_last_features]
  y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

  # освободим память от "тяжелых" и ненужных файлов
  del X_train_pd, y_train_pd
  gc.collect()

  # я не воспользовался параметром timeout метода optimize потому, что  
  # optimize отставливает поиск параметров, если время обучения 
  # превышает timeout. Мне нужно чтобы поиск параметров продолжился.
  try:
    # для модуля func_time_out упакуем обучение модели в функцию try_func
    def try_func():
            return optuna_random_forest.fit(X_train_balanced,y_train_balanced)
    # обучаем модель с ограничением по времени 10 минут (600 секунд)
    optuna_random_forest = func_timeout.func_timeout(600, try_func)

    # удаляем крупные файлы чтобы высвободить память 
    del X_train_balanced, y_train_balanced
    gc.collect()

    # подгружаем данные для тестирования
    X_train = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas().to_numpy()[:,list_n_last_features]
    y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
    X_valid = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_valid').to_pandas().to_numpy()[:,list_n_last_features]
    y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()

    # делаем предсказание на обучающем и валидационном наборе и считаем метрики
    roc_train = metrics.roc_auc_score(y_train, optuna_random_forest.predict_proba(X_train)[:,1])
    roc_valid = metrics.roc_auc_score(y_valid, optuna_random_forest.predict_proba(X_valid)[:,1])
    f1_score = metrics.f1_score(y_valid, optuna_random_forest.predict(X_valid))
    recall_1 = metrics.recall_score(y_valid, optuna_random_forest.predict(X_valid),average=None,zero_division=0)[1]
    precision_1 = metrics.precision_score(y_valid, optuna_random_forest.predict(X_valid),average=None,zero_division=0)[1]

    # удаляем крупные файлы чтобы высвободить память 
    del X_train, X_valid, y_train, y_valid 
    gc.collect()

  except func_timeout.FunctionTimedOut:
    # освободим память от "тяжелых" и ненужных файлов
    del X_train_balanced, y_train_balanced
    gc.collect()

    # обьявим метрики пустыми значениями
    roc_train = 0
    roc_valid = 0
    f1_score = 0
    recall_1 = 0
    precision_1 = 0
    pass

  return roc_train, roc_valid, f1_score, recall_1, precision_1

In [ ]:
# так как Random Forest склонен к переобучению 
# попробуем направить optimize в сторону уменьшения roc_train
# cоздаем объект исследования
optuna_study_rf = optuna.create_study(study_name='RandomForestClassifier_'+feature_space+'_torow', 
                               directions=['minimize','maximize','maximize','maximize','maximize'], 
                               sampler=optuna.samplers.TPESampler(),
                               pruner='Hyperband',
                               storage='sqlite:///optuna_studies.db',
                               load_if_exists=True)
# на случай удаления 
# optuna.delete_study(study_name='RandomForestClassifier_'+feature_space+'_torow', storage='sqlite:///optuna_studies.db')

In [ ]:
# ищем лучшую комбинацию гиперпараметров n_trials раз
optuna_study_rf.optimize(optuna_rf, n_trials=300)

In [ ]:
# из полученного результат соврмируем Data Frame
optuna_study_rf_pd = optuna_study_rf.trials_dataframe().dropna()
# переименуем столбы
optuna_study_rf_pd.rename(columns={
    'values_0': 'roc_train',
    'values_1': 'roc_valid',
    'values_2': 'f1_score',
    'values_3': 'recall_1',
    'values_4': 'precision_1'
},inplace=True)
optuna_study_rf_pd.sort_values(by='precision_1').drop(['datetime_start','datetime_complete','duration'],axis=1)

In [ ]:
# покаждем статистику обучения
print('Максимальное значение метрики ROC AUC на валидационном наборе:',round(optuna_study_rf_pd['roc_valid'].max(),3))
print('Среднее значение метрики ROC AUC на валидационном наборе:',round(optuna_study_rf_pd['roc_valid'].mean(),3))
print('Максимальное значение метрики f1_score на валидационном наборе:',round(optuna_study_rf_pd['f1_score'].max(),3))
print('Среднее значение метрики f1_score на валидационном наборе:',round(optuna_study_rf_pd['f1_score'].mean(),3))
print('Максимальное значение метрики recall_1 на валидационном наборе:',round(optuna_study_rf_pd['recall_1'].max(),3))
print('Среднее значение метрики recall_1 на валидационном наборе:',round(optuna_study_rf_pd['recall_1'].mean(),3))
print('Максимальное значение метрики precision_1 на валидационном наборе:',round(optuna_study_rf_pd['precision_1'].max(),3))
print('Среднее значение метрики precision_1 на валидационном наборе:',round(optuna_study_rf_pd['precision_1'].mean(),3))

In [ ]:
# построим график важности гиперпарметров
optuna.visualization.plot_param_importances(optuna_study_rf, target_name='Score')

In [ ]:
# Построим зависимость метрики precision_1

fig = px.scatter(
    data_frame=optuna_study_rf_pd,
    x='precision_1', #ось абсцисс
    y=['f1_score', 'recall_1'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision класса 1 от recall класса 1', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision на классе 1',
    yaxis_title='Величина метрики recall и f1_score на классе 1 ')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики precision_1 от гиперпараметров

fig = px.scatter(
    data_frame=optuna_study_rf_pd,
    x='precision_1', #ось абсцисс
    y=['params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight','params_max_depth','params_max_features',
        'params_max_samples', 'params_min_samples_leaf', 'params_min_samples_split', 
        'params_n_estimators', 'params_n_last'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision класса 1 от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision на классе 1',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики f1_score от гиперпараметров

fig = px.scatter(
    data_frame=optuna_study_rf_pd,
    x='f1_score', #ось абсцисс
    y=['params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight','params_max_depth','params_max_features',
        'params_max_samples', 'params_min_samples_leaf', 'params_min_samples_split', 
        'params_n_estimators', 'params_n_last'],
)
fig.update_layout(
    title ={
        'text':'Зависимость f1 - score от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики f1 score',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики roc_valid от гиперпараметров

fig = px.scatter(
    data_frame=optuna_study_rf_pd,
    x='roc_valid', #ось абсцисс
    y=['params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight','params_max_depth','params_max_features',
        'params_max_samples', 'params_min_samples_leaf', 'params_min_samples_split', 
        'params_n_estimators', 'params_n_last'],
)
fig.update_layout(
    title ={
        'text':'Зависимость roc valid от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики roc valid',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрик качества модели от number

fig = px.scatter(
    data_frame=optuna_study_rf_pd[(optuna_study_rf_pd['roc_valid']>=0.657)],
    x='number', #ось абсцисс
    y=['roc_valid','roc_train','precision_1','recall_1'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость метрик качества модели от number', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики ROC Valid',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# определим номер лучшге варианта
best_optuna_number = 246

# сформируем словарь лучших гипирпарметров RandomForestClassifier
best_param_rf = {
    'n_estimators' : int(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_n_estimators'].iloc[0]),
    'criterion': optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_criterion'].iloc[0],
    'max_depth' : optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_max_depth'].iloc[0],
    'min_samples_split': optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_min_samples_split'].iloc[0],
    'min_samples_leaf' : optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_min_samples_leaf'].iloc[0],
    'max_features': optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_max_features'].iloc[0],
    'max_samples' : optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_max_samples'].iloc[0],
    'class_weight' : {0:optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_class_0_weight'].iloc[0],
                      1:optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_class_1_weight'].iloc[0]}
    }
# определим перменные для лучших значений параметров
best_n_last = optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_n_last'].iloc[0]
best_class_1_percent = optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_class_1_percent'].iloc[0]
best_random_state = optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_random_state'].iloc[0]


# Выведем принятые наилучшие праметры
print('best n estimators:',best_param_rf['n_estimators'])
print('best criterion:',best_param_rf['criterion'])
print('best max depth:',best_param_rf['max_depth'])
print('best min samples split:',best_param_rf['min_samples_split'])
print('best min samples leaf:',best_param_rf['min_samples_leaf'])
print('best max features(%):',round(best_param_rf['max_features'],3))
print('best max samples(%):',round(best_param_rf['max_samples'],3))
print('best class 0 weight:',round(best_param_rf['class_weight'][0],3))
print('best class 1 weight:',round(best_param_rf['class_weight'][1],3))

print('best class 1 percent:',round(best_class_1_percent,3))
print('best n last:',best_n_last)
print('best random state:',best_random_state)
print('time for best train:',round(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['duration'].iloc[0].seconds/60),'minutes')

print()
print('ROC AUC на обучающем наборе:', round(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['roc_train'].iloc[0],3))
print('ROC AUC на валидационном наборе:', round(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['roc_valid'].iloc[0],3))
print('precision класса 1:', round(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['precision_1'].iloc[0],3))
print('recall класса 1:', round(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['recall_1'].iloc[0],3))

##### <span style="color:MediumSlateBlue">Обучение модели с лучшими параметрами</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'relative'
count_features = dict_spaces[feature_space]

# с помощью функции features_from_transform_data_torow извлечем  
# из данных transform_data_torow
# признаки сооствествующие n_last последним клиенским операциям
list_n_last_features = features_from_transform_data_torow(best_n_last,count_features,25)

# загружаем обучующие наборы
X_train_pd = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas()
y_train_pd = pd.read_csv('target/target_train.csv')

# поготовим данные y_train_pd к работе с функцией class_1_percent_samples
y_train_pd.set_index('id',drop=False,inplace=True)

# подготовим данные для обучения модели
# с помощью функции class_1_percent_samples зададим долю 
# класса 1
list_c1_percent_id = class_1_percent_samples(y_train_pd,best_class_1_percent,random_state=best_random_state)[:1000000]

# сформируем данные для обучения базовой модели
X_train_balanced = np.array(X_train_pd.loc[list_c1_percent_id])[:,list_n_last_features]
y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

# освободим память от "тяжелых" и ненужных файлов
del X_train_pd, y_train_pd
gc.collect()

# обучаем модель RandomForestClassifier с наилучшеми параметрами
random_forest = RandomForestClassifier(
        **best_param_rf,
        n_jobs=-1,
        random_state=best_random_state)
random_forest.fit(X_train_balanced, y_train_balanced)

# освободим память от "тяжелых" и ненужных файлов
del X_train_balanced, y_train_balanced
gc.collect()


# подгружаем данные для тестирования
X_train = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas().to_numpy()[:,list_n_last_features]
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
X_valid = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_valid').to_pandas().to_numpy()[:,list_n_last_features]
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()

# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_rf__pred_proba = random_forest.predict_proba(X_train)[:,1]
y_valid_rf_pred_proba = random_forest.predict_proba(X_valid)[:,1]

# Делаем предсказание для валидационной выборки
y_valid_rf_pred = random_forest.predict(X_valid)

# удаляем крупные файлы чтобы высвободить память 
del X_train, X_valid
gc.collect()

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_rf__pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_rf_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_valid,y_valid_rf_pred,zero_division=0))

# time: 2m 10s

##### <span style="color:MediumSlateBlue">Анализ влияния порога классификации на качество модели</span>

In [ ]:
# чтобы проше обращаться к каждому обьекту в данных
# преобразуем y_valid_LR_pred к Series типу
y_valid_rf_pred_proba = pd.Series(y_valid_rf_pred_proba)

# формируем список из необходимых значений thresholds
thresholds = np.arange(0.01, 1, 0.01)

# сформируем списко для заполнения данными результатов анализа
threshold_data = []


for threshold in thresholds:
        # сформируем список для заполнения данными текушего threshold
        threshold_current_data = [threshold]

        #клиентов, для которых вероятность дефолта > threshold, относим к классу 1
        #в противном случае — к классу 0
        y_valid_rf_pred = y_valid_rf_pred_proba.apply(lambda x: 1 if x>threshold else 0)


        #Считаем метрики для класса 0 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(y_valid, y_valid_rf_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.precision_score(y_valid, y_valid_rf_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.f1_score(y_valid, y_valid_rf_pred,average=None,zero_division=0)[0],3))

        #Считаем метрики для класса 1 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(y_valid, y_valid_rf_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.precision_score(y_valid, y_valid_rf_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.f1_score(y_valid, y_valid_rf_pred,average=None,zero_division=0)[1],3))

        # добавляем полученные данные в список threshold_data
        threshold_data.append(threshold_current_data)

        # из полученных данных сформируем dataframe
        thresholds_pd =pd.DataFrame(
        columns=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'],
        data = threshold_data)

# time: 17s

In [ ]:
# Визуализируем метрики на валидационном наборе при различных threshold
fig_threshold = px.line(
    data_frame=thresholds_pd,
    x='threshold', #ось абсцисс
    y=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'], #ось ординат
)
fig_threshold.update_layout(
    title ={
        'text':'Зависимость качества модели от threshold', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Порог классификации threshold',
    yaxis_title='Величина метрики')
fig_threshold.update_xaxes(showspikes=True)
fig_threshold.update_yaxes(showspikes=True)

fig_threshold.show()

Метрика $precision$ показывает на сколько сильно ошиблась модель при определиние класса.  
Чем выше метрика, тем меньше ошиблась модель. 

$$precision = \frac{TP}{TP+FP}$$

Метрика $recall$ показывает способность модели найти класс.  
Чем выше метрика, тем выше способность модели находить класс.  

$$recall = \frac{TP}{TP+FN}$$

Метрика $f1-score$ показывает баланс между $precision$ и $recall$ 

$threshold$- порог класификации модели.  
Все объекты для которых $у_{\text{proba pred}} < threshold$, модель относит к класс 0.  
Соотвественно, если $у_{\text{proba pred}}  > threshold$ модель относит обьект к классу 1.  
Для идельаной модели $threshold$, является границей между областями класса 0 и класса 1.  

<span style="color:Blue">

Выводы по результам анализа:  

Для улучшения качества модели по метрике $ROC AUC$, нет смысла сдвигать порог   
классификации. Но зависимость метрик качества модели от $threshold$, может рассказать о  
способности модели искать класс 1 и класс 0.

При фокусировке модели на классе 1 (более низкий $threshold$):  
- на классе 1 метрика $precision$ уменьшается, метрика $recall$ растет;  
- на классе 0 метрика $precision$ растет, метрика $recall$ уменьшается.  

При фокусировке модели на классе 0 (более высокий $threshold$):  
- на классе 1 метрика $precision$ увеличивается, метрика $recall$ уменьшается;  
- на классе 0 метрика $precision$ уменьшается, метрика $recall$ растет.  

Обьсняется тем, что в условия дисбаланса модель научилась хорошо определять класс 0 и плохо класс 1.  

При фокусировке модели на классе 1 (более низкий $threshold$), все больше обьектов отнесены к классу 1.   
Так как $precision$ класса 1 уменьшается, значит в области класса 1 стало больше обектов класса 0,  
что явлется закономерным для "идеальной" модели.  
Однако при этом, $recall$ класса 1 растет, значит в добавленных обьектах также присутвует класс 1.  
Это как раз указывает на то, что модель не нучилась находить среди класса 0 класс 1. В области с     
высокой вероятностью класса 0 (более низкий $threshold$), находятся обьекты класса 1.  

Фокусировка на классе 0 (более высокий $threshold$), заставляет модель все больше обьектов из области класса 1  
относить к классу 0. При этом увеличение метрики $precision$ класса 1 говорит о том, что в области класса 1     
становится меньше 0, а значит модель умеет среди класса 1, находить класс 0.  
После  $threshold=0.79$ модель впринципе теряет способность отличать класс 1. У модели нет обьектов класса 1   
в которых она "уверена" на 80+%.

Обратим внимание на $threshold=0.53$, в этой точке находится баланс межу метриками качества модели:
$$precision_1 = recall_1 = \text{f1-score}_1 = 0.1$$
$$precision_0 = recall_0 = \text{f1-score}_0 = 0.966$$

Для модели с лучшим качеством эти две точки должны находится максимально близко к друг другу и иметь   
высокое значение метрик $precision_1$ и $recall_1$.  
Для класса 0 это условие выполнется, что также говорит о том, что модель научилась искать этот класс.    
Для класса 1 это условие не выполнется, что также говорит о том, что модель плохо ищет класс 1.

##### <span style="color:MediumSlateBlue">Построение ROC-кривой выбранной модели на тестовом наборе</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'relative'
count_features = dict_spaces[feature_space]

In [ ]:
# подгружаем данные для тестирования
X_train = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas().to_numpy()[:,list_n_last_features]
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
X_valid = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_valid').to_pandas().to_numpy()[:,list_n_last_features]
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()
X_test = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_test').to_pandas().to_numpy()[:,list_n_last_features]
y_test = pd.read_csv('target/target_test.csv')['flag'].to_numpy()

In [ ]:
print("Размеры обучающего набора",X_train.shape,y_train.shape)
print("Размеры валидационного набора",X_valid.shape,y_valid.shape)
print("Размеры тестового набора",X_test.shape,y_test.shape)
# time: 1m 43s

In [ ]:
# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_rf_pred_proba = random_forest.predict_proba(X_train)[:,1]
y_valid_rf_pred_proba = random_forest.predict_proba(X_valid)[:,1]
y_test_rf_pred_proba = random_forest.predict_proba(X_test)[:,1]

# Делаем предсказание для валидационной выборки
y_test_lr_pred = random_forest.predict(X_test)

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_rf_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_rf_pred_proba),3))
print('ROC AUC на тестовом наборе', round(metrics.roc_auc_score(y_test, y_test_rf_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_test,y_test_lr_pred,zero_division=0))

In [ ]:
# Для расчета значений TPR и FPR используем функцию roc_curve
fpr_train, tpr_train, _ = metrics.roc_curve(y_train, y_train_rf_pred_proba)
fpr_valid, tpr_valid, _ = metrics.roc_curve(y_valid, y_valid_rf_pred_proba)
fpr_test, tpr_test, _ = metrics.roc_curve(y_test, y_test_rf_pred_proba)

# рассчитаем площадь под кривой
auc_train = round(metrics.roc_auc_score(y_train, y_train_rf_pred_proba),3)
auc_valid = round(metrics.roc_auc_score(y_valid, y_valid_rf_pred_proba),3)
auc_test = round(metrics.roc_auc_score(y_test, y_test_rf_pred_proba),3)

trace_train = go.Scatter(x=fpr_train, y=tpr_train, mode='lines', name='AUC_train = %0.2f' % auc_train,
                   line=dict(color='red', width=2),
                   hovertemplate='train<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_train])

trace_valid = go.Scatter(x=fpr_valid, y=tpr_valid, mode='lines', name='AUC_valid = %0.2f' % auc_valid,
                   line=dict(color='green', width=2),
                   hovertemplate='valid<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_valid])

trace_test = go.Scatter(x=fpr_test, y=tpr_test, mode='lines', name='AUC_test = %0.2f' % auc_test,
                   line=dict(color='darkorange', width=2),
                   hovertemplate='test<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_test])


reference_line = go.Scatter(x=[0,1], y=[0,1], mode='lines', name='Reference Line',
                            line=dict(color='navy', width=2, dash='dash'))

fig_roc = go.Figure(data=[trace_train,trace_valid,trace_test,reference_line])

fig_roc.update_layout(
    title ={
        'text':'ROC-кривая', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 1350,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate')

fig_roc.update_xaxes(showspikes=True)
fig_roc.update_yaxes(showspikes=True)


fig_roc.show()

<span style="color:Blue">

Вывод по ROC-кривой:
1. Модель выдает сталибильное качество на валидационном и тестовом наборах, а значит  
не переобучена - разброс ($variance$) не большой. Хотя, качество модели на обучающем наборе,  
заметно лучше.
2. Смещение ($bias$) модели чуть лучше, чем у модели $Logicstic$ $Regression$ $lassifier$, обученной   
на тех же данных.  
На тестовом наборе полученно качество $ROC AUC = 0.65$, против $ROC AUC = 0.61$

#### <span style="color:MediumBlue">Построение модели Gradient Boosting Classifier</span>

##### <span style="color:MediumSlateBlue">Baseline обучение модели Hist Gradient Boosting Classifier</span>

In [ ]:
# обучаем модель Gradient Boosting Classifier
gradient_boosting = HistGradientBoostingClassifier(random_state=42)
gradient_boosting.fit(np.array(X_train_balanced)[:,list_n_last_features],np.array(y_train_balanced))

# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_rf__pred_proba = gradient_boosting.predict_proba(X_train[:,list_n_last_features])[:,1]
y_valid_rf_pred_proba = gradient_boosting.predict_proba(X_valid[:,list_n_last_features])[:,1]

# Делаем предсказание для валидационной выборки
y_valid_rf_pred = gradient_boosting.predict(X_valid[:,list_n_last_features])

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_rf__pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_rf_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_valid,y_valid_rf_pred))

# time: 

##### <span style="color:MediumSlateBlue">Подбор гиперпараметров модели</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'relative'
count_features = dict_spaces[feature_space]

# настроим оптимизацию гипер параметров
def optuna_hgbc(trial):
  # задаем пространства поиска гиперпараметров
  learning_rate = trial.suggest_float('learning_rate',0.2,0.6)
  max_iter = trial.suggest_int('max_iter', 50, 300,step = 1)
  max_leaf_nodes = trial.suggest_int('max_leaf_nodes', 2, 50,step = 1)
  max_depth = trial.suggest_int('max_depth', 1, 15,step = 1)
  min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 50,step = 1)
  max_features = trial.suggest_float('max_features',0.1,1)
  l2_regularization = trial.suggest_float('l2_regularization',0.01,1)
  class_0_weight = trial.suggest_float('class_0_weight',0.01,1)
  class_1_weight = trial.suggest_float('class_1_weight',0.01,1)
  n_last = trial.suggest_int('n_last', 5, 25,step = 1)
  class_1_percent = trial.suggest_float('class_1_percent',0.01,1)
  random_state = trial.suggest_int('random_state', 1, 1000000)


  # создаем модель
  optuna_gradient_boosting = HistGradientBoostingClassifier(
      learning_rate= learning_rate,
      max_iter = max_iter,
      max_leaf_nodes =max_leaf_nodes,
      max_depth = max_depth,
      min_samples_leaf = min_samples_leaf,
      max_features=max_features,
      l2_regularization = l2_regularization,
      class_weight={0:class_0_weight,1:class_1_weight},
      random_state=42)

  # с помощью функции features_from_transform_data_torow извлечем  
  # из данных transform_data_torow
  # признаки сооствествующие n_last последним клиенским операциям
  list_n_last_features = features_from_transform_data_torow(n_last,count_features,25)

  # загружаем обучующие наборы
  X_train_pd = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas()
  y_train_pd = pd.read_csv('target/target_train.csv')

  # поготовим данные y_train_pd к работе с функцией class_1_percent_samples
  y_train_pd.set_index('id',drop=False,inplace=True)

  # подготовим данные для обучения модели
  # с помощью функции class_1_percent_samples зададим долю класса 1
  list_c1_percent_id = class_1_percent_samples(y_train_pd,class_1_percent,random_state=random_state)[:1000000]

  # сформируем данные для обучения модели
  X_train_balanced = np.array(X_train_pd.loc[list_c1_percent_id])[:,list_n_last_features]
  y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

  # освободим память от "тяжелых" и ненужных файлов
  del X_train_pd, y_train_pd
  gc.collect()

  # я не воспользовался параметром timeout метода optimize потому, что  
  # optimize отставливает поиск параметров, если время обучения 
  # превышает timeout. Мне нужно чтобы поиск параметров продолжился.
  try:
    # для модуля func_time_out упакуем обучение модели в функцию try_func
    def try_func():
            return optuna_gradient_boosting.fit(X_train_balanced,y_train_balanced)
    # обучаем модель с ограничением по времени 10 минут (600 секунд)
    optuna_gradient_boosting = func_timeout.func_timeout(600, try_func)

    # удаляем крупные файлы чтобы высвободить память 
    del X_train_balanced, y_train_balanced
    gc.collect()

    # подгружаем данные для тестирования
    X_train = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas().to_numpy()[:,list_n_last_features]
    y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
    X_valid = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_valid').to_pandas().to_numpy()[:,list_n_last_features]
    y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()

    # делаем предсказание на обучающем и валидационном наборе и считаем метрики
    roc_train = metrics.roc_auc_score(y_train, optuna_gradient_boosting.predict_proba(X_train)[:,1])
    roc_valid = metrics.roc_auc_score(y_valid, optuna_gradient_boosting.predict_proba(X_valid)[:,1])
    f1_score = metrics.f1_score(y_valid, optuna_gradient_boosting.predict(X_valid))
    recall_1 = metrics.recall_score(y_valid, optuna_gradient_boosting.predict(X_valid),average=None,zero_division=0)[1]
    precision_1 = metrics.precision_score(y_valid, optuna_gradient_boosting.predict(X_valid),average=None,zero_division=0)[1]

    # удаляем крупные файлы чтобы высвободить память 
    del X_train, X_valid, y_train, y_valid 
    gc.collect()

  except func_timeout.FunctionTimedOut:
    # освободим память от "тяжелых" и ненужных файлов
    del X_train_balanced, y_train_balanced
    gc.collect()

    # обьявим метрики пустыми значениями
    roc_train = 0
    roc_valid = 0
    f1_score = 0
    recall_1 = 0
    precision_1 = 0
    pass

  return roc_train, roc_valid, f1_score, recall_1, precision_1

In [ ]:
# cоздаем объект исследования
# чтобы модель не переобучалась минимизируем roc_train 
# и максимизируем roc_valid
optuna_study_hsbs = optuna.create_study(study_name='HistGradientBoostingClassifier_'+feature_space+'_torow', 
                               directions=['maximize','maximize','maximize','maximize','maximize'], 
                               sampler=optuna.samplers.TPESampler(),
                               pruner='Hyperband',
                               storage='sqlite:///optuna_studies.db',
                               load_if_exists=True)

# ну случай удаления обучения
# optuna.delete_study(study_name='HistGradientBoostingClassifier_'+feature_space+'_torow', storage='sqlite:///optuna_studies.db')

In [ ]:
# ищем лучшую комбинацию гиперпараметров n_trials раз
optuna_study_hsbs.optimize(optuna_hgbc, n_trials=300)

In [ ]:
# из полученного результат соврмируем Data Frame
optuna_study_hsbs_pd = optuna_study_hsbs.trials_dataframe()
# переименуем столбы
optuna_study_hsbs_pd.rename(columns={
    'values_0': 'roc_train',
    'values_1': 'roc_valid',
    'values_2': 'f1_score',
    'values_3': 'recall_1',
    'values_4': 'precision_1'
},inplace=True)
# Выделим время потраченное на обучение модели
last_trail = optuna_study_hsbs_pd['number'].max()
optuna_study_hsbs_pd.tail(5)


In [ ]:
# покаждем статистику обучения
print('Максимальное значение метрики ROC AUC на валидационном наборе:',round(optuna_study_hsbs_pd['roc_valid'].max(),3))
print('Среднее значение метрики ROC AUC на валидационном наборе:',round(optuna_study_hsbs_pd['roc_valid'].mean(),3))
print('Максимальное значение метрики f1_score на валидационном наборе:',round(optuna_study_hsbs_pd['f1_score'].max(),3))
print('Среднее значение метрики f1_score на валидационном наборе:',round(optuna_study_hsbs_pd['f1_score'].mean(),3))
print('Максимальное значение метрики recall_1 на валидационном наборе:',round(optuna_study_hsbs_pd['recall_1'].max(),3))
print('Среднее значение метрики recall_1 на валидационном наборе:',round(optuna_study_hsbs_pd['recall_1'].mean(),3))
print('Максимальное значение метрики precision_1 на валидационном наборе:',round(optuna_study_hsbs_pd['precision_1'].max(),3))
print('Среднее значение метрики precision_1 на валидационном наборе:',round(optuna_study_hsbs_pd['precision_1'].mean(),3))

In [ ]:
# построим график важности гиперпарметров
optuna.visualization.plot_param_importances(optuna_study_hsbs, target_name='Score')

In [ ]:
# Построим зависимость метрики precision_1 
# max_iter

fig = px.scatter(
    data_frame=optuna_study_hsbs_pd,
    x='precision_1', #ось абсцисс
    y=['f1_score','recall_1'
    ], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision класса 1 от recall класса 1', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision класса 1',
    yaxis_title='Величина метрики recall и f1-score класса 1')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики precision_1 от гиперпараметров
# max_iter

fig = px.scatter(
    data_frame=optuna_study_hsbs_pd,
    x='precision_1', #ось абсцисс
    y=['params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight', 'params_l2_regularization',
       'params_learning_rate', 'params_max_depth', 'params_max_features',
       'params_max_iter', 'params_max_leaf_nodes', 'params_min_samples_leaf',
       'params_n_last'
    ], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision класса 1 от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина precision класса 1 ',
    yaxis_title='Величина гиперпараметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики f1_score от гиперпараметров
fig = px.scatter(
    data_frame=optuna_study_hsbs_pd,
    x='f1_score', #ось абсцисс
    y=['params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight', 'params_l2_regularization',
       'params_learning_rate', 'params_max_depth', 'params_max_features',
       'params_max_iter', 'params_max_leaf_nodes', 'params_min_samples_leaf',
       'params_n_last'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость f1-score класса 1 от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики f1-score класса 1',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики roc_valid от гиперпараметров
fig = px.scatter(
    data_frame=optuna_study_hsbs_pd,
    x='roc_valid', #ось абсцисс
    y=['params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight', 'params_l2_regularization',
       'params_learning_rate', 'params_max_depth', 'params_max_features',
       'params_max_iter', 'params_max_leaf_nodes', 'params_min_samples_leaf',
       'params_n_last'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость roc valid от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики roc valid',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрик качества модели от number

fig = px.scatter(
    data_frame=optuna_study_hsbs_pd[(optuna_study_hsbs_pd['roc_valid']>0.67)],
    x='number', #ось абсцисс
    y=['roc_train','roc_valid','recall_1','precision_1'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision_1',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

<span style="color:Blue">

Вывод:  

Их всех выбираем точку $number =124$.   
В этой точке оптимальные значения метрик $recall_1$ и $precision_1$, а  
также относительно высокая метрика $ROC AUC$.   
Посмотрим каким значениям гиперпараметров соотвествует выбранная точка.

In [ ]:
# определим номер лучшге варианта
best_optuna_number = 124

# сформируем словарь лучших гипирпарметров RandomForestClassifier
best_param_rf = {
    'learning_rate' : optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['params_learning_rate'].iloc[0],
    'max_iter' : optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['params_max_iter'].iloc[0],
    'max_leaf_nodes' : optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['params_max_leaf_nodes'].iloc[0],
    'max_depth' : optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['params_max_depth'].iloc[0],
    'min_samples_leaf' : optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['params_min_samples_leaf'].iloc[0],
    'max_features' : optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['params_max_features'].iloc[0],
    'l2_regularization' : optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['params_l2_regularization'].iloc[0],
    'class_weight' : {0: optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['params_class_0_weight'].iloc[0],
                      1: optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['params_class_1_weight'].iloc[0],}
    }

# определим перменные для лучших значений параметров
best_n_last = optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['params_n_last'].iloc[0]
best_class_1_percent = optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['params_class_1_percent'].iloc[0]
best_random_state = optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['params_random_state'].iloc[0]


# Выведем принятые наилучшие праметры
print('best learning rate:',round(best_param_rf['learning_rate'],3))
print('best max iter:',best_param_rf['max_iter'])
print('best max leaf nodes:',best_param_rf['max_leaf_nodes'])
print('best max depth:',best_param_rf['max_depth'])
print('best min samples leaf:',best_param_rf['min_samples_leaf'])
print('best max features:',round(best_param_rf['max_features'],3))
print('best l2 regularization:',round(best_param_rf['l2_regularization'],3))
print('best class 0 weight:',round(best_param_rf['class_weight'][0],3))
print('best class 1 weight:',round(best_param_rf['class_weight'][1],3))

print('best class 1 percent:',round(best_class_1_percent,3))
print('best n last:',best_n_last)
print('best random state:',best_random_state)
print('time for best train:',round(optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['duration'].iloc[0].seconds/60),'minutes')

print()
print('ROC AUC на обучающем наборе:', round(optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['roc_train'].iloc[0],3))
print('ROC AUC на валидационном наборе:', round(optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['roc_valid'].iloc[0],3))
print('precision класса 1:', round(optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['precision_1'].iloc[0],3))
print('recall класса 1:', round(optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['recall_1'].iloc[0],3))

##### <span style="color:MediumSlateBlue">Обучение модели с лучшими параметрами</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'relative'
count_features = dict_spaces[feature_space]

# с помощью функции features_from_transform_data_torow извлечем  
# из данных transform_data_torow
# признаки сооствествующие n_last последним клиенским операциям
list_n_last_features = features_from_transform_data_torow(best_n_last,count_features,25)

# загружаем обучующие наборы
X_train_pd = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas()
y_train_pd = pd.read_csv('target/target_train.csv')

# поготовим данные y_train_pd к работе с функцией class_1_percent_samples
y_train_pd.set_index('id',drop=False,inplace=True)

# подготовим данные для обучения модели
# с помощью функции class_1_percent_samples зададим долю класса 1
list_c1_percent_id = class_1_percent_samples(y_train_pd,best_class_1_percent,random_state=best_random_state)[:1000000]

# сформируем данные для обучения модели
X_train_balanced = np.array(X_train_pd.loc[list_c1_percent_id])[:,list_n_last_features]
y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

# освободим память от "тяжелых" и ненужных файлов
del X_train_pd, y_train_pd
gc.collect()


# обучаем модель HistGradientBoostingClassifier с наилучшеми параметрами
gradient_boosting = HistGradientBoostingClassifier(
        **best_param_rf,
        random_state=42)
gradient_boosting.fit(X_train_balanced,y_train_balanced)

# освободим память от "тяжелых" и ненужных файлов
del X_train_balanced, y_train_balanced
gc.collect()

# подгружаем данные для тестирования
X_train = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas().to_numpy()[:,list_n_last_features]
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
X_valid = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_valid').to_pandas().to_numpy()[:,list_n_last_features]
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()

# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_gb__pred_proba = gradient_boosting.predict_proba(X_train)[:,1]
y_valid_gb_pred_proba = gradient_boosting.predict_proba(X_valid)[:,1]

# Делаем предсказание для валидационной выборки
y_valid_gb_pred = gradient_boosting.predict(X_valid)

# освободим память от "тяжелых" и ненужных файлов
del X_train, X_valid
gc.collect()

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_gb__pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_gb_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_valid,y_valid_gb_pred,zero_division=0))

# time: 2m 30s

##### <span style="color:MediumSlateBlue">Анализ влияния порога классификации на качество модели</span>

In [ ]:
# чтобы проше обращаться к каждому обьекту в данных
# преобразуем y_valid_LR_pred к Series типу
y_valid_gb_pred_proba = pd.Series(y_valid_gb_pred_proba)

# формируем список из необходимых значений thresholds
thresholds = np.arange(0.01, 1, 0.01)

# сформируем списко для заполнения данными результатов анализа
threshold_data = []


for threshold in thresholds:
        # сформируем список для заполнения данными текушего threshold
        threshold_current_data = [threshold]

        #клиентов, для которых вероятность дефолта > threshold, относим к классу 1
        #в противном случае — к классу 0
        y_valid_gb_pred = y_valid_gb_pred_proba.apply(lambda x: 1 if x>threshold else 0)


        #Считаем метрики для класса 0 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(y_valid, y_valid_gb_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.precision_score(y_valid, y_valid_gb_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.f1_score(y_valid, y_valid_gb_pred,average=None,zero_division=0)[0],3))

        #Считаем метрики для класса 1 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(y_valid, y_valid_gb_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.precision_score(y_valid, y_valid_gb_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.f1_score(y_valid, y_valid_gb_pred,average=None,zero_division=0)[1],3))

        # добавляем полученные данные в список threshold_data
        threshold_data.append(threshold_current_data)

        # из полученных данных сформируем dataframe
        thresholds_pd =pd.DataFrame(
        columns=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'],
        data = threshold_data)

# time: 17s

In [ ]:
# Визуализируем метрики на валидационном наборе при различных threshold
fig_threshold = px.line(
    data_frame=thresholds_pd,
    x='threshold', #ось абсцисс
    y=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'], #ось ординат
)
fig_threshold.update_layout(
    title ={
        'text':'Зависимость качества модели от threshold', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Порог классификации threshold',
    yaxis_title='Величина метрики')
fig_threshold.update_xaxes(showspikes=True)
fig_threshold.update_yaxes(showspikes=True)

fig_threshold.show()

Метрика $precision$ показывает на сколько сильно ошиблась модель при определиние класса.  
Чем выше метрика, тем меньше ошиблась модель. 

$$precision = \frac{TP}{TP+FP}$$

Метрика $recall$ показывает способность модели найти класс.  
Чем выше метрика, тем выше способность модели находить класс.  

$$recall = \frac{TP}{TP+FN}$$

Метрика $f1-score$ показывает баланс между $precision$ и $recall$ 

$threshold$- порог класификации модели.  
Все объекты для которых $у_{\text{proba pred}} < threshold$, модель относит к класс 0.  
Соотвественно, если $у_{\text{proba pred}}  > threshold$ модель относит обьект к классу 1.  
Для идельаной модели $threshold$, является границей между областями класса 0 и класса 1.  

<span style="color:Blue">

Выводы по результам анализа:  

Для улучшения качества модели по метрике $ROC AUC$, нет смысла сдвигать порог   
классификации. Но зависимость метрик качества модели от $threshold$, может рассказать о  
способности модели искать класс 1 и класс 0.

При фокусировке модели на классе 1 (более низкий $threshold$):  
- на классе 1 метрика $precision$ уменьшается, метрика $recall$ растет;  
- на классе 0 метрика $precision$ растет, метрика $recall$ уменьшается.  

При фокусировке модели на классе 0 (более высокий $threshold$):  
- на классе 1 метрика $precision$ увеличивается, метрика $recall$ уменьшается;  
- на классе 0 метрика $precision$ уменьшается, метрика $recall$ растет.  

Обьсняется тем, что в условия дисбаланса модель научилась хорошо определять класс 0 и плохо класс 1.  

При фокусировке модели на классе 1 (более низкий $threshold$), все больше обьектов отнесены к классу 1.   
Так как $precision$ класса 1 уменьшается, значит в области класса 1 стало больше обектов класса 0,  
что явлется закономерным для "идеальной" модели.  
Однако при этом, $recall$ класса 1 растет, значит в добавленных обьектах также присутвует класс 1.  
Это как раз указывает на то, что модель не нучилась находить среди класса 0 класс 1. В области с     
высокой вероятностью класса 0 (более низкий $threshold$), находятся обьекты класса 1.  

Фокусировка на классе 0 (более высокий $threshold$), заставляет модель все больше обьектов из области класса 1  
относить к классу 0. При этом увеличение метрики $precision$ класса 1 говорит о том, что в области класса 1     
становится меньше 0, а значит модель умеет среди класса 1, находить класс 0.  
После  $threshold=0.92$ модель впринципе теряет способность отличать класс 1. У модели нет обьектов класса 1   
в которых она "уверена" на 95+%.

Обратим внимание на $threshold=0.52$, в этой точке находится баланс межу метриками качества модели:
$$precision_1 = recall_1 = \text{f1-score}_1 = 0.107$$
$$precision_0 = recall_0 = \text{f1-score}_0 = 0.966$$

Для модели с лучшим качеством эти две точки должны находится максимально близко к друг другу и иметь   
высокое значение метрик $precision_1$ и $recall_1$.  
Для класса 0 это условие выполнется, что также говорит о том, что модель научилась искать этот класс.    
Для класса 1 это условие не выполнется, что также говорит о том, что модель плохо ищет класс 1.

##### <span style="color:MediumSlateBlue">Построение ROC-кривой выбранной модели на тестовом наборе</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'relative'
count_features = dict_spaces[feature_space]

In [ ]:
# подгружаем данные для тестирования
X_train = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas().to_numpy()[:,list_n_last_features]
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
X_valid = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_valid').to_pandas().to_numpy()[:,list_n_last_features]
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()
X_test = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_test').to_pandas().to_numpy()[:,list_n_last_features]
y_test = pd.read_csv('target/target_test.csv')['flag'].to_numpy()

In [ ]:
print("Размеры обучающего набора",X_train.shape,y_train.shape)
print("Размеры валидационного набора",X_valid.shape,y_valid.shape)
print("Размеры тестового набора",X_test.shape,y_test.shape)
# time: 1m 43s

In [ ]:
# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_gb_pred_proba = gradient_boosting.predict_proba(X_train)[:,1]
y_valid_gb_pred_proba = gradient_boosting.predict_proba(X_valid)[:,1]
y_test_gb_pred_proba = gradient_boosting.predict_proba(X_test)[:,1]

# Делаем предсказание для валидационной выборки
y_test_gb_pred = gradient_boosting.predict(X_test)

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_gb_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_gb_pred_proba),3))
print('ROC AUC на тестовом наборе', round(metrics.roc_auc_score(y_test, y_test_gb_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_test,y_test_gb_pred,zero_division=0))

In [ ]:
# Для расчета значений TPR и FPR используем функцию roc_curve
fpr_train, tpr_train, _ = metrics.roc_curve(y_train, y_train_gb_pred_proba)
fpr_valid, tpr_valid, _ = metrics.roc_curve(y_valid, y_valid_gb_pred_proba)
fpr_test, tpr_test, _ = metrics.roc_curve(y_test, y_test_gb_pred_proba)

# рассчитаем площадь под кривой
auc_train = round(metrics.roc_auc_score(y_train, y_train_gb_pred_proba),3)
auc_valid = round(metrics.roc_auc_score(y_valid, y_valid_gb_pred_proba),3)
auc_test = round(metrics.roc_auc_score(y_test, y_test_gb_pred_proba),3)

trace_train = go.Scatter(x=fpr_train, y=tpr_train, mode='lines', name='AUC_train = %0.2f' % auc_train,
                   line=dict(color='red', width=2),
                   hovertemplate='train<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_train])

trace_valid = go.Scatter(x=fpr_valid, y=tpr_valid, mode='lines', name='AUC_valid = %0.2f' % auc_valid,
                   line=dict(color='green', width=2),
                   hovertemplate='valid<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_valid])

trace_test = go.Scatter(x=fpr_test, y=tpr_test, mode='lines', name='AUC_test = %0.2f' % auc_test,
                   line=dict(color='darkorange', width=2),
                   hovertemplate='test<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_test])


reference_line = go.Scatter(x=[0,1], y=[0,1], mode='lines', name='Reference Line',
                            line=dict(color='navy', width=2, dash='dash'))

fig_roc = go.Figure(data=[trace_train,trace_valid,trace_test,reference_line])

fig_roc.update_layout(
    title ={
        'text':'ROC-кривая', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 1350,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate')

fig_roc.update_xaxes(showspikes=True)
fig_roc.update_yaxes(showspikes=True)


fig_roc.show()

<span style="color:Blue">

Вывод по ROC-кривой:
1. Модель выдает сталибильное качество на валидационном и тестовом наборах, а значит  
не переобучена - разброс ($variance$) не большой. Хотя, качество модели на обучающем наборе,  
заметно лучше.
2. Смещение ($bias$) модели чуть лучше, чем у модели $Random$ $Forest$ $Classifier$,  
обученной на тех же данных.  
На тестовом наборе полученно качество $ROC AUC = 0.66$, против $ROC AUC = 0.65$

### <span style="color:RoyalBlue">Построение модели на сбалансированных данных подпространства payments torow</span>

#### <span style="color:MediumBlue">Формирование данных для обучения</span>

Анализ исходных данных, в частности target, показал что представленные данные   
имееют перекос: 96% клиентов у которых не случился дефолт (flag = 0),    
против 4 % - для которых произошел дефолт (flag = 1).  
С помощью функции class_1_percent_samples, сформируем сбаланисрованную выборку  
для обучения моделей.
За основу сбалансировных данных возьмем данные клиентов с дефолтом  
и к ним будем добирать необходимое количество клиентов до сбалансированности. 


Функция class_1_percent_samples принемает две переменные:
- target_data - массив с id и целевой переменной
- class_1_percent - процент класса 1 в результирующем массиве

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'service'
count_features = dict_spaces[feature_space]

In [ ]:
# сформируем данные для анализа сбалансированности обучающих данных
X_train_pd = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas()
y_train_pd = pd.read_csv('target/target_train.csv')

In [ ]:
# поготовим данные y_train_pd к работе с функцией class_1_percent_samples
y_train_pd.set_index('id',drop=False,inplace=True)
y_train_pd.head(4)

In [ ]:
# взглянем на сблансиорованность данных в y_train
y_train_pd['flag'].value_counts(normalize=True)

In [ ]:
# сбалансируем данные в y_train_pd
list_c1_percent_id = class_1_percent_samples(y_train_pd,0.5)
# list_c1_percent_features - список 'id' из y_train_pd соотвествующих  
# заданному проценту класса 1
len(list_c1_percent_id)

In [ ]:
# проверим сбалансированность полученного y_train
y_train_pd.loc[list_c1_percent_id]['flag'].value_counts(normalize=True)

*transform_data_torow*, показываеются последние N операций клиента.  
В данных в *date_torow*, собраны последние 25 операций клиентов.

Предварительный анализ показывает, что медиальное значение количества клиентских  
операций равно 7. Поэтому, для начала, будем показывать моделе $n_{last} = 7$   
последних клиентских операций.

Для извлечения из *transform_data_torow_25* необходимого количества последних клиенских  
операций ($n_{last}$) используем функцию features_from_transform_data_torow.

Функция features_from_transform_data_torow примает следующие аргументы:
- $n_{last} = 7$ - необходимое количество последних клиентских операций; 
- $n_{groups} = 8$ - количество признаков в $date$ $features$;
- $N_{last} = 25$ - количество $n_{last}$ операций, показанных в transform_data_torow_25


In [ ]:
# с помощью функции features_from_transform_data_torow извлечем  
# из данных transform_data_torow_25
# признаки сооствествующие 7 последним клиенским операциям
list_n_last_features = features_from_transform_data_torow(7,count_features,25)

In [ ]:
# подготовим данные для обучения
X_train_balanced = X_train_pd.loc[list_c1_percent_id]
y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag']
# сформируем данные для проверики модели
X_train = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas().to_numpy()
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
X_valid = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_valid').to_pandas().to_numpy()
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()

In [ ]:
# посмотрим на структуру данных в X_train_balanced
X_train_balanced.head(2)

In [ ]:
# посмотрим на структуру данных в y_train_balanced
y_train_balanced.head(2)

In [ ]:
# проверим что выборки действительно совпадают по id
X_train_balanced.index.to_list() == y_train_balanced.index.to_list()

#### <span style="color:MediumBlue">Построение модели Logistic Regression</span>

##### <span style="color:MediumSlateBlue">Baseline обучение модели LogisticRegression</span>

In [ ]:
# обучаем модель логистической регрессии
logistic_regression = linear_model.LogisticRegression(random_state=42, max_iter=10000)
logistic_regression.fit(np.array(X_train_balanced)[:,list_n_last_features],np.array(y_train_balanced))

# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_LR_pred_proba = logistic_regression.predict_proba(X_train[:,list_n_last_features])[:,1]
y_valid_LR_pred_proba = logistic_regression.predict_proba(X_valid[:,list_n_last_features])[:,1]

# Делаем предсказание для валидационной выборки
y_valid_LR_pred = logistic_regression.predict(X_valid[:,list_n_last_features])

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_LR_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_LR_pred_proba),3))

print('Основные метрики на тестовом наборе:')
print(metrics.classification_report(y_valid,y_valid_LR_pred))

# time: 1m 12s

##### <span style="color:MediumSlateBlue">Анализ влияния scaler преобразования на качество и скорость схождения модели</span>

In [ ]:
# формируем список из необходимых scaler
scalers = [MinMaxScaler(),RobustScaler() ,StandardScaler()]

# сформируем списко для заполнения данными результатов анализа
scalers_data = []

# установим ограничение на время обучения модели в 600 секунд (10 минут)
time_out = 600

for scaler in scalers:
        # для того чтобы следить за выполнением цикла введем индикатор
        print('current scaler: ', scaler)

         # сформируем список для заполнения данными текушего scaler
        current_scaler_data = [scaler]
        
        # выполним scaler преобразование
        X_train_s_balanced = scaler.fit_transform(np.array(X_train_balanced)[:,list_n_last_features])
        X_train_s = scaler.transform(X_train[:,list_n_last_features])
        X_valid_s = scaler.transform(X_valid[:,list_n_last_features])

        try:
                # для модуля func_time_out упакуем обучение модели в функцию try_func
                def try_func():
                        logistic_regression = linear_model.LogisticRegression(random_state=42, max_iter=10000)
                        return logistic_regression.fit(X_train_s_balanced,y_train_balanced)
                    
                # фиксируем время начала
                start_time = time.time()
                
                # запускаем обучение модели с ограничением по времени
                logistic_regression = func_timeout.func_timeout(time_out, try_func)
                
                # фиксируем время окончания обучения
                end_time = time.time()
                
                # запишем время обучения модели
                current_scaler_data.append(round(end_time-start_time))

                # Делаем предсказание для обучающей и валидационной выборки
                y_valid_lr_pred = logistic_regression.predict(X_valid_s)
                
                # для метрик ROC AUC делаем предсказание модели для обучающей и валидационной выборки  
                # в виде вероятности принадлежности к классу 1 
                y_train_lr_pred_proba = logistic_regression.predict_proba(X_train_s)[:,1]
                y_valid_lr_pred_proba = logistic_regression.predict_proba(X_valid_s)[:,1]

                #Считаем метрики ROC AUC yна обуающем и валидационном наборе и добавляем их в спискок
                current_scaler_data.append(round(metrics.roc_auc_score(y_train, y_train_lr_pred_proba),3))
                current_scaler_data.append(round(metrics.roc_auc_score(y_valid, y_valid_lr_pred_proba),3))

                #Считаем метрики для класса 0 на валидационном наборе и добавляем их в список
                current_scaler_data.append(round(metrics.recall_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))
                current_scaler_data.append(round(metrics.precision_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))


                #Считаем метрики для класса 1 на валидационном набопе и добавляем их в список
                current_scaler_data.append(round(metrics.recall_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))
                current_scaler_data.append(round(metrics.precision_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))

                # добавляем полученные данные в список scalers_data
                scalers_data.append(current_scaler_data)

                # чтобы подстраховаться на случай ошибки : "The Kernel crashed while executing code"
                # сохраним в локальную папку удачную итерацию
                # для этого из полученных данных сформируем dataframe
                # из полученных данных сформируем dataframe
                scaler_pd =pd.DataFrame(
                        columns=['scaler','time_fit',
                                 'roc_train','roc_valid', 
                                 'recall_0','precision_0',
                                 'recall_1','precision_1'],
                        data = scalers_data)
                # сохраняем dataframe
                scaler_pd.to_csv('data_recovery/scaler_pd.csv',index=False)
               
                # очистим output от прошедшей итерации
                clear_output()

        except func_timeout.FunctionTimedOut:
                # если время обучения привышает лимит
                # запишем в current_scaler_data соотвествующую данные
                current_scaler_data = [scaler,'>30m',0,0,0,0,0,0]
                # добавляем полученные данные в список scalers_data
                scalers_data.append(current_scaler_data)

                # чтобы подстраховаться на случай ошибки : "The Kernel crashed while executing code"
                # сохраним в локальную папку удачную итерацию
                # для этого из полученных данных сформируем dataframe
                # из полученных данных сформируем dataframe
                scaler_pd =pd.DataFrame(
                        columns=['scaler','time_fit',
                                 'roc_train','roc_valid', 
                                 'recall_0','precision_0',
                                 'recall_1','precision_1'],
                        data = scalers_data)
                # сохраняем dataframe
                scaler_pd.to_csv('data_recovery/scaler_pd.csv',index=False)
                
                # очистим output от прошедшей итерации
                clear_output()
                # переходим к следующему scaler
                pass
# очистим output
clear_output()

# сформируем данные соотвествующие лучщему ROC AUC на валидацинном наборе
best_roc_valid_data = scaler_pd[scaler_pd['roc_valid']==scaler_pd['roc_valid'].max()]

# выведем лучший результат на валидационном наборе
print('result:')
print('best scaler on valid:',best_roc_valid_data.iloc[0]['scaler'])
print('ROC AUC on train:', best_roc_valid_data.iloc[0]['roc_train'])
print('ROC AUC on valid:', best_roc_valid_data.iloc[0]['roc_valid'])
print('Time fit for best scaler:', best_roc_valid_data.iloc[0]['time_fit'],' seconds')

# выведем полученный dataframe
scaler_pd

# time: 

##### <span style="color:MediumSlateBlue"> Анализ влияния solver оптимизатора на качество и скорость обучения модели</span>

Проверим качество и скорость сходимости модели при разных solver  
Проверку осуществим через цикл  
Чтобы модель не сходилась бесконечно с помощью модуля func_timeout  
поставим ограничение времени схождения в 10 минут

In [ ]:
# подготовим данные для обучения модели

# обьявим scaler
scaler = StandardScaler()
# выполним scaler преобразование
X_train_s_balanced = scaler.fit_transform(np.array(X_train_balanced)[:,list_n_last_features])
X_train_s = scaler.transform(X_train[:,list_n_last_features])
X_valid_s = scaler.transform(X_valid[:,list_n_last_features])

# time: 10s

In [ ]:
# Зададим список solvers
solvers_list = ['lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga']

# сформируем список для заполнения
solvers_data = []

# установим ограничение на время обучения модели в 300 секунд (5 минут)
time_out = 300

for solver in solvers_list:
        # для того чтобы следить за выполнением цикла введем индикатор
        print('current solver: ', solver)

         # сформируем список для заполнения данными текушего solver
        current_solver_data = [solver]
        
        try:
                # для модуля func_time_out упакуем обучение модели в функцию try_func
                def try_func():
                        logistic_regression = linear_model.LogisticRegression(solver= solver,
                                                                  random_state=42, 
                                                                  max_iter=10000)
                        return logistic_regression.fit(X_train_s_balanced,y_train_balanced)
                    
                # фиксируем время начала
                start_time = time.time()
                
                # запускаем обучение модели с ограничением по времени
                logistic_regression = func_timeout.func_timeout(time_out, try_func)
                
                # фиксируем время окончания обучения
                end_time = time.time()
                
                # запишем время обучения модели
                current_solver_data.append(round(end_time-start_time))

                # Делаем предсказание для обучающей и валидационной выборки
                y_valid_lr_pred = logistic_regression.predict(X_valid_s)
                
                # для метрик ROC AUC делаем предсказание модели для обучающей и валидационной выборки  
                # в виде вероятности принадлежности к классу 1 
                y_train_lr_pred_proba = logistic_regression.predict_proba(X_train_s)[:,1]
                y_valid_lr_pred_proba = logistic_regression.predict_proba(X_valid_s)[:,1]

                #Считаем метрики ROC AUC yна обуающем и валидационном наборе и добавляем их в спискок
                current_solver_data.append(round(metrics.roc_auc_score(y_train, y_train_lr_pred_proba),3))
                current_solver_data.append(round(metrics.roc_auc_score(y_valid, y_valid_lr_pred_proba),3))

                #Считаем метрики для класса 0 на валидационном наборе и добавляем их в список
                current_solver_data.append(round(metrics.recall_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))
                current_solver_data.append(round(metrics.precision_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))


                #Считаем метрики для класса 1 на валидационном набопе и добавляем их в список
                current_solver_data.append(round(metrics.recall_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))
                current_solver_data.append(round(metrics.precision_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))

                # запишем данные полученные на одной итерации
                solvers_data.append(current_solver_data)

                # чтобы подстраховаться на случай ошибки : "The Kernel crashed while executing code"
                # сохраним в локальную папку удачную итерацию
                # для этого из полученных данных сформируем dataframe
                # из полученных данных сформируем dataframe
                solvers_pd =pd.DataFrame(
                        columns=['solver','time_fit',
                                 'roc_train','roc_valid', 
                                 'recall_0','precision_0',
                                 'recall_1','precision_1'],
                        data = solvers_data)
                # сохраняем dataframe
                solvers_pd.to_csv('data_recovery/solvers_pd.csv',index=False)
               
                # очистим output от прошедшей итерации
                clear_output()

        except func_timeout.FunctionTimedOut:
                # если время обучения привышает лимит
                # запишем в current_solver_data соотвествующую данные
                current_solver_data = [solver,'>10m',0,0,0,0,0,0]
                # добавляем полученные данные в список solvers_data
                solvers_data.append(current_solver_data)

                # чтобы подстраховаться на случай ошибки : "The Kernel crashed while executing code"
                # сохраним в локальную папку удачную итерацию
                # для этого из полученных данных сформируем dataframe
                # из полученных данных сформируем dataframe
                solvers_pd =pd.DataFrame(
                        columns=['solver','time_fit',
                                 'roc_train','roc_valid', 
                                 'recall_0','precision_0',
                                 'recall_1','precision_1'],
                        data = solvers_data)
                # сохраняем dataframe
                solvers_pd.to_csv('data_recovery/solvers_pd.csv',index=False)
                
                # очистим output от прошедшей итерации
                clear_output()
                # переходим к следующему solver
                pass
# очистим output
clear_output()

# сформируем данные соотвествующие лучщему ROC AUC на валидацинном наборе
best_roc_valid_data = solvers_pd[solvers_pd['roc_valid']==solvers_pd['roc_valid'].max()]

# выведем лучший результат на валидационном наборе
print('result:')
print('best solver on valid:',best_roc_valid_data.iloc[0]['solver'])
print('ROC AUC on train:', best_roc_valid_data.iloc[0]['roc_train'])
print('ROC AUC on valid:', best_roc_valid_data.iloc[0]['roc_valid'])
print('Time fit for best solver:', best_roc_valid_data.iloc[0]['time_fit'],' seconds')

# выведем полученный dataframe
solvers_pd

# time: 11m 35s

##### <span style="color:MediumSlateBlue">Подбор гиперпараметров модели</span>

In [ ]:
# для выбора sceler преобразвания на понадобится словарь
dict_scalers ={
    'MinMaxScaler':MinMaxScaler(),
    'RobustScaler':RobustScaler(),
    'StandardScaler':StandardScaler(),
}
# определим пространство признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'payments'
count_features = dict_spaces[feature_space]

# настроим оптимизацию гипер параметров
def optuna_lg(trial):
  # задаем пространства поиска гиперпараметров
  C = trial.suggest_float('C',0.01,1)
  solver = trial.suggest_categorical('solver', ['lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky'])
  class_0_weight = trial.suggest_float('class_0_weight',0.01,1)
  class_1_weight = trial.suggest_float('class_1_weight',0.01,1)
  n_last = trial.suggest_int('n_last', 10, 25,step = 1)
  class_1_percent = trial.suggest_float('class_1_percent',0.01,1)
  scaler = trial.suggest_categorical('scaler', ['MinMaxScaler', 'RobustScaler','StandardScaler'])
  random_state = trial.suggest_int('random_state', 1, 1000000)

  # создаем модель
  optuna_log_reg = linear_model.LogisticRegression(
      solver=solver,
      C=C,
      class_weight={0:class_0_weight,1:class_1_weight},
      random_state=random_state,
      max_iter=10000)

  # с помощью функции features_from_transform_data_torow извлечем  
  # из данных transform_data_torow
  # признаки сооствествующие n_last последним клиенским операциям
  list_n_last_features = features_from_transform_data_torow(n_last,count_features,25)

  # обьявим scaler
  scaler = dict_scalers[scaler]

  # загружаем обучующие наборы
  X_train_pd = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas()
  y_train_pd = pd.read_csv('target/target_train.csv')

  # поготовим данные y_train_pd к работе с функцией class_1_percent_samples
  y_train_pd.set_index('id',drop=False,inplace=True)

  # подготовим данные для обучения модели
  # с помощью функции class_1_percent_samples зададим долю 
  # класса 1
  list_c1_percent_id = class_1_percent_samples(y_train_pd,class_1_percent,random_state=random_state)[:1000000]

  # сформируем данные для обучения базовой модели
  X_train_s_balanced = scaler.fit_transform(np.array(X_train_pd.loc[list_c1_percent_id])[:,list_n_last_features])
  y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

  # освободим память от "тяжелых" и ненужных файлов
  del X_train_pd, y_train_pd
  gc.collect()

  # я не воспользовался параметром timeout метода optimize потому, что  
  # optimize отставливает поиск параметров, если время обучения 
  # превышает timeout. Мне нужно чтобы поиск параметров продолжился.
  try:
    # для модуля func_time_out упакуем обучение модели в функцию try_func
    def try_func():
            return optuna_log_reg.fit(X_train_s_balanced, y_train_balanced)
    # обучаем модель с ограничением по времени 10 минут (600 секунд)
    optuna_log_reg = func_timeout.func_timeout(600, try_func)

    # удаляем крупные файлы чтобы высвободить память 
    del X_train_s_balanced,y_train_balanced
    gc.collect()

    # подгружаем данные для тестирования
    X_train = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas().to_numpy()[:,list_n_last_features]
    y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
    X_valid = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_valid').to_pandas().to_numpy()[:,list_n_last_features]
    y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()

    # сформируем данные для проверки модели
    X_train_s = scaler.transform(X_train)
    X_valid_s = scaler.transform(X_valid)

    # удаляем крупные файлы чтобы высвободить память 
    del X_train, X_valid
    gc.collect()

    # делаем предсказание на обучающем и валидационном наборе и считаем метрики
    roc_train = metrics.roc_auc_score(y_train, optuna_log_reg.predict_proba(X_train_s)[:,1])
    roc_valid = metrics.roc_auc_score(y_valid, optuna_log_reg.predict_proba(X_valid_s)[:,1])
    f1_score = metrics.f1_score(y_valid, optuna_log_reg.predict(X_valid_s))
    recall_1 = metrics.recall_score(y_valid, optuna_log_reg.predict(X_valid_s),average=None,zero_division=0)[1]
    precision_1 = metrics.precision_score(y_valid, optuna_log_reg.predict(X_valid_s),average=None,zero_division=0)[1]

    del X_train_s, X_valid_s, y_train, y_valid
    gc.collect()

  except func_timeout.FunctionTimedOut:
    # освободим память от "тяжелых" и ненужных файлов
    del X_train_s_balanced, y_train_balanced
    gc.collect()

    # обьявим метрики пустыми значениями
    roc_train = 0
    roc_valid = 0
    f1_score = 0
    recall_1 = 0
    precision_1 = 0
    pass

  return roc_train, roc_valid, f1_score, recall_1, precision_1

In [ ]:
# cоздаем объект исследования
# чтобы модель не переобучалась минимизируем roc_train 
# и максимизируем roc_valid
optuna_study_lg = optuna.create_study(study_name='LogisticRegression_'+feature_space+'_torow', 
                               directions=['maximize','maximize','maximize','maximize','maximize'], 
                               sampler=optuna.samplers.TPESampler(),
                               pruner='Hyperband',
                               storage='sqlite:///optuna_studies.db',
                               load_if_exists=True)
# на случай удаления 
# optuna.delete_study(study_name='LogisticRegression_'+feature_space+'_torow', storage='sqlite:///optuna_studies.db')

In [ ]:
# ищем лучшую комбинацию гиперпараметров n_trials раз
optuna_study_lg.optimize(optuna_lg, n_trials=300)

In [ ]:
# из полученного результат соврмируем Data Frame
optuna_study_lg_pd = optuna_study_lg.trials_dataframe().dropna()
# переименуем столбы
optuna_study_lg_pd.rename(columns={
    'values_0': 'roc_train',
    'values_1': 'roc_valid',
    'values_2': 'f1_score',
    'values_3': 'recall_1',
    'values_4': 'precision_1'
},inplace=True)
# Выделим время потраченное на обучение модели
optuna_study_lg_pd.head(5)

In [ ]:
# покаждем статистику обучения
print('Максимальное значение метрики ROC AUC на валидационном наборе:',round(optuna_study_lg_pd['roc_valid'].max(),3))
print('Среднее значение метрики ROC AUC на валидационном наборе:',round(optuna_study_lg_pd['roc_valid'].mean(),3))
print('Максимальное значение метрики f1_score на валидационном наборе:',round(optuna_study_lg_pd['f1_score'].max(),3))
print('Среднее значение метрики f1_score на валидационном наборе:',round(optuna_study_lg_pd['f1_score'].mean(),3))
print('Максимальное значение метрики recall_1 на валидационном наборе:',round(optuna_study_lg_pd['recall_1'].max(),3))
print('Среднее значение метрики recall_1 на валидационном наборе:',round(optuna_study_lg_pd['recall_1'].mean(),3))
print('Максимальное значение метрики precision_1 на валидационном наборе:',round(optuna_study_lg_pd['precision_1'].max(),3))
print('Среднее значение метрики precision_1 на валидационном наборе:',round(optuna_study_lg_pd['precision_1'].mean(),3))

In [ ]:
# построим график важности гиперпарметров
optuna.visualization.plot_param_importances(optuna_study_lg, target_name='Score')

In [ ]:
# Построим зависимость метрики precision_1

fig= px.scatter(
    data_frame=optuna_study_lg_pd,
    x='precision_1', #ось абсцисс
    y=['recall_1','f1_score'
    ], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision от recall класса 1', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision 1',
    yaxis_title='Величина метрики recall и f1-score')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики precision_1 от гипер параметров

fig= px.scatter(
    data_frame=optuna_study_lg_pd,
    x='precision_1', #ось абсцисс
    y=['params_C','params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight', 'params_n_last',
    ], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision класса 1 от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision 1',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики f1_score от гипер параметров

fig= px.scatter(
    data_frame=optuna_study_lg_pd,
    x='f1_score', #ось абсцисс
    y=['params_C','params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight', 'params_n_last',
    ], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость f1-score от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики f1-score',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики roc valid от гипер параметров

fig= px.scatter(
    data_frame=optuna_study_lg_pd,
    x='roc_valid', #ось абсцисс
    y=['params_C','params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight', 'params_n_last',
    ], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость roc valid от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики roc valid',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Визуализируем метрики полученные при optuna оптимизации
fig = px.scatter(
    data_frame=optuna_study_lg_pd[
                                (optuna_study_lg_pd['roc_valid']>0.99*optuna_study_lg_pd['roc_valid'].max())],
    x='number', #ось абсцисс
    y=['roc_train','roc_valid','recall_1','precision_1'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость качества модели от trials optuna', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='trials optuna',
    yaxis_title='Величина метрики')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

<span style="color:Blue">

Вывод:  

Их всех выбираем точку $number =294$.   
В этой точке одно из самых больших значений $ROC AUC$.  
Посмотрим каким значениям гиперпараметров соотвествует выбранная точка.

In [ ]:
# определим номер лучшге варианта
best_optuna_number = 294

# сформируем словарь лучших гипирпарметров RandomForestClassifier
best_param_lr_torow = {
    'C' : round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_C'].iloc[0],3),
    'class_weight' : {0:round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_class_0_weight'].iloc[0],3),
                      1:round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_class_1_weight'].iloc[0],3)},
    }
# создадим перменные
best_n_last = int(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_n_last'].iloc[0])
best_scaler = optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_scaler'].iloc[0]
best_class_1_percent = optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_class_1_percent'].iloc[0]
best_random_state = int(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_random_state'].iloc[0])
# Выведем принятые наилучшие праметры
print('best C:',best_param_lr_torow['C'])
print('best class 0 weight:',best_param_lr_torow['class_weight'][0])
print('best class 1 weight:',best_param_lr_torow['class_weight'][1])

print('best class 1 percent:',round(best_class_1_percent,3))
print('best scaler:',best_scaler)
print('best n_last:',round(best_n_last,3))
print('best best random state:',best_random_state)
print('time for best train:',round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['duration'].iloc[0].seconds/60),'minutes')

print()
print('ROC AUC на обучающем наборе:', round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['roc_train'].iloc[0],3))
print('ROC AUC на валидационном наборе:', round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['roc_valid'].iloc[0],3))
print('precision класса 1:', round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['precision_1'].iloc[0],3))
print('recall класса 1:', round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['recall_1'].iloc[0],3))

##### <span style="color:MediumSlateBlue">Обучение модели с лучшими параметрами</span>

In [ ]:
# для выбора sceler преобразвания на понадобится словарь
dict_scalers ={
    'MinMaxScaler':MinMaxScaler(),
    'RobustScaler':RobustScaler(),
    'StandardScaler':StandardScaler(),
}

# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'payments'
count_features = dict_spaces[feature_space]

# с помощью функции features_from_transform_data_torow извлечем  
# из данных transform_data_torow_25
# признаки сооствествующие n_last последним клиенским операциям
list_n_last_features = features_from_transform_data_torow(best_n_last,count_features,25)

# обьявим scaler
scaler = dict_scalers[best_scaler]

# загружаем обучующие наборы
X_train_pd = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas()
y_train_pd = pd.read_csv('target/target_train.csv')

# поготовим данные y_train_pd к работе с функцией class_1_percent_samples
y_train_pd.set_index('id',drop=False,inplace=True)

# подготовим данные для обучения модели
# с помощью функции class_1_percent_samples зададим долю 
# класса 1
list_c1_percent_id = class_1_percent_samples(y_train_pd,best_class_1_percent,random_state=best_random_state)[:1000000]

# сформируем данные для обучения базовой модели
X_train_s_balanced = scaler.fit_transform(np.array(X_train_pd.loc[list_c1_percent_id])[:,list_n_last_features])
y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

# освободим память от "тяжелых" и ненужных файлов
del X_train_pd, y_train_pd
gc.collect()


# обучаем модель LogisticRegression с наилучшеми параметрами
logistic_regression = linear_model.LogisticRegression(
        **best_param_lr_torow,
        random_state=best_random_state,
        max_iter=10000)
logistic_regression.fit(X_train_s_balanced,y_train_balanced)

# удаляем крупные файлы чтобы высвободить память 
del X_train_s_balanced,y_train_balanced
gc.collect()

# подгружаем данные для тестирования
X_train = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas().to_numpy()[:,list_n_last_features]
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
X_valid = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_valid').to_pandas().to_numpy()[:,list_n_last_features]
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()
    
# сформируем данные для проверки модели
X_train_s = scaler.transform(X_train)
X_valid_s = scaler.transform(X_valid)

# удаляем крупные файлы чтобы высвободить память 
del X_train, X_valid
gc.collect()

# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_lr_pred_proba = logistic_regression.predict_proba(X_train_s)[:,1]
y_valid_lr_pred_proba = logistic_regression.predict_proba(X_valid_s)[:,1]

# Делаем предсказание для валидационной выборки
y_valid_lr_pred = logistic_regression.predict(X_valid_s)

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_lr_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_lr_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_valid,y_valid_lr_pred,zero_division=0))

# time: 2m 40s

##### <span style="color:MediumSlateBlue">Анализ влияния порога классификации на качество модели</span>

In [ ]:
# чтобы проше обращаться к каждому обьекту в данных
# преобразуем y_valid_LR_pred к Series типу
y_valid_lr_pred_proba = pd.Series(y_valid_lr_pred_proba)

# формируем список из необходимых значений thresholds
thresholds = np.arange(0.01, 1, 0.01)

# сформируем списко для заполнения данными результатов анализа
threshold_data = []


for threshold in thresholds:
        # сформируем список для заполнения данными текушего threshold
        threshold_current_data = [threshold]

        #клиентов, для которых вероятность дефолта > threshold, относим к классу 1
        #в противном случае — к классу 0
        y_valid_lr_pred = y_valid_lr_pred_proba.apply(lambda x: 1 if x>threshold else 0)


        #Считаем метрики для класса 0 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.precision_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.f1_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))

        #Считаем метрики для класса 1 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.precision_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.f1_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))

        # добавляем полученные данные в список threshold_data
        threshold_data.append(threshold_current_data)

        # из полученных данных сформируем dataframe
        thresholds_pd =pd.DataFrame(
        columns=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'],
        data = threshold_data)

# time: 17s

In [ ]:
# Визуализируем метрики на валидационном наборе при различных threshold
fig_threshold = px.line(
    data_frame=thresholds_pd,
    x='threshold', #ось абсцисс
    y=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'], #ось ординат
)
fig_threshold.update_layout(
    title ={
        'text':'Зависимость качества модели от threshold', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Порог классификации threshold',
    yaxis_title='Величина метрики')
fig_threshold.update_xaxes(showspikes=True)
fig_threshold.update_yaxes(showspikes=True)

fig_threshold.show()

Метрика $precision$ показывает на сколько сильно ошиблась модель при определиние класса.  
Чем выше метрика, тем меньше ошиблась модель. 

$$precision = \frac{TP}{TP+FP}$$

Метрика $recall$ показывает способность модели найти класс.  
Чем выше метрика, тем выше способность модели находить класс.  

$$recall = \frac{TP}{TP+FN}$$

Метрика $f1-score$ показывает баланс между $precision$ и $recall$ 

$threshold$- порог класификации модели.  
Все объекты для которых $у_{\text{proba pred}} < threshold$, модель относит к класс 0.  
Соотвественно, если $у_{\text{proba pred}}  > threshold$ модель относит обьект к классу 1.  
Для идельаной модели $threshold$, является границей между областями класса 0 и класса 1.  

<span style="color:Blue">

Выводы по результам анализа:  

Для улучшения качества модели по метрике $ROC AUC$, нет смысла сдвигать порог   
классификации. Но зависимость метрик качества модели от $threshold$, может рассказать о  
спосбоности модели искать класс 1 и класс 0.

При фокусировке модели на классе 1 (более низкий $threshold$):  
- на классе 1 метрика $precision$ уменьшается, метрика $recall$ растет;  
- на классе 0 метрика $precision$ растет, метрика $recall$ уменьшается.  

При фокусировке модели на классе 0 (более высокий $threshold$):  
- на классе 1 метрика $precision$ увеличивается, метрика $recall$ уменьшается;  
- на классе 0 метрика $precision$ уменьшается, метрика $recall$ растет.  

Обьсняется тем, что в условия дисбаланса модель научилась хорошо определять класс 0 и плохо класс 1.  

При фокусировке модели на классе 1 (более низкий $threshold$), все больше обьектов отнесены к классу 1.   
Так как $precision$ класса 1 уменьшается, значит в области класса 1 стало больше обектов класса 0,  
что явлется закономерным для "идеальной" модели.  
Однако при этом, $recall$ класса 1 растет, значит в добавленных обьектах также присутвует класс 1.  
Это как раз указывает на то, что модель не нучилась находить среди класса 0 класс 1. В области с     
высокой вероятностью класса 0 (более низкий $threshold$), находятся обьекты класса 1.  

Фокусировка на классе 0 (более высокий $threshold$), заставляет модель все больше обьектов из области класса 1  
относить к классу 0. При этом увеличение метрики $precision$ класса 1 говорит о том, что в области класса 1     
становится меньше 0, а значит модель умеет среди класса 1, находить класс 0.  


Обратим внимание на $threshold=0.5$, в этой точке находится баланс межу метриками качества модели:
$$precision_1 = recall_1 = \text{f1-score}_1 = 0.083$$
$$precision_0 = recall_0 = \text{f1-score}_0 = 0.995$$

Для модели с лучшим качеством эти две точки должны находится максимально близко к друг другу и иметь  
высокое значение метрик $precision_1$ и $recall_1$.  
Для класса 0 это условие выполнется, что также говорит о том, что модель научилась искать этот класс.   
Для класса 1 это условие не выполнется, что также говорит о том, что модель плохо ищет класс 1.

##### <span style="color:MediumSlateBlue">Построение ROC-кривой выбранной модели на тестовом наборе</span>

In [ ]:
# определим пространство признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'payments'
count_features = dict_spaces[feature_space]

In [ ]:
# подгружаем данные для тестирования
X_train = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas().to_numpy()[:,list_n_last_features]
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
X_valid = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_valid').to_pandas().to_numpy()[:,list_n_last_features]
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()
X_test = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_test').to_pandas().to_numpy()[:,list_n_last_features]
y_test = pd.read_csv('target/target_test.csv')['flag'].to_numpy()

In [ ]:
# выполним scaler преобразование
X_train_s = scaler.transform(X_train)
X_valid_s = scaler.transform(X_valid)
X_test_s = scaler.transform(X_test)

print("Размеры обучающего набора",X_train_s.shape)
print("Размеры валидационного набора",X_valid_s.shape)
print("Размеры тестового набора",X_test_s.shape)
# time: 1m 43s

In [ ]:
# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_lr_pred_proba = logistic_regression.predict_proba(X_train_s)[:,1]
y_valid_lr_pred_proba = logistic_regression.predict_proba(X_valid_s)[:,1]
y_test_lr_pred_proba = logistic_regression.predict_proba(X_test_s)[:,1]

# Делаем предсказание для валидационной выборки
y_test_lr_pred = logistic_regression.predict(X_test_s)

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_lr_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_lr_pred_proba),3))
print('ROC AUC на тестовом наборе', round(metrics.roc_auc_score(y_test, y_test_lr_pred_proba),3))

print('Основные метрики на тестовом наборе:')
print(metrics.classification_report(y_test,y_test_lr_pred,zero_division=0))

In [ ]:
# Для расчета значений TPR и FPR используем функцию roc_curve
fpr_train, tpr_train, _ = metrics.roc_curve(y_train, y_train_lr_pred_proba)
fpr_valid, tpr_valid, _ = metrics.roc_curve(y_valid, y_valid_lr_pred_proba)
fpr_test, tpr_test, _ = metrics.roc_curve(y_test, y_test_lr_pred_proba)

# рассчитаем площадь под кривой
auc_train = round(metrics.roc_auc_score(y_train, y_train_lr_pred_proba),3)
auc_valid = round(metrics.roc_auc_score(y_valid, y_valid_lr_pred_proba),3)
auc_test = round(metrics.roc_auc_score(y_test, y_test_lr_pred_proba),3)

trace_train = go.Scatter(x=fpr_train, y=tpr_train, mode='lines', name='AUC_train = %0.2f' % auc_train,
                   line=dict(color='red', width=2),
                   hovertemplate='train<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_train])

trace_valid = go.Scatter(x=fpr_valid, y=tpr_valid, mode='lines', name='AUC_valid = %0.2f' % auc_valid,
                   line=dict(color='green', width=2),
                   hovertemplate='valid<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_valid])

trace_test = go.Scatter(x=fpr_test, y=tpr_test, mode='lines', name='AUC_test = %0.2f' % auc_test,
                   line=dict(color='darkorange', width=2),
                   hovertemplate='test<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_test])


reference_line = go.Scatter(x=[0,1], y=[0,1], mode='lines', name='Reference Line',
                            line=dict(color='navy', width=2, dash='dash'))

fig_roc = go.Figure(data=[trace_train,trace_valid,trace_test,reference_line])

fig_roc.update_layout(
    title ={
        'text':'ROC-кривая', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 1350,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate')

fig_roc.update_xaxes(showspikes=True)
fig_roc.update_yaxes(showspikes=True)


fig_roc.show()

<span style="color:Blue">

Вывод по ROC-кривой:
1. Модель выдает сталибильное качество на всех наборах, а значит  
не переобучена - разброс ($variance$) не большой.  
2. Смещение ($bias$) модели относительно большое.   
На тестовом наборе полученно качество $ROC AUC = 0.64$.

#### <span style="color:MediumBlue">Построение модели Random Forest Classifier</span>

##### <span style="color:MediumSlateBlue">Baseline обучение модели RandomForestClassifier</span>

In [ ]:
# обучаем модель логистической регрессии
# чтобы модель пыталась найти связь во всех признаках
random_forest = RandomForestClassifier(random_state=42, n_jobs=-1)
random_forest.fit(np.array(X_train_balanced)[:,list_n_last_features],np.array(y_train_balanced))

# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_rf__pred_proba = random_forest.predict_proba(X_train[:,list_n_last_features])[:,1]
y_valid_rf_pred_proba = random_forest.predict_proba(X_valid[:,list_n_last_features])[:,1]

# Делаем предсказание для валидационной выборки
y_valid_rf_pred = random_forest.predict(X_valid[:,list_n_last_features])

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_rf__pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_rf_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_valid,y_valid_rf_pred))

# time: 2m 5s

##### <span style="color:MediumSlateBlue">Подбор гиперпараметров модели</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'payments'
count_features = dict_spaces[feature_space]

# настроим оптимизацию гипер параметров
def optuna_rf(trial):
  # задаем пространства поиска гиперпараметров
  # задаем пространства поиска гиперпараметров
  n_estimators = trial.suggest_int('n_estimators', 50, 150,step = 5)
  criterion = trial.suggest_categorical('criterion',['gini', 'entropy', 'log_loss'])
  max_depth = trial.suggest_int('max_depth', 9, 20,step = 1)
  min_samples_split = trial.suggest_int('min_samples_split', 5, 15,step = 1)
  min_samples_leaf = trial.suggest_int('min_samples_leaf', 5, 25,step = 1)
  max_features = trial.suggest_float('max_features',0.1,1)
  max_samples = trial.suggest_float('max_samples',0.1,1)
  class_0_weight = trial.suggest_float('class_0_weight',0.01,1)
  class_1_weight = trial.suggest_float('class_1_weight',0.01,1)
  n_last = trial.suggest_int('n_last', 5, 25,step = 1)
  class_1_percent = trial.suggest_float('class_1_percent',0.01,1)
  random_state = trial.suggest_int('random_state', 1, 1000000)


  # создаем модель
  optuna_random_forest = RandomForestClassifier(
      n_estimators = n_estimators, 
      criterion = criterion,
      max_depth = max_depth,
      min_samples_split = min_samples_split,  
      min_samples_leaf = min_samples_leaf,
      max_features=max_features,
      max_samples=max_samples,
      class_weight={0:class_0_weight,1:class_1_weight},
      n_jobs=-1,
      random_state=random_state)

  # с помощью функции features_from_transform_data_torow извлечем  
  # из данных transform_data_torow
  # признаки сооствествующие n_last последним клиенским операциям
  list_n_last_features = features_from_transform_data_torow(n_last,count_features,25)

  # загружаем обучующие наборы
  X_train_pd = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas()
  y_train_pd = pd.read_csv('target/target_train.csv')

  # поготовим данные y_train_pd к работе с функцией class_1_percent_samples
  y_train_pd.set_index('id',drop=False,inplace=True)

  # подготовим данные для обучения модели
  # с помощью функции class_1_percent_samples зададим долю 
  # класса 1
  list_c1_percent_id = class_1_percent_samples(y_train_pd,class_1_percent,random_state=random_state)[:1000000]

  # сформируем данные для обучения базовой модели
  X_train_balanced = np.array(X_train_pd.loc[list_c1_percent_id])[:,list_n_last_features]
  y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

  # освободим память от "тяжелых" и ненужных файлов
  del X_train_pd, y_train_pd
  gc.collect()

  # я не воспользовался параметром timeout метода optimize потому, что  
  # optimize отставливает поиск параметров, если время обучения 
  # превышает timeout. Мне нужно чтобы поиск параметров продолжился.
  try:
    # для модуля func_time_out упакуем обучение модели в функцию try_func
    def try_func():
            return optuna_random_forest.fit(X_train_balanced,y_train_balanced)
    # обучаем модель с ограничением по времени 10 минут (600 секунд)
    optuna_random_forest = func_timeout.func_timeout(600, try_func)

    # удаляем крупные файлы чтобы высвободить память 
    del X_train_balanced, y_train_balanced
    gc.collect()

    # подгружаем данные для тестирования
    X_train = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas().to_numpy()[:,list_n_last_features]
    y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
    X_valid = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_valid').to_pandas().to_numpy()[:,list_n_last_features]
    y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()

    # делаем предсказание на обучающем и валидационном наборе и считаем метрики
    roc_train = metrics.roc_auc_score(y_train, optuna_random_forest.predict_proba(X_train)[:,1])
    roc_valid = metrics.roc_auc_score(y_valid, optuna_random_forest.predict_proba(X_valid)[:,1])
    f1_score = metrics.f1_score(y_valid, optuna_random_forest.predict(X_valid))
    recall_1 = metrics.recall_score(y_valid, optuna_random_forest.predict(X_valid),average=None,zero_division=0)[1]
    precision_1 = metrics.precision_score(y_valid, optuna_random_forest.predict(X_valid),average=None,zero_division=0)[1]

    # удаляем крупные файлы чтобы высвободить память 
    del X_train, X_valid, y_train, y_valid 
    gc.collect()

  except func_timeout.FunctionTimedOut:
    # освободим память от "тяжелых" и ненужных файлов
    del X_train_balanced, y_train_balanced
    gc.collect()

    # обьявим метрики пустыми значениями
    roc_train = 0
    roc_valid = 0
    f1_score = 0
    recall_1 = 0
    precision_1 = 0
    pass

  return roc_train, roc_valid, f1_score, recall_1, precision_1

In [ ]:
# так как Random Forest склонен к переобучению 
# попробуем направить optimize в сторону уменьшения roc_train
# cоздаем объект исследования
optuna_study_rf = optuna.create_study(study_name='RandomForestClassifier_'+feature_space+'_torow', 
                               directions=['minimize','maximize','maximize','maximize','maximize'], 
                               sampler=optuna.samplers.TPESampler(),
                               pruner='Hyperband',
                               storage='sqlite:///optuna_studies.db',
                               load_if_exists=True)
# на случай удаления 
# optuna.delete_study(study_name='RandomForestClassifier_'+feature_space+'_torow', storage='sqlite:///optuna_studies.db')

In [ ]:
# ищем лучшую комбинацию гиперпараметров n_trials раз
optuna_study_rf.optimize(optuna_rf, n_trials=300)

In [ ]:
# из полученного результат соврмируем Data Frame
optuna_study_rf_pd = optuna_study_rf.trials_dataframe().dropna()
# переименуем столбы
optuna_study_rf_pd.rename(columns={
    'values_0': 'roc_train',
    'values_1': 'roc_valid',
    'values_2': 'f1_score',
    'values_3': 'recall_1',
    'values_4': 'precision_1'
},inplace=True)
optuna_study_rf_pd.sort_values(by='precision_1').drop(['datetime_start','datetime_complete','duration'],axis=1)

In [ ]:
# покаждем статистику обучения
print('Максимальное значение метрики ROC AUC на валидационном наборе:',round(optuna_study_rf_pd['roc_valid'].max(),3))
print('Среднее значение метрики ROC AUC на валидационном наборе:',round(optuna_study_rf_pd['roc_valid'].mean(),3))
print('Максимальное значение метрики f1_score на валидационном наборе:',round(optuna_study_rf_pd['f1_score'].max(),3))
print('Среднее значение метрики f1_score на валидационном наборе:',round(optuna_study_rf_pd['f1_score'].mean(),3))
print('Максимальное значение метрики recall_1 на валидационном наборе:',round(optuna_study_rf_pd['recall_1'].max(),3))
print('Среднее значение метрики recall_1 на валидационном наборе:',round(optuna_study_rf_pd['recall_1'].mean(),3))
print('Максимальное значение метрики precision_1 на валидационном наборе:',round(optuna_study_rf_pd['precision_1'].max(),3))
print('Среднее значение метрики precision_1 на валидационном наборе:',round(optuna_study_rf_pd['precision_1'].mean(),3))

In [ ]:
# построим график важности гиперпарметров
optuna.visualization.plot_param_importances(optuna_study_rf, target_name='Score')

In [ ]:
# Построим зависимость метрики precision_1

fig = px.scatter(
    data_frame=optuna_study_rf_pd,
    x='precision_1', #ось абсцисс
    y=['f1_score','recall_1'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision от recall', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision',
    yaxis_title='Величина метрики recall и f1-score')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики precision_1 от гиперпараметров

fig = px.scatter(
    data_frame=optuna_study_rf_pd,
    x='precision_1', #ось абсцисс
    y=['params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight','params_max_depth','params_max_features',
        'params_max_samples', 'params_min_samples_leaf', 'params_min_samples_split', 
        'params_n_estimators', 'params_n_last'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision класса 1 от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision на классе 1',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики f1_score от гиперпараметров

fig = px.scatter(
    data_frame=optuna_study_rf_pd,
    x='f1_score', #ось абсцисс
    y=['params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight','params_max_depth','params_max_features',
        'params_max_samples', 'params_min_samples_leaf', 'params_min_samples_split', 
        'params_n_estimators', 'params_n_last'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость f1-score от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики f1-score',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики roc_valid от гиперпараметров

fig = px.scatter(
    data_frame=optuna_study_rf_pd,
    x='roc_valid', #ось абсцисс
    y=['params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight','params_max_depth','params_max_features',
        'params_max_samples', 'params_min_samples_leaf', 'params_min_samples_split', 
        'params_n_estimators', 'params_n_last'],
)
fig.update_layout(
    title ={
        'text':'Зависимость ROC AUC на валидационном наборе от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики ROC AUC',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрик качества модели от number

fig = px.scatter(
    data_frame=optuna_study_rf_pd[(optuna_study_rf_pd['roc_valid']>=0.68) 
                                  & (optuna_study_rf_pd['precision_1']>0.1)
                                  & (optuna_study_rf_pd['recall_1']>0.1)],
    x='number', #ось абсцисс
    y=['roc_valid','roc_train','precision_1','recall_1'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость метрик качества модели от number', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики ROC Valid',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

<span style="color:Blue">

Вывод:  

Их всех выбираем точку $number =163$.   
В этой точке оптимальные значения метрик $recall_1$ и $precision_1$, а  
также относительно высокая метрика $ROC AUC$.   
Посмотрим каким значениям гиперпараметров соотвествует выбраннаяточка.

In [ ]:
# определим номер лучшге варианта
best_optuna_number = 163

# сформируем словарь лучших гипирпарметров RandomForestClassifier
best_param_rf = {
    'n_estimators' : int(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_n_estimators'].iloc[0]),
    'criterion': optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_criterion'].iloc[0],
    'max_depth' : optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_max_depth'].iloc[0],
    'min_samples_split': optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_min_samples_split'].iloc[0],
    'min_samples_leaf' : optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_min_samples_leaf'].iloc[0],
    'max_features': optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_max_features'].iloc[0],
    'max_samples' : optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_max_samples'].iloc[0],
    'class_weight' : {0:optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_class_0_weight'].iloc[0],
                      1:optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_class_1_weight'].iloc[0]}
    }
# определим перменные для лучших значений параметров
best_n_last = optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_n_last'].iloc[0]
best_class_1_percent = optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_class_1_percent'].iloc[0]
best_random_state = optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_random_state'].iloc[0]


# Выведем принятые наилучшие праметры
print('best n estimators:',best_param_rf['n_estimators'])
print('best criterion:',best_param_rf['criterion'])
print('best max depth:',best_param_rf['max_depth'])
print('best min samples split:',best_param_rf['min_samples_split'])
print('best min samples leaf:',best_param_rf['min_samples_leaf'])
print('best max features(%):',round(best_param_rf['max_features'],3))
print('best max samples(%):',round(best_param_rf['max_samples'],3))
print('best class 0 weight:',round(best_param_rf['class_weight'][0],3))
print('best class 1 weight:',round(best_param_rf['class_weight'][1],3))

print('best class 1 percent:',round(best_class_1_percent,3))
print('best n last:',best_n_last)
print('best random state:',best_random_state)
print('time for best train:',round(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['duration'].iloc[0].seconds/60),'minutes')

print()
print('ROC AUC на обучающем наборе:', round(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['roc_train'].iloc[0],3))
print('ROC AUC на валидационном наборе:', round(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['roc_valid'].iloc[0],3))
print('precision класса 1:', round(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['precision_1'].iloc[0],3))
print('recall класса 1:', round(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['recall_1'].iloc[0],3))

##### <span style="color:MediumSlateBlue">Обучение модели с лучшими параметрами</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'payments'
count_features = dict_spaces[feature_space]

# с помощью функции features_from_transform_data_torow извлечем  
# из данных transform_data_torow
# признаки сооствествующие n_last последним клиенским операциям
list_n_last_features = features_from_transform_data_torow(best_n_last,count_features,25)

# загружаем обучующие наборы
X_train_pd = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas()
y_train_pd = pd.read_csv('target/target_train.csv')

# поготовим данные y_train_pd к работе с функцией class_1_percent_samples
y_train_pd.set_index('id',drop=False,inplace=True)

# подготовим данные для обучения модели
# с помощью функции class_1_percent_samples зададим долю 
# класса 1
list_c1_percent_id = class_1_percent_samples(y_train_pd,best_class_1_percent,random_state=best_random_state)[:1000000]

# сформируем данные для обучения базовой модели
X_train_balanced = np.array(X_train_pd.loc[list_c1_percent_id])[:,list_n_last_features]
y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

# освободим память от "тяжелых" и ненужных файлов
del X_train_pd, y_train_pd
gc.collect()

# обучаем модель RandomForestClassifier с наилучшеми параметрами
random_forest = RandomForestClassifier(
        **best_param_rf,
        n_jobs=-1,
        random_state=best_random_state)
random_forest.fit(X_train_balanced, y_train_balanced)

# освободим память от "тяжелых" и ненужных файлов
del X_train_balanced, y_train_balanced
gc.collect()


# подгружаем данные для тестирования
X_train = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas().to_numpy()[:,list_n_last_features]
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
X_valid = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_valid').to_pandas().to_numpy()[:,list_n_last_features]
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()

# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_rf__pred_proba = random_forest.predict_proba(X_train)[:,1]
y_valid_rf_pred_proba = random_forest.predict_proba(X_valid)[:,1]

# Делаем предсказание для валидационной выборки
y_valid_rf_pred = random_forest.predict(X_valid)

# удаляем крупные файлы чтобы высвободить память 
del X_train, X_valid
gc.collect()

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_rf__pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_rf_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_valid,y_valid_rf_pred,zero_division=0))

# time: 2m 10s

##### <span style="color:MediumSlateBlue">Анализ влияния порога классификации на качество модели</span>

In [ ]:
# чтобы проше обращаться к каждому обьекту в данных
# преобразуем y_valid_LR_pred к Series типу
y_valid_rf_pred_proba = pd.Series(y_valid_rf_pred_proba)

# формируем список из необходимых значений thresholds
thresholds = np.arange(0.01, 1, 0.01)

# сформируем списко для заполнения данными результатов анализа
threshold_data = []


for threshold in thresholds:
        # сформируем список для заполнения данными текушего threshold
        threshold_current_data = [threshold]

        #клиентов, для которых вероятность дефолта > threshold, относим к классу 1
        #в противном случае — к классу 0
        y_valid_rf_pred = y_valid_rf_pred_proba.apply(lambda x: 1 if x>threshold else 0)


        #Считаем метрики для класса 0 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(y_valid, y_valid_rf_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.precision_score(y_valid, y_valid_rf_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.f1_score(y_valid, y_valid_rf_pred,average=None,zero_division=0)[0],3))

        #Считаем метрики для класса 1 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(y_valid, y_valid_rf_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.precision_score(y_valid, y_valid_rf_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.f1_score(y_valid, y_valid_rf_pred,average=None,zero_division=0)[1],3))

        # добавляем полученные данные в список threshold_data
        threshold_data.append(threshold_current_data)

        # из полученных данных сформируем dataframe
        thresholds_pd =pd.DataFrame(
        columns=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'],
        data = threshold_data)

# time: 17s

In [ ]:
# Визуализируем метрики на валидационном наборе при различных threshold
fig_threshold = px.line(
    data_frame=thresholds_pd,
    x='threshold', #ось абсцисс
    y=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'], #ось ординат
)
fig_threshold.update_layout(
    title ={
        'text':'Зависимость качества модели от threshold', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Порог классификации threshold',
    yaxis_title='Величина метрики')
fig_threshold.update_xaxes(showspikes=True)
fig_threshold.update_yaxes(showspikes=True)

fig_threshold.show()

Метрика $precision$ показывает на сколько сильно ошиблась модель при определиние класса.  
Чем выше метрика, тем меньше ошиблась модель. 

$$precision = \frac{TP}{TP+FP}$$

Метрика $recall$ показывает способность модели найти класс.  
Чем выше метрика, тем выше способность модели находить класс.  

$$recall = \frac{TP}{TP+FN}$$

Метрика $f1-score$ показывает баланс между $precision$ и $recall$ 

$threshold$- порог класификации модели.  
Все объекты для которых $у_{\text{proba pred}} < threshold$, модель относит к класс 0.  
Соотвественно, если $у_{\text{proba pred}}  > threshold$ модель относит обьект к классу 1.  
Для идельаной модели $threshold$, является границей между областями класса 0 и класса 1.  

<span style="color:Blue">

Выводы по результам анализа:  

Для улучшения качества модели по метрике $ROC AUC$, нет смысла сдвигать порог   
классификации. Но зависимость метрик качества модели от $threshold$, может рассказать о  
способности модели искать класс 1 и класс 0.

При фокусировке модели на классе 1 (более низкий $threshold$):  
- на классе 1 метрика $precision$ уменьшается, метрика $recall$ растет;  
- на классе 0 метрика $precision$ растет, метрика $recall$ уменьшается.  

При фокусировке модели на классе 0 (более высокий $threshold$):  
- на классе 1 метрика $precision$ увеличивается, метрика $recall$ уменьшается;  
- на классе 0 метрика $precision$ уменьшается, метрика $recall$ растет.  

Обьсняется тем, что в условия дисбаланса модель научилась хорошо определять класс 0 и плохо класс 1.  

При фокусировке модели на классе 1 (более низкий $threshold$), все больше обьектов отнесены к классу 1.   
Так как $precision$ класса 1 уменьшается, значит в области класса 1 стало больше обектов класса 0,  
что явлется закономерным для "идеальной" модели.  
Однако при этом, $recall$ класса 1 растет, значит в добавленных обьектах также присутвует класс 1.  
Это как раз указывает на то, что модель не нучилась находить среди класса 0 класс 1. В области с     
высокой вероятностью класса 0 (более низкий $threshold$), находятся обьекты класса 1.  

Фокусировка на классе 0 (более высокий $threshold$), заставляет модель все больше обьектов из области класса 1  
относить к классу 0. При этом увеличение метрики $precision$ класса 1 говорит о том, что в области класса 1     
становится меньше 0, а значит модель умеет среди класса 1, находить класс 0.  
После  $threshold=0.65$ модель впринципе теряет способность отличать класс 1. У модели нет обьектов класса 1   
в которых она "уверена" на 70+%.

Обратим внимание на $threshold=0.38$, в этой точке находится баланс межу метриками качества модели:
$$precision_1 = recall_1 = \text{f1-score}_1 = 0.096$$
$$precision_0 = recall_0 = \text{f1-score}_0 = 0.966$$

Для модели с лучшим качеством эти две точки должны находится максимально близко к друг другу и иметь   
высокое значение метрик $precision_1$ и $recall_1$.  
Для класса 0 это условие выполнется, что также говорит о том, что модель научилась искать этот класс.    
Для класса 1 это условие не выполнется, что также говорит о том, что модель плохо ищет класс 1.

##### <span style="color:MediumSlateBlue">Построение ROC-кривой выбранной модели на тестовом наборе</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'payments'
count_features = dict_spaces[feature_space]

In [ ]:
# подгружаем данные для тестирования
X_train = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas().to_numpy()[:,list_n_last_features]
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
X_valid = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_valid').to_pandas().to_numpy()[:,list_n_last_features]
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()
X_test = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_test').to_pandas().to_numpy()[:,list_n_last_features]
y_test = pd.read_csv('target/target_test.csv')['flag'].to_numpy()

In [ ]:
print("Размеры обучающего набора",X_train.shape,y_train.shape)
print("Размеры валидационного набора",X_valid.shape,y_valid.shape)
print("Размеры тестового набора",X_test.shape,y_test.shape)
# time: 1m 43s

In [ ]:
# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_rf_pred_proba = random_forest.predict_proba(X_train)[:,1]
y_valid_rf_pred_proba = random_forest.predict_proba(X_valid)[:,1]
y_test_rf_pred_proba = random_forest.predict_proba(X_test)[:,1]

# Делаем предсказание для валидационной выборки
y_test_lr_pred = random_forest.predict(X_test)

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_rf_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_rf_pred_proba),3))
print('ROC AUC на тестовом наборе', round(metrics.roc_auc_score(y_test, y_test_rf_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_test,y_test_lr_pred,zero_division=0))

In [ ]:
# Для расчета значений TPR и FPR используем функцию roc_curve
fpr_train, tpr_train, _ = metrics.roc_curve(y_train, y_train_rf_pred_proba)
fpr_valid, tpr_valid, _ = metrics.roc_curve(y_valid, y_valid_rf_pred_proba)
fpr_test, tpr_test, _ = metrics.roc_curve(y_test, y_test_rf_pred_proba)

# рассчитаем площадь под кривой
auc_train = round(metrics.roc_auc_score(y_train, y_train_rf_pred_proba),3)
auc_valid = round(metrics.roc_auc_score(y_valid, y_valid_rf_pred_proba),3)
auc_test = round(metrics.roc_auc_score(y_test, y_test_rf_pred_proba),3)

trace_train = go.Scatter(x=fpr_train, y=tpr_train, mode='lines', name='AUC_train = %0.2f' % auc_train,
                   line=dict(color='red', width=2),
                   hovertemplate='train<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_train])

trace_valid = go.Scatter(x=fpr_valid, y=tpr_valid, mode='lines', name='AUC_valid = %0.2f' % auc_valid,
                   line=dict(color='green', width=2),
                   hovertemplate='valid<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_valid])

trace_test = go.Scatter(x=fpr_test, y=tpr_test, mode='lines', name='AUC_test = %0.2f' % auc_test,
                   line=dict(color='darkorange', width=2),
                   hovertemplate='test<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_test])


reference_line = go.Scatter(x=[0,1], y=[0,1], mode='lines', name='Reference Line',
                            line=dict(color='navy', width=2, dash='dash'))

fig_roc = go.Figure(data=[trace_train,trace_valid,trace_test,reference_line])

fig_roc.update_layout(
    title ={
        'text':'ROC-кривая', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 1350,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate')

fig_roc.update_xaxes(showspikes=True)
fig_roc.update_yaxes(showspikes=True)


fig_roc.show()

<span style="color:Blue">

Вывод по ROC-кривой:
1. Модель выдает сталибильное качество на валидационном и тестовом наборах, а значит  
не переобучена - разброс ($variance$) не большой. Хотя, качество модели на обучающем наборе,  
заметно лучше.
2. Смещение ($bias$) модели лучше, чем у модели $Logicstic$ $Regression$ $lassifier$, обученной   
на тех же данных.  
На тестовом наборе полученно качество $ROC AUC = 0.69$, против $ROC AUC = 0.64$

#### <span style="color:MediumBlue">Построение модели Gradient Boosting Classifier</span>

##### <span style="color:MediumSlateBlue">Baseline обучение модели Hist Gradient Boosting Classifier</span>

In [ ]:
# обучаем модель Gradient Boosting Classifier
gradient_boosting = HistGradientBoostingClassifier(random_state=42)
gradient_boosting.fit(np.array(X_train_balanced)[:,list_n_last_features],np.array(y_train_balanced))

# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_rf__pred_proba = gradient_boosting.predict_proba(X_train[:,list_n_last_features])[:,1]
y_valid_rf_pred_proba = gradient_boosting.predict_proba(X_valid[:,list_n_last_features])[:,1]

# Делаем предсказание для валидационной выборки
y_valid_rf_pred = gradient_boosting.predict(X_valid[:,list_n_last_features])

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_rf__pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_rf_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_valid,y_valid_rf_pred))

# time: 

##### <span style="color:MediumSlateBlue">Подбор гиперпараметров модели</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'payments'
count_features = dict_spaces[feature_space]

# настроим оптимизацию гипер параметров
def optuna_hgbc(trial):
  # задаем пространства поиска гиперпараметров
  learning_rate = trial.suggest_float('learning_rate',0.2,0.6)
  max_iter = trial.suggest_int('max_iter', 50, 300,step = 1)
  max_leaf_nodes = trial.suggest_int('max_leaf_nodes', 2, 50,step = 1)
  max_depth = trial.suggest_int('max_depth', 1, 15,step = 1)
  min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 50,step = 1)
  max_features = trial.suggest_float('max_features',0.1,1)
  l2_regularization = trial.suggest_float('l2_regularization',0.01,1)
  class_0_weight = trial.suggest_float('class_0_weight',0.01,1)
  class_1_weight = trial.suggest_float('class_1_weight',0.01,1)
  n_last = trial.suggest_int('n_last', 5, 25,step = 1)
  class_1_percent = trial.suggest_float('class_1_percent',0.01,1)
  random_state = trial.suggest_int('random_state', 1, 1000000)


  # создаем модель
  optuna_gradient_boosting = HistGradientBoostingClassifier(
      learning_rate= learning_rate,
      max_iter = max_iter,
      max_leaf_nodes =max_leaf_nodes,
      max_depth = max_depth,
      min_samples_leaf = min_samples_leaf,
      max_features=max_features,
      l2_regularization = l2_regularization,
      class_weight={0:class_0_weight,1:class_1_weight},
      random_state=42)

  # с помощью функции features_from_transform_data_torow извлечем  
  # из данных transform_data_torow
  # признаки сооствествующие n_last последним клиенским операциям
  list_n_last_features = features_from_transform_data_torow(n_last,count_features,25)

  # загружаем обучующие наборы
  X_train_pd = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas()
  y_train_pd = pd.read_csv('target/target_train.csv')

  # поготовим данные y_train_pd к работе с функцией class_1_percent_samples
  y_train_pd.set_index('id',drop=False,inplace=True)

  # подготовим данные для обучения модели
  # с помощью функции class_1_percent_samples зададим долю класса 1
  list_c1_percent_id = class_1_percent_samples(y_train_pd,class_1_percent,random_state=random_state)[:1000000]

  # сформируем данные для обучения модели
  X_train_balanced = np.array(X_train_pd.loc[list_c1_percent_id])[:,list_n_last_features]
  y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

  # освободим память от "тяжелых" и ненужных файлов
  del X_train_pd, y_train_pd
  gc.collect()

  # я не воспользовался параметром timeout метода optimize потому, что  
  # optimize отставливает поиск параметров, если время обучения 
  # превышает timeout. Мне нужно чтобы поиск параметров продолжился.
  try:
    # для модуля func_time_out упакуем обучение модели в функцию try_func
    def try_func():
            return optuna_gradient_boosting.fit(X_train_balanced,y_train_balanced)
    # обучаем модель с ограничением по времени 10 минут (600 секунд)
    optuna_gradient_boosting = func_timeout.func_timeout(600, try_func)

    # удаляем крупные файлы чтобы высвободить память 
    del X_train_balanced, y_train_balanced
    gc.collect()

    # подгружаем данные для тестирования
    X_train = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas().to_numpy()[:,list_n_last_features]
    y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
    X_valid = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_valid').to_pandas().to_numpy()[:,list_n_last_features]
    y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()

    # делаем предсказание на обучающем и валидационном наборе и считаем метрики
    roc_train = metrics.roc_auc_score(y_train, optuna_gradient_boosting.predict_proba(X_train)[:,1])
    roc_valid = metrics.roc_auc_score(y_valid, optuna_gradient_boosting.predict_proba(X_valid)[:,1])
    f1_score = metrics.f1_score(y_valid, optuna_gradient_boosting.predict(X_valid))
    recall_1 = metrics.recall_score(y_valid, optuna_gradient_boosting.predict(X_valid),average=None,zero_division=0)[1]
    precision_1 = metrics.precision_score(y_valid, optuna_gradient_boosting.predict(X_valid),average=None,zero_division=0)[1]

    # удаляем крупные файлы чтобы высвободить память 
    del X_train, X_valid, y_train, y_valid 
    gc.collect()

  except func_timeout.FunctionTimedOut:
    # освободим память от "тяжелых" и ненужных файлов
    del X_train_balanced, y_train_balanced
    gc.collect()

    # обьявим метрики пустыми значениями
    roc_train = 0
    roc_valid = 0
    f1_score = 0
    recall_1 = 0
    precision_1 = 0
    pass

  return roc_train, roc_valid, f1_score, recall_1, precision_1

In [ ]:
# cоздаем объект исследования
# чтобы модель не переобучалась минимизируем roc_train 
# и максимизируем roc_valid
optuna_study_hsbs = optuna.create_study(study_name='HistGradientBoostingClassifier_'+feature_space+'_torow', 
                               directions=['maximize','maximize','maximize','maximize','maximize'], 
                               sampler=optuna.samplers.TPESampler(),
                               pruner='Hyperband',
                               storage='sqlite:///optuna_studies.db',
                               load_if_exists=True)

# ну случай удаления обучения
# optuna.delete_study(study_name='HistGradientBoostingClassifier_'+feature_space+'_torow', storage='sqlite:///optuna_studies.db')

In [ ]:
# ищем лучшую комбинацию гиперпараметров n_trials раз
optuna_study_hsbs.optimize(optuna_hgbc, n_trials=300)

In [ ]:
# из полученного результат соврмируем Data Frame
optuna_study_hsbs_pd = optuna_study_hsbs.trials_dataframe()
# переименуем столбы
optuna_study_hsbs_pd.rename(columns={
    'values_0': 'roc_train',
    'values_1': 'roc_valid',
    'values_2': 'f1_score',
    'values_3': 'recall_1',
    'values_4': 'precision_1'
},inplace=True)
# Выделим время потраченное на обучение модели
last_trail = optuna_study_hsbs_pd['number'].max()
optuna_study_hsbs_pd.tail(5)


In [ ]:
# покаждем статистику обучения
print('Максимальное значение метрики ROC AUC на валидационном наборе:',round(optuna_study_hsbs_pd['roc_valid'].max(),3))
print('Среднее значение метрики ROC AUC на валидационном наборе:',round(optuna_study_hsbs_pd['roc_valid'].mean(),3))
print('Максимальное значение метрики f1_score на валидационном наборе:',round(optuna_study_hsbs_pd['f1_score'].max(),3))
print('Среднее значение метрики f1_score на валидационном наборе:',round(optuna_study_hsbs_pd['f1_score'].mean(),3))
print('Максимальное значение метрики recall_1 на валидационном наборе:',round(optuna_study_hsbs_pd['recall_1'].max(),3))
print('Среднее значение метрики recall_1 на валидационном наборе:',round(optuna_study_hsbs_pd['recall_1'].mean(),3))
print('Максимальное значение метрики precision_1 на валидационном наборе:',round(optuna_study_hsbs_pd['precision_1'].max(),3))
print('Среднее значение метрики precision_1 на валидационном наборе:',round(optuna_study_hsbs_pd['precision_1'].mean(),3))

In [ ]:
# построим график важности гиперпарметров
optuna.visualization.plot_param_importances(optuna_study_hsbs, target_name='Score')

In [ ]:
# Построим зависимость метрики precision_1
# max_iter

fig = px.scatter(
    data_frame=optuna_study_hsbs_pd,
    x='precision_1', #ось абсцисс
    y=['f1_score', 'recall_1'
    ], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision от recall', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision',
    yaxis_title='Величина метрики recall и f1-score')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики precision_1 от гиперпараметров
fig = px.scatter(
    data_frame=optuna_study_hsbs_pd,
    x='precision_1', #ось абсцисс
    y=['params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight', 'params_l2_regularization',
       'params_learning_rate', 'params_max_depth', 'params_max_features',
       'params_max_iter', 'params_max_leaf_nodes', 'params_min_samples_leaf',
       'params_n_last'
    ], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision',
    yaxis_title='Величина параметров')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики f1_score от гиперпараметров
fig = px.scatter(
    data_frame=optuna_study_hsbs_pd,
    x='f1_score', #ось абсцисс
    y=['params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight', 'params_l2_regularization',
       'params_learning_rate', 'params_max_depth', 'params_max_features',
       'params_max_iter', 'params_max_leaf_nodes', 'params_min_samples_leaf',
       'params_n_last'
    ], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость f1-score от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики f1-score',
    yaxis_title='Величина параметров')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики roc_valid от гиперпараметров
fig = px.scatter(
    data_frame=optuna_study_hsbs_pd,
    x='roc_valid', #ось абсцисс
    y=['params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight', 'params_l2_regularization',
       'params_learning_rate', 'params_max_depth', 'params_max_features',
       'params_max_iter', 'params_max_leaf_nodes', 'params_min_samples_leaf',
       'params_n_last'
    ], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость roc valid от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики roc valid',
    yaxis_title='Величина параметров')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрик качества модели от number

fig = px.scatter(
    data_frame=optuna_study_hsbs_pd[(optuna_study_hsbs_pd['recall_1']>0.11)&
                                    (optuna_study_hsbs_pd['precision_1']>0.11)&
                                    (optuna_study_hsbs_pd['roc_valid']>0.68)],
    x='number', #ось абсцисс
    y=['roc_train','roc_valid','recall_1','precision_1'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision_1',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

<span style="color:Blue">

Вывод:  

Их всех выбираем точку $number =161$.   
В этой точке оптимальные значения метрик $recall_1$ и $precision_1$, а  
также относительно высокая метрика $ROC AUC$.   
Посмотрим каким значениям гиперпараметров соотвествует выбранная точка.

In [ ]:
# определим номер лучшге варианта
best_optuna_number = 161

# сформируем словарь лучших гипирпарметров RandomForestClassifier
best_param_rf = {
    'learning_rate' : optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['params_learning_rate'].iloc[0],
    'max_iter' : optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['params_max_iter'].iloc[0],
    'max_leaf_nodes' : optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['params_max_leaf_nodes'].iloc[0],
    'max_depth' : optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['params_max_depth'].iloc[0],
    'min_samples_leaf' : optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['params_min_samples_leaf'].iloc[0],
    'max_features' : optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['params_max_features'].iloc[0],
    'l2_regularization' : optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['params_l2_regularization'].iloc[0],
    'class_weight' : {0: optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['params_class_0_weight'].iloc[0],
                      1: optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['params_class_1_weight'].iloc[0],}
    }

# определим перменные для лучших значений параметров
best_n_last = optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['params_n_last'].iloc[0]
best_class_1_percent = optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['params_class_1_percent'].iloc[0]
best_random_state = optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['params_random_state'].iloc[0]


# Выведем принятые наилучшие праметры
print('best learning rate:',round(best_param_rf['learning_rate'],3))
print('best max iter:',best_param_rf['max_iter'])
print('best max leaf nodes:',best_param_rf['max_leaf_nodes'])
print('best max depth:',best_param_rf['max_depth'])
print('best min samples leaf:',best_param_rf['min_samples_leaf'])
print('best max features:',round(best_param_rf['max_features'],3))
print('best l2 regularization:',round(best_param_rf['l2_regularization'],3))
print('best class 0 weight:',round(best_param_rf['class_weight'][0],3))
print('best class 1 weight:',round(best_param_rf['class_weight'][1],3))

print('best class 1 percent:',round(best_class_1_percent,3))
print('best n last:',best_n_last)
print('best random state:',best_random_state)
print('time for best train:',round(optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['duration'].iloc[0].seconds/60),'minutes')

print()
print('ROC AUC на обучающем наборе:', round(optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['roc_train'].iloc[0],3))
print('ROC AUC на валидационном наборе:', round(optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['roc_valid'].iloc[0],3))
print('precision класса 1:', round(optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['precision_1'].iloc[0],3))
print('recall класса 1:', round(optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['recall_1'].iloc[0],3))

##### <span style="color:MediumSlateBlue">Обучение модели с лучшими параметрами</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'payments'
count_features = dict_spaces[feature_space]

# с помощью функции features_from_transform_data_torow извлечем  
# из данных transform_data_torow
# признаки сооствествующие n_last последним клиенским операциям
list_n_last_features = features_from_transform_data_torow(best_n_last,count_features,25)

# загружаем обучующие наборы
X_train_pd = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas()
y_train_pd = pd.read_csv('target/target_train.csv')

# поготовим данные y_train_pd к работе с функцией class_1_percent_samples
y_train_pd.set_index('id',drop=False,inplace=True)

# подготовим данные для обучения модели
# с помощью функции class_1_percent_samples зададим долю класса 1
list_c1_percent_id = class_1_percent_samples(y_train_pd,best_class_1_percent,random_state=best_random_state)[:1000000]

# сформируем данные для обучения модели
X_train_balanced = np.array(X_train_pd.loc[list_c1_percent_id])[:,list_n_last_features]
y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

# освободим память от "тяжелых" и ненужных файлов
del X_train_pd, y_train_pd
gc.collect()


# обучаем модель HistGradientBoostingClassifier с наилучшеми параметрами
gradient_boosting = HistGradientBoostingClassifier(
        **best_param_rf,
        random_state=42)
gradient_boosting.fit(X_train_balanced,y_train_balanced)

# освободим память от "тяжелых" и ненужных файлов
del X_train_balanced, y_train_balanced
gc.collect()

# подгружаем данные для тестирования
X_train = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas().to_numpy()[:,list_n_last_features]
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
X_valid = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_valid').to_pandas().to_numpy()[:,list_n_last_features]
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()

# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_gb__pred_proba = gradient_boosting.predict_proba(X_train)[:,1]
y_valid_gb_pred_proba = gradient_boosting.predict_proba(X_valid)[:,1]

# Делаем предсказание для валидационной выборки
y_valid_gb_pred = gradient_boosting.predict(X_valid)

# освободим память от "тяжелых" и ненужных файлов
del X_train, X_valid
gc.collect()

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_gb__pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_gb_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_valid,y_valid_gb_pred,zero_division=0))

# time: 2m 30s

##### <span style="color:MediumSlateBlue">Анализ влияния порога классификации на качество модели</span>

In [ ]:
# чтобы проше обращаться к каждому обьекту в данных
# преобразуем y_valid_LR_pred к Series типу
y_valid_gb_pred_proba = pd.Series(y_valid_gb_pred_proba)

# формируем список из необходимых значений thresholds
thresholds = np.arange(0.01, 1, 0.01)

# сформируем списко для заполнения данными результатов анализа
threshold_data = []


for threshold in thresholds:
        # сформируем список для заполнения данными текушего threshold
        threshold_current_data = [threshold]

        #клиентов, для которых вероятность дефолта > threshold, относим к классу 1
        #в противном случае — к классу 0
        y_valid_gb_pred = y_valid_gb_pred_proba.apply(lambda x: 1 if x>threshold else 0)


        #Считаем метрики для класса 0 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(y_valid, y_valid_gb_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.precision_score(y_valid, y_valid_gb_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.f1_score(y_valid, y_valid_gb_pred,average=None,zero_division=0)[0],3))

        #Считаем метрики для класса 1 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(y_valid, y_valid_gb_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.precision_score(y_valid, y_valid_gb_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.f1_score(y_valid, y_valid_gb_pred,average=None,zero_division=0)[1],3))

        # добавляем полученные данные в список threshold_data
        threshold_data.append(threshold_current_data)

        # из полученных данных сформируем dataframe
        thresholds_pd =pd.DataFrame(
        columns=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'],
        data = threshold_data)

# time: 17s

In [ ]:
# Визуализируем метрики на валидационном наборе при различных threshold
fig_threshold = px.line(
    data_frame=thresholds_pd,
    x='threshold', #ось абсцисс
    y=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'], #ось ординат
)
fig_threshold.update_layout(
    title ={
        'text':'Зависимость качества модели от threshold', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Порог классификации threshold',
    yaxis_title='Величина метрики')
fig_threshold.update_xaxes(showspikes=True)
fig_threshold.update_yaxes(showspikes=True)

fig_threshold.show()

Метрика $precision$ показывает на сколько сильно ошиблась модель при определиние класса.  
Чем выше метрика, тем меньше ошиблась модель. 

$$precision = \frac{TP}{TP+FP}$$

Метрика $recall$ показывает способность модели найти класс.  
Чем выше метрика, тем выше способность модели находить класс.  

$$recall = \frac{TP}{TP+FN}$$

Метрика $f1-score$ показывает баланс между $precision$ и $recall$ 

$threshold$- порог класификации модели.  
Все объекты для которых $у_{\text{proba pred}} < threshold$, модель относит к класс 0.  
Соотвественно, если $у_{\text{proba pred}}  > threshold$ модель относит обьект к классу 1.  
Для идельаной модели $threshold$, является границей между областями класса 0 и класса 1.  

<span style="color:Blue">

Выводы по результам анализа:  

Для улучшения качества модели по метрике $ROC AUC$, нет смысла сдвигать порог   
классификации. Но зависимость метрик качества модели от $threshold$, может рассказать о  
способности модели искать класс 1 и класс 0.

При фокусировке модели на классе 1 (более низкий $threshold$):  
- на классе 1 метрика $precision$ уменьшается, метрика $recall$ растет;  
- на классе 0 метрика $precision$ растет, метрика $recall$ уменьшается.  

При фокусировке модели на классе 0 (более высокий $threshold$):  
- на классе 1 метрика $precision$ увеличивается, метрика $recall$ уменьшается;  
- на классе 0 метрика $precision$ уменьшается, метрика $recall$ растет.  

Обьсняется тем, что в условия дисбаланса модель научилась хорошо определять класс 0 и плохо класс 1.  

При фокусировке модели на классе 1 (более низкий $threshold$), все больше обьектов отнесены к классу 1.   
Так как $precision$ класса 1 уменьшается, значит в области класса 1 стало больше обектов класса 0,  
что явлется закономерным для "идеальной" модели.  
Однако при этом, $recall$ класса 1 растет, значит в добавленных обьектах также присутвует класс 1.  
Это как раз указывает на то, что модель не нучилась находить среди класса 0 класс 1. В области с     
высокой вероятностью класса 0 (более низкий $threshold$), находятся обьекты класса 1.  

Фокусировка на классе 0 (более высокий $threshold$), заставляет модель все больше обьектов из области класса 1  
относить к классу 0. При этом увеличение метрики $precision$ класса 1 говорит о том, что в области класса 1     
становится меньше 0, а значит модель умеет среди класса 1, находить класс 0.  
После  $threshold=0.83$ модель впринципе теряет способность отличать класс 1. У модели нет обьектов класса 1   
в которых она "уверена" на 95+%.

Обратим внимание на $threshold=0.51$, в этой точке находится баланс межу метриками качества модели:
$$precision_1 = recall_1 = \text{f1-score}_1 = 0.12$$
$$precision_0 = recall_0 = \text{f1-score}_0 = 0.966$$

Для модели с лучшим качеством эти две точки должны находится максимально близко к друг другу и иметь   
высокое значение метрик $precision_1$ и $recall_1$.  
Для класса 0 это условие выполнется, что также говорит о том, что модель научилась искать этот класс.    
Для класса 1 это условие не выполнется, что также говорит о том, что модель плохо ищет класс 1.

##### <span style="color:MediumSlateBlue">Построение ROC-кривой выбранной модели на тестовом наборе</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'payments'
count_features = dict_spaces[feature_space]

In [ ]:
# подгружаем данные для тестирования
X_train = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas().to_numpy()[:,list_n_last_features]
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
X_valid = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_valid').to_pandas().to_numpy()[:,list_n_last_features]
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()
X_test = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_test').to_pandas().to_numpy()[:,list_n_last_features]
y_test = pd.read_csv('target/target_test.csv')['flag'].to_numpy()

In [ ]:
print("Размеры обучающего набора",X_train.shape,y_train.shape)
print("Размеры валидационного набора",X_valid.shape,y_valid.shape)
print("Размеры тестового набора",X_test.shape,y_test.shape)
# time: 1m 43s

In [ ]:
# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_gb_pred_proba = gradient_boosting.predict_proba(X_train)[:,1]
y_valid_gb_pred_proba = gradient_boosting.predict_proba(X_valid)[:,1]
y_test_gb_pred_proba = gradient_boosting.predict_proba(X_test)[:,1]

# Делаем предсказание для валидационной выборки
y_test_gb_pred = gradient_boosting.predict(X_test)

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_gb_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_gb_pred_proba),3))
print('ROC AUC на тестовом наборе', round(metrics.roc_auc_score(y_test, y_test_gb_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_test,y_test_gb_pred,zero_division=0))

In [ ]:
# Для расчета значений TPR и FPR используем функцию roc_curve
fpr_train, tpr_train, _ = metrics.roc_curve(y_train, y_train_gb_pred_proba)
fpr_valid, tpr_valid, _ = metrics.roc_curve(y_valid, y_valid_gb_pred_proba)
fpr_test, tpr_test, _ = metrics.roc_curve(y_test, y_test_gb_pred_proba)

# рассчитаем площадь под кривой
auc_train = round(metrics.roc_auc_score(y_train, y_train_gb_pred_proba),3)
auc_valid = round(metrics.roc_auc_score(y_valid, y_valid_gb_pred_proba),3)
auc_test = round(metrics.roc_auc_score(y_test, y_test_gb_pred_proba),3)

trace_train = go.Scatter(x=fpr_train, y=tpr_train, mode='lines', name='AUC_train = %0.2f' % auc_train,
                   line=dict(color='red', width=2),
                   hovertemplate='train<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_train])

trace_valid = go.Scatter(x=fpr_valid, y=tpr_valid, mode='lines', name='AUC_valid = %0.2f' % auc_valid,
                   line=dict(color='green', width=2),
                   hovertemplate='valid<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_valid])

trace_test = go.Scatter(x=fpr_test, y=tpr_test, mode='lines', name='AUC_test = %0.2f' % auc_test,
                   line=dict(color='darkorange', width=2),
                   hovertemplate='test<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_test])


reference_line = go.Scatter(x=[0,1], y=[0,1], mode='lines', name='Reference Line',
                            line=dict(color='navy', width=2, dash='dash'))

fig_roc = go.Figure(data=[trace_train,trace_valid,trace_test,reference_line])

fig_roc.update_layout(
    title ={
        'text':'ROC-кривая', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 1350,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate')

fig_roc.update_xaxes(showspikes=True)
fig_roc.update_yaxes(showspikes=True)


fig_roc.show()

<span style="color:Blue">

Вывод по ROC-кривой:
1. Модель выдает сталибильное качество на валидационном и тестовом наборах, а значит  
не переобучена - разброс ($variance$) не большой. Хотя, качество модели на обучающем наборе,  
заметно лучше.
2. Смещение ($bias$) модели такое же как и у модели $Random$ $Forest$ $Classifier$,  
обученной на тех же данных.  
На тестовом наборе полученно качество $ROC AUC = 0.69$.

### <span style="color:RoyalBlue">Построение модели на сбалансированных данных подпространства service torow</span>

#### <span style="color:MediumBlue">Формирование данных для обучения</span>

Анализ исходных данных, в частности target, показал что представленные данные   
имееют перекос: 96% клиентов у которых не случился дефолт (flag = 0),    
против 4 % - для которых произошел дефолт (flag = 1).  
С помощью функции class_1_percent_samples, сформируем сбаланисрованную выборку  
для обучения моделей.
За основу сбалансировных данных возьмем данные клиентов с дефолтом  
и к ним будем добирать необходимое количество клиентов до сбалансированности. 


Функция class_1_percent_samples принемает две переменные:
- target_data - массив с id и целевой переменной
- class_1_percent - процент класса 1 в результирующем массиве

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'service'
count_features = dict_spaces[feature_space]

In [ ]:
# сформируем данные для анализа сбалансированности обучающих данных
X_train_pd = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas()
y_train_pd = pd.read_csv('target/target_train.csv')

In [ ]:
# поготовим данные y_train_pd к работе с функцией class_1_percent_samples
y_train_pd.set_index('id',drop=False,inplace=True)
y_train_pd.head(4)

In [ ]:
# взглянем на сблансиорованность данных в y_train
y_train_pd['flag'].value_counts(normalize=True)

In [ ]:
# сбалансируем данные в y_train_pd
list_c1_percent_id = class_1_percent_samples(y_train_pd,0.5)
# list_c1_percent_features - список 'id' из y_train_pd соотвествующих  
# заданному проценту класса 1
len(list_c1_percent_id)

In [ ]:
# проверим сбалансированность полученного y_train
y_train_pd.loc[list_c1_percent_id]['flag'].value_counts(normalize=True)

*transform_data_torow*, показываеются последние N операций клиента.  
В данных в *date_torow*, собраны последние 25 операций клиентов.

Предварительный анализ показывает, что медиальное значение количества клиентских  
операций равно 7. Поэтому, для начала, будем показывать моделе $n_{last} = 7$   
последних клиентских операций.

Для извлечения из *transform_data_torow_25* необходимого количества последних клиенских  
операций ($n_{last}$) используем функцию features_from_transform_data_torow.

Функция features_from_transform_data_torow примает следующие аргументы:
- $n_{last} = 7$ - необходимое количество последних клиентских операций; 
- $n_{groups} = 8$ - количество признаков в $date$ $features$;
- $N_{last} = 25$ - количество $n_{last}$ операций, показанных в transform_data_torow_25


In [ ]:
# с помощью функции features_from_transform_data_torow извлечем  
# из данных transform_data_torow_25
# признаки сооствествующие 7 последним клиенским операциям
list_n_last_features = features_from_transform_data_torow(7,count_features,25)

In [ ]:
# подготовим данные для обучения
X_train_balanced = X_train_pd.loc[list_c1_percent_id]
y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag']
# сформируем данные для проверики модели
X_train = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas().to_numpy()
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
X_valid = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_valid').to_pandas().to_numpy()
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()

In [ ]:
# посмотрим на структуру данных в X_train_balanced
X_train_balanced.head(2)

In [ ]:
# посмотрим на структуру данных в y_train_balanced
y_train_balanced.head(2)

In [ ]:
# проверим что выборки действительно совпадают по id
X_train_balanced.index.to_list() == y_train_balanced.index.to_list()

#### <span style="color:MediumBlue">Построение модели Logistic Regression</span>

##### <span style="color:MediumSlateBlue">Baseline обучение модели LogisticRegression</span>

In [ ]:
# обучаем модель логистической регрессии
logistic_regression = linear_model.LogisticRegression(random_state=42, max_iter=10000)
logistic_regression.fit(np.array(X_train_balanced)[:,list_n_last_features],np.array(y_train_balanced))

# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_LR_pred_proba = logistic_regression.predict_proba(X_train[:,list_n_last_features])[:,1]
y_valid_LR_pred_proba = logistic_regression.predict_proba(X_valid[:,list_n_last_features])[:,1]

# Делаем предсказание для валидационной выборки
y_valid_LR_pred = logistic_regression.predict(X_valid[:,list_n_last_features])

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_LR_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_LR_pred_proba),3))

print('Основные метрики на тестовом наборе:')
print(metrics.classification_report(y_valid,y_valid_LR_pred))

# time: 1m 12s

##### <span style="color:MediumSlateBlue">Анализ влияния scaler преобразования на качество и скорость схождения модели</span>

In [ ]:
# формируем список из необходимых scaler
scalers = [MinMaxScaler(),RobustScaler() ,StandardScaler()]

# сформируем списко для заполнения данными результатов анализа
scalers_data = []

# установим ограничение на время обучения модели в 600 секунд (10 минут)
time_out = 600

for scaler in scalers:
        # для того чтобы следить за выполнением цикла введем индикатор
        print('current scaler: ', scaler)

         # сформируем список для заполнения данными текушего scaler
        current_scaler_data = [scaler]
        
        # выполним scaler преобразование
        X_train_s_balanced = scaler.fit_transform(np.array(X_train_balanced)[:,list_n_last_features])
        X_train_s = scaler.transform(X_train[:,list_n_last_features])
        X_valid_s = scaler.transform(X_valid[:,list_n_last_features])

        try:
                # для модуля func_time_out упакуем обучение модели в функцию try_func
                def try_func():
                        logistic_regression = linear_model.LogisticRegression(random_state=42, max_iter=10000)
                        return logistic_regression.fit(X_train_s_balanced,y_train_balanced)
                    
                # фиксируем время начала
                start_time = time.time()
                
                # запускаем обучение модели с ограничением по времени
                logistic_regression = func_timeout.func_timeout(time_out, try_func)
                
                # фиксируем время окончания обучения
                end_time = time.time()
                
                # запишем время обучения модели
                current_scaler_data.append(round(end_time-start_time))

                # Делаем предсказание для обучающей и валидационной выборки
                y_valid_lr_pred = logistic_regression.predict(X_valid_s)
                
                # для метрик ROC AUC делаем предсказание модели для обучающей и валидационной выборки  
                # в виде вероятности принадлежности к классу 1 
                y_train_lr_pred_proba = logistic_regression.predict_proba(X_train_s)[:,1]
                y_valid_lr_pred_proba = logistic_regression.predict_proba(X_valid_s)[:,1]

                #Считаем метрики ROC AUC yна обуающем и валидационном наборе и добавляем их в спискок
                current_scaler_data.append(round(metrics.roc_auc_score(y_train, y_train_lr_pred_proba),3))
                current_scaler_data.append(round(metrics.roc_auc_score(y_valid, y_valid_lr_pred_proba),3))

                #Считаем метрики для класса 0 на валидационном наборе и добавляем их в список
                current_scaler_data.append(round(metrics.recall_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))
                current_scaler_data.append(round(metrics.precision_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))


                #Считаем метрики для класса 1 на валидационном набопе и добавляем их в список
                current_scaler_data.append(round(metrics.recall_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))
                current_scaler_data.append(round(metrics.precision_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))

                # добавляем полученные данные в список scalers_data
                scalers_data.append(current_scaler_data)

                # чтобы подстраховаться на случай ошибки : "The Kernel crashed while executing code"
                # сохраним в локальную папку удачную итерацию
                # для этого из полученных данных сформируем dataframe
                # из полученных данных сформируем dataframe
                scaler_pd =pd.DataFrame(
                        columns=['scaler','time_fit',
                                 'roc_train','roc_valid', 
                                 'recall_0','precision_0',
                                 'recall_1','precision_1'],
                        data = scalers_data)
                # сохраняем dataframe
                scaler_pd.to_csv('data_recovery/scaler_pd.csv',index=False)
               
                # очистим output от прошедшей итерации
                clear_output()

        except func_timeout.FunctionTimedOut:
                # если время обучения привышает лимит
                # запишем в current_scaler_data соотвествующую данные
                current_scaler_data = [scaler,'>30m',0,0,0,0,0,0]
                # добавляем полученные данные в список scalers_data
                scalers_data.append(current_scaler_data)

                # чтобы подстраховаться на случай ошибки : "The Kernel crashed while executing code"
                # сохраним в локальную папку удачную итерацию
                # для этого из полученных данных сформируем dataframe
                # из полученных данных сформируем dataframe
                scaler_pd =pd.DataFrame(
                        columns=['scaler','time_fit',
                                 'roc_train','roc_valid', 
                                 'recall_0','precision_0',
                                 'recall_1','precision_1'],
                        data = scalers_data)
                # сохраняем dataframe
                scaler_pd.to_csv('data_recovery/scaler_pd.csv',index=False)
                
                # очистим output от прошедшей итерации
                clear_output()
                # переходим к следующему scaler
                pass
# очистим output
clear_output()

# сформируем данные соотвествующие лучщему ROC AUC на валидацинном наборе
best_roc_valid_data = scaler_pd[scaler_pd['roc_valid']==scaler_pd['roc_valid'].max()]

# выведем лучший результат на валидационном наборе
print('result:')
print('best scaler on valid:',best_roc_valid_data.iloc[0]['scaler'])
print('ROC AUC on train:', best_roc_valid_data.iloc[0]['roc_train'])
print('ROC AUC on valid:', best_roc_valid_data.iloc[0]['roc_valid'])
print('Time fit for best scaler:', best_roc_valid_data.iloc[0]['time_fit'],' seconds')

# выведем полученный dataframe
scaler_pd

# time: 

##### <span style="color:MediumSlateBlue"> Анализ влияния solver оптимизатора на качество и скорость обучения модели</span>

Проверим качество и скорость сходимости модели при разных solver  
Проверку осуществим через цикл  
Чтобы модель не сходилась бесконечно с помощью модуля func_timeout  
поставим ограничение времени схождения в 10 минут

In [ ]:
# подготовим данные для обучения модели

# обьявим scaler
scaler = StandardScaler()
# выполним scaler преобразование
X_train_s_balanced = scaler.fit_transform(np.array(X_train_balanced)[:,list_n_last_features])
X_train_s = scaler.transform(X_train[:,list_n_last_features])
X_valid_s = scaler.transform(X_valid[:,list_n_last_features])

# time: 10s

In [ ]:
# Зададим список solvers
solvers_list = ['lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga']

# сформируем список для заполнения
solvers_data = []

# установим ограничение на время обучения модели в 300 секунд (5 минут)
time_out = 300

for solver in solvers_list:
        # для того чтобы следить за выполнением цикла введем индикатор
        print('current solver: ', solver)

         # сформируем список для заполнения данными текушего solver
        current_solver_data = [solver]
        
        try:
                # для модуля func_time_out упакуем обучение модели в функцию try_func
                def try_func():
                        logistic_regression = linear_model.LogisticRegression(solver= solver,
                                                                  random_state=42, 
                                                                  max_iter=10000)
                        return logistic_regression.fit(X_train_s_balanced,y_train_balanced)
                    
                # фиксируем время начала
                start_time = time.time()
                
                # запускаем обучение модели с ограничением по времени
                logistic_regression = func_timeout.func_timeout(time_out, try_func)
                
                # фиксируем время окончания обучения
                end_time = time.time()
                
                # запишем время обучения модели
                current_solver_data.append(round(end_time-start_time))

                # Делаем предсказание для обучающей и валидационной выборки
                y_valid_lr_pred = logistic_regression.predict(X_valid_s)
                
                # для метрик ROC AUC делаем предсказание модели для обучающей и валидационной выборки  
                # в виде вероятности принадлежности к классу 1 
                y_train_lr_pred_proba = logistic_regression.predict_proba(X_train_s)[:,1]
                y_valid_lr_pred_proba = logistic_regression.predict_proba(X_valid_s)[:,1]

                #Считаем метрики ROC AUC yна обуающем и валидационном наборе и добавляем их в спискок
                current_solver_data.append(round(metrics.roc_auc_score(y_train, y_train_lr_pred_proba),3))
                current_solver_data.append(round(metrics.roc_auc_score(y_valid, y_valid_lr_pred_proba),3))

                #Считаем метрики для класса 0 на валидационном наборе и добавляем их в список
                current_solver_data.append(round(metrics.recall_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))
                current_solver_data.append(round(metrics.precision_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))


                #Считаем метрики для класса 1 на валидационном набопе и добавляем их в список
                current_solver_data.append(round(metrics.recall_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))
                current_solver_data.append(round(metrics.precision_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))

                # запишем данные полученные на одной итерации
                solvers_data.append(current_solver_data)

                # чтобы подстраховаться на случай ошибки : "The Kernel crashed while executing code"
                # сохраним в локальную папку удачную итерацию
                # для этого из полученных данных сформируем dataframe
                # из полученных данных сформируем dataframe
                solvers_pd =pd.DataFrame(
                        columns=['solver','time_fit',
                                 'roc_train','roc_valid', 
                                 'recall_0','precision_0',
                                 'recall_1','precision_1'],
                        data = solvers_data)
                # сохраняем dataframe
                solvers_pd.to_csv('data_recovery/solvers_pd.csv',index=False)
               
                # очистим output от прошедшей итерации
                clear_output()

        except func_timeout.FunctionTimedOut:
                # если время обучения привышает лимит
                # запишем в current_solver_data соотвествующую данные
                current_solver_data = [solver,'>10m',0,0,0,0,0,0]
                # добавляем полученные данные в список solvers_data
                solvers_data.append(current_solver_data)

                # чтобы подстраховаться на случай ошибки : "The Kernel crashed while executing code"
                # сохраним в локальную папку удачную итерацию
                # для этого из полученных данных сформируем dataframe
                # из полученных данных сформируем dataframe
                solvers_pd =pd.DataFrame(
                        columns=['solver','time_fit',
                                 'roc_train','roc_valid', 
                                 'recall_0','precision_0',
                                 'recall_1','precision_1'],
                        data = solvers_data)
                # сохраняем dataframe
                solvers_pd.to_csv('data_recovery/solvers_pd.csv',index=False)
                
                # очистим output от прошедшей итерации
                clear_output()
                # переходим к следующему solver
                pass
# очистим output
clear_output()

# сформируем данные соотвествующие лучщему ROC AUC на валидацинном наборе
best_roc_valid_data = solvers_pd[solvers_pd['roc_valid']==solvers_pd['roc_valid'].max()]

# выведем лучший результат на валидационном наборе
print('result:')
print('best solver on valid:',best_roc_valid_data.iloc[0]['solver'])
print('ROC AUC on train:', best_roc_valid_data.iloc[0]['roc_train'])
print('ROC AUC on valid:', best_roc_valid_data.iloc[0]['roc_valid'])
print('Time fit for best solver:', best_roc_valid_data.iloc[0]['time_fit'],' seconds')

# выведем полученный dataframe
solvers_pd

# time: 11m 35s

##### <span style="color:MediumSlateBlue">Подбор гиперпараметров модели</span>

In [ ]:
# для выбора sceler преобразвания на понадобится словарь
dict_scalers ={
    'MinMaxScaler':MinMaxScaler(),
    'RobustScaler':RobustScaler(),
    'StandardScaler':StandardScaler(),
}
# определим пространство признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'service'
count_features = dict_spaces[feature_space]

# настроим оптимизацию гипер параметров
def optuna_lg(trial):
  # задаем пространства поиска гиперпараметров
  C = trial.suggest_float('C',0.01,1)
  solver = trial.suggest_categorical('solver', ['lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky'])
  class_0_weight = trial.suggest_float('class_0_weight',0.01,1)
  class_1_weight = trial.suggest_float('class_1_weight',0.01,1)
  n_last = trial.suggest_int('n_last', 10, 25,step = 1)
  class_1_percent = trial.suggest_float('class_1_percent',0.01,1)
  scaler = trial.suggest_categorical('scaler', ['MinMaxScaler', 'RobustScaler','StandardScaler'])
  random_state = trial.suggest_int('random_state', 1, 1000000)

  # создаем модель
  optuna_log_reg = linear_model.LogisticRegression(
      solver=solver,
      C=C,
      class_weight={0:class_0_weight,1:class_1_weight},
      random_state=random_state,
      max_iter=10000)

  # с помощью функции features_from_transform_data_torow извлечем  
  # из данных transform_data_torow
  # признаки сооствествующие n_last последним клиенским операциям
  list_n_last_features = features_from_transform_data_torow(n_last,count_features,25)

  # обьявим scaler
  scaler = dict_scalers[scaler]

  # загружаем обучующие наборы
  X_train_pd = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas()
  y_train_pd = pd.read_csv('target/target_train.csv')

  # поготовим данные y_train_pd к работе с функцией class_1_percent_samples
  y_train_pd.set_index('id',drop=False,inplace=True)

  # подготовим данные для обучения модели
  # с помощью функции class_1_percent_samples зададим долю 
  # класса 1
  list_c1_percent_id = class_1_percent_samples(y_train_pd,class_1_percent,random_state=random_state)[:1000000]

  # сформируем данные для обучения базовой модели
  X_train_s_balanced = scaler.fit_transform(np.array(X_train_pd.loc[list_c1_percent_id])[:,list_n_last_features])
  y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

  # освободим память от "тяжелых" и ненужных файлов
  del X_train_pd, y_train_pd
  gc.collect()

  # я не воспользовался параметром timeout метода optimize потому, что  
  # optimize отставливает поиск параметров, если время обучения 
  # превышает timeout. Мне нужно чтобы поиск параметров продолжился.
  try:
    # для модуля func_time_out упакуем обучение модели в функцию try_func
    def try_func():
            return optuna_log_reg.fit(X_train_s_balanced, y_train_balanced)
    # обучаем модель с ограничением по времени 10 минут (600 секунд)
    optuna_log_reg = func_timeout.func_timeout(600, try_func)

    # удаляем крупные файлы чтобы высвободить память 
    del X_train_s_balanced,y_train_balanced
    gc.collect()

    # подгружаем данные для тестирования
    X_train = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas().to_numpy()[:,list_n_last_features]
    y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
    X_valid = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_valid').to_pandas().to_numpy()[:,list_n_last_features]
    y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()

    # сформируем данные для проверки модели
    X_train_s = scaler.transform(X_train)
    X_valid_s = scaler.transform(X_valid)

    # удаляем крупные файлы чтобы высвободить память 
    del X_train, X_valid
    gc.collect()

    # делаем предсказание на обучающем и валидационном наборе и считаем метрики
    roc_train = metrics.roc_auc_score(y_train, optuna_log_reg.predict_proba(X_train_s)[:,1])
    roc_valid = metrics.roc_auc_score(y_valid, optuna_log_reg.predict_proba(X_valid_s)[:,1])
    f1_score = metrics.f1_score(y_valid, optuna_log_reg.predict(X_valid_s))
    recall_1 = metrics.recall_score(y_valid, optuna_log_reg.predict(X_valid_s),average=None,zero_division=0)[1]
    precision_1 = metrics.precision_score(y_valid, optuna_log_reg.predict(X_valid_s),average=None,zero_division=0)[1]

    del X_train_s, X_valid_s, y_train, y_valid
    gc.collect()

  except func_timeout.FunctionTimedOut:
    # освободим память от "тяжелых" и ненужных файлов
    del X_train_s_balanced, y_train_balanced
    gc.collect()

    # обьявим метрики пустыми значениями
    roc_train = 0
    roc_valid = 0
    f1_score = 0
    recall_1 = 0
    precision_1 = 0
    pass

  return roc_train, roc_valid, f1_score, recall_1, precision_1

In [ ]:
# cоздаем объект исследования
# чтобы модель не переобучалась минимизируем roc_train 
# и максимизируем roc_valid
optuna_study_lg = optuna.create_study(study_name='LogisticRegression_'+feature_space+'_torow', 
                               directions=['maximize','maximize','maximize','maximize','maximize'], 
                               sampler=optuna.samplers.TPESampler(),
                               pruner='Hyperband',
                               storage='sqlite:///optuna_studies.db',
                               load_if_exists=True)
# на случай удаления 
# optuna.delete_study(study_name='LogisticRegression_'+feature_space+'_torow', storage='sqlite:///optuna_studies.db')

In [ ]:
# ищем лучшую комбинацию гиперпараметров n_trials раз
optuna_study_lg.optimize(optuna_lg, n_trials=300)

In [ ]:
# из полученного результат соврмируем Data Frame
optuna_study_lg_pd = optuna_study_lg.trials_dataframe().dropna()
# переименуем столбы
optuna_study_lg_pd.rename(columns={
    'values_0': 'roc_train',
    'values_1': 'roc_valid',
    'values_2': 'f1_score',
    'values_3': 'recall_1',
    'values_4': 'precision_1'
},inplace=True)
# Выделим время потраченное на обучение модели
optuna_study_lg_pd.head(5)

In [ ]:
# покаждем статистику обучения
print('Максимальное значение метрики ROC AUC на валидационном наборе:',round(optuna_study_lg_pd['roc_valid'].max(),3))
print('Среднее значение метрики ROC AUC на валидационном наборе:',round(optuna_study_lg_pd['roc_valid'].mean(),3))
print('Максимальное значение метрики f1_score на валидационном наборе:',round(optuna_study_lg_pd['f1_score'].max(),3))
print('Среднее значение метрики f1_score на валидационном наборе:',round(optuna_study_lg_pd['f1_score'].mean(),3))
print('Максимальное значение метрики recall_1 на валидационном наборе:',round(optuna_study_lg_pd['recall_1'].max(),3))
print('Среднее значение метрики recall_1 на валидационном наборе:',round(optuna_study_lg_pd['recall_1'].mean(),3))
print('Максимальное значение метрики precision_1 на валидационном наборе:',round(optuna_study_lg_pd['precision_1'].max(),3))
print('Среднее значение метрики precision_1 на валидационном наборе:',round(optuna_study_lg_pd['precision_1'].mean(),3))

In [ ]:
# построим график важности гиперпарметров
optuna.visualization.plot_param_importances(optuna_study_lg, target_name='Score')

In [ ]:
# Построим зависимость метрики precision_1

fig= px.scatter(
    data_frame=optuna_study_lg_pd,
    x='precision_1', #ось абсцисс
    y=['recall_1','f1_score'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision от recall на классе 1', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision',
    yaxis_title='Величина метрик recall и f1-score')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики precision_1 от гипер параметров

fig= px.scatter(
    data_frame=optuna_study_lg_pd,
    x='precision_1', #ось абсцисс
    y=['params_C','params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight', 'params_n_last',
    ], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики f1_score от гипер параметров

fig= px.scatter(
    data_frame=optuna_study_lg_pd,
    x='f1_score', #ось абсцисс
    y=['params_C','params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight', 'params_n_last',
    ], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость f1-score от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики f1-score',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики roc_valid от гипер параметров

fig= px.scatter(
    data_frame=optuna_study_lg_pd,
    x='roc_valid', #ось абсцисс
    y=['params_C','params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight', 'params_n_last',
    ], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость ROC AUC от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики ROC AUC',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Визуализируем метрики полученные при optuna оптимизации
fig = px.scatter(
    data_frame=optuna_study_lg_pd[
                                (optuna_study_lg_pd['roc_valid']>0.9*optuna_study_lg_pd['roc_valid'].max())& 
                                (optuna_study_lg_pd['precision_1']>0.065)&
                                (optuna_study_lg_pd['recall_1']>0.1)],
    x='number', #ось абсцисс
    y=['roc_train','roc_valid','recall_1','precision_1'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость качества модели от trials optuna', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='trials optuna',
    yaxis_title='Величина метрики')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

<span style="color:Blue">

Вывод:  

Их всех выбираем точку $number =24$.   
В этой точке одно из самых больших значений $ROC AUC$.  
Посмотрим каким значениям гиперпараметров соотвествует выбранная точка.

In [ ]:
# определим номер лучшге варианта
best_optuna_number = 24

# сформируем словарь лучших гипирпарметров RandomForestClassifier
best_param_lr_torow = {
    'C' : round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_C'].iloc[0],3),
    'class_weight' : {0:round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_class_0_weight'].iloc[0],3),
                      1:round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_class_1_weight'].iloc[0],3)},
    }
# создадим перменные
best_n_last = int(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_n_last'].iloc[0])
best_scaler = optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_scaler'].iloc[0]
best_class_1_percent = optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_class_1_percent'].iloc[0]
best_random_state = int(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_random_state'].iloc[0])
# Выведем принятые наилучшие праметры
print('best C:',best_param_lr_torow['C'])
print('best class 0 weight:',best_param_lr_torow['class_weight'][0])
print('best class 1 weight:',best_param_lr_torow['class_weight'][1])

print('best class 1 percent:',round(best_class_1_percent,3))
print('best scaler:',best_scaler)
print('best n_last:',round(best_n_last,3))
print('best best random state:',best_random_state)
print('time for best train:',round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['duration'].iloc[0].seconds/60),'minutes')

print()
print('ROC AUC на обучающем наборе:', round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['roc_train'].iloc[0],3))
print('ROC AUC на валидационном наборе:', round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['roc_valid'].iloc[0],3))
print('precision класса 1:', round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['precision_1'].iloc[0],3))
print('recall класса 1:', round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['recall_1'].iloc[0],3))

##### <span style="color:MediumSlateBlue">Обучение модели с лучшими параметрами</span>

In [ ]:
# для выбора sceler преобразвания на понадобится словарь
dict_scalers ={
    'MinMaxScaler':MinMaxScaler(),
    'RobustScaler':RobustScaler(),
    'StandardScaler':StandardScaler(),
}

# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'service'
count_features = dict_spaces[feature_space]

# с помощью функции features_from_transform_data_torow извлечем  
# из данных transform_data_torow_25
# признаки сооствествующие n_last последним клиенским операциям
list_n_last_features = features_from_transform_data_torow(best_n_last,count_features,25)

# обьявим scaler
scaler = dict_scalers[best_scaler]

# загружаем обучующие наборы
X_train_pd = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas()
y_train_pd = pd.read_csv('target/target_train.csv')

# поготовим данные y_train_pd к работе с функцией class_1_percent_samples
y_train_pd.set_index('id',drop=False,inplace=True)

# подготовим данные для обучения модели
# с помощью функции class_1_percent_samples зададим долю 
# класса 1
list_c1_percent_id = class_1_percent_samples(y_train_pd,best_class_1_percent,random_state=best_random_state)[:1000000]

# сформируем данные для обучения базовой модели
X_train_s_balanced = scaler.fit_transform(np.array(X_train_pd.loc[list_c1_percent_id])[:,list_n_last_features])
y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

# освободим память от "тяжелых" и ненужных файлов
del X_train_pd, y_train_pd
gc.collect()


# обучаем модель LogisticRegression с наилучшеми параметрами
logistic_regression = linear_model.LogisticRegression(
        **best_param_lr_torow,
        random_state=best_random_state,
        max_iter=10000)
logistic_regression.fit(X_train_s_balanced,y_train_balanced)

# удаляем крупные файлы чтобы высвободить память 
del X_train_s_balanced,y_train_balanced
gc.collect()

# подгружаем данные для тестирования
X_train = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas().to_numpy()[:,list_n_last_features]
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
X_valid = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_valid').to_pandas().to_numpy()[:,list_n_last_features]
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()
    
# сформируем данные для проверки модели
X_train_s = scaler.transform(X_train)
X_valid_s = scaler.transform(X_valid)

# удаляем крупные файлы чтобы высвободить память 
del X_train, X_valid
gc.collect()

# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_lr_pred_proba = logistic_regression.predict_proba(X_train_s)[:,1]
y_valid_lr_pred_proba = logistic_regression.predict_proba(X_valid_s)[:,1]

# Делаем предсказание для валидационной выборки
y_valid_lr_pred = logistic_regression.predict(X_valid_s)

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_lr_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_lr_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_valid,y_valid_lr_pred,zero_division=0))

# time: 2m 40s

##### <span style="color:MediumSlateBlue">Анализ влияния порога классификации на качество модели</span>

In [ ]:
# чтобы проше обращаться к каждому обьекту в данных
# преобразуем y_valid_LR_pred к Series типу
y_valid_lr_pred_proba = pd.Series(y_valid_lr_pred_proba)

# формируем список из необходимых значений thresholds
thresholds = np.arange(0.01, 1, 0.01)

# сформируем списко для заполнения данными результатов анализа
threshold_data = []


for threshold in thresholds:
        # сформируем список для заполнения данными текушего threshold
        threshold_current_data = [threshold]

        #клиентов, для которых вероятность дефолта > threshold, относим к классу 1
        #в противном случае — к классу 0
        y_valid_lr_pred = y_valid_lr_pred_proba.apply(lambda x: 1 if x>threshold else 0)


        #Считаем метрики для класса 0 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.precision_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.f1_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))

        #Считаем метрики для класса 1 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.precision_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.f1_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))

        # добавляем полученные данные в список threshold_data
        threshold_data.append(threshold_current_data)

        # из полученных данных сформируем dataframe
        thresholds_pd =pd.DataFrame(
        columns=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'],
        data = threshold_data)

# time: 17s

In [ ]:
# Визуализируем метрики на валидационном наборе при различных threshold
fig_threshold = px.line(
    data_frame=thresholds_pd,
    x='threshold', #ось абсцисс
    y=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'], #ось ординат
)
fig_threshold.update_layout(
    title ={
        'text':'Зависимость качества модели от threshold', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Порог классификации threshold',
    yaxis_title='Величина метрики')
fig_threshold.update_xaxes(showspikes=True)
fig_threshold.update_yaxes(showspikes=True)

fig_threshold.show()

Метрика $precision$ показывает на сколько сильно ошиблась модель при определиние класса.  
Чем выше метрика, тем меньше ошиблась модель. 

$$precision = \frac{TP}{TP+FP}$$

Метрика $recall$ показывает способность модели найти класс.  
Чем выше метрика, тем выше способность модели находить класс.  

$$recall = \frac{TP}{TP+FN}$$

Метрика $f1-score$ показывает баланс между $precision$ и $recall$ 

$threshold$- порог класификации модели.  
Все объекты для которых $у_{\text{proba pred}} < threshold$, модель относит к класс 0.  
Соотвественно, если $у_{\text{proba pred}}  > threshold$ модель относит обьект к классу 1.  
Для идельаной модели $threshold$, является границей между областями класса 0 и класса 1.  

<span style="color:Blue">

Выводы по результам анализа:  

Для улучшения качества модели по метрике $ROC AUC$, нет смысла сдвигать порог   
классификации. Но зависимость метрик качества модели от $threshold$, может рассказать о  
спосбоности модели искать класс 1 и класс 0.

При фокусировке модели на классе 1 (более низкий $threshold$):  
- на классе 1 метрика $precision$ уменьшается, метрика $recall$ растет;  
- на классе 0 метрика $precision$ растет, метрика $recall$ уменьшается.  

При фокусировке модели на классе 0 (более высокий $threshold$):  
- на классе 1 метрика $precision$ увеличивается, метрика $recall$ уменьшается;  
- на классе 0 метрика $precision$ уменьшается, метрика $recall$ растет.  

Обьсняется тем, что в условия дисбаланса модель научилась хорошо определять класс 0 и плохо класс 1.  

При фокусировке модели на классе 1 (более низкий $threshold$), все больше обьектов отнесены к классу 1.   
Так как $precision$ класса 1 уменьшается, значит в области класса 1 стало больше обектов класса 0,  
что явлется закономерным для "идеальной" модели.  
Однако при этом, $recall$ класса 1 растет, значит в добавленных обьектах также присутвует класс 1.  
Это как раз указывает на то, что модель не нучилась находить среди класса 0 класс 1. В области с     
высокой вероятностью класса 0 (более низкий $threshold$), находятся обьекты класса 1.  

Фокусировка на классе 0 (более высокий $threshold$), заставляет модель все больше обьектов из области класса 1  
относить к классу 0. При этом увеличение метрики $precision$ класса 1 говорит о том, что в области класса 1     
становится меньше 0, а значит модель умеет среди класса 1, находить класс 0.  


Обратим внимание на $threshold=0.52$, в этой точке находится баланс межу метриками качества модели:
$$precision_1 = recall_1 = \text{f1-score}_1 = 0.067$$
$$precision_0 = recall_0 = \text{f1-score}_0 = 0.965$$

Для модели с лучшим качеством эти две точки должны находится максимально близко к друг другу и иметь  
высокое значение метрик $precision_1$ и $recall_1$.  
Для класса 0 это условие выполнется, что также говорит о том, что модель научилась искать этот класс.   
Для класса 1 это условие не выполнется, что также говорит о том, что модель плохо ищет класс 1.

##### <span style="color:MediumSlateBlue">Построение ROC-кривой выбранной модели на тестовом наборе</span>

In [ ]:
# определим пространство признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'service'
count_features = dict_spaces[feature_space]

In [ ]:
# подгружаем данные для тестирования
X_train = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas().to_numpy()[:,list_n_last_features]
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
X_valid = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_valid').to_pandas().to_numpy()[:,list_n_last_features]
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()
X_test = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_test').to_pandas().to_numpy()[:,list_n_last_features]
y_test = pd.read_csv('target/target_test.csv')['flag'].to_numpy()

In [ ]:
# выполним scaler преобразование
X_train_s = scaler.transform(X_train)
X_valid_s = scaler.transform(X_valid)
X_test_s = scaler.transform(X_test)

print("Размеры обучающего набора",X_train_s.shape)
print("Размеры валидационного набора",X_valid_s.shape)
print("Размеры тестового набора",X_test_s.shape)
# time: 1m 43s

In [ ]:
# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_lr_pred_proba = logistic_regression.predict_proba(X_train_s)[:,1]
y_valid_lr_pred_proba = logistic_regression.predict_proba(X_valid_s)[:,1]
y_test_lr_pred_proba = logistic_regression.predict_proba(X_test_s)[:,1]

# Делаем предсказание для валидационной выборки
y_test_lr_pred = logistic_regression.predict(X_test_s)

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_lr_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_lr_pred_proba),3))
print('ROC AUC на тестовом наборе', round(metrics.roc_auc_score(y_test, y_test_lr_pred_proba),3))

print('Основные метрики на тестовом наборе:')
print(metrics.classification_report(y_test,y_test_lr_pred,zero_division=0))

In [ ]:
# Для расчета значений TPR и FPR используем функцию roc_curve
fpr_train, tpr_train, _ = metrics.roc_curve(y_train, y_train_lr_pred_proba)
fpr_valid, tpr_valid, _ = metrics.roc_curve(y_valid, y_valid_lr_pred_proba)
fpr_test, tpr_test, _ = metrics.roc_curve(y_test, y_test_lr_pred_proba)

# рассчитаем площадь под кривой
auc_train = round(metrics.roc_auc_score(y_train, y_train_lr_pred_proba),3)
auc_valid = round(metrics.roc_auc_score(y_valid, y_valid_lr_pred_proba),3)
auc_test = round(metrics.roc_auc_score(y_test, y_test_lr_pred_proba),3)

trace_train = go.Scatter(x=fpr_train, y=tpr_train, mode='lines', name='AUC_train = %0.2f' % auc_train,
                   line=dict(color='red', width=2),
                   hovertemplate='train<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_train])

trace_valid = go.Scatter(x=fpr_valid, y=tpr_valid, mode='lines', name='AUC_valid = %0.2f' % auc_valid,
                   line=dict(color='green', width=2),
                   hovertemplate='valid<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_valid])

trace_test = go.Scatter(x=fpr_test, y=tpr_test, mode='lines', name='AUC_test = %0.2f' % auc_test,
                   line=dict(color='darkorange', width=2),
                   hovertemplate='test<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_test])


reference_line = go.Scatter(x=[0,1], y=[0,1], mode='lines', name='Reference Line',
                            line=dict(color='navy', width=2, dash='dash'))

fig_roc = go.Figure(data=[trace_train,trace_valid,trace_test,reference_line])

fig_roc.update_layout(
    title ={
        'text':'ROC-кривая', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 1350,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate')

fig_roc.update_xaxes(showspikes=True)
fig_roc.update_yaxes(showspikes=True)


fig_roc.show()

<span style="color:Blue">

Вывод по ROC-кривой:
1. Модель выдает сталибильное качество на всех наборах, а значит  
не переобучена - разброс ($variance$) не большой.  
2. Смещение ($bias$) модели относительно большое. Качество чуть лучше, чем  
случайное угадываение.
На тестовом наборе полученно качество $ROC AUC = 0.59$.

#### <span style="color:MediumBlue">Построение модели Random Forest Classifier</span>

##### <span style="color:MediumSlateBlue">Baseline обучение модели RandomForestClassifier</span>

In [ ]:
# обучаем модель логистической регрессии
# чтобы модель пыталась найти связь во всех признаках
random_forest = RandomForestClassifier(random_state=42, n_jobs=-1)
random_forest.fit(np.array(X_train_balanced)[:,list_n_last_features],np.array(y_train_balanced))

# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_rf__pred_proba = random_forest.predict_proba(X_train[:,list_n_last_features])[:,1]
y_valid_rf_pred_proba = random_forest.predict_proba(X_valid[:,list_n_last_features])[:,1]

# Делаем предсказание для валидационной выборки
y_valid_rf_pred = random_forest.predict(X_valid[:,list_n_last_features])

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_rf__pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_rf_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_valid,y_valid_rf_pred))

# time: 2m 5s

##### <span style="color:MediumSlateBlue">Подбор гиперпараметров модели</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'service'
count_features = dict_spaces[feature_space]

# настроим оптимизацию гипер параметров
def optuna_rf(trial):
  # задаем пространства поиска гиперпараметров
  # задаем пространства поиска гиперпараметров
  n_estimators = trial.suggest_int('n_estimators', 50, 150,step = 5)
  criterion = trial.suggest_categorical('criterion',['gini', 'entropy', 'log_loss'])
  max_depth = trial.suggest_int('max_depth', 9, 20,step = 1)
  min_samples_split = trial.suggest_int('min_samples_split', 5, 15,step = 1)
  min_samples_leaf = trial.suggest_int('min_samples_leaf', 5, 25,step = 1)
  max_features = trial.suggest_float('max_features',0.1,1)
  max_samples = trial.suggest_float('max_samples',0.1,1)
  class_0_weight = trial.suggest_float('class_0_weight',0.01,1)
  class_1_weight = trial.suggest_float('class_1_weight',0.01,1)
  n_last = trial.suggest_int('n_last', 5, 25,step = 1)
  class_1_percent = trial.suggest_float('class_1_percent',0.01,1)
  random_state = trial.suggest_int('random_state', 1, 1000000)


  # создаем модель
  optuna_random_forest = RandomForestClassifier(
      n_estimators = n_estimators, 
      criterion = criterion,
      max_depth = max_depth,
      min_samples_split = min_samples_split,  
      min_samples_leaf = min_samples_leaf,
      max_features=max_features,
      max_samples=max_samples,
      class_weight={0:class_0_weight,1:class_1_weight},
      n_jobs=-1,
      random_state=random_state)

  # с помощью функции features_from_transform_data_torow извлечем  
  # из данных transform_data_torow
  # признаки сооствествующие n_last последним клиенским операциям
  list_n_last_features = features_from_transform_data_torow(n_last,count_features,25)

  # загружаем обучующие наборы
  X_train_pd = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas()
  y_train_pd = pd.read_csv('target/target_train.csv')

  # поготовим данные y_train_pd к работе с функцией class_1_percent_samples
  y_train_pd.set_index('id',drop=False,inplace=True)

  # подготовим данные для обучения модели
  # с помощью функции class_1_percent_samples зададим долю 
  # класса 1
  list_c1_percent_id = class_1_percent_samples(y_train_pd,class_1_percent,random_state=random_state)[:1000000]

  # сформируем данные для обучения базовой модели
  X_train_balanced = np.array(X_train_pd.loc[list_c1_percent_id])[:,list_n_last_features]
  y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

  # освободим память от "тяжелых" и ненужных файлов
  del X_train_pd, y_train_pd
  gc.collect()

  # я не воспользовался параметром timeout метода optimize потому, что  
  # optimize отставливает поиск параметров, если время обучения 
  # превышает timeout. Мне нужно чтобы поиск параметров продолжился.
  try:
    # для модуля func_time_out упакуем обучение модели в функцию try_func
    def try_func():
            return optuna_random_forest.fit(X_train_balanced,y_train_balanced)
    # обучаем модель с ограничением по времени 10 минут (600 секунд)
    optuna_random_forest = func_timeout.func_timeout(600, try_func)

    # удаляем крупные файлы чтобы высвободить память 
    del X_train_balanced, y_train_balanced
    gc.collect()

    # подгружаем данные для тестирования
    X_train = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas().to_numpy()[:,list_n_last_features]
    y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
    X_valid = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_valid').to_pandas().to_numpy()[:,list_n_last_features]
    y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()

    # делаем предсказание на обучающем и валидационном наборе и считаем метрики
    roc_train = metrics.roc_auc_score(y_train, optuna_random_forest.predict_proba(X_train)[:,1])
    roc_valid = metrics.roc_auc_score(y_valid, optuna_random_forest.predict_proba(X_valid)[:,1])
    f1_score = metrics.f1_score(y_valid, optuna_random_forest.predict(X_valid))
    recall_1 = metrics.recall_score(y_valid, optuna_random_forest.predict(X_valid),average=None,zero_division=0)[1]
    precision_1 = metrics.precision_score(y_valid, optuna_random_forest.predict(X_valid),average=None,zero_division=0)[1]

    # удаляем крупные файлы чтобы высвободить память 
    del X_train, X_valid, y_train, y_valid 
    gc.collect()

  except func_timeout.FunctionTimedOut:
    # освободим память от "тяжелых" и ненужных файлов
    del X_train_balanced, y_train_balanced
    gc.collect()

    # обьявим метрики пустыми значениями
    roc_train = 0
    roc_valid = 0
    f1_score = 0
    recall_1 = 0
    precision_1 = 0
    pass

  return roc_train, roc_valid, f1_score, recall_1, precision_1

In [ ]:
# так как Random Forest склонен к переобучению 
# попробуем направить optimize в сторону уменьшения roc_train
# cоздаем объект исследования
optuna_study_rf = optuna.create_study(study_name='RandomForestClassifier_'+feature_space+'_torow', 
                               directions=['minimize','maximize','maximize','maximize','maximize'], 
                               sampler=optuna.samplers.TPESampler(),
                               pruner='Hyperband',
                               storage='sqlite:///optuna_studies.db',
                               load_if_exists=True)
# на случай удаления 
# optuna.delete_study(study_name='RandomForestClassifier_'+feature_space+'_torow', storage='sqlite:///optuna_studies.db')

In [ ]:
# ищем лучшую комбинацию гиперпараметров n_trials раз
optuna_study_rf.optimize(optuna_rf, n_trials=300)

In [ ]:
# из полученного результат соврмируем Data Frame
optuna_study_rf_pd = optuna_study_rf.trials_dataframe().dropna()
# переименуем столбы
optuna_study_rf_pd.rename(columns={
    'values_0': 'roc_train',
    'values_1': 'roc_valid',
    'values_2': 'f1_score',
    'values_3': 'recall_1',
    'values_4': 'precision_1'
},inplace=True)
optuna_study_rf_pd.sort_values(by='precision_1').drop(['datetime_start','datetime_complete','duration'],axis=1)

In [ ]:
# покаждем статистику обучения
print('Максимальное значение метрики ROC AUC на валидационном наборе:',round(optuna_study_rf_pd['roc_valid'].max(),3))
print('Среднее значение метрики ROC AUC на валидационном наборе:',round(optuna_study_rf_pd['roc_valid'].mean(),3))
print('Максимальное значение метрики f1_score на валидационном наборе:',round(optuna_study_rf_pd['f1_score'].max(),3))
print('Среднее значение метрики f1_score на валидационном наборе:',round(optuna_study_rf_pd['f1_score'].mean(),3))
print('Максимальное значение метрики recall_1 на валидационном наборе:',round(optuna_study_rf_pd['recall_1'].max(),3))
print('Среднее значение метрики recall_1 на валидационном наборе:',round(optuna_study_rf_pd['recall_1'].mean(),3))
print('Максимальное значение метрики precision_1 на валидационном наборе:',round(optuna_study_rf_pd['precision_1'].max(),3))
print('Среднее значение метрики precision_1 на валидационном наборе:',round(optuna_study_rf_pd['precision_1'].mean(),3))

In [ ]:
# построим график важности гиперпарметров
optuna.visualization.plot_param_importances(optuna_study_rf, target_name='Score')

In [ ]:
# Построим зависимость метрики precision_1

fig = px.scatter(
    data_frame=optuna_study_rf_pd,
    x='precision_1', #ось абсцисс
    y=['f1_score','recall_1'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision от recall', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision',
    yaxis_title='Величина метрик recall и f1-score')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики precision_1 от гиперпараметров

fig = px.scatter(
    data_frame=optuna_study_rf_pd,
    x='precision_1', #ось абсцисс
    y=['params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight','params_max_depth','params_max_features',
        'params_max_samples', 'params_min_samples_leaf', 'params_min_samples_split', 
        'params_n_estimators', 'params_n_last'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision класса 1 от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision на классе 1',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики f1_score от гиперпараметров

fig = px.scatter(
    data_frame=optuna_study_rf_pd,
    x='f1_score', #ось абсцисс
    y=['params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight','params_max_depth','params_max_features',
        'params_max_samples', 'params_min_samples_leaf', 'params_min_samples_split', 
        'params_n_estimators', 'params_n_last'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость f1 - score от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики f1-score',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики roc_valid от гиперпараметров

fig = px.scatter(
    data_frame=optuna_study_rf_pd,
    x='roc_valid', #ось абсцисс
    y=['params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight','params_max_depth','params_max_features',
        'params_max_samples', 'params_min_samples_leaf', 'params_min_samples_split', 
        'params_n_estimators', 'params_n_last'],
)
fig.update_layout(
    title ={
        'text':'Зависимость ROC AUC на валидационном наборе от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики ROC AUC',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрик качества модели от number

fig = px.scatter(
    data_frame=optuna_study_rf_pd[(optuna_study_rf_pd['roc_valid']>=0.63) 
                                  & (optuna_study_rf_pd['precision_1']>0.09)
                                  & (optuna_study_rf_pd['recall_1']>0.09)],
    x='number', #ось абсцисс
    y=['roc_valid','roc_train','precision_1','recall_1'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость метрик качества модели от number', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики ROC Valid',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

<span style="color:Blue">

Вывод:  

Их всех выбираем точку $number =245$.   
В этой точке оптимальные значения метрик $recall_1$ и $precision_1$, а  
также относительно высокая метрика $ROC AUC$.   
Посмотрим каким значениям гиперпараметров соотвествует выбраннаяточка.

In [ ]:
# определим номер лучшге варианта
best_optuna_number = 245

# сформируем словарь лучших гипирпарметров RandomForestClassifier
best_param_rf = {
    'n_estimators' : int(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_n_estimators'].iloc[0]),
    'criterion': optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_criterion'].iloc[0],
    'max_depth' : optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_max_depth'].iloc[0],
    'min_samples_split': optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_min_samples_split'].iloc[0],
    'min_samples_leaf' : optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_min_samples_leaf'].iloc[0],
    'max_features': optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_max_features'].iloc[0],
    'max_samples' : optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_max_samples'].iloc[0],
    'class_weight' : {0:optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_class_0_weight'].iloc[0],
                      1:optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_class_1_weight'].iloc[0]}
    }
# определим перменные для лучших значений параметров
best_n_last = optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_n_last'].iloc[0]
best_class_1_percent = optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_class_1_percent'].iloc[0]
best_random_state = optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_random_state'].iloc[0]


# Выведем принятые наилучшие праметры
print('best n estimators:',best_param_rf['n_estimators'])
print('best criterion:',best_param_rf['criterion'])
print('best max depth:',best_param_rf['max_depth'])
print('best min samples split:',best_param_rf['min_samples_split'])
print('best min samples leaf:',best_param_rf['min_samples_leaf'])
print('best max features(%):',round(best_param_rf['max_features'],3))
print('best max samples(%):',round(best_param_rf['max_samples'],3))
print('best class 0 weight:',round(best_param_rf['class_weight'][0],3))
print('best class 1 weight:',round(best_param_rf['class_weight'][1],3))

print('best class 1 percent:',round(best_class_1_percent,3))
print('best n last:',best_n_last)
print('best random state:',best_random_state)
print('time for best train:',round(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['duration'].iloc[0].seconds/60),'minutes')

print()
print('ROC AUC на обучающем наборе:', round(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['roc_train'].iloc[0],3))
print('ROC AUC на валидационном наборе:', round(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['roc_valid'].iloc[0],3))
print('precision класса 1:', round(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['precision_1'].iloc[0],3))
print('recall класса 1:', round(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['recall_1'].iloc[0],3))

##### <span style="color:MediumSlateBlue">Обучение модели с лучшими параметрами</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'service'
count_features = dict_spaces[feature_space]

# с помощью функции features_from_transform_data_torow извлечем  
# из данных transform_data_torow
# признаки сооствествующие n_last последним клиенским операциям
list_n_last_features = features_from_transform_data_torow(best_n_last,count_features,25)

# загружаем обучующие наборы
X_train_pd = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas()
y_train_pd = pd.read_csv('target/target_train.csv')

# поготовим данные y_train_pd к работе с функцией class_1_percent_samples
y_train_pd.set_index('id',drop=False,inplace=True)

# подготовим данные для обучения модели
# с помощью функции class_1_percent_samples зададим долю 
# класса 1
list_c1_percent_id = class_1_percent_samples(y_train_pd,best_class_1_percent,random_state=best_random_state)[:1000000]

# сформируем данные для обучения базовой модели
X_train_balanced = np.array(X_train_pd.loc[list_c1_percent_id])[:,list_n_last_features]
y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

# освободим память от "тяжелых" и ненужных файлов
del X_train_pd, y_train_pd
gc.collect()

# обучаем модель RandomForestClassifier с наилучшеми параметрами
random_forest = RandomForestClassifier(
        **best_param_rf,
        n_jobs=-1,
        random_state=best_random_state)
random_forest.fit(X_train_balanced, y_train_balanced)

# освободим память от "тяжелых" и ненужных файлов
del X_train_balanced, y_train_balanced
gc.collect()


# подгружаем данные для тестирования
X_train = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas().to_numpy()[:,list_n_last_features]
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
X_valid = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_valid').to_pandas().to_numpy()[:,list_n_last_features]
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()

# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_rf__pred_proba = random_forest.predict_proba(X_train)[:,1]
y_valid_rf_pred_proba = random_forest.predict_proba(X_valid)[:,1]

# Делаем предсказание для валидационной выборки
y_valid_rf_pred = random_forest.predict(X_valid)

# удаляем крупные файлы чтобы высвободить память 
del X_train, X_valid
gc.collect()

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_rf__pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_rf_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_valid,y_valid_rf_pred,zero_division=0))

# time: 2m 10s

##### <span style="color:MediumSlateBlue">Анализ влияния порога классификации на качество модели</span>

In [ ]:
# чтобы проше обращаться к каждому обьекту в данных
# преобразуем y_valid_LR_pred к Series типу
y_valid_rf_pred_proba = pd.Series(y_valid_rf_pred_proba)

# формируем список из необходимых значений thresholds
thresholds = np.arange(0.01, 1, 0.01)

# сформируем списко для заполнения данными результатов анализа
threshold_data = []


for threshold in thresholds:
        # сформируем список для заполнения данными текушего threshold
        threshold_current_data = [threshold]

        #клиентов, для которых вероятность дефолта > threshold, относим к классу 1
        #в противном случае — к классу 0
        y_valid_rf_pred = y_valid_rf_pred_proba.apply(lambda x: 1 if x>threshold else 0)


        #Считаем метрики для класса 0 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(y_valid, y_valid_rf_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.precision_score(y_valid, y_valid_rf_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.f1_score(y_valid, y_valid_rf_pred,average=None,zero_division=0)[0],3))

        #Считаем метрики для класса 1 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(y_valid, y_valid_rf_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.precision_score(y_valid, y_valid_rf_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.f1_score(y_valid, y_valid_rf_pred,average=None,zero_division=0)[1],3))

        # добавляем полученные данные в список threshold_data
        threshold_data.append(threshold_current_data)

        # из полученных данных сформируем dataframe
        thresholds_pd =pd.DataFrame(
        columns=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'],
        data = threshold_data)

# time: 17s

In [ ]:
# Визуализируем метрики на валидационном наборе при различных threshold
fig_threshold = px.line(
    data_frame=thresholds_pd,
    x='threshold', #ось абсцисс
    y=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'], #ось ординат
)
fig_threshold.update_layout(
    title ={
        'text':'Зависимость качества модели от threshold', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Порог классификации threshold',
    yaxis_title='Величина метрики')
fig_threshold.update_xaxes(showspikes=True)
fig_threshold.update_yaxes(showspikes=True)

fig_threshold.show()

Метрика $precision$ показывает на сколько сильно ошиблась модель при определиние класса.  
Чем выше метрика, тем меньше ошиблась модель. 

$$precision = \frac{TP}{TP+FP}$$

Метрика $recall$ показывает способность модели найти класс.  
Чем выше метрика, тем выше способность модели находить класс.  

$$recall = \frac{TP}{TP+FN}$$

Метрика $f1-score$ показывает баланс между $precision$ и $recall$ 

$threshold$- порог класификации модели.  
Все объекты для которых $у_{\text{proba pred}} < threshold$, модель относит к класс 0.  
Соотвественно, если $у_{\text{proba pred}}  > threshold$ модель относит обьект к классу 1.  
Для идельаной модели $threshold$, является границей между областями класса 0 и класса 1.  

<span style="color:Blue">

Выводы по результам анализа:  

Для улучшения качества модели по метрике $ROC AUC$, нет смысла сдвигать порог   
классификации. Но зависимость метрик качества модели от $threshold$, может рассказать о  
способности модели искать класс 1 и класс 0.

При фокусировке модели на классе 1 (более низкий $threshold$):  
- на классе 1 метрика $precision$ уменьшается, метрика $recall$ растет;  
- на классе 0 метрика $precision$ растет, метрика $recall$ уменьшается.  

При фокусировке модели на классе 0 (более высокий $threshold$):  
- на классе 1 метрика $precision$ увеличивается, метрика $recall$ уменьшается;  
- на классе 0 метрика $precision$ уменьшается, метрика $recall$ растет.  

Обьсняется тем, что в условия дисбаланса модель научилась хорошо определять класс 0 и плохо класс 1.  

При фокусировке модели на классе 1 (более низкий $threshold$), все больше обьектов отнесены к классу 1.   
Так как $precision$ класса 1 уменьшается, значит в области класса 1 стало больше обектов класса 0,  
что явлется закономерным для "идеальной" модели.  
Однако при этом, $recall$ класса 1 растет, значит в добавленных обьектах также присутвует класс 1.  
Это как раз указывает на то, что модель не нучилась находить среди класса 0 класс 1. В области с     
высокой вероятностью класса 0 (более низкий $threshold$), находятся обьекты класса 1.  

Фокусировка на классе 0 (более высокий $threshold$), заставляет модель все больше обьектов из области класса 1  
относить к классу 0. При этом увеличение метрики $precision$ класса 1 говорит о том, что в области класса 1     
становится меньше 0, а значит модель умеет среди класса 1, находить класс 0.  
После  $threshold=0.8$ модель впринципе теряет способность отличать класс 1. У модели нет обьектов класса 1   
в которых она "уверена" на 85+%.

Обратим внимание на $threshold=0.5$, в этой точке находится баланс межу метриками качества модели:
$$precision_1 = recall_1 = \text{f1-score}_1 = 0.1$$
$$precision_0 = recall_0 = \text{f1-score}_0 = 0.968$$

Для модели с лучшим качеством эти две точки должны находится максимально близко к друг другу и иметь   
высокое значение метрик $precision_1$ и $recall_1$.  
Для класса 0 это условие выполнется, что также говорит о том, что модель научилась искать этот класс.    
Для класса 1 это условие не выполнется, что также говорит о том, что модель плохо ищет класс 1.

##### <span style="color:MediumSlateBlue">Построение ROC-кривой выбранной модели на тестовом наборе</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'service'
count_features = dict_spaces[feature_space]

In [ ]:
# подгружаем данные для тестирования
X_train = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas().to_numpy()[:,list_n_last_features]
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
X_valid = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_valid').to_pandas().to_numpy()[:,list_n_last_features]
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()
X_test = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_test').to_pandas().to_numpy()[:,list_n_last_features]
y_test = pd.read_csv('target/target_test.csv')['flag'].to_numpy()

In [ ]:
print("Размеры обучающего набора",X_train.shape,y_train.shape)
print("Размеры валидационного набора",X_valid.shape,y_valid.shape)
print("Размеры тестового набора",X_test.shape,y_test.shape)
# time: 1m 43s

In [ ]:
# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_rf_pred_proba = random_forest.predict_proba(X_train)[:,1]
y_valid_rf_pred_proba = random_forest.predict_proba(X_valid)[:,1]
y_test_rf_pred_proba = random_forest.predict_proba(X_test)[:,1]

# Делаем предсказание для валидационной выборки
y_test_lr_pred = random_forest.predict(X_test)

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_rf_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_rf_pred_proba),3))
print('ROC AUC на тестовом наборе', round(metrics.roc_auc_score(y_test, y_test_rf_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_test,y_test_lr_pred,zero_division=0))

In [ ]:
# Для расчета значений TPR и FPR используем функцию roc_curve
fpr_train, tpr_train, _ = metrics.roc_curve(y_train, y_train_rf_pred_proba)
fpr_valid, tpr_valid, _ = metrics.roc_curve(y_valid, y_valid_rf_pred_proba)
fpr_test, tpr_test, _ = metrics.roc_curve(y_test, y_test_rf_pred_proba)

# рассчитаем площадь под кривой
auc_train = round(metrics.roc_auc_score(y_train, y_train_rf_pred_proba),3)
auc_valid = round(metrics.roc_auc_score(y_valid, y_valid_rf_pred_proba),3)
auc_test = round(metrics.roc_auc_score(y_test, y_test_rf_pred_proba),3)

trace_train = go.Scatter(x=fpr_train, y=tpr_train, mode='lines', name='AUC_train = %0.2f' % auc_train,
                   line=dict(color='red', width=2),
                   hovertemplate='train<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_train])

trace_valid = go.Scatter(x=fpr_valid, y=tpr_valid, mode='lines', name='AUC_valid = %0.2f' % auc_valid,
                   line=dict(color='green', width=2),
                   hovertemplate='valid<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_valid])

trace_test = go.Scatter(x=fpr_test, y=tpr_test, mode='lines', name='AUC_test = %0.2f' % auc_test,
                   line=dict(color='darkorange', width=2),
                   hovertemplate='test<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_test])


reference_line = go.Scatter(x=[0,1], y=[0,1], mode='lines', name='Reference Line',
                            line=dict(color='navy', width=2, dash='dash'))

fig_roc = go.Figure(data=[trace_train,trace_valid,trace_test,reference_line])

fig_roc.update_layout(
    title ={
        'text':'ROC-кривая', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 1350,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate')

fig_roc.update_xaxes(showspikes=True)
fig_roc.update_yaxes(showspikes=True)


fig_roc.show()

<span style="color:Blue">

Вывод по ROC-кривой:
1. Модель выдает сталибильное качество на валидационном и тестовом наборах, а значит  
не переобучена - разброс ($variance$) не большой. Хотя, качество модели на обучающем наборе,  
заметно лучше.
2. Смещение ($bias$) модели чуть лучше, чем у модели $Logicstic$ $Regression$ $lassifier$, обученной   
на тех же данных.  
На тестовом наборе полученно качество $ROC AUC = 0.63$, против $ROC AUC = 0.59$

#### <span style="color:MediumBlue">Построение модели Gradient Boosting Classifier</span>

##### <span style="color:MediumSlateBlue">Baseline обучение модели Hist Gradient Boosting Classifier</span>

In [ ]:
# обучаем модель Gradient Boosting Classifier
gradient_boosting = HistGradientBoostingClassifier(random_state=42)
gradient_boosting.fit(np.array(X_train_balanced)[:,list_n_last_features],np.array(y_train_balanced))

# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_rf__pred_proba = gradient_boosting.predict_proba(X_train[:,list_n_last_features])[:,1]
y_valid_rf_pred_proba = gradient_boosting.predict_proba(X_valid[:,list_n_last_features])[:,1]

# Делаем предсказание для валидационной выборки
y_valid_rf_pred = gradient_boosting.predict(X_valid[:,list_n_last_features])

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_rf__pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_rf_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_valid,y_valid_rf_pred))

# time: 

##### <span style="color:MediumSlateBlue">Подбор гиперпараметров модели</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'service'
count_features = dict_spaces[feature_space]

# настроим оптимизацию гипер параметров
def optuna_hgbc(trial):
  # задаем пространства поиска гиперпараметров
  learning_rate = trial.suggest_float('learning_rate',0.2,0.6)
  max_iter = trial.suggest_int('max_iter', 50, 300,step = 1)
  max_leaf_nodes = trial.suggest_int('max_leaf_nodes', 2, 50,step = 1)
  max_depth = trial.suggest_int('max_depth', 1, 15,step = 1)
  min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 50,step = 1)
  max_features = trial.suggest_float('max_features',0.1,1)
  l2_regularization = trial.suggest_float('l2_regularization',0.01,1)
  class_0_weight = trial.suggest_float('class_0_weight',0.01,1)
  class_1_weight = trial.suggest_float('class_1_weight',0.01,1)
  n_last = trial.suggest_int('n_last', 5, 25,step = 1)
  class_1_percent = trial.suggest_float('class_1_percent',0.01,1)
  random_state = trial.suggest_int('random_state', 1, 1000000)


  # создаем модель
  optuna_gradient_boosting = HistGradientBoostingClassifier(
      learning_rate= learning_rate,
      max_iter = max_iter,
      max_leaf_nodes =max_leaf_nodes,
      max_depth = max_depth,
      min_samples_leaf = min_samples_leaf,
      max_features=max_features,
      l2_regularization = l2_regularization,
      class_weight={0:class_0_weight,1:class_1_weight},
      random_state=42)

  # с помощью функции features_from_transform_data_torow извлечем  
  # из данных transform_data_torow
  # признаки сооствествующие n_last последним клиенским операциям
  list_n_last_features = features_from_transform_data_torow(n_last,count_features,25)

  # загружаем обучующие наборы
  X_train_pd = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas()
  y_train_pd = pd.read_csv('target/target_train.csv')

  # поготовим данные y_train_pd к работе с функцией class_1_percent_samples
  y_train_pd.set_index('id',drop=False,inplace=True)

  # подготовим данные для обучения модели
  # с помощью функции class_1_percent_samples зададим долю класса 1
  list_c1_percent_id = class_1_percent_samples(y_train_pd,class_1_percent,random_state=random_state)[:1000000]

  # сформируем данные для обучения модели
  X_train_balanced = np.array(X_train_pd.loc[list_c1_percent_id])[:,list_n_last_features]
  y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

  # освободим память от "тяжелых" и ненужных файлов
  del X_train_pd, y_train_pd
  gc.collect()

  # я не воспользовался параметром timeout метода optimize потому, что  
  # optimize отставливает поиск параметров, если время обучения 
  # превышает timeout. Мне нужно чтобы поиск параметров продолжился.
  try:
    # для модуля func_time_out упакуем обучение модели в функцию try_func
    def try_func():
            return optuna_gradient_boosting.fit(X_train_balanced,y_train_balanced)
    # обучаем модель с ограничением по времени 10 минут (600 секунд)
    optuna_gradient_boosting = func_timeout.func_timeout(600, try_func)

    # удаляем крупные файлы чтобы высвободить память 
    del X_train_balanced, y_train_balanced
    gc.collect()

    # подгружаем данные для тестирования
    X_train = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas().to_numpy()[:,list_n_last_features]
    y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
    X_valid = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_valid').to_pandas().to_numpy()[:,list_n_last_features]
    y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()

    # делаем предсказание на обучающем и валидационном наборе и считаем метрики
    roc_train = metrics.roc_auc_score(y_train, optuna_gradient_boosting.predict_proba(X_train)[:,1])
    roc_valid = metrics.roc_auc_score(y_valid, optuna_gradient_boosting.predict_proba(X_valid)[:,1])
    f1_score = metrics.f1_score(y_valid, optuna_gradient_boosting.predict(X_valid))
    recall_1 = metrics.recall_score(y_valid, optuna_gradient_boosting.predict(X_valid),average=None,zero_division=0)[1]
    precision_1 = metrics.precision_score(y_valid, optuna_gradient_boosting.predict(X_valid),average=None,zero_division=0)[1]

    # удаляем крупные файлы чтобы высвободить память 
    del X_train, X_valid, y_train, y_valid 
    gc.collect()

  except func_timeout.FunctionTimedOut:
    # освободим память от "тяжелых" и ненужных файлов
    del X_train_balanced, y_train_balanced
    gc.collect()

    # обьявим метрики пустыми значениями
    roc_train = 0
    roc_valid = 0
    f1_score = 0
    recall_1 = 0
    precision_1 = 0
    pass

  return roc_train, roc_valid, f1_score, recall_1, precision_1

In [ ]:
# cоздаем объект исследования
# чтобы модель не переобучалась минимизируем roc_train 
# и максимизируем roc_valid
optuna_study_hsbs = optuna.create_study(study_name='HistGradientBoostingClassifier_'+feature_space+'_torow', 
                               directions=['maximize','maximize','maximize','maximize','maximize'], 
                               sampler=optuna.samplers.TPESampler(),
                               pruner='Hyperband',
                               storage='sqlite:///optuna_studies.db',
                               load_if_exists=True)

# ну случай удаления обучения
# optuna.delete_study(study_name='HistGradientBoostingClassifier_'+feature_space+'_torow', storage='sqlite:///optuna_studies.db')

In [ ]:
# ищем лучшую комбинацию гиперпараметров n_trials раз
optuna_study_hsbs.optimize(optuna_hgbc, n_trials=300)

In [ ]:
# из полученного результат соврмируем Data Frame
optuna_study_hsbs_pd = optuna_study_hsbs.trials_dataframe()
# переименуем столбы
optuna_study_hsbs_pd.rename(columns={
    'values_0': 'roc_train',
    'values_1': 'roc_valid',
    'values_2': 'f1_score',
    'values_3': 'recall_1',
    'values_4': 'precision_1'
},inplace=True)
# Выделим время потраченное на обучение модели
last_trail = optuna_study_hsbs_pd['number'].max()
optuna_study_hsbs_pd.tail(5)


In [ ]:
# покаждем статистику обучения
print('Максимальное значение метрики ROC AUC на валидационном наборе:',round(optuna_study_hsbs_pd['roc_valid'].max(),3))
print('Среднее значение метрики ROC AUC на валидационном наборе:',round(optuna_study_hsbs_pd['roc_valid'].mean(),3))
print('Максимальное значение метрики f1_score на валидационном наборе:',round(optuna_study_hsbs_pd['f1_score'].max(),3))
print('Среднее значение метрики f1_score на валидационном наборе:',round(optuna_study_hsbs_pd['f1_score'].mean(),3))
print('Максимальное значение метрики recall_1 на валидационном наборе:',round(optuna_study_hsbs_pd['recall_1'].max(),3))
print('Среднее значение метрики recall_1 на валидационном наборе:',round(optuna_study_hsbs_pd['recall_1'].mean(),3))
print('Максимальное значение метрики precision_1 на валидационном наборе:',round(optuna_study_hsbs_pd['precision_1'].max(),3))
print('Среднее значение метрики precision_1 на валидационном наборе:',round(optuna_study_hsbs_pd['precision_1'].mean(),3))

In [ ]:
# построим график важности гиперпарметров
optuna.visualization.plot_param_importances(optuna_study_hsbs, target_name='Score')

In [ ]:
# Построим зависимость метрики precision_1
# max_iter

fig = px.scatter(
    data_frame=optuna_study_hsbs_pd,
    x='precision_1', #ось абсцисс
    y=['f1_score', 'recall_1'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость метрик precision от recall', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision',
    yaxis_title='Величина метрик recall и f1-score')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики precision_1 от гиперпараметров
# max_iter

fig = px.scatter(
    data_frame=optuna_study_hsbs_pd,
    x='precision_1', #ось абсцисс
    y=['params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight', 'params_l2_regularization',
       'params_learning_rate', 'params_max_depth', 'params_max_features',
       'params_max_iter', 'params_max_leaf_nodes', 'params_min_samples_leaf',
       'params_n_last'
    ], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики f1_score от гиперпараметров
# max_iter

fig = px.scatter(
    data_frame=optuna_study_hsbs_pd,
    x='f1_score', #ось абсцисс
    y=['params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight', 'params_l2_regularization',
       'params_learning_rate', 'params_max_depth', 'params_max_features',
       'params_max_iter', 'params_max_leaf_nodes', 'params_min_samples_leaf',
       'params_n_last'
    ], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость f1-score от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики f1-score',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики roc_valid от гиперпараметров
fig = px.scatter(
    data_frame=optuna_study_hsbs_pd,
    x='roc_valid', #ось абсцисс
    y=['params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight', 'params_l2_regularization',
       'params_learning_rate', 'params_max_depth', 'params_max_features',
       'params_max_iter', 'params_max_leaf_nodes', 'params_min_samples_leaf',
       'params_n_last'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость ROC AUC от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики ROC AUC',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрик качества модели от number

fig = px.scatter(
    data_frame=optuna_study_hsbs_pd[(optuna_study_hsbs_pd['recall_1']>0.09)&
                                    (optuna_study_hsbs_pd['precision_1']>0.09)&
                                    (optuna_study_hsbs_pd['roc_valid']>0.63)
                                    ],
    x='number', #ось абсцисс
    y=['roc_train','roc_valid','recall_1','precision_1'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина optuna number',
    yaxis_title='Величина метрики')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

<span style="color:Blue">

Вывод:  

Их всех выбираем точку $number =91$.   
В этой точке оптимальные значения метрик $recall_1$ и $precision_1$, а  
также относительно высокая метрика $ROC AUC$.   
Посмотрим каким значениям гиперпараметров соотвествует выбранная точка.

In [ ]:
# определим номер лучшге варианта
best_optuna_number = 91

# сформируем словарь лучших гипирпарметров RandomForestClassifier
best_param_rf = {
    'learning_rate' : optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['params_learning_rate'].iloc[0],
    'max_iter' : optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['params_max_iter'].iloc[0],
    'max_leaf_nodes' : optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['params_max_leaf_nodes'].iloc[0],
    'max_depth' : optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['params_max_depth'].iloc[0],
    'min_samples_leaf' : optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['params_min_samples_leaf'].iloc[0],
    'max_features' : optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['params_max_features'].iloc[0],
    'l2_regularization' : optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['params_l2_regularization'].iloc[0],
    'class_weight' : {0: optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['params_class_0_weight'].iloc[0],
                      1: optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['params_class_1_weight'].iloc[0],}
    }

# определим перменные для лучших значений параметров
best_n_last = optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['params_n_last'].iloc[0]
best_class_1_percent = optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['params_class_1_percent'].iloc[0]
best_random_state = optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['params_random_state'].iloc[0]


# Выведем принятые наилучшие праметры
print('best learning rate:',round(best_param_rf['learning_rate'],3))
print('best max iter:',best_param_rf['max_iter'])
print('best max leaf nodes:',best_param_rf['max_leaf_nodes'])
print('best max depth:',best_param_rf['max_depth'])
print('best min samples leaf:',best_param_rf['min_samples_leaf'])
print('best max features:',round(best_param_rf['max_features'],3))
print('best l2 regularization:',round(best_param_rf['l2_regularization'],3))
print('best class 0 weight:',round(best_param_rf['class_weight'][0],3))
print('best class 1 weight:',round(best_param_rf['class_weight'][1],3))

print('best class 1 percent:',round(best_class_1_percent,3))
print('best n last:',best_n_last)
print('best random state:',best_random_state)
print('time for best train:',round(optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['duration'].iloc[0].seconds/60),'minutes')

print()
print('ROC AUC на обучающем наборе:', round(optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['roc_train'].iloc[0],3))
print('ROC AUC на валидационном наборе:', round(optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['roc_valid'].iloc[0],3))
print('precision класса 1:', round(optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['precision_1'].iloc[0],3))
print('recall класса 1:', round(optuna_study_hsbs_pd[optuna_study_hsbs_pd['number']==best_optuna_number]['recall_1'].iloc[0],3))

##### <span style="color:MediumSlateBlue">Обучение модели с лучшими параметрами</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'service'
count_features = dict_spaces[feature_space]

# с помощью функции features_from_transform_data_torow извлечем  
# из данных transform_data_torow
# признаки сооствествующие n_last последним клиенским операциям
list_n_last_features = features_from_transform_data_torow(best_n_last,count_features,25)

# загружаем обучующие наборы
X_train_pd = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas()
y_train_pd = pd.read_csv('target/target_train.csv')

# поготовим данные y_train_pd к работе с функцией class_1_percent_samples
y_train_pd.set_index('id',drop=False,inplace=True)

# подготовим данные для обучения модели
# с помощью функции class_1_percent_samples зададим долю класса 1
list_c1_percent_id = class_1_percent_samples(y_train_pd,best_class_1_percent,random_state=best_random_state)[:1000000]

# сформируем данные для обучения модели
X_train_balanced = np.array(X_train_pd.loc[list_c1_percent_id])[:,list_n_last_features]
y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

# освободим память от "тяжелых" и ненужных файлов
del X_train_pd, y_train_pd
gc.collect()


# обучаем модель HistGradientBoostingClassifier с наилучшеми параметрами
gradient_boosting = HistGradientBoostingClassifier(
        **best_param_rf,
        random_state=42)
gradient_boosting.fit(X_train_balanced,y_train_balanced)

# освободим память от "тяжелых" и ненужных файлов
del X_train_balanced, y_train_balanced
gc.collect()

# подгружаем данные для тестирования
X_train = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas().to_numpy()[:,list_n_last_features]
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
X_valid = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_valid').to_pandas().to_numpy()[:,list_n_last_features]
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()

# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_gb__pred_proba = gradient_boosting.predict_proba(X_train)[:,1]
y_valid_gb_pred_proba = gradient_boosting.predict_proba(X_valid)[:,1]

# Делаем предсказание для валидационной выборки
y_valid_gb_pred = gradient_boosting.predict(X_valid)

# освободим память от "тяжелых" и ненужных файлов
del X_train, X_valid
gc.collect()

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_gb__pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_gb_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_valid,y_valid_gb_pred,zero_division=0))

# time: 2m 30s

##### <span style="color:MediumSlateBlue">Анализ влияния порога классификации на качество модели</span>

In [ ]:
# чтобы проше обращаться к каждому обьекту в данных
# преобразуем y_valid_LR_pred к Series типу
y_valid_gb_pred_proba = pd.Series(y_valid_gb_pred_proba)

# формируем список из необходимых значений thresholds
thresholds = np.arange(0.01, 1, 0.01)

# сформируем списко для заполнения данными результатов анализа
threshold_data = []


for threshold in thresholds:
        # сформируем список для заполнения данными текушего threshold
        threshold_current_data = [threshold]

        #клиентов, для которых вероятность дефолта > threshold, относим к классу 1
        #в противном случае — к классу 0
        y_valid_gb_pred = y_valid_gb_pred_proba.apply(lambda x: 1 if x>threshold else 0)


        #Считаем метрики для класса 0 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(y_valid, y_valid_gb_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.precision_score(y_valid, y_valid_gb_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.f1_score(y_valid, y_valid_gb_pred,average=None,zero_division=0)[0],3))

        #Считаем метрики для класса 1 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(y_valid, y_valid_gb_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.precision_score(y_valid, y_valid_gb_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.f1_score(y_valid, y_valid_gb_pred,average=None,zero_division=0)[1],3))

        # добавляем полученные данные в список threshold_data
        threshold_data.append(threshold_current_data)

        # из полученных данных сформируем dataframe
        thresholds_pd =pd.DataFrame(
        columns=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'],
        data = threshold_data)

# time: 17s

In [ ]:
# Визуализируем метрики на валидационном наборе при различных threshold
fig_threshold = px.line(
    data_frame=thresholds_pd,
    x='threshold', #ось абсцисс
    y=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'], #ось ординат
)
fig_threshold.update_layout(
    title ={
        'text':'Зависимость качества модели от threshold', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Порог классификации threshold',
    yaxis_title='Величина метрики')
fig_threshold.update_xaxes(showspikes=True)
fig_threshold.update_yaxes(showspikes=True)

fig_threshold.show()

Метрика $precision$ показывает на сколько сильно ошиблась модель при определиние класса.  
Чем выше метрика, тем меньше ошиблась модель. 

$$precision = \frac{TP}{TP+FP}$$

Метрика $recall$ показывает способность модели найти класс.  
Чем выше метрика, тем выше способность модели находить класс.  

$$recall = \frac{TP}{TP+FN}$$

Метрика $f1-score$ показывает баланс между $precision$ и $recall$ 

$threshold$- порог класификации модели.  
Все объекты для которых $у_{\text{proba pred}} < threshold$, модель относит к класс 0.  
Соотвественно, если $у_{\text{proba pred}}  > threshold$ модель относит обьект к классу 1.  
Для идельаной модели $threshold$, является границей между областями класса 0 и класса 1.  

<span style="color:Blue">

Выводы по результам анализа:  

Для улучшения качества модели по метрике $ROC AUC$, нет смысла сдвигать порог   
классификации. Но зависимость метрик качества модели от $threshold$, может рассказать о  
способности модели искать класс 1 и класс 0.

При фокусировке модели на классе 1 (более низкий $threshold$):  
- на классе 1 метрика $precision$ уменьшается, метрика $recall$ растет;  
- на классе 0 метрика $precision$ растет, метрика $recall$ уменьшается.  

При фокусировке модели на классе 0 (более высокий $threshold$):  
- на классе 1 метрика $precision$ увеличивается, метрика $recall$ уменьшается;  
- на классе 0 метрика $precision$ уменьшается, метрика $recall$ растет.  

Обьсняется тем, что в условия дисбаланса модель научилась хорошо определять класс 0 и плохо класс 1.  

При фокусировке модели на классе 1 (более низкий $threshold$), все больше обьектов отнесены к классу 1.   
Так как $precision$ класса 1 уменьшается, значит в области класса 1 стало больше обектов класса 0,  
что явлется закономерным для "идеальной" модели.  
Однако при этом, $recall$ класса 1 растет, значит в добавленных обьектах также присутвует класс 1.  
Это как раз указывает на то, что модель не нучилась находить среди класса 0 класс 1. В области с     
высокой вероятностью класса 0 (более низкий $threshold$), находятся обьекты класса 1.  

Фокусировка на классе 0 (более высокий $threshold$), заставляет модель все больше обьектов из области класса 1  
относить к классу 0. При этом увеличение метрики $precision$ класса 1 говорит о том, что в области класса 1     
становится меньше 0, а значит модель умеет среди класса 1, находить класс 0.  
После  $threshold=0.83$ модель впринципе теряет способность отличать класс 1. У модели нет обьектов класса 1   
в которых она "уверена" на 90+%.

Обратим внимание на $threshold=0.49$, в этой точке находится баланс межу метриками качества модели:
$$precision_1 = recall_1 = \text{f1-score}_1 = 0.103$$
$$precision_0 = recall_0 = \text{f1-score}_0 = 0.966$$

Для модели с лучшим качеством эти две точки должны находится максимально близко к друг другу и иметь   
высокое значение метрик $precision_1$ и $recall_1$.  
Для класса 0 это условие выполнется, что также говорит о том, что модель научилась искать этот класс.    
Для класса 1 это условие не выполнется, что также говорит о том, что модель плохо ищет класс 1.

##### <span style="color:MediumSlateBlue">Построение ROC-кривой выбранной модели на тестовом наборе</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'service'
count_features = dict_spaces[feature_space]

In [ ]:
# подгружаем данные для тестирования
X_train = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas().to_numpy()[:,list_n_last_features]
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
X_valid = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_valid').to_pandas().to_numpy()[:,list_n_last_features]
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()
X_test = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_test').to_pandas().to_numpy()[:,list_n_last_features]
y_test = pd.read_csv('target/target_test.csv')['flag'].to_numpy()

In [ ]:
print("Размеры обучающего набора",X_train.shape,y_train.shape)
print("Размеры валидационного набора",X_valid.shape,y_valid.shape)
print("Размеры тестового набора",X_test.shape,y_test.shape)
# time: 1m 43s

In [ ]:
# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_gb_pred_proba = gradient_boosting.predict_proba(X_train)[:,1]
y_valid_gb_pred_proba = gradient_boosting.predict_proba(X_valid)[:,1]
y_test_gb_pred_proba = gradient_boosting.predict_proba(X_test)[:,1]

# Делаем предсказание для валидационной выборки
y_test_gb_pred = gradient_boosting.predict(X_test)

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_gb_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_gb_pred_proba),3))
print('ROC AUC на тестовом наборе', round(metrics.roc_auc_score(y_test, y_test_gb_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_test,y_test_gb_pred,zero_division=0))

In [ ]:
# Для расчета значений TPR и FPR используем функцию roc_curve
fpr_train, tpr_train, _ = metrics.roc_curve(y_train, y_train_gb_pred_proba)
fpr_valid, tpr_valid, _ = metrics.roc_curve(y_valid, y_valid_gb_pred_proba)
fpr_test, tpr_test, _ = metrics.roc_curve(y_test, y_test_gb_pred_proba)

# рассчитаем площадь под кривой
auc_train = round(metrics.roc_auc_score(y_train, y_train_gb_pred_proba),3)
auc_valid = round(metrics.roc_auc_score(y_valid, y_valid_gb_pred_proba),3)
auc_test = round(metrics.roc_auc_score(y_test, y_test_gb_pred_proba),3)

trace_train = go.Scatter(x=fpr_train, y=tpr_train, mode='lines', name='AUC_train = %0.2f' % auc_train,
                   line=dict(color='red', width=2),
                   hovertemplate='train<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_train])

trace_valid = go.Scatter(x=fpr_valid, y=tpr_valid, mode='lines', name='AUC_valid = %0.2f' % auc_valid,
                   line=dict(color='green', width=2),
                   hovertemplate='valid<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_valid])

trace_test = go.Scatter(x=fpr_test, y=tpr_test, mode='lines', name='AUC_test = %0.2f' % auc_test,
                   line=dict(color='darkorange', width=2),
                   hovertemplate='test<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_test])


reference_line = go.Scatter(x=[0,1], y=[0,1], mode='lines', name='Reference Line',
                            line=dict(color='navy', width=2, dash='dash'))

fig_roc = go.Figure(data=[trace_train,trace_valid,trace_test,reference_line])

fig_roc.update_layout(
    title ={
        'text':'ROC-кривая', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 1350,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate')

fig_roc.update_xaxes(showspikes=True)
fig_roc.update_yaxes(showspikes=True)


fig_roc.show()

<span style="color:Blue">

Вывод по ROC-кривой:
1. Модель выдает сталибильное качество на валидационном и тестовом наборах, а значит  
не переобучена - разброс ($variance$) не большой. Хотя, качество модели на обучающем наборе,  
заметно лучше.
2. Смещение ($bias$) модели чуть лучше, чем у модели $Random$ $Forest$ $Classifier$,  
обученной на тех же данных.  
На тестовом наборе полученно качество $ROC AUC = 0.64$, против $ROC AUC = 0.63$

### <span style="color:RoyalBlue">Построение модели на сбалансированных данных подпрострнства date stat</span>

#### <span style="color:MediumBlue">Формирование данных для обучения</span>

Анализ исходных данных, в частности target, показал что представленные данные   
имееют перекос: 96% клиентов у которых не случился дефолт (flag = 0),    
против 4 % - для которых произошел дефолт (flag = 1).  
С помощью функции class_1_percent_samples, сформируем сбаланисрованную выборку  
для обучения моделей.
За основу сбалансировных данных возьмем данные клиентов с дефолтом  
и к ним будем добирать необходимое количество клиентов до сбалансированности. 


Функция class_1_percent_samples принемает две переменные:
- target_data - массив с id и целевой переменной
- class_1_percent - процент класса 1 в результирующем массиве

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'date'
count_features = dict_spaces[feature_space]

In [ ]:
# сформируем данные для анализа сбалансированности обучающих данных
X_train_pd = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()
y_train_pd = pd.read_csv('target/target_train.csv')

In [ ]:
# поготовим данные y_train_pd к работе с функцией class_1_percent_samples
y_train_pd.set_index('id',drop=False,inplace=True)
y_train_pd.head(4)

In [ ]:
# взглянем на сблансиорованность данных в y_train
y_train_pd['flag'].value_counts(normalize=True)

In [ ]:
# сбалансируем данные в y_train_pd
list_c1_percent_id = class_1_percent_samples(y_train_pd,0.5)
# list_c1_percent_features - список 'id' из y_train_pd соотвествующих  
# заданному проценту класса 1
len(list_c1_percent_id)

In [ ]:
# проверим сбалансированность полученного y_train
y_train_pd.loc[list_c1_percent_id]['flag'].value_counts(normalize=True)

В отличие от данных в *transform_data_torow*, где модели показываеются последние N  
операций клиента, в данные в *transform_data_stat* сфомированы из различных статистических   
характеристик ряда определяющего клиентскую историю в банке. 

Соотвественно, если в данных *transform_data_torow* была сильная корреляция между признаками,  
то я не смог бы ее убрать, иначе потярется смысл "показать последовательный ряд в историии клиента".  
Признаки в *transform_data_stat* не имеют такой особенности. Поэтому, необходимо проанализировать  
данные в *transform_data_stat* и, в случае сильной коррреляции признаков, устранить ее.

In [ ]:
# подгрузим DataFrame с матрицей корреляции признаков
corr_matrix = fp.ParquetFile('base_models/data/stat/corr_matrix_'+feature_space).to_pandas()
corr_matrix.shape

In [ ]:
# с помощью функции corr_transrom_to_power получим
# силу корреляции матрицы и список не коррелируемых признаков

# для начального анализа зададим порог силы корреляции, при котором признаки 
# будут отнесены к "сильно скоррелированными" равным 0.6
ncorr_matrix,list_ncorr_features,corr_power =corr_transform_to_force(corr_matrix,threshold=0.6)

# Выведем результаты преобразования
print('Исходное количество признаков: ', corr_matrix.shape[0])
print('Количество не коррелируемых признаков: ', len(list_ncorr_features))
# сила корреляции признаков
print('Отношение коррелируемых исходному количеству признаков (сила корреляции): ',corr_power)

In [ ]:
# подготовим данные для обучения
X_train_balanced = X_train_pd.loc[list_c1_percent_id]
y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag']
# сформируем данные для проверики модели
# сформируем данные для анализа сбалансированности обучающих данных
X_train = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()
y_train = pd.read_csv('target/target_train.csv')['flag']
X_valid = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_valid').to_pandas()
y_valid = pd.read_csv('target/target_valid.csv')['flag']

In [ ]:
# посмотрим на структуру данных в X_train_balanced
X_train_balanced.head(4)

In [ ]:
# посмотрим на структуру данных в y_train_balanced
y_train_balanced.head(4)

In [ ]:
# проверим что выборки действительно совпадают по id
X_train_balanced.index.to_list() == y_train_balanced.index.to_list()

#### <span style="color:MediumBlue">Построение модели Logistic Regression</span>

##### <span style="color:MediumSlateBlue">Baseline обучение модели LogisticRegression</span>

In [ ]:
# обучаем модель логистической регрессии
logistic_regression = linear_model.LogisticRegression(random_state=42, max_iter=10000)
logistic_regression.fit(X_train_balanced[list_ncorr_features],y_train_balanced)

# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_LR_pred_proba = logistic_regression.predict_proba(X_train[list_ncorr_features])[:,1]
y_valid_LR_pred_proba = logistic_regression.predict_proba(X_valid[list_ncorr_features])[:,1]

# Делаем предсказание для валидационной выборки
y_valid_LR_pred = logistic_regression.predict(X_valid[list_ncorr_features])

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_LR_pred_proba),3))
print('ROC AUC на тестовом наборе', round(metrics.roc_auc_score(y_valid, y_valid_LR_pred_proba),3))

print('Основные метрики на тестовом наборе:')
print(metrics.classification_report(y_valid,y_valid_LR_pred))

# time: 3m 5s

##### <span style="color:MediumSlateBlue">Анализ влияния scaler преобразования на качество и скорость схождения модели</span>

In [ ]:
# формируем список из необходимых scaler
scalers = [MinMaxScaler(),RobustScaler() ,StandardScaler()]

# сформируем списко для заполнения данными результатов анализа
scalers_data = []

# установим ограничение на время обучения модели в 600 секунд (10 минут)
time_out = 600

for scaler in scalers:
        # для того чтобы следить за выполнением цикла введем индикатор
        print('current scaler: ', scaler)

         # сформируем список для заполнения данными текушего scaler
        current_scaler_data = [scaler]
        
        # выполним scaler преобразование
        X_train_s_balanced = scaler.fit_transform(X_train_balanced[list_ncorr_features])
        X_train_s = scaler.transform(X_train[list_ncorr_features])
        X_valid_s = scaler.transform(X_valid[list_ncorr_features])

        try:
                # для модуля func_time_out упакуем обучение модели в функцию try_func
                def try_func():
                        logistic_regression = linear_model.LogisticRegression(random_state=42, max_iter=10000)
                        return logistic_regression.fit(X_train_s_balanced,y_train_balanced)
                    
                # фиксируем время начала
                start_time = time.time()
                
                # запускаем обучение модели с ограничением по времени
                logistic_regression = func_timeout.func_timeout(time_out, try_func)
                
                # фиксируем время окончания обучения
                end_time = time.time()
                
                # запишем время обучения модели
                current_scaler_data.append(round(end_time-start_time))

                # Делаем предсказание для обучающей и валидационной выборки
                y_valid_lr_pred = logistic_regression.predict(X_valid_s)
                
                # для метрик ROC AUC делаем предсказание модели для обучающей и валидационной выборки  
                # в виде вероятности принадлежности к классу 1 
                y_train_lr_pred_proba = logistic_regression.predict_proba(X_train_s)[:,1]
                y_valid_lr_pred_proba = logistic_regression.predict_proba(X_valid_s)[:,1]

                #Считаем метрики ROC AUC yна обуающем и валидационном наборе и добавляем их в спискок
                current_scaler_data.append(round(metrics.roc_auc_score(y_train, y_train_lr_pred_proba),3))
                current_scaler_data.append(round(metrics.roc_auc_score(y_valid, y_valid_lr_pred_proba),3))

                #Считаем метрики для класса 0 на валидационном наборе и добавляем их в список
                current_scaler_data.append(round(metrics.recall_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))
                current_scaler_data.append(round(metrics.precision_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))


                #Считаем метрики для класса 1 на валидационном набопе и добавляем их в список
                current_scaler_data.append(round(metrics.recall_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))
                current_scaler_data.append(round(metrics.precision_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))

                # добавляем полученные данные в список scalers_data
                scalers_data.append(current_scaler_data)

                # чтобы подстраховаться на случай ошибки : "The Kernel crashed while executing code"
                # сохраним в локальную папку удачную итерацию
                # для этого из полученных данных сформируем dataframe
                # из полученных данных сформируем dataframe
                scaler_pd =pd.DataFrame(
                        columns=['scaler','time_fit',
                                 'roc_train','roc_valid', 
                                 'recall_0','precision_0',
                                 'recall_1','precision_1'],
                        data = scalers_data)
                # сохраняем dataframe
                scaler_pd.to_csv('data_recovery/scaler_pd.csv',index=False)
               
                # очистим output от прошедшей итерации
                clear_output()

        except func_timeout.FunctionTimedOut:
                # если время обучения привышает лимит
                # запишем в current_scaler_data соотвествующую данные
                current_scaler_data = [scaler,'>30m',0,0,0,0,0,0]
                # добавляем полученные данные в список scalers_data
                scalers_data.append(current_scaler_data)

                # чтобы подстраховаться на случай ошибки : "The Kernel crashed while executing code"
                # сохраним в локальную папку удачную итерацию
                # для этого из полученных данных сформируем dataframe
                # из полученных данных сформируем dataframe
                scaler_pd =pd.DataFrame(
                        columns=['scaler','time_fit',
                                 'roc_train','roc_valid', 
                                 'recall_0','precision_0',
                                 'recall_1','precision_1'],
                        data = scalers_data)
                # сохраняем dataframe
                scaler_pd.to_csv('data_recovery/scaler_pd.csv',index=False)
                
                # очистим output от прошедшей итерации
                clear_output()
                # переходим к следующему scaler
                pass
# очистим output
clear_output()

# сформируем данные соотвествующие лучщему ROC AUC на валидацинном наборе
best_roc_valid_data = scaler_pd[scaler_pd['roc_valid']==scaler_pd['roc_valid'].max()]

# выведем лучший результат на валидационном наборе
print('result:')
print('best scaler on valid:',best_roc_valid_data.iloc[0]['scaler'])
print('ROC AUC on train:', best_roc_valid_data.iloc[0]['roc_train'])
print('ROC AUC on valid:', best_roc_valid_data.iloc[0]['roc_valid'])
print('Time fit for best scaler:', best_roc_valid_data.iloc[0]['time_fit'],' seconds')

# выведем полученный dataframe
scaler_pd

# time: 

##### <span style="color:MediumSlateBlue"> Анализ влияния solver оптимизатора на качество и скорость обучения модели</span>

Проверим качество и скорость сходимости модели при разных solver  
Проверку осуществим через цикл  
Чтобы модель не сходилась бесконечно с помощью модуля func_timeout  
поставим ограничение времени схождения в 10 минут

In [ ]:
# подготовим данные для обучения модели

# обьявим scaler
scaler = StandardScaler()
# выполним scaler преобразование
X_train_s_balanced = scaler.fit_transform(X_train_balanced[list_ncorr_features])
X_train_s = scaler.transform(X_train[list_ncorr_features])
X_valid_s = scaler.transform(X_valid[list_ncorr_features])

# time: 10s

In [ ]:
# Зададим список solvers
solvers_list = ['lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga']

# сформируем список для заполнения
solvers_data = []

# установим ограничение на время обучения модели в 180 секунд (3 минут)
time_out = 180

for solver in solvers_list:
        # для того чтобы следить за выполнением цикла введем индикатор
        print('current solver: ', solver)

         # сформируем список для заполнения данными текушего solver
        current_solver_data = [solver]
        
        try:
                # для модуля func_time_out упакуем обучение модели в функцию try_func
                def try_func():
                        logistic_regression = linear_model.LogisticRegression(solver= solver,
                                                                  random_state=42, 
                                                                  max_iter=10000)
                        return logistic_regression.fit(X_train_s_balanced,y_train_balanced)
                    
                # фиксируем время начала
                start_time = time.time()
                
                # запускаем обучение модели с ограничением по времени
                logistic_regression = func_timeout.func_timeout(time_out, try_func)
                
                # фиксируем время окончания обучения
                end_time = time.time()
                
                # запишем время обучения модели
                current_solver_data.append(round(end_time-start_time))

                # Делаем предсказание для обучающей и валидационной выборки
                y_valid_lr_pred = logistic_regression.predict(X_valid_s)
                
                # для метрик ROC AUC делаем предсказание модели для обучающей и валидационной выборки  
                # в виде вероятности принадлежности к классу 1 
                y_train_lr_pred_proba = logistic_regression.predict_proba(X_train_s)[:,1]
                y_valid_lr_pred_proba = logistic_regression.predict_proba(X_valid_s)[:,1]

                #Считаем метрики ROC AUC yна обуающем и валидационном наборе и добавляем их в спискок
                current_solver_data.append(round(metrics.roc_auc_score(y_train, y_train_lr_pred_proba),3))
                current_solver_data.append(round(metrics.roc_auc_score(y_valid, y_valid_lr_pred_proba),3))

                #Считаем метрики для класса 0 на валидационном наборе и добавляем их в список
                current_solver_data.append(round(metrics.recall_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))
                current_solver_data.append(round(metrics.precision_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))


                #Считаем метрики для класса 1 на валидационном набопе и добавляем их в список
                current_solver_data.append(round(metrics.recall_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))
                current_solver_data.append(round(metrics.precision_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))

                # запишем данные полученные на одной итерации
                solvers_data.append(current_solver_data)

                # чтобы подстраховаться на случай ошибки : "The Kernel crashed while executing code"
                # сохраним в локальную папку удачную итерацию
                # для этого из полученных данных сформируем dataframe
                # из полученных данных сформируем dataframe
                solvers_pd =pd.DataFrame(
                        columns=['solver','time_fit',
                                 'roc_train','roc_valid', 
                                 'recall_0','precision_0',
                                 'recall_1','precision_1'],
                        data = solvers_data)
                # сохраняем dataframe
                solvers_pd.to_csv('data_recovery/solvers_pd.csv',index=False)
               
                # очистим output от прошедшей итерации
                clear_output()

        except func_timeout.FunctionTimedOut:
                # если время обучения привышает лимит
                # запишем в current_solver_data соотвествующую данные
                current_solver_data = [solver,'>3m',0,0,0,0,0,0]
                # добавляем полученные данные в список solvers_data
                solvers_data.append(current_solver_data)

                # чтобы подстраховаться на случай ошибки : "The Kernel crashed while executing code"
                # сохраним в локальную папку удачную итерацию
                # для этого из полученных данных сформируем dataframe
                # из полученных данных сформируем dataframe
                solvers_pd =pd.DataFrame(
                        columns=['solver','time_fit',
                                 'roc_train','roc_valid', 
                                 'recall_0','precision_0',
                                 'recall_1','precision_1'],
                        data = solvers_data)
                # сохраняем dataframe
                solvers_pd.to_csv('data_recovery/solvers_pd.csv',index=False)
                
                # очистим output от прошедшей итерации
                clear_output()
                # переходим к следующему solver
                pass
# очистим output
clear_output()

# сформируем данные соотвествующие лучщему ROC AUC на валидацинном наборе
best_roc_valid_data = solvers_pd[solvers_pd['roc_valid']==solvers_pd['roc_valid'].max()]

# выведем лучший результат на валидационном наборе
print('result:')
print('best solver on valid:',best_roc_valid_data.iloc[0]['solver'])
print('ROC AUC on train:', best_roc_valid_data.iloc[0]['roc_train'])
print('ROC AUC on valid:', best_roc_valid_data.iloc[0]['roc_valid'])
print('Time fit for best solver:', best_roc_valid_data.iloc[0]['time_fit'],' seconds')

# выведем полученный dataframe
solvers_pd

# time: 2m 35s

##### <span style="color:MediumSlateBlue">Подбор гиперпараметров модели</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'date'
count_features = dict_spaces[feature_space]

# для работы функции corr_transform_to_force
# подгрузим DataFrame с матрицей корреляции признаков
corr_matrix = fp.ParquetFile('base_models/data/stat/corr_matrix_'+feature_space).to_pandas()

# для выбора sceler преобразвания нам понадобится словарь
dict_scalers ={
    'MinMaxScaler': MinMaxScaler(),
    'RobustScaler':RobustScaler(),
    'StandardScaler':StandardScaler()}

# настроим оптимизацию гипер параметров
def optuna_lg(trial):
  # задаем пространства поиска гиперпараметров
  solver = trial.suggest_categorical("solver", ['lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky'])
  C = trial.suggest_float('C',0.01,1)
  class_0_weight = trial.suggest_float('class_0_weight',0.2,0.6)
  class_1_weight = trial.suggest_float('class_1_weight',0.2,0.6)
  threshold = trial.suggest_float('threshold',0.6,1)
  class_1_percent = trial.suggest_float('class_1_percent',0.01,0.3)
  scaler = trial.suggest_categorical("scaler", ['MinMaxScaler','RobustScaler','StandardScaler'])
  random_state = trial.suggest_int('random_state', 1, 1000000)


  # создаем модель
  optuna_log_reg = linear_model.LogisticRegression(
      solver=solver,
      C=C,
      class_weight={0:class_0_weight,1:class_1_weight},
      random_state=random_state,
      max_iter=10000)

  # с помощью функции corr_transform_to_force получим
  # список не коррелируемых признаков
  list_ncorr_features = corr_transform_to_force(corr_matrix,threshold=threshold)[1]

  # обьявим scaler
  scaler = dict_scalers[scaler]

  # сформируем данные для анализа сбалансированности обучающих данных
  X_train_pd = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
  y_train_pd = pd.read_csv('target/target_train.csv')

  # поготовим данные y_train_pd к работе с функцией class_1_percent_samples
  y_train_pd.set_index('id',drop=False,inplace=True)

  # подготовим данные для обучения модели
  # с помощью функции class_1_percent_samples зададим долю 
  # класса 1
  list_c1_percent_id = class_1_percent_samples(y_train_pd,class_1_percent,random_state=random_state)[:1000000]

  # подготовим данные для обучения
  X_train_s_balanced = scaler.fit_transform(X_train_pd.loc[list_c1_percent_id])
  y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

  # освободим память от "тяжелых" и ненужных файлов
  del X_train_pd, y_train_pd
  gc.collect()

  # я не воспользовался параметром timeout метода optimize потому, что  
  # optimize отставливает поиск параметров, если время обучения 
  # превышает timeout. Мне нужно чтобы поиск параметров продолжился.
  try:
    # для модуля func_time_out упакуем обучение модели в функцию try_func
    def try_func():
            return optuna_log_reg.fit(X_train_s_balanced, y_train_balanced)
    # обучаем модель с ограничением по времени 10 минут (600 секунд)
    optuna_log_reg = func_timeout.func_timeout(600, try_func)

    # удаляем крупные файлы чтобы высвободить память 
    del X_train_s_balanced, y_train_balanced
    gc.collect()

    # загружаем тестовые наборы
    # сформируем данные для анализа сбалансированности обучающих данных
    X_train = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
    X_valid = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_valid').to_pandas()[list_ncorr_features]
    y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
    y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()

    # подготовим данные для проверки модели
    X_train_s = scaler.transform(X_train)
    X_valid_s = scaler.transform(X_valid)

    # удаляем крупные файлы чтобы высвободить память 
    del X_train, X_valid
    gc.collect()

    # делаем предсказание на обучающем и валидационном наборе и считаем метрики
    roc_train = metrics.roc_auc_score(y_train, optuna_log_reg.predict_proba(X_train_s)[:,1])
    roc_valid = metrics.roc_auc_score(y_valid, optuna_log_reg.predict_proba(X_valid_s)[:,1])
    f1_score = metrics.f1_score(y_valid, optuna_log_reg.predict(X_valid_s))
    recall_1 = metrics.recall_score(y_valid, optuna_log_reg.predict(X_valid_s),average=None,zero_division=0)[1]
    precision_1 = metrics.precision_score(y_valid, optuna_log_reg.predict(X_valid_s),average=None,zero_division=0)[1]

    # удаляем крупные файлы чтобы высвободить память 
    del X_train_s, X_valid_s, y_train, y_valid
    gc.collect()

  except func_timeout.FunctionTimedOut:
    # удаляем крупные файлы чтобы высвободить память 
    del X_train_s_balanced, y_train_balanced
    gc.collect()
    
    # фиксируем пустые значения метрик
    roc_train = 0
    roc_valid = 0
    f1_score = 0
    recall_1 = 0
    precision_1 = 0
    pass

  return roc_train, roc_valid, f1_score, recall_1, precision_1

In [ ]:
# cоздаем объект исследования
# чтобы модель не переобучалась минимизируем roc_train 
# и максимизируем roc_valid
optuna_study_lg = optuna.create_study(study_name='LogisticRegression_'+feature_space+'_stat', 
                               directions=['maximize','maximize','maximize','maximize','maximize'], 
                               sampler=optuna.samplers.TPESampler(),
                               pruner='Hyperband',
                               storage='sqlite:///optuna_studies.db',
                               load_if_exists=True)
# optuna.delete_study(study_name='LogisticRegression_'+feature_space+'_stat', storage='sqlite:///optuna_studies.db')

In [ ]:
# ищем лучшую комбинацию гиперпараметров n_trials раз
optuna_study_lg.optimize(optuna_lg, n_trials=300)

In [ ]:
# из полученного результат соврмируем Data Frame
optuna_study_lg_pd = optuna_study_lg.trials_dataframe().dropna()
# переименуем столбы
optuna_study_lg_pd.rename(columns={
    'values_0': 'roc_train',
    'values_1': 'roc_valid',
    'values_2': 'f1_score',
    'values_3': 'recall_1',
    'values_4': 'precision_1'
},inplace=True)
# Выделим время потраченное на обучение модели
optuna_study_lg_pd.tail()

In [ ]:
# покаждем статистику обучения
print('Максимальное значение метрики ROC AUC на валидационном наборе:',round(optuna_study_lg_pd['roc_valid'].max(),3))
print('Среднее значение метрики ROC AUC на валидационном наборе:',round(optuna_study_lg_pd['roc_valid'].mean(),3))
print('Максимальное значение метрики f1_score на валидационном наборе:',round(optuna_study_lg_pd['f1_score'].max(),3))
print('Среднее значение метрики f1_score на валидационном наборе:',round(optuna_study_lg_pd['f1_score'].mean(),3))
print('Максимальное значение метрики recall_1 на валидационном наборе:',round(optuna_study_lg_pd['recall_1'].max(),3))
print('Среднее значение метрики recall_1 на валидационном наборе:',round(optuna_study_lg_pd['recall_1'].mean(),3))
print('Максимальное значение метрики precision_1 на валидационном наборе:',round(optuna_study_lg_pd['precision_1'].max(),3))
print('Среднее значение метрики precision_1 на валидационном наборе:',round(optuna_study_lg_pd['precision_1'].mean(),3))

In [ ]:
# построим график важности гиперпарметров
optuna.visualization.plot_param_importances(optuna_study_lg, target_name='Score')

In [ ]:
# Построим зависимость метрики precision_1
fig = px.scatter(
    data_frame=optuna_study_lg_pd,
    x='precision_1', #ось абсцисс
    y=['f1_score', 'recall_1'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision от recall на классе 1', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision',
    yaxis_title='Величина метрик recall и f1-score')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики precision_1 от гипер параметров

fig = px.scatter(
    data_frame=optuna_study_lg_pd,
    x='precision_1', #ось абсцисс
    y=['params_C', 'params_class_0_weight', 
       'params_class_1_weight','params_class_1_percent',
       'params_threshold'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики f1_score от гипер параметров

fig = px.scatter(
    data_frame=optuna_study_lg_pd,
    x='f1_score', #ось абсцисс
    y=['params_C', 'params_class_0_weight', 
       'params_class_1_weight','params_class_1_percent',
       'params_threshold'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость f1-score от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики f1-score',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики roc_valid от гипер параметров

fig = px.scatter(
    data_frame=optuna_study_lg_pd,
    x='roc_valid', #ось абсцисс
    y=['params_C', 'params_class_0_weight', 
       'params_class_1_weight','params_class_1_percent',
       'params_threshold'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость ROC AUC на валидационном наборе от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики ROC AUC',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Визуализируем метрики полученные при optuna оптимизации
fig = px.scatter(
    data_frame=optuna_study_lg_pd[(optuna_study_lg_pd['precision_1']>0.08)
                                  & (optuna_study_lg_pd['recall_1']>0.08) 
                                  & (optuna_study_lg_pd['roc_valid']>0.8*optuna_study_lg_pd['roc_valid'].max())],
    x='number', #ось абсцисс
    y=['roc_train','roc_valid','recall_1','precision_1'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость качества модели от trials optuna', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='trials optuna',
    yaxis_title='Величина метрики')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

<span style="color:Blue">

Вывод:  

Их всех выбираем точку $number =234$.   
В этой точке самые оптимальные значения $recall$ и $precision$.  
Посмотрим каким значениям гиперпараметров соотвествует выбранная точка.

In [ ]:
# определим номер лучшге варианта
best_optuna_number = 234

# сформируем словарь лучших гипирпарметров RandomForestClassifier
best_param_lr = {
    'solver' : optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_solver'].iloc[0],
    'C' : round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_C'].iloc[0],3),
    'class_weight' : {0:round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_class_0_weight'].iloc[0],3),
                      1:round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_class_1_weight'].iloc[0],3)},
    }

# создадим перменные
best_threshold = optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_threshold'].iloc[0]
best_scaler = optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_scaler'].iloc[0]
best_class_1_percent = optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_class_1_percent'].iloc[0]
best_random_state = optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_random_state'].iloc[0]

# Выведем принятые наилучшие праметры
print('best solver:',best_param_lr['solver'])
print('best C:',best_param_lr['C'])
print('best class 0 weight:',best_param_lr['class_weight'][0])
print('best class 1 weight:',best_param_lr['class_weight'][1])

print('best class 1 percent:',round(best_class_1_percent,3))
print('best scaler:',best_scaler)
print('best threshold:',round(best_threshold,3))
print('best best random state:',best_random_state)
print('time for best train:',round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['duration'].iloc[0].seconds/60),'minutes')

print()
print('ROC AUC на обучающем наборе:', round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['roc_train'].iloc[0],3))
print('ROC AUC на валидационном наборе:', round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['roc_valid'].iloc[0],3))
print('precision класса 1:', round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['precision_1'].iloc[0],3))
print('recall класса 1:', round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['recall_1'].iloc[0],3))


##### <span style="color:MediumSlateBlue">Обучение модели с лучшими параметрами</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'date'
count_features = dict_spaces[feature_space]

# для работы функции corr_transform_to_force
# подгрузим DataFrame с матрицей корреляции признаков
corr_matrix = fp.ParquetFile('base_models/data/stat/corr_matrix_'+feature_space).to_pandas()

# подготовим данные
# для выбора sceler преобразвания нам понадобится словарь
dict_scalers ={
    'MinMaxScaler': MinMaxScaler(),
    'RobustScaler':RobustScaler(),
    'StandardScaler':StandardScaler(),
}

# с помощью функции corr_transform_to_force получим
# список не коррелируемых признаков
list_ncorr_features = corr_transform_to_force(corr_matrix,threshold=best_threshold)[1]

# обьявим scaler
scaler = dict_scalers[best_scaler]

# сформируем данные для анализа сбалансированности обучающих данных
X_train_pd = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
y_train_pd = pd.read_csv('target/target_train.csv')

# поготовим данные y_train_pd к работе с функцией class_1_percent_samples
y_train_pd.set_index('id',drop=False,inplace=True)

# подготовим данные для обучения модели
# с помощью функции class_1_percent_samples зададим долю 
# класса 1
list_c1_percent_id = class_1_percent_samples(y_train_pd,best_class_1_percent,random_state=best_random_state)[:1000000]

# подготовим данные для обучения
X_train_s_balanced = scaler.fit_transform(X_train_pd.loc[list_c1_percent_id])
y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

# освободим память от "тяжелых" и ненужных файлов
del X_train_pd, y_train_pd
gc.collect()


# обучаем модель LogisticRegression с наилучшеми параметрами
logistic_regression = linear_model.LogisticRegression(
        **best_param_lr,
        random_state=42,
        max_iter=10000)
logistic_regression.fit(X_train_s_balanced, y_train_balanced)

# удаляем крупные файлы чтобы высвободить память 
del X_train_s_balanced, y_train_balanced
gc.collect()

# загружаем тестовые наборы
# сформируем данные для анализа сбалансированности обучающих данных
X_train = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
X_valid = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_valid').to_pandas()[list_ncorr_features]
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()

# подготовим данные для проверки модели
X_train_s = scaler.transform(X_train)
X_valid_s = scaler.transform(X_valid)

# удаляем крупные файлы чтобы высвободить память 
del X_train, X_valid
gc.collect()

# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_lr_pred_proba = logistic_regression.predict_proba(X_train_s)[:,1]
y_valid_lr_pred_proba = logistic_regression.predict_proba(X_valid_s)[:,1]

# Делаем предсказание для валидационной выборки
y_valid_lr_pred = logistic_regression.predict(X_valid_s)

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_lr_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_lr_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_valid,y_valid_lr_pred,zero_division=0))

# time: 2m 30s

##### <span style="color:MediumSlateBlue">Анализ влияния порога классификации на качество модели</span>

In [ ]:
# чтобы проше обращаться к каждому обьекту в данных
# преобразуем y_valid_LR_pred к Series типу
y_valid_lr_pred_proba = pd.Series(y_valid_lr_pred_proba)

# формируем список из необходимых значений thresholds
thresholds = np.arange(0.01, 1, 0.01)

# сформируем списко для заполнения данными результатов анализа
threshold_data = []


for threshold in thresholds:
        # сформируем список для заполнения данными текушего threshold
        threshold_current_data = [threshold]

        #клиентов, для которых вероятность дефолта > threshold, относим к классу 1
        #в противном случае — к классу 0
        y_valid_lr_pred = y_valid_lr_pred_proba.apply(lambda x: 1 if x>threshold else 0)


        #Считаем метрики для класса 0 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.precision_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.f1_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))

        #Считаем метрики для класса 1 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.precision_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.f1_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))

        # добавляем полученные данные в список threshold_data
        threshold_data.append(threshold_current_data)

        # из полученных данных сформируем dataframe
        thresholds_pd =pd.DataFrame(
        columns=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'],
        data = threshold_data)

# time: 17s

In [ ]:
# Визуализируем метрики на валидационном наборе при различных threshold
fig_threshold = px.line(
    data_frame=thresholds_pd,
    x='threshold', #ось абсцисс
    y=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'], #ось ординат
)
fig_threshold.update_layout(
    title ={
        'text':'Зависимость качества модели от threshold', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Порог классификации threshold',
    yaxis_title='Величина метрики')
fig_threshold.update_xaxes(showspikes=True)
fig_threshold.update_yaxes(showspikes=True)

fig_threshold.show()

Метрика $precision$ показывает на сколько сильно ошиблась модель при определиние класса.  
Чем выше метрика, тем меньше ошиблась модель. 

$$precision = \frac{TP}{TP+FP}$$

Метрика $recall$ показывает способность модели найти класс.  
Чем выше метрика, тем выше способность модели находить класс.  

$$recall = \frac{TP}{TP+FN}$$

Метрика $f1-score$ показывает баланс между $precision$ и $recall$ 

$threshold$- порог класификации модели.  
Все объекты для которых $у_{\text{proba pred}} < threshold$, модель относит к класс 0.  
Соотвественно, если $у_{\text{proba pred}}  > threshold$ модель относит обьект к классу 1.  
Для идельаной модели $threshold$, является границей между областями класса 0 и класса 1.  

<span style="color:Blue">

Выводы по результам анализа:  

Для улучшения качества модели по метрике $ROC AUC$, нет смысла сдвигать порог   
классификации. Но зависимость метрик качества модели от $threshold$, может рассказать о  
спосбоности модели искать класс 1 и класс 0.

При фокусировке модели на классе 1 (более низкий $threshold$):  
- на классе 1 метрика $precision$ уменьшается, метрика $recall$ растет;  
- на классе 0 метрика $precision$ растет, метрика $recall$ уменьшается.  

При фокусировке модели на классе 0 (более высокий $threshold$):  
- на классе 1 метрика $precision$ увеличивается, метрика $recall$ уменьшается;  
- на классе 0 метрика $precision$ уменьшается, метрика $recall$ растет.  

Обьсняется тем, что в условия дисбаланса модель научилась хорошо определять класс 0 и плохо класс 1.  

При фокусировке модели на классе 1 (более низкий $threshold$), все больше обьектов отнесены к классу 1.   
Так как $precision$ класса 1 уменьшается, значит в области класса 1 стало больше обектов класса 0,  
что явлется закономерным для "идеальной" модели.  
Однако при этом, $recall$ класса 1 растет, значит в добавленных обьектах также присутвует класс 1.  
Это как раз указывает на то, что модель не научилась находить среди класса 0 класс 1. В области с     
высокой вероятностью класса 0 (более низкий $threshold$), находятся обьекты класса 1.  

Фокусировка на классе 0 (более высокий $threshold$), заставляет модель все больше обьектов из области класса 1  
относить к классу 0. При этом увеличение метрики $precision$ класса 1 говорит о том, что в области класса 1     
становится меньше 0, а значит модель умеет среди класса 1, находить класс 0.  
После  $threshold=0.64$ модель впринципе теряет способность отличать класс 1. У модели нет обьектов класса 1   
в которых она "уверена" на 80+%.

Обратим внимание на $threshold=0.52$, в этой точке находится баланс межу метриками качества модели:
$$precision_1 = recall_1 = \text{f1-score}_1 = 0.082$$
$$precision_0 = recall_0 = \text{f1-score}_0 = 0.965$$

Для модели с лучшим качеством эти две точки должны находится максимально близко к друг другу и иметь  
высокое значение метрик $precision_1$ и $recall_1$.
Для класса 0 это условие выполнется, что также говорит о том, что модель научилась искать этот класс.  
Для класса 1 это условие не выполнется, что также говорит о том, что модель плохо ищет класс 1.

##### <span style="color:MediumSlateBlue">Построение ROC-кривой выбранной модели на тестовом наборе</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'date'
count_features = dict_spaces[feature_space]

In [ ]:
# сформируем данные для проверки модели
X_train = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
X_valid = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_valid').to_pandas()[list_ncorr_features]
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()
X_test = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_test').to_pandas()[list_ncorr_features]
y_test = pd.read_csv('target/target_test.csv')['flag'].to_numpy()

In [ ]:
# выполним scaler преобразование
X_train_s = scaler.transform(X_train)
X_valid_s = scaler.transform(X_valid)
X_test_s = scaler.transform(X_test)

print("Размеры обучающего набора",X_train_s.shape)
print("Размеры валидационного набора",X_valid_s.shape)
print("Размеры тестового набора",X_test_s.shape)
# time: 1m 43s

In [ ]:
# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_lr_pred_proba = logistic_regression.predict_proba(X_train_s)[:,1]
y_valid_lr_pred_proba = logistic_regression.predict_proba(X_valid_s)[:,1]
y_test_lr_pred_proba = logistic_regression.predict_proba(X_test_s)[:,1]

# Делаем предсказание для валидационной выборки
y_test_lr_pred = logistic_regression.predict(X_test_s)

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_lr_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_lr_pred_proba),3))
print('ROC AUC на тестовом наборе', round(metrics.roc_auc_score(y_test, y_test_lr_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_test,y_test_lr_pred,zero_division=0))

In [ ]:
# Для расчета значений TPR и FPR используем функцию roc_curve
fpr_train, tpr_train, _ = metrics.roc_curve(y_train, y_train_lr_pred_proba)
fpr_valid, tpr_valid, _ = metrics.roc_curve(y_valid, y_valid_lr_pred_proba)
fpr_test, tpr_test, _ = metrics.roc_curve(y_test, y_test_lr_pred_proba)

# рассчитаем площадь под кривой
auc_train = round(metrics.roc_auc_score(y_train, y_train_lr_pred_proba),3)
auc_valid = round(metrics.roc_auc_score(y_valid, y_valid_lr_pred_proba),3)
auc_test = round(metrics.roc_auc_score(y_test, y_test_lr_pred_proba),3)

trace_train = go.Scatter(x=fpr_train, y=tpr_train, mode='lines', name='AUC_train = %0.2f' % auc_train,
                   line=dict(color='red', width=2),
                   hovertemplate='train<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_train])

trace_valid = go.Scatter(x=fpr_valid, y=tpr_valid, mode='lines', name='AUC_valid = %0.2f' % auc_valid,
                   line=dict(color='green', width=2),
                   hovertemplate='valid<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_valid])

trace_test = go.Scatter(x=fpr_test, y=tpr_test, mode='lines', name='AUC_test = %0.2f' % auc_test,
                   line=dict(color='darkorange', width=2),
                   hovertemplate='test<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_test])


reference_line = go.Scatter(x=[0,1], y=[0,1], mode='lines', name='Reference Line',
                            line=dict(color='navy', width=2, dash='dash'))

fig_roc = go.Figure(data=[trace_train,trace_valid,trace_test,reference_line])

fig_roc.update_layout(
    title ={
        'text':'ROC-кривая', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 1350,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate')

fig_roc.update_xaxes(showspikes=True)
fig_roc.update_yaxes(showspikes=True)


fig_roc.show()

<span style="color:Blue">

Вывод по ROC-кривой:
1. Модель выдает сталибильное качество на всех наборах, а значит  
не переобучена - разброс ($variance$) не большой.  
2. Смещение ($bias$) модели относительно большое.  
На тестовом наборе полученно качество $ROC AUC = 0.6$.

#### <span style="color:MediumBlue">Построение модели Random Forest Classifier</span>

##### <span style="color:MediumSlateBlue">Baseline обучение модели RandomForestClassifier</span>

In [ ]:
# обучаем модель логистической регрессии
# чтобы модель пыталась найти связь во всех признаках
random_forest = RandomForestClassifier(random_state=42, n_jobs=-1)
random_forest.fit(X_train_balanced[list_ncorr_features],y_train_balanced)

# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_rf__pred_proba = random_forest.predict_proba(X_train[list_ncorr_features])[:,1]
y_valid_rf_pred_proba = random_forest.predict_proba(X_valid[list_ncorr_features])[:,1]

# Делаем предсказание для валидационной выборки
y_valid_rf_pred = random_forest.predict(X_valid[list_ncorr_features])

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_rf__pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_rf_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_valid,y_valid_rf_pred))

# time: 17s

##### <span style="color:MediumSlateBlue">Подбор гиперпараметров модели</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'date'
count_features = dict_spaces[feature_space]

# для работы функции corr_transform_to_force
# подгрузим DataFrame с матрицей корреляции признаков
corr_matrix = fp.ParquetFile('base_models/data/stat/corr_matrix_'+feature_space).to_pandas()

# настроим оптимизацию гипер параметров
def optuna_rf(trial):
  # задаем пространства поиска гиперпараметров
  # задаем пространства поиска гиперпараметров
  n_estimators = trial.suggest_int('n_estimators', 40, 100,step = 5)
  criterion = trial.suggest_categorical('criterion',['gini', 'entropy', 'log_loss'])
  max_depth = trial.suggest_int('max_depth', 10, 30,step = 1)
  min_samples_split = trial.suggest_int('min_samples_split', 5, 15,step = 1)
  min_samples_leaf = trial.suggest_int('min_samples_leaf', 5, 15,step = 1)
  max_features = trial.suggest_float('max_features',0.4,0.8)
  max_samples = trial.suggest_float('max_samples',0.4,0.8)
  class_0_weight = trial.suggest_float('class_0_weight',0.1,1)
  class_1_weight = trial.suggest_float('class_1_weight',0.1,1)
  threshold = trial.suggest_float('threshold',0.4,0.8)
  class_1_percent = trial.suggest_float('class_1_percent',0.01,0.5)
  random_state = trial.suggest_int('random_state', 1, 1000000)


  # создаем модель
  optuna_random_forest = RandomForestClassifier(
      n_estimators = n_estimators, 
      criterion = criterion,
      max_depth = max_depth,
      min_samples_split = min_samples_split,  
      min_samples_leaf = min_samples_leaf,
      max_features=max_features,
      max_samples=max_samples,
      class_weight={0:class_0_weight,1:class_1_weight},
      n_jobs=-1,
      random_state=42)

  # с помощью функции corr_transform_to_force получим
  # список не коррелируемых признаков
  list_ncorr_features = corr_transform_to_force(corr_matrix,threshold=threshold)[1]

  # сформируем данные для анализа сбалансированности обучающих данных
  X_train_pd = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
  y_train_pd = pd.read_csv('target/target_train.csv')

  # поготовим данные y_train_pd к работе с функцией class_1_percent_samples
  y_train_pd.set_index('id',drop=False,inplace=True)

  # подготовим данные для обучения модели
  # с помощью функции class_1_percent_samples зададим долю 
  # класса 1
  list_c1_percent_id = class_1_percent_samples(y_train_pd,class_1_percent,random_state=random_state)[:1000000]

  # подготовим данные для обучения
  X_train_balanced = X_train_pd.loc[list_c1_percent_id]
  y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag']

  # освободим память от "тяжелых" и ненужных файлов
  del X_train_pd, y_train_pd
  gc.collect()

  # я не воспользовался параметром timeout метода optimize потому, что  
  # optimize отставливает поиск параметров, если время обучения 
  # превышает timeout. Мне нужно чтобы поиск параметров продолжился.
  try:
    # для модуля func_time_out упакуем обучение модели в функцию try_func
    def try_func():
            return optuna_random_forest.fit(X_train_balanced, y_train_balanced)
    # обучаем модель с ограничением по времени 10 минут (600 секунд)
    optuna_random_forest = func_timeout.func_timeout(600, try_func)

    # удаляем крупные файлы чтобы высвободить память 
    del X_train_balanced, y_train_balanced
    gc.collect()

    # загружаем тестовые наборы
    # сформируем данные для анализа сбалансированности обучающих данных
    X_train = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
    X_valid = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_valid').to_pandas()[list_ncorr_features]
    y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
    y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()
    
    # делаем предсказание на обучающем и валидационном наборе
    #Считаем метрики для класса 1 добавляем их в список
    roc_train = metrics.roc_auc_score(y_train, optuna_random_forest.predict_proba(X_train)[:,1])
    roc_valid = metrics.roc_auc_score(y_valid, optuna_random_forest.predict_proba(X_valid)[:,1])
    f1_score = metrics.f1_score(y_valid, optuna_random_forest.predict(X_valid))
    recall_1 = metrics.recall_score(y_valid, optuna_random_forest.predict(X_valid),average=None,zero_division=0)[1]
    precision_1 = metrics.precision_score(y_valid, optuna_random_forest.predict(X_valid),average=None,zero_division=0)[1]

    # удаляем крупные файлы чтобы высвободить память 
    del X_train, X_valid, y_train, y_valid
    gc.collect()

  except func_timeout.FunctionTimedOut:
    # удаляем крупные файлы чтобы высвободить память 
    del X_train_balanced, y_train_balanced
    gc.collect()
    
    # фиксируем пустые значения метрик
    roc_train = 0
    roc_valid = 0
    f1_score = 0
    recall_1 = 0
    precision_1 = 0
    pass

  return roc_train, roc_valid, f1_score, recall_1, precision_1

In [ ]:
# cоздаем объект исследования
optuna_study_rf = optuna.create_study(study_name= 'RandomForestClassifier_'+feature_space+'_stat', 
                               directions=['maximize','maximize','maximize','maximize','maximize'], 
                               sampler=optuna.samplers.TPESampler(),
                               pruner='Hyperband',
                               storage='sqlite:///optuna_studies.db',
                               load_if_exists=True)
# на случай удаления 
# optuna.delete_study(study_name='RandomForestClassifier_'+feature_space+'_stat', storage='sqlite:///optuna_studies.db')

In [ ]:
# ищем лучшую комбинацию гиперпараметров n_trials раз
optuna_study_rf.optimize(optuna_rf, n_trials=300)

In [ ]:
# из полученного результат соврмируем Data Frame
optuna_study_rf_pd = optuna_study_rf.trials_dataframe().dropna()
# переименуем столбы
optuna_study_rf_pd.rename(columns={
    'values_0': 'roc_train',
    'values_1': 'roc_valid',
    'values_2': 'f1_score',
    'values_3': 'recall_1',
    'values_4': 'precision_1'
},inplace=True)
optuna_study_rf_pd.tail()

In [ ]:
# покаждем статистику обучения
print('Максимальное значение метрики ROC AUC на валидационном наборе:',round(optuna_study_rf_pd['roc_valid'].max(),3))
print('Среднее значение метрики ROC AUC на валидационном наборе:',round(optuna_study_rf_pd['roc_valid'].mean(),3))
print('Максимальное значение метрики f1_score на валидационном наборе:',round(optuna_study_rf_pd['f1_score'].max(),3))
print('Среднее значение метрики f1_score на валидационном наборе:',round(optuna_study_rf_pd['f1_score'].mean(),3))
print('Максимальное значение метрики recall_1 на валидационном наборе:',round(optuna_study_rf_pd['recall_1'].max(),3))
print('Среднее значение метрики recall_1 на валидационном наборе:',round(optuna_study_rf_pd['recall_1'].mean(),3))
print('Максимальное значение метрики precision_1 на валидационном наборе:',round(optuna_study_rf_pd['precision_1'].max(),3))
print('Среднее значение метрики precision_1 на валидационном наборе:',round(optuna_study_rf_pd['precision_1'].mean(),3))

In [ ]:
# Построим зависимость метрики precision_1

fig = px.scatter(
    data_frame=optuna_study_rf_pd,
    x='precision_1', #ось абсцисс
    y=['recall_1','f1_score'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision от recall на классе 1', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision на классе 1',
    yaxis_title='Величина метрик recall и f1-score')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики precision_1 от гиперпараметров

fig = px.scatter(
    data_frame=optuna_study_rf_pd,
    x='precision_1', #ось абсцисс
    y=['params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight', 'params_max_depth',
       'params_max_features', 'params_max_samples', 'params_min_samples_leaf',
       'params_min_samples_split', 'params_n_estimators', 'params_threshold'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision класса 1 от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision на классе 1',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики f1_score от гиперпараметров

fig = px.scatter(
    data_frame=optuna_study_rf_pd,
    x='f1_score', #ось абсцисс
    y=['params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight', 'params_max_depth',
       'params_max_features', 'params_max_samples', 'params_min_samples_leaf',
       'params_min_samples_split', 'params_n_estimators', 'params_threshold'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость f1-score класса 1 от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики f1-score на классе 1',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики roc_valid от гиперпараметров

fig = px.scatter(
    data_frame=optuna_study_rf_pd,
    x='roc_valid', #ось абсцисс
    y=['params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight', 'params_max_depth',
       'params_max_features', 'params_max_samples', 'params_min_samples_leaf',
       'params_min_samples_split', 'params_n_estimators', 'params_threshold'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость ROC AUC на валидационном наборе от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики ROC AUV на валидационном наборе',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрик качества модели от number

fig = px.scatter(
    data_frame=optuna_study_rf_pd[(optuna_study_rf_pd['recall_1']>=0.08) & (optuna_study_rf_pd['precision_1']>0.08)],
    x='number', #ось абсцисс
    y=['roc_valid','roc_train','precision_1','recall_1'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость качества модели от trials optuna', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='trials optuna',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

<span style="color:Blue">

Вывод:  

Их всех выбираем точку $number =188$.   
Не смотря на, то что в этой точке модель подает признаки переобучеености,  
в этой точке оптимальные значения метрик $recall_1$ и $precision_1$, а  
также относительно высокая метрика $ROC AUC$.   
Посмотрим каким значениям гиперпараметров соотвествует выбраннаяточка.

In [ ]:
# определим номер лучшге варианта
best_optuna_number = 188

# сформируем словарь лучших гипирпарметров RandomForestClassifier
best_param_rf = {
    'n_estimators' : int(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_n_estimators'].iloc[0]),
    'criterion': optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_criterion'].iloc[0],
    'max_depth' : int(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_max_depth'].iloc[0]),
    'min_samples_split': int(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_min_samples_split'].iloc[0]),
    'min_samples_leaf' : int(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_min_samples_leaf'].iloc[0]),
    'max_features': optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_max_features'].iloc[0],
    'max_samples' : optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_max_samples'].iloc[0],
    'class_weight' : {0:optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_class_0_weight'].iloc[0],
                      1:optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_class_1_weight'].iloc[0]}
    }

# определим перменные для лучших значений параметров
best_threshold = optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_threshold'].iloc[0]
best_class_1_percent = optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_class_1_percent'].iloc[0]
best_random_state = optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_random_state'].iloc[0]


# Выведем принятые наилучшие праметры
print('best n estimators:',best_param_rf['n_estimators'])
print('best criterion:',best_param_rf['criterion'])
print('best max depth:',best_param_rf['max_depth'])
print('best min samples split:',best_param_rf['min_samples_split'])
print('best min samples leaf:',best_param_rf['min_samples_leaf'])
print('best max features(%):',round(best_param_rf['max_features'],3))
print('best max samples(%):',round(best_param_rf['max_samples'],3))
print('best class 0 weight:',round(best_param_rf['class_weight'][0],3))
print('best class 1 weight:',round(best_param_rf['class_weight'][1],3))

print('best class 1 percent:',round(best_class_1_percent,3))
print('best threshold:',round(best_threshold,3))
print('best random state:', best_random_state)
print('time for best train:',round(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['duration'].iloc[0].seconds/60),'minutes')

print()
print('ROC AUC на обучающем наборе:', round(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['roc_train'].iloc[0],3))
print('ROC AUC на валидационном наборе:', round(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['roc_valid'].iloc[0],3))
print('precision класса 1:', round(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['precision_1'].iloc[0],3))
print('recall класса 1:', round(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['recall_1'].iloc[0],3))

##### <span style="color:MediumSlateBlue">Обучение модели с лучшими параметрами</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'date'
count_features = dict_spaces[feature_space]

# для работы функции corr_transform_to_force
# подгрузим DataFrame с матрицей корреляции признаков
corr_matrix = fp.ParquetFile('base_models/data/stat/corr_matrix_'+feature_space).to_pandas()

# с помощью функции corr_transform_to_force получим
# список не коррелируемых признаков
list_ncorr_features = corr_transform_to_force(corr_matrix,threshold=best_threshold)[1]

# сформируем данные для анализа сбалансированности обучающих данных
X_train_pd = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
y_train_pd = pd.read_csv('target/target_train.csv')

# поготовим данные y_train_pd к работе с функцией class_1_percent_samples
y_train_pd.set_index('id',drop=False,inplace=True)

# подготовим данные для обучения модели
# с помощью функции class_1_percent_samples зададим долю 
# класса 1
list_c1_percent_id = class_1_percent_samples(y_train_pd,best_class_1_percent,random_state=best_random_state)[:1000000]

# подготовим данные для обучения
X_train_balanced = X_train_pd.loc[list_c1_percent_id]
y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag']

# освободим память от "тяжелых" и ненужных файлов
del X_train_pd, y_train_pd
gc.collect()


# обучаем модель RandomForestClassifier с наилучшеми параметрами
random_forest = RandomForestClassifier(
        **best_param_rf,
        n_jobs=-1,
        random_state=42)
random_forest.fit(X_train_balanced, y_train_balanced)

# удаляем крупные файлы чтобы высвободить память 
del X_train_balanced, y_train_balanced
gc.collect()

# загружаем тестовые наборы
# сформируем данные для анализа сбалансированности обучающих данных
X_train = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
X_valid = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_valid').to_pandas()[list_ncorr_features]
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()

# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_rf_pred_proba = random_forest.predict_proba(X_train)[:,1]
y_valid_rf_pred_proba = random_forest.predict_proba(X_valid)[:,1]

# Делаем предсказание для валидационной выборки
y_valid_rf_pred = random_forest.predict(X_valid)

# удаляем крупные файлы чтобы высвободить память 
del X_train, X_valid
gc.collect()

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_rf_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_rf_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_valid,y_valid_rf_pred,zero_division=0))

# time: 5m 30s

##### <span style="color:MediumSlateBlue">Анализ влияния порога классификации на качество модели</span>

In [ ]:
# чтобы проше обращаться к каждому обьекту в данных
# преобразуем y_valid_LR_pred к Series типу
y_valid_rf_pred_proba = pd.Series(y_valid_rf_pred_proba)

# формируем список из необходимых значений thresholds
thresholds = np.arange(0.01, 1, 0.01)

# сформируем списко для заполнения данными результатов анализа
threshold_data = []


for threshold in thresholds:
        # сформируем список для заполнения данными текушего threshold
        threshold_current_data = [threshold]

        #клиентов, для которых вероятность дефолта > threshold, относим к классу 1
        #в противном случае — к классу 0
        y_valid_rf_pred = y_valid_rf_pred_proba.apply(lambda x: 1 if x>threshold else 0)


        #Считаем метрики для класса 0 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(y_valid, y_valid_rf_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.precision_score(y_valid, y_valid_rf_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.f1_score(y_valid, y_valid_rf_pred,average=None,zero_division=0)[0],3))

        #Считаем метрики для класса 1 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(y_valid, y_valid_rf_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.precision_score(y_valid, y_valid_rf_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.f1_score(y_valid, y_valid_rf_pred,average=None,zero_division=0)[1],3))

        # добавляем полученные данные в список threshold_data
        threshold_data.append(threshold_current_data)

        # из полученных данных сформируем dataframe
        thresholds_pd =pd.DataFrame(
        columns=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'],
        data = threshold_data)

# time: 17s

In [ ]:
# Визуализируем метрики на валидационном наборе при различных threshold
fig_threshold = px.line(
    data_frame=thresholds_pd,
    x='threshold', #ось абсцисс
    y=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'], #ось ординат
)
fig_threshold.update_layout(
    title ={
        'text':'Зависимость качества модели от threshold', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Порог классификации threshold',
    yaxis_title='Величина метрики')
fig_threshold.update_xaxes(showspikes=True)
fig_threshold.update_yaxes(showspikes=True)

fig_threshold.show()

Метрика $precision$ показывает на сколько сильно ошиблась модель при определиние класса.  
Чем выше метрика, тем меньше ошиблась модель. 

$$precision = \frac{TP}{TP+FP}$$

Метрика $recall$ показывает способность модели найти класс.  
Чем выше метрика, тем выше способность модели находить класс.  

$$recall = \frac{TP}{TP+FN}$$

Метрика $f1-score$ показывает баланс между $precision$ и $recall$ 

$threshold$- порог класификации модели.  
Все объекты для которых $у_{\text{proba pred}} < threshold$, модель относит к класс 0.  
Соотвественно, если $у_{\text{proba pred}}  > threshold$ модель относит обьект к классу 1.  
Для идельаной модели $threshold$, является границей между областями класса 0 и класса 1.  

<span style="color:Blue">

Выводы по результам анализа:  

Для улучшения качества модели по метрике $ROC AUC$, нет смысла сдвигать порог   
классификации. Но зависимость метрик качества модели от $threshold$, может рассказать о  
способности модели искать класс 1 и класс 0.

При фокусировке модели на классе 1 (более низкий $threshold$):  
- на классе 1 метрика $precision$ уменьшается, метрика $recall$ растет;  
- на классе 0 метрика $precision$ растет, метрика $recall$ уменьшается.  

При фокусировке модели на классе 0 (более высокий $threshold$):  
- на классе 1 метрика $precision$ увеличивается, метрика $recall$ уменьшается;  
- на классе 0 метрика $precision$ уменьшается, метрика $recall$ растет.  

Обьсняется тем, что в условия дисбаланса модель научилась хорошо определять класс 0 и плохо класс 1.  

При фокусировке модели на классе 1 (более низкий $threshold$), все больше обьектов отнесены к классу 1.   
Так как $precision$ класса 1 уменьшается, значит в области класса 1 стало больше обектов класса 0,  
что явлется закономерным для "идеальной" модели.  
Однако при этом, $recall$ класса 1 растет, значит в добавленных обьектах также присутвует класс 1.  
Это как раз указывает на то, что модель не нучилась находить среди класса 0 класс 1. В области с     
высокой вероятностью класса 0 (более низкий $threshold$), находятся обьекты класса 1.  

Фокусировка на классе 0 (более высокий $threshold$), заставляет модель все больше обьектов из области класса 1  
относить к классу 0. При этом увеличение метрики $precision$ класса 1 говорит о том, что в области класса 1     
становится меньше 0, а значит модель умеет среди класса 1, находить класс 0.  
После  $threshold=0.77$ модель впринципе теряет способность отличать класс 1. У модели нет обьектов класса 1   
в которых она "уверена" на 80+%.

Обратим внимание на $threshold=0.49$, в этой точке находится баланс межу метриками качества модели:
$$precision_1 = recall_1 = \text{f1-score}_1 = 0.09$$
$$precision_0 = recall_0 = \text{f1-score}_0 = 0.968$$

Для модели с лучшим качеством эти две точки должны находится максимально близко к друг другу и иметь   
высокое значение метрик $precision_1$ и $recall_1$.  
Для класса 0 это условие выполнется, что также говорит о том, что модель научилась искать этот класс.    
Для класса 1 это условие не выполнется, что также говорит о том, что модель плохо ищет класс 1.

##### <span style="color:MediumSlateBlue">Построение ROC-кривой выбранной модели на тестовом наборе</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'date'
count_features = dict_spaces[feature_space]

In [ ]:
# сформируем данные для проверки модели
X_train = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
X_valid = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_valid').to_pandas()[list_ncorr_features]
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()
X_test = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_test').to_pandas()[list_ncorr_features]
y_test = pd.read_csv('target/target_test.csv')['flag'].to_numpy()

In [ ]:
print("Размеры обучающего набора",X_train.shape,y_train.shape)
print("Размеры валидационного набора",X_valid.shape,y_valid.shape)
print("Размеры тестового набора",X_test.shape,y_test.shape)
# time: 1m 43s

In [ ]:
# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_rf_pred_proba = random_forest.predict_proba(X_train)[:,1]
y_valid_rf_pred_proba = random_forest.predict_proba(X_valid)[:,1]
y_test_rf_pred_proba = random_forest.predict_proba(X_test)[:,1]

# Делаем предсказание для валидационной выборки
y_test_lr_pred = random_forest.predict(X_test)

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_rf_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_rf_pred_proba),3))
print('ROC AUC на тестовом наборе', round(metrics.roc_auc_score(y_test, y_test_rf_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_test,y_test_lr_pred,zero_division=0))

In [ ]:
# Для расчета значений TPR и FPR используем функцию roc_curve
fpr_train, tpr_train, _ = metrics.roc_curve(y_train, y_train_rf_pred_proba)
fpr_valid, tpr_valid, _ = metrics.roc_curve(y_valid, y_valid_rf_pred_proba)
fpr_test, tpr_test, _ = metrics.roc_curve(y_test, y_test_rf_pred_proba)

# рассчитаем площадь под кривой
auc_train = round(metrics.roc_auc_score(y_train, y_train_rf_pred_proba),3)
auc_valid = round(metrics.roc_auc_score(y_valid, y_valid_rf_pred_proba),3)
auc_test = round(metrics.roc_auc_score(y_test, y_test_rf_pred_proba),3)

trace_train = go.Scatter(x=fpr_train, y=tpr_train, mode='lines', name='AUC_train = %0.2f' % auc_train,
                   line=dict(color='red', width=2),
                   hovertemplate='train<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_train])

trace_valid = go.Scatter(x=fpr_valid, y=tpr_valid, mode='lines', name='AUC_valid = %0.2f' % auc_valid,
                   line=dict(color='green', width=2),
                   hovertemplate='valid<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_valid])

trace_test = go.Scatter(x=fpr_test, y=tpr_test, mode='lines', name='AUC_test = %0.2f' % auc_test,
                   line=dict(color='darkorange', width=2),
                   hovertemplate='test<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_test])


reference_line = go.Scatter(x=[0,1], y=[0,1], mode='lines', name='Reference Line',
                            line=dict(color='navy', width=2, dash='dash'))

fig_roc = go.Figure(data=[trace_train,trace_valid,trace_test,reference_line])

fig_roc.update_layout(
    title ={
        'text':'ROC-кривая', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 1350,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate')

fig_roc.update_xaxes(showspikes=True)
fig_roc.update_yaxes(showspikes=True)


fig_roc.show()

<span style="color:Blue">

Вывод по ROC-кривой:
1. Не смотря на то, что модель показывает, очень хорошую $ROC$ кривую на обучающем наборе,  
модель, так же выдает сталибильное качество на валидационном и тестовом наборах, а значит  
не переобучена - разброс ($variance$) не большой.
2. Смещение ($bias$) модели такое же как у модели $Logistic$ $Regression$ $Classifier$,  
обученной  на тех же данных.  
На тестовом наборе полученно качество $ROC AUC = 0.61$.

#### <span style="color:MediumBlue">Построение модели Gradient Boosting Classifier</span>

##### <span style="color:MediumSlateBlue">Baseline обучение модели Hist Gradient Boosting Classifier</span>

In [ ]:
# обучаем модель Gradient Boosting Classifier
gradient_boosting = HistGradientBoostingClassifier(random_state=42)
gradient_boosting.fit(X_train_balanced[list_ncorr_features],y_train_balanced)

# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_rf__pred_proba = gradient_boosting.predict_proba(X_train[list_ncorr_features])[:,1]
y_valid_rf_pred_proba = gradient_boosting.predict_proba(X_valid[list_ncorr_features])[:,1]

# Делаем предсказание для валидационной выборки
y_valid_rf_pred = gradient_boosting.predict(X_valid[list_ncorr_features])

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_rf__pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_rf_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_valid,y_valid_rf_pred))

# time: 30s

##### <span style="color:MediumSlateBlue">Подбор гиперпараметров модели</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'date'
count_features = dict_spaces[feature_space]

# для работы функции corr_transform_to_force
# подгрузим DataFrame с матрицей корреляции признаков
corr_matrix = fp.ParquetFile('base_models/data/stat/corr_matrix_'+feature_space).to_pandas()

# настроим оптимизацию гипер параметров
def optuna_hgbc(trial):
  # задаем пространства поиска гиперпараметров
  learning_rate = trial.suggest_float('learning_rate',0.01,0.1)
  max_iter = trial.suggest_int('max_iter', 150, 250,step = 1)
  max_leaf_nodes = trial.suggest_int('max_leaf_nodes', 2, 60,step = 1)
  max_depth = trial.suggest_int('max_depth', 1, 10,step = 1)
  min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 60,step = 1)
  max_features = trial.suggest_float('max_features',0.6,1)
  l2_regularization = trial.suggest_float('l2_regularization',0.01,1)
  class_0_weight = trial.suggest_float('class_0_weight',0.01,1)
  class_1_weight = trial.suggest_float('class_1_weight',0.5,1)
  threshold = trial.suggest_float('threshold',0.7,1)
  class_1_percent = trial.suggest_float('class_1_percent',0.01,0.25)
  random_state = trial.suggest_int('random_state', 1, 1000000)

  # создаем модель
  optuna_gradient_boosting = HistGradientBoostingClassifier(
      learning_rate= learning_rate,
      max_iter = max_iter,
      max_leaf_nodes =max_leaf_nodes,
      max_depth = max_depth,
      min_samples_leaf = min_samples_leaf,
      max_features=max_features,
      l2_regularization = l2_regularization,
      class_weight={0:class_0_weight,1:class_1_weight},
      random_state=42)

  # с помощью функции corr_transform_to_force получим
  # список не коррелируемых признаков
  list_ncorr_features = corr_transform_to_force(corr_matrix,threshold=threshold)[1]

  # сформируем данные для анализа сбалансированности обучающих данных
  X_train_pd = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
  y_train_pd = pd.read_csv('target/target_train.csv')

  # поготовим данные y_train_pd к работе с функцией class_1_percent_samples
  y_train_pd.set_index('id',drop=False,inplace=True)

  # подготовим данные для обучения модели
  # с помощью функции class_1_percent_samples зададим долю 
  # класса 1
  list_c1_percent_id = class_1_percent_samples(y_train_pd,class_1_percent,random_state=random_state)[:1000000]

  # подготовим данные для обучения
  X_train_balanced = X_train_pd.loc[list_c1_percent_id]
  y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag']

  # освободим память от "тяжелых" и ненужных файлов
  del X_train_pd, y_train_pd
  gc.collect()

  # я не воспользовался параметром timeout метода optimize потому, что  
  # optimize отставливает поиск параметров, если время обучения 
  # превышает timeout. Мне нужно чтобы поиск параметров продолжился.
  try:
    # для модуля func_time_out упакуем обучение модели в функцию try_func
    def try_func():
            return optuna_gradient_boosting.fit(X_train_balanced,y_train_balanced)
    # обучаем модель с ограничением по времени 10 минут (600 секунд)
    optuna_gradient_boosting = func_timeout.func_timeout(600, try_func)

    # удаляем крупные файлы чтобы высвободить память 
    del X_train_balanced, y_train_balanced
    gc.collect()

        # загружаем тестовые наборы
    # сформируем данные для анализа сбалансированности обучающих данных
    X_train = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
    X_valid = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_valid').to_pandas()[list_ncorr_features]
    y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
    y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()
    
    # делаем предсказание на обучающем и валидационном наборе и считаем метрики
    roc_train = metrics.roc_auc_score(y_train, optuna_gradient_boosting.predict_proba(X_train)[:,1])
    roc_valid = metrics.roc_auc_score(y_valid, optuna_gradient_boosting.predict_proba(X_valid)[:,1])
    f1_score = metrics.f1_score(y_valid, optuna_gradient_boosting.predict(X_valid))
    recall_1 = metrics.recall_score(y_valid, optuna_gradient_boosting.predict(X_valid),average=None,zero_division=0)[1]
    precision_1 = metrics.precision_score(y_valid, optuna_gradient_boosting.predict(X_valid),average=None,zero_division=0)[1]


    # удаляем крупные файлы чтобы высвободить память 
    del X_train, X_valid, y_train, y_valid
    gc.collect()

  except func_timeout.FunctionTimedOut:
    # удаляем крупные файлы чтобы высвободить память 
    del X_train_balanced, y_train_balanced
    gc.collect()
    
    # фиксируем пустые значения метрик
    roc_train = 0
    roc_valid = 0
    f1_score = 0
    recall_1 = 0
    precision_1 = 0
    pass

  return roc_train, roc_valid, f1_score, recall_1, precision_1

In [ ]:
# cоздаем объект исследования
# чтобы модель не переобучалась минимизируем roc_train 
# и максимизируем roc_valid
optuna_study_hgbc = optuna.create_study(study_name='HistGradientBoostingClassifier_'+feature_space+'_stat', 
                               directions=['maximize','maximize','maximize','maximize','maximize'], 
                               sampler=optuna.samplers.TPESampler(),
                               pruner='Hyperband',
                               storage='sqlite:///optuna_studies.db',
                               load_if_exists=True)

# ну случай удаления обучения
# optuna.delete_study(study_name='HistGradientBoostingClassifier'+feature_space+'_stat', storage='sqlite:///optuna_studies.db')

In [ ]:
# ищем лучшую комбинацию гиперпараметров n_trials раз
optuna_study_hgbc.optimize(optuna_hgbc, n_trials=300)

In [ ]:
# из полученного результат соврмируем Data Frame
optuna_study_hgbc_pd = optuna_study_hgbc.trials_dataframe()
# переименуем столбы
optuna_study_hgbc_pd.rename(columns={
    'values_0': 'roc_train',
    'values_1': 'roc_valid',
    'values_2': 'f1_score',
    'values_3': 'recall_1',
    'values_4': 'precision_1'
},inplace=True)
optuna_study_hgbc_pd.tail(5)

In [ ]:
# покаждем статистику обучения
print('Максимальное значение метрики ROC AUC на валидационном наборе:',round(optuna_study_hgbc_pd['roc_valid'].max(),3))
print('Среднее значение метрики ROC AUC на валидационном наборе:',round(optuna_study_hgbc_pd['roc_valid'].mean(),3))
print('Максимальное значение метрики f1_score на валидационном наборе:',round(optuna_study_hgbc_pd['f1_score'].max(),3))
print('Среднее значение метрики f1_score на валидационном наборе:',round(optuna_study_hgbc_pd['f1_score'].mean(),3))
print('Максимальное значение метрики recall_1 на валидационном наборе:',round(optuna_study_hgbc_pd['recall_1'].max(),3))
print('Среднее значение метрики recall_1 на валидационном наборе:',round(optuna_study_hgbc_pd['recall_1'].mean(),3))
print('Максимальное значение метрики precision_1 на валидационном наборе:',round(optuna_study_hgbc_pd['precision_1'].max(),3))
print('Среднее значение метрики precision_1 на валидационном наборе:',round(optuna_study_hgbc_pd['precision_1'].mean(),3))

In [ ]:
# построим график важности гиперпарметров
optuna.visualization.plot_param_importances(optuna_study_hgbc, target_name='Score')

In [ ]:
# Построим зависимость метрики precision_1
fig = px.scatter(
    data_frame=optuna_study_hgbc_pd,
    x='precision_1', #ось абсцисс
    y=['f1_score','recall_1'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision от recall на классе 1', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision класса 1',
    yaxis_title='Величина метрик recall и f1-score класса 1')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики precision_1 от гиперпараметров
fig = px.scatter(
    data_frame=optuna_study_hgbc_pd,
    x='precision_1', #ось абсцисс
    y=['params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight', 'params_l2_regularization',
       'params_learning_rate', 'params_max_depth', 'params_max_features',
       'params_max_iter', 'params_max_leaf_nodes', 'params_min_samples_leaf',
       'params_threshold'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision класса 1 от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision класса 1',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики f1_score от гиперпараметров
fig = px.scatter(
    data_frame=optuna_study_hgbc_pd,
    x='f1_score', #ось абсцисс
    y=['params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight', 'params_l2_regularization',
       'params_learning_rate', 'params_max_depth', 'params_max_features',
       'params_max_iter', 'params_max_leaf_nodes', 'params_min_samples_leaf',
       'params_threshold'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость f1-score класса 1 от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики f1-score класса 1',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики roc_valid от гиперпараметров
fig = px.scatter(
    data_frame=optuna_study_hgbc_pd,
    x='roc_valid', #ось абсцисс
    y=['params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight', 'params_l2_regularization',
       'params_learning_rate', 'params_max_depth', 'params_max_features',
       'params_max_iter', 'params_max_leaf_nodes', 'params_min_samples_leaf',
       'params_threshold'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость ROC AUC на валидационном наборе от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики ROC AUC на валидационном наборе',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрик качества модели от number

fig = px.scatter(
    data_frame=optuna_study_hgbc_pd[(optuna_study_hgbc_pd['recall_1']>0.1)&(optuna_study_hgbc_pd['precision_1']>0.1)],
    x='number', #ось абсцисс
    y=['roc_train','roc_valid','recall_1','precision_1'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость качества модели от trials optuna', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='trials optuna',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

<span style="color:Blue">

Вывод:  

Их всех выбираем точку $number =127$.   
В этой точке оптимальные значения метрик $recall_1$ и $precision_1$, а  
также относительно высокая метрика $ROC AUC$.   
Посмотрим каким значениям гиперпараметров соотвествует выбранная точка.

In [ ]:
# определим номер лучшге варианта
best_optuna_number = 127

# сформируем словарь лучших гипирпарметров HistGradientBoostingClassifier
best_param_hgbc = {
    'learning_rate' : optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['params_learning_rate'].iloc[0],
    'max_iter' : optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['params_max_iter'].iloc[0],
    'max_leaf_nodes' : optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['params_max_leaf_nodes'].iloc[0],
    'max_depth' : optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['params_max_depth'].iloc[0],
    'min_samples_leaf' : optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['params_min_samples_leaf'].iloc[0],
    'max_features' : optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['params_max_features'].iloc[0],
    'l2_regularization' : optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['params_l2_regularization'].iloc[0],
    'class_weight' : {0: optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['params_class_0_weight'].iloc[0],
                      1: optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['params_class_1_weight'].iloc[0],}
    }

# определим перменные для лучших значений параметров
best_threshold = optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['params_threshold'].iloc[0]
best_class_1_percent = optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['params_class_1_percent'].iloc[0]
best_random_state = optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['params_random_state'].iloc[0]


# Выведем принятые наилучшие параметры
print('best learning rate:',round(best_param_hgbc['learning_rate'],3))
print('best max iter:',best_param_hgbc['max_iter'])
print('best max leaf nodes:',best_param_hgbc['max_leaf_nodes'])
print('best max depth:',best_param_hgbc['max_depth'])
print('best min samples leaf:',best_param_hgbc['min_samples_leaf'])
print('best max features:',round(best_param_hgbc['max_features'],3))
print('best l2 regularization:',round(best_param_hgbc['l2_regularization'],3))
print('best class 0 weight:',round(best_param_hgbc['class_weight'][0],3))
print('best class 1 weight:',round(best_param_hgbc['class_weight'][1],3))

print('best class 1 percent:',round(best_class_1_percent,3))
print('best threshold:',round(best_threshold,3))
print('time for best train:',round(optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['duration'].iloc[0].seconds/60),'minutes')

print()
print('ROC AUC на обучающем наборе:', round(optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['roc_train'].iloc[0],3))
print('ROC AUC на валидационном наборе:', round(optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['roc_valid'].iloc[0],3))
print('precision класса 1:', round(optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['precision_1'].iloc[0],3))
print('recall класса 1:', round(optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['recall_1'].iloc[0],3))

##### <span style="color:MediumSlateBlue">Обучение модели с лучшими параметрами</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'date'
count_features = dict_spaces[feature_space]

# для работы функции corr_transform_to_force
# подгрузим DataFrame с матрицей корреляции признаков
corr_matrix = fp.ParquetFile('base_models/data/stat/corr_matrix_'+feature_space).to_pandas()

# с помощью функции corr_transform_to_force получим
# список не коррелируемых признаков
list_ncorr_features = corr_transform_to_force(corr_matrix,threshold=best_threshold)[1]

# сформируем данные для анализа сбалансированности обучающих данных
X_train_pd = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
y_train_pd = pd.read_csv('target/target_train.csv')

# поготовим данные y_train_pd к работе с функцией class_1_percent_samples
y_train_pd.set_index('id',drop=False,inplace=True)

# подготовим данные для обучения модели
# с помощью функции class_1_percent_samples зададим долю 
# класса 1
list_c1_percent_id = class_1_percent_samples(y_train_pd,best_class_1_percent,random_state=best_random_state)[:1000000]

# подготовим данные для обучения
X_train_balanced = X_train_pd.loc[list_c1_percent_id]
y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag']

# освободим память от "тяжелых" и ненужных файлов
del X_train_pd, y_train_pd
gc.collect()

# обучаем модель HistGradientBoostingClassifier с наилучшеми параметрами
gradient_boosting = HistGradientBoostingClassifier(
        **best_param_hgbc,
        random_state=42)
gradient_boosting.fit(X_train_balanced,y_train_balanced)

# удаляем крупные файлы чтобы высвободить память 
del X_train_balanced, y_train_balanced
gc.collect()

# сформируем данные для анализа сбалансированности обучающих данных
X_train = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
X_valid = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_valid').to_pandas()[list_ncorr_features]
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()

# делаем предсказание на обучающем и валидационном наборе и считаем метрики
y_train_gb_pred_proba = gradient_boosting.predict_proba(X_train)[:,1]
y_valid_gb_pred_proba = gradient_boosting.predict_proba(X_valid)[:,1]
y_valid_gb_pred = gradient_boosting.predict(X_valid)

# удаляем крупные файлы чтобы высвободить память 
del X_train, X_valid
gc.collect()

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_gb_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_gb_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_valid,y_valid_gb_pred,zero_division=0))

# time: 1m 40s

##### <span style="color:MediumSlateBlue">Анализ влияния порога классификации на качество модели</span>

In [ ]:
# чтобы проше обращаться к каждому обьекту в данных
# преобразуем y_valid_LR_pred к Series типу
y_valid_gb_pred_proba = pd.Series(y_valid_gb_pred_proba)

# формируем список из необходимых значений thresholds
thresholds = np.arange(0.01, 1, 0.01)

# сформируем списко для заполнения данными результатов анализа
threshold_data = []


for threshold in thresholds:
        # сформируем список для заполнения данными текушего threshold
        threshold_current_data = [threshold]

        #клиентов, для которых вероятность дефолта > threshold, относим к классу 1
        #в противном случае — к классу 0
        y_valid_gb_pred = y_valid_gb_pred_proba.apply(lambda x: 1 if x>threshold else 0)


        #Считаем метрики для класса 0 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(y_valid, y_valid_gb_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.precision_score(y_valid, y_valid_gb_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.f1_score(y_valid, y_valid_gb_pred,average=None,zero_division=0)[0],3))

        #Считаем метрики для класса 1 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(y_valid, y_valid_gb_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.precision_score(y_valid, y_valid_gb_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.f1_score(y_valid, y_valid_gb_pred,average=None,zero_division=0)[1],3))

        # добавляем полученные данные в список threshold_data
        threshold_data.append(threshold_current_data)

        # из полученных данных сформируем dataframe
        thresholds_pd =pd.DataFrame(
        columns=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'],
        data = threshold_data)

# time: 17s

In [ ]:
# Визуализируем метрики на валидационном наборе при различных threshold
fig_threshold = px.line(
    data_frame=thresholds_pd,
    x='threshold', #ось абсцисс
    y=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'], #ось ординат
)
fig_threshold.update_layout(
    title ={
        'text':'Зависимость качества модели от threshold', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Порог классификации threshold',
    yaxis_title='Величина метрики')
fig_threshold.update_xaxes(showspikes=True)
fig_threshold.update_yaxes(showspikes=True)

fig_threshold.show()

Метрика $precision$ показывает на сколько сильно ошиблась модель при определиние класса.  
Чем выше метрика, тем меньше ошиблась модель. 

$$precision = \frac{TP}{TP+FP}$$

Метрика $recall$ показывает способность модели найти класс.  
Чем выше метрика, тем выше способность модели находить класс.  

$$recall = \frac{TP}{TP+FN}$$

Метрика $f1-score$ показывает баланс между $precision$ и $recall$ 

$threshold$- порог класификации модели.  
Все объекты для которых $у_{\text{proba pred}} < threshold$, модель относит к класс 0.  
Соотвественно, если $у_{\text{proba pred}}  > threshold$ модель относит обьект к классу 1.  
Для идельаной модели $threshold$, является границей между областями класса 0 и класса 1.  

<span style="color:Blue">

Выводы по результам анализа:  

Для улучшения качества модели по метрике $ROC AUC$, нет смысла сдвигать порог   
классификации. Но зависимость метрик качества модели от $threshold$, может рассказать о  
способности модели искать класс 1 и класс 0.

При фокусировке модели на классе 1 (более низкий $threshold$):  
- на классе 1 метрика $precision$ уменьшается, метрика $recall$ растет;  
- на классе 0 метрика $precision$ растет, метрика $recall$ уменьшается.  

При фокусировке модели на классе 0 (более высокий $threshold$):  
- на классе 1 метрика $precision$ увеличивается, метрика $recall$ уменьшается;  
- на классе 0 метрика $precision$ уменьшается, метрика $recall$ растет.  

Обьсняется тем, что в условия дисбаланса модель научилась хорошо определять класс 0 и плохо класс 1.  

При фокусировке модели на классе 1 (более низкий $threshold$), все больше обьектов отнесены к классу 1.   
Так как $precision$ класса 1 уменьшается, значит в области класса 1 стало больше обектов класса 0,  
что явлется закономерным для "идеальной" модели.  
Однако при этом, $recall$ класса 1 растет, значит в добавленных обьектах также присутвует класс 1.  
Это как раз указывает на то, что модель не нучилась находить среди класса 0 класс 1. В области с     
высокой вероятностью класса 0 (более низкий $threshold$), находятся обьекты класса 1.  

Фокусировка на классе 0 (более высокий $threshold$), заставляет модель все больше обьектов из области класса 1  
относить к классу 0. При этом увеличение метрики $precision$ класса 1 говорит о том, что в области класса 1     
становится меньше 0, а значит модель умеет среди класса 1, находить класс 0.  
После  $threshold=0.88$ модель впринципе теряет способность отличать класс 1. У модели нет обьектов класса 1   
в которых она "уверена" на 90+%.


Обратим внимание на $threshold=0.51$, в этой точке находится баланс межу метриками качества модели:
$$precision_1 = recall_1 = \text{f1-score}_1 = 0.103$$
$$precision_0 = recall_0 = \text{f1-score}_0 = 0.97$$

Для модели с лучшим качеством эти две точки должны находится максимально близко к друг другу и иметь   
высокое значение метрик $precision_1$ и $recall_1$.  
Для класса 0 это условие выполнется, что также говорит о том, что модель научилась искать этот класс.    
Для класса 1 это условие не выполнется, что также говорит о том, что модель плохо ищет класс 1.

##### <span style="color:MediumSlateBlue">Построение ROC-кривой выбранной модели на тестовом наборе</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'date'
count_features = dict_spaces[feature_space]

In [ ]:
# сформируем данные для проверки модели
X_train = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
X_valid = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_valid').to_pandas()[list_ncorr_features]
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()
X_test = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_test').to_pandas()[list_ncorr_features]
y_test = pd.read_csv('target/target_test.csv')['flag'].to_numpy()

In [ ]:
print("Размеры обучающего набора",X_train.shape,y_train.shape)
print("Размеры валидационного набора",X_valid.shape,y_valid.shape)
print("Размеры тестового набора",X_test.shape,y_test.shape)
# time: 1m 43s

In [ ]:
# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_gb_pred_proba = gradient_boosting.predict_proba(X_train)[:,1]
y_valid_gb_pred_proba = gradient_boosting.predict_proba(X_valid)[:,1]
y_test_gb_pred_proba = gradient_boosting.predict_proba(X_test)[:,1]

# Делаем предсказание для валидационной выборки
y_test_gb_pred = gradient_boosting.predict(X_test)

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_gb_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_gb_pred_proba),3))
print('ROC AUC на тестовом наборе', round(metrics.roc_auc_score(y_test, y_test_gb_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_test,y_test_gb_pred,zero_division=0))

In [ ]:
# Для расчета значений TPR и FPR используем функцию roc_curve
fpr_train, tpr_train, _ = metrics.roc_curve(y_train, y_train_gb_pred_proba)
fpr_valid, tpr_valid, _ = metrics.roc_curve(y_valid, y_valid_gb_pred_proba)
fpr_test, tpr_test, _ = metrics.roc_curve(y_test, y_test_gb_pred_proba)

# рассчитаем площадь под кривой
auc_train = round(metrics.roc_auc_score(y_train, y_train_gb_pred_proba),3)
auc_valid = round(metrics.roc_auc_score(y_valid, y_valid_gb_pred_proba),3)
auc_test = round(metrics.roc_auc_score(y_test, y_test_gb_pred_proba),3)

trace_train = go.Scatter(x=fpr_train, y=tpr_train, mode='lines', name='AUC_train = %0.2f' % auc_train,
                   line=dict(color='red', width=2),
                   hovertemplate='train<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_train])

trace_valid = go.Scatter(x=fpr_valid, y=tpr_valid, mode='lines', name='AUC_valid = %0.2f' % auc_valid,
                   line=dict(color='green', width=2),
                   hovertemplate='valid<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_valid])

trace_test = go.Scatter(x=fpr_test, y=tpr_test, mode='lines', name='AUC_test = %0.2f' % auc_test,
                   line=dict(color='darkorange', width=2),
                   hovertemplate='test<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_test])


reference_line = go.Scatter(x=[0,1], y=[0,1], mode='lines', name='Reference Line',
                            line=dict(color='navy', width=2, dash='dash'))

fig_roc = go.Figure(data=[trace_train,trace_valid,trace_test,reference_line])

fig_roc.update_layout(
    title ={
        'text':'ROC-кривая', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 1350,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate')

fig_roc.update_xaxes(showspikes=True)
fig_roc.update_yaxes(showspikes=True)


fig_roc.show()

<span style="color:Blue">

Вывод по ROC-кривой:
1. Модель выдает сталибильное качество на валидационном и тестовом наборах, а значит  
не переобучена - разброс ($variance$) не большой. Хотя, качество модели на обучающем наборе,  
заметно лучше.
2. Смещение ($bias$) модели чуть лучше, чем у модели $Random$ $Forest$ $Classifier$,  
обученной на тех же данных.    
На тестовом наборе полученно качество $ROC AUC = 0.63$, против $ROC AUC = 0.61$.

### <span style="color:RoyalBlue">Построение модели на сбалансированных данных подпрострнства late stat</span>

#### <span style="color:MediumBlue">Формирование данных для обучения</span>

Анализ исходных данных, в частности target, показал что представленные данные   
имееют перекос: 96% клиентов у которых не случился дефолт (flag = 0),    
против 4 % - для которых произошел дефолт (flag = 1).  
С помощью функции class_1_percent_samples, сформируем сбаланисрованную выборку  
для обучения моделей.
За основу сбалансировных данных возьмем данные клиентов с дефолтом  
и к ним будем добирать необходимое количество клиентов до сбалансированности. 


Функция class_1_percent_samples принемает две переменные:
- target_data - массив с id и целевой переменной
- class_1_percent - процент класса 1 в результирующем массиве

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'late'
count_features = dict_spaces[feature_space]

In [ ]:
# сформируем данные для анализа сбалансированности обучающих данных
X_train_pd = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()
y_train_pd = pd.read_csv('target/target_train.csv')

In [ ]:
# поготовим данные y_train_pd к работе с функцией class_1_percent_samples
y_train_pd.set_index('id',drop=False,inplace=True)
y_train_pd.head(4)

In [ ]:
# взглянем на сблансиорованность данных в y_train
y_train_pd['flag'].value_counts(normalize=True)

In [ ]:
# сбалансируем данные в y_train_pd
list_c1_percent_id = class_1_percent_samples(y_train_pd,0.5)
# list_c1_percent_features - список 'id' из y_train_pd соотвествующих  
# заданному проценту класса 1
len(list_c1_percent_id)

In [ ]:
# проверим сбалансированность полученного y_train
y_train_pd.loc[list_c1_percent_id]['flag'].value_counts(normalize=True)

В отличие от данных в *transform_data_torow*, где модели показываеются последние N  
операций клиента, в данные в *transform_data_stat* сфомированы из различных статистических   
характеристик ряда определяющего клиентскую историю в банке. 

Соотвественно, если в данных *transform_data_torow* была сильная корреляция между признаками,  
то я не смог бы ее убрать, иначе потярется смысл "показать последовательный ряд в историии клиента".  
Признаки в *transform_data_stat* не имеют такой особенности. Поэтому, необходимо проанализировать  
данные в *transform_data_stat* и, в случае сильной коррреляции признаков, устранить ее.

In [ ]:
# подгрузим DataFrame с матрицей корреляции признаков
corr_matrix = fp.ParquetFile('base_models/data/stat/corr_matrix_'+feature_space).to_pandas()
corr_matrix.shape

In [ ]:
# с помощью функции corr_transrom_to_power получим
# силу корреляции матрицы и список не коррелируемых признаков

# для начального анализа зададим порог силы корреляции, при котором признаки 
# будут отнесены к "сильно скоррелированными" равным 0.6
ncorr_matrix,list_ncorr_features,corr_power =corr_transform_to_force(corr_matrix,threshold=0.6)

# Выведем результаты преобразования
print('Исходное количество признаков: ', corr_matrix.shape[0])
print('Количество не коррелируемых признаков: ', len(list_ncorr_features))
# сила корреляции признаков
print('Отношение коррелируемых исходному количеству признаков (сила корреляции): ',corr_power)

In [ ]:
# подготовим данные для обучения
X_train_balanced = X_train_pd.loc[list_c1_percent_id]
y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag']
# сформируем данные для проверики модели
# сформируем данные для анализа сбалансированности обучающих данных
X_train = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()
y_train = pd.read_csv('target/target_train.csv')['flag']
X_valid = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_valid').to_pandas()
y_valid = pd.read_csv('target/target_valid.csv')['flag']

In [ ]:
# посмотрим на структуру данных в X_train_balanced
X_train_balanced.head(4)

In [ ]:
# посмотрим на структуру данных в y_train_balanced
y_train_balanced.head(4)

In [ ]:
# проверим что выборки действительно совпадают по id
X_train_balanced.index.to_list() == y_train_balanced.index.to_list()

#### <span style="color:MediumBlue">Построение модели Logistic Regression</span>

##### <span style="color:MediumSlateBlue">Baseline обучение модели LogisticRegression</span>

In [ ]:
# обучаем модель логистической регрессии
logistic_regression = linear_model.LogisticRegression(random_state=42, max_iter=10000)
logistic_regression.fit(X_train_balanced[list_ncorr_features],y_train_balanced)

# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_LR_pred_proba = logistic_regression.predict_proba(X_train[list_ncorr_features])[:,1]
y_valid_LR_pred_proba = logistic_regression.predict_proba(X_valid[list_ncorr_features])[:,1]

# Делаем предсказание для валидационной выборки
y_valid_LR_pred = logistic_regression.predict(X_valid[list_ncorr_features])

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_LR_pred_proba),3))
print('ROC AUC на тестовом наборе', round(metrics.roc_auc_score(y_valid, y_valid_LR_pred_proba),3))

print('Основные метрики на тестовом наборе:')
print(metrics.classification_report(y_valid,y_valid_LR_pred))

# time: 3m 5s

##### <span style="color:MediumSlateBlue">Анализ влияния scaler преобразования на качество и скорость схождения модели</span>

In [ ]:
# формируем список из необходимых scaler
scalers = [MinMaxScaler(),RobustScaler() ,StandardScaler()]

# сформируем списко для заполнения данными результатов анализа
scalers_data = []

# установим ограничение на время обучения модели в 600 секунд (10 минут)
time_out = 600

for scaler in scalers:
        # для того чтобы следить за выполнением цикла введем индикатор
        print('current scaler: ', scaler)

         # сформируем список для заполнения данными текушего scaler
        current_scaler_data = [scaler]
        
        # выполним scaler преобразование
        X_train_s_balanced = scaler.fit_transform(X_train_balanced[list_ncorr_features])
        X_train_s = scaler.transform(X_train[list_ncorr_features])
        X_valid_s = scaler.transform(X_valid[list_ncorr_features])

        try:
                # для модуля func_time_out упакуем обучение модели в функцию try_func
                def try_func():
                        logistic_regression = linear_model.LogisticRegression(random_state=42, max_iter=10000)
                        return logistic_regression.fit(X_train_s_balanced,y_train_balanced)
                    
                # фиксируем время начала
                start_time = time.time()
                
                # запускаем обучение модели с ограничением по времени
                logistic_regression = func_timeout.func_timeout(time_out, try_func)
                
                # фиксируем время окончания обучения
                end_time = time.time()
                
                # запишем время обучения модели
                current_scaler_data.append(round(end_time-start_time))

                # Делаем предсказание для обучающей и валидационной выборки
                y_valid_lr_pred = logistic_regression.predict(X_valid_s)
                
                # для метрик ROC AUC делаем предсказание модели для обучающей и валидационной выборки  
                # в виде вероятности принадлежности к классу 1 
                y_train_lr_pred_proba = logistic_regression.predict_proba(X_train_s)[:,1]
                y_valid_lr_pred_proba = logistic_regression.predict_proba(X_valid_s)[:,1]

                #Считаем метрики ROC AUC yна обуающем и валидационном наборе и добавляем их в спискок
                current_scaler_data.append(round(metrics.roc_auc_score(y_train, y_train_lr_pred_proba),3))
                current_scaler_data.append(round(metrics.roc_auc_score(y_valid, y_valid_lr_pred_proba),3))

                #Считаем метрики для класса 0 на валидационном наборе и добавляем их в список
                current_scaler_data.append(round(metrics.recall_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))
                current_scaler_data.append(round(metrics.precision_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))


                #Считаем метрики для класса 1 на валидационном набопе и добавляем их в список
                current_scaler_data.append(round(metrics.recall_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))
                current_scaler_data.append(round(metrics.precision_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))

                # добавляем полученные данные в список scalers_data
                scalers_data.append(current_scaler_data)

                # чтобы подстраховаться на случай ошибки : "The Kernel crashed while executing code"
                # сохраним в локальную папку удачную итерацию
                # для этого из полученных данных сформируем dataframe
                # из полученных данных сформируем dataframe
                scaler_pd =pd.DataFrame(
                        columns=['scaler','time_fit',
                                 'roc_train','roc_valid', 
                                 'recall_0','precision_0',
                                 'recall_1','precision_1'],
                        data = scalers_data)
                # сохраняем dataframe
                scaler_pd.to_csv('data_recovery/scaler_pd.csv',index=False)
               
                # очистим output от прошедшей итерации
                clear_output()

        except func_timeout.FunctionTimedOut:
                # если время обучения привышает лимит
                # запишем в current_scaler_data соотвествующую данные
                current_scaler_data = [scaler,'>30m',0,0,0,0,0,0]
                # добавляем полученные данные в список scalers_data
                scalers_data.append(current_scaler_data)

                # чтобы подстраховаться на случай ошибки : "The Kernel crashed while executing code"
                # сохраним в локальную папку удачную итерацию
                # для этого из полученных данных сформируем dataframe
                # из полученных данных сформируем dataframe
                scaler_pd =pd.DataFrame(
                        columns=['scaler','time_fit',
                                 'roc_train','roc_valid', 
                                 'recall_0','precision_0',
                                 'recall_1','precision_1'],
                        data = scalers_data)
                # сохраняем dataframe
                scaler_pd.to_csv('data_recovery/scaler_pd.csv',index=False)
                
                # очистим output от прошедшей итерации
                clear_output()
                # переходим к следующему scaler
                pass
# очистим output
clear_output()

# сформируем данные соотвествующие лучщему ROC AUC на валидацинном наборе
best_roc_valid_data = scaler_pd[scaler_pd['roc_valid']==scaler_pd['roc_valid'].max()]

# выведем лучший результат на валидационном наборе
print('result:')
print('best scaler on valid:',best_roc_valid_data.iloc[0]['scaler'])
print('ROC AUC on train:', best_roc_valid_data.iloc[0]['roc_train'])
print('ROC AUC on valid:', best_roc_valid_data.iloc[0]['roc_valid'])
print('Time fit for best scaler:', best_roc_valid_data.iloc[0]['time_fit'],' seconds')

# выведем полученный dataframe
scaler_pd

# time: 

##### <span style="color:MediumSlateBlue"> Анализ влияния solver оптимизатора на качество и скорость обучения модели</span>

Проверим качество и скорость сходимости модели при разных solver  
Проверку осуществим через цикл  
Чтобы модель не сходилась бесконечно с помощью модуля func_timeout  
поставим ограничение времени схождения в 10 минут

In [ ]:
# подготовим данные для обучения модели

# обьявим scaler
scaler = StandardScaler()
# выполним scaler преобразование
X_train_s_balanced = scaler.fit_transform(X_train_balanced[list_ncorr_features])
X_train_s = scaler.transform(X_train[list_ncorr_features])
X_valid_s = scaler.transform(X_valid[list_ncorr_features])

# time: 10s

In [ ]:
# Зададим список solvers
solvers_list = ['lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga']

# сформируем список для заполнения
solvers_data = []

# установим ограничение на время обучения модели в 180 секунд (3 минут)
time_out = 180

for solver in solvers_list:
        # для того чтобы следить за выполнением цикла введем индикатор
        print('current solver: ', solver)

         # сформируем список для заполнения данными текушего solver
        current_solver_data = [solver]
        
        try:
                # для модуля func_time_out упакуем обучение модели в функцию try_func
                def try_func():
                        logistic_regression = linear_model.LogisticRegression(solver= solver,
                                                                  random_state=42, 
                                                                  max_iter=10000)
                        return logistic_regression.fit(X_train_s_balanced,y_train_balanced)
                    
                # фиксируем время начала
                start_time = time.time()
                
                # запускаем обучение модели с ограничением по времени
                logistic_regression = func_timeout.func_timeout(time_out, try_func)
                
                # фиксируем время окончания обучения
                end_time = time.time()
                
                # запишем время обучения модели
                current_solver_data.append(round(end_time-start_time))

                # Делаем предсказание для обучающей и валидационной выборки
                y_valid_lr_pred = logistic_regression.predict(X_valid_s)
                
                # для метрик ROC AUC делаем предсказание модели для обучающей и валидационной выборки  
                # в виде вероятности принадлежности к классу 1 
                y_train_lr_pred_proba = logistic_regression.predict_proba(X_train_s)[:,1]
                y_valid_lr_pred_proba = logistic_regression.predict_proba(X_valid_s)[:,1]

                #Считаем метрики ROC AUC yна обуающем и валидационном наборе и добавляем их в спискок
                current_solver_data.append(round(metrics.roc_auc_score(y_train, y_train_lr_pred_proba),3))
                current_solver_data.append(round(metrics.roc_auc_score(y_valid, y_valid_lr_pred_proba),3))

                #Считаем метрики для класса 0 на валидационном наборе и добавляем их в список
                current_solver_data.append(round(metrics.recall_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))
                current_solver_data.append(round(metrics.precision_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))


                #Считаем метрики для класса 1 на валидационном набопе и добавляем их в список
                current_solver_data.append(round(metrics.recall_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))
                current_solver_data.append(round(metrics.precision_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))

                # запишем данные полученные на одной итерации
                solvers_data.append(current_solver_data)

                # чтобы подстраховаться на случай ошибки : "The Kernel crashed while executing code"
                # сохраним в локальную папку удачную итерацию
                # для этого из полученных данных сформируем dataframe
                # из полученных данных сформируем dataframe
                solvers_pd =pd.DataFrame(
                        columns=['solver','time_fit',
                                 'roc_train','roc_valid', 
                                 'recall_0','precision_0',
                                 'recall_1','precision_1'],
                        data = solvers_data)
                # сохраняем dataframe
                solvers_pd.to_csv('data_recovery/solvers_pd.csv',index=False)
               
                # очистим output от прошедшей итерации
                clear_output()

        except func_timeout.FunctionTimedOut:
                # если время обучения привышает лимит
                # запишем в current_solver_data соотвествующую данные
                current_solver_data = [solver,'>3m',0,0,0,0,0,0]
                # добавляем полученные данные в список solvers_data
                solvers_data.append(current_solver_data)

                # чтобы подстраховаться на случай ошибки : "The Kernel crashed while executing code"
                # сохраним в локальную папку удачную итерацию
                # для этого из полученных данных сформируем dataframe
                # из полученных данных сформируем dataframe
                solvers_pd =pd.DataFrame(
                        columns=['solver','time_fit',
                                 'roc_train','roc_valid', 
                                 'recall_0','precision_0',
                                 'recall_1','precision_1'],
                        data = solvers_data)
                # сохраняем dataframe
                solvers_pd.to_csv('data_recovery/solvers_pd.csv',index=False)
                
                # очистим output от прошедшей итерации
                clear_output()
                # переходим к следующему solver
                pass
# очистим output
clear_output()

# сформируем данные соотвествующие лучщему ROC AUC на валидацинном наборе
best_roc_valid_data = solvers_pd[solvers_pd['roc_valid']==solvers_pd['roc_valid'].max()]

# выведем лучший результат на валидационном наборе
print('result:')
print('best solver on valid:',best_roc_valid_data.iloc[0]['solver'])
print('ROC AUC on train:', best_roc_valid_data.iloc[0]['roc_train'])
print('ROC AUC on valid:', best_roc_valid_data.iloc[0]['roc_valid'])
print('Time fit for best solver:', best_roc_valid_data.iloc[0]['time_fit'],' seconds')

# выведем полученный dataframe
solvers_pd

# time: 2m 35s

##### <span style="color:MediumSlateBlue">Подбор гиперпараметров модели</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'late'
count_features = dict_spaces[feature_space]

# для работы функции corr_transform_to_force
# подгрузим DataFrame с матрицей корреляции признаков
corr_matrix = fp.ParquetFile('base_models/data/stat/corr_matrix_'+feature_space).to_pandas()

# для выбора sceler преобразвания нам понадобится словарь
dict_scalers ={
    'MinMaxScaler': MinMaxScaler(),
    'RobustScaler':RobustScaler(),
    'StandardScaler':StandardScaler()}

# настроим оптимизацию гипер параметров
def optuna_lg(trial):
  # задаем пространства поиска гиперпараметров
  solver = trial.suggest_categorical("solver", ['lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky'])
  C = trial.suggest_float('C',0.01,1)
  class_0_weight = trial.suggest_float('class_0_weight',0.2,0.6)
  class_1_weight = trial.suggest_float('class_1_weight',0.2,0.6)
  threshold = trial.suggest_float('threshold',0.6,1)
  class_1_percent = trial.suggest_float('class_1_percent',0.01,0.3)
  scaler = trial.suggest_categorical("scaler", ['MinMaxScaler','RobustScaler','StandardScaler'])
  random_state = trial.suggest_int('random_state', 1, 1000000)


  # создаем модель
  optuna_log_reg = linear_model.LogisticRegression(
      solver=solver,
      C=C,
      class_weight={0:class_0_weight,1:class_1_weight},
      random_state=random_state,
      max_iter=10000)

  # с помощью функции corr_transform_to_force получим
  # список не коррелируемых признаков
  list_ncorr_features = corr_transform_to_force(corr_matrix,threshold=threshold)[1]

  # обьявим scaler
  scaler = dict_scalers[scaler]

  # сформируем данные для анализа сбалансированности обучающих данных
  X_train_pd = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
  y_train_pd = pd.read_csv('target/target_train.csv')

  # поготовим данные y_train_pd к работе с функцией class_1_percent_samples
  y_train_pd.set_index('id',drop=False,inplace=True)

  # подготовим данные для обучения модели
  # с помощью функции class_1_percent_samples зададим долю 
  # класса 1
  list_c1_percent_id = class_1_percent_samples(y_train_pd,class_1_percent,random_state=random_state)[:1000000]

  # подготовим данные для обучения
  X_train_s_balanced = scaler.fit_transform(X_train_pd.loc[list_c1_percent_id])
  y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

  # освободим память от "тяжелых" и ненужных файлов
  del X_train_pd, y_train_pd
  gc.collect()

  # я не воспользовался параметром timeout метода optimize потому, что  
  # optimize отставливает поиск параметров, если время обучения 
  # превышает timeout. Мне нужно чтобы поиск параметров продолжился.
  try:
    # для модуля func_time_out упакуем обучение модели в функцию try_func
    def try_func():
            return optuna_log_reg.fit(X_train_s_balanced, y_train_balanced)
    # обучаем модель с ограничением по времени 10 минут (600 секунд)
    optuna_log_reg = func_timeout.func_timeout(600, try_func)

    # удаляем крупные файлы чтобы высвободить память 
    del X_train_s_balanced, y_train_balanced
    gc.collect()

    # загружаем тестовые наборы
    # сформируем данные для анализа сбалансированности обучающих данных
    X_train = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
    X_valid = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_valid').to_pandas()[list_ncorr_features]
    y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
    y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()

    # подготовим данные для проверки модели
    X_train_s = scaler.transform(X_train)
    X_valid_s = scaler.transform(X_valid)

    # удаляем крупные файлы чтобы высвободить память 
    del X_train, X_valid
    gc.collect()

    # делаем предсказание на обучающем и валидационном наборе и считаем метрики
    roc_train = metrics.roc_auc_score(y_train, optuna_log_reg.predict_proba(X_train_s)[:,1])
    roc_valid = metrics.roc_auc_score(y_valid, optuna_log_reg.predict_proba(X_valid_s)[:,1])
    f1_score = metrics.f1_score(y_valid, optuna_log_reg.predict(X_valid_s))
    recall_1 = metrics.recall_score(y_valid, optuna_log_reg.predict(X_valid_s),average=None,zero_division=0)[1]
    precision_1 = metrics.precision_score(y_valid, optuna_log_reg.predict(X_valid_s),average=None,zero_division=0)[1]

    # удаляем крупные файлы чтобы высвободить память 
    del X_train_s, X_valid_s, y_train, y_valid
    gc.collect()

  except func_timeout.FunctionTimedOut:
    # удаляем крупные файлы чтобы высвободить память 
    del X_train_s_balanced, y_train_balanced
    gc.collect()
    
    # фиксируем пустые значения метрик
    roc_train = 0
    roc_valid = 0
    f1_score = 0
    recall_1 = 0
    precision_1 = 0
    pass

  return roc_train, roc_valid, f1_score, recall_1, precision_1

In [ ]:
# cоздаем объект исследования
# чтобы модель не переобучалась минимизируем roc_train 
# и максимизируем roc_valid
optuna_study_lg = optuna.create_study(study_name='LogisticRegression_'+feature_space+'_stat', 
                               directions=['maximize','maximize','maximize','maximize','maximize'], 
                               sampler=optuna.samplers.TPESampler(),
                               pruner='Hyperband',
                               storage='sqlite:///optuna_studies.db',
                               load_if_exists=True)
# optuna.delete_study(study_name='LogisticRegression_'+feature_space+'_stat', storage='sqlite:///optuna_studies.db')

In [ ]:
# ищем лучшую комбинацию гиперпараметров n_trials раз
optuna_study_lg.optimize(optuna_lg, n_trials=300)

In [ ]:
# из полученного результат соврмируем Data Frame
optuna_study_lg_pd = optuna_study_lg.trials_dataframe().dropna()
# переименуем столбы
optuna_study_lg_pd.rename(columns={
    'values_0': 'roc_train',
    'values_1': 'roc_valid',
    'values_2': 'f1_score',
    'values_3': 'recall_1',
    'values_4': 'precision_1'
},inplace=True)
# Выделим время потраченное на обучение модели
optuna_study_lg_pd.tail()

In [ ]:
# покаждем статистику обучения
print('Максимальное значение метрики ROC AUC на валидационном наборе:',round(optuna_study_lg_pd['roc_valid'].max(),3))
print('Среднее значение метрики ROC AUC на валидационном наборе:',round(optuna_study_lg_pd['roc_valid'].mean(),3))
print('Максимальное значение метрики f1_score на валидационном наборе:',round(optuna_study_lg_pd['f1_score'].max(),3))
print('Среднее значение метрики f1_score на валидационном наборе:',round(optuna_study_lg_pd['f1_score'].mean(),3))
print('Максимальное значение метрики recall_1 на валидационном наборе:',round(optuna_study_lg_pd['recall_1'].max(),3))
print('Среднее значение метрики recall_1 на валидационном наборе:',round(optuna_study_lg_pd['recall_1'].mean(),3))
print('Максимальное значение метрики precision_1 на валидационном наборе:',round(optuna_study_lg_pd['precision_1'].max(),3))
print('Среднее значение метрики precision_1 на валидационном наборе:',round(optuna_study_lg_pd['precision_1'].mean(),3))

In [ ]:
# построим график важности гиперпарметров
optuna.visualization.plot_param_importances(optuna_study_lg, target_name='Score')

In [ ]:
# Построим зависимость метрики precision_1
fig = px.scatter(
    data_frame=optuna_study_lg_pd,
    x='precision_1', #ось абсцисс
    y=['f1_score', 'recall_1'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision от recall на классе 1', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision',
    yaxis_title='Величина метрик recall и f1-score')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики precision_1 от гипер параметров

fig = px.scatter(
    data_frame=optuna_study_lg_pd,
    x='precision_1', #ось абсцисс
    y=['params_C', 'params_class_0_weight', 
       'params_class_1_weight','params_class_1_percent',
       'params_threshold'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики f1_score от гипер параметров

fig = px.scatter(
    data_frame=optuna_study_lg_pd,
    x='f1_score', #ось абсцисс
    y=['params_C', 'params_class_0_weight', 
       'params_class_1_weight','params_class_1_percent',
       'params_threshold'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость f1-score от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики f1-score',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики roc_valid от гипер параметров

fig = px.scatter(
    data_frame=optuna_study_lg_pd,
    x='roc_valid', #ось абсцисс
    y=['params_C', 'params_class_0_weight', 
       'params_class_1_weight','params_class_1_percent',
       'params_threshold'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость ROC AUC на валидационном наборе от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики ROC AUC',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Визуализируем метрики полученные при optuna оптимизации
fig = px.scatter(
    data_frame=optuna_study_lg_pd[(optuna_study_lg_pd['precision_1']>0.08)
                                  & (optuna_study_lg_pd['recall_1']>0.08) 
                                  & (optuna_study_lg_pd['roc_valid']>0.8*optuna_study_lg_pd['roc_valid'].max())],
    x='number', #ось абсцисс
    y=['roc_train','roc_valid','recall_1','precision_1'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость качества модели от trials optuna', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='trials optuna',
    yaxis_title='Величина метрики')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

<span style="color:Blue">

Вывод:  

Их всех выбираем точку $number =234$.   
В этой точке самые оптимальные значения $recall$ и $precision$.  
Посмотрим каким значениям гиперпараметров соотвествует выбранная точка.

In [ ]:
# определим номер лучшге варианта
best_optuna_number = 234

# сформируем словарь лучших гипирпарметров RandomForestClassifier
best_param_lr = {
    'solver' : optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_solver'].iloc[0],
    'C' : round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_C'].iloc[0],3),
    'class_weight' : {0:round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_class_0_weight'].iloc[0],3),
                      1:round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_class_1_weight'].iloc[0],3)},
    }

# создадим перменные
best_threshold = optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_threshold'].iloc[0]
best_scaler = optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_scaler'].iloc[0]
best_class_1_percent = optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_class_1_percent'].iloc[0]
best_random_state = optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_random_state'].iloc[0]

# Выведем принятые наилучшие праметры
print('best solver:',best_param_lr['solver'])
print('best C:',best_param_lr['C'])
print('best class 0 weight:',best_param_lr['class_weight'][0])
print('best class 1 weight:',best_param_lr['class_weight'][1])

print('best class 1 percent:',round(best_class_1_percent,3))
print('best scaler:',best_scaler)
print('best threshold:',round(best_threshold,3))
print('best best random state:',best_random_state)
print('time for best train:',round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['duration'].iloc[0].seconds/60),'minutes')

print()
print('ROC AUC на обучающем наборе:', round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['roc_train'].iloc[0],3))
print('ROC AUC на валидационном наборе:', round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['roc_valid'].iloc[0],3))
print('precision класса 1:', round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['precision_1'].iloc[0],3))
print('recall класса 1:', round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['recall_1'].iloc[0],3))


##### <span style="color:MediumSlateBlue">Обучение модели с лучшими параметрами</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'late'
count_features = dict_spaces[feature_space]

# для работы функции corr_transform_to_force
# подгрузим DataFrame с матрицей корреляции признаков
corr_matrix = fp.ParquetFile('base_models/data/stat/corr_matrix_'+feature_space).to_pandas()

# подготовим данные
# для выбора sceler преобразвания нам понадобится словарь
dict_scalers ={
    'MinMaxScaler': MinMaxScaler(),
    'RobustScaler':RobustScaler(),
    'StandardScaler':StandardScaler(),
}

# с помощью функции corr_transform_to_force получим
# список не коррелируемых признаков
list_ncorr_features = corr_transform_to_force(corr_matrix,threshold=best_threshold)[1]

# обьявим scaler
scaler = dict_scalers[best_scaler]

# сформируем данные для анализа сбалансированности обучающих данных
X_train_pd = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
y_train_pd = pd.read_csv('target/target_train.csv')

# поготовим данные y_train_pd к работе с функцией class_1_percent_samples
y_train_pd.set_index('id',drop=False,inplace=True)

# подготовим данные для обучения модели
# с помощью функции class_1_percent_samples зададим долю 
# класса 1
list_c1_percent_id = class_1_percent_samples(y_train_pd,best_class_1_percent,random_state=best_random_state)[:1000000]

# подготовим данные для обучения
X_train_s_balanced = scaler.fit_transform(X_train_pd.loc[list_c1_percent_id])
y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

# освободим память от "тяжелых" и ненужных файлов
del X_train_pd, y_train_pd
gc.collect()


# обучаем модель LogisticRegression с наилучшеми параметрами
logistic_regression = linear_model.LogisticRegression(
        **best_param_lr,
        random_state=42,
        max_iter=10000)
logistic_regression.fit(X_train_s_balanced, y_train_balanced)

# удаляем крупные файлы чтобы высвободить память 
del X_train_s_balanced, y_train_balanced
gc.collect()

# загружаем тестовые наборы
# сформируем данные для анализа сбалансированности обучающих данных
X_train = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
X_valid = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_valid').to_pandas()[list_ncorr_features]
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()

# подготовим данные для проверки модели
X_train_s = scaler.transform(X_train)
X_valid_s = scaler.transform(X_valid)

# удаляем крупные файлы чтобы высвободить память 
del X_train, X_valid
gc.collect()

# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_lr_pred_proba = logistic_regression.predict_proba(X_train_s)[:,1]
y_valid_lr_pred_proba = logistic_regression.predict_proba(X_valid_s)[:,1]

# Делаем предсказание для валидационной выборки
y_valid_lr_pred = logistic_regression.predict(X_valid_s)

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_lr_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_lr_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_valid,y_valid_lr_pred,zero_division=0))

# time: 2m 30s

##### <span style="color:MediumSlateBlue">Анализ влияния порога классификации на качество модели</span>

In [ ]:
# чтобы проше обращаться к каждому обьекту в данных
# преобразуем y_valid_LR_pred к Series типу
y_valid_lr_pred_proba = pd.Series(y_valid_lr_pred_proba)

# формируем список из необходимых значений thresholds
thresholds = np.arange(0.01, 1, 0.01)

# сформируем списко для заполнения данными результатов анализа
threshold_data = []


for threshold in thresholds:
        # сформируем список для заполнения данными текушего threshold
        threshold_current_data = [threshold]

        #клиентов, для которых вероятность дефолта > threshold, относим к классу 1
        #в противном случае — к классу 0
        y_valid_lr_pred = y_valid_lr_pred_proba.apply(lambda x: 1 if x>threshold else 0)


        #Считаем метрики для класса 0 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.precision_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.f1_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))

        #Считаем метрики для класса 1 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.precision_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.f1_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))

        # добавляем полученные данные в список threshold_data
        threshold_data.append(threshold_current_data)

        # из полученных данных сформируем dataframe
        thresholds_pd =pd.DataFrame(
        columns=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'],
        data = threshold_data)

# time: 17s

In [ ]:
# Визуализируем метрики на валидационном наборе при различных threshold
fig_threshold = px.line(
    data_frame=thresholds_pd,
    x='threshold', #ось абсцисс
    y=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'], #ось ординат
)
fig_threshold.update_layout(
    title ={
        'text':'Зависимость качества модели от threshold', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Порог классификации threshold',
    yaxis_title='Величина метрики')
fig_threshold.update_xaxes(showspikes=True)
fig_threshold.update_yaxes(showspikes=True)

fig_threshold.show()

Метрика $precision$ показывает на сколько сильно ошиблась модель при определиние класса.  
Чем выше метрика, тем меньше ошиблась модель. 

$$precision = \frac{TP}{TP+FP}$$

Метрика $recall$ показывает способность модели найти класс.  
Чем выше метрика, тем выше способность модели находить класс.  

$$recall = \frac{TP}{TP+FN}$$

Метрика $f1-score$ показывает баланс между $precision$ и $recall$ 

$threshold$- порог класификации модели.  
Все объекты для которых $у_{\text{proba pred}} < threshold$, модель относит к класс 0.  
Соотвественно, если $у_{\text{proba pred}}  > threshold$ модель относит обьект к классу 1.  
Для идельаной модели $threshold$, является границей между областями класса 0 и класса 1.  

<span style="color:Blue">

Выводы по результам анализа:  

Для улучшения качества модели по метрике $ROC AUC$, нет смысла сдвигать порог   
классификации. Но зависимость метрик качества модели от $threshold$, может рассказать о  
спосбоности модели искать класс 1 и класс 0.

При фокусировке модели на классе 1 (более низкий $threshold$):  
- на классе 1 метрика $precision$ уменьшается, метрика $recall$ растет;  
- на классе 0 метрика $precision$ растет, метрика $recall$ уменьшается.  

При фокусировке модели на классе 0 (более высокий $threshold$):  
- на классе 1 метрика $precision$ увеличивается, метрика $recall$ уменьшается;  
- на классе 0 метрика $precision$ уменьшается, метрика $recall$ растет.  

Обьсняется тем, что в условия дисбаланса модель научилась хорошо определять класс 0 и плохо класс 1.  

При фокусировке модели на классе 1 (более низкий $threshold$), все больше обьектов отнесены к классу 1.   
Так как $precision$ класса 1 уменьшается, значит в области класса 1 стало больше обектов класса 0,  
что явлется закономерным для "идеальной" модели.  
Однако при этом, $recall$ класса 1 растет, значит в добавленных обьектах также присутвует класс 1.  
Это как раз указывает на то, что модель не научилась находить среди класса 0 класс 1. В области с     
высокой вероятностью класса 0 (более низкий $threshold$), находятся обьекты класса 1.  

Фокусировка на классе 0 (более высокий $threshold$), заставляет модель все больше обьектов из области класса 1  
относить к классу 0. При этом увеличение метрики $precision$ класса 1 говорит о том, что в области класса 1     
становится меньше 0, а значит модель умеет среди класса 1, находить класс 0.  
После  $threshold=0.64$ модель впринципе теряет способность отличать класс 1. У модели нет обьектов класса 1   
в которых она "уверена" на 80+%.

Обратим внимание на $threshold=0.52$, в этой точке находится баланс межу метриками качества модели:
$$precision_1 = recall_1 = \text{f1-score}_1 = 0.082$$
$$precision_0 = recall_0 = \text{f1-score}_0 = 0.965$$

Для модели с лучшим качеством эти две точки должны находится максимально близко к друг другу и иметь  
высокое значение метрик $precision_1$ и $recall_1$.
Для класса 0 это условие выполнется, что также говорит о том, что модель научилась искать этот класс.  
Для класса 1 это условие не выполнется, что также говорит о том, что модель плохо ищет класс 1.

##### <span style="color:MediumSlateBlue">Построение ROC-кривой выбранной модели на тестовом наборе</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'late'
count_features = dict_spaces[feature_space]

In [ ]:
# сформируем данные для проверки модели
X_train = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
X_valid = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_valid').to_pandas()[list_ncorr_features]
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()
X_test = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_test').to_pandas()[list_ncorr_features]
y_test = pd.read_csv('target/target_test.csv')['flag'].to_numpy()

In [ ]:
# выполним scaler преобразование
X_train_s = scaler.transform(X_train)
X_valid_s = scaler.transform(X_valid)
X_test_s = scaler.transform(X_test)

print("Размеры обучающего набора",X_train_s.shape)
print("Размеры валидационного набора",X_valid_s.shape)
print("Размеры тестового набора",X_test_s.shape)
# time: 1m 43s

In [ ]:
# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_lr_pred_proba = logistic_regression.predict_proba(X_train_s)[:,1]
y_valid_lr_pred_proba = logistic_regression.predict_proba(X_valid_s)[:,1]
y_test_lr_pred_proba = logistic_regression.predict_proba(X_test_s)[:,1]

# Делаем предсказание для валидационной выборки
y_test_lr_pred = logistic_regression.predict(X_test_s)

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_lr_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_lr_pred_proba),3))
print('ROC AUC на тестовом наборе', round(metrics.roc_auc_score(y_test, y_test_lr_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_test,y_test_lr_pred,zero_division=0))

In [ ]:
# Для расчета значений TPR и FPR используем функцию roc_curve
fpr_train, tpr_train, _ = metrics.roc_curve(y_train, y_train_lr_pred_proba)
fpr_valid, tpr_valid, _ = metrics.roc_curve(y_valid, y_valid_lr_pred_proba)
fpr_test, tpr_test, _ = metrics.roc_curve(y_test, y_test_lr_pred_proba)

# рассчитаем площадь под кривой
auc_train = round(metrics.roc_auc_score(y_train, y_train_lr_pred_proba),3)
auc_valid = round(metrics.roc_auc_score(y_valid, y_valid_lr_pred_proba),3)
auc_test = round(metrics.roc_auc_score(y_test, y_test_lr_pred_proba),3)

trace_train = go.Scatter(x=fpr_train, y=tpr_train, mode='lines', name='AUC_train = %0.2f' % auc_train,
                   line=dict(color='red', width=2),
                   hovertemplate='train<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_train])

trace_valid = go.Scatter(x=fpr_valid, y=tpr_valid, mode='lines', name='AUC_valid = %0.2f' % auc_valid,
                   line=dict(color='green', width=2),
                   hovertemplate='valid<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_valid])

trace_test = go.Scatter(x=fpr_test, y=tpr_test, mode='lines', name='AUC_test = %0.2f' % auc_test,
                   line=dict(color='darkorange', width=2),
                   hovertemplate='test<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_test])


reference_line = go.Scatter(x=[0,1], y=[0,1], mode='lines', name='Reference Line',
                            line=dict(color='navy', width=2, dash='dash'))

fig_roc = go.Figure(data=[trace_train,trace_valid,trace_test,reference_line])

fig_roc.update_layout(
    title ={
        'text':'ROC-кривая', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 1350,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate')

fig_roc.update_xaxes(showspikes=True)
fig_roc.update_yaxes(showspikes=True)


fig_roc.show()

<span style="color:Blue">

Вывод по ROC-кривой:
1. Модель выдает сталибильное качество на всех наборах, а значит  
не переобучена - разброс ($variance$) не большой.  
2. Смещение ($bias$) модели относительно большое.  
На тестовом наборе полученно качество $ROC AUC = 0.6$.

#### <span style="color:MediumBlue">Построение модели Random Forest Classifier</span>

##### <span style="color:MediumSlateBlue">Baseline обучение модели RandomForestClassifier</span>

In [ ]:
# обучаем модель логистической регрессии
# чтобы модель пыталась найти связь во всех признаках
random_forest = RandomForestClassifier(random_state=42, n_jobs=-1)
random_forest.fit(X_train_balanced[list_ncorr_features],y_train_balanced)

# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_rf__pred_proba = random_forest.predict_proba(X_train[list_ncorr_features])[:,1]
y_valid_rf_pred_proba = random_forest.predict_proba(X_valid[list_ncorr_features])[:,1]

# Делаем предсказание для валидационной выборки
y_valid_rf_pred = random_forest.predict(X_valid[list_ncorr_features])

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_rf__pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_rf_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_valid,y_valid_rf_pred))

# time: 17s

##### <span style="color:MediumSlateBlue">Подбор гиперпараметров модели</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'late'
count_features = dict_spaces[feature_space]

# для работы функции corr_transform_to_force
# подгрузим DataFrame с матрицей корреляции признаков
corr_matrix = fp.ParquetFile('base_models/data/stat/corr_matrix_'+feature_space).to_pandas()

# настроим оптимизацию гипер параметров
def optuna_rf(trial):
  # задаем пространства поиска гиперпараметров
  # задаем пространства поиска гиперпараметров
  n_estimators = trial.suggest_int('n_estimators', 40, 100,step = 5)
  criterion = trial.suggest_categorical('criterion',['gini', 'entropy', 'log_loss'])
  max_depth = trial.suggest_int('max_depth', 10, 30,step = 1)
  min_samples_split = trial.suggest_int('min_samples_split', 5, 15,step = 1)
  min_samples_leaf = trial.suggest_int('min_samples_leaf', 5, 15,step = 1)
  max_features = trial.suggest_float('max_features',0.4,0.8)
  max_samples = trial.suggest_float('max_samples',0.4,0.8)
  class_0_weight = trial.suggest_float('class_0_weight',0.1,1)
  class_1_weight = trial.suggest_float('class_1_weight',0.1,1)
  threshold = trial.suggest_float('threshold',0.4,0.8)
  class_1_percent = trial.suggest_float('class_1_percent',0.01,0.5)
  random_state = trial.suggest_int('random_state', 1, 1000000)


  # создаем модель
  optuna_random_forest = RandomForestClassifier(
      n_estimators = n_estimators, 
      criterion = criterion,
      max_depth = max_depth,
      min_samples_split = min_samples_split,  
      min_samples_leaf = min_samples_leaf,
      max_features=max_features,
      max_samples=max_samples,
      class_weight={0:class_0_weight,1:class_1_weight},
      n_jobs=-1,
      random_state=42)

  # с помощью функции corr_transform_to_force получим
  # список не коррелируемых признаков
  list_ncorr_features = corr_transform_to_force(corr_matrix,threshold=threshold)[1]

  # сформируем данные для анализа сбалансированности обучающих данных
  X_train_pd = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
  y_train_pd = pd.read_csv('target/target_train.csv')

  # поготовим данные y_train_pd к работе с функцией class_1_percent_samples
  y_train_pd.set_index('id',drop=False,inplace=True)

  # подготовим данные для обучения модели
  # с помощью функции class_1_percent_samples зададим долю 
  # класса 1
  list_c1_percent_id = class_1_percent_samples(y_train_pd,class_1_percent,random_state=random_state)[:1000000]

  # подготовим данные для обучения
  X_train_balanced = X_train_pd.loc[list_c1_percent_id]
  y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag']

  # освободим память от "тяжелых" и ненужных файлов
  del X_train_pd, y_train_pd
  gc.collect()

  # я не воспользовался параметром timeout метода optimize потому, что  
  # optimize отставливает поиск параметров, если время обучения 
  # превышает timeout. Мне нужно чтобы поиск параметров продолжился.
  try:
    # для модуля func_time_out упакуем обучение модели в функцию try_func
    def try_func():
            return optuna_random_forest.fit(X_train_balanced, y_train_balanced)
    # обучаем модель с ограничением по времени 10 минут (600 секунд)
    optuna_random_forest = func_timeout.func_timeout(600, try_func)

    # удаляем крупные файлы чтобы высвободить память 
    del X_train_balanced, y_train_balanced
    gc.collect()

    # загружаем тестовые наборы
    # сформируем данные для анализа сбалансированности обучающих данных
    X_train = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
    X_valid = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_valid').to_pandas()[list_ncorr_features]
    y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
    y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()
    
    # делаем предсказание на обучающем и валидационном наборе
    #Считаем метрики для класса 1 добавляем их в список
    roc_train = metrics.roc_auc_score(y_train, optuna_random_forest.predict_proba(X_train)[:,1])
    roc_valid = metrics.roc_auc_score(y_valid, optuna_random_forest.predict_proba(X_valid)[:,1])
    f1_score = metrics.f1_score(y_valid, optuna_random_forest.predict(X_valid))
    recall_1 = metrics.recall_score(y_valid, optuna_random_forest.predict(X_valid),average=None,zero_division=0)[1]
    precision_1 = metrics.precision_score(y_valid, optuna_random_forest.predict(X_valid),average=None,zero_division=0)[1]

    # удаляем крупные файлы чтобы высвободить память 
    del X_train, X_valid, y_train, y_valid
    gc.collect()

  except func_timeout.FunctionTimedOut:
    # удаляем крупные файлы чтобы высвободить память 
    del X_train_balanced, y_train_balanced
    gc.collect()
    
    # фиксируем пустые значения метрик
    roc_train = 0
    roc_valid = 0
    f1_score = 0
    recall_1 = 0
    precision_1 = 0
    pass

  return roc_train, roc_valid, f1_score, recall_1, precision_1

In [ ]:
# cоздаем объект исследования
optuna_study_rf = optuna.create_study(study_name= 'RandomForestClassifier_'+feature_space+'_stat', 
                               directions=['maximize','maximize','maximize','maximize','maximize'], 
                               sampler=optuna.samplers.TPESampler(),
                               pruner='Hyperband',
                               storage='sqlite:///optuna_studies.db',
                               load_if_exists=True)
# на случай удаления 
# optuna.delete_study(study_name='RandomForestClassifier_'+feature_space+'_stat', storage='sqlite:///optuna_studies.db')

In [ ]:
# ищем лучшую комбинацию гиперпараметров n_trials раз
optuna_study_rf.optimize(optuna_rf, n_trials=300)

In [ ]:
# из полученного результат соврмируем Data Frame
optuna_study_rf_pd = optuna_study_rf.trials_dataframe().dropna()
# переименуем столбы
optuna_study_rf_pd.rename(columns={
    'values_0': 'roc_train',
    'values_1': 'roc_valid',
    'values_2': 'f1_score',
    'values_3': 'recall_1',
    'values_4': 'precision_1'
},inplace=True)
optuna_study_rf_pd.tail()

In [ ]:
# покаждем статистику обучения
print('Максимальное значение метрики ROC AUC на валидационном наборе:',round(optuna_study_rf_pd['roc_valid'].max(),3))
print('Среднее значение метрики ROC AUC на валидационном наборе:',round(optuna_study_rf_pd['roc_valid'].mean(),3))
print('Максимальное значение метрики f1_score на валидационном наборе:',round(optuna_study_rf_pd['f1_score'].max(),3))
print('Среднее значение метрики f1_score на валидационном наборе:',round(optuna_study_rf_pd['f1_score'].mean(),3))
print('Максимальное значение метрики recall_1 на валидационном наборе:',round(optuna_study_rf_pd['recall_1'].max(),3))
print('Среднее значение метрики recall_1 на валидационном наборе:',round(optuna_study_rf_pd['recall_1'].mean(),3))
print('Максимальное значение метрики precision_1 на валидационном наборе:',round(optuna_study_rf_pd['precision_1'].max(),3))
print('Среднее значение метрики precision_1 на валидационном наборе:',round(optuna_study_rf_pd['precision_1'].mean(),3))

In [ ]:
# Построим зависимость метрики precision_1

fig = px.scatter(
    data_frame=optuna_study_rf_pd,
    x='precision_1', #ось абсцисс
    y=['recall_1','f1_score'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision от recall на классе 1', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision на классе 1',
    yaxis_title='Величина метрик recall и f1-score')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики precision_1 от гиперпараметров

fig = px.scatter(
    data_frame=optuna_study_rf_pd,
    x='precision_1', #ось абсцисс
    y=['params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight', 'params_max_depth',
       'params_max_features', 'params_max_samples', 'params_min_samples_leaf',
       'params_min_samples_split', 'params_n_estimators', 'params_threshold'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision класса 1 от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision на классе 1',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики f1_score от гиперпараметров

fig = px.scatter(
    data_frame=optuna_study_rf_pd,
    x='f1_score', #ось абсцисс
    y=['params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight', 'params_max_depth',
       'params_max_features', 'params_max_samples', 'params_min_samples_leaf',
       'params_min_samples_split', 'params_n_estimators', 'params_threshold'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость f1-score класса 1 от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики f1-score на классе 1',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики roc_valid от гиперпараметров

fig = px.scatter(
    data_frame=optuna_study_rf_pd,
    x='roc_valid', #ось абсцисс
    y=['params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight', 'params_max_depth',
       'params_max_features', 'params_max_samples', 'params_min_samples_leaf',
       'params_min_samples_split', 'params_n_estimators', 'params_threshold'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость ROC AUC на валидационном наборе от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики ROC AUV на валидационном наборе',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрик качества модели от number

fig = px.scatter(
    data_frame=optuna_study_rf_pd[(optuna_study_rf_pd['recall_1']>=0.08) & (optuna_study_rf_pd['precision_1']>0.08)],
    x='number', #ось абсцисс
    y=['roc_valid','roc_train','precision_1','recall_1'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость качества модели от trials optuna', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='trials optuna',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

<span style="color:Blue">

Вывод:  

Их всех выбираем точку $number =188$.   
Не смотря на, то что в этой точке модель подает признаки переобучеености,  
в этой точке оптимальные значения метрик $recall_1$ и $precision_1$, а  
также относительно высокая метрика $ROC AUC$.   
Посмотрим каким значениям гиперпараметров соотвествует выбраннаяточка.

In [ ]:
# определим номер лучшге варианта
best_optuna_number = 188

# сформируем словарь лучших гипирпарметров RandomForestClassifier
best_param_rf = {
    'n_estimators' : int(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_n_estimators'].iloc[0]),
    'criterion': optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_criterion'].iloc[0],
    'max_depth' : int(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_max_depth'].iloc[0]),
    'min_samples_split': int(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_min_samples_split'].iloc[0]),
    'min_samples_leaf' : int(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_min_samples_leaf'].iloc[0]),
    'max_features': optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_max_features'].iloc[0],
    'max_samples' : optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_max_samples'].iloc[0],
    'class_weight' : {0:optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_class_0_weight'].iloc[0],
                      1:optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_class_1_weight'].iloc[0]}
    }

# определим перменные для лучших значений параметров
best_threshold = optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_threshold'].iloc[0]
best_class_1_percent = optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_class_1_percent'].iloc[0]
best_random_state = optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_random_state'].iloc[0]


# Выведем принятые наилучшие праметры
print('best n estimators:',best_param_rf['n_estimators'])
print('best criterion:',best_param_rf['criterion'])
print('best max depth:',best_param_rf['max_depth'])
print('best min samples split:',best_param_rf['min_samples_split'])
print('best min samples leaf:',best_param_rf['min_samples_leaf'])
print('best max features(%):',round(best_param_rf['max_features'],3))
print('best max samples(%):',round(best_param_rf['max_samples'],3))
print('best class 0 weight:',round(best_param_rf['class_weight'][0],3))
print('best class 1 weight:',round(best_param_rf['class_weight'][1],3))

print('best class 1 percent:',round(best_class_1_percent,3))
print('best threshold:',round(best_threshold,3))
print('best random state:', best_random_state)
print('time for best train:',round(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['duration'].iloc[0].seconds/60),'minutes')

print()
print('ROC AUC на обучающем наборе:', round(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['roc_train'].iloc[0],3))
print('ROC AUC на валидационном наборе:', round(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['roc_valid'].iloc[0],3))
print('precision класса 1:', round(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['precision_1'].iloc[0],3))
print('recall класса 1:', round(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['recall_1'].iloc[0],3))

##### <span style="color:MediumSlateBlue">Обучение модели с лучшими параметрами</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'late'
count_features = dict_spaces[feature_space]

# для работы функции corr_transform_to_force
# подгрузим DataFrame с матрицей корреляции признаков
corr_matrix = fp.ParquetFile('base_models/data/stat/corr_matrix_'+feature_space).to_pandas()

# с помощью функции corr_transform_to_force получим
# список не коррелируемых признаков
list_ncorr_features = corr_transform_to_force(corr_matrix,threshold=best_threshold)[1]

# сформируем данные для анализа сбалансированности обучающих данных
X_train_pd = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
y_train_pd = pd.read_csv('target/target_train.csv')

# поготовим данные y_train_pd к работе с функцией class_1_percent_samples
y_train_pd.set_index('id',drop=False,inplace=True)

# подготовим данные для обучения модели
# с помощью функции class_1_percent_samples зададим долю 
# класса 1
list_c1_percent_id = class_1_percent_samples(y_train_pd,best_class_1_percent,random_state=best_random_state)[:1000000]

# подготовим данные для обучения
X_train_balanced = X_train_pd.loc[list_c1_percent_id]
y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag']

# освободим память от "тяжелых" и ненужных файлов
del X_train_pd, y_train_pd
gc.collect()


# обучаем модель RandomForestClassifier с наилучшеми параметрами
random_forest = RandomForestClassifier(
        **best_param_rf,
        n_jobs=-1,
        random_state=42)
random_forest.fit(X_train_balanced, y_train_balanced)

# удаляем крупные файлы чтобы высвободить память 
del X_train_balanced, y_train_balanced
gc.collect()

# загружаем тестовые наборы
# сформируем данные для анализа сбалансированности обучающих данных
X_train = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
X_valid = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_valid').to_pandas()[list_ncorr_features]
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()

# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_rf_pred_proba = random_forest.predict_proba(X_train)[:,1]
y_valid_rf_pred_proba = random_forest.predict_proba(X_valid)[:,1]

# Делаем предсказание для валидационной выборки
y_valid_rf_pred = random_forest.predict(X_valid)

# удаляем крупные файлы чтобы высвободить память 
del X_train, X_valid
gc.collect()

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_rf_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_rf_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_valid,y_valid_rf_pred,zero_division=0))

# time: 5m 30s

##### <span style="color:MediumSlateBlue">Анализ влияния порога классификации на качество модели</span>

In [ ]:
# чтобы проше обращаться к каждому обьекту в данных
# преобразуем y_valid_LR_pred к Series типу
y_valid_rf_pred_proba = pd.Series(y_valid_rf_pred_proba)

# формируем список из необходимых значений thresholds
thresholds = np.arange(0.01, 1, 0.01)

# сформируем списко для заполнения данными результатов анализа
threshold_data = []


for threshold in thresholds:
        # сформируем список для заполнения данными текушего threshold
        threshold_current_data = [threshold]

        #клиентов, для которых вероятность дефолта > threshold, относим к классу 1
        #в противном случае — к классу 0
        y_valid_rf_pred = y_valid_rf_pred_proba.apply(lambda x: 1 if x>threshold else 0)


        #Считаем метрики для класса 0 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(y_valid, y_valid_rf_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.precision_score(y_valid, y_valid_rf_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.f1_score(y_valid, y_valid_rf_pred,average=None,zero_division=0)[0],3))

        #Считаем метрики для класса 1 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(y_valid, y_valid_rf_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.precision_score(y_valid, y_valid_rf_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.f1_score(y_valid, y_valid_rf_pred,average=None,zero_division=0)[1],3))

        # добавляем полученные данные в список threshold_data
        threshold_data.append(threshold_current_data)

        # из полученных данных сформируем dataframe
        thresholds_pd =pd.DataFrame(
        columns=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'],
        data = threshold_data)

# time: 17s

In [ ]:
# Визуализируем метрики на валидационном наборе при различных threshold
fig_threshold = px.line(
    data_frame=thresholds_pd,
    x='threshold', #ось абсцисс
    y=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'], #ось ординат
)
fig_threshold.update_layout(
    title ={
        'text':'Зависимость качества модели от threshold', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Порог классификации threshold',
    yaxis_title='Величина метрики')
fig_threshold.update_xaxes(showspikes=True)
fig_threshold.update_yaxes(showspikes=True)

fig_threshold.show()

Метрика $precision$ показывает на сколько сильно ошиблась модель при определиние класса.  
Чем выше метрика, тем меньше ошиблась модель. 

$$precision = \frac{TP}{TP+FP}$$

Метрика $recall$ показывает способность модели найти класс.  
Чем выше метрика, тем выше способность модели находить класс.  

$$recall = \frac{TP}{TP+FN}$$

Метрика $f1-score$ показывает баланс между $precision$ и $recall$ 

$threshold$- порог класификации модели.  
Все объекты для которых $у_{\text{proba pred}} < threshold$, модель относит к класс 0.  
Соотвественно, если $у_{\text{proba pred}}  > threshold$ модель относит обьект к классу 1.  
Для идельаной модели $threshold$, является границей между областями класса 0 и класса 1.  

<span style="color:Blue">

Выводы по результам анализа:  

Для улучшения качества модели по метрике $ROC AUC$, нет смысла сдвигать порог   
классификации. Но зависимость метрик качества модели от $threshold$, может рассказать о  
способности модели искать класс 1 и класс 0.

При фокусировке модели на классе 1 (более низкий $threshold$):  
- на классе 1 метрика $precision$ уменьшается, метрика $recall$ растет;  
- на классе 0 метрика $precision$ растет, метрика $recall$ уменьшается.  

При фокусировке модели на классе 0 (более высокий $threshold$):  
- на классе 1 метрика $precision$ увеличивается, метрика $recall$ уменьшается;  
- на классе 0 метрика $precision$ уменьшается, метрика $recall$ растет.  

Обьсняется тем, что в условия дисбаланса модель научилась хорошо определять класс 0 и плохо класс 1.  

При фокусировке модели на классе 1 (более низкий $threshold$), все больше обьектов отнесены к классу 1.   
Так как $precision$ класса 1 уменьшается, значит в области класса 1 стало больше обектов класса 0,  
что явлется закономерным для "идеальной" модели.  
Однако при этом, $recall$ класса 1 растет, значит в добавленных обьектах также присутвует класс 1.  
Это как раз указывает на то, что модель не нучилась находить среди класса 0 класс 1. В области с     
высокой вероятностью класса 0 (более низкий $threshold$), находятся обьекты класса 1.  

Фокусировка на классе 0 (более высокий $threshold$), заставляет модель все больше обьектов из области класса 1  
относить к классу 0. При этом увеличение метрики $precision$ класса 1 говорит о том, что в области класса 1     
становится меньше 0, а значит модель умеет среди класса 1, находить класс 0.  
После  $threshold=0.77$ модель впринципе теряет способность отличать класс 1. У модели нет обьектов класса 1   
в которых она "уверена" на 80+%.

Обратим внимание на $threshold=0.49$, в этой точке находится баланс межу метриками качества модели:
$$precision_1 = recall_1 = \text{f1-score}_1 = 0.09$$
$$precision_0 = recall_0 = \text{f1-score}_0 = 0.968$$

Для модели с лучшим качеством эти две точки должны находится максимально близко к друг другу и иметь   
высокое значение метрик $precision_1$ и $recall_1$.  
Для класса 0 это условие выполнется, что также говорит о том, что модель научилась искать этот класс.    
Для класса 1 это условие не выполнется, что также говорит о том, что модель плохо ищет класс 1.

##### <span style="color:MediumSlateBlue">Построение ROC-кривой выбранной модели на тестовом наборе</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'late'
count_features = dict_spaces[feature_space]

In [ ]:
# сформируем данные для проверки модели
X_train = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
X_valid = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_valid').to_pandas()[list_ncorr_features]
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()
X_test = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_test').to_pandas()[list_ncorr_features]
y_test = pd.read_csv('target/target_test.csv')['flag'].to_numpy()

In [ ]:
print("Размеры обучающего набора",X_train.shape,y_train.shape)
print("Размеры валидационного набора",X_valid.shape,y_valid.shape)
print("Размеры тестового набора",X_test.shape,y_test.shape)
# time: 1m 43s

In [ ]:
# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_rf_pred_proba = random_forest.predict_proba(X_train)[:,1]
y_valid_rf_pred_proba = random_forest.predict_proba(X_valid)[:,1]
y_test_rf_pred_proba = random_forest.predict_proba(X_test)[:,1]

# Делаем предсказание для валидационной выборки
y_test_lr_pred = random_forest.predict(X_test)

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_rf_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_rf_pred_proba),3))
print('ROC AUC на тестовом наборе', round(metrics.roc_auc_score(y_test, y_test_rf_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_test,y_test_lr_pred,zero_division=0))

In [ ]:
# Для расчета значений TPR и FPR используем функцию roc_curve
fpr_train, tpr_train, _ = metrics.roc_curve(y_train, y_train_rf_pred_proba)
fpr_valid, tpr_valid, _ = metrics.roc_curve(y_valid, y_valid_rf_pred_proba)
fpr_test, tpr_test, _ = metrics.roc_curve(y_test, y_test_rf_pred_proba)

# рассчитаем площадь под кривой
auc_train = round(metrics.roc_auc_score(y_train, y_train_rf_pred_proba),3)
auc_valid = round(metrics.roc_auc_score(y_valid, y_valid_rf_pred_proba),3)
auc_test = round(metrics.roc_auc_score(y_test, y_test_rf_pred_proba),3)

trace_train = go.Scatter(x=fpr_train, y=tpr_train, mode='lines', name='AUC_train = %0.2f' % auc_train,
                   line=dict(color='red', width=2),
                   hovertemplate='train<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_train])

trace_valid = go.Scatter(x=fpr_valid, y=tpr_valid, mode='lines', name='AUC_valid = %0.2f' % auc_valid,
                   line=dict(color='green', width=2),
                   hovertemplate='valid<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_valid])

trace_test = go.Scatter(x=fpr_test, y=tpr_test, mode='lines', name='AUC_test = %0.2f' % auc_test,
                   line=dict(color='darkorange', width=2),
                   hovertemplate='test<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_test])


reference_line = go.Scatter(x=[0,1], y=[0,1], mode='lines', name='Reference Line',
                            line=dict(color='navy', width=2, dash='dash'))

fig_roc = go.Figure(data=[trace_train,trace_valid,trace_test,reference_line])

fig_roc.update_layout(
    title ={
        'text':'ROC-кривая', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 1350,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate')

fig_roc.update_xaxes(showspikes=True)
fig_roc.update_yaxes(showspikes=True)


fig_roc.show()

<span style="color:Blue">

Вывод по ROC-кривой:
1. Не смотря на то, что модель показывает, очень хорошую $ROC$ кривую на обучающем наборе,  
модель, так же выдает сталибильное качество на валидационном и тестовом наборах, а значит  
не переобучена - разброс ($variance$) не большой.
2. Смещение ($bias$) модели такое же как у модели $Logistic$ $Regression$ $Classifier$,  
обученной  на тех же данных.  
На тестовом наборе полученно качество $ROC AUC = 0.61$.

#### <span style="color:MediumBlue">Построение модели Gradient Boosting Classifier</span>

##### <span style="color:MediumSlateBlue">Baseline обучение модели Hist Gradient Boosting Classifier</span>

In [ ]:
# обучаем модель Gradient Boosting Classifier
gradient_boosting = HistGradientBoostingClassifier(random_state=42)
gradient_boosting.fit(X_train_balanced[list_ncorr_features],y_train_balanced)

# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_rf__pred_proba = gradient_boosting.predict_proba(X_train[list_ncorr_features])[:,1]
y_valid_rf_pred_proba = gradient_boosting.predict_proba(X_valid[list_ncorr_features])[:,1]

# Делаем предсказание для валидационной выборки
y_valid_rf_pred = gradient_boosting.predict(X_valid[list_ncorr_features])

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_rf__pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_rf_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_valid,y_valid_rf_pred))

# time: 30s

##### <span style="color:MediumSlateBlue">Подбор гиперпараметров модели</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'late'
count_features = dict_spaces[feature_space]

# для работы функции corr_transform_to_force
# подгрузим DataFrame с матрицей корреляции признаков
corr_matrix = fp.ParquetFile('base_models/data/stat/corr_matrix_'+feature_space).to_pandas()

# настроим оптимизацию гипер параметров
def optuna_hgbc(trial):
  # задаем пространства поиска гиперпараметров
  learning_rate = trial.suggest_float('learning_rate',0.01,0.1)
  max_iter = trial.suggest_int('max_iter', 150, 250,step = 1)
  max_leaf_nodes = trial.suggest_int('max_leaf_nodes', 2, 60,step = 1)
  max_depth = trial.suggest_int('max_depth', 1, 10,step = 1)
  min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 60,step = 1)
  max_features = trial.suggest_float('max_features',0.6,1)
  l2_regularization = trial.suggest_float('l2_regularization',0.01,1)
  class_0_weight = trial.suggest_float('class_0_weight',0.01,1)
  class_1_weight = trial.suggest_float('class_1_weight',0.5,1)
  threshold = trial.suggest_float('threshold',0.7,1)
  class_1_percent = trial.suggest_float('class_1_percent',0.01,0.25)
  random_state = trial.suggest_int('random_state', 1, 1000000)

  # создаем модель
  optuna_gradient_boosting = HistGradientBoostingClassifier(
      learning_rate= learning_rate,
      max_iter = max_iter,
      max_leaf_nodes =max_leaf_nodes,
      max_depth = max_depth,
      min_samples_leaf = min_samples_leaf,
      max_features=max_features,
      l2_regularization = l2_regularization,
      class_weight={0:class_0_weight,1:class_1_weight},
      random_state=42)

  # с помощью функции corr_transform_to_force получим
  # список не коррелируемых признаков
  list_ncorr_features = corr_transform_to_force(corr_matrix,threshold=threshold)[1]

  # сформируем данные для анализа сбалансированности обучающих данных
  X_train_pd = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
  y_train_pd = pd.read_csv('target/target_train.csv')

  # поготовим данные y_train_pd к работе с функцией class_1_percent_samples
  y_train_pd.set_index('id',drop=False,inplace=True)

  # подготовим данные для обучения модели
  # с помощью функции class_1_percent_samples зададим долю 
  # класса 1
  list_c1_percent_id = class_1_percent_samples(y_train_pd,class_1_percent,random_state=random_state)[:1000000]

  # подготовим данные для обучения
  X_train_balanced = X_train_pd.loc[list_c1_percent_id]
  y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag']

  # освободим память от "тяжелых" и ненужных файлов
  del X_train_pd, y_train_pd
  gc.collect()

  # я не воспользовался параметром timeout метода optimize потому, что  
  # optimize отставливает поиск параметров, если время обучения 
  # превышает timeout. Мне нужно чтобы поиск параметров продолжился.
  try:
    # для модуля func_time_out упакуем обучение модели в функцию try_func
    def try_func():
            return optuna_gradient_boosting.fit(X_train_balanced,y_train_balanced)
    # обучаем модель с ограничением по времени 10 минут (600 секунд)
    optuna_gradient_boosting = func_timeout.func_timeout(600, try_func)

    # удаляем крупные файлы чтобы высвободить память 
    del X_train_balanced, y_train_balanced
    gc.collect()

        # загружаем тестовые наборы
    # сформируем данные для анализа сбалансированности обучающих данных
    X_train = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
    X_valid = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_valid').to_pandas()[list_ncorr_features]
    y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
    y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()
    
    # делаем предсказание на обучающем и валидационном наборе и считаем метрики
    roc_train = metrics.roc_auc_score(y_train, optuna_gradient_boosting.predict_proba(X_train)[:,1])
    roc_valid = metrics.roc_auc_score(y_valid, optuna_gradient_boosting.predict_proba(X_valid)[:,1])
    f1_score = metrics.f1_score(y_valid, optuna_gradient_boosting.predict(X_valid))
    recall_1 = metrics.recall_score(y_valid, optuna_gradient_boosting.predict(X_valid),average=None,zero_division=0)[1]
    precision_1 = metrics.precision_score(y_valid, optuna_gradient_boosting.predict(X_valid),average=None,zero_division=0)[1]


    # удаляем крупные файлы чтобы высвободить память 
    del X_train, X_valid, y_train, y_valid
    gc.collect()

  except func_timeout.FunctionTimedOut:
    # удаляем крупные файлы чтобы высвободить память 
    del X_train_balanced, y_train_balanced
    gc.collect()
    
    # фиксируем пустые значения метрик
    roc_train = 0
    roc_valid = 0
    f1_score = 0
    recall_1 = 0
    precision_1 = 0
    pass

  return roc_train, roc_valid, f1_score, recall_1, precision_1

In [ ]:
# cоздаем объект исследования
# чтобы модель не переобучалась минимизируем roc_train 
# и максимизируем roc_valid
optuna_study_hgbc = optuna.create_study(study_name='HistGradientBoostingClassifier_'+feature_space+'_stat', 
                               directions=['maximize','maximize','maximize','maximize','maximize'], 
                               sampler=optuna.samplers.TPESampler(),
                               pruner='Hyperband',
                               storage='sqlite:///optuna_studies.db',
                               load_if_exists=True)

# ну случай удаления обучения
# optuna.delete_study(study_name='HistGradientBoostingClassifier'+feature_space+'_stat', storage='sqlite:///optuna_studies.db')

In [ ]:
# ищем лучшую комбинацию гиперпараметров n_trials раз
optuna_study_hgbc.optimize(optuna_hgbc, n_trials=300)

In [ ]:
# из полученного результат соврмируем Data Frame
optuna_study_hgbc_pd = optuna_study_hgbc.trials_dataframe()
# переименуем столбы
optuna_study_hgbc_pd.rename(columns={
    'values_0': 'roc_train',
    'values_1': 'roc_valid',
    'values_2': 'f1_score',
    'values_3': 'recall_1',
    'values_4': 'precision_1'
},inplace=True)
optuna_study_hgbc_pd.tail(5)

In [ ]:
# покаждем статистику обучения
print('Максимальное значение метрики ROC AUC на валидационном наборе:',round(optuna_study_hgbc_pd['roc_valid'].max(),3))
print('Среднее значение метрики ROC AUC на валидационном наборе:',round(optuna_study_hgbc_pd['roc_valid'].mean(),3))
print('Максимальное значение метрики f1_score на валидационном наборе:',round(optuna_study_hgbc_pd['f1_score'].max(),3))
print('Среднее значение метрики f1_score на валидационном наборе:',round(optuna_study_hgbc_pd['f1_score'].mean(),3))
print('Максимальное значение метрики recall_1 на валидационном наборе:',round(optuna_study_hgbc_pd['recall_1'].max(),3))
print('Среднее значение метрики recall_1 на валидационном наборе:',round(optuna_study_hgbc_pd['recall_1'].mean(),3))
print('Максимальное значение метрики precision_1 на валидационном наборе:',round(optuna_study_hgbc_pd['precision_1'].max(),3))
print('Среднее значение метрики precision_1 на валидационном наборе:',round(optuna_study_hgbc_pd['precision_1'].mean(),3))

In [ ]:
# построим график важности гиперпарметров
optuna.visualization.plot_param_importances(optuna_study_hgbc, target_name='Score')

In [ ]:
# Построим зависимость метрики precision_1
fig = px.scatter(
    data_frame=optuna_study_hgbc_pd,
    x='precision_1', #ось абсцисс
    y=['f1_score','recall_1'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision от recall на классе 1', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision класса 1',
    yaxis_title='Величина метрик recall и f1-score класса 1')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики precision_1 от гиперпараметров
fig = px.scatter(
    data_frame=optuna_study_hgbc_pd,
    x='precision_1', #ось абсцисс
    y=['params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight', 'params_l2_regularization',
       'params_learning_rate', 'params_max_depth', 'params_max_features',
       'params_max_iter', 'params_max_leaf_nodes', 'params_min_samples_leaf',
       'params_threshold'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision класса 1 от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision класса 1',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики f1_score от гиперпараметров
fig = px.scatter(
    data_frame=optuna_study_hgbc_pd,
    x='f1_score', #ось абсцисс
    y=['params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight', 'params_l2_regularization',
       'params_learning_rate', 'params_max_depth', 'params_max_features',
       'params_max_iter', 'params_max_leaf_nodes', 'params_min_samples_leaf',
       'params_threshold'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость f1-score класса 1 от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики f1-score класса 1',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики roc_valid от гиперпараметров
fig = px.scatter(
    data_frame=optuna_study_hgbc_pd,
    x='roc_valid', #ось абсцисс
    y=['params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight', 'params_l2_regularization',
       'params_learning_rate', 'params_max_depth', 'params_max_features',
       'params_max_iter', 'params_max_leaf_nodes', 'params_min_samples_leaf',
       'params_threshold'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость ROC AUC на валидационном наборе от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики ROC AUC на валидационном наборе',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрик качества модели от number

fig = px.scatter(
    data_frame=optuna_study_hgbc_pd[(optuna_study_hgbc_pd['recall_1']>0.1)&(optuna_study_hgbc_pd['precision_1']>0.1)],
    x='number', #ось абсцисс
    y=['roc_train','roc_valid','recall_1','precision_1'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость качества модели от trials optuna', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='trials optuna',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

<span style="color:Blue">

Вывод:  

Их всех выбираем точку $number =127$.   
В этой точке оптимальные значения метрик $recall_1$ и $precision_1$, а  
также относительно высокая метрика $ROC AUC$.   
Посмотрим каким значениям гиперпараметров соотвествует выбранная точка.

In [ ]:
# определим номер лучшге варианта
best_optuna_number = 127

# сформируем словарь лучших гипирпарметров HistGradientBoostingClassifier
best_param_hgbc = {
    'learning_rate' : optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['params_learning_rate'].iloc[0],
    'max_iter' : optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['params_max_iter'].iloc[0],
    'max_leaf_nodes' : optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['params_max_leaf_nodes'].iloc[0],
    'max_depth' : optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['params_max_depth'].iloc[0],
    'min_samples_leaf' : optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['params_min_samples_leaf'].iloc[0],
    'max_features' : optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['params_max_features'].iloc[0],
    'l2_regularization' : optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['params_l2_regularization'].iloc[0],
    'class_weight' : {0: optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['params_class_0_weight'].iloc[0],
                      1: optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['params_class_1_weight'].iloc[0],}
    }

# определим перменные для лучших значений параметров
best_threshold = optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['params_threshold'].iloc[0]
best_class_1_percent = optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['params_class_1_percent'].iloc[0]
best_random_state = optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['params_random_state'].iloc[0]


# Выведем принятые наилучшие параметры
print('best learning rate:',round(best_param_hgbc['learning_rate'],3))
print('best max iter:',best_param_hgbc['max_iter'])
print('best max leaf nodes:',best_param_hgbc['max_leaf_nodes'])
print('best max depth:',best_param_hgbc['max_depth'])
print('best min samples leaf:',best_param_hgbc['min_samples_leaf'])
print('best max features:',round(best_param_hgbc['max_features'],3))
print('best l2 regularization:',round(best_param_hgbc['l2_regularization'],3))
print('best class 0 weight:',round(best_param_hgbc['class_weight'][0],3))
print('best class 1 weight:',round(best_param_hgbc['class_weight'][1],3))

print('best class 1 percent:',round(best_class_1_percent,3))
print('best threshold:',round(best_threshold,3))
print('time for best train:',round(optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['duration'].iloc[0].seconds/60),'minutes')

print()
print('ROC AUC на обучающем наборе:', round(optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['roc_train'].iloc[0],3))
print('ROC AUC на валидационном наборе:', round(optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['roc_valid'].iloc[0],3))
print('precision класса 1:', round(optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['precision_1'].iloc[0],3))
print('recall класса 1:', round(optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['recall_1'].iloc[0],3))

##### <span style="color:MediumSlateBlue">Обучение модели с лучшими параметрами</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'late'
count_features = dict_spaces[feature_space]

# для работы функции corr_transform_to_force
# подгрузим DataFrame с матрицей корреляции признаков
corr_matrix = fp.ParquetFile('base_models/data/stat/corr_matrix_'+feature_space).to_pandas()

# с помощью функции corr_transform_to_force получим
# список не коррелируемых признаков
list_ncorr_features = corr_transform_to_force(corr_matrix,threshold=best_threshold)[1]

# сформируем данные для анализа сбалансированности обучающих данных
X_train_pd = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
y_train_pd = pd.read_csv('target/target_train.csv')

# поготовим данные y_train_pd к работе с функцией class_1_percent_samples
y_train_pd.set_index('id',drop=False,inplace=True)

# подготовим данные для обучения модели
# с помощью функции class_1_percent_samples зададим долю 
# класса 1
list_c1_percent_id = class_1_percent_samples(y_train_pd,best_class_1_percent,random_state=best_random_state)[:1000000]

# подготовим данные для обучения
X_train_balanced = X_train_pd.loc[list_c1_percent_id]
y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag']

# освободим память от "тяжелых" и ненужных файлов
del X_train_pd, y_train_pd
gc.collect()

# обучаем модель HistGradientBoostingClassifier с наилучшеми параметрами
gradient_boosting = HistGradientBoostingClassifier(
        **best_param_hgbc,
        random_state=42)
gradient_boosting.fit(X_train_balanced,y_train_balanced)

# удаляем крупные файлы чтобы высвободить память 
del X_train_balanced, y_train_balanced
gc.collect()

# сформируем данные для анализа сбалансированности обучающих данных
X_train = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
X_valid = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_valid').to_pandas()[list_ncorr_features]
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()

# делаем предсказание на обучающем и валидационном наборе и считаем метрики
y_train_gb_pred_proba = gradient_boosting.predict_proba(X_train)[:,1]
y_valid_gb_pred_proba = gradient_boosting.predict_proba(X_valid)[:,1]
y_valid_gb_pred = gradient_boosting.predict(X_valid)

# удаляем крупные файлы чтобы высвободить память 
del X_train, X_valid
gc.collect()

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_gb_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_gb_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_valid,y_valid_gb_pred,zero_division=0))

# time: 1m 40s

##### <span style="color:MediumSlateBlue">Анализ влияния порога классификации на качество модели</span>

In [ ]:
# чтобы проше обращаться к каждому обьекту в данных
# преобразуем y_valid_LR_pred к Series типу
y_valid_gb_pred_proba = pd.Series(y_valid_gb_pred_proba)

# формируем список из необходимых значений thresholds
thresholds = np.arange(0.01, 1, 0.01)

# сформируем списко для заполнения данными результатов анализа
threshold_data = []


for threshold in thresholds:
        # сформируем список для заполнения данными текушего threshold
        threshold_current_data = [threshold]

        #клиентов, для которых вероятность дефолта > threshold, относим к классу 1
        #в противном случае — к классу 0
        y_valid_gb_pred = y_valid_gb_pred_proba.apply(lambda x: 1 if x>threshold else 0)


        #Считаем метрики для класса 0 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(y_valid, y_valid_gb_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.precision_score(y_valid, y_valid_gb_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.f1_score(y_valid, y_valid_gb_pred,average=None,zero_division=0)[0],3))

        #Считаем метрики для класса 1 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(y_valid, y_valid_gb_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.precision_score(y_valid, y_valid_gb_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.f1_score(y_valid, y_valid_gb_pred,average=None,zero_division=0)[1],3))

        # добавляем полученные данные в список threshold_data
        threshold_data.append(threshold_current_data)

        # из полученных данных сформируем dataframe
        thresholds_pd =pd.DataFrame(
        columns=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'],
        data = threshold_data)

# time: 17s

In [ ]:
# Визуализируем метрики на валидационном наборе при различных threshold
fig_threshold = px.line(
    data_frame=thresholds_pd,
    x='threshold', #ось абсцисс
    y=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'], #ось ординат
)
fig_threshold.update_layout(
    title ={
        'text':'Зависимость качества модели от threshold', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Порог классификации threshold',
    yaxis_title='Величина метрики')
fig_threshold.update_xaxes(showspikes=True)
fig_threshold.update_yaxes(showspikes=True)

fig_threshold.show()

Метрика $precision$ показывает на сколько сильно ошиблась модель при определиние класса.  
Чем выше метрика, тем меньше ошиблась модель. 

$$precision = \frac{TP}{TP+FP}$$

Метрика $recall$ показывает способность модели найти класс.  
Чем выше метрика, тем выше способность модели находить класс.  

$$recall = \frac{TP}{TP+FN}$$

Метрика $f1-score$ показывает баланс между $precision$ и $recall$ 

$threshold$- порог класификации модели.  
Все объекты для которых $у_{\text{proba pred}} < threshold$, модель относит к класс 0.  
Соотвественно, если $у_{\text{proba pred}}  > threshold$ модель относит обьект к классу 1.  
Для идельаной модели $threshold$, является границей между областями класса 0 и класса 1.  

<span style="color:Blue">

Выводы по результам анализа:  

Для улучшения качества модели по метрике $ROC AUC$, нет смысла сдвигать порог   
классификации. Но зависимость метрик качества модели от $threshold$, может рассказать о  
способности модели искать класс 1 и класс 0.

При фокусировке модели на классе 1 (более низкий $threshold$):  
- на классе 1 метрика $precision$ уменьшается, метрика $recall$ растет;  
- на классе 0 метрика $precision$ растет, метрика $recall$ уменьшается.  

При фокусировке модели на классе 0 (более высокий $threshold$):  
- на классе 1 метрика $precision$ увеличивается, метрика $recall$ уменьшается;  
- на классе 0 метрика $precision$ уменьшается, метрика $recall$ растет.  

Обьсняется тем, что в условия дисбаланса модель научилась хорошо определять класс 0 и плохо класс 1.  

При фокусировке модели на классе 1 (более низкий $threshold$), все больше обьектов отнесены к классу 1.   
Так как $precision$ класса 1 уменьшается, значит в области класса 1 стало больше обектов класса 0,  
что явлется закономерным для "идеальной" модели.  
Однако при этом, $recall$ класса 1 растет, значит в добавленных обьектах также присутвует класс 1.  
Это как раз указывает на то, что модель не нучилась находить среди класса 0 класс 1. В области с     
высокой вероятностью класса 0 (более низкий $threshold$), находятся обьекты класса 1.  

Фокусировка на классе 0 (более высокий $threshold$), заставляет модель все больше обьектов из области класса 1  
относить к классу 0. При этом увеличение метрики $precision$ класса 1 говорит о том, что в области класса 1     
становится меньше 0, а значит модель умеет среди класса 1, находить класс 0.  
После  $threshold=0.88$ модель впринципе теряет способность отличать класс 1. У модели нет обьектов класса 1   
в которых она "уверена" на 90+%.


Обратим внимание на $threshold=0.51$, в этой точке находится баланс межу метриками качества модели:
$$precision_1 = recall_1 = \text{f1-score}_1 = 0.103$$
$$precision_0 = recall_0 = \text{f1-score}_0 = 0.97$$

Для модели с лучшим качеством эти две точки должны находится максимально близко к друг другу и иметь   
высокое значение метрик $precision_1$ и $recall_1$.  
Для класса 0 это условие выполнется, что также говорит о том, что модель научилась искать этот класс.    
Для класса 1 это условие не выполнется, что также говорит о том, что модель плохо ищет класс 1.

##### <span style="color:MediumSlateBlue">Построение ROC-кривой выбранной модели на тестовом наборе</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'late'
count_features = dict_spaces[feature_space]

In [ ]:
# сформируем данные для проверки модели
X_train = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
X_valid = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_valid').to_pandas()[list_ncorr_features]
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()
X_test = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_test').to_pandas()[list_ncorr_features]
y_test = pd.read_csv('target/target_test.csv')['flag'].to_numpy()

In [ ]:
print("Размеры обучающего набора",X_train.shape,y_train.shape)
print("Размеры валидационного набора",X_valid.shape,y_valid.shape)
print("Размеры тестового набора",X_test.shape,y_test.shape)
# time: 1m 43s

In [ ]:
# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_gb_pred_proba = gradient_boosting.predict_proba(X_train)[:,1]
y_valid_gb_pred_proba = gradient_boosting.predict_proba(X_valid)[:,1]
y_test_gb_pred_proba = gradient_boosting.predict_proba(X_test)[:,1]

# Делаем предсказание для валидационной выборки
y_test_gb_pred = gradient_boosting.predict(X_test)

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_gb_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_gb_pred_proba),3))
print('ROC AUC на тестовом наборе', round(metrics.roc_auc_score(y_test, y_test_gb_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_test,y_test_gb_pred,zero_division=0))

In [ ]:
# Для расчета значений TPR и FPR используем функцию roc_curve
fpr_train, tpr_train, _ = metrics.roc_curve(y_train, y_train_gb_pred_proba)
fpr_valid, tpr_valid, _ = metrics.roc_curve(y_valid, y_valid_gb_pred_proba)
fpr_test, tpr_test, _ = metrics.roc_curve(y_test, y_test_gb_pred_proba)

# рассчитаем площадь под кривой
auc_train = round(metrics.roc_auc_score(y_train, y_train_gb_pred_proba),3)
auc_valid = round(metrics.roc_auc_score(y_valid, y_valid_gb_pred_proba),3)
auc_test = round(metrics.roc_auc_score(y_test, y_test_gb_pred_proba),3)

trace_train = go.Scatter(x=fpr_train, y=tpr_train, mode='lines', name='AUC_train = %0.2f' % auc_train,
                   line=dict(color='red', width=2),
                   hovertemplate='train<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_train])

trace_valid = go.Scatter(x=fpr_valid, y=tpr_valid, mode='lines', name='AUC_valid = %0.2f' % auc_valid,
                   line=dict(color='green', width=2),
                   hovertemplate='valid<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_valid])

trace_test = go.Scatter(x=fpr_test, y=tpr_test, mode='lines', name='AUC_test = %0.2f' % auc_test,
                   line=dict(color='darkorange', width=2),
                   hovertemplate='test<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_test])


reference_line = go.Scatter(x=[0,1], y=[0,1], mode='lines', name='Reference Line',
                            line=dict(color='navy', width=2, dash='dash'))

fig_roc = go.Figure(data=[trace_train,trace_valid,trace_test,reference_line])

fig_roc.update_layout(
    title ={
        'text':'ROC-кривая', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 1350,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate')

fig_roc.update_xaxes(showspikes=True)
fig_roc.update_yaxes(showspikes=True)


fig_roc.show()

<span style="color:Blue">

Вывод по ROC-кривой:
1. Модель выдает сталибильное качество на валидационном и тестовом наборах, а значит  
не переобучена - разброс ($variance$) не большой. Хотя, качество модели на обучающем наборе,  
заметно лучше.
2. Смещение ($bias$) модели чуть лучше, чем у модели $Random$ $Forest$ $Classifier$,  
обученной на тех же данных.    
На тестовом наборе полученно качество $ROC AUC = 0.63$, против $ROC AUC = 0.61$.

### <span style="color:RoyalBlue">Построение модели на сбалансированных данных подпрострнства credit stat</span>

#### <span style="color:MediumBlue">Формирование данных для обучения</span>

Анализ исходных данных, в частности target, показал что представленные данные   
имееют перекос: 96% клиентов у которых не случился дефолт (flag = 0),    
против 4 % - для которых произошел дефолт (flag = 1).  
С помощью функции class_1_percent_samples, сформируем сбаланисрованную выборку  
для обучения моделей.
За основу сбалансировных данных возьмем данные клиентов с дефолтом  
и к ним будем добирать необходимое количество клиентов до сбалансированности. 


Функция class_1_percent_samples принемает две переменные:
- target_data - массив с id и целевой переменной
- class_1_percent - процент класса 1 в результирующем массиве

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'credit'
count_features = dict_spaces[feature_space]

In [ ]:
# сформируем данные для анализа сбалансированности обучающих данных
X_train_pd = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()
y_train_pd = pd.read_csv('target/target_train.csv')

In [ ]:
# поготовим данные y_train_pd к работе с функцией class_1_percent_samples
y_train_pd.set_index('id',drop=False,inplace=True)
y_train_pd.head(4)

In [ ]:
# взглянем на сблансиорованность данных в y_train
y_train_pd['flag'].value_counts(normalize=True)

In [ ]:
# сбалансируем данные в y_train_pd
list_c1_percent_id = class_1_percent_samples(y_train_pd,0.5)
# list_c1_percent_features - список 'id' из y_train_pd соотвествующих  
# заданному проценту класса 1
len(list_c1_percent_id)

In [ ]:
# проверим сбалансированность полученного y_train
y_train_pd.loc[list_c1_percent_id]['flag'].value_counts(normalize=True)

В отличие от данных в *transform_data_torow*, где модели показываеются последние N  
операций клиента, в данные в *transform_data_stat* сфомированы из различных статистических   
характеристик ряда определяющего клиентскую историю в банке. 

Соотвественно, если в данных *transform_data_torow* была сильная корреляция между признаками,  
то я не смог бы ее убрать, иначе потярется смысл "показать последовательный ряд в историии клиента".  
Признаки в *transform_data_stat* не имеют такой особенности. Поэтому, необходимо проанализировать  
данные в *transform_data_stat* и, в случае сильной коррреляции признаков, устранить ее.

In [ ]:
# подгрузим DataFrame с матрицей корреляции признаков
corr_matrix = fp.ParquetFile('base_models/data/stat/corr_matrix_'+feature_space).to_pandas()
corr_matrix.shape

In [ ]:
# с помощью функции corr_transrom_to_power получим
# силу корреляции матрицы и список не коррелируемых признаков

# для начального анализа зададим порог силы корреляции, при котором признаки 
# будут отнесены к "сильно скоррелированными" равным 0.6
ncorr_matrix,list_ncorr_features,corr_power =corr_transform_to_force(corr_matrix,threshold=0.6)

# Выведем результаты преобразования
print('Исходное количество признаков: ', corr_matrix.shape[0])
print('Количество не коррелируемых признаков: ', len(list_ncorr_features))
# сила корреляции признаков
print('Отношение коррелируемых исходному количеству признаков (сила корреляции): ',corr_power)

In [ ]:
# подготовим данные для обучения
X_train_balanced = X_train_pd.loc[list_c1_percent_id]
y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag']
# сформируем данные для проверики модели
# сформируем данные для анализа сбалансированности обучающих данных
X_train = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()
y_train = pd.read_csv('target/target_train.csv')['flag']
X_valid = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_valid').to_pandas()
y_valid = pd.read_csv('target/target_valid.csv')['flag']

In [ ]:
# посмотрим на структуру данных в X_train_balanced
X_train_balanced.head(4)

In [ ]:
# посмотрим на структуру данных в y_train_balanced
y_train_balanced.head(4)

In [ ]:
# проверим что выборки действительно совпадают по id
X_train_balanced.index.to_list() == y_train_balanced.index.to_list()

#### <span style="color:MediumBlue">Построение модели Logistic Regression</span>

##### <span style="color:MediumSlateBlue">Baseline обучение модели LogisticRegression</span>

In [ ]:
# обучаем модель логистической регрессии
logistic_regression = linear_model.LogisticRegression(random_state=42, max_iter=10000)
logistic_regression.fit(X_train_balanced[list_ncorr_features],y_train_balanced)

# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_LR_pred_proba = logistic_regression.predict_proba(X_train[list_ncorr_features])[:,1]
y_valid_LR_pred_proba = logistic_regression.predict_proba(X_valid[list_ncorr_features])[:,1]

# Делаем предсказание для валидационной выборки
y_valid_LR_pred = logistic_regression.predict(X_valid[list_ncorr_features])

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_LR_pred_proba),3))
print('ROC AUC на тестовом наборе', round(metrics.roc_auc_score(y_valid, y_valid_LR_pred_proba),3))

print('Основные метрики на тестовом наборе:')
print(metrics.classification_report(y_valid,y_valid_LR_pred))

# time: 3m 5s

##### <span style="color:MediumSlateBlue">Анализ влияния scaler преобразования на качество и скорость схождения модели</span>

In [ ]:
# формируем список из необходимых scaler
scalers = [MinMaxScaler(),RobustScaler() ,StandardScaler()]

# сформируем списко для заполнения данными результатов анализа
scalers_data = []

# установим ограничение на время обучения модели в 600 секунд (10 минут)
time_out = 600

for scaler in scalers:
        # для того чтобы следить за выполнением цикла введем индикатор
        print('current scaler: ', scaler)

         # сформируем список для заполнения данными текушего scaler
        current_scaler_data = [scaler]
        
        # выполним scaler преобразование
        X_train_s_balanced = scaler.fit_transform(X_train_balanced[list_ncorr_features])
        X_train_s = scaler.transform(X_train[list_ncorr_features])
        X_valid_s = scaler.transform(X_valid[list_ncorr_features])

        try:
                # для модуля func_time_out упакуем обучение модели в функцию try_func
                def try_func():
                        logistic_regression = linear_model.LogisticRegression(random_state=42, max_iter=10000)
                        return logistic_regression.fit(X_train_s_balanced,y_train_balanced)
                    
                # фиксируем время начала
                start_time = time.time()
                
                # запускаем обучение модели с ограничением по времени
                logistic_regression = func_timeout.func_timeout(time_out, try_func)
                
                # фиксируем время окончания обучения
                end_time = time.time()
                
                # запишем время обучения модели
                current_scaler_data.append(round(end_time-start_time))

                # Делаем предсказание для обучающей и валидационной выборки
                y_valid_lr_pred = logistic_regression.predict(X_valid_s)
                
                # для метрик ROC AUC делаем предсказание модели для обучающей и валидационной выборки  
                # в виде вероятности принадлежности к классу 1 
                y_train_lr_pred_proba = logistic_regression.predict_proba(X_train_s)[:,1]
                y_valid_lr_pred_proba = logistic_regression.predict_proba(X_valid_s)[:,1]

                #Считаем метрики ROC AUC yна обуающем и валидационном наборе и добавляем их в спискок
                current_scaler_data.append(round(metrics.roc_auc_score(y_train, y_train_lr_pred_proba),3))
                current_scaler_data.append(round(metrics.roc_auc_score(y_valid, y_valid_lr_pred_proba),3))

                #Считаем метрики для класса 0 на валидационном наборе и добавляем их в список
                current_scaler_data.append(round(metrics.recall_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))
                current_scaler_data.append(round(metrics.precision_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))


                #Считаем метрики для класса 1 на валидационном набопе и добавляем их в список
                current_scaler_data.append(round(metrics.recall_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))
                current_scaler_data.append(round(metrics.precision_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))

                # добавляем полученные данные в список scalers_data
                scalers_data.append(current_scaler_data)

                # чтобы подстраховаться на случай ошибки : "The Kernel crashed while executing code"
                # сохраним в локальную папку удачную итерацию
                # для этого из полученных данных сформируем dataframe
                # из полученных данных сформируем dataframe
                scaler_pd =pd.DataFrame(
                        columns=['scaler','time_fit',
                                 'roc_train','roc_valid', 
                                 'recall_0','precision_0',
                                 'recall_1','precision_1'],
                        data = scalers_data)
                # сохраняем dataframe
                scaler_pd.to_csv('data_recovery/scaler_pd.csv',index=False)
               
                # очистим output от прошедшей итерации
                clear_output()

        except func_timeout.FunctionTimedOut:
                # если время обучения привышает лимит
                # запишем в current_scaler_data соотвествующую данные
                current_scaler_data = [scaler,'>30m',0,0,0,0,0,0]
                # добавляем полученные данные в список scalers_data
                scalers_data.append(current_scaler_data)

                # чтобы подстраховаться на случай ошибки : "The Kernel crashed while executing code"
                # сохраним в локальную папку удачную итерацию
                # для этого из полученных данных сформируем dataframe
                # из полученных данных сформируем dataframe
                scaler_pd =pd.DataFrame(
                        columns=['scaler','time_fit',
                                 'roc_train','roc_valid', 
                                 'recall_0','precision_0',
                                 'recall_1','precision_1'],
                        data = scalers_data)
                # сохраняем dataframe
                scaler_pd.to_csv('data_recovery/scaler_pd.csv',index=False)
                
                # очистим output от прошедшей итерации
                clear_output()
                # переходим к следующему scaler
                pass
# очистим output
clear_output()

# сформируем данные соотвествующие лучщему ROC AUC на валидацинном наборе
best_roc_valid_data = scaler_pd[scaler_pd['roc_valid']==scaler_pd['roc_valid'].max()]

# выведем лучший результат на валидационном наборе
print('result:')
print('best scaler on valid:',best_roc_valid_data.iloc[0]['scaler'])
print('ROC AUC on train:', best_roc_valid_data.iloc[0]['roc_train'])
print('ROC AUC on valid:', best_roc_valid_data.iloc[0]['roc_valid'])
print('Time fit for best scaler:', best_roc_valid_data.iloc[0]['time_fit'],' seconds')

# выведем полученный dataframe
scaler_pd

# time: 

##### <span style="color:MediumSlateBlue"> Анализ влияния solver оптимизатора на качество и скорость обучения модели</span>

Проверим качество и скорость сходимости модели при разных solver  
Проверку осуществим через цикл  
Чтобы модель не сходилась бесконечно с помощью модуля func_timeout  
поставим ограничение времени схождения в 10 минут

In [ ]:
# подготовим данные для обучения модели

# обьявим scaler
scaler = StandardScaler()
# выполним scaler преобразование
X_train_s_balanced = scaler.fit_transform(X_train_balanced[list_ncorr_features])
X_train_s = scaler.transform(X_train[list_ncorr_features])
X_valid_s = scaler.transform(X_valid[list_ncorr_features])

# time: 10s

In [ ]:
# Зададим список solvers
solvers_list = ['lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga']

# сформируем список для заполнения
solvers_data = []

# установим ограничение на время обучения модели в 180 секунд (3 минут)
time_out = 180

for solver in solvers_list:
        # для того чтобы следить за выполнением цикла введем индикатор
        print('current solver: ', solver)

         # сформируем список для заполнения данными текушего solver
        current_solver_data = [solver]
        
        try:
                # для модуля func_time_out упакуем обучение модели в функцию try_func
                def try_func():
                        logistic_regression = linear_model.LogisticRegression(solver= solver,
                                                                  random_state=42, 
                                                                  max_iter=10000)
                        return logistic_regression.fit(X_train_s_balanced,y_train_balanced)
                    
                # фиксируем время начала
                start_time = time.time()
                
                # запускаем обучение модели с ограничением по времени
                logistic_regression = func_timeout.func_timeout(time_out, try_func)
                
                # фиксируем время окончания обучения
                end_time = time.time()
                
                # запишем время обучения модели
                current_solver_data.append(round(end_time-start_time))

                # Делаем предсказание для обучающей и валидационной выборки
                y_valid_lr_pred = logistic_regression.predict(X_valid_s)
                
                # для метрик ROC AUC делаем предсказание модели для обучающей и валидационной выборки  
                # в виде вероятности принадлежности к классу 1 
                y_train_lr_pred_proba = logistic_regression.predict_proba(X_train_s)[:,1]
                y_valid_lr_pred_proba = logistic_regression.predict_proba(X_valid_s)[:,1]

                #Считаем метрики ROC AUC yна обуающем и валидационном наборе и добавляем их в спискок
                current_solver_data.append(round(metrics.roc_auc_score(y_train, y_train_lr_pred_proba),3))
                current_solver_data.append(round(metrics.roc_auc_score(y_valid, y_valid_lr_pred_proba),3))

                #Считаем метрики для класса 0 на валидационном наборе и добавляем их в список
                current_solver_data.append(round(metrics.recall_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))
                current_solver_data.append(round(metrics.precision_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))


                #Считаем метрики для класса 1 на валидационном набопе и добавляем их в список
                current_solver_data.append(round(metrics.recall_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))
                current_solver_data.append(round(metrics.precision_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))

                # запишем данные полученные на одной итерации
                solvers_data.append(current_solver_data)

                # чтобы подстраховаться на случай ошибки : "The Kernel crashed while executing code"
                # сохраним в локальную папку удачную итерацию
                # для этого из полученных данных сформируем dataframe
                # из полученных данных сформируем dataframe
                solvers_pd =pd.DataFrame(
                        columns=['solver','time_fit',
                                 'roc_train','roc_valid', 
                                 'recall_0','precision_0',
                                 'recall_1','precision_1'],
                        data = solvers_data)
                # сохраняем dataframe
                solvers_pd.to_csv('data_recovery/solvers_pd.csv',index=False)
               
                # очистим output от прошедшей итерации
                clear_output()

        except func_timeout.FunctionTimedOut:
                # если время обучения привышает лимит
                # запишем в current_solver_data соотвествующую данные
                current_solver_data = [solver,'>3m',0,0,0,0,0,0]
                # добавляем полученные данные в список solvers_data
                solvers_data.append(current_solver_data)

                # чтобы подстраховаться на случай ошибки : "The Kernel crashed while executing code"
                # сохраним в локальную папку удачную итерацию
                # для этого из полученных данных сформируем dataframe
                # из полученных данных сформируем dataframe
                solvers_pd =pd.DataFrame(
                        columns=['solver','time_fit',
                                 'roc_train','roc_valid', 
                                 'recall_0','precision_0',
                                 'recall_1','precision_1'],
                        data = solvers_data)
                # сохраняем dataframe
                solvers_pd.to_csv('data_recovery/solvers_pd.csv',index=False)
                
                # очистим output от прошедшей итерации
                clear_output()
                # переходим к следующему solver
                pass
# очистим output
clear_output()

# сформируем данные соотвествующие лучщему ROC AUC на валидацинном наборе
best_roc_valid_data = solvers_pd[solvers_pd['roc_valid']==solvers_pd['roc_valid'].max()]

# выведем лучший результат на валидационном наборе
print('result:')
print('best solver on valid:',best_roc_valid_data.iloc[0]['solver'])
print('ROC AUC on train:', best_roc_valid_data.iloc[0]['roc_train'])
print('ROC AUC on valid:', best_roc_valid_data.iloc[0]['roc_valid'])
print('Time fit for best solver:', best_roc_valid_data.iloc[0]['time_fit'],' seconds')

# выведем полученный dataframe
solvers_pd

# time: 2m 35s

##### <span style="color:MediumSlateBlue">Подбор гиперпараметров модели</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'credit'
count_features = dict_spaces[feature_space]

# для работы функции corr_transform_to_force
# подгрузим DataFrame с матрицей корреляции признаков
corr_matrix = fp.ParquetFile('base_models/data/stat/corr_matrix_'+feature_space).to_pandas()

# для выбора sceler преобразвания нам понадобится словарь
dict_scalers ={
    'MinMaxScaler': MinMaxScaler(),
    'RobustScaler':RobustScaler(),
    'StandardScaler':StandardScaler()}

# настроим оптимизацию гипер параметров
def optuna_lg(trial):
  # задаем пространства поиска гиперпараметров
  solver = trial.suggest_categorical("solver", ['lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky'])
  C = trial.suggest_float('C',0.01,1)
  class_0_weight = trial.suggest_float('class_0_weight',0.2,0.6)
  class_1_weight = trial.suggest_float('class_1_weight',0.2,0.6)
  threshold = trial.suggest_float('threshold',0.6,1)
  class_1_percent = trial.suggest_float('class_1_percent',0.01,0.3)
  scaler = trial.suggest_categorical("scaler", ['MinMaxScaler','RobustScaler','StandardScaler'])
  random_state = trial.suggest_int('random_state', 1, 1000000)


  # создаем модель
  optuna_log_reg = linear_model.LogisticRegression(
      solver=solver,
      C=C,
      class_weight={0:class_0_weight,1:class_1_weight},
      random_state=random_state,
      max_iter=10000)

  # с помощью функции corr_transform_to_force получим
  # список не коррелируемых признаков
  list_ncorr_features = corr_transform_to_force(corr_matrix,threshold=threshold)[1]

  # обьявим scaler
  scaler = dict_scalers[scaler]

  # сформируем данные для анализа сбалансированности обучающих данных
  X_train_pd = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
  y_train_pd = pd.read_csv('target/target_train.csv')

  # поготовим данные y_train_pd к работе с функцией class_1_percent_samples
  y_train_pd.set_index('id',drop=False,inplace=True)

  # подготовим данные для обучения модели
  # с помощью функции class_1_percent_samples зададим долю 
  # класса 1
  list_c1_percent_id = class_1_percent_samples(y_train_pd,class_1_percent,random_state=random_state)[:1000000]

  # подготовим данные для обучения
  X_train_s_balanced = scaler.fit_transform(X_train_pd.loc[list_c1_percent_id])
  y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

  # освободим память от "тяжелых" и ненужных файлов
  del X_train_pd, y_train_pd
  gc.collect()

  # я не воспользовался параметром timeout метода optimize потому, что  
  # optimize отставливает поиск параметров, если время обучения 
  # превышает timeout. Мне нужно чтобы поиск параметров продолжился.
  try:
    # для модуля func_time_out упакуем обучение модели в функцию try_func
    def try_func():
            return optuna_log_reg.fit(X_train_s_balanced, y_train_balanced)
    # обучаем модель с ограничением по времени 10 минут (600 секунд)
    optuna_log_reg = func_timeout.func_timeout(600, try_func)

    # удаляем крупные файлы чтобы высвободить память 
    del X_train_s_balanced, y_train_balanced
    gc.collect()

    # загружаем тестовые наборы
    # сформируем данные для анализа сбалансированности обучающих данных
    X_train = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
    X_valid = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_valid').to_pandas()[list_ncorr_features]
    y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
    y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()

    # подготовим данные для проверки модели
    X_train_s = scaler.transform(X_train)
    X_valid_s = scaler.transform(X_valid)

    # удаляем крупные файлы чтобы высвободить память 
    del X_train, X_valid
    gc.collect()

    # делаем предсказание на обучающем и валидационном наборе и считаем метрики
    roc_train = metrics.roc_auc_score(y_train, optuna_log_reg.predict_proba(X_train_s)[:,1])
    roc_valid = metrics.roc_auc_score(y_valid, optuna_log_reg.predict_proba(X_valid_s)[:,1])
    f1_score = metrics.f1_score(y_valid, optuna_log_reg.predict(X_valid_s))
    recall_1 = metrics.recall_score(y_valid, optuna_log_reg.predict(X_valid_s),average=None,zero_division=0)[1]
    precision_1 = metrics.precision_score(y_valid, optuna_log_reg.predict(X_valid_s),average=None,zero_division=0)[1]

    # удаляем крупные файлы чтобы высвободить память 
    del X_train_s, X_valid_s, y_train, y_valid
    gc.collect()

  except func_timeout.FunctionTimedOut:
    # удаляем крупные файлы чтобы высвободить память 
    del X_train_s_balanced, y_train_balanced
    gc.collect()
    
    # фиксируем пустые значения метрик
    roc_train = 0
    roc_valid = 0
    f1_score = 0
    recall_1 = 0
    precision_1 = 0
    pass

  return roc_train, roc_valid, f1_score, recall_1, precision_1

In [ ]:
# cоздаем объект исследования
# чтобы модель не переобучалась минимизируем roc_train 
# и максимизируем roc_valid
optuna_study_lg = optuna.create_study(study_name='LogisticRegression_'+feature_space+'_stat', 
                               directions=['maximize','maximize','maximize','maximize','maximize'], 
                               sampler=optuna.samplers.TPESampler(),
                               pruner='Hyperband',
                               storage='sqlite:///optuna_studies.db',
                               load_if_exists=True)
# optuna.delete_study(study_name='LogisticRegression_'+feature_space+'_stat', storage='sqlite:///optuna_studies.db')

In [ ]:
# ищем лучшую комбинацию гиперпараметров n_trials раз
optuna_study_lg.optimize(optuna_lg, n_trials=300)

In [ ]:
# из полученного результат соврмируем Data Frame
optuna_study_lg_pd = optuna_study_lg.trials_dataframe().dropna()
# переименуем столбы
optuna_study_lg_pd.rename(columns={
    'values_0': 'roc_train',
    'values_1': 'roc_valid',
    'values_2': 'f1_score',
    'values_3': 'recall_1',
    'values_4': 'precision_1'
},inplace=True)
# Выделим время потраченное на обучение модели
optuna_study_lg_pd.tail()

In [ ]:
# покаждем статистику обучения
print('Максимальное значение метрики ROC AUC на валидационном наборе:',round(optuna_study_lg_pd['roc_valid'].max(),3))
print('Среднее значение метрики ROC AUC на валидационном наборе:',round(optuna_study_lg_pd['roc_valid'].mean(),3))
print('Максимальное значение метрики f1_score на валидационном наборе:',round(optuna_study_lg_pd['f1_score'].max(),3))
print('Среднее значение метрики f1_score на валидационном наборе:',round(optuna_study_lg_pd['f1_score'].mean(),3))
print('Максимальное значение метрики recall_1 на валидационном наборе:',round(optuna_study_lg_pd['recall_1'].max(),3))
print('Среднее значение метрики recall_1 на валидационном наборе:',round(optuna_study_lg_pd['recall_1'].mean(),3))
print('Максимальное значение метрики precision_1 на валидационном наборе:',round(optuna_study_lg_pd['precision_1'].max(),3))
print('Среднее значение метрики precision_1 на валидационном наборе:',round(optuna_study_lg_pd['precision_1'].mean(),3))

In [ ]:
# построим график важности гиперпарметров
optuna.visualization.plot_param_importances(optuna_study_lg, target_name='Score')

In [ ]:
# Построим зависимость метрики precision_1
fig = px.scatter(
    data_frame=optuna_study_lg_pd,
    x='precision_1', #ось абсцисс
    y=['f1_score', 'recall_1'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision от recall на классе 1', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision',
    yaxis_title='Величина метрик recall и f1-score')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики precision_1 от гипер параметров

fig = px.scatter(
    data_frame=optuna_study_lg_pd,
    x='precision_1', #ось абсцисс
    y=['params_C', 'params_class_0_weight', 
       'params_class_1_weight','params_class_1_percent',
       'params_threshold'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики f1_score от гипер параметров

fig = px.scatter(
    data_frame=optuna_study_lg_pd,
    x='f1_score', #ось абсцисс
    y=['params_C', 'params_class_0_weight', 
       'params_class_1_weight','params_class_1_percent',
       'params_threshold'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость f1-score от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики f1-score',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики roc_valid от гипер параметров

fig = px.scatter(
    data_frame=optuna_study_lg_pd,
    x='roc_valid', #ось абсцисс
    y=['params_C', 'params_class_0_weight', 
       'params_class_1_weight','params_class_1_percent',
       'params_threshold'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость ROC AUC на валидационном наборе от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики ROC AUC',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Визуализируем метрики полученные при optuna оптимизации
fig = px.scatter(
    data_frame=optuna_study_lg_pd[(optuna_study_lg_pd['precision_1']>0.08)
                                  & (optuna_study_lg_pd['recall_1']>0.08) 
                                  & (optuna_study_lg_pd['roc_valid']>0.8*optuna_study_lg_pd['roc_valid'].max())],
    x='number', #ось абсцисс
    y=['roc_train','roc_valid','recall_1','precision_1'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость качества модели от trials optuna', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='trials optuna',
    yaxis_title='Величина метрики')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

<span style="color:Blue">

Вывод:  

Их всех выбираем точку $number =234$.   
В этой точке самые оптимальные значения $recall$ и $precision$.  
Посмотрим каким значениям гиперпараметров соотвествует выбранная точка.

In [ ]:
# определим номер лучшге варианта
best_optuna_number = 234

# сформируем словарь лучших гипирпарметров RandomForestClassifier
best_param_lr = {
    'solver' : optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_solver'].iloc[0],
    'C' : round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_C'].iloc[0],3),
    'class_weight' : {0:round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_class_0_weight'].iloc[0],3),
                      1:round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_class_1_weight'].iloc[0],3)},
    }

# создадим перменные
best_threshold = optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_threshold'].iloc[0]
best_scaler = optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_scaler'].iloc[0]
best_class_1_percent = optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_class_1_percent'].iloc[0]
best_random_state = optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_random_state'].iloc[0]

# Выведем принятые наилучшие праметры
print('best solver:',best_param_lr['solver'])
print('best C:',best_param_lr['C'])
print('best class 0 weight:',best_param_lr['class_weight'][0])
print('best class 1 weight:',best_param_lr['class_weight'][1])

print('best class 1 percent:',round(best_class_1_percent,3))
print('best scaler:',best_scaler)
print('best threshold:',round(best_threshold,3))
print('best best random state:',best_random_state)
print('time for best train:',round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['duration'].iloc[0].seconds/60),'minutes')

print()
print('ROC AUC на обучающем наборе:', round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['roc_train'].iloc[0],3))
print('ROC AUC на валидационном наборе:', round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['roc_valid'].iloc[0],3))
print('precision класса 1:', round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['precision_1'].iloc[0],3))
print('recall класса 1:', round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['recall_1'].iloc[0],3))


##### <span style="color:MediumSlateBlue">Обучение модели с лучшими параметрами</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'credit'
count_features = dict_spaces[feature_space]

# для работы функции corr_transform_to_force
# подгрузим DataFrame с матрицей корреляции признаков
corr_matrix = fp.ParquetFile('base_models/data/stat/corr_matrix_'+feature_space).to_pandas()

# подготовим данные
# для выбора sceler преобразвания нам понадобится словарь
dict_scalers ={
    'MinMaxScaler': MinMaxScaler(),
    'RobustScaler':RobustScaler(),
    'StandardScaler':StandardScaler(),
}

# с помощью функции corr_transform_to_force получим
# список не коррелируемых признаков
list_ncorr_features = corr_transform_to_force(corr_matrix,threshold=best_threshold)[1]

# обьявим scaler
scaler = dict_scalers[best_scaler]

# сформируем данные для анализа сбалансированности обучающих данных
X_train_pd = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
y_train_pd = pd.read_csv('target/target_train.csv')

# поготовим данные y_train_pd к работе с функцией class_1_percent_samples
y_train_pd.set_index('id',drop=False,inplace=True)

# подготовим данные для обучения модели
# с помощью функции class_1_percent_samples зададим долю 
# класса 1
list_c1_percent_id = class_1_percent_samples(y_train_pd,best_class_1_percent,random_state=best_random_state)[:1000000]

# подготовим данные для обучения
X_train_s_balanced = scaler.fit_transform(X_train_pd.loc[list_c1_percent_id])
y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

# освободим память от "тяжелых" и ненужных файлов
del X_train_pd, y_train_pd
gc.collect()


# обучаем модель LogisticRegression с наилучшеми параметрами
logistic_regression = linear_model.LogisticRegression(
        **best_param_lr,
        random_state=42,
        max_iter=10000)
logistic_regression.fit(X_train_s_balanced, y_train_balanced)

# удаляем крупные файлы чтобы высвободить память 
del X_train_s_balanced, y_train_balanced
gc.collect()

# загружаем тестовые наборы
# сформируем данные для анализа сбалансированности обучающих данных
X_train = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
X_valid = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_valid').to_pandas()[list_ncorr_features]
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()

# подготовим данные для проверки модели
X_train_s = scaler.transform(X_train)
X_valid_s = scaler.transform(X_valid)

# удаляем крупные файлы чтобы высвободить память 
del X_train, X_valid
gc.collect()

# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_lr_pred_proba = logistic_regression.predict_proba(X_train_s)[:,1]
y_valid_lr_pred_proba = logistic_regression.predict_proba(X_valid_s)[:,1]

# Делаем предсказание для валидационной выборки
y_valid_lr_pred = logistic_regression.predict(X_valid_s)

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_lr_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_lr_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_valid,y_valid_lr_pred,zero_division=0))

# time: 2m 30s

##### <span style="color:MediumSlateBlue">Анализ влияния порога классификации на качество модели</span>

In [ ]:
# чтобы проше обращаться к каждому обьекту в данных
# преобразуем y_valid_LR_pred к Series типу
y_valid_lr_pred_proba = pd.Series(y_valid_lr_pred_proba)

# формируем список из необходимых значений thresholds
thresholds = np.arange(0.01, 1, 0.01)

# сформируем списко для заполнения данными результатов анализа
threshold_data = []


for threshold in thresholds:
        # сформируем список для заполнения данными текушего threshold
        threshold_current_data = [threshold]

        #клиентов, для которых вероятность дефолта > threshold, относим к классу 1
        #в противном случае — к классу 0
        y_valid_lr_pred = y_valid_lr_pred_proba.apply(lambda x: 1 if x>threshold else 0)


        #Считаем метрики для класса 0 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.precision_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.f1_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))

        #Считаем метрики для класса 1 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.precision_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.f1_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))

        # добавляем полученные данные в список threshold_data
        threshold_data.append(threshold_current_data)

        # из полученных данных сформируем dataframe
        thresholds_pd =pd.DataFrame(
        columns=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'],
        data = threshold_data)

# time: 17s

In [ ]:
# Визуализируем метрики на валидационном наборе при различных threshold
fig_threshold = px.line(
    data_frame=thresholds_pd,
    x='threshold', #ось абсцисс
    y=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'], #ось ординат
)
fig_threshold.update_layout(
    title ={
        'text':'Зависимость качества модели от threshold', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Порог классификации threshold',
    yaxis_title='Величина метрики')
fig_threshold.update_xaxes(showspikes=True)
fig_threshold.update_yaxes(showspikes=True)

fig_threshold.show()

Метрика $precision$ показывает на сколько сильно ошиблась модель при определиние класса.  
Чем выше метрика, тем меньше ошиблась модель. 

$$precision = \frac{TP}{TP+FP}$$

Метрика $recall$ показывает способность модели найти класс.  
Чем выше метрика, тем выше способность модели находить класс.  

$$recall = \frac{TP}{TP+FN}$$

Метрика $f1-score$ показывает баланс между $precision$ и $recall$ 

$threshold$- порог класификации модели.  
Все объекты для которых $у_{\text{proba pred}} < threshold$, модель относит к класс 0.  
Соотвественно, если $у_{\text{proba pred}}  > threshold$ модель относит обьект к классу 1.  
Для идельаной модели $threshold$, является границей между областями класса 0 и класса 1.  

<span style="color:Blue">

Выводы по результам анализа:  

Для улучшения качества модели по метрике $ROC AUC$, нет смысла сдвигать порог   
классификации. Но зависимость метрик качества модели от $threshold$, может рассказать о  
спосбоности модели искать класс 1 и класс 0.

При фокусировке модели на классе 1 (более низкий $threshold$):  
- на классе 1 метрика $precision$ уменьшается, метрика $recall$ растет;  
- на классе 0 метрика $precision$ растет, метрика $recall$ уменьшается.  

При фокусировке модели на классе 0 (более высокий $threshold$):  
- на классе 1 метрика $precision$ увеличивается, метрика $recall$ уменьшается;  
- на классе 0 метрика $precision$ уменьшается, метрика $recall$ растет.  

Обьсняется тем, что в условия дисбаланса модель научилась хорошо определять класс 0 и плохо класс 1.  

При фокусировке модели на классе 1 (более низкий $threshold$), все больше обьектов отнесены к классу 1.   
Так как $precision$ класса 1 уменьшается, значит в области класса 1 стало больше обектов класса 0,  
что явлется закономерным для "идеальной" модели.  
Однако при этом, $recall$ класса 1 растет, значит в добавленных обьектах также присутвует класс 1.  
Это как раз указывает на то, что модель не научилась находить среди класса 0 класс 1. В области с     
высокой вероятностью класса 0 (более низкий $threshold$), находятся обьекты класса 1.  

Фокусировка на классе 0 (более высокий $threshold$), заставляет модель все больше обьектов из области класса 1  
относить к классу 0. При этом увеличение метрики $precision$ класса 1 говорит о том, что в области класса 1     
становится меньше 0, а значит модель умеет среди класса 1, находить класс 0.  
После  $threshold=0.64$ модель впринципе теряет способность отличать класс 1. У модели нет обьектов класса 1   
в которых она "уверена" на 80+%.

Обратим внимание на $threshold=0.52$, в этой точке находится баланс межу метриками качества модели:
$$precision_1 = recall_1 = \text{f1-score}_1 = 0.082$$
$$precision_0 = recall_0 = \text{f1-score}_0 = 0.965$$

Для модели с лучшим качеством эти две точки должны находится максимально близко к друг другу и иметь  
высокое значение метрик $precision_1$ и $recall_1$.
Для класса 0 это условие выполнется, что также говорит о том, что модель научилась искать этот класс.  
Для класса 1 это условие не выполнется, что также говорит о том, что модель плохо ищет класс 1.

##### <span style="color:MediumSlateBlue">Построение ROC-кривой выбранной модели на тестовом наборе</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'credit'
count_features = dict_spaces[feature_space]

In [ ]:
# сформируем данные для проверки модели
X_train = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
X_valid = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_valid').to_pandas()[list_ncorr_features]
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()
X_test = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_test').to_pandas()[list_ncorr_features]
y_test = pd.read_csv('target/target_test.csv')['flag'].to_numpy()

In [ ]:
# выполним scaler преобразование
X_train_s = scaler.transform(X_train)
X_valid_s = scaler.transform(X_valid)
X_test_s = scaler.transform(X_test)

print("Размеры обучающего набора",X_train_s.shape)
print("Размеры валидационного набора",X_valid_s.shape)
print("Размеры тестового набора",X_test_s.shape)
# time: 1m 43s

In [ ]:
# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_lr_pred_proba = logistic_regression.predict_proba(X_train_s)[:,1]
y_valid_lr_pred_proba = logistic_regression.predict_proba(X_valid_s)[:,1]
y_test_lr_pred_proba = logistic_regression.predict_proba(X_test_s)[:,1]

# Делаем предсказание для валидационной выборки
y_test_lr_pred = logistic_regression.predict(X_test_s)

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_lr_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_lr_pred_proba),3))
print('ROC AUC на тестовом наборе', round(metrics.roc_auc_score(y_test, y_test_lr_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_test,y_test_lr_pred,zero_division=0))

In [ ]:
# Для расчета значений TPR и FPR используем функцию roc_curve
fpr_train, tpr_train, _ = metrics.roc_curve(y_train, y_train_lr_pred_proba)
fpr_valid, tpr_valid, _ = metrics.roc_curve(y_valid, y_valid_lr_pred_proba)
fpr_test, tpr_test, _ = metrics.roc_curve(y_test, y_test_lr_pred_proba)

# рассчитаем площадь под кривой
auc_train = round(metrics.roc_auc_score(y_train, y_train_lr_pred_proba),3)
auc_valid = round(metrics.roc_auc_score(y_valid, y_valid_lr_pred_proba),3)
auc_test = round(metrics.roc_auc_score(y_test, y_test_lr_pred_proba),3)

trace_train = go.Scatter(x=fpr_train, y=tpr_train, mode='lines', name='AUC_train = %0.2f' % auc_train,
                   line=dict(color='red', width=2),
                   hovertemplate='train<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_train])

trace_valid = go.Scatter(x=fpr_valid, y=tpr_valid, mode='lines', name='AUC_valid = %0.2f' % auc_valid,
                   line=dict(color='green', width=2),
                   hovertemplate='valid<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_valid])

trace_test = go.Scatter(x=fpr_test, y=tpr_test, mode='lines', name='AUC_test = %0.2f' % auc_test,
                   line=dict(color='darkorange', width=2),
                   hovertemplate='test<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_test])


reference_line = go.Scatter(x=[0,1], y=[0,1], mode='lines', name='Reference Line',
                            line=dict(color='navy', width=2, dash='dash'))

fig_roc = go.Figure(data=[trace_train,trace_valid,trace_test,reference_line])

fig_roc.update_layout(
    title ={
        'text':'ROC-кривая', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 1350,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate')

fig_roc.update_xaxes(showspikes=True)
fig_roc.update_yaxes(showspikes=True)


fig_roc.show()

<span style="color:Blue">

Вывод по ROC-кривой:
1. Модель выдает сталибильное качество на всех наборах, а значит  
не переобучена - разброс ($variance$) не большой.  
2. Смещение ($bias$) модели относительно большое.  
На тестовом наборе полученно качество $ROC AUC = 0.6$.

#### <span style="color:MediumBlue">Построение модели Random Forest Classifier</span>

##### <span style="color:MediumSlateBlue">Baseline обучение модели RandomForestClassifier</span>

In [ ]:
# обучаем модель логистической регрессии
# чтобы модель пыталась найти связь во всех признаках
random_forest = RandomForestClassifier(random_state=42, n_jobs=-1)
random_forest.fit(X_train_balanced[list_ncorr_features],y_train_balanced)

# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_rf__pred_proba = random_forest.predict_proba(X_train[list_ncorr_features])[:,1]
y_valid_rf_pred_proba = random_forest.predict_proba(X_valid[list_ncorr_features])[:,1]

# Делаем предсказание для валидационной выборки
y_valid_rf_pred = random_forest.predict(X_valid[list_ncorr_features])

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_rf__pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_rf_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_valid,y_valid_rf_pred))

# time: 17s

##### <span style="color:MediumSlateBlue">Подбор гиперпараметров модели</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'credit'
count_features = dict_spaces[feature_space]

# для работы функции corr_transform_to_force
# подгрузим DataFrame с матрицей корреляции признаков
corr_matrix = fp.ParquetFile('base_models/data/stat/corr_matrix_'+feature_space).to_pandas()

# настроим оптимизацию гипер параметров
def optuna_rf(trial):
  # задаем пространства поиска гиперпараметров
  # задаем пространства поиска гиперпараметров
  n_estimators = trial.suggest_int('n_estimators', 40, 100,step = 5)
  criterion = trial.suggest_categorical('criterion',['gini', 'entropy', 'log_loss'])
  max_depth = trial.suggest_int('max_depth', 10, 30,step = 1)
  min_samples_split = trial.suggest_int('min_samples_split', 5, 15,step = 1)
  min_samples_leaf = trial.suggest_int('min_samples_leaf', 5, 15,step = 1)
  max_features = trial.suggest_float('max_features',0.4,0.8)
  max_samples = trial.suggest_float('max_samples',0.4,0.8)
  class_0_weight = trial.suggest_float('class_0_weight',0.1,1)
  class_1_weight = trial.suggest_float('class_1_weight',0.1,1)
  threshold = trial.suggest_float('threshold',0.4,0.8)
  class_1_percent = trial.suggest_float('class_1_percent',0.01,0.5)
  random_state = trial.suggest_int('random_state', 1, 1000000)


  # создаем модель
  optuna_random_forest = RandomForestClassifier(
      n_estimators = n_estimators, 
      criterion = criterion,
      max_depth = max_depth,
      min_samples_split = min_samples_split,  
      min_samples_leaf = min_samples_leaf,
      max_features=max_features,
      max_samples=max_samples,
      class_weight={0:class_0_weight,1:class_1_weight},
      n_jobs=-1,
      random_state=42)

  # с помощью функции corr_transform_to_force получим
  # список не коррелируемых признаков
  list_ncorr_features = corr_transform_to_force(corr_matrix,threshold=threshold)[1]

  # сформируем данные для анализа сбалансированности обучающих данных
  X_train_pd = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
  y_train_pd = pd.read_csv('target/target_train.csv')

  # поготовим данные y_train_pd к работе с функцией class_1_percent_samples
  y_train_pd.set_index('id',drop=False,inplace=True)

  # подготовим данные для обучения модели
  # с помощью функции class_1_percent_samples зададим долю 
  # класса 1
  list_c1_percent_id = class_1_percent_samples(y_train_pd,class_1_percent,random_state=random_state)[:1000000]

  # подготовим данные для обучения
  X_train_balanced = X_train_pd.loc[list_c1_percent_id]
  y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag']

  # освободим память от "тяжелых" и ненужных файлов
  del X_train_pd, y_train_pd
  gc.collect()

  # я не воспользовался параметром timeout метода optimize потому, что  
  # optimize отставливает поиск параметров, если время обучения 
  # превышает timeout. Мне нужно чтобы поиск параметров продолжился.
  try:
    # для модуля func_time_out упакуем обучение модели в функцию try_func
    def try_func():
            return optuna_random_forest.fit(X_train_balanced, y_train_balanced)
    # обучаем модель с ограничением по времени 10 минут (600 секунд)
    optuna_random_forest = func_timeout.func_timeout(600, try_func)

    # удаляем крупные файлы чтобы высвободить память 
    del X_train_balanced, y_train_balanced
    gc.collect()

    # загружаем тестовые наборы
    # сформируем данные для анализа сбалансированности обучающих данных
    X_train = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
    X_valid = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_valid').to_pandas()[list_ncorr_features]
    y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
    y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()
    
    # делаем предсказание на обучающем и валидационном наборе
    #Считаем метрики для класса 1 добавляем их в список
    roc_train = metrics.roc_auc_score(y_train, optuna_random_forest.predict_proba(X_train)[:,1])
    roc_valid = metrics.roc_auc_score(y_valid, optuna_random_forest.predict_proba(X_valid)[:,1])
    f1_score = metrics.f1_score(y_valid, optuna_random_forest.predict(X_valid))
    recall_1 = metrics.recall_score(y_valid, optuna_random_forest.predict(X_valid),average=None,zero_division=0)[1]
    precision_1 = metrics.precision_score(y_valid, optuna_random_forest.predict(X_valid),average=None,zero_division=0)[1]

    # удаляем крупные файлы чтобы высвободить память 
    del X_train, X_valid, y_train, y_valid
    gc.collect()

  except func_timeout.FunctionTimedOut:
    # удаляем крупные файлы чтобы высвободить память 
    del X_train_balanced, y_train_balanced
    gc.collect()
    
    # фиксируем пустые значения метрик
    roc_train = 0
    roc_valid = 0
    f1_score = 0
    recall_1 = 0
    precision_1 = 0
    pass

  return roc_train, roc_valid, f1_score, recall_1, precision_1

In [ ]:
# cоздаем объект исследования
optuna_study_rf = optuna.create_study(study_name= 'RandomForestClassifier_'+feature_space+'_stat', 
                               directions=['maximize','maximize','maximize','maximize','maximize'], 
                               sampler=optuna.samplers.TPESampler(),
                               pruner='Hyperband',
                               storage='sqlite:///optuna_studies.db',
                               load_if_exists=True)
# на случай удаления 
# optuna.delete_study(study_name='RandomForestClassifier_'+feature_space+'_stat', storage='sqlite:///optuna_studies.db')

In [ ]:
# ищем лучшую комбинацию гиперпараметров n_trials раз
optuna_study_rf.optimize(optuna_rf, n_trials=300)

In [ ]:
# из полученного результат соврмируем Data Frame
optuna_study_rf_pd = optuna_study_rf.trials_dataframe().dropna()
# переименуем столбы
optuna_study_rf_pd.rename(columns={
    'values_0': 'roc_train',
    'values_1': 'roc_valid',
    'values_2': 'f1_score',
    'values_3': 'recall_1',
    'values_4': 'precision_1'
},inplace=True)
optuna_study_rf_pd.tail()

In [ ]:
# покаждем статистику обучения
print('Максимальное значение метрики ROC AUC на валидационном наборе:',round(optuna_study_rf_pd['roc_valid'].max(),3))
print('Среднее значение метрики ROC AUC на валидационном наборе:',round(optuna_study_rf_pd['roc_valid'].mean(),3))
print('Максимальное значение метрики f1_score на валидационном наборе:',round(optuna_study_rf_pd['f1_score'].max(),3))
print('Среднее значение метрики f1_score на валидационном наборе:',round(optuna_study_rf_pd['f1_score'].mean(),3))
print('Максимальное значение метрики recall_1 на валидационном наборе:',round(optuna_study_rf_pd['recall_1'].max(),3))
print('Среднее значение метрики recall_1 на валидационном наборе:',round(optuna_study_rf_pd['recall_1'].mean(),3))
print('Максимальное значение метрики precision_1 на валидационном наборе:',round(optuna_study_rf_pd['precision_1'].max(),3))
print('Среднее значение метрики precision_1 на валидационном наборе:',round(optuna_study_rf_pd['precision_1'].mean(),3))

In [ ]:
# Построим зависимость метрики precision_1

fig = px.scatter(
    data_frame=optuna_study_rf_pd,
    x='precision_1', #ось абсцисс
    y=['recall_1','f1_score'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision от recall на классе 1', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision на классе 1',
    yaxis_title='Величина метрик recall и f1-score')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики precision_1 от гиперпараметров

fig = px.scatter(
    data_frame=optuna_study_rf_pd,
    x='precision_1', #ось абсцисс
    y=['params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight', 'params_max_depth',
       'params_max_features', 'params_max_samples', 'params_min_samples_leaf',
       'params_min_samples_split', 'params_n_estimators', 'params_threshold'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision класса 1 от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision на классе 1',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики f1_score от гиперпараметров

fig = px.scatter(
    data_frame=optuna_study_rf_pd,
    x='f1_score', #ось абсцисс
    y=['params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight', 'params_max_depth',
       'params_max_features', 'params_max_samples', 'params_min_samples_leaf',
       'params_min_samples_split', 'params_n_estimators', 'params_threshold'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость f1-score класса 1 от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики f1-score на классе 1',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики roc_valid от гиперпараметров

fig = px.scatter(
    data_frame=optuna_study_rf_pd,
    x='roc_valid', #ось абсцисс
    y=['params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight', 'params_max_depth',
       'params_max_features', 'params_max_samples', 'params_min_samples_leaf',
       'params_min_samples_split', 'params_n_estimators', 'params_threshold'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость ROC AUC на валидационном наборе от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики ROC AUV на валидационном наборе',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрик качества модели от number

fig = px.scatter(
    data_frame=optuna_study_rf_pd[(optuna_study_rf_pd['recall_1']>=0.08) & (optuna_study_rf_pd['precision_1']>0.08)],
    x='number', #ось абсцисс
    y=['roc_valid','roc_train','precision_1','recall_1'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость качества модели от trials optuna', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='trials optuna',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

<span style="color:Blue">

Вывод:  

Их всех выбираем точку $number =188$.   
Не смотря на, то что в этой точке модель подает признаки переобучеености,  
в этой точке оптимальные значения метрик $recall_1$ и $precision_1$, а  
также относительно высокая метрика $ROC AUC$.   
Посмотрим каким значениям гиперпараметров соотвествует выбраннаяточка.

In [ ]:
# определим номер лучшге варианта
best_optuna_number = 188

# сформируем словарь лучших гипирпарметров RandomForestClassifier
best_param_rf = {
    'n_estimators' : int(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_n_estimators'].iloc[0]),
    'criterion': optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_criterion'].iloc[0],
    'max_depth' : int(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_max_depth'].iloc[0]),
    'min_samples_split': int(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_min_samples_split'].iloc[0]),
    'min_samples_leaf' : int(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_min_samples_leaf'].iloc[0]),
    'max_features': optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_max_features'].iloc[0],
    'max_samples' : optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_max_samples'].iloc[0],
    'class_weight' : {0:optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_class_0_weight'].iloc[0],
                      1:optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_class_1_weight'].iloc[0]}
    }

# определим перменные для лучших значений параметров
best_threshold = optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_threshold'].iloc[0]
best_class_1_percent = optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_class_1_percent'].iloc[0]
best_random_state = optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_random_state'].iloc[0]


# Выведем принятые наилучшие праметры
print('best n estimators:',best_param_rf['n_estimators'])
print('best criterion:',best_param_rf['criterion'])
print('best max depth:',best_param_rf['max_depth'])
print('best min samples split:',best_param_rf['min_samples_split'])
print('best min samples leaf:',best_param_rf['min_samples_leaf'])
print('best max features(%):',round(best_param_rf['max_features'],3))
print('best max samples(%):',round(best_param_rf['max_samples'],3))
print('best class 0 weight:',round(best_param_rf['class_weight'][0],3))
print('best class 1 weight:',round(best_param_rf['class_weight'][1],3))

print('best class 1 percent:',round(best_class_1_percent,3))
print('best threshold:',round(best_threshold,3))
print('best random state:', best_random_state)
print('time for best train:',round(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['duration'].iloc[0].seconds/60),'minutes')

print()
print('ROC AUC на обучающем наборе:', round(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['roc_train'].iloc[0],3))
print('ROC AUC на валидационном наборе:', round(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['roc_valid'].iloc[0],3))
print('precision класса 1:', round(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['precision_1'].iloc[0],3))
print('recall класса 1:', round(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['recall_1'].iloc[0],3))

##### <span style="color:MediumSlateBlue">Обучение модели с лучшими параметрами</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'credit'
count_features = dict_spaces[feature_space]

# для работы функции corr_transform_to_force
# подгрузим DataFrame с матрицей корреляции признаков
corr_matrix = fp.ParquetFile('base_models/data/stat/corr_matrix_'+feature_space).to_pandas()

# с помощью функции corr_transform_to_force получим
# список не коррелируемых признаков
list_ncorr_features = corr_transform_to_force(corr_matrix,threshold=best_threshold)[1]

# сформируем данные для анализа сбалансированности обучающих данных
X_train_pd = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
y_train_pd = pd.read_csv('target/target_train.csv')

# поготовим данные y_train_pd к работе с функцией class_1_percent_samples
y_train_pd.set_index('id',drop=False,inplace=True)

# подготовим данные для обучения модели
# с помощью функции class_1_percent_samples зададим долю 
# класса 1
list_c1_percent_id = class_1_percent_samples(y_train_pd,best_class_1_percent,random_state=best_random_state)[:1000000]

# подготовим данные для обучения
X_train_balanced = X_train_pd.loc[list_c1_percent_id]
y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag']

# освободим память от "тяжелых" и ненужных файлов
del X_train_pd, y_train_pd
gc.collect()


# обучаем модель RandomForestClassifier с наилучшеми параметрами
random_forest = RandomForestClassifier(
        **best_param_rf,
        n_jobs=-1,
        random_state=42)
random_forest.fit(X_train_balanced, y_train_balanced)

# удаляем крупные файлы чтобы высвободить память 
del X_train_balanced, y_train_balanced
gc.collect()

# загружаем тестовые наборы
# сформируем данные для анализа сбалансированности обучающих данных
X_train = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
X_valid = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_valid').to_pandas()[list_ncorr_features]
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()

# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_rf_pred_proba = random_forest.predict_proba(X_train)[:,1]
y_valid_rf_pred_proba = random_forest.predict_proba(X_valid)[:,1]

# Делаем предсказание для валидационной выборки
y_valid_rf_pred = random_forest.predict(X_valid)

# удаляем крупные файлы чтобы высвободить память 
del X_train, X_valid
gc.collect()

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_rf_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_rf_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_valid,y_valid_rf_pred,zero_division=0))

# time: 5m 30s

##### <span style="color:MediumSlateBlue">Анализ влияния порога классификации на качество модели</span>

In [ ]:
# чтобы проше обращаться к каждому обьекту в данных
# преобразуем y_valid_LR_pred к Series типу
y_valid_rf_pred_proba = pd.Series(y_valid_rf_pred_proba)

# формируем список из необходимых значений thresholds
thresholds = np.arange(0.01, 1, 0.01)

# сформируем списко для заполнения данными результатов анализа
threshold_data = []


for threshold in thresholds:
        # сформируем список для заполнения данными текушего threshold
        threshold_current_data = [threshold]

        #клиентов, для которых вероятность дефолта > threshold, относим к классу 1
        #в противном случае — к классу 0
        y_valid_rf_pred = y_valid_rf_pred_proba.apply(lambda x: 1 if x>threshold else 0)


        #Считаем метрики для класса 0 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(y_valid, y_valid_rf_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.precision_score(y_valid, y_valid_rf_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.f1_score(y_valid, y_valid_rf_pred,average=None,zero_division=0)[0],3))

        #Считаем метрики для класса 1 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(y_valid, y_valid_rf_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.precision_score(y_valid, y_valid_rf_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.f1_score(y_valid, y_valid_rf_pred,average=None,zero_division=0)[1],3))

        # добавляем полученные данные в список threshold_data
        threshold_data.append(threshold_current_data)

        # из полученных данных сформируем dataframe
        thresholds_pd =pd.DataFrame(
        columns=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'],
        data = threshold_data)

# time: 17s

In [ ]:
# Визуализируем метрики на валидационном наборе при различных threshold
fig_threshold = px.line(
    data_frame=thresholds_pd,
    x='threshold', #ось абсцисс
    y=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'], #ось ординат
)
fig_threshold.update_layout(
    title ={
        'text':'Зависимость качества модели от threshold', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Порог классификации threshold',
    yaxis_title='Величина метрики')
fig_threshold.update_xaxes(showspikes=True)
fig_threshold.update_yaxes(showspikes=True)

fig_threshold.show()

Метрика $precision$ показывает на сколько сильно ошиблась модель при определиние класса.  
Чем выше метрика, тем меньше ошиблась модель. 

$$precision = \frac{TP}{TP+FP}$$

Метрика $recall$ показывает способность модели найти класс.  
Чем выше метрика, тем выше способность модели находить класс.  

$$recall = \frac{TP}{TP+FN}$$

Метрика $f1-score$ показывает баланс между $precision$ и $recall$ 

$threshold$- порог класификации модели.  
Все объекты для которых $у_{\text{proba pred}} < threshold$, модель относит к класс 0.  
Соотвественно, если $у_{\text{proba pred}}  > threshold$ модель относит обьект к классу 1.  
Для идельаной модели $threshold$, является границей между областями класса 0 и класса 1.  

<span style="color:Blue">

Выводы по результам анализа:  

Для улучшения качества модели по метрике $ROC AUC$, нет смысла сдвигать порог   
классификации. Но зависимость метрик качества модели от $threshold$, может рассказать о  
способности модели искать класс 1 и класс 0.

При фокусировке модели на классе 1 (более низкий $threshold$):  
- на классе 1 метрика $precision$ уменьшается, метрика $recall$ растет;  
- на классе 0 метрика $precision$ растет, метрика $recall$ уменьшается.  

При фокусировке модели на классе 0 (более высокий $threshold$):  
- на классе 1 метрика $precision$ увеличивается, метрика $recall$ уменьшается;  
- на классе 0 метрика $precision$ уменьшается, метрика $recall$ растет.  

Обьсняется тем, что в условия дисбаланса модель научилась хорошо определять класс 0 и плохо класс 1.  

При фокусировке модели на классе 1 (более низкий $threshold$), все больше обьектов отнесены к классу 1.   
Так как $precision$ класса 1 уменьшается, значит в области класса 1 стало больше обектов класса 0,  
что явлется закономерным для "идеальной" модели.  
Однако при этом, $recall$ класса 1 растет, значит в добавленных обьектах также присутвует класс 1.  
Это как раз указывает на то, что модель не нучилась находить среди класса 0 класс 1. В области с     
высокой вероятностью класса 0 (более низкий $threshold$), находятся обьекты класса 1.  

Фокусировка на классе 0 (более высокий $threshold$), заставляет модель все больше обьектов из области класса 1  
относить к классу 0. При этом увеличение метрики $precision$ класса 1 говорит о том, что в области класса 1     
становится меньше 0, а значит модель умеет среди класса 1, находить класс 0.  
После  $threshold=0.77$ модель впринципе теряет способность отличать класс 1. У модели нет обьектов класса 1   
в которых она "уверена" на 80+%.

Обратим внимание на $threshold=0.49$, в этой точке находится баланс межу метриками качества модели:
$$precision_1 = recall_1 = \text{f1-score}_1 = 0.09$$
$$precision_0 = recall_0 = \text{f1-score}_0 = 0.968$$

Для модели с лучшим качеством эти две точки должны находится максимально близко к друг другу и иметь   
высокое значение метрик $precision_1$ и $recall_1$.  
Для класса 0 это условие выполнется, что также говорит о том, что модель научилась искать этот класс.    
Для класса 1 это условие не выполнется, что также говорит о том, что модель плохо ищет класс 1.

##### <span style="color:MediumSlateBlue">Построение ROC-кривой выбранной модели на тестовом наборе</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'credit'
count_features = dict_spaces[feature_space]

In [ ]:
# сформируем данные для проверки модели
X_train = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
X_valid = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_valid').to_pandas()[list_ncorr_features]
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()
X_test = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_test').to_pandas()[list_ncorr_features]
y_test = pd.read_csv('target/target_test.csv')['flag'].to_numpy()

In [ ]:
print("Размеры обучающего набора",X_train.shape,y_train.shape)
print("Размеры валидационного набора",X_valid.shape,y_valid.shape)
print("Размеры тестового набора",X_test.shape,y_test.shape)
# time: 1m 43s

In [ ]:
# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_rf_pred_proba = random_forest.predict_proba(X_train)[:,1]
y_valid_rf_pred_proba = random_forest.predict_proba(X_valid)[:,1]
y_test_rf_pred_proba = random_forest.predict_proba(X_test)[:,1]

# Делаем предсказание для валидационной выборки
y_test_lr_pred = random_forest.predict(X_test)

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_rf_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_rf_pred_proba),3))
print('ROC AUC на тестовом наборе', round(metrics.roc_auc_score(y_test, y_test_rf_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_test,y_test_lr_pred,zero_division=0))

In [ ]:
# Для расчета значений TPR и FPR используем функцию roc_curve
fpr_train, tpr_train, _ = metrics.roc_curve(y_train, y_train_rf_pred_proba)
fpr_valid, tpr_valid, _ = metrics.roc_curve(y_valid, y_valid_rf_pred_proba)
fpr_test, tpr_test, _ = metrics.roc_curve(y_test, y_test_rf_pred_proba)

# рассчитаем площадь под кривой
auc_train = round(metrics.roc_auc_score(y_train, y_train_rf_pred_proba),3)
auc_valid = round(metrics.roc_auc_score(y_valid, y_valid_rf_pred_proba),3)
auc_test = round(metrics.roc_auc_score(y_test, y_test_rf_pred_proba),3)

trace_train = go.Scatter(x=fpr_train, y=tpr_train, mode='lines', name='AUC_train = %0.2f' % auc_train,
                   line=dict(color='red', width=2),
                   hovertemplate='train<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_train])

trace_valid = go.Scatter(x=fpr_valid, y=tpr_valid, mode='lines', name='AUC_valid = %0.2f' % auc_valid,
                   line=dict(color='green', width=2),
                   hovertemplate='valid<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_valid])

trace_test = go.Scatter(x=fpr_test, y=tpr_test, mode='lines', name='AUC_test = %0.2f' % auc_test,
                   line=dict(color='darkorange', width=2),
                   hovertemplate='test<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_test])


reference_line = go.Scatter(x=[0,1], y=[0,1], mode='lines', name='Reference Line',
                            line=dict(color='navy', width=2, dash='dash'))

fig_roc = go.Figure(data=[trace_train,trace_valid,trace_test,reference_line])

fig_roc.update_layout(
    title ={
        'text':'ROC-кривая', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 1350,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate')

fig_roc.update_xaxes(showspikes=True)
fig_roc.update_yaxes(showspikes=True)


fig_roc.show()

<span style="color:Blue">

Вывод по ROC-кривой:
1. Не смотря на то, что модель показывает, очень хорошую $ROC$ кривую на обучающем наборе,  
модель, так же выдает сталибильное качество на валидационном и тестовом наборах, а значит  
не переобучена - разброс ($variance$) не большой.
2. Смещение ($bias$) модели такое же как у модели $Logistic$ $Regression$ $Classifier$,  
обученной  на тех же данных.  
На тестовом наборе полученно качество $ROC AUC = 0.61$.

#### <span style="color:MediumBlue">Построение модели Gradient Boosting Classifier</span>

##### <span style="color:MediumSlateBlue">Baseline обучение модели Hist Gradient Boosting Classifier</span>

In [ ]:
# обучаем модель Gradient Boosting Classifier
gradient_boosting = HistGradientBoostingClassifier(random_state=42)
gradient_boosting.fit(X_train_balanced[list_ncorr_features],y_train_balanced)

# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_rf__pred_proba = gradient_boosting.predict_proba(X_train[list_ncorr_features])[:,1]
y_valid_rf_pred_proba = gradient_boosting.predict_proba(X_valid[list_ncorr_features])[:,1]

# Делаем предсказание для валидационной выборки
y_valid_rf_pred = gradient_boosting.predict(X_valid[list_ncorr_features])

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_rf__pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_rf_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_valid,y_valid_rf_pred))

# time: 30s

##### <span style="color:MediumSlateBlue">Подбор гиперпараметров модели</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'credit'
count_features = dict_spaces[feature_space]

# для работы функции corr_transform_to_force
# подгрузим DataFrame с матрицей корреляции признаков
corr_matrix = fp.ParquetFile('base_models/data/stat/corr_matrix_'+feature_space).to_pandas()

# настроим оптимизацию гипер параметров
def optuna_hgbc(trial):
  # задаем пространства поиска гиперпараметров
  learning_rate = trial.suggest_float('learning_rate',0.01,0.1)
  max_iter = trial.suggest_int('max_iter', 150, 250,step = 1)
  max_leaf_nodes = trial.suggest_int('max_leaf_nodes', 2, 60,step = 1)
  max_depth = trial.suggest_int('max_depth', 1, 10,step = 1)
  min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 60,step = 1)
  max_features = trial.suggest_float('max_features',0.6,1)
  l2_regularization = trial.suggest_float('l2_regularization',0.01,1)
  class_0_weight = trial.suggest_float('class_0_weight',0.01,1)
  class_1_weight = trial.suggest_float('class_1_weight',0.5,1)
  threshold = trial.suggest_float('threshold',0.7,1)
  class_1_percent = trial.suggest_float('class_1_percent',0.01,0.25)
  random_state = trial.suggest_int('random_state', 1, 1000000)

  # создаем модель
  optuna_gradient_boosting = HistGradientBoostingClassifier(
      learning_rate= learning_rate,
      max_iter = max_iter,
      max_leaf_nodes =max_leaf_nodes,
      max_depth = max_depth,
      min_samples_leaf = min_samples_leaf,
      max_features=max_features,
      l2_regularization = l2_regularization,
      class_weight={0:class_0_weight,1:class_1_weight},
      random_state=42)

  # с помощью функции corr_transform_to_force получим
  # список не коррелируемых признаков
  list_ncorr_features = corr_transform_to_force(corr_matrix,threshold=threshold)[1]

  # сформируем данные для анализа сбалансированности обучающих данных
  X_train_pd = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
  y_train_pd = pd.read_csv('target/target_train.csv')

  # поготовим данные y_train_pd к работе с функцией class_1_percent_samples
  y_train_pd.set_index('id',drop=False,inplace=True)

  # подготовим данные для обучения модели
  # с помощью функции class_1_percent_samples зададим долю 
  # класса 1
  list_c1_percent_id = class_1_percent_samples(y_train_pd,class_1_percent,random_state=random_state)[:1000000]

  # подготовим данные для обучения
  X_train_balanced = X_train_pd.loc[list_c1_percent_id]
  y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag']

  # освободим память от "тяжелых" и ненужных файлов
  del X_train_pd, y_train_pd
  gc.collect()

  # я не воспользовался параметром timeout метода optimize потому, что  
  # optimize отставливает поиск параметров, если время обучения 
  # превышает timeout. Мне нужно чтобы поиск параметров продолжился.
  try:
    # для модуля func_time_out упакуем обучение модели в функцию try_func
    def try_func():
            return optuna_gradient_boosting.fit(X_train_balanced,y_train_balanced)
    # обучаем модель с ограничением по времени 10 минут (600 секунд)
    optuna_gradient_boosting = func_timeout.func_timeout(600, try_func)

    # удаляем крупные файлы чтобы высвободить память 
    del X_train_balanced, y_train_balanced
    gc.collect()

        # загружаем тестовые наборы
    # сформируем данные для анализа сбалансированности обучающих данных
    X_train = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
    X_valid = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_valid').to_pandas()[list_ncorr_features]
    y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
    y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()
    
    # делаем предсказание на обучающем и валидационном наборе и считаем метрики
    roc_train = metrics.roc_auc_score(y_train, optuna_gradient_boosting.predict_proba(X_train)[:,1])
    roc_valid = metrics.roc_auc_score(y_valid, optuna_gradient_boosting.predict_proba(X_valid)[:,1])
    f1_score = metrics.f1_score(y_valid, optuna_gradient_boosting.predict(X_valid))
    recall_1 = metrics.recall_score(y_valid, optuna_gradient_boosting.predict(X_valid),average=None,zero_division=0)[1]
    precision_1 = metrics.precision_score(y_valid, optuna_gradient_boosting.predict(X_valid),average=None,zero_division=0)[1]


    # удаляем крупные файлы чтобы высвободить память 
    del X_train, X_valid, y_train, y_valid
    gc.collect()

  except func_timeout.FunctionTimedOut:
    # удаляем крупные файлы чтобы высвободить память 
    del X_train_balanced, y_train_balanced
    gc.collect()
    
    # фиксируем пустые значения метрик
    roc_train = 0
    roc_valid = 0
    f1_score = 0
    recall_1 = 0
    precision_1 = 0
    pass

  return roc_train, roc_valid, f1_score, recall_1, precision_1

In [ ]:
# cоздаем объект исследования
# чтобы модель не переобучалась минимизируем roc_train 
# и максимизируем roc_valid
optuna_study_hgbc = optuna.create_study(study_name='HistGradientBoostingClassifier_'+feature_space+'_stat', 
                               directions=['maximize','maximize','maximize','maximize','maximize'], 
                               sampler=optuna.samplers.TPESampler(),
                               pruner='Hyperband',
                               storage='sqlite:///optuna_studies.db',
                               load_if_exists=True)

# ну случай удаления обучения
# optuna.delete_study(study_name='HistGradientBoostingClassifier'+feature_space+'_stat', storage='sqlite:///optuna_studies.db')

In [ ]:
# ищем лучшую комбинацию гиперпараметров n_trials раз
optuna_study_hgbc.optimize(optuna_hgbc, n_trials=300)

In [ ]:
# из полученного результат соврмируем Data Frame
optuna_study_hgbc_pd = optuna_study_hgbc.trials_dataframe()
# переименуем столбы
optuna_study_hgbc_pd.rename(columns={
    'values_0': 'roc_train',
    'values_1': 'roc_valid',
    'values_2': 'f1_score',
    'values_3': 'recall_1',
    'values_4': 'precision_1'
},inplace=True)
optuna_study_hgbc_pd.tail(5)

In [ ]:
# покаждем статистику обучения
print('Максимальное значение метрики ROC AUC на валидационном наборе:',round(optuna_study_hgbc_pd['roc_valid'].max(),3))
print('Среднее значение метрики ROC AUC на валидационном наборе:',round(optuna_study_hgbc_pd['roc_valid'].mean(),3))
print('Максимальное значение метрики f1_score на валидационном наборе:',round(optuna_study_hgbc_pd['f1_score'].max(),3))
print('Среднее значение метрики f1_score на валидационном наборе:',round(optuna_study_hgbc_pd['f1_score'].mean(),3))
print('Максимальное значение метрики recall_1 на валидационном наборе:',round(optuna_study_hgbc_pd['recall_1'].max(),3))
print('Среднее значение метрики recall_1 на валидационном наборе:',round(optuna_study_hgbc_pd['recall_1'].mean(),3))
print('Максимальное значение метрики precision_1 на валидационном наборе:',round(optuna_study_hgbc_pd['precision_1'].max(),3))
print('Среднее значение метрики precision_1 на валидационном наборе:',round(optuna_study_hgbc_pd['precision_1'].mean(),3))

In [ ]:
# построим график важности гиперпарметров
optuna.visualization.plot_param_importances(optuna_study_hgbc, target_name='Score')

In [ ]:
# Построим зависимость метрики precision_1
fig = px.scatter(
    data_frame=optuna_study_hgbc_pd,
    x='precision_1', #ось абсцисс
    y=['f1_score','recall_1'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision от recall на классе 1', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision класса 1',
    yaxis_title='Величина метрик recall и f1-score класса 1')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики precision_1 от гиперпараметров
fig = px.scatter(
    data_frame=optuna_study_hgbc_pd,
    x='precision_1', #ось абсцисс
    y=['params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight', 'params_l2_regularization',
       'params_learning_rate', 'params_max_depth', 'params_max_features',
       'params_max_iter', 'params_max_leaf_nodes', 'params_min_samples_leaf',
       'params_threshold'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision класса 1 от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision класса 1',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики f1_score от гиперпараметров
fig = px.scatter(
    data_frame=optuna_study_hgbc_pd,
    x='f1_score', #ось абсцисс
    y=['params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight', 'params_l2_regularization',
       'params_learning_rate', 'params_max_depth', 'params_max_features',
       'params_max_iter', 'params_max_leaf_nodes', 'params_min_samples_leaf',
       'params_threshold'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость f1-score класса 1 от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики f1-score класса 1',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики roc_valid от гиперпараметров
fig = px.scatter(
    data_frame=optuna_study_hgbc_pd,
    x='roc_valid', #ось абсцисс
    y=['params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight', 'params_l2_regularization',
       'params_learning_rate', 'params_max_depth', 'params_max_features',
       'params_max_iter', 'params_max_leaf_nodes', 'params_min_samples_leaf',
       'params_threshold'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость ROC AUC на валидационном наборе от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики ROC AUC на валидационном наборе',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрик качества модели от number

fig = px.scatter(
    data_frame=optuna_study_hgbc_pd[(optuna_study_hgbc_pd['recall_1']>0.1)&(optuna_study_hgbc_pd['precision_1']>0.1)],
    x='number', #ось абсцисс
    y=['roc_train','roc_valid','recall_1','precision_1'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость качества модели от trials optuna', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='trials optuna',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

<span style="color:Blue">

Вывод:  

Их всех выбираем точку $number =127$.   
В этой точке оптимальные значения метрик $recall_1$ и $precision_1$, а  
также относительно высокая метрика $ROC AUC$.   
Посмотрим каким значениям гиперпараметров соотвествует выбранная точка.

In [ ]:
# определим номер лучшге варианта
best_optuna_number = 127

# сформируем словарь лучших гипирпарметров HistGradientBoostingClassifier
best_param_hgbc = {
    'learning_rate' : optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['params_learning_rate'].iloc[0],
    'max_iter' : optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['params_max_iter'].iloc[0],
    'max_leaf_nodes' : optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['params_max_leaf_nodes'].iloc[0],
    'max_depth' : optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['params_max_depth'].iloc[0],
    'min_samples_leaf' : optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['params_min_samples_leaf'].iloc[0],
    'max_features' : optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['params_max_features'].iloc[0],
    'l2_regularization' : optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['params_l2_regularization'].iloc[0],
    'class_weight' : {0: optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['params_class_0_weight'].iloc[0],
                      1: optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['params_class_1_weight'].iloc[0],}
    }

# определим перменные для лучших значений параметров
best_threshold = optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['params_threshold'].iloc[0]
best_class_1_percent = optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['params_class_1_percent'].iloc[0]
best_random_state = optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['params_random_state'].iloc[0]


# Выведем принятые наилучшие параметры
print('best learning rate:',round(best_param_hgbc['learning_rate'],3))
print('best max iter:',best_param_hgbc['max_iter'])
print('best max leaf nodes:',best_param_hgbc['max_leaf_nodes'])
print('best max depth:',best_param_hgbc['max_depth'])
print('best min samples leaf:',best_param_hgbc['min_samples_leaf'])
print('best max features:',round(best_param_hgbc['max_features'],3))
print('best l2 regularization:',round(best_param_hgbc['l2_regularization'],3))
print('best class 0 weight:',round(best_param_hgbc['class_weight'][0],3))
print('best class 1 weight:',round(best_param_hgbc['class_weight'][1],3))

print('best class 1 percent:',round(best_class_1_percent,3))
print('best threshold:',round(best_threshold,3))
print('time for best train:',round(optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['duration'].iloc[0].seconds/60),'minutes')

print()
print('ROC AUC на обучающем наборе:', round(optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['roc_train'].iloc[0],3))
print('ROC AUC на валидационном наборе:', round(optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['roc_valid'].iloc[0],3))
print('precision класса 1:', round(optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['precision_1'].iloc[0],3))
print('recall класса 1:', round(optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['recall_1'].iloc[0],3))

##### <span style="color:MediumSlateBlue">Обучение модели с лучшими параметрами</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'credit'
count_features = dict_spaces[feature_space]

# для работы функции corr_transform_to_force
# подгрузим DataFrame с матрицей корреляции признаков
corr_matrix = fp.ParquetFile('base_models/data/stat/corr_matrix_'+feature_space).to_pandas()

# с помощью функции corr_transform_to_force получим
# список не коррелируемых признаков
list_ncorr_features = corr_transform_to_force(corr_matrix,threshold=best_threshold)[1]

# сформируем данные для анализа сбалансированности обучающих данных
X_train_pd = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
y_train_pd = pd.read_csv('target/target_train.csv')

# поготовим данные y_train_pd к работе с функцией class_1_percent_samples
y_train_pd.set_index('id',drop=False,inplace=True)

# подготовим данные для обучения модели
# с помощью функции class_1_percent_samples зададим долю 
# класса 1
list_c1_percent_id = class_1_percent_samples(y_train_pd,best_class_1_percent,random_state=best_random_state)[:1000000]

# подготовим данные для обучения
X_train_balanced = X_train_pd.loc[list_c1_percent_id]
y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag']

# освободим память от "тяжелых" и ненужных файлов
del X_train_pd, y_train_pd
gc.collect()

# обучаем модель HistGradientBoostingClassifier с наилучшеми параметрами
gradient_boosting = HistGradientBoostingClassifier(
        **best_param_hgbc,
        random_state=42)
gradient_boosting.fit(X_train_balanced,y_train_balanced)

# удаляем крупные файлы чтобы высвободить память 
del X_train_balanced, y_train_balanced
gc.collect()

# сформируем данные для анализа сбалансированности обучающих данных
X_train = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
X_valid = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_valid').to_pandas()[list_ncorr_features]
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()

# делаем предсказание на обучающем и валидационном наборе и считаем метрики
y_train_gb_pred_proba = gradient_boosting.predict_proba(X_train)[:,1]
y_valid_gb_pred_proba = gradient_boosting.predict_proba(X_valid)[:,1]
y_valid_gb_pred = gradient_boosting.predict(X_valid)

# удаляем крупные файлы чтобы высвободить память 
del X_train, X_valid
gc.collect()

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_gb_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_gb_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_valid,y_valid_gb_pred,zero_division=0))

# time: 1m 40s

##### <span style="color:MediumSlateBlue">Анализ влияния порога классификации на качество модели</span>

In [ ]:
# чтобы проше обращаться к каждому обьекту в данных
# преобразуем y_valid_LR_pred к Series типу
y_valid_gb_pred_proba = pd.Series(y_valid_gb_pred_proba)

# формируем список из необходимых значений thresholds
thresholds = np.arange(0.01, 1, 0.01)

# сформируем списко для заполнения данными результатов анализа
threshold_data = []


for threshold in thresholds:
        # сформируем список для заполнения данными текушего threshold
        threshold_current_data = [threshold]

        #клиентов, для которых вероятность дефолта > threshold, относим к классу 1
        #в противном случае — к классу 0
        y_valid_gb_pred = y_valid_gb_pred_proba.apply(lambda x: 1 if x>threshold else 0)


        #Считаем метрики для класса 0 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(y_valid, y_valid_gb_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.precision_score(y_valid, y_valid_gb_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.f1_score(y_valid, y_valid_gb_pred,average=None,zero_division=0)[0],3))

        #Считаем метрики для класса 1 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(y_valid, y_valid_gb_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.precision_score(y_valid, y_valid_gb_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.f1_score(y_valid, y_valid_gb_pred,average=None,zero_division=0)[1],3))

        # добавляем полученные данные в список threshold_data
        threshold_data.append(threshold_current_data)

        # из полученных данных сформируем dataframe
        thresholds_pd =pd.DataFrame(
        columns=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'],
        data = threshold_data)

# time: 17s

In [ ]:
# Визуализируем метрики на валидационном наборе при различных threshold
fig_threshold = px.line(
    data_frame=thresholds_pd,
    x='threshold', #ось абсцисс
    y=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'], #ось ординат
)
fig_threshold.update_layout(
    title ={
        'text':'Зависимость качества модели от threshold', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Порог классификации threshold',
    yaxis_title='Величина метрики')
fig_threshold.update_xaxes(showspikes=True)
fig_threshold.update_yaxes(showspikes=True)

fig_threshold.show()

Метрика $precision$ показывает на сколько сильно ошиблась модель при определиние класса.  
Чем выше метрика, тем меньше ошиблась модель. 

$$precision = \frac{TP}{TP+FP}$$

Метрика $recall$ показывает способность модели найти класс.  
Чем выше метрика, тем выше способность модели находить класс.  

$$recall = \frac{TP}{TP+FN}$$

Метрика $f1-score$ показывает баланс между $precision$ и $recall$ 

$threshold$- порог класификации модели.  
Все объекты для которых $у_{\text{proba pred}} < threshold$, модель относит к класс 0.  
Соотвественно, если $у_{\text{proba pred}}  > threshold$ модель относит обьект к классу 1.  
Для идельаной модели $threshold$, является границей между областями класса 0 и класса 1.  

<span style="color:Blue">

Выводы по результам анализа:  

Для улучшения качества модели по метрике $ROC AUC$, нет смысла сдвигать порог   
классификации. Но зависимость метрик качества модели от $threshold$, может рассказать о  
способности модели искать класс 1 и класс 0.

При фокусировке модели на классе 1 (более низкий $threshold$):  
- на классе 1 метрика $precision$ уменьшается, метрика $recall$ растет;  
- на классе 0 метрика $precision$ растет, метрика $recall$ уменьшается.  

При фокусировке модели на классе 0 (более высокий $threshold$):  
- на классе 1 метрика $precision$ увеличивается, метрика $recall$ уменьшается;  
- на классе 0 метрика $precision$ уменьшается, метрика $recall$ растет.  

Обьсняется тем, что в условия дисбаланса модель научилась хорошо определять класс 0 и плохо класс 1.  

При фокусировке модели на классе 1 (более низкий $threshold$), все больше обьектов отнесены к классу 1.   
Так как $precision$ класса 1 уменьшается, значит в области класса 1 стало больше обектов класса 0,  
что явлется закономерным для "идеальной" модели.  
Однако при этом, $recall$ класса 1 растет, значит в добавленных обьектах также присутвует класс 1.  
Это как раз указывает на то, что модель не нучилась находить среди класса 0 класс 1. В области с     
высокой вероятностью класса 0 (более низкий $threshold$), находятся обьекты класса 1.  

Фокусировка на классе 0 (более высокий $threshold$), заставляет модель все больше обьектов из области класса 1  
относить к классу 0. При этом увеличение метрики $precision$ класса 1 говорит о том, что в области класса 1     
становится меньше 0, а значит модель умеет среди класса 1, находить класс 0.  
После  $threshold=0.88$ модель впринципе теряет способность отличать класс 1. У модели нет обьектов класса 1   
в которых она "уверена" на 90+%.


Обратим внимание на $threshold=0.51$, в этой точке находится баланс межу метриками качества модели:
$$precision_1 = recall_1 = \text{f1-score}_1 = 0.103$$
$$precision_0 = recall_0 = \text{f1-score}_0 = 0.97$$

Для модели с лучшим качеством эти две точки должны находится максимально близко к друг другу и иметь   
высокое значение метрик $precision_1$ и $recall_1$.  
Для класса 0 это условие выполнется, что также говорит о том, что модель научилась искать этот класс.    
Для класса 1 это условие не выполнется, что также говорит о том, что модель плохо ищет класс 1.

##### <span style="color:MediumSlateBlue">Построение ROC-кривой выбранной модели на тестовом наборе</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'credit'
count_features = dict_spaces[feature_space]

In [ ]:
# сформируем данные для проверки модели
X_train = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
X_valid = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_valid').to_pandas()[list_ncorr_features]
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()
X_test = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_test').to_pandas()[list_ncorr_features]
y_test = pd.read_csv('target/target_test.csv')['flag'].to_numpy()

In [ ]:
print("Размеры обучающего набора",X_train.shape,y_train.shape)
print("Размеры валидационного набора",X_valid.shape,y_valid.shape)
print("Размеры тестового набора",X_test.shape,y_test.shape)
# time: 1m 43s

In [ ]:
# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_gb_pred_proba = gradient_boosting.predict_proba(X_train)[:,1]
y_valid_gb_pred_proba = gradient_boosting.predict_proba(X_valid)[:,1]
y_test_gb_pred_proba = gradient_boosting.predict_proba(X_test)[:,1]

# Делаем предсказание для валидационной выборки
y_test_gb_pred = gradient_boosting.predict(X_test)

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_gb_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_gb_pred_proba),3))
print('ROC AUC на тестовом наборе', round(metrics.roc_auc_score(y_test, y_test_gb_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_test,y_test_gb_pred,zero_division=0))

In [ ]:
# Для расчета значений TPR и FPR используем функцию roc_curve
fpr_train, tpr_train, _ = metrics.roc_curve(y_train, y_train_gb_pred_proba)
fpr_valid, tpr_valid, _ = metrics.roc_curve(y_valid, y_valid_gb_pred_proba)
fpr_test, tpr_test, _ = metrics.roc_curve(y_test, y_test_gb_pred_proba)

# рассчитаем площадь под кривой
auc_train = round(metrics.roc_auc_score(y_train, y_train_gb_pred_proba),3)
auc_valid = round(metrics.roc_auc_score(y_valid, y_valid_gb_pred_proba),3)
auc_test = round(metrics.roc_auc_score(y_test, y_test_gb_pred_proba),3)

trace_train = go.Scatter(x=fpr_train, y=tpr_train, mode='lines', name='AUC_train = %0.2f' % auc_train,
                   line=dict(color='red', width=2),
                   hovertemplate='train<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_train])

trace_valid = go.Scatter(x=fpr_valid, y=tpr_valid, mode='lines', name='AUC_valid = %0.2f' % auc_valid,
                   line=dict(color='green', width=2),
                   hovertemplate='valid<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_valid])

trace_test = go.Scatter(x=fpr_test, y=tpr_test, mode='lines', name='AUC_test = %0.2f' % auc_test,
                   line=dict(color='darkorange', width=2),
                   hovertemplate='test<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_test])


reference_line = go.Scatter(x=[0,1], y=[0,1], mode='lines', name='Reference Line',
                            line=dict(color='navy', width=2, dash='dash'))

fig_roc = go.Figure(data=[trace_train,trace_valid,trace_test,reference_line])

fig_roc.update_layout(
    title ={
        'text':'ROC-кривая', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 1350,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate')

fig_roc.update_xaxes(showspikes=True)
fig_roc.update_yaxes(showspikes=True)


fig_roc.show()

<span style="color:Blue">

Вывод по ROC-кривой:
1. Модель выдает сталибильное качество на валидационном и тестовом наборах, а значит  
не переобучена - разброс ($variance$) не большой. Хотя, качество модели на обучающем наборе,  
заметно лучше.
2. Смещение ($bias$) модели чуть лучше, чем у модели $Random$ $Forest$ $Classifier$,  
обученной на тех же данных.    
На тестовом наборе полученно качество $ROC AUC = 0.63$, против $ROC AUC = 0.61$.

### <span style="color:RoyalBlue">Построение модели на сбалансированных данных подпрострнства relative stat</span>

#### <span style="color:MediumBlue">Формирование данных для обучения</span>

Анализ исходных данных, в частности target, показал что представленные данные   
имееют перекос: 96% клиентов у которых не случился дефолт (flag = 0),    
против 4 % - для которых произошел дефолт (flag = 1).  
С помощью функции class_1_percent_samples, сформируем сбаланисрованную выборку  
для обучения моделей.
За основу сбалансировных данных возьмем данные клиентов с дефолтом  
и к ним будем добирать необходимое количество клиентов до сбалансированности. 


Функция class_1_percent_samples принемает две переменные:
- target_data - массив с id и целевой переменной
- class_1_percent - процент класса 1 в результирующем массиве

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'relative'
count_features = dict_spaces[feature_space]

In [ ]:
# сформируем данные для анализа сбалансированности обучающих данных
X_train_pd = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()
y_train_pd = pd.read_csv('target/target_train.csv')

In [ ]:
# поготовим данные y_train_pd к работе с функцией class_1_percent_samples
y_train_pd.set_index('id',drop=False,inplace=True)
y_train_pd.head(4)

In [ ]:
# взглянем на сблансиорованность данных в y_train
y_train_pd['flag'].value_counts(normalize=True)

In [ ]:
# сбалансируем данные в y_train_pd
list_c1_percent_id = class_1_percent_samples(y_train_pd,0.5)
# list_c1_percent_features - список 'id' из y_train_pd соотвествующих  
# заданному проценту класса 1
len(list_c1_percent_id)

In [ ]:
# проверим сбалансированность полученного y_train
y_train_pd.loc[list_c1_percent_id]['flag'].value_counts(normalize=True)

В отличие от данных в *transform_data_torow*, где модели показываеются последние N  
операций клиента, в данные в *transform_data_stat* сфомированы из различных статистических   
характеристик ряда определяющего клиентскую историю в банке. 

Соотвественно, если в данных *transform_data_torow* была сильная корреляция между признаками,  
то я не смог бы ее убрать, иначе потярется смысл "показать последовательный ряд в историии клиента".  
Признаки в *transform_data_stat* не имеют такой особенности. Поэтому, необходимо проанализировать  
данные в *transform_data_stat* и, в случае сильной коррреляции признаков, устранить ее.

In [ ]:
# подгрузим DataFrame с матрицей корреляции признаков
corr_matrix = fp.ParquetFile('base_models/data/stat/corr_matrix_'+feature_space).to_pandas()
corr_matrix.shape

In [ ]:
# с помощью функции corr_transrom_to_power получим
# силу корреляции матрицы и список не коррелируемых признаков

# для начального анализа зададим порог силы корреляции, при котором признаки 
# будут отнесены к "сильно скоррелированными" равным 0.6
ncorr_matrix,list_ncorr_features,corr_power =corr_transform_to_force(corr_matrix,threshold=0.6)

# Выведем результаты преобразования
print('Исходное количество признаков: ', corr_matrix.shape[0])
print('Количество не коррелируемых признаков: ', len(list_ncorr_features))
# сила корреляции признаков
print('Отношение коррелируемых исходному количеству признаков (сила корреляции): ',corr_power)

In [ ]:
# подготовим данные для обучения
X_train_balanced = X_train_pd.loc[list_c1_percent_id]
y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag']
# сформируем данные для проверики модели
# сформируем данные для анализа сбалансированности обучающих данных
X_train = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()
y_train = pd.read_csv('target/target_train.csv')['flag']
X_valid = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_valid').to_pandas()
y_valid = pd.read_csv('target/target_valid.csv')['flag']

In [ ]:
# посмотрим на структуру данных в X_train_balanced
X_train_balanced.head(4)

In [ ]:
# посмотрим на структуру данных в y_train_balanced
y_train_balanced.head(4)

In [ ]:
# проверим что выборки действительно совпадают по id
X_train_balanced.index.to_list() == y_train_balanced.index.to_list()

#### <span style="color:MediumBlue">Построение модели Logistic Regression</span>

##### <span style="color:MediumSlateBlue">Baseline обучение модели LogisticRegression</span>

In [ ]:
# обучаем модель логистической регрессии
logistic_regression = linear_model.LogisticRegression(random_state=42, max_iter=10000)
logistic_regression.fit(X_train_balanced[list_ncorr_features],y_train_balanced)

# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_LR_pred_proba = logistic_regression.predict_proba(X_train[list_ncorr_features])[:,1]
y_valid_LR_pred_proba = logistic_regression.predict_proba(X_valid[list_ncorr_features])[:,1]

# Делаем предсказание для валидационной выборки
y_valid_LR_pred = logistic_regression.predict(X_valid[list_ncorr_features])

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_LR_pred_proba),3))
print('ROC AUC на тестовом наборе', round(metrics.roc_auc_score(y_valid, y_valid_LR_pred_proba),3))

print('Основные метрики на тестовом наборе:')
print(metrics.classification_report(y_valid,y_valid_LR_pred))

# time: 3m 5s

##### <span style="color:MediumSlateBlue">Анализ влияния scaler преобразования на качество и скорость схождения модели</span>

In [ ]:
# формируем список из необходимых scaler
scalers = [MinMaxScaler(),RobustScaler() ,StandardScaler()]

# сформируем списко для заполнения данными результатов анализа
scalers_data = []

# установим ограничение на время обучения модели в 600 секунд (10 минут)
time_out = 600

for scaler in scalers:
        # для того чтобы следить за выполнением цикла введем индикатор
        print('current scaler: ', scaler)

         # сформируем список для заполнения данными текушего scaler
        current_scaler_data = [scaler]
        
        # выполним scaler преобразование
        X_train_s_balanced = scaler.fit_transform(X_train_balanced[list_ncorr_features])
        X_train_s = scaler.transform(X_train[list_ncorr_features])
        X_valid_s = scaler.transform(X_valid[list_ncorr_features])

        try:
                # для модуля func_time_out упакуем обучение модели в функцию try_func
                def try_func():
                        logistic_regression = linear_model.LogisticRegression(random_state=42, max_iter=10000)
                        return logistic_regression.fit(X_train_s_balanced,y_train_balanced)
                    
                # фиксируем время начала
                start_time = time.time()
                
                # запускаем обучение модели с ограничением по времени
                logistic_regression = func_timeout.func_timeout(time_out, try_func)
                
                # фиксируем время окончания обучения
                end_time = time.time()
                
                # запишем время обучения модели
                current_scaler_data.append(round(end_time-start_time))

                # Делаем предсказание для обучающей и валидационной выборки
                y_valid_lr_pred = logistic_regression.predict(X_valid_s)
                
                # для метрик ROC AUC делаем предсказание модели для обучающей и валидационной выборки  
                # в виде вероятности принадлежности к классу 1 
                y_train_lr_pred_proba = logistic_regression.predict_proba(X_train_s)[:,1]
                y_valid_lr_pred_proba = logistic_regression.predict_proba(X_valid_s)[:,1]

                #Считаем метрики ROC AUC yна обуающем и валидационном наборе и добавляем их в спискок
                current_scaler_data.append(round(metrics.roc_auc_score(y_train, y_train_lr_pred_proba),3))
                current_scaler_data.append(round(metrics.roc_auc_score(y_valid, y_valid_lr_pred_proba),3))

                #Считаем метрики для класса 0 на валидационном наборе и добавляем их в список
                current_scaler_data.append(round(metrics.recall_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))
                current_scaler_data.append(round(metrics.precision_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))


                #Считаем метрики для класса 1 на валидационном набопе и добавляем их в список
                current_scaler_data.append(round(metrics.recall_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))
                current_scaler_data.append(round(metrics.precision_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))

                # добавляем полученные данные в список scalers_data
                scalers_data.append(current_scaler_data)

                # чтобы подстраховаться на случай ошибки : "The Kernel crashed while executing code"
                # сохраним в локальную папку удачную итерацию
                # для этого из полученных данных сформируем dataframe
                # из полученных данных сформируем dataframe
                scaler_pd =pd.DataFrame(
                        columns=['scaler','time_fit',
                                 'roc_train','roc_valid', 
                                 'recall_0','precision_0',
                                 'recall_1','precision_1'],
                        data = scalers_data)
                # сохраняем dataframe
                scaler_pd.to_csv('data_recovery/scaler_pd.csv',index=False)
               
                # очистим output от прошедшей итерации
                clear_output()

        except func_timeout.FunctionTimedOut:
                # если время обучения привышает лимит
                # запишем в current_scaler_data соотвествующую данные
                current_scaler_data = [scaler,'>30m',0,0,0,0,0,0]
                # добавляем полученные данные в список scalers_data
                scalers_data.append(current_scaler_data)

                # чтобы подстраховаться на случай ошибки : "The Kernel crashed while executing code"
                # сохраним в локальную папку удачную итерацию
                # для этого из полученных данных сформируем dataframe
                # из полученных данных сформируем dataframe
                scaler_pd =pd.DataFrame(
                        columns=['scaler','time_fit',
                                 'roc_train','roc_valid', 
                                 'recall_0','precision_0',
                                 'recall_1','precision_1'],
                        data = scalers_data)
                # сохраняем dataframe
                scaler_pd.to_csv('data_recovery/scaler_pd.csv',index=False)
                
                # очистим output от прошедшей итерации
                clear_output()
                # переходим к следующему scaler
                pass
# очистим output
clear_output()

# сформируем данные соотвествующие лучщему ROC AUC на валидацинном наборе
best_roc_valid_data = scaler_pd[scaler_pd['roc_valid']==scaler_pd['roc_valid'].max()]

# выведем лучший результат на валидационном наборе
print('result:')
print('best scaler on valid:',best_roc_valid_data.iloc[0]['scaler'])
print('ROC AUC on train:', best_roc_valid_data.iloc[0]['roc_train'])
print('ROC AUC on valid:', best_roc_valid_data.iloc[0]['roc_valid'])
print('Time fit for best scaler:', best_roc_valid_data.iloc[0]['time_fit'],' seconds')

# выведем полученный dataframe
scaler_pd

# time: 

##### <span style="color:MediumSlateBlue"> Анализ влияния solver оптимизатора на качество и скорость обучения модели</span>

Проверим качество и скорость сходимости модели при разных solver  
Проверку осуществим через цикл  
Чтобы модель не сходилась бесконечно с помощью модуля func_timeout  
поставим ограничение времени схождения в 10 минут

In [ ]:
# подготовим данные для обучения модели

# обьявим scaler
scaler = StandardScaler()
# выполним scaler преобразование
X_train_s_balanced = scaler.fit_transform(X_train_balanced[list_ncorr_features])
X_train_s = scaler.transform(X_train[list_ncorr_features])
X_valid_s = scaler.transform(X_valid[list_ncorr_features])

# time: 10s

In [ ]:
# Зададим список solvers
solvers_list = ['lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga']

# сформируем список для заполнения
solvers_data = []

# установим ограничение на время обучения модели в 180 секунд (3 минут)
time_out = 180

for solver in solvers_list:
        # для того чтобы следить за выполнением цикла введем индикатор
        print('current solver: ', solver)

         # сформируем список для заполнения данными текушего solver
        current_solver_data = [solver]
        
        try:
                # для модуля func_time_out упакуем обучение модели в функцию try_func
                def try_func():
                        logistic_regression = linear_model.LogisticRegression(solver= solver,
                                                                  random_state=42, 
                                                                  max_iter=10000)
                        return logistic_regression.fit(X_train_s_balanced,y_train_balanced)
                    
                # фиксируем время начала
                start_time = time.time()
                
                # запускаем обучение модели с ограничением по времени
                logistic_regression = func_timeout.func_timeout(time_out, try_func)
                
                # фиксируем время окончания обучения
                end_time = time.time()
                
                # запишем время обучения модели
                current_solver_data.append(round(end_time-start_time))

                # Делаем предсказание для обучающей и валидационной выборки
                y_valid_lr_pred = logistic_regression.predict(X_valid_s)
                
                # для метрик ROC AUC делаем предсказание модели для обучающей и валидационной выборки  
                # в виде вероятности принадлежности к классу 1 
                y_train_lr_pred_proba = logistic_regression.predict_proba(X_train_s)[:,1]
                y_valid_lr_pred_proba = logistic_regression.predict_proba(X_valid_s)[:,1]

                #Считаем метрики ROC AUC yна обуающем и валидационном наборе и добавляем их в спискок
                current_solver_data.append(round(metrics.roc_auc_score(y_train, y_train_lr_pred_proba),3))
                current_solver_data.append(round(metrics.roc_auc_score(y_valid, y_valid_lr_pred_proba),3))

                #Считаем метрики для класса 0 на валидационном наборе и добавляем их в список
                current_solver_data.append(round(metrics.recall_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))
                current_solver_data.append(round(metrics.precision_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))


                #Считаем метрики для класса 1 на валидационном набопе и добавляем их в список
                current_solver_data.append(round(metrics.recall_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))
                current_solver_data.append(round(metrics.precision_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))

                # запишем данные полученные на одной итерации
                solvers_data.append(current_solver_data)

                # чтобы подстраховаться на случай ошибки : "The Kernel crashed while executing code"
                # сохраним в локальную папку удачную итерацию
                # для этого из полученных данных сформируем dataframe
                # из полученных данных сформируем dataframe
                solvers_pd =pd.DataFrame(
                        columns=['solver','time_fit',
                                 'roc_train','roc_valid', 
                                 'recall_0','precision_0',
                                 'recall_1','precision_1'],
                        data = solvers_data)
                # сохраняем dataframe
                solvers_pd.to_csv('data_recovery/solvers_pd.csv',index=False)
               
                # очистим output от прошедшей итерации
                clear_output()

        except func_timeout.FunctionTimedOut:
                # если время обучения привышает лимит
                # запишем в current_solver_data соотвествующую данные
                current_solver_data = [solver,'>3m',0,0,0,0,0,0]
                # добавляем полученные данные в список solvers_data
                solvers_data.append(current_solver_data)

                # чтобы подстраховаться на случай ошибки : "The Kernel crashed while executing code"
                # сохраним в локальную папку удачную итерацию
                # для этого из полученных данных сформируем dataframe
                # из полученных данных сформируем dataframe
                solvers_pd =pd.DataFrame(
                        columns=['solver','time_fit',
                                 'roc_train','roc_valid', 
                                 'recall_0','precision_0',
                                 'recall_1','precision_1'],
                        data = solvers_data)
                # сохраняем dataframe
                solvers_pd.to_csv('data_recovery/solvers_pd.csv',index=False)
                
                # очистим output от прошедшей итерации
                clear_output()
                # переходим к следующему solver
                pass
# очистим output
clear_output()

# сформируем данные соотвествующие лучщему ROC AUC на валидацинном наборе
best_roc_valid_data = solvers_pd[solvers_pd['roc_valid']==solvers_pd['roc_valid'].max()]

# выведем лучший результат на валидационном наборе
print('result:')
print('best solver on valid:',best_roc_valid_data.iloc[0]['solver'])
print('ROC AUC on train:', best_roc_valid_data.iloc[0]['roc_train'])
print('ROC AUC on valid:', best_roc_valid_data.iloc[0]['roc_valid'])
print('Time fit for best solver:', best_roc_valid_data.iloc[0]['time_fit'],' seconds')

# выведем полученный dataframe
solvers_pd

# time: 2m 35s

##### <span style="color:MediumSlateBlue">Подбор гиперпараметров модели</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'relative'
count_features = dict_spaces[feature_space]

# для работы функции corr_transform_to_force
# подгрузим DataFrame с матрицей корреляции признаков
corr_matrix = fp.ParquetFile('base_models/data/stat/corr_matrix_'+feature_space).to_pandas()

# для выбора sceler преобразвания нам понадобится словарь
dict_scalers ={
    'MinMaxScaler': MinMaxScaler(),
    'RobustScaler':RobustScaler(),
    'StandardScaler':StandardScaler()}

# настроим оптимизацию гипер параметров
def optuna_lg(trial):
  # задаем пространства поиска гиперпараметров
  solver = trial.suggest_categorical("solver", ['lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky'])
  C = trial.suggest_float('C',0.01,1)
  class_0_weight = trial.suggest_float('class_0_weight',0.2,0.6)
  class_1_weight = trial.suggest_float('class_1_weight',0.2,0.6)
  threshold = trial.suggest_float('threshold',0.6,1)
  class_1_percent = trial.suggest_float('class_1_percent',0.01,0.3)
  scaler = trial.suggest_categorical("scaler", ['MinMaxScaler','RobustScaler','StandardScaler'])
  random_state = trial.suggest_int('random_state', 1, 1000000)


  # создаем модель
  optuna_log_reg = linear_model.LogisticRegression(
      solver=solver,
      C=C,
      class_weight={0:class_0_weight,1:class_1_weight},
      random_state=random_state,
      max_iter=10000)

  # с помощью функции corr_transform_to_force получим
  # список не коррелируемых признаков
  list_ncorr_features = corr_transform_to_force(corr_matrix,threshold=threshold)[1]

  # обьявим scaler
  scaler = dict_scalers[scaler]

  # сформируем данные для анализа сбалансированности обучающих данных
  X_train_pd = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
  y_train_pd = pd.read_csv('target/target_train.csv')

  # поготовим данные y_train_pd к работе с функцией class_1_percent_samples
  y_train_pd.set_index('id',drop=False,inplace=True)

  # подготовим данные для обучения модели
  # с помощью функции class_1_percent_samples зададим долю 
  # класса 1
  list_c1_percent_id = class_1_percent_samples(y_train_pd,class_1_percent,random_state=random_state)[:1000000]

  # подготовим данные для обучения
  X_train_s_balanced = scaler.fit_transform(X_train_pd.loc[list_c1_percent_id])
  y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

  # освободим память от "тяжелых" и ненужных файлов
  del X_train_pd, y_train_pd
  gc.collect()

  # я не воспользовался параметром timeout метода optimize потому, что  
  # optimize отставливает поиск параметров, если время обучения 
  # превышает timeout. Мне нужно чтобы поиск параметров продолжился.
  try:
    # для модуля func_time_out упакуем обучение модели в функцию try_func
    def try_func():
            return optuna_log_reg.fit(X_train_s_balanced, y_train_balanced)
    # обучаем модель с ограничением по времени 10 минут (600 секунд)
    optuna_log_reg = func_timeout.func_timeout(600, try_func)

    # удаляем крупные файлы чтобы высвободить память 
    del X_train_s_balanced, y_train_balanced
    gc.collect()

    # загружаем тестовые наборы
    # сформируем данные для анализа сбалансированности обучающих данных
    X_train = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
    X_valid = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_valid').to_pandas()[list_ncorr_features]
    y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
    y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()

    # подготовим данные для проверки модели
    X_train_s = scaler.transform(X_train)
    X_valid_s = scaler.transform(X_valid)

    # удаляем крупные файлы чтобы высвободить память 
    del X_train, X_valid
    gc.collect()

    # делаем предсказание на обучающем и валидационном наборе и считаем метрики
    roc_train = metrics.roc_auc_score(y_train, optuna_log_reg.predict_proba(X_train_s)[:,1])
    roc_valid = metrics.roc_auc_score(y_valid, optuna_log_reg.predict_proba(X_valid_s)[:,1])
    f1_score = metrics.f1_score(y_valid, optuna_log_reg.predict(X_valid_s))
    recall_1 = metrics.recall_score(y_valid, optuna_log_reg.predict(X_valid_s),average=None,zero_division=0)[1]
    precision_1 = metrics.precision_score(y_valid, optuna_log_reg.predict(X_valid_s),average=None,zero_division=0)[1]

    # удаляем крупные файлы чтобы высвободить память 
    del X_train_s, X_valid_s, y_train, y_valid
    gc.collect()

  except func_timeout.FunctionTimedOut:
    # удаляем крупные файлы чтобы высвободить память 
    del X_train_s_balanced, y_train_balanced
    gc.collect()
    
    # фиксируем пустые значения метрик
    roc_train = 0
    roc_valid = 0
    f1_score = 0
    recall_1 = 0
    precision_1 = 0
    pass

  return roc_train, roc_valid, f1_score, recall_1, precision_1

In [ ]:
# cоздаем объект исследования
# чтобы модель не переобучалась минимизируем roc_train 
# и максимизируем roc_valid
optuna_study_lg = optuna.create_study(study_name='LogisticRegression_'+feature_space+'_stat', 
                               directions=['maximize','maximize','maximize','maximize','maximize'], 
                               sampler=optuna.samplers.TPESampler(),
                               pruner='Hyperband',
                               storage='sqlite:///optuna_studies.db',
                               load_if_exists=True)
# optuna.delete_study(study_name='LogisticRegression_'+feature_space+'_stat', storage='sqlite:///optuna_studies.db')

In [ ]:
# ищем лучшую комбинацию гиперпараметров n_trials раз
optuna_study_lg.optimize(optuna_lg, n_trials=300)

In [ ]:
# из полученного результат соврмируем Data Frame
optuna_study_lg_pd = optuna_study_lg.trials_dataframe().dropna()
# переименуем столбы
optuna_study_lg_pd.rename(columns={
    'values_0': 'roc_train',
    'values_1': 'roc_valid',
    'values_2': 'f1_score',
    'values_3': 'recall_1',
    'values_4': 'precision_1'
},inplace=True)
# Выделим время потраченное на обучение модели
optuna_study_lg_pd.tail()

In [ ]:
# покаждем статистику обучения
print('Максимальное значение метрики ROC AUC на валидационном наборе:',round(optuna_study_lg_pd['roc_valid'].max(),3))
print('Среднее значение метрики ROC AUC на валидационном наборе:',round(optuna_study_lg_pd['roc_valid'].mean(),3))
print('Максимальное значение метрики f1_score на валидационном наборе:',round(optuna_study_lg_pd['f1_score'].max(),3))
print('Среднее значение метрики f1_score на валидационном наборе:',round(optuna_study_lg_pd['f1_score'].mean(),3))
print('Максимальное значение метрики recall_1 на валидационном наборе:',round(optuna_study_lg_pd['recall_1'].max(),3))
print('Среднее значение метрики recall_1 на валидационном наборе:',round(optuna_study_lg_pd['recall_1'].mean(),3))
print('Максимальное значение метрики precision_1 на валидационном наборе:',round(optuna_study_lg_pd['precision_1'].max(),3))
print('Среднее значение метрики precision_1 на валидационном наборе:',round(optuna_study_lg_pd['precision_1'].mean(),3))

In [ ]:
# построим график важности гиперпарметров
optuna.visualization.plot_param_importances(optuna_study_lg, target_name='Score')

In [ ]:
# Построим зависимость метрики precision_1
fig = px.scatter(
    data_frame=optuna_study_lg_pd,
    x='precision_1', #ось абсцисс
    y=['f1_score', 'recall_1'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision от recall на классе 1', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision',
    yaxis_title='Величина метрик recall и f1-score')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики precision_1 от гипер параметров

fig = px.scatter(
    data_frame=optuna_study_lg_pd,
    x='precision_1', #ось абсцисс
    y=['params_C', 'params_class_0_weight', 
       'params_class_1_weight','params_class_1_percent',
       'params_threshold'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики f1_score от гипер параметров

fig = px.scatter(
    data_frame=optuna_study_lg_pd,
    x='f1_score', #ось абсцисс
    y=['params_C', 'params_class_0_weight', 
       'params_class_1_weight','params_class_1_percent',
       'params_threshold'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость f1-score от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики f1-score',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики roc_valid от гипер параметров

fig = px.scatter(
    data_frame=optuna_study_lg_pd,
    x='roc_valid', #ось абсцисс
    y=['params_C', 'params_class_0_weight', 
       'params_class_1_weight','params_class_1_percent',
       'params_threshold'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость ROC AUC на валидационном наборе от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики ROC AUC',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Визуализируем метрики полученные при optuna оптимизации
fig = px.scatter(
    data_frame=optuna_study_lg_pd[(optuna_study_lg_pd['precision_1']>0.08)
                                  & (optuna_study_lg_pd['recall_1']>0.08) 
                                  & (optuna_study_lg_pd['roc_valid']>0.8*optuna_study_lg_pd['roc_valid'].max())],
    x='number', #ось абсцисс
    y=['roc_train','roc_valid','recall_1','precision_1'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость качества модели от trials optuna', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='trials optuna',
    yaxis_title='Величина метрики')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

<span style="color:Blue">

Вывод:  

Их всех выбираем точку $number =234$.   
В этой точке самые оптимальные значения $recall$ и $precision$.  
Посмотрим каким значениям гиперпараметров соотвествует выбранная точка.

In [ ]:
# определим номер лучшге варианта
best_optuna_number = 234

# сформируем словарь лучших гипирпарметров RandomForestClassifier
best_param_lr = {
    'solver' : optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_solver'].iloc[0],
    'C' : round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_C'].iloc[0],3),
    'class_weight' : {0:round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_class_0_weight'].iloc[0],3),
                      1:round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_class_1_weight'].iloc[0],3)},
    }

# создадим перменные
best_threshold = optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_threshold'].iloc[0]
best_scaler = optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_scaler'].iloc[0]
best_class_1_percent = optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_class_1_percent'].iloc[0]
best_random_state = optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_random_state'].iloc[0]

# Выведем принятые наилучшие праметры
print('best solver:',best_param_lr['solver'])
print('best C:',best_param_lr['C'])
print('best class 0 weight:',best_param_lr['class_weight'][0])
print('best class 1 weight:',best_param_lr['class_weight'][1])

print('best class 1 percent:',round(best_class_1_percent,3))
print('best scaler:',best_scaler)
print('best threshold:',round(best_threshold,3))
print('best best random state:',best_random_state)
print('time for best train:',round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['duration'].iloc[0].seconds/60),'minutes')

print()
print('ROC AUC на обучающем наборе:', round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['roc_train'].iloc[0],3))
print('ROC AUC на валидационном наборе:', round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['roc_valid'].iloc[0],3))
print('precision класса 1:', round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['precision_1'].iloc[0],3))
print('recall класса 1:', round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['recall_1'].iloc[0],3))


##### <span style="color:MediumSlateBlue">Обучение модели с лучшими параметрами</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'relative'
count_features = dict_spaces[feature_space]

# для работы функции corr_transform_to_force
# подгрузим DataFrame с матрицей корреляции признаков
corr_matrix = fp.ParquetFile('base_models/data/stat/corr_matrix_'+feature_space).to_pandas()

# подготовим данные
# для выбора sceler преобразвания нам понадобится словарь
dict_scalers ={
    'MinMaxScaler': MinMaxScaler(),
    'RobustScaler':RobustScaler(),
    'StandardScaler':StandardScaler(),
}

# с помощью функции corr_transform_to_force получим
# список не коррелируемых признаков
list_ncorr_features = corr_transform_to_force(corr_matrix,threshold=best_threshold)[1]

# обьявим scaler
scaler = dict_scalers[best_scaler]

# сформируем данные для анализа сбалансированности обучающих данных
X_train_pd = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
y_train_pd = pd.read_csv('target/target_train.csv')

# поготовим данные y_train_pd к работе с функцией class_1_percent_samples
y_train_pd.set_index('id',drop=False,inplace=True)

# подготовим данные для обучения модели
# с помощью функции class_1_percent_samples зададим долю 
# класса 1
list_c1_percent_id = class_1_percent_samples(y_train_pd,best_class_1_percent,random_state=best_random_state)[:1000000]

# подготовим данные для обучения
X_train_s_balanced = scaler.fit_transform(X_train_pd.loc[list_c1_percent_id])
y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

# освободим память от "тяжелых" и ненужных файлов
del X_train_pd, y_train_pd
gc.collect()


# обучаем модель LogisticRegression с наилучшеми параметрами
logistic_regression = linear_model.LogisticRegression(
        **best_param_lr,
        random_state=42,
        max_iter=10000)
logistic_regression.fit(X_train_s_balanced, y_train_balanced)

# удаляем крупные файлы чтобы высвободить память 
del X_train_s_balanced, y_train_balanced
gc.collect()

# загружаем тестовые наборы
# сформируем данные для анализа сбалансированности обучающих данных
X_train = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
X_valid = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_valid').to_pandas()[list_ncorr_features]
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()

# подготовим данные для проверки модели
X_train_s = scaler.transform(X_train)
X_valid_s = scaler.transform(X_valid)

# удаляем крупные файлы чтобы высвободить память 
del X_train, X_valid
gc.collect()

# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_lr_pred_proba = logistic_regression.predict_proba(X_train_s)[:,1]
y_valid_lr_pred_proba = logistic_regression.predict_proba(X_valid_s)[:,1]

# Делаем предсказание для валидационной выборки
y_valid_lr_pred = logistic_regression.predict(X_valid_s)

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_lr_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_lr_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_valid,y_valid_lr_pred,zero_division=0))

# time: 2m 30s

##### <span style="color:MediumSlateBlue">Анализ влияния порога классификации на качество модели</span>

In [ ]:
# чтобы проше обращаться к каждому обьекту в данных
# преобразуем y_valid_LR_pred к Series типу
y_valid_lr_pred_proba = pd.Series(y_valid_lr_pred_proba)

# формируем список из необходимых значений thresholds
thresholds = np.arange(0.01, 1, 0.01)

# сформируем списко для заполнения данными результатов анализа
threshold_data = []


for threshold in thresholds:
        # сформируем список для заполнения данными текушего threshold
        threshold_current_data = [threshold]

        #клиентов, для которых вероятность дефолта > threshold, относим к классу 1
        #в противном случае — к классу 0
        y_valid_lr_pred = y_valid_lr_pred_proba.apply(lambda x: 1 if x>threshold else 0)


        #Считаем метрики для класса 0 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.precision_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.f1_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))

        #Считаем метрики для класса 1 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.precision_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.f1_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))

        # добавляем полученные данные в список threshold_data
        threshold_data.append(threshold_current_data)

        # из полученных данных сформируем dataframe
        thresholds_pd =pd.DataFrame(
        columns=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'],
        data = threshold_data)

# time: 17s

In [ ]:
# Визуализируем метрики на валидационном наборе при различных threshold
fig_threshold = px.line(
    data_frame=thresholds_pd,
    x='threshold', #ось абсцисс
    y=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'], #ось ординат
)
fig_threshold.update_layout(
    title ={
        'text':'Зависимость качества модели от threshold', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Порог классификации threshold',
    yaxis_title='Величина метрики')
fig_threshold.update_xaxes(showspikes=True)
fig_threshold.update_yaxes(showspikes=True)

fig_threshold.show()

Метрика $precision$ показывает на сколько сильно ошиблась модель при определиние класса.  
Чем выше метрика, тем меньше ошиблась модель. 

$$precision = \frac{TP}{TP+FP}$$

Метрика $recall$ показывает способность модели найти класс.  
Чем выше метрика, тем выше способность модели находить класс.  

$$recall = \frac{TP}{TP+FN}$$

Метрика $f1-score$ показывает баланс между $precision$ и $recall$ 

$threshold$- порог класификации модели.  
Все объекты для которых $у_{\text{proba pred}} < threshold$, модель относит к класс 0.  
Соотвественно, если $у_{\text{proba pred}}  > threshold$ модель относит обьект к классу 1.  
Для идельаной модели $threshold$, является границей между областями класса 0 и класса 1.  

<span style="color:Blue">

Выводы по результам анализа:  

Для улучшения качества модели по метрике $ROC AUC$, нет смысла сдвигать порог   
классификации. Но зависимость метрик качества модели от $threshold$, может рассказать о  
спосбоности модели искать класс 1 и класс 0.

При фокусировке модели на классе 1 (более низкий $threshold$):  
- на классе 1 метрика $precision$ уменьшается, метрика $recall$ растет;  
- на классе 0 метрика $precision$ растет, метрика $recall$ уменьшается.  

При фокусировке модели на классе 0 (более высокий $threshold$):  
- на классе 1 метрика $precision$ увеличивается, метрика $recall$ уменьшается;  
- на классе 0 метрика $precision$ уменьшается, метрика $recall$ растет.  

Обьсняется тем, что в условия дисбаланса модель научилась хорошо определять класс 0 и плохо класс 1.  

При фокусировке модели на классе 1 (более низкий $threshold$), все больше обьектов отнесены к классу 1.   
Так как $precision$ класса 1 уменьшается, значит в области класса 1 стало больше обектов класса 0,  
что явлется закономерным для "идеальной" модели.  
Однако при этом, $recall$ класса 1 растет, значит в добавленных обьектах также присутвует класс 1.  
Это как раз указывает на то, что модель не научилась находить среди класса 0 класс 1. В области с     
высокой вероятностью класса 0 (более низкий $threshold$), находятся обьекты класса 1.  

Фокусировка на классе 0 (более высокий $threshold$), заставляет модель все больше обьектов из области класса 1  
относить к классу 0. При этом увеличение метрики $precision$ класса 1 говорит о том, что в области класса 1     
становится меньше 0, а значит модель умеет среди класса 1, находить класс 0.  
После  $threshold=0.64$ модель впринципе теряет способность отличать класс 1. У модели нет обьектов класса 1   
в которых она "уверена" на 80+%.

Обратим внимание на $threshold=0.52$, в этой точке находится баланс межу метриками качества модели:
$$precision_1 = recall_1 = \text{f1-score}_1 = 0.082$$
$$precision_0 = recall_0 = \text{f1-score}_0 = 0.965$$

Для модели с лучшим качеством эти две точки должны находится максимально близко к друг другу и иметь  
высокое значение метрик $precision_1$ и $recall_1$.
Для класса 0 это условие выполнется, что также говорит о том, что модель научилась искать этот класс.  
Для класса 1 это условие не выполнется, что также говорит о том, что модель плохо ищет класс 1.

##### <span style="color:MediumSlateBlue">Построение ROC-кривой выбранной модели на тестовом наборе</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'relative'
count_features = dict_spaces[feature_space]

In [ ]:
# сформируем данные для проверки модели
X_train = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
X_valid = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_valid').to_pandas()[list_ncorr_features]
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()
X_test = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_test').to_pandas()[list_ncorr_features]
y_test = pd.read_csv('target/target_test.csv')['flag'].to_numpy()

In [ ]:
# выполним scaler преобразование
X_train_s = scaler.transform(X_train)
X_valid_s = scaler.transform(X_valid)
X_test_s = scaler.transform(X_test)

print("Размеры обучающего набора",X_train_s.shape)
print("Размеры валидационного набора",X_valid_s.shape)
print("Размеры тестового набора",X_test_s.shape)
# time: 1m 43s

In [ ]:
# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_lr_pred_proba = logistic_regression.predict_proba(X_train_s)[:,1]
y_valid_lr_pred_proba = logistic_regression.predict_proba(X_valid_s)[:,1]
y_test_lr_pred_proba = logistic_regression.predict_proba(X_test_s)[:,1]

# Делаем предсказание для валидационной выборки
y_test_lr_pred = logistic_regression.predict(X_test_s)

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_lr_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_lr_pred_proba),3))
print('ROC AUC на тестовом наборе', round(metrics.roc_auc_score(y_test, y_test_lr_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_test,y_test_lr_pred,zero_division=0))

In [ ]:
# Для расчета значений TPR и FPR используем функцию roc_curve
fpr_train, tpr_train, _ = metrics.roc_curve(y_train, y_train_lr_pred_proba)
fpr_valid, tpr_valid, _ = metrics.roc_curve(y_valid, y_valid_lr_pred_proba)
fpr_test, tpr_test, _ = metrics.roc_curve(y_test, y_test_lr_pred_proba)

# рассчитаем площадь под кривой
auc_train = round(metrics.roc_auc_score(y_train, y_train_lr_pred_proba),3)
auc_valid = round(metrics.roc_auc_score(y_valid, y_valid_lr_pred_proba),3)
auc_test = round(metrics.roc_auc_score(y_test, y_test_lr_pred_proba),3)

trace_train = go.Scatter(x=fpr_train, y=tpr_train, mode='lines', name='AUC_train = %0.2f' % auc_train,
                   line=dict(color='red', width=2),
                   hovertemplate='train<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_train])

trace_valid = go.Scatter(x=fpr_valid, y=tpr_valid, mode='lines', name='AUC_valid = %0.2f' % auc_valid,
                   line=dict(color='green', width=2),
                   hovertemplate='valid<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_valid])

trace_test = go.Scatter(x=fpr_test, y=tpr_test, mode='lines', name='AUC_test = %0.2f' % auc_test,
                   line=dict(color='darkorange', width=2),
                   hovertemplate='test<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_test])


reference_line = go.Scatter(x=[0,1], y=[0,1], mode='lines', name='Reference Line',
                            line=dict(color='navy', width=2, dash='dash'))

fig_roc = go.Figure(data=[trace_train,trace_valid,trace_test,reference_line])

fig_roc.update_layout(
    title ={
        'text':'ROC-кривая', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 1350,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate')

fig_roc.update_xaxes(showspikes=True)
fig_roc.update_yaxes(showspikes=True)


fig_roc.show()

<span style="color:Blue">

Вывод по ROC-кривой:
1. Модель выдает сталибильное качество на всех наборах, а значит  
не переобучена - разброс ($variance$) не большой.  
2. Смещение ($bias$) модели относительно большое.  
На тестовом наборе полученно качество $ROC AUC = 0.6$.

#### <span style="color:MediumBlue">Построение модели Random Forest Classifier</span>

##### <span style="color:MediumSlateBlue">Baseline обучение модели RandomForestClassifier</span>

In [ ]:
# обучаем модель логистической регрессии
# чтобы модель пыталась найти связь во всех признаках
random_forest = RandomForestClassifier(random_state=42, n_jobs=-1)
random_forest.fit(X_train_balanced[list_ncorr_features],y_train_balanced)

# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_rf__pred_proba = random_forest.predict_proba(X_train[list_ncorr_features])[:,1]
y_valid_rf_pred_proba = random_forest.predict_proba(X_valid[list_ncorr_features])[:,1]

# Делаем предсказание для валидационной выборки
y_valid_rf_pred = random_forest.predict(X_valid[list_ncorr_features])

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_rf__pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_rf_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_valid,y_valid_rf_pred))

# time: 17s

##### <span style="color:MediumSlateBlue">Подбор гиперпараметров модели</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'relative'
count_features = dict_spaces[feature_space]

# для работы функции corr_transform_to_force
# подгрузим DataFrame с матрицей корреляции признаков
corr_matrix = fp.ParquetFile('base_models/data/stat/corr_matrix_'+feature_space).to_pandas()

# настроим оптимизацию гипер параметров
def optuna_rf(trial):
  # задаем пространства поиска гиперпараметров
  # задаем пространства поиска гиперпараметров
  n_estimators = trial.suggest_int('n_estimators', 40, 100,step = 5)
  criterion = trial.suggest_categorical('criterion',['gini', 'entropy', 'log_loss'])
  max_depth = trial.suggest_int('max_depth', 10, 30,step = 1)
  min_samples_split = trial.suggest_int('min_samples_split', 5, 15,step = 1)
  min_samples_leaf = trial.suggest_int('min_samples_leaf', 5, 15,step = 1)
  max_features = trial.suggest_float('max_features',0.4,0.8)
  max_samples = trial.suggest_float('max_samples',0.4,0.8)
  class_0_weight = trial.suggest_float('class_0_weight',0.1,1)
  class_1_weight = trial.suggest_float('class_1_weight',0.1,1)
  threshold = trial.suggest_float('threshold',0.4,0.8)
  class_1_percent = trial.suggest_float('class_1_percent',0.01,0.5)
  random_state = trial.suggest_int('random_state', 1, 1000000)


  # создаем модель
  optuna_random_forest = RandomForestClassifier(
      n_estimators = n_estimators, 
      criterion = criterion,
      max_depth = max_depth,
      min_samples_split = min_samples_split,  
      min_samples_leaf = min_samples_leaf,
      max_features=max_features,
      max_samples=max_samples,
      class_weight={0:class_0_weight,1:class_1_weight},
      n_jobs=-1,
      random_state=42)

  # с помощью функции corr_transform_to_force получим
  # список не коррелируемых признаков
  list_ncorr_features = corr_transform_to_force(corr_matrix,threshold=threshold)[1]

  # сформируем данные для анализа сбалансированности обучающих данных
  X_train_pd = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
  y_train_pd = pd.read_csv('target/target_train.csv')

  # поготовим данные y_train_pd к работе с функцией class_1_percent_samples
  y_train_pd.set_index('id',drop=False,inplace=True)

  # подготовим данные для обучения модели
  # с помощью функции class_1_percent_samples зададим долю 
  # класса 1
  list_c1_percent_id = class_1_percent_samples(y_train_pd,class_1_percent,random_state=random_state)[:1000000]

  # подготовим данные для обучения
  X_train_balanced = X_train_pd.loc[list_c1_percent_id]
  y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag']

  # освободим память от "тяжелых" и ненужных файлов
  del X_train_pd, y_train_pd
  gc.collect()

  # я не воспользовался параметром timeout метода optimize потому, что  
  # optimize отставливает поиск параметров, если время обучения 
  # превышает timeout. Мне нужно чтобы поиск параметров продолжился.
  try:
    # для модуля func_time_out упакуем обучение модели в функцию try_func
    def try_func():
            return optuna_random_forest.fit(X_train_balanced, y_train_balanced)
    # обучаем модель с ограничением по времени 10 минут (600 секунд)
    optuna_random_forest = func_timeout.func_timeout(600, try_func)

    # удаляем крупные файлы чтобы высвободить память 
    del X_train_balanced, y_train_balanced
    gc.collect()

    # загружаем тестовые наборы
    # сформируем данные для анализа сбалансированности обучающих данных
    X_train = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
    X_valid = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_valid').to_pandas()[list_ncorr_features]
    y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
    y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()
    
    # делаем предсказание на обучающем и валидационном наборе
    #Считаем метрики для класса 1 добавляем их в список
    roc_train = metrics.roc_auc_score(y_train, optuna_random_forest.predict_proba(X_train)[:,1])
    roc_valid = metrics.roc_auc_score(y_valid, optuna_random_forest.predict_proba(X_valid)[:,1])
    f1_score = metrics.f1_score(y_valid, optuna_random_forest.predict(X_valid))
    recall_1 = metrics.recall_score(y_valid, optuna_random_forest.predict(X_valid),average=None,zero_division=0)[1]
    precision_1 = metrics.precision_score(y_valid, optuna_random_forest.predict(X_valid),average=None,zero_division=0)[1]

    # удаляем крупные файлы чтобы высвободить память 
    del X_train, X_valid, y_train, y_valid
    gc.collect()

  except func_timeout.FunctionTimedOut:
    # удаляем крупные файлы чтобы высвободить память 
    del X_train_balanced, y_train_balanced
    gc.collect()
    
    # фиксируем пустые значения метрик
    roc_train = 0
    roc_valid = 0
    f1_score = 0
    recall_1 = 0
    precision_1 = 0
    pass

  return roc_train, roc_valid, f1_score, recall_1, precision_1

In [ ]:
# cоздаем объект исследования
optuna_study_rf = optuna.create_study(study_name= 'RandomForestClassifier_'+feature_space+'_stat', 
                               directions=['maximize','maximize','maximize','maximize','maximize'], 
                               sampler=optuna.samplers.TPESampler(),
                               pruner='Hyperband',
                               storage='sqlite:///optuna_studies.db',
                               load_if_exists=True)
# на случай удаления 
# optuna.delete_study(study_name='RandomForestClassifier_'+feature_space+'_stat', storage='sqlite:///optuna_studies.db')

In [ ]:
# ищем лучшую комбинацию гиперпараметров n_trials раз
optuna_study_rf.optimize(optuna_rf, n_trials=300)

In [ ]:
# из полученного результат соврмируем Data Frame
optuna_study_rf_pd = optuna_study_rf.trials_dataframe().dropna()
# переименуем столбы
optuna_study_rf_pd.rename(columns={
    'values_0': 'roc_train',
    'values_1': 'roc_valid',
    'values_2': 'f1_score',
    'values_3': 'recall_1',
    'values_4': 'precision_1'
},inplace=True)
optuna_study_rf_pd.tail()

In [ ]:
# покаждем статистику обучения
print('Максимальное значение метрики ROC AUC на валидационном наборе:',round(optuna_study_rf_pd['roc_valid'].max(),3))
print('Среднее значение метрики ROC AUC на валидационном наборе:',round(optuna_study_rf_pd['roc_valid'].mean(),3))
print('Максимальное значение метрики f1_score на валидационном наборе:',round(optuna_study_rf_pd['f1_score'].max(),3))
print('Среднее значение метрики f1_score на валидационном наборе:',round(optuna_study_rf_pd['f1_score'].mean(),3))
print('Максимальное значение метрики recall_1 на валидационном наборе:',round(optuna_study_rf_pd['recall_1'].max(),3))
print('Среднее значение метрики recall_1 на валидационном наборе:',round(optuna_study_rf_pd['recall_1'].mean(),3))
print('Максимальное значение метрики precision_1 на валидационном наборе:',round(optuna_study_rf_pd['precision_1'].max(),3))
print('Среднее значение метрики precision_1 на валидационном наборе:',round(optuna_study_rf_pd['precision_1'].mean(),3))

In [ ]:
# Построим зависимость метрики precision_1

fig = px.scatter(
    data_frame=optuna_study_rf_pd,
    x='precision_1', #ось абсцисс
    y=['recall_1','f1_score'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision от recall на классе 1', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision на классе 1',
    yaxis_title='Величина метрик recall и f1-score')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики precision_1 от гиперпараметров

fig = px.scatter(
    data_frame=optuna_study_rf_pd,
    x='precision_1', #ось абсцисс
    y=['params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight', 'params_max_depth',
       'params_max_features', 'params_max_samples', 'params_min_samples_leaf',
       'params_min_samples_split', 'params_n_estimators', 'params_threshold'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision класса 1 от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision на классе 1',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики f1_score от гиперпараметров

fig = px.scatter(
    data_frame=optuna_study_rf_pd,
    x='f1_score', #ось абсцисс
    y=['params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight', 'params_max_depth',
       'params_max_features', 'params_max_samples', 'params_min_samples_leaf',
       'params_min_samples_split', 'params_n_estimators', 'params_threshold'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость f1-score класса 1 от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики f1-score на классе 1',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики roc_valid от гиперпараметров

fig = px.scatter(
    data_frame=optuna_study_rf_pd,
    x='roc_valid', #ось абсцисс
    y=['params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight', 'params_max_depth',
       'params_max_features', 'params_max_samples', 'params_min_samples_leaf',
       'params_min_samples_split', 'params_n_estimators', 'params_threshold'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость ROC AUC на валидационном наборе от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики ROC AUV на валидационном наборе',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрик качества модели от number

fig = px.scatter(
    data_frame=optuna_study_rf_pd[(optuna_study_rf_pd['recall_1']>=0.08) & (optuna_study_rf_pd['precision_1']>0.08)],
    x='number', #ось абсцисс
    y=['roc_valid','roc_train','precision_1','recall_1'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость качества модели от trials optuna', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='trials optuna',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

<span style="color:Blue">

Вывод:  

Их всех выбираем точку $number =188$.   
Не смотря на, то что в этой точке модель подает признаки переобучеености,  
в этой точке оптимальные значения метрик $recall_1$ и $precision_1$, а  
также относительно высокая метрика $ROC AUC$.   
Посмотрим каким значениям гиперпараметров соотвествует выбраннаяточка.

In [ ]:
# определим номер лучшге варианта
best_optuna_number = 188

# сформируем словарь лучших гипирпарметров RandomForestClassifier
best_param_rf = {
    'n_estimators' : int(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_n_estimators'].iloc[0]),
    'criterion': optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_criterion'].iloc[0],
    'max_depth' : int(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_max_depth'].iloc[0]),
    'min_samples_split': int(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_min_samples_split'].iloc[0]),
    'min_samples_leaf' : int(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_min_samples_leaf'].iloc[0]),
    'max_features': optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_max_features'].iloc[0],
    'max_samples' : optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_max_samples'].iloc[0],
    'class_weight' : {0:optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_class_0_weight'].iloc[0],
                      1:optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_class_1_weight'].iloc[0]}
    }

# определим перменные для лучших значений параметров
best_threshold = optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_threshold'].iloc[0]
best_class_1_percent = optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_class_1_percent'].iloc[0]
best_random_state = optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_random_state'].iloc[0]


# Выведем принятые наилучшие праметры
print('best n estimators:',best_param_rf['n_estimators'])
print('best criterion:',best_param_rf['criterion'])
print('best max depth:',best_param_rf['max_depth'])
print('best min samples split:',best_param_rf['min_samples_split'])
print('best min samples leaf:',best_param_rf['min_samples_leaf'])
print('best max features(%):',round(best_param_rf['max_features'],3))
print('best max samples(%):',round(best_param_rf['max_samples'],3))
print('best class 0 weight:',round(best_param_rf['class_weight'][0],3))
print('best class 1 weight:',round(best_param_rf['class_weight'][1],3))

print('best class 1 percent:',round(best_class_1_percent,3))
print('best threshold:',round(best_threshold,3))
print('best random state:', best_random_state)
print('time for best train:',round(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['duration'].iloc[0].seconds/60),'minutes')

print()
print('ROC AUC на обучающем наборе:', round(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['roc_train'].iloc[0],3))
print('ROC AUC на валидационном наборе:', round(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['roc_valid'].iloc[0],3))
print('precision класса 1:', round(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['precision_1'].iloc[0],3))
print('recall класса 1:', round(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['recall_1'].iloc[0],3))

##### <span style="color:MediumSlateBlue">Обучение модели с лучшими параметрами</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'relative'
count_features = dict_spaces[feature_space]

# для работы функции corr_transform_to_force
# подгрузим DataFrame с матрицей корреляции признаков
corr_matrix = fp.ParquetFile('base_models/data/stat/corr_matrix_'+feature_space).to_pandas()

# с помощью функции corr_transform_to_force получим
# список не коррелируемых признаков
list_ncorr_features = corr_transform_to_force(corr_matrix,threshold=best_threshold)[1]

# сформируем данные для анализа сбалансированности обучающих данных
X_train_pd = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
y_train_pd = pd.read_csv('target/target_train.csv')

# поготовим данные y_train_pd к работе с функцией class_1_percent_samples
y_train_pd.set_index('id',drop=False,inplace=True)

# подготовим данные для обучения модели
# с помощью функции class_1_percent_samples зададим долю 
# класса 1
list_c1_percent_id = class_1_percent_samples(y_train_pd,best_class_1_percent,random_state=best_random_state)[:1000000]

# подготовим данные для обучения
X_train_balanced = X_train_pd.loc[list_c1_percent_id]
y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag']

# освободим память от "тяжелых" и ненужных файлов
del X_train_pd, y_train_pd
gc.collect()


# обучаем модель RandomForestClassifier с наилучшеми параметрами
random_forest = RandomForestClassifier(
        **best_param_rf,
        n_jobs=-1,
        random_state=42)
random_forest.fit(X_train_balanced, y_train_balanced)

# удаляем крупные файлы чтобы высвободить память 
del X_train_balanced, y_train_balanced
gc.collect()

# загружаем тестовые наборы
# сформируем данные для анализа сбалансированности обучающих данных
X_train = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
X_valid = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_valid').to_pandas()[list_ncorr_features]
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()

# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_rf_pred_proba = random_forest.predict_proba(X_train)[:,1]
y_valid_rf_pred_proba = random_forest.predict_proba(X_valid)[:,1]

# Делаем предсказание для валидационной выборки
y_valid_rf_pred = random_forest.predict(X_valid)

# удаляем крупные файлы чтобы высвободить память 
del X_train, X_valid
gc.collect()

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_rf_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_rf_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_valid,y_valid_rf_pred,zero_division=0))

# time: 5m 30s

##### <span style="color:MediumSlateBlue">Анализ влияния порога классификации на качество модели</span>

In [ ]:
# чтобы проше обращаться к каждому обьекту в данных
# преобразуем y_valid_LR_pred к Series типу
y_valid_rf_pred_proba = pd.Series(y_valid_rf_pred_proba)

# формируем список из необходимых значений thresholds
thresholds = np.arange(0.01, 1, 0.01)

# сформируем списко для заполнения данными результатов анализа
threshold_data = []


for threshold in thresholds:
        # сформируем список для заполнения данными текушего threshold
        threshold_current_data = [threshold]

        #клиентов, для которых вероятность дефолта > threshold, относим к классу 1
        #в противном случае — к классу 0
        y_valid_rf_pred = y_valid_rf_pred_proba.apply(lambda x: 1 if x>threshold else 0)


        #Считаем метрики для класса 0 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(y_valid, y_valid_rf_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.precision_score(y_valid, y_valid_rf_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.f1_score(y_valid, y_valid_rf_pred,average=None,zero_division=0)[0],3))

        #Считаем метрики для класса 1 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(y_valid, y_valid_rf_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.precision_score(y_valid, y_valid_rf_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.f1_score(y_valid, y_valid_rf_pred,average=None,zero_division=0)[1],3))

        # добавляем полученные данные в список threshold_data
        threshold_data.append(threshold_current_data)

        # из полученных данных сформируем dataframe
        thresholds_pd =pd.DataFrame(
        columns=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'],
        data = threshold_data)

# time: 17s

In [ ]:
# Визуализируем метрики на валидационном наборе при различных threshold
fig_threshold = px.line(
    data_frame=thresholds_pd,
    x='threshold', #ось абсцисс
    y=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'], #ось ординат
)
fig_threshold.update_layout(
    title ={
        'text':'Зависимость качества модели от threshold', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Порог классификации threshold',
    yaxis_title='Величина метрики')
fig_threshold.update_xaxes(showspikes=True)
fig_threshold.update_yaxes(showspikes=True)

fig_threshold.show()

Метрика $precision$ показывает на сколько сильно ошиблась модель при определиние класса.  
Чем выше метрика, тем меньше ошиблась модель. 

$$precision = \frac{TP}{TP+FP}$$

Метрика $recall$ показывает способность модели найти класс.  
Чем выше метрика, тем выше способность модели находить класс.  

$$recall = \frac{TP}{TP+FN}$$

Метрика $f1-score$ показывает баланс между $precision$ и $recall$ 

$threshold$- порог класификации модели.  
Все объекты для которых $у_{\text{proba pred}} < threshold$, модель относит к класс 0.  
Соотвественно, если $у_{\text{proba pred}}  > threshold$ модель относит обьект к классу 1.  
Для идельаной модели $threshold$, является границей между областями класса 0 и класса 1.  

<span style="color:Blue">

Выводы по результам анализа:  

Для улучшения качества модели по метрике $ROC AUC$, нет смысла сдвигать порог   
классификации. Но зависимость метрик качества модели от $threshold$, может рассказать о  
способности модели искать класс 1 и класс 0.

При фокусировке модели на классе 1 (более низкий $threshold$):  
- на классе 1 метрика $precision$ уменьшается, метрика $recall$ растет;  
- на классе 0 метрика $precision$ растет, метрика $recall$ уменьшается.  

При фокусировке модели на классе 0 (более высокий $threshold$):  
- на классе 1 метрика $precision$ увеличивается, метрика $recall$ уменьшается;  
- на классе 0 метрика $precision$ уменьшается, метрика $recall$ растет.  

Обьсняется тем, что в условия дисбаланса модель научилась хорошо определять класс 0 и плохо класс 1.  

При фокусировке модели на классе 1 (более низкий $threshold$), все больше обьектов отнесены к классу 1.   
Так как $precision$ класса 1 уменьшается, значит в области класса 1 стало больше обектов класса 0,  
что явлется закономерным для "идеальной" модели.  
Однако при этом, $recall$ класса 1 растет, значит в добавленных обьектах также присутвует класс 1.  
Это как раз указывает на то, что модель не нучилась находить среди класса 0 класс 1. В области с     
высокой вероятностью класса 0 (более низкий $threshold$), находятся обьекты класса 1.  

Фокусировка на классе 0 (более высокий $threshold$), заставляет модель все больше обьектов из области класса 1  
относить к классу 0. При этом увеличение метрики $precision$ класса 1 говорит о том, что в области класса 1     
становится меньше 0, а значит модель умеет среди класса 1, находить класс 0.  
После  $threshold=0.77$ модель впринципе теряет способность отличать класс 1. У модели нет обьектов класса 1   
в которых она "уверена" на 80+%.

Обратим внимание на $threshold=0.49$, в этой точке находится баланс межу метриками качества модели:
$$precision_1 = recall_1 = \text{f1-score}_1 = 0.09$$
$$precision_0 = recall_0 = \text{f1-score}_0 = 0.968$$

Для модели с лучшим качеством эти две точки должны находится максимально близко к друг другу и иметь   
высокое значение метрик $precision_1$ и $recall_1$.  
Для класса 0 это условие выполнется, что также говорит о том, что модель научилась искать этот класс.    
Для класса 1 это условие не выполнется, что также говорит о том, что модель плохо ищет класс 1.

##### <span style="color:MediumSlateBlue">Построение ROC-кривой выбранной модели на тестовом наборе</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'relative'
count_features = dict_spaces[feature_space]

In [ ]:
# сформируем данные для проверки модели
X_train = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
X_valid = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_valid').to_pandas()[list_ncorr_features]
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()
X_test = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_test').to_pandas()[list_ncorr_features]
y_test = pd.read_csv('target/target_test.csv')['flag'].to_numpy()

In [ ]:
print("Размеры обучающего набора",X_train.shape,y_train.shape)
print("Размеры валидационного набора",X_valid.shape,y_valid.shape)
print("Размеры тестового набора",X_test.shape,y_test.shape)
# time: 1m 43s

In [ ]:
# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_rf_pred_proba = random_forest.predict_proba(X_train)[:,1]
y_valid_rf_pred_proba = random_forest.predict_proba(X_valid)[:,1]
y_test_rf_pred_proba = random_forest.predict_proba(X_test)[:,1]

# Делаем предсказание для валидационной выборки
y_test_lr_pred = random_forest.predict(X_test)

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_rf_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_rf_pred_proba),3))
print('ROC AUC на тестовом наборе', round(metrics.roc_auc_score(y_test, y_test_rf_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_test,y_test_lr_pred,zero_division=0))

In [ ]:
# Для расчета значений TPR и FPR используем функцию roc_curve
fpr_train, tpr_train, _ = metrics.roc_curve(y_train, y_train_rf_pred_proba)
fpr_valid, tpr_valid, _ = metrics.roc_curve(y_valid, y_valid_rf_pred_proba)
fpr_test, tpr_test, _ = metrics.roc_curve(y_test, y_test_rf_pred_proba)

# рассчитаем площадь под кривой
auc_train = round(metrics.roc_auc_score(y_train, y_train_rf_pred_proba),3)
auc_valid = round(metrics.roc_auc_score(y_valid, y_valid_rf_pred_proba),3)
auc_test = round(metrics.roc_auc_score(y_test, y_test_rf_pred_proba),3)

trace_train = go.Scatter(x=fpr_train, y=tpr_train, mode='lines', name='AUC_train = %0.2f' % auc_train,
                   line=dict(color='red', width=2),
                   hovertemplate='train<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_train])

trace_valid = go.Scatter(x=fpr_valid, y=tpr_valid, mode='lines', name='AUC_valid = %0.2f' % auc_valid,
                   line=dict(color='green', width=2),
                   hovertemplate='valid<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_valid])

trace_test = go.Scatter(x=fpr_test, y=tpr_test, mode='lines', name='AUC_test = %0.2f' % auc_test,
                   line=dict(color='darkorange', width=2),
                   hovertemplate='test<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_test])


reference_line = go.Scatter(x=[0,1], y=[0,1], mode='lines', name='Reference Line',
                            line=dict(color='navy', width=2, dash='dash'))

fig_roc = go.Figure(data=[trace_train,trace_valid,trace_test,reference_line])

fig_roc.update_layout(
    title ={
        'text':'ROC-кривая', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 1350,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate')

fig_roc.update_xaxes(showspikes=True)
fig_roc.update_yaxes(showspikes=True)


fig_roc.show()

<span style="color:Blue">

Вывод по ROC-кривой:
1. Не смотря на то, что модель показывает, очень хорошую $ROC$ кривую на обучающем наборе,  
модель, так же выдает сталибильное качество на валидационном и тестовом наборах, а значит  
не переобучена - разброс ($variance$) не большой.
2. Смещение ($bias$) модели такое же как у модели $Logistic$ $Regression$ $Classifier$,  
обученной  на тех же данных.  
На тестовом наборе полученно качество $ROC AUC = 0.61$.

#### <span style="color:MediumBlue">Построение модели Gradient Boosting Classifier</span>

##### <span style="color:MediumSlateBlue">Baseline обучение модели Hist Gradient Boosting Classifier</span>

In [ ]:
# обучаем модель Gradient Boosting Classifier
gradient_boosting = HistGradientBoostingClassifier(random_state=42)
gradient_boosting.fit(X_train_balanced[list_ncorr_features],y_train_balanced)

# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_rf__pred_proba = gradient_boosting.predict_proba(X_train[list_ncorr_features])[:,1]
y_valid_rf_pred_proba = gradient_boosting.predict_proba(X_valid[list_ncorr_features])[:,1]

# Делаем предсказание для валидационной выборки
y_valid_rf_pred = gradient_boosting.predict(X_valid[list_ncorr_features])

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_rf__pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_rf_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_valid,y_valid_rf_pred))

# time: 30s

##### <span style="color:MediumSlateBlue">Подбор гиперпараметров модели</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'relative'
count_features = dict_spaces[feature_space]

# для работы функции corr_transform_to_force
# подгрузим DataFrame с матрицей корреляции признаков
corr_matrix = fp.ParquetFile('base_models/data/stat/corr_matrix_'+feature_space).to_pandas()

# настроим оптимизацию гипер параметров
def optuna_hgbc(trial):
  # задаем пространства поиска гиперпараметров
  learning_rate = trial.suggest_float('learning_rate',0.01,0.1)
  max_iter = trial.suggest_int('max_iter', 150, 250,step = 1)
  max_leaf_nodes = trial.suggest_int('max_leaf_nodes', 2, 60,step = 1)
  max_depth = trial.suggest_int('max_depth', 1, 10,step = 1)
  min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 60,step = 1)
  max_features = trial.suggest_float('max_features',0.6,1)
  l2_regularization = trial.suggest_float('l2_regularization',0.01,1)
  class_0_weight = trial.suggest_float('class_0_weight',0.01,1)
  class_1_weight = trial.suggest_float('class_1_weight',0.5,1)
  threshold = trial.suggest_float('threshold',0.7,1)
  class_1_percent = trial.suggest_float('class_1_percent',0.01,0.25)
  random_state = trial.suggest_int('random_state', 1, 1000000)

  # создаем модель
  optuna_gradient_boosting = HistGradientBoostingClassifier(
      learning_rate= learning_rate,
      max_iter = max_iter,
      max_leaf_nodes =max_leaf_nodes,
      max_depth = max_depth,
      min_samples_leaf = min_samples_leaf,
      max_features=max_features,
      l2_regularization = l2_regularization,
      class_weight={0:class_0_weight,1:class_1_weight},
      random_state=42)

  # с помощью функции corr_transform_to_force получим
  # список не коррелируемых признаков
  list_ncorr_features = corr_transform_to_force(corr_matrix,threshold=threshold)[1]

  # сформируем данные для анализа сбалансированности обучающих данных
  X_train_pd = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
  y_train_pd = pd.read_csv('target/target_train.csv')

  # поготовим данные y_train_pd к работе с функцией class_1_percent_samples
  y_train_pd.set_index('id',drop=False,inplace=True)

  # подготовим данные для обучения модели
  # с помощью функции class_1_percent_samples зададим долю 
  # класса 1
  list_c1_percent_id = class_1_percent_samples(y_train_pd,class_1_percent,random_state=random_state)[:1000000]

  # подготовим данные для обучения
  X_train_balanced = X_train_pd.loc[list_c1_percent_id]
  y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag']

  # освободим память от "тяжелых" и ненужных файлов
  del X_train_pd, y_train_pd
  gc.collect()

  # я не воспользовался параметром timeout метода optimize потому, что  
  # optimize отставливает поиск параметров, если время обучения 
  # превышает timeout. Мне нужно чтобы поиск параметров продолжился.
  try:
    # для модуля func_time_out упакуем обучение модели в функцию try_func
    def try_func():
            return optuna_gradient_boosting.fit(X_train_balanced,y_train_balanced)
    # обучаем модель с ограничением по времени 10 минут (600 секунд)
    optuna_gradient_boosting = func_timeout.func_timeout(600, try_func)

    # удаляем крупные файлы чтобы высвободить память 
    del X_train_balanced, y_train_balanced
    gc.collect()

        # загружаем тестовые наборы
    # сформируем данные для анализа сбалансированности обучающих данных
    X_train = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
    X_valid = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_valid').to_pandas()[list_ncorr_features]
    y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
    y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()
    
    # делаем предсказание на обучающем и валидационном наборе и считаем метрики
    roc_train = metrics.roc_auc_score(y_train, optuna_gradient_boosting.predict_proba(X_train)[:,1])
    roc_valid = metrics.roc_auc_score(y_valid, optuna_gradient_boosting.predict_proba(X_valid)[:,1])
    f1_score = metrics.f1_score(y_valid, optuna_gradient_boosting.predict(X_valid))
    recall_1 = metrics.recall_score(y_valid, optuna_gradient_boosting.predict(X_valid),average=None,zero_division=0)[1]
    precision_1 = metrics.precision_score(y_valid, optuna_gradient_boosting.predict(X_valid),average=None,zero_division=0)[1]


    # удаляем крупные файлы чтобы высвободить память 
    del X_train, X_valid, y_train, y_valid
    gc.collect()

  except func_timeout.FunctionTimedOut:
    # удаляем крупные файлы чтобы высвободить память 
    del X_train_balanced, y_train_balanced
    gc.collect()
    
    # фиксируем пустые значения метрик
    roc_train = 0
    roc_valid = 0
    f1_score = 0
    recall_1 = 0
    precision_1 = 0
    pass

  return roc_train, roc_valid, f1_score, recall_1, precision_1

In [ ]:
# cоздаем объект исследования
# чтобы модель не переобучалась минимизируем roc_train 
# и максимизируем roc_valid
optuna_study_hgbc = optuna.create_study(study_name='HistGradientBoostingClassifier_'+feature_space+'_stat', 
                               directions=['maximize','maximize','maximize','maximize','maximize'], 
                               sampler=optuna.samplers.TPESampler(),
                               pruner='Hyperband',
                               storage='sqlite:///optuna_studies.db',
                               load_if_exists=True)

# ну случай удаления обучения
# optuna.delete_study(study_name='HistGradientBoostingClassifier'+feature_space+'_stat', storage='sqlite:///optuna_studies.db')

In [ ]:
# ищем лучшую комбинацию гиперпараметров n_trials раз
optuna_study_hgbc.optimize(optuna_hgbc, n_trials=300)

In [ ]:
# из полученного результат соврмируем Data Frame
optuna_study_hgbc_pd = optuna_study_hgbc.trials_dataframe()
# переименуем столбы
optuna_study_hgbc_pd.rename(columns={
    'values_0': 'roc_train',
    'values_1': 'roc_valid',
    'values_2': 'f1_score',
    'values_3': 'recall_1',
    'values_4': 'precision_1'
},inplace=True)
optuna_study_hgbc_pd.tail(5)

In [ ]:
# покаждем статистику обучения
print('Максимальное значение метрики ROC AUC на валидационном наборе:',round(optuna_study_hgbc_pd['roc_valid'].max(),3))
print('Среднее значение метрики ROC AUC на валидационном наборе:',round(optuna_study_hgbc_pd['roc_valid'].mean(),3))
print('Максимальное значение метрики f1_score на валидационном наборе:',round(optuna_study_hgbc_pd['f1_score'].max(),3))
print('Среднее значение метрики f1_score на валидационном наборе:',round(optuna_study_hgbc_pd['f1_score'].mean(),3))
print('Максимальное значение метрики recall_1 на валидационном наборе:',round(optuna_study_hgbc_pd['recall_1'].max(),3))
print('Среднее значение метрики recall_1 на валидационном наборе:',round(optuna_study_hgbc_pd['recall_1'].mean(),3))
print('Максимальное значение метрики precision_1 на валидационном наборе:',round(optuna_study_hgbc_pd['precision_1'].max(),3))
print('Среднее значение метрики precision_1 на валидационном наборе:',round(optuna_study_hgbc_pd['precision_1'].mean(),3))

In [ ]:
# построим график важности гиперпарметров
optuna.visualization.plot_param_importances(optuna_study_hgbc, target_name='Score')

In [ ]:
# Построим зависимость метрики precision_1
fig = px.scatter(
    data_frame=optuna_study_hgbc_pd,
    x='precision_1', #ось абсцисс
    y=['f1_score','recall_1'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision от recall на классе 1', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision класса 1',
    yaxis_title='Величина метрик recall и f1-score класса 1')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики precision_1 от гиперпараметров
fig = px.scatter(
    data_frame=optuna_study_hgbc_pd,
    x='precision_1', #ось абсцисс
    y=['params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight', 'params_l2_regularization',
       'params_learning_rate', 'params_max_depth', 'params_max_features',
       'params_max_iter', 'params_max_leaf_nodes', 'params_min_samples_leaf',
       'params_threshold'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision класса 1 от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision класса 1',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики f1_score от гиперпараметров
fig = px.scatter(
    data_frame=optuna_study_hgbc_pd,
    x='f1_score', #ось абсцисс
    y=['params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight', 'params_l2_regularization',
       'params_learning_rate', 'params_max_depth', 'params_max_features',
       'params_max_iter', 'params_max_leaf_nodes', 'params_min_samples_leaf',
       'params_threshold'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость f1-score класса 1 от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики f1-score класса 1',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики roc_valid от гиперпараметров
fig = px.scatter(
    data_frame=optuna_study_hgbc_pd,
    x='roc_valid', #ось абсцисс
    y=['params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight', 'params_l2_regularization',
       'params_learning_rate', 'params_max_depth', 'params_max_features',
       'params_max_iter', 'params_max_leaf_nodes', 'params_min_samples_leaf',
       'params_threshold'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость ROC AUC на валидационном наборе от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики ROC AUC на валидационном наборе',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрик качества модели от number

fig = px.scatter(
    data_frame=optuna_study_hgbc_pd[(optuna_study_hgbc_pd['recall_1']>0.1)&(optuna_study_hgbc_pd['precision_1']>0.1)],
    x='number', #ось абсцисс
    y=['roc_train','roc_valid','recall_1','precision_1'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость качества модели от trials optuna', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='trials optuna',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

<span style="color:Blue">

Вывод:  

Их всех выбираем точку $number =127$.   
В этой точке оптимальные значения метрик $recall_1$ и $precision_1$, а  
также относительно высокая метрика $ROC AUC$.   
Посмотрим каким значениям гиперпараметров соотвествует выбранная точка.

In [ ]:
# определим номер лучшге варианта
best_optuna_number = 127

# сформируем словарь лучших гипирпарметров HistGradientBoostingClassifier
best_param_hgbc = {
    'learning_rate' : optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['params_learning_rate'].iloc[0],
    'max_iter' : optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['params_max_iter'].iloc[0],
    'max_leaf_nodes' : optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['params_max_leaf_nodes'].iloc[0],
    'max_depth' : optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['params_max_depth'].iloc[0],
    'min_samples_leaf' : optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['params_min_samples_leaf'].iloc[0],
    'max_features' : optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['params_max_features'].iloc[0],
    'l2_regularization' : optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['params_l2_regularization'].iloc[0],
    'class_weight' : {0: optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['params_class_0_weight'].iloc[0],
                      1: optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['params_class_1_weight'].iloc[0],}
    }

# определим перменные для лучших значений параметров
best_threshold = optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['params_threshold'].iloc[0]
best_class_1_percent = optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['params_class_1_percent'].iloc[0]
best_random_state = optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['params_random_state'].iloc[0]


# Выведем принятые наилучшие параметры
print('best learning rate:',round(best_param_hgbc['learning_rate'],3))
print('best max iter:',best_param_hgbc['max_iter'])
print('best max leaf nodes:',best_param_hgbc['max_leaf_nodes'])
print('best max depth:',best_param_hgbc['max_depth'])
print('best min samples leaf:',best_param_hgbc['min_samples_leaf'])
print('best max features:',round(best_param_hgbc['max_features'],3))
print('best l2 regularization:',round(best_param_hgbc['l2_regularization'],3))
print('best class 0 weight:',round(best_param_hgbc['class_weight'][0],3))
print('best class 1 weight:',round(best_param_hgbc['class_weight'][1],3))

print('best class 1 percent:',round(best_class_1_percent,3))
print('best threshold:',round(best_threshold,3))
print('time for best train:',round(optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['duration'].iloc[0].seconds/60),'minutes')

print()
print('ROC AUC на обучающем наборе:', round(optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['roc_train'].iloc[0],3))
print('ROC AUC на валидационном наборе:', round(optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['roc_valid'].iloc[0],3))
print('precision класса 1:', round(optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['precision_1'].iloc[0],3))
print('recall класса 1:', round(optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['recall_1'].iloc[0],3))

##### <span style="color:MediumSlateBlue">Обучение модели с лучшими параметрами</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'relative'
count_features = dict_spaces[feature_space]

# для работы функции corr_transform_to_force
# подгрузим DataFrame с матрицей корреляции признаков
corr_matrix = fp.ParquetFile('base_models/data/stat/corr_matrix_'+feature_space).to_pandas()

# с помощью функции corr_transform_to_force получим
# список не коррелируемых признаков
list_ncorr_features = corr_transform_to_force(corr_matrix,threshold=best_threshold)[1]

# сформируем данные для анализа сбалансированности обучающих данных
X_train_pd = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
y_train_pd = pd.read_csv('target/target_train.csv')

# поготовим данные y_train_pd к работе с функцией class_1_percent_samples
y_train_pd.set_index('id',drop=False,inplace=True)

# подготовим данные для обучения модели
# с помощью функции class_1_percent_samples зададим долю 
# класса 1
list_c1_percent_id = class_1_percent_samples(y_train_pd,best_class_1_percent,random_state=best_random_state)[:1000000]

# подготовим данные для обучения
X_train_balanced = X_train_pd.loc[list_c1_percent_id]
y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag']

# освободим память от "тяжелых" и ненужных файлов
del X_train_pd, y_train_pd
gc.collect()

# обучаем модель HistGradientBoostingClassifier с наилучшеми параметрами
gradient_boosting = HistGradientBoostingClassifier(
        **best_param_hgbc,
        random_state=42)
gradient_boosting.fit(X_train_balanced,y_train_balanced)

# удаляем крупные файлы чтобы высвободить память 
del X_train_balanced, y_train_balanced
gc.collect()

# сформируем данные для анализа сбалансированности обучающих данных
X_train = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
X_valid = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_valid').to_pandas()[list_ncorr_features]
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()

# делаем предсказание на обучающем и валидационном наборе и считаем метрики
y_train_gb_pred_proba = gradient_boosting.predict_proba(X_train)[:,1]
y_valid_gb_pred_proba = gradient_boosting.predict_proba(X_valid)[:,1]
y_valid_gb_pred = gradient_boosting.predict(X_valid)

# удаляем крупные файлы чтобы высвободить память 
del X_train, X_valid
gc.collect()

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_gb_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_gb_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_valid,y_valid_gb_pred,zero_division=0))

# time: 1m 40s

##### <span style="color:MediumSlateBlue">Анализ влияния порога классификации на качество модели</span>

In [ ]:
# чтобы проше обращаться к каждому обьекту в данных
# преобразуем y_valid_LR_pred к Series типу
y_valid_gb_pred_proba = pd.Series(y_valid_gb_pred_proba)

# формируем список из необходимых значений thresholds
thresholds = np.arange(0.01, 1, 0.01)

# сформируем списко для заполнения данными результатов анализа
threshold_data = []


for threshold in thresholds:
        # сформируем список для заполнения данными текушего threshold
        threshold_current_data = [threshold]

        #клиентов, для которых вероятность дефолта > threshold, относим к классу 1
        #в противном случае — к классу 0
        y_valid_gb_pred = y_valid_gb_pred_proba.apply(lambda x: 1 if x>threshold else 0)


        #Считаем метрики для класса 0 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(y_valid, y_valid_gb_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.precision_score(y_valid, y_valid_gb_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.f1_score(y_valid, y_valid_gb_pred,average=None,zero_division=0)[0],3))

        #Считаем метрики для класса 1 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(y_valid, y_valid_gb_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.precision_score(y_valid, y_valid_gb_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.f1_score(y_valid, y_valid_gb_pred,average=None,zero_division=0)[1],3))

        # добавляем полученные данные в список threshold_data
        threshold_data.append(threshold_current_data)

        # из полученных данных сформируем dataframe
        thresholds_pd =pd.DataFrame(
        columns=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'],
        data = threshold_data)

# time: 17s

In [ ]:
# Визуализируем метрики на валидационном наборе при различных threshold
fig_threshold = px.line(
    data_frame=thresholds_pd,
    x='threshold', #ось абсцисс
    y=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'], #ось ординат
)
fig_threshold.update_layout(
    title ={
        'text':'Зависимость качества модели от threshold', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Порог классификации threshold',
    yaxis_title='Величина метрики')
fig_threshold.update_xaxes(showspikes=True)
fig_threshold.update_yaxes(showspikes=True)

fig_threshold.show()

Метрика $precision$ показывает на сколько сильно ошиблась модель при определиние класса.  
Чем выше метрика, тем меньше ошиблась модель. 

$$precision = \frac{TP}{TP+FP}$$

Метрика $recall$ показывает способность модели найти класс.  
Чем выше метрика, тем выше способность модели находить класс.  

$$recall = \frac{TP}{TP+FN}$$

Метрика $f1-score$ показывает баланс между $precision$ и $recall$ 

$threshold$- порог класификации модели.  
Все объекты для которых $у_{\text{proba pred}} < threshold$, модель относит к класс 0.  
Соотвественно, если $у_{\text{proba pred}}  > threshold$ модель относит обьект к классу 1.  
Для идельаной модели $threshold$, является границей между областями класса 0 и класса 1.  

<span style="color:Blue">

Выводы по результам анализа:  

Для улучшения качества модели по метрике $ROC AUC$, нет смысла сдвигать порог   
классификации. Но зависимость метрик качества модели от $threshold$, может рассказать о  
способности модели искать класс 1 и класс 0.

При фокусировке модели на классе 1 (более низкий $threshold$):  
- на классе 1 метрика $precision$ уменьшается, метрика $recall$ растет;  
- на классе 0 метрика $precision$ растет, метрика $recall$ уменьшается.  

При фокусировке модели на классе 0 (более высокий $threshold$):  
- на классе 1 метрика $precision$ увеличивается, метрика $recall$ уменьшается;  
- на классе 0 метрика $precision$ уменьшается, метрика $recall$ растет.  

Обьсняется тем, что в условия дисбаланса модель научилась хорошо определять класс 0 и плохо класс 1.  

При фокусировке модели на классе 1 (более низкий $threshold$), все больше обьектов отнесены к классу 1.   
Так как $precision$ класса 1 уменьшается, значит в области класса 1 стало больше обектов класса 0,  
что явлется закономерным для "идеальной" модели.  
Однако при этом, $recall$ класса 1 растет, значит в добавленных обьектах также присутвует класс 1.  
Это как раз указывает на то, что модель не нучилась находить среди класса 0 класс 1. В области с     
высокой вероятностью класса 0 (более низкий $threshold$), находятся обьекты класса 1.  

Фокусировка на классе 0 (более высокий $threshold$), заставляет модель все больше обьектов из области класса 1  
относить к классу 0. При этом увеличение метрики $precision$ класса 1 говорит о том, что в области класса 1     
становится меньше 0, а значит модель умеет среди класса 1, находить класс 0.  
После  $threshold=0.88$ модель впринципе теряет способность отличать класс 1. У модели нет обьектов класса 1   
в которых она "уверена" на 90+%.


Обратим внимание на $threshold=0.51$, в этой точке находится баланс межу метриками качества модели:
$$precision_1 = recall_1 = \text{f1-score}_1 = 0.103$$
$$precision_0 = recall_0 = \text{f1-score}_0 = 0.97$$

Для модели с лучшим качеством эти две точки должны находится максимально близко к друг другу и иметь   
высокое значение метрик $precision_1$ и $recall_1$.  
Для класса 0 это условие выполнется, что также говорит о том, что модель научилась искать этот класс.    
Для класса 1 это условие не выполнется, что также говорит о том, что модель плохо ищет класс 1.

##### <span style="color:MediumSlateBlue">Построение ROC-кривой выбранной модели на тестовом наборе</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'relative'
count_features = dict_spaces[feature_space]

In [ ]:
# сформируем данные для проверки модели
X_train = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
X_valid = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_valid').to_pandas()[list_ncorr_features]
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()
X_test = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_test').to_pandas()[list_ncorr_features]
y_test = pd.read_csv('target/target_test.csv')['flag'].to_numpy()

In [ ]:
print("Размеры обучающего набора",X_train.shape,y_train.shape)
print("Размеры валидационного набора",X_valid.shape,y_valid.shape)
print("Размеры тестового набора",X_test.shape,y_test.shape)
# time: 1m 43s

In [ ]:
# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_gb_pred_proba = gradient_boosting.predict_proba(X_train)[:,1]
y_valid_gb_pred_proba = gradient_boosting.predict_proba(X_valid)[:,1]
y_test_gb_pred_proba = gradient_boosting.predict_proba(X_test)[:,1]

# Делаем предсказание для валидационной выборки
y_test_gb_pred = gradient_boosting.predict(X_test)

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_gb_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_gb_pred_proba),3))
print('ROC AUC на тестовом наборе', round(metrics.roc_auc_score(y_test, y_test_gb_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_test,y_test_gb_pred,zero_division=0))

In [ ]:
# Для расчета значений TPR и FPR используем функцию roc_curve
fpr_train, tpr_train, _ = metrics.roc_curve(y_train, y_train_gb_pred_proba)
fpr_valid, tpr_valid, _ = metrics.roc_curve(y_valid, y_valid_gb_pred_proba)
fpr_test, tpr_test, _ = metrics.roc_curve(y_test, y_test_gb_pred_proba)

# рассчитаем площадь под кривой
auc_train = round(metrics.roc_auc_score(y_train, y_train_gb_pred_proba),3)
auc_valid = round(metrics.roc_auc_score(y_valid, y_valid_gb_pred_proba),3)
auc_test = round(metrics.roc_auc_score(y_test, y_test_gb_pred_proba),3)

trace_train = go.Scatter(x=fpr_train, y=tpr_train, mode='lines', name='AUC_train = %0.2f' % auc_train,
                   line=dict(color='red', width=2),
                   hovertemplate='train<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_train])

trace_valid = go.Scatter(x=fpr_valid, y=tpr_valid, mode='lines', name='AUC_valid = %0.2f' % auc_valid,
                   line=dict(color='green', width=2),
                   hovertemplate='valid<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_valid])

trace_test = go.Scatter(x=fpr_test, y=tpr_test, mode='lines', name='AUC_test = %0.2f' % auc_test,
                   line=dict(color='darkorange', width=2),
                   hovertemplate='test<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_test])


reference_line = go.Scatter(x=[0,1], y=[0,1], mode='lines', name='Reference Line',
                            line=dict(color='navy', width=2, dash='dash'))

fig_roc = go.Figure(data=[trace_train,trace_valid,trace_test,reference_line])

fig_roc.update_layout(
    title ={
        'text':'ROC-кривая', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 1350,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate')

fig_roc.update_xaxes(showspikes=True)
fig_roc.update_yaxes(showspikes=True)


fig_roc.show()

<span style="color:Blue">

Вывод по ROC-кривой:
1. Модель выдает сталибильное качество на валидационном и тестовом наборах, а значит  
не переобучена - разброс ($variance$) не большой. Хотя, качество модели на обучающем наборе,  
заметно лучше.
2. Смещение ($bias$) модели чуть лучше, чем у модели $Random$ $Forest$ $Classifier$,  
обученной на тех же данных.    
На тестовом наборе полученно качество $ROC AUC = 0.63$, против $ROC AUC = 0.61$.

### <span style="color:RoyalBlue">Построение модели на сбалансированных данных подпрострнства payments stat</span>

#### <span style="color:MediumBlue">Формирование данных для обучения</span>

Анализ исходных данных, в частности target, показал что представленные данные   
имееют перекос: 96% клиентов у которых не случился дефолт (flag = 0),    
против 4 % - для которых произошел дефолт (flag = 1).  
С помощью функции class_1_percent_samples, сформируем сбаланисрованную выборку  
для обучения моделей.
За основу сбалансировных данных возьмем данные клиентов с дефолтом  
и к ним будем добирать необходимое количество клиентов до сбалансированности. 


Функция class_1_percent_samples принемает две переменные:
- target_data - массив с id и целевой переменной
- class_1_percent - процент класса 1 в результирующем массиве

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'payments'
count_features = dict_spaces[feature_space]

In [ ]:
# сформируем данные для анализа сбалансированности обучающих данных
X_train_pd = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()
y_train_pd = pd.read_csv('target/target_train.csv')

In [ ]:
# поготовим данные y_train_pd к работе с функцией class_1_percent_samples
y_train_pd.set_index('id',drop=False,inplace=True)
y_train_pd.head(4)

In [ ]:
# взглянем на сблансиорованность данных в y_train
y_train_pd['flag'].value_counts(normalize=True)

In [ ]:
# сбалансируем данные в y_train_pd
list_c1_percent_id = class_1_percent_samples(y_train_pd,0.5)
# list_c1_percent_features - список 'id' из y_train_pd соотвествующих  
# заданному проценту класса 1
len(list_c1_percent_id)

In [ ]:
# проверим сбалансированность полученного y_train
y_train_pd.loc[list_c1_percent_id]['flag'].value_counts(normalize=True)

В отличие от данных в *transform_data_torow*, где модели показываеются последние N  
операций клиента, в данные в *transform_data_stat* сфомированы из различных статистических   
характеристик ряда определяющего клиентскую историю в банке. 

Соотвественно, если в данных *transform_data_torow* была сильная корреляция между признаками,  
то я не смог бы ее убрать, иначе потярется смысл "показать последовательный ряд в историии клиента".  
Признаки в *transform_data_stat* не имеют такой особенности. Поэтому, необходимо проанализировать  
данные в *transform_data_stat* и, в случае сильной коррреляции признаков, устранить ее.

In [ ]:
# подгрузим DataFrame с матрицей корреляции признаков
corr_matrix = fp.ParquetFile('base_models/data/stat/corr_matrix_'+feature_space).to_pandas()
corr_matrix.shape

In [ ]:
# с помощью функции corr_transrom_to_power получим
# силу корреляции матрицы и список не коррелируемых признаков

# для начального анализа зададим порог силы корреляции, при котором признаки 
# будут отнесены к "сильно скоррелированными" равным 0.6
ncorr_matrix,list_ncorr_features,corr_power =corr_transform_to_force(corr_matrix,threshold=0.6)

# Выведем результаты преобразования
print('Исходное количество признаков: ', corr_matrix.shape[0])
print('Количество не коррелируемых признаков: ', len(list_ncorr_features))
# сила корреляции признаков
print('Отношение коррелируемых исходному количеству признаков (сила корреляции): ',corr_power)

In [ ]:
# подготовим данные для обучения
X_train_balanced = X_train_pd.loc[list_c1_percent_id]
y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag']
# сформируем данные для проверики модели
# сформируем данные для анализа сбалансированности обучающих данных
X_train = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()
y_train = pd.read_csv('target/target_train.csv')['flag']
X_valid = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_valid').to_pandas()
y_valid = pd.read_csv('target/target_valid.csv')['flag']

In [ ]:
# посмотрим на структуру данных в X_train_balanced
X_train_balanced.head(4)

In [ ]:
# посмотрим на структуру данных в y_train_balanced
y_train_balanced.head(4)

In [ ]:
# проверим что выборки действительно совпадают по id
X_train_balanced.index.to_list() == y_train_balanced.index.to_list()

#### <span style="color:MediumBlue">Построение модели Logistic Regression</span>

##### <span style="color:MediumSlateBlue">Baseline обучение модели LogisticRegression</span>

In [ ]:
# обучаем модель логистической регрессии
logistic_regression = linear_model.LogisticRegression(random_state=42, max_iter=10000)
logistic_regression.fit(X_train_balanced[list_ncorr_features],y_train_balanced)

# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_LR_pred_proba = logistic_regression.predict_proba(X_train[list_ncorr_features])[:,1]
y_valid_LR_pred_proba = logistic_regression.predict_proba(X_valid[list_ncorr_features])[:,1]

# Делаем предсказание для валидационной выборки
y_valid_LR_pred = logistic_regression.predict(X_valid[list_ncorr_features])

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_LR_pred_proba),3))
print('ROC AUC на тестовом наборе', round(metrics.roc_auc_score(y_valid, y_valid_LR_pred_proba),3))

print('Основные метрики на тестовом наборе:')
print(metrics.classification_report(y_valid,y_valid_LR_pred))

# time: 3m 5s

##### <span style="color:MediumSlateBlue">Анализ влияния scaler преобразования на качество и скорость схождения модели</span>

In [ ]:
# формируем список из необходимых scaler
scalers = [MinMaxScaler(),RobustScaler() ,StandardScaler()]

# сформируем списко для заполнения данными результатов анализа
scalers_data = []

# установим ограничение на время обучения модели в 600 секунд (10 минут)
time_out = 600

for scaler in scalers:
        # для того чтобы следить за выполнением цикла введем индикатор
        print('current scaler: ', scaler)

         # сформируем список для заполнения данными текушего scaler
        current_scaler_data = [scaler]
        
        # выполним scaler преобразование
        X_train_s_balanced = scaler.fit_transform(X_train_balanced[list_ncorr_features])
        X_train_s = scaler.transform(X_train[list_ncorr_features])
        X_valid_s = scaler.transform(X_valid[list_ncorr_features])

        try:
                # для модуля func_time_out упакуем обучение модели в функцию try_func
                def try_func():
                        logistic_regression = linear_model.LogisticRegression(random_state=42, max_iter=10000)
                        return logistic_regression.fit(X_train_s_balanced,y_train_balanced)
                    
                # фиксируем время начала
                start_time = time.time()
                
                # запускаем обучение модели с ограничением по времени
                logistic_regression = func_timeout.func_timeout(time_out, try_func)
                
                # фиксируем время окончания обучения
                end_time = time.time()
                
                # запишем время обучения модели
                current_scaler_data.append(round(end_time-start_time))

                # Делаем предсказание для обучающей и валидационной выборки
                y_valid_lr_pred = logistic_regression.predict(X_valid_s)
                
                # для метрик ROC AUC делаем предсказание модели для обучающей и валидационной выборки  
                # в виде вероятности принадлежности к классу 1 
                y_train_lr_pred_proba = logistic_regression.predict_proba(X_train_s)[:,1]
                y_valid_lr_pred_proba = logistic_regression.predict_proba(X_valid_s)[:,1]

                #Считаем метрики ROC AUC yна обуающем и валидационном наборе и добавляем их в спискок
                current_scaler_data.append(round(metrics.roc_auc_score(y_train, y_train_lr_pred_proba),3))
                current_scaler_data.append(round(metrics.roc_auc_score(y_valid, y_valid_lr_pred_proba),3))

                #Считаем метрики для класса 0 на валидационном наборе и добавляем их в список
                current_scaler_data.append(round(metrics.recall_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))
                current_scaler_data.append(round(metrics.precision_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))


                #Считаем метрики для класса 1 на валидационном набопе и добавляем их в список
                current_scaler_data.append(round(metrics.recall_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))
                current_scaler_data.append(round(metrics.precision_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))

                # добавляем полученные данные в список scalers_data
                scalers_data.append(current_scaler_data)

                # чтобы подстраховаться на случай ошибки : "The Kernel crashed while executing code"
                # сохраним в локальную папку удачную итерацию
                # для этого из полученных данных сформируем dataframe
                # из полученных данных сформируем dataframe
                scaler_pd =pd.DataFrame(
                        columns=['scaler','time_fit',
                                 'roc_train','roc_valid', 
                                 'recall_0','precision_0',
                                 'recall_1','precision_1'],
                        data = scalers_data)
                # сохраняем dataframe
                scaler_pd.to_csv('data_recovery/scaler_pd.csv',index=False)
               
                # очистим output от прошедшей итерации
                clear_output()

        except func_timeout.FunctionTimedOut:
                # если время обучения привышает лимит
                # запишем в current_scaler_data соотвествующую данные
                current_scaler_data = [scaler,'>30m',0,0,0,0,0,0]
                # добавляем полученные данные в список scalers_data
                scalers_data.append(current_scaler_data)

                # чтобы подстраховаться на случай ошибки : "The Kernel crashed while executing code"
                # сохраним в локальную папку удачную итерацию
                # для этого из полученных данных сформируем dataframe
                # из полученных данных сформируем dataframe
                scaler_pd =pd.DataFrame(
                        columns=['scaler','time_fit',
                                 'roc_train','roc_valid', 
                                 'recall_0','precision_0',
                                 'recall_1','precision_1'],
                        data = scalers_data)
                # сохраняем dataframe
                scaler_pd.to_csv('data_recovery/scaler_pd.csv',index=False)
                
                # очистим output от прошедшей итерации
                clear_output()
                # переходим к следующему scaler
                pass
# очистим output
clear_output()

# сформируем данные соотвествующие лучщему ROC AUC на валидацинном наборе
best_roc_valid_data = scaler_pd[scaler_pd['roc_valid']==scaler_pd['roc_valid'].max()]

# выведем лучший результат на валидационном наборе
print('result:')
print('best scaler on valid:',best_roc_valid_data.iloc[0]['scaler'])
print('ROC AUC on train:', best_roc_valid_data.iloc[0]['roc_train'])
print('ROC AUC on valid:', best_roc_valid_data.iloc[0]['roc_valid'])
print('Time fit for best scaler:', best_roc_valid_data.iloc[0]['time_fit'],' seconds')

# выведем полученный dataframe
scaler_pd

# time: 

##### <span style="color:MediumSlateBlue"> Анализ влияния solver оптимизатора на качество и скорость обучения модели</span>

Проверим качество и скорость сходимости модели при разных solver  
Проверку осуществим через цикл  
Чтобы модель не сходилась бесконечно с помощью модуля func_timeout  
поставим ограничение времени схождения в 10 минут

In [ ]:
# подготовим данные для обучения модели

# обьявим scaler
scaler = StandardScaler()
# выполним scaler преобразование
X_train_s_balanced = scaler.fit_transform(X_train_balanced[list_ncorr_features])
X_train_s = scaler.transform(X_train[list_ncorr_features])
X_valid_s = scaler.transform(X_valid[list_ncorr_features])

# time: 10s

In [ ]:
# Зададим список solvers
solvers_list = ['lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga']

# сформируем список для заполнения
solvers_data = []

# установим ограничение на время обучения модели в 180 секунд (3 минут)
time_out = 180

for solver in solvers_list:
        # для того чтобы следить за выполнением цикла введем индикатор
        print('current solver: ', solver)

         # сформируем список для заполнения данными текушего solver
        current_solver_data = [solver]
        
        try:
                # для модуля func_time_out упакуем обучение модели в функцию try_func
                def try_func():
                        logistic_regression = linear_model.LogisticRegression(solver= solver,
                                                                  random_state=42, 
                                                                  max_iter=10000)
                        return logistic_regression.fit(X_train_s_balanced,y_train_balanced)
                    
                # фиксируем время начала
                start_time = time.time()
                
                # запускаем обучение модели с ограничением по времени
                logistic_regression = func_timeout.func_timeout(time_out, try_func)
                
                # фиксируем время окончания обучения
                end_time = time.time()
                
                # запишем время обучения модели
                current_solver_data.append(round(end_time-start_time))

                # Делаем предсказание для обучающей и валидационной выборки
                y_valid_lr_pred = logistic_regression.predict(X_valid_s)
                
                # для метрик ROC AUC делаем предсказание модели для обучающей и валидационной выборки  
                # в виде вероятности принадлежности к классу 1 
                y_train_lr_pred_proba = logistic_regression.predict_proba(X_train_s)[:,1]
                y_valid_lr_pred_proba = logistic_regression.predict_proba(X_valid_s)[:,1]

                #Считаем метрики ROC AUC yна обуающем и валидационном наборе и добавляем их в спискок
                current_solver_data.append(round(metrics.roc_auc_score(y_train, y_train_lr_pred_proba),3))
                current_solver_data.append(round(metrics.roc_auc_score(y_valid, y_valid_lr_pred_proba),3))

                #Считаем метрики для класса 0 на валидационном наборе и добавляем их в список
                current_solver_data.append(round(metrics.recall_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))
                current_solver_data.append(round(metrics.precision_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))


                #Считаем метрики для класса 1 на валидационном набопе и добавляем их в список
                current_solver_data.append(round(metrics.recall_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))
                current_solver_data.append(round(metrics.precision_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))

                # запишем данные полученные на одной итерации
                solvers_data.append(current_solver_data)

                # чтобы подстраховаться на случай ошибки : "The Kernel crashed while executing code"
                # сохраним в локальную папку удачную итерацию
                # для этого из полученных данных сформируем dataframe
                # из полученных данных сформируем dataframe
                solvers_pd =pd.DataFrame(
                        columns=['solver','time_fit',
                                 'roc_train','roc_valid', 
                                 'recall_0','precision_0',
                                 'recall_1','precision_1'],
                        data = solvers_data)
                # сохраняем dataframe
                solvers_pd.to_csv('data_recovery/solvers_pd.csv',index=False)
               
                # очистим output от прошедшей итерации
                clear_output()

        except func_timeout.FunctionTimedOut:
                # если время обучения привышает лимит
                # запишем в current_solver_data соотвествующую данные
                current_solver_data = [solver,'>3m',0,0,0,0,0,0]
                # добавляем полученные данные в список solvers_data
                solvers_data.append(current_solver_data)

                # чтобы подстраховаться на случай ошибки : "The Kernel crashed while executing code"
                # сохраним в локальную папку удачную итерацию
                # для этого из полученных данных сформируем dataframe
                # из полученных данных сформируем dataframe
                solvers_pd =pd.DataFrame(
                        columns=['solver','time_fit',
                                 'roc_train','roc_valid', 
                                 'recall_0','precision_0',
                                 'recall_1','precision_1'],
                        data = solvers_data)
                # сохраняем dataframe
                solvers_pd.to_csv('data_recovery/solvers_pd.csv',index=False)
                
                # очистим output от прошедшей итерации
                clear_output()
                # переходим к следующему solver
                pass
# очистим output
clear_output()

# сформируем данные соотвествующие лучщему ROC AUC на валидацинном наборе
best_roc_valid_data = solvers_pd[solvers_pd['roc_valid']==solvers_pd['roc_valid'].max()]

# выведем лучший результат на валидационном наборе
print('result:')
print('best solver on valid:',best_roc_valid_data.iloc[0]['solver'])
print('ROC AUC on train:', best_roc_valid_data.iloc[0]['roc_train'])
print('ROC AUC on valid:', best_roc_valid_data.iloc[0]['roc_valid'])
print('Time fit for best solver:', best_roc_valid_data.iloc[0]['time_fit'],' seconds')

# выведем полученный dataframe
solvers_pd

# time: 2m 35s

##### <span style="color:MediumSlateBlue">Подбор гиперпараметров модели</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'payments'
count_features = dict_spaces[feature_space]

# для работы функции corr_transform_to_force
# подгрузим DataFrame с матрицей корреляции признаков
corr_matrix = fp.ParquetFile('base_models/data/stat/corr_matrix_'+feature_space).to_pandas()

# для выбора sceler преобразвания нам понадобится словарь
dict_scalers ={
    'MinMaxScaler': MinMaxScaler(),
    'RobustScaler':RobustScaler(),
    'StandardScaler':StandardScaler()}

# настроим оптимизацию гипер параметров
def optuna_lg(trial):
  # задаем пространства поиска гиперпараметров
  solver = trial.suggest_categorical("solver", ['lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky'])
  C = trial.suggest_float('C',0.01,1)
  class_0_weight = trial.suggest_float('class_0_weight',0.2,0.6)
  class_1_weight = trial.suggest_float('class_1_weight',0.2,0.6)
  threshold = trial.suggest_float('threshold',0.6,1)
  class_1_percent = trial.suggest_float('class_1_percent',0.01,0.3)
  scaler = trial.suggest_categorical("scaler", ['MinMaxScaler','RobustScaler','StandardScaler'])
  random_state = trial.suggest_int('random_state', 1, 1000000)


  # создаем модель
  optuna_log_reg = linear_model.LogisticRegression(
      solver=solver,
      C=C,
      class_weight={0:class_0_weight,1:class_1_weight},
      random_state=random_state,
      max_iter=10000)

  # с помощью функции corr_transform_to_force получим
  # список не коррелируемых признаков
  list_ncorr_features = corr_transform_to_force(corr_matrix,threshold=threshold)[1]

  # обьявим scaler
  scaler = dict_scalers[scaler]

  # сформируем данные для анализа сбалансированности обучающих данных
  X_train_pd = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
  y_train_pd = pd.read_csv('target/target_train.csv')

  # поготовим данные y_train_pd к работе с функцией class_1_percent_samples
  y_train_pd.set_index('id',drop=False,inplace=True)

  # подготовим данные для обучения модели
  # с помощью функции class_1_percent_samples зададим долю 
  # класса 1
  list_c1_percent_id = class_1_percent_samples(y_train_pd,class_1_percent,random_state=random_state)[:1000000]

  # подготовим данные для обучения
  X_train_s_balanced = scaler.fit_transform(X_train_pd.loc[list_c1_percent_id])
  y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

  # освободим память от "тяжелых" и ненужных файлов
  del X_train_pd, y_train_pd
  gc.collect()

  # я не воспользовался параметром timeout метода optimize потому, что  
  # optimize отставливает поиск параметров, если время обучения 
  # превышает timeout. Мне нужно чтобы поиск параметров продолжился.
  try:
    # для модуля func_time_out упакуем обучение модели в функцию try_func
    def try_func():
            return optuna_log_reg.fit(X_train_s_balanced, y_train_balanced)
    # обучаем модель с ограничением по времени 10 минут (600 секунд)
    optuna_log_reg = func_timeout.func_timeout(600, try_func)

    # удаляем крупные файлы чтобы высвободить память 
    del X_train_s_balanced, y_train_balanced
    gc.collect()

    # загружаем тестовые наборы
    # сформируем данные для анализа сбалансированности обучающих данных
    X_train = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
    X_valid = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_valid').to_pandas()[list_ncorr_features]
    y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
    y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()

    # подготовим данные для проверки модели
    X_train_s = scaler.transform(X_train)
    X_valid_s = scaler.transform(X_valid)

    # удаляем крупные файлы чтобы высвободить память 
    del X_train, X_valid
    gc.collect()

    # делаем предсказание на обучающем и валидационном наборе и считаем метрики
    roc_train = metrics.roc_auc_score(y_train, optuna_log_reg.predict_proba(X_train_s)[:,1])
    roc_valid = metrics.roc_auc_score(y_valid, optuna_log_reg.predict_proba(X_valid_s)[:,1])
    f1_score = metrics.f1_score(y_valid, optuna_log_reg.predict(X_valid_s))
    recall_1 = metrics.recall_score(y_valid, optuna_log_reg.predict(X_valid_s),average=None,zero_division=0)[1]
    precision_1 = metrics.precision_score(y_valid, optuna_log_reg.predict(X_valid_s),average=None,zero_division=0)[1]

    # удаляем крупные файлы чтобы высвободить память 
    del X_train_s, X_valid_s, y_train, y_valid
    gc.collect()

  except func_timeout.FunctionTimedOut:
    # удаляем крупные файлы чтобы высвободить память 
    del X_train_s_balanced, y_train_balanced
    gc.collect()
    
    # фиксируем пустые значения метрик
    roc_train = 0
    roc_valid = 0
    f1_score = 0
    recall_1 = 0
    precision_1 = 0
    pass

  return roc_train, roc_valid, f1_score, recall_1, precision_1

In [ ]:
# cоздаем объект исследования
# чтобы модель не переобучалась минимизируем roc_train 
# и максимизируем roc_valid
optuna_study_lg = optuna.create_study(study_name='LogisticRegression_'+feature_space+'_stat', 
                               directions=['maximize','maximize','maximize','maximize','maximize'], 
                               sampler=optuna.samplers.TPESampler(),
                               pruner='Hyperband',
                               storage='sqlite:///optuna_studies.db',
                               load_if_exists=True)
# optuna.delete_study(study_name='LogisticRegression_'+feature_space+'_stat', storage='sqlite:///optuna_studies.db')

In [ ]:
# ищем лучшую комбинацию гиперпараметров n_trials раз
optuna_study_lg.optimize(optuna_lg, n_trials=300)

In [ ]:
# из полученного результат соврмируем Data Frame
optuna_study_lg_pd = optuna_study_lg.trials_dataframe().dropna()
# переименуем столбы
optuna_study_lg_pd.rename(columns={
    'values_0': 'roc_train',
    'values_1': 'roc_valid',
    'values_2': 'f1_score',
    'values_3': 'recall_1',
    'values_4': 'precision_1'
},inplace=True)
# Выделим время потраченное на обучение модели
optuna_study_lg_pd.tail()

In [ ]:
# покаждем статистику обучения
print('Максимальное значение метрики ROC AUC на валидационном наборе:',round(optuna_study_lg_pd['roc_valid'].max(),3))
print('Среднее значение метрики ROC AUC на валидационном наборе:',round(optuna_study_lg_pd['roc_valid'].mean(),3))
print('Максимальное значение метрики f1_score на валидационном наборе:',round(optuna_study_lg_pd['f1_score'].max(),3))
print('Среднее значение метрики f1_score на валидационном наборе:',round(optuna_study_lg_pd['f1_score'].mean(),3))
print('Максимальное значение метрики recall_1 на валидационном наборе:',round(optuna_study_lg_pd['recall_1'].max(),3))
print('Среднее значение метрики recall_1 на валидационном наборе:',round(optuna_study_lg_pd['recall_1'].mean(),3))
print('Максимальное значение метрики precision_1 на валидационном наборе:',round(optuna_study_lg_pd['precision_1'].max(),3))
print('Среднее значение метрики precision_1 на валидационном наборе:',round(optuna_study_lg_pd['precision_1'].mean(),3))

In [ ]:
# построим график важности гиперпарметров
optuna.visualization.plot_param_importances(optuna_study_lg, target_name='Score')

In [ ]:
# Построим зависимость метрики precision_1
fig = px.scatter(
    data_frame=optuna_study_lg_pd,
    x='precision_1', #ось абсцисс
    y=['f1_score', 'recall_1'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision от recall на классе 1', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision',
    yaxis_title='Величина метрик recall и f1-score')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики precision_1 от гипер параметров

fig = px.scatter(
    data_frame=optuna_study_lg_pd,
    x='precision_1', #ось абсцисс
    y=['params_C', 'params_class_0_weight', 
       'params_class_1_weight','params_class_1_percent',
       'params_threshold'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики f1_score от гипер параметров

fig = px.scatter(
    data_frame=optuna_study_lg_pd,
    x='f1_score', #ось абсцисс
    y=['params_C', 'params_class_0_weight', 
       'params_class_1_weight','params_class_1_percent',
       'params_threshold'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость f1-score от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики f1-score',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики roc_valid от гипер параметров

fig = px.scatter(
    data_frame=optuna_study_lg_pd,
    x='roc_valid', #ось абсцисс
    y=['params_C', 'params_class_0_weight', 
       'params_class_1_weight','params_class_1_percent',
       'params_threshold'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость ROC AUC на валидационном наборе от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики ROC AUC',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Визуализируем метрики полученные при optuna оптимизации
fig = px.scatter(
    data_frame=optuna_study_lg_pd[(optuna_study_lg_pd['precision_1']>0.08)
                                  & (optuna_study_lg_pd['recall_1']>0.08) 
                                  & (optuna_study_lg_pd['roc_valid']>0.8*optuna_study_lg_pd['roc_valid'].max())],
    x='number', #ось абсцисс
    y=['roc_train','roc_valid','recall_1','precision_1'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость качества модели от trials optuna', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='trials optuna',
    yaxis_title='Величина метрики')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

<span style="color:Blue">

Вывод:  

Их всех выбираем точку $number =234$.   
В этой точке самые оптимальные значения $recall$ и $precision$.  
Посмотрим каким значениям гиперпараметров соотвествует выбранная точка.

In [ ]:
# определим номер лучшге варианта
best_optuna_number = 234

# сформируем словарь лучших гипирпарметров RandomForestClassifier
best_param_lr = {
    'solver' : optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_solver'].iloc[0],
    'C' : round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_C'].iloc[0],3),
    'class_weight' : {0:round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_class_0_weight'].iloc[0],3),
                      1:round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_class_1_weight'].iloc[0],3)},
    }

# создадим перменные
best_threshold = optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_threshold'].iloc[0]
best_scaler = optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_scaler'].iloc[0]
best_class_1_percent = optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_class_1_percent'].iloc[0]
best_random_state = optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_random_state'].iloc[0]

# Выведем принятые наилучшие праметры
print('best solver:',best_param_lr['solver'])
print('best C:',best_param_lr['C'])
print('best class 0 weight:',best_param_lr['class_weight'][0])
print('best class 1 weight:',best_param_lr['class_weight'][1])

print('best class 1 percent:',round(best_class_1_percent,3))
print('best scaler:',best_scaler)
print('best threshold:',round(best_threshold,3))
print('best best random state:',best_random_state)
print('time for best train:',round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['duration'].iloc[0].seconds/60),'minutes')

print()
print('ROC AUC на обучающем наборе:', round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['roc_train'].iloc[0],3))
print('ROC AUC на валидационном наборе:', round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['roc_valid'].iloc[0],3))
print('precision класса 1:', round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['precision_1'].iloc[0],3))
print('recall класса 1:', round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['recall_1'].iloc[0],3))


##### <span style="color:MediumSlateBlue">Обучение модели с лучшими параметрами</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'payments'
count_features = dict_spaces[feature_space]

# для работы функции corr_transform_to_force
# подгрузим DataFrame с матрицей корреляции признаков
corr_matrix = fp.ParquetFile('base_models/data/stat/corr_matrix_'+feature_space).to_pandas()

# подготовим данные
# для выбора sceler преобразвания нам понадобится словарь
dict_scalers ={
    'MinMaxScaler': MinMaxScaler(),
    'RobustScaler':RobustScaler(),
    'StandardScaler':StandardScaler(),
}

# с помощью функции corr_transform_to_force получим
# список не коррелируемых признаков
list_ncorr_features = corr_transform_to_force(corr_matrix,threshold=best_threshold)[1]

# обьявим scaler
scaler = dict_scalers[best_scaler]

# сформируем данные для анализа сбалансированности обучающих данных
X_train_pd = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
y_train_pd = pd.read_csv('target/target_train.csv')

# поготовим данные y_train_pd к работе с функцией class_1_percent_samples
y_train_pd.set_index('id',drop=False,inplace=True)

# подготовим данные для обучения модели
# с помощью функции class_1_percent_samples зададим долю 
# класса 1
list_c1_percent_id = class_1_percent_samples(y_train_pd,best_class_1_percent,random_state=best_random_state)[:1000000]

# подготовим данные для обучения
X_train_s_balanced = scaler.fit_transform(X_train_pd.loc[list_c1_percent_id])
y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

# освободим память от "тяжелых" и ненужных файлов
del X_train_pd, y_train_pd
gc.collect()


# обучаем модель LogisticRegression с наилучшеми параметрами
logistic_regression = linear_model.LogisticRegression(
        **best_param_lr,
        random_state=42,
        max_iter=10000)
logistic_regression.fit(X_train_s_balanced, y_train_balanced)

# удаляем крупные файлы чтобы высвободить память 
del X_train_s_balanced, y_train_balanced
gc.collect()

# загружаем тестовые наборы
# сформируем данные для анализа сбалансированности обучающих данных
X_train = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
X_valid = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_valid').to_pandas()[list_ncorr_features]
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()

# подготовим данные для проверки модели
X_train_s = scaler.transform(X_train)
X_valid_s = scaler.transform(X_valid)

# удаляем крупные файлы чтобы высвободить память 
del X_train, X_valid
gc.collect()

# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_lr_pred_proba = logistic_regression.predict_proba(X_train_s)[:,1]
y_valid_lr_pred_proba = logistic_regression.predict_proba(X_valid_s)[:,1]

# Делаем предсказание для валидационной выборки
y_valid_lr_pred = logistic_regression.predict(X_valid_s)

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_lr_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_lr_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_valid,y_valid_lr_pred,zero_division=0))

# time: 2m 30s

##### <span style="color:MediumSlateBlue">Анализ влияния порога классификации на качество модели</span>

In [ ]:
# чтобы проше обращаться к каждому обьекту в данных
# преобразуем y_valid_LR_pred к Series типу
y_valid_lr_pred_proba = pd.Series(y_valid_lr_pred_proba)

# формируем список из необходимых значений thresholds
thresholds = np.arange(0.01, 1, 0.01)

# сформируем списко для заполнения данными результатов анализа
threshold_data = []


for threshold in thresholds:
        # сформируем список для заполнения данными текушего threshold
        threshold_current_data = [threshold]

        #клиентов, для которых вероятность дефолта > threshold, относим к классу 1
        #в противном случае — к классу 0
        y_valid_lr_pred = y_valid_lr_pred_proba.apply(lambda x: 1 if x>threshold else 0)


        #Считаем метрики для класса 0 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.precision_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.f1_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))

        #Считаем метрики для класса 1 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.precision_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.f1_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))

        # добавляем полученные данные в список threshold_data
        threshold_data.append(threshold_current_data)

        # из полученных данных сформируем dataframe
        thresholds_pd =pd.DataFrame(
        columns=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'],
        data = threshold_data)

# time: 17s

In [ ]:
# Визуализируем метрики на валидационном наборе при различных threshold
fig_threshold = px.line(
    data_frame=thresholds_pd,
    x='threshold', #ось абсцисс
    y=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'], #ось ординат
)
fig_threshold.update_layout(
    title ={
        'text':'Зависимость качества модели от threshold', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Порог классификации threshold',
    yaxis_title='Величина метрики')
fig_threshold.update_xaxes(showspikes=True)
fig_threshold.update_yaxes(showspikes=True)

fig_threshold.show()

Метрика $precision$ показывает на сколько сильно ошиблась модель при определиние класса.  
Чем выше метрика, тем меньше ошиблась модель. 

$$precision = \frac{TP}{TP+FP}$$

Метрика $recall$ показывает способность модели найти класс.  
Чем выше метрика, тем выше способность модели находить класс.  

$$recall = \frac{TP}{TP+FN}$$

Метрика $f1-score$ показывает баланс между $precision$ и $recall$ 

$threshold$- порог класификации модели.  
Все объекты для которых $у_{\text{proba pred}} < threshold$, модель относит к класс 0.  
Соотвественно, если $у_{\text{proba pred}}  > threshold$ модель относит обьект к классу 1.  
Для идельаной модели $threshold$, является границей между областями класса 0 и класса 1.  

<span style="color:Blue">

Выводы по результам анализа:  

Для улучшения качества модели по метрике $ROC AUC$, нет смысла сдвигать порог   
классификации. Но зависимость метрик качества модели от $threshold$, может рассказать о  
спосбоности модели искать класс 1 и класс 0.

При фокусировке модели на классе 1 (более низкий $threshold$):  
- на классе 1 метрика $precision$ уменьшается, метрика $recall$ растет;  
- на классе 0 метрика $precision$ растет, метрика $recall$ уменьшается.  

При фокусировке модели на классе 0 (более высокий $threshold$):  
- на классе 1 метрика $precision$ увеличивается, метрика $recall$ уменьшается;  
- на классе 0 метрика $precision$ уменьшается, метрика $recall$ растет.  

Обьсняется тем, что в условия дисбаланса модель научилась хорошо определять класс 0 и плохо класс 1.  

При фокусировке модели на классе 1 (более низкий $threshold$), все больше обьектов отнесены к классу 1.   
Так как $precision$ класса 1 уменьшается, значит в области класса 1 стало больше обектов класса 0,  
что явлется закономерным для "идеальной" модели.  
Однако при этом, $recall$ класса 1 растет, значит в добавленных обьектах также присутвует класс 1.  
Это как раз указывает на то, что модель не научилась находить среди класса 0 класс 1. В области с     
высокой вероятностью класса 0 (более низкий $threshold$), находятся обьекты класса 1.  

Фокусировка на классе 0 (более высокий $threshold$), заставляет модель все больше обьектов из области класса 1  
относить к классу 0. При этом увеличение метрики $precision$ класса 1 говорит о том, что в области класса 1     
становится меньше 0, а значит модель умеет среди класса 1, находить класс 0.  
После  $threshold=0.64$ модель впринципе теряет способность отличать класс 1. У модели нет обьектов класса 1   
в которых она "уверена" на 80+%.

Обратим внимание на $threshold=0.52$, в этой точке находится баланс межу метриками качества модели:
$$precision_1 = recall_1 = \text{f1-score}_1 = 0.082$$
$$precision_0 = recall_0 = \text{f1-score}_0 = 0.965$$

Для модели с лучшим качеством эти две точки должны находится максимально близко к друг другу и иметь  
высокое значение метрик $precision_1$ и $recall_1$.
Для класса 0 это условие выполнется, что также говорит о том, что модель научилась искать этот класс.  
Для класса 1 это условие не выполнется, что также говорит о том, что модель плохо ищет класс 1.

##### <span style="color:MediumSlateBlue">Построение ROC-кривой выбранной модели на тестовом наборе</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'payments'
count_features = dict_spaces[feature_space]

In [ ]:
# сформируем данные для проверки модели
X_train = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
X_valid = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_valid').to_pandas()[list_ncorr_features]
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()
X_test = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_test').to_pandas()[list_ncorr_features]
y_test = pd.read_csv('target/target_test.csv')['flag'].to_numpy()

In [ ]:
# выполним scaler преобразование
X_train_s = scaler.transform(X_train)
X_valid_s = scaler.transform(X_valid)
X_test_s = scaler.transform(X_test)

print("Размеры обучающего набора",X_train_s.shape)
print("Размеры валидационного набора",X_valid_s.shape)
print("Размеры тестового набора",X_test_s.shape)
# time: 1m 43s

In [ ]:
# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_lr_pred_proba = logistic_regression.predict_proba(X_train_s)[:,1]
y_valid_lr_pred_proba = logistic_regression.predict_proba(X_valid_s)[:,1]
y_test_lr_pred_proba = logistic_regression.predict_proba(X_test_s)[:,1]

# Делаем предсказание для валидационной выборки
y_test_lr_pred = logistic_regression.predict(X_test_s)

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_lr_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_lr_pred_proba),3))
print('ROC AUC на тестовом наборе', round(metrics.roc_auc_score(y_test, y_test_lr_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_test,y_test_lr_pred,zero_division=0))

In [ ]:
# Для расчета значений TPR и FPR используем функцию roc_curve
fpr_train, tpr_train, _ = metrics.roc_curve(y_train, y_train_lr_pred_proba)
fpr_valid, tpr_valid, _ = metrics.roc_curve(y_valid, y_valid_lr_pred_proba)
fpr_test, tpr_test, _ = metrics.roc_curve(y_test, y_test_lr_pred_proba)

# рассчитаем площадь под кривой
auc_train = round(metrics.roc_auc_score(y_train, y_train_lr_pred_proba),3)
auc_valid = round(metrics.roc_auc_score(y_valid, y_valid_lr_pred_proba),3)
auc_test = round(metrics.roc_auc_score(y_test, y_test_lr_pred_proba),3)

trace_train = go.Scatter(x=fpr_train, y=tpr_train, mode='lines', name='AUC_train = %0.2f' % auc_train,
                   line=dict(color='red', width=2),
                   hovertemplate='train<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_train])

trace_valid = go.Scatter(x=fpr_valid, y=tpr_valid, mode='lines', name='AUC_valid = %0.2f' % auc_valid,
                   line=dict(color='green', width=2),
                   hovertemplate='valid<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_valid])

trace_test = go.Scatter(x=fpr_test, y=tpr_test, mode='lines', name='AUC_test = %0.2f' % auc_test,
                   line=dict(color='darkorange', width=2),
                   hovertemplate='test<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_test])


reference_line = go.Scatter(x=[0,1], y=[0,1], mode='lines', name='Reference Line',
                            line=dict(color='navy', width=2, dash='dash'))

fig_roc = go.Figure(data=[trace_train,trace_valid,trace_test,reference_line])

fig_roc.update_layout(
    title ={
        'text':'ROC-кривая', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 1350,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate')

fig_roc.update_xaxes(showspikes=True)
fig_roc.update_yaxes(showspikes=True)


fig_roc.show()

<span style="color:Blue">

Вывод по ROC-кривой:
1. Модель выдает сталибильное качество на всех наборах, а значит  
не переобучена - разброс ($variance$) не большой.  
2. Смещение ($bias$) модели относительно большое.  
На тестовом наборе полученно качество $ROC AUC = 0.6$.

#### <span style="color:MediumBlue">Построение модели Random Forest Classifier</span>

##### <span style="color:MediumSlateBlue">Baseline обучение модели RandomForestClassifier</span>

In [ ]:
# обучаем модель логистической регрессии
# чтобы модель пыталась найти связь во всех признаках
random_forest = RandomForestClassifier(random_state=42, n_jobs=-1)
random_forest.fit(X_train_balanced[list_ncorr_features],y_train_balanced)

# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_rf__pred_proba = random_forest.predict_proba(X_train[list_ncorr_features])[:,1]
y_valid_rf_pred_proba = random_forest.predict_proba(X_valid[list_ncorr_features])[:,1]

# Делаем предсказание для валидационной выборки
y_valid_rf_pred = random_forest.predict(X_valid[list_ncorr_features])

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_rf__pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_rf_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_valid,y_valid_rf_pred))

# time: 17s

##### <span style="color:MediumSlateBlue">Подбор гиперпараметров модели</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'payments'
count_features = dict_spaces[feature_space]

# для работы функции corr_transform_to_force
# подгрузим DataFrame с матрицей корреляции признаков
corr_matrix = fp.ParquetFile('base_models/data/stat/corr_matrix_'+feature_space).to_pandas()

# настроим оптимизацию гипер параметров
def optuna_rf(trial):
  # задаем пространства поиска гиперпараметров
  # задаем пространства поиска гиперпараметров
  n_estimators = trial.suggest_int('n_estimators', 40, 100,step = 5)
  criterion = trial.suggest_categorical('criterion',['gini', 'entropy', 'log_loss'])
  max_depth = trial.suggest_int('max_depth', 10, 30,step = 1)
  min_samples_split = trial.suggest_int('min_samples_split', 5, 15,step = 1)
  min_samples_leaf = trial.suggest_int('min_samples_leaf', 5, 15,step = 1)
  max_features = trial.suggest_float('max_features',0.4,0.8)
  max_samples = trial.suggest_float('max_samples',0.4,0.8)
  class_0_weight = trial.suggest_float('class_0_weight',0.1,1)
  class_1_weight = trial.suggest_float('class_1_weight',0.1,1)
  threshold = trial.suggest_float('threshold',0.4,0.8)
  class_1_percent = trial.suggest_float('class_1_percent',0.01,0.5)
  random_state = trial.suggest_int('random_state', 1, 1000000)


  # создаем модель
  optuna_random_forest = RandomForestClassifier(
      n_estimators = n_estimators, 
      criterion = criterion,
      max_depth = max_depth,
      min_samples_split = min_samples_split,  
      min_samples_leaf = min_samples_leaf,
      max_features=max_features,
      max_samples=max_samples,
      class_weight={0:class_0_weight,1:class_1_weight},
      n_jobs=-1,
      random_state=42)

  # с помощью функции corr_transform_to_force получим
  # список не коррелируемых признаков
  list_ncorr_features = corr_transform_to_force(corr_matrix,threshold=threshold)[1]

  # сформируем данные для анализа сбалансированности обучающих данных
  X_train_pd = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
  y_train_pd = pd.read_csv('target/target_train.csv')

  # поготовим данные y_train_pd к работе с функцией class_1_percent_samples
  y_train_pd.set_index('id',drop=False,inplace=True)

  # подготовим данные для обучения модели
  # с помощью функции class_1_percent_samples зададим долю 
  # класса 1
  list_c1_percent_id = class_1_percent_samples(y_train_pd,class_1_percent,random_state=random_state)[:1000000]

  # подготовим данные для обучения
  X_train_balanced = X_train_pd.loc[list_c1_percent_id]
  y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag']

  # освободим память от "тяжелых" и ненужных файлов
  del X_train_pd, y_train_pd
  gc.collect()

  # я не воспользовался параметром timeout метода optimize потому, что  
  # optimize отставливает поиск параметров, если время обучения 
  # превышает timeout. Мне нужно чтобы поиск параметров продолжился.
  try:
    # для модуля func_time_out упакуем обучение модели в функцию try_func
    def try_func():
            return optuna_random_forest.fit(X_train_balanced, y_train_balanced)
    # обучаем модель с ограничением по времени 10 минут (600 секунд)
    optuna_random_forest = func_timeout.func_timeout(600, try_func)

    # удаляем крупные файлы чтобы высвободить память 
    del X_train_balanced, y_train_balanced
    gc.collect()

    # загружаем тестовые наборы
    # сформируем данные для анализа сбалансированности обучающих данных
    X_train = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
    X_valid = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_valid').to_pandas()[list_ncorr_features]
    y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
    y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()
    
    # делаем предсказание на обучающем и валидационном наборе
    #Считаем метрики для класса 1 добавляем их в список
    roc_train = metrics.roc_auc_score(y_train, optuna_random_forest.predict_proba(X_train)[:,1])
    roc_valid = metrics.roc_auc_score(y_valid, optuna_random_forest.predict_proba(X_valid)[:,1])
    f1_score = metrics.f1_score(y_valid, optuna_random_forest.predict(X_valid))
    recall_1 = metrics.recall_score(y_valid, optuna_random_forest.predict(X_valid),average=None,zero_division=0)[1]
    precision_1 = metrics.precision_score(y_valid, optuna_random_forest.predict(X_valid),average=None,zero_division=0)[1]

    # удаляем крупные файлы чтобы высвободить память 
    del X_train, X_valid, y_train, y_valid
    gc.collect()

  except func_timeout.FunctionTimedOut:
    # удаляем крупные файлы чтобы высвободить память 
    del X_train_balanced, y_train_balanced
    gc.collect()
    
    # фиксируем пустые значения метрик
    roc_train = 0
    roc_valid = 0
    f1_score = 0
    recall_1 = 0
    precision_1 = 0
    pass

  return roc_train, roc_valid, f1_score, recall_1, precision_1

In [ ]:
# cоздаем объект исследования
optuna_study_rf = optuna.create_study(study_name= 'RandomForestClassifier_'+feature_space+'_stat', 
                               directions=['maximize','maximize','maximize','maximize','maximize'], 
                               sampler=optuna.samplers.TPESampler(),
                               pruner='Hyperband',
                               storage='sqlite:///optuna_studies.db',
                               load_if_exists=True)
# на случай удаления 
# optuna.delete_study(study_name='RandomForestClassifier_'+feature_space+'_stat', storage='sqlite:///optuna_studies.db')

In [ ]:
# ищем лучшую комбинацию гиперпараметров n_trials раз
optuna_study_rf.optimize(optuna_rf, n_trials=300)

In [ ]:
# из полученного результат соврмируем Data Frame
optuna_study_rf_pd = optuna_study_rf.trials_dataframe().dropna()
# переименуем столбы
optuna_study_rf_pd.rename(columns={
    'values_0': 'roc_train',
    'values_1': 'roc_valid',
    'values_2': 'f1_score',
    'values_3': 'recall_1',
    'values_4': 'precision_1'
},inplace=True)
optuna_study_rf_pd.tail()

In [ ]:
# покаждем статистику обучения
print('Максимальное значение метрики ROC AUC на валидационном наборе:',round(optuna_study_rf_pd['roc_valid'].max(),3))
print('Среднее значение метрики ROC AUC на валидационном наборе:',round(optuna_study_rf_pd['roc_valid'].mean(),3))
print('Максимальное значение метрики f1_score на валидационном наборе:',round(optuna_study_rf_pd['f1_score'].max(),3))
print('Среднее значение метрики f1_score на валидационном наборе:',round(optuna_study_rf_pd['f1_score'].mean(),3))
print('Максимальное значение метрики recall_1 на валидационном наборе:',round(optuna_study_rf_pd['recall_1'].max(),3))
print('Среднее значение метрики recall_1 на валидационном наборе:',round(optuna_study_rf_pd['recall_1'].mean(),3))
print('Максимальное значение метрики precision_1 на валидационном наборе:',round(optuna_study_rf_pd['precision_1'].max(),3))
print('Среднее значение метрики precision_1 на валидационном наборе:',round(optuna_study_rf_pd['precision_1'].mean(),3))

In [ ]:
# Построим зависимость метрики precision_1

fig = px.scatter(
    data_frame=optuna_study_rf_pd,
    x='precision_1', #ось абсцисс
    y=['recall_1','f1_score'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision от recall на классе 1', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision на классе 1',
    yaxis_title='Величина метрик recall и f1-score')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики precision_1 от гиперпараметров

fig = px.scatter(
    data_frame=optuna_study_rf_pd,
    x='precision_1', #ось абсцисс
    y=['params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight', 'params_max_depth',
       'params_max_features', 'params_max_samples', 'params_min_samples_leaf',
       'params_min_samples_split', 'params_n_estimators', 'params_threshold'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision класса 1 от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision на классе 1',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики f1_score от гиперпараметров

fig = px.scatter(
    data_frame=optuna_study_rf_pd,
    x='f1_score', #ось абсцисс
    y=['params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight', 'params_max_depth',
       'params_max_features', 'params_max_samples', 'params_min_samples_leaf',
       'params_min_samples_split', 'params_n_estimators', 'params_threshold'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость f1-score класса 1 от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики f1-score на классе 1',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики roc_valid от гиперпараметров

fig = px.scatter(
    data_frame=optuna_study_rf_pd,
    x='roc_valid', #ось абсцисс
    y=['params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight', 'params_max_depth',
       'params_max_features', 'params_max_samples', 'params_min_samples_leaf',
       'params_min_samples_split', 'params_n_estimators', 'params_threshold'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость ROC AUC на валидационном наборе от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики ROC AUV на валидационном наборе',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрик качества модели от number

fig = px.scatter(
    data_frame=optuna_study_rf_pd[(optuna_study_rf_pd['recall_1']>=0.08) & (optuna_study_rf_pd['precision_1']>0.08)],
    x='number', #ось абсцисс
    y=['roc_valid','roc_train','precision_1','recall_1'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость качества модели от trials optuna', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='trials optuna',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

<span style="color:Blue">

Вывод:  

Их всех выбираем точку $number =188$.   
Не смотря на, то что в этой точке модель подает признаки переобучеености,  
в этой точке оптимальные значения метрик $recall_1$ и $precision_1$, а  
также относительно высокая метрика $ROC AUC$.   
Посмотрим каким значениям гиперпараметров соотвествует выбраннаяточка.

In [ ]:
# определим номер лучшге варианта
best_optuna_number = 188

# сформируем словарь лучших гипирпарметров RandomForestClassifier
best_param_rf = {
    'n_estimators' : int(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_n_estimators'].iloc[0]),
    'criterion': optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_criterion'].iloc[0],
    'max_depth' : int(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_max_depth'].iloc[0]),
    'min_samples_split': int(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_min_samples_split'].iloc[0]),
    'min_samples_leaf' : int(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_min_samples_leaf'].iloc[0]),
    'max_features': optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_max_features'].iloc[0],
    'max_samples' : optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_max_samples'].iloc[0],
    'class_weight' : {0:optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_class_0_weight'].iloc[0],
                      1:optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_class_1_weight'].iloc[0]}
    }

# определим перменные для лучших значений параметров
best_threshold = optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_threshold'].iloc[0]
best_class_1_percent = optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_class_1_percent'].iloc[0]
best_random_state = optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_random_state'].iloc[0]


# Выведем принятые наилучшие праметры
print('best n estimators:',best_param_rf['n_estimators'])
print('best criterion:',best_param_rf['criterion'])
print('best max depth:',best_param_rf['max_depth'])
print('best min samples split:',best_param_rf['min_samples_split'])
print('best min samples leaf:',best_param_rf['min_samples_leaf'])
print('best max features(%):',round(best_param_rf['max_features'],3))
print('best max samples(%):',round(best_param_rf['max_samples'],3))
print('best class 0 weight:',round(best_param_rf['class_weight'][0],3))
print('best class 1 weight:',round(best_param_rf['class_weight'][1],3))

print('best class 1 percent:',round(best_class_1_percent,3))
print('best threshold:',round(best_threshold,3))
print('best random state:', best_random_state)
print('time for best train:',round(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['duration'].iloc[0].seconds/60),'minutes')

print()
print('ROC AUC на обучающем наборе:', round(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['roc_train'].iloc[0],3))
print('ROC AUC на валидационном наборе:', round(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['roc_valid'].iloc[0],3))
print('precision класса 1:', round(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['precision_1'].iloc[0],3))
print('recall класса 1:', round(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['recall_1'].iloc[0],3))

##### <span style="color:MediumSlateBlue">Обучение модели с лучшими параметрами</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'payments'
count_features = dict_spaces[feature_space]

# для работы функции corr_transform_to_force
# подгрузим DataFrame с матрицей корреляции признаков
corr_matrix = fp.ParquetFile('base_models/data/stat/corr_matrix_'+feature_space).to_pandas()

# с помощью функции corr_transform_to_force получим
# список не коррелируемых признаков
list_ncorr_features = corr_transform_to_force(corr_matrix,threshold=best_threshold)[1]

# сформируем данные для анализа сбалансированности обучающих данных
X_train_pd = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
y_train_pd = pd.read_csv('target/target_train.csv')

# поготовим данные y_train_pd к работе с функцией class_1_percent_samples
y_train_pd.set_index('id',drop=False,inplace=True)

# подготовим данные для обучения модели
# с помощью функции class_1_percent_samples зададим долю 
# класса 1
list_c1_percent_id = class_1_percent_samples(y_train_pd,best_class_1_percent,random_state=best_random_state)[:1000000]

# подготовим данные для обучения
X_train_balanced = X_train_pd.loc[list_c1_percent_id]
y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag']

# освободим память от "тяжелых" и ненужных файлов
del X_train_pd, y_train_pd
gc.collect()


# обучаем модель RandomForestClassifier с наилучшеми параметрами
random_forest = RandomForestClassifier(
        **best_param_rf,
        n_jobs=-1,
        random_state=42)
random_forest.fit(X_train_balanced, y_train_balanced)

# удаляем крупные файлы чтобы высвободить память 
del X_train_balanced, y_train_balanced
gc.collect()

# загружаем тестовые наборы
# сформируем данные для анализа сбалансированности обучающих данных
X_train = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
X_valid = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_valid').to_pandas()[list_ncorr_features]
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()

# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_rf_pred_proba = random_forest.predict_proba(X_train)[:,1]
y_valid_rf_pred_proba = random_forest.predict_proba(X_valid)[:,1]

# Делаем предсказание для валидационной выборки
y_valid_rf_pred = random_forest.predict(X_valid)

# удаляем крупные файлы чтобы высвободить память 
del X_train, X_valid
gc.collect()

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_rf_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_rf_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_valid,y_valid_rf_pred,zero_division=0))

# time: 5m 30s

##### <span style="color:MediumSlateBlue">Анализ влияния порога классификации на качество модели</span>

In [ ]:
# чтобы проше обращаться к каждому обьекту в данных
# преобразуем y_valid_LR_pred к Series типу
y_valid_rf_pred_proba = pd.Series(y_valid_rf_pred_proba)

# формируем список из необходимых значений thresholds
thresholds = np.arange(0.01, 1, 0.01)

# сформируем списко для заполнения данными результатов анализа
threshold_data = []


for threshold in thresholds:
        # сформируем список для заполнения данными текушего threshold
        threshold_current_data = [threshold]

        #клиентов, для которых вероятность дефолта > threshold, относим к классу 1
        #в противном случае — к классу 0
        y_valid_rf_pred = y_valid_rf_pred_proba.apply(lambda x: 1 if x>threshold else 0)


        #Считаем метрики для класса 0 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(y_valid, y_valid_rf_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.precision_score(y_valid, y_valid_rf_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.f1_score(y_valid, y_valid_rf_pred,average=None,zero_division=0)[0],3))

        #Считаем метрики для класса 1 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(y_valid, y_valid_rf_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.precision_score(y_valid, y_valid_rf_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.f1_score(y_valid, y_valid_rf_pred,average=None,zero_division=0)[1],3))

        # добавляем полученные данные в список threshold_data
        threshold_data.append(threshold_current_data)

        # из полученных данных сформируем dataframe
        thresholds_pd =pd.DataFrame(
        columns=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'],
        data = threshold_data)

# time: 17s

In [ ]:
# Визуализируем метрики на валидационном наборе при различных threshold
fig_threshold = px.line(
    data_frame=thresholds_pd,
    x='threshold', #ось абсцисс
    y=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'], #ось ординат
)
fig_threshold.update_layout(
    title ={
        'text':'Зависимость качества модели от threshold', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Порог классификации threshold',
    yaxis_title='Величина метрики')
fig_threshold.update_xaxes(showspikes=True)
fig_threshold.update_yaxes(showspikes=True)

fig_threshold.show()

Метрика $precision$ показывает на сколько сильно ошиблась модель при определиние класса.  
Чем выше метрика, тем меньше ошиблась модель. 

$$precision = \frac{TP}{TP+FP}$$

Метрика $recall$ показывает способность модели найти класс.  
Чем выше метрика, тем выше способность модели находить класс.  

$$recall = \frac{TP}{TP+FN}$$

Метрика $f1-score$ показывает баланс между $precision$ и $recall$ 

$threshold$- порог класификации модели.  
Все объекты для которых $у_{\text{proba pred}} < threshold$, модель относит к класс 0.  
Соотвественно, если $у_{\text{proba pred}}  > threshold$ модель относит обьект к классу 1.  
Для идельаной модели $threshold$, является границей между областями класса 0 и класса 1.  

<span style="color:Blue">

Выводы по результам анализа:  

Для улучшения качества модели по метрике $ROC AUC$, нет смысла сдвигать порог   
классификации. Но зависимость метрик качества модели от $threshold$, может рассказать о  
способности модели искать класс 1 и класс 0.

При фокусировке модели на классе 1 (более низкий $threshold$):  
- на классе 1 метрика $precision$ уменьшается, метрика $recall$ растет;  
- на классе 0 метрика $precision$ растет, метрика $recall$ уменьшается.  

При фокусировке модели на классе 0 (более высокий $threshold$):  
- на классе 1 метрика $precision$ увеличивается, метрика $recall$ уменьшается;  
- на классе 0 метрика $precision$ уменьшается, метрика $recall$ растет.  

Обьсняется тем, что в условия дисбаланса модель научилась хорошо определять класс 0 и плохо класс 1.  

При фокусировке модели на классе 1 (более низкий $threshold$), все больше обьектов отнесены к классу 1.   
Так как $precision$ класса 1 уменьшается, значит в области класса 1 стало больше обектов класса 0,  
что явлется закономерным для "идеальной" модели.  
Однако при этом, $recall$ класса 1 растет, значит в добавленных обьектах также присутвует класс 1.  
Это как раз указывает на то, что модель не нучилась находить среди класса 0 класс 1. В области с     
высокой вероятностью класса 0 (более низкий $threshold$), находятся обьекты класса 1.  

Фокусировка на классе 0 (более высокий $threshold$), заставляет модель все больше обьектов из области класса 1  
относить к классу 0. При этом увеличение метрики $precision$ класса 1 говорит о том, что в области класса 1     
становится меньше 0, а значит модель умеет среди класса 1, находить класс 0.  
После  $threshold=0.77$ модель впринципе теряет способность отличать класс 1. У модели нет обьектов класса 1   
в которых она "уверена" на 80+%.

Обратим внимание на $threshold=0.49$, в этой точке находится баланс межу метриками качества модели:
$$precision_1 = recall_1 = \text{f1-score}_1 = 0.09$$
$$precision_0 = recall_0 = \text{f1-score}_0 = 0.968$$

Для модели с лучшим качеством эти две точки должны находится максимально близко к друг другу и иметь   
высокое значение метрик $precision_1$ и $recall_1$.  
Для класса 0 это условие выполнется, что также говорит о том, что модель научилась искать этот класс.    
Для класса 1 это условие не выполнется, что также говорит о том, что модель плохо ищет класс 1.

##### <span style="color:MediumSlateBlue">Построение ROC-кривой выбранной модели на тестовом наборе</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'payments'
count_features = dict_spaces[feature_space]

In [ ]:
# сформируем данные для проверки модели
X_train = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
X_valid = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_valid').to_pandas()[list_ncorr_features]
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()
X_test = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_test').to_pandas()[list_ncorr_features]
y_test = pd.read_csv('target/target_test.csv')['flag'].to_numpy()

In [ ]:
print("Размеры обучающего набора",X_train.shape,y_train.shape)
print("Размеры валидационного набора",X_valid.shape,y_valid.shape)
print("Размеры тестового набора",X_test.shape,y_test.shape)
# time: 1m 43s

In [ ]:
# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_rf_pred_proba = random_forest.predict_proba(X_train)[:,1]
y_valid_rf_pred_proba = random_forest.predict_proba(X_valid)[:,1]
y_test_rf_pred_proba = random_forest.predict_proba(X_test)[:,1]

# Делаем предсказание для валидационной выборки
y_test_lr_pred = random_forest.predict(X_test)

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_rf_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_rf_pred_proba),3))
print('ROC AUC на тестовом наборе', round(metrics.roc_auc_score(y_test, y_test_rf_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_test,y_test_lr_pred,zero_division=0))

In [ ]:
# Для расчета значений TPR и FPR используем функцию roc_curve
fpr_train, tpr_train, _ = metrics.roc_curve(y_train, y_train_rf_pred_proba)
fpr_valid, tpr_valid, _ = metrics.roc_curve(y_valid, y_valid_rf_pred_proba)
fpr_test, tpr_test, _ = metrics.roc_curve(y_test, y_test_rf_pred_proba)

# рассчитаем площадь под кривой
auc_train = round(metrics.roc_auc_score(y_train, y_train_rf_pred_proba),3)
auc_valid = round(metrics.roc_auc_score(y_valid, y_valid_rf_pred_proba),3)
auc_test = round(metrics.roc_auc_score(y_test, y_test_rf_pred_proba),3)

trace_train = go.Scatter(x=fpr_train, y=tpr_train, mode='lines', name='AUC_train = %0.2f' % auc_train,
                   line=dict(color='red', width=2),
                   hovertemplate='train<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_train])

trace_valid = go.Scatter(x=fpr_valid, y=tpr_valid, mode='lines', name='AUC_valid = %0.2f' % auc_valid,
                   line=dict(color='green', width=2),
                   hovertemplate='valid<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_valid])

trace_test = go.Scatter(x=fpr_test, y=tpr_test, mode='lines', name='AUC_test = %0.2f' % auc_test,
                   line=dict(color='darkorange', width=2),
                   hovertemplate='test<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_test])


reference_line = go.Scatter(x=[0,1], y=[0,1], mode='lines', name='Reference Line',
                            line=dict(color='navy', width=2, dash='dash'))

fig_roc = go.Figure(data=[trace_train,trace_valid,trace_test,reference_line])

fig_roc.update_layout(
    title ={
        'text':'ROC-кривая', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 1350,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate')

fig_roc.update_xaxes(showspikes=True)
fig_roc.update_yaxes(showspikes=True)


fig_roc.show()

<span style="color:Blue">

Вывод по ROC-кривой:
1. Не смотря на то, что модель показывает, очень хорошую $ROC$ кривую на обучающем наборе,  
модель, так же выдает сталибильное качество на валидационном и тестовом наборах, а значит  
не переобучена - разброс ($variance$) не большой.
2. Смещение ($bias$) модели такое же как у модели $Logistic$ $Regression$ $Classifier$,  
обученной  на тех же данных.  
На тестовом наборе полученно качество $ROC AUC = 0.61$.

#### <span style="color:MediumBlue">Построение модели Gradient Boosting Classifier</span>

##### <span style="color:MediumSlateBlue">Baseline обучение модели Hist Gradient Boosting Classifier</span>

In [ ]:
# обучаем модель Gradient Boosting Classifier
gradient_boosting = HistGradientBoostingClassifier(random_state=42)
gradient_boosting.fit(X_train_balanced[list_ncorr_features],y_train_balanced)

# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_rf__pred_proba = gradient_boosting.predict_proba(X_train[list_ncorr_features])[:,1]
y_valid_rf_pred_proba = gradient_boosting.predict_proba(X_valid[list_ncorr_features])[:,1]

# Делаем предсказание для валидационной выборки
y_valid_rf_pred = gradient_boosting.predict(X_valid[list_ncorr_features])

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_rf__pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_rf_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_valid,y_valid_rf_pred))

# time: 30s

##### <span style="color:MediumSlateBlue">Подбор гиперпараметров модели</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'payments'
count_features = dict_spaces[feature_space]

# для работы функции corr_transform_to_force
# подгрузим DataFrame с матрицей корреляции признаков
corr_matrix = fp.ParquetFile('base_models/data/stat/corr_matrix_'+feature_space).to_pandas()

# настроим оптимизацию гипер параметров
def optuna_hgbc(trial):
  # задаем пространства поиска гиперпараметров
  learning_rate = trial.suggest_float('learning_rate',0.01,0.1)
  max_iter = trial.suggest_int('max_iter', 150, 250,step = 1)
  max_leaf_nodes = trial.suggest_int('max_leaf_nodes', 2, 60,step = 1)
  max_depth = trial.suggest_int('max_depth', 1, 10,step = 1)
  min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 60,step = 1)
  max_features = trial.suggest_float('max_features',0.6,1)
  l2_regularization = trial.suggest_float('l2_regularization',0.01,1)
  class_0_weight = trial.suggest_float('class_0_weight',0.01,1)
  class_1_weight = trial.suggest_float('class_1_weight',0.5,1)
  threshold = trial.suggest_float('threshold',0.7,1)
  class_1_percent = trial.suggest_float('class_1_percent',0.01,0.25)
  random_state = trial.suggest_int('random_state', 1, 1000000)

  # создаем модель
  optuna_gradient_boosting = HistGradientBoostingClassifier(
      learning_rate= learning_rate,
      max_iter = max_iter,
      max_leaf_nodes =max_leaf_nodes,
      max_depth = max_depth,
      min_samples_leaf = min_samples_leaf,
      max_features=max_features,
      l2_regularization = l2_regularization,
      class_weight={0:class_0_weight,1:class_1_weight},
      random_state=42)

  # с помощью функции corr_transform_to_force получим
  # список не коррелируемых признаков
  list_ncorr_features = corr_transform_to_force(corr_matrix,threshold=threshold)[1]

  # сформируем данные для анализа сбалансированности обучающих данных
  X_train_pd = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
  y_train_pd = pd.read_csv('target/target_train.csv')

  # поготовим данные y_train_pd к работе с функцией class_1_percent_samples
  y_train_pd.set_index('id',drop=False,inplace=True)

  # подготовим данные для обучения модели
  # с помощью функции class_1_percent_samples зададим долю 
  # класса 1
  list_c1_percent_id = class_1_percent_samples(y_train_pd,class_1_percent,random_state=random_state)[:1000000]

  # подготовим данные для обучения
  X_train_balanced = X_train_pd.loc[list_c1_percent_id]
  y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag']

  # освободим память от "тяжелых" и ненужных файлов
  del X_train_pd, y_train_pd
  gc.collect()

  # я не воспользовался параметром timeout метода optimize потому, что  
  # optimize отставливает поиск параметров, если время обучения 
  # превышает timeout. Мне нужно чтобы поиск параметров продолжился.
  try:
    # для модуля func_time_out упакуем обучение модели в функцию try_func
    def try_func():
            return optuna_gradient_boosting.fit(X_train_balanced,y_train_balanced)
    # обучаем модель с ограничением по времени 10 минут (600 секунд)
    optuna_gradient_boosting = func_timeout.func_timeout(600, try_func)

    # удаляем крупные файлы чтобы высвободить память 
    del X_train_balanced, y_train_balanced
    gc.collect()

        # загружаем тестовые наборы
    # сформируем данные для анализа сбалансированности обучающих данных
    X_train = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
    X_valid = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_valid').to_pandas()[list_ncorr_features]
    y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
    y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()
    
    # делаем предсказание на обучающем и валидационном наборе и считаем метрики
    roc_train = metrics.roc_auc_score(y_train, optuna_gradient_boosting.predict_proba(X_train)[:,1])
    roc_valid = metrics.roc_auc_score(y_valid, optuna_gradient_boosting.predict_proba(X_valid)[:,1])
    f1_score = metrics.f1_score(y_valid, optuna_gradient_boosting.predict(X_valid))
    recall_1 = metrics.recall_score(y_valid, optuna_gradient_boosting.predict(X_valid),average=None,zero_division=0)[1]
    precision_1 = metrics.precision_score(y_valid, optuna_gradient_boosting.predict(X_valid),average=None,zero_division=0)[1]


    # удаляем крупные файлы чтобы высвободить память 
    del X_train, X_valid, y_train, y_valid
    gc.collect()

  except func_timeout.FunctionTimedOut:
    # удаляем крупные файлы чтобы высвободить память 
    del X_train_balanced, y_train_balanced
    gc.collect()
    
    # фиксируем пустые значения метрик
    roc_train = 0
    roc_valid = 0
    f1_score = 0
    recall_1 = 0
    precision_1 = 0
    pass

  return roc_train, roc_valid, f1_score, recall_1, precision_1

In [ ]:
# cоздаем объект исследования
# чтобы модель не переобучалась минимизируем roc_train 
# и максимизируем roc_valid
optuna_study_hgbc = optuna.create_study(study_name='HistGradientBoostingClassifier_'+feature_space+'_stat', 
                               directions=['maximize','maximize','maximize','maximize','maximize'], 
                               sampler=optuna.samplers.TPESampler(),
                               pruner='Hyperband',
                               storage='sqlite:///optuna_studies.db',
                               load_if_exists=True)

# ну случай удаления обучения
# optuna.delete_study(study_name='HistGradientBoostingClassifier'+feature_space+'_stat', storage='sqlite:///optuna_studies.db')

In [ ]:
# ищем лучшую комбинацию гиперпараметров n_trials раз
optuna_study_hgbc.optimize(optuna_hgbc, n_trials=300)

In [ ]:
# из полученного результат соврмируем Data Frame
optuna_study_hgbc_pd = optuna_study_hgbc.trials_dataframe()
# переименуем столбы
optuna_study_hgbc_pd.rename(columns={
    'values_0': 'roc_train',
    'values_1': 'roc_valid',
    'values_2': 'f1_score',
    'values_3': 'recall_1',
    'values_4': 'precision_1'
},inplace=True)
optuna_study_hgbc_pd.tail(5)

In [ ]:
# покаждем статистику обучения
print('Максимальное значение метрики ROC AUC на валидационном наборе:',round(optuna_study_hgbc_pd['roc_valid'].max(),3))
print('Среднее значение метрики ROC AUC на валидационном наборе:',round(optuna_study_hgbc_pd['roc_valid'].mean(),3))
print('Максимальное значение метрики f1_score на валидационном наборе:',round(optuna_study_hgbc_pd['f1_score'].max(),3))
print('Среднее значение метрики f1_score на валидационном наборе:',round(optuna_study_hgbc_pd['f1_score'].mean(),3))
print('Максимальное значение метрики recall_1 на валидационном наборе:',round(optuna_study_hgbc_pd['recall_1'].max(),3))
print('Среднее значение метрики recall_1 на валидационном наборе:',round(optuna_study_hgbc_pd['recall_1'].mean(),3))
print('Максимальное значение метрики precision_1 на валидационном наборе:',round(optuna_study_hgbc_pd['precision_1'].max(),3))
print('Среднее значение метрики precision_1 на валидационном наборе:',round(optuna_study_hgbc_pd['precision_1'].mean(),3))

In [ ]:
# построим график важности гиперпарметров
optuna.visualization.plot_param_importances(optuna_study_hgbc, target_name='Score')

In [ ]:
# Построим зависимость метрики precision_1
fig = px.scatter(
    data_frame=optuna_study_hgbc_pd,
    x='precision_1', #ось абсцисс
    y=['f1_score','recall_1'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision от recall на классе 1', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision класса 1',
    yaxis_title='Величина метрик recall и f1-score класса 1')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики precision_1 от гиперпараметров
fig = px.scatter(
    data_frame=optuna_study_hgbc_pd,
    x='precision_1', #ось абсцисс
    y=['params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight', 'params_l2_regularization',
       'params_learning_rate', 'params_max_depth', 'params_max_features',
       'params_max_iter', 'params_max_leaf_nodes', 'params_min_samples_leaf',
       'params_threshold'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision класса 1 от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision класса 1',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики f1_score от гиперпараметров
fig = px.scatter(
    data_frame=optuna_study_hgbc_pd,
    x='f1_score', #ось абсцисс
    y=['params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight', 'params_l2_regularization',
       'params_learning_rate', 'params_max_depth', 'params_max_features',
       'params_max_iter', 'params_max_leaf_nodes', 'params_min_samples_leaf',
       'params_threshold'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость f1-score класса 1 от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики f1-score класса 1',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики roc_valid от гиперпараметров
fig = px.scatter(
    data_frame=optuna_study_hgbc_pd,
    x='roc_valid', #ось абсцисс
    y=['params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight', 'params_l2_regularization',
       'params_learning_rate', 'params_max_depth', 'params_max_features',
       'params_max_iter', 'params_max_leaf_nodes', 'params_min_samples_leaf',
       'params_threshold'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость ROC AUC на валидационном наборе от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики ROC AUC на валидационном наборе',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрик качества модели от number

fig = px.scatter(
    data_frame=optuna_study_hgbc_pd[(optuna_study_hgbc_pd['recall_1']>0.1)&(optuna_study_hgbc_pd['precision_1']>0.1)],
    x='number', #ось абсцисс
    y=['roc_train','roc_valid','recall_1','precision_1'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость качества модели от trials optuna', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='trials optuna',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

<span style="color:Blue">

Вывод:  

Их всех выбираем точку $number =127$.   
В этой точке оптимальные значения метрик $recall_1$ и $precision_1$, а  
также относительно высокая метрика $ROC AUC$.   
Посмотрим каким значениям гиперпараметров соотвествует выбранная точка.

In [ ]:
# определим номер лучшге варианта
best_optuna_number = 191

# сформируем словарь лучших гипирпарметров HistGradientBoostingClassifier
best_param_hgbc = {
    'learning_rate' : optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['params_learning_rate'].iloc[0],
    'max_iter' : optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['params_max_iter'].iloc[0],
    'max_leaf_nodes' : optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['params_max_leaf_nodes'].iloc[0],
    'max_depth' : optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['params_max_depth'].iloc[0],
    'min_samples_leaf' : optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['params_min_samples_leaf'].iloc[0],
    'max_features' : optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['params_max_features'].iloc[0],
    'l2_regularization' : optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['params_l2_regularization'].iloc[0],
    'class_weight' : {0: optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['params_class_0_weight'].iloc[0],
                      1: optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['params_class_1_weight'].iloc[0],}
    }

# определим перменные для лучших значений параметров
best_threshold = optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['params_threshold'].iloc[0]
best_class_1_percent = optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['params_class_1_percent'].iloc[0]
best_random_state = optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['params_random_state'].iloc[0]


# Выведем принятые наилучшие параметры
print('best learning rate:',round(best_param_hgbc['learning_rate'],3))
print('best max iter:',best_param_hgbc['max_iter'])
print('best max leaf nodes:',best_param_hgbc['max_leaf_nodes'])
print('best max depth:',best_param_hgbc['max_depth'])
print('best min samples leaf:',best_param_hgbc['min_samples_leaf'])
print('best max features:',round(best_param_hgbc['max_features'],3))
print('best l2 regularization:',round(best_param_hgbc['l2_regularization'],3))
print('best class 0 weight:',round(best_param_hgbc['class_weight'][0],3))
print('best class 1 weight:',round(best_param_hgbc['class_weight'][1],3))

print('best class 1 percent:',round(best_class_1_percent,3))
print('best threshold:',round(best_threshold,3))
print('time for best train:',round(optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['duration'].iloc[0].seconds/60),'minutes')

print()
print('ROC AUC на обучающем наборе:', round(optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['roc_train'].iloc[0],3))
print('ROC AUC на валидационном наборе:', round(optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['roc_valid'].iloc[0],3))
print('precision класса 1:', round(optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['precision_1'].iloc[0],3))
print('recall класса 1:', round(optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['recall_1'].iloc[0],3))

##### <span style="color:MediumSlateBlue">Обучение модели с лучшими параметрами</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'payments'
count_features = dict_spaces[feature_space]

# для работы функции corr_transform_to_force
# подгрузим DataFrame с матрицей корреляции признаков
corr_matrix = fp.ParquetFile('base_models/data/stat/corr_matrix_'+feature_space).to_pandas()

# с помощью функции corr_transform_to_force получим
# список не коррелируемых признаков
list_ncorr_features = corr_transform_to_force(corr_matrix,threshold=best_threshold)[1]

# сформируем данные для анализа сбалансированности обучающих данных
X_train_pd = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
y_train_pd = pd.read_csv('target/target_train.csv')

# поготовим данные y_train_pd к работе с функцией class_1_percent_samples
y_train_pd.set_index('id',drop=False,inplace=True)

# подготовим данные для обучения модели
# с помощью функции class_1_percent_samples зададим долю 
# класса 1
list_c1_percent_id = class_1_percent_samples(y_train_pd,best_class_1_percent,random_state=best_random_state)[:1000000]

# подготовим данные для обучения
X_train_balanced = X_train_pd.loc[list_c1_percent_id]
y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag']

# освободим память от "тяжелых" и ненужных файлов
del X_train_pd, y_train_pd
gc.collect()

# обучаем модель HistGradientBoostingClassifier с наилучшеми параметрами
gradient_boosting = HistGradientBoostingClassifier(
        **best_param_hgbc,
        random_state=42)
gradient_boosting.fit(X_train_balanced,y_train_balanced)

# удаляем крупные файлы чтобы высвободить память 
del X_train_balanced, y_train_balanced
gc.collect()

# сформируем данные для анализа сбалансированности обучающих данных
X_train = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
X_valid = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_valid').to_pandas()[list_ncorr_features]
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()

# делаем предсказание на обучающем и валидационном наборе и считаем метрики
y_train_gb_pred_proba = gradient_boosting.predict_proba(X_train)[:,1]
y_valid_gb_pred_proba = gradient_boosting.predict_proba(X_valid)[:,1]
y_valid_gb_pred = gradient_boosting.predict(X_valid)

# удаляем крупные файлы чтобы высвободить память 
del X_train, X_valid
gc.collect()

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_gb_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_gb_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_valid,y_valid_gb_pred,zero_division=0))

# time: 1m 40s

##### <span style="color:MediumSlateBlue">Анализ влияния порога классификации на качество модели</span>

In [ ]:
# чтобы проше обращаться к каждому обьекту в данных
# преобразуем y_valid_LR_pred к Series типу
y_valid_gb_pred_proba = pd.Series(y_valid_gb_pred_proba)

# формируем список из необходимых значений thresholds
thresholds = np.arange(0.01, 1, 0.01)

# сформируем списко для заполнения данными результатов анализа
threshold_data = []


for threshold in thresholds:
        # сформируем список для заполнения данными текушего threshold
        threshold_current_data = [threshold]

        #клиентов, для которых вероятность дефолта > threshold, относим к классу 1
        #в противном случае — к классу 0
        y_valid_gb_pred = y_valid_gb_pred_proba.apply(lambda x: 1 if x>threshold else 0)


        #Считаем метрики для класса 0 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(y_valid, y_valid_gb_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.precision_score(y_valid, y_valid_gb_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.f1_score(y_valid, y_valid_gb_pred,average=None,zero_division=0)[0],3))

        #Считаем метрики для класса 1 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(y_valid, y_valid_gb_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.precision_score(y_valid, y_valid_gb_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.f1_score(y_valid, y_valid_gb_pred,average=None,zero_division=0)[1],3))

        # добавляем полученные данные в список threshold_data
        threshold_data.append(threshold_current_data)

        # из полученных данных сформируем dataframe
        thresholds_pd =pd.DataFrame(
        columns=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'],
        data = threshold_data)

# time: 17s

In [ ]:
# Визуализируем метрики на валидационном наборе при различных threshold
fig_threshold = px.line(
    data_frame=thresholds_pd,
    x='threshold', #ось абсцисс
    y=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'], #ось ординат
)
fig_threshold.update_layout(
    title ={
        'text':'Зависимость качества модели от threshold', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Порог классификации threshold',
    yaxis_title='Величина метрики')
fig_threshold.update_xaxes(showspikes=True)
fig_threshold.update_yaxes(showspikes=True)

fig_threshold.show()

Метрика $precision$ показывает на сколько сильно ошиблась модель при определиние класса.  
Чем выше метрика, тем меньше ошиблась модель. 

$$precision = \frac{TP}{TP+FP}$$

Метрика $recall$ показывает способность модели найти класс.  
Чем выше метрика, тем выше способность модели находить класс.  

$$recall = \frac{TP}{TP+FN}$$

Метрика $f1-score$ показывает баланс между $precision$ и $recall$ 

$threshold$- порог класификации модели.  
Все объекты для которых $у_{\text{proba pred}} < threshold$, модель относит к класс 0.  
Соотвественно, если $у_{\text{proba pred}}  > threshold$ модель относит обьект к классу 1.  
Для идельаной модели $threshold$, является границей между областями класса 0 и класса 1.  

<span style="color:Blue">

Выводы по результам анализа:  

Для улучшения качества модели по метрике $ROC AUC$, нет смысла сдвигать порог   
классификации. Но зависимость метрик качества модели от $threshold$, может рассказать о  
способности модели искать класс 1 и класс 0.

При фокусировке модели на классе 1 (более низкий $threshold$):  
- на классе 1 метрика $precision$ уменьшается, метрика $recall$ растет;  
- на классе 0 метрика $precision$ растет, метрика $recall$ уменьшается.  

При фокусировке модели на классе 0 (более высокий $threshold$):  
- на классе 1 метрика $precision$ увеличивается, метрика $recall$ уменьшается;  
- на классе 0 метрика $precision$ уменьшается, метрика $recall$ растет.  

Обьсняется тем, что в условия дисбаланса модель научилась хорошо определять класс 0 и плохо класс 1.  

При фокусировке модели на классе 1 (более низкий $threshold$), все больше обьектов отнесены к классу 1.   
Так как $precision$ класса 1 уменьшается, значит в области класса 1 стало больше обектов класса 0,  
что явлется закономерным для "идеальной" модели.  
Однако при этом, $recall$ класса 1 растет, значит в добавленных обьектах также присутвует класс 1.  
Это как раз указывает на то, что модель не нучилась находить среди класса 0 класс 1. В области с     
высокой вероятностью класса 0 (более низкий $threshold$), находятся обьекты класса 1.  

Фокусировка на классе 0 (более высокий $threshold$), заставляет модель все больше обьектов из области класса 1  
относить к классу 0. При этом увеличение метрики $precision$ класса 1 говорит о том, что в области класса 1     
становится меньше 0, а значит модель умеет среди класса 1, находить класс 0.  
После  $threshold=0.88$ модель впринципе теряет способность отличать класс 1. У модели нет обьектов класса 1   
в которых она "уверена" на 90+%.


Обратим внимание на $threshold=0.51$, в этой точке находится баланс межу метриками качества модели:
$$precision_1 = recall_1 = \text{f1-score}_1 = 0.103$$
$$precision_0 = recall_0 = \text{f1-score}_0 = 0.97$$

Для модели с лучшим качеством эти две точки должны находится максимально близко к друг другу и иметь   
высокое значение метрик $precision_1$ и $recall_1$.  
Для класса 0 это условие выполнется, что также говорит о том, что модель научилась искать этот класс.    
Для класса 1 это условие не выполнется, что также говорит о том, что модель плохо ищет класс 1.

##### <span style="color:MediumSlateBlue">Построение ROC-кривой выбранной модели на тестовом наборе</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'payments'
count_features = dict_spaces[feature_space]

In [ ]:
# сформируем данные для проверки модели
X_train = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
X_valid = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_valid').to_pandas()[list_ncorr_features]
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()
X_test = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_test').to_pandas()[list_ncorr_features]
y_test = pd.read_csv('target/target_test.csv')['flag'].to_numpy()

In [ ]:
print("Размеры обучающего набора",X_train.shape,y_train.shape)
print("Размеры валидационного набора",X_valid.shape,y_valid.shape)
print("Размеры тестового набора",X_test.shape,y_test.shape)
# time: 1m 43s

In [ ]:
# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_gb_pred_proba = gradient_boosting.predict_proba(X_train)[:,1]
y_valid_gb_pred_proba = gradient_boosting.predict_proba(X_valid)[:,1]
y_test_gb_pred_proba = gradient_boosting.predict_proba(X_test)[:,1]

# Делаем предсказание для валидационной выборки
y_test_gb_pred = gradient_boosting.predict(X_test)

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_gb_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_gb_pred_proba),3))
print('ROC AUC на тестовом наборе', round(metrics.roc_auc_score(y_test, y_test_gb_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_test,y_test_gb_pred,zero_division=0))

In [ ]:
# Для расчета значений TPR и FPR используем функцию roc_curve
fpr_train, tpr_train, _ = metrics.roc_curve(y_train, y_train_gb_pred_proba)
fpr_valid, tpr_valid, _ = metrics.roc_curve(y_valid, y_valid_gb_pred_proba)
fpr_test, tpr_test, _ = metrics.roc_curve(y_test, y_test_gb_pred_proba)

# рассчитаем площадь под кривой
auc_train = round(metrics.roc_auc_score(y_train, y_train_gb_pred_proba),3)
auc_valid = round(metrics.roc_auc_score(y_valid, y_valid_gb_pred_proba),3)
auc_test = round(metrics.roc_auc_score(y_test, y_test_gb_pred_proba),3)

trace_train = go.Scatter(x=fpr_train, y=tpr_train, mode='lines', name='AUC_train = %0.2f' % auc_train,
                   line=dict(color='red', width=2),
                   hovertemplate='train<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_train])

trace_valid = go.Scatter(x=fpr_valid, y=tpr_valid, mode='lines', name='AUC_valid = %0.2f' % auc_valid,
                   line=dict(color='green', width=2),
                   hovertemplate='valid<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_valid])

trace_test = go.Scatter(x=fpr_test, y=tpr_test, mode='lines', name='AUC_test = %0.2f' % auc_test,
                   line=dict(color='darkorange', width=2),
                   hovertemplate='test<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_test])


reference_line = go.Scatter(x=[0,1], y=[0,1], mode='lines', name='Reference Line',
                            line=dict(color='navy', width=2, dash='dash'))

fig_roc = go.Figure(data=[trace_train,trace_valid,trace_test,reference_line])

fig_roc.update_layout(
    title ={
        'text':'ROC-кривая', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 1350,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate')

fig_roc.update_xaxes(showspikes=True)
fig_roc.update_yaxes(showspikes=True)


fig_roc.show()

<span style="color:Blue">

Вывод по ROC-кривой:
1. Модель выдает сталибильное качество на валидационном и тестовом наборах, а значит  
не переобучена - разброс ($variance$) не большой. Хотя, качество модели на обучающем наборе,  
заметно лучше.
2. Смещение ($bias$) модели чуть лучше, чем у модели $Random$ $Forest$ $Classifier$,  
обученной на тех же данных.    
На тестовом наборе полученно качество $ROC AUC = 0.63$, против $ROC AUC = 0.61$.

### <span style="color:RoyalBlue">Построение модели на сбалансированных данных подпрострнства service stat</span>

#### <span style="color:MediumBlue">Формирование данных для обучения</span>

Анализ исходных данных, в частности target, показал что представленные данные   
имееют перекос: 96% клиентов у которых не случился дефолт (flag = 0),    
против 4 % - для которых произошел дефолт (flag = 1).  
С помощью функции class_1_percent_samples, сформируем сбаланисрованную выборку  
для обучения моделей.
За основу сбалансировных данных возьмем данные клиентов с дефолтом  
и к ним будем добирать необходимое количество клиентов до сбалансированности. 


Функция class_1_percent_samples принемает две переменные:
- target_data - массив с id и целевой переменной
- class_1_percent - процент класса 1 в результирующем массиве

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'service'
count_features = dict_spaces[feature_space]

In [ ]:
# сформируем данные для анализа сбалансированности обучающих данных
X_train_pd = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()
y_train_pd = pd.read_csv('target/target_train.csv')

In [ ]:
# поготовим данные y_train_pd к работе с функцией class_1_percent_samples
y_train_pd.set_index('id',drop=False,inplace=True)
y_train_pd.head(4)

In [ ]:
# взглянем на сблансиорованность данных в y_train
y_train_pd['flag'].value_counts(normalize=True)

In [ ]:
# сбалансируем данные в y_train_pd
list_c1_percent_id = class_1_percent_samples(y_train_pd,0.5)
# list_c1_percent_features - список 'id' из y_train_pd соотвествующих  
# заданному проценту класса 1
len(list_c1_percent_id)

In [ ]:
# проверим сбалансированность полученного y_train
y_train_pd.loc[list_c1_percent_id]['flag'].value_counts(normalize=True)

В отличие от данных в *transform_data_torow*, где модели показываеются последние N  
операций клиента, в данные в *transform_data_stat* сфомированы из различных статистических   
характеристик ряда определяющего клиентскую историю в банке. 

Соотвественно, если в данных *transform_data_torow* была сильная корреляция между признаками,  
то я не смог бы ее убрать, иначе потярется смысл "показать последовательный ряд в историии клиента".  
Признаки в *transform_data_stat* не имеют такой особенности. Поэтому, необходимо проанализировать  
данные в *transform_data_stat* и, в случае сильной коррреляции признаков, устранить ее.

In [ ]:
# подгрузим DataFrame с матрицей корреляции признаков
corr_matrix = fp.ParquetFile('base_models/data/stat/corr_matrix_'+feature_space).to_pandas()
corr_matrix.shape

In [ ]:
# с помощью функции corr_transrom_to_power получим
# силу корреляции матрицы и список не коррелируемых признаков

# для начального анализа зададим порог силы корреляции, при котором признаки 
# будут отнесены к "сильно скоррелированными" равным 0.6
ncorr_matrix,list_ncorr_features,corr_power =corr_transform_to_force(corr_matrix,threshold=0.6)

# Выведем результаты преобразования
print('Исходное количество признаков: ', corr_matrix.shape[0])
print('Количество не коррелируемых признаков: ', len(list_ncorr_features))
# сила корреляции признаков
print('Отношение коррелируемых исходному количеству признаков (сила корреляции): ',corr_power)

In [ ]:
# подготовим данные для обучения
X_train_balanced = X_train_pd.loc[list_c1_percent_id]
y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag']
# сформируем данные для проверики модели
# сформируем данные для анализа сбалансированности обучающих данных
X_train = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()
y_train = pd.read_csv('target/target_train.csv')['flag']
X_valid = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_valid').to_pandas()
y_valid = pd.read_csv('target/target_valid.csv')['flag']

In [ ]:
# посмотрим на структуру данных в X_train_balanced
X_train_balanced.head(4)

In [ ]:
# посмотрим на структуру данных в y_train_balanced
y_train_balanced.head(4)

In [ ]:
# проверим что выборки действительно совпадают по id
X_train_balanced.index.to_list() == y_train_balanced.index.to_list()

#### <span style="color:MediumBlue">Построение модели Logistic Regression</span>

##### <span style="color:MediumSlateBlue">Baseline обучение модели LogisticRegression</span>

In [ ]:
# обучаем модель логистической регрессии
logistic_regression = linear_model.LogisticRegression(random_state=42, max_iter=10000)
logistic_regression.fit(X_train_balanced[list_ncorr_features],y_train_balanced)

# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_LR_pred_proba = logistic_regression.predict_proba(X_train[list_ncorr_features])[:,1]
y_valid_LR_pred_proba = logistic_regression.predict_proba(X_valid[list_ncorr_features])[:,1]

# Делаем предсказание для валидационной выборки
y_valid_LR_pred = logistic_regression.predict(X_valid[list_ncorr_features])

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_LR_pred_proba),3))
print('ROC AUC на тестовом наборе', round(metrics.roc_auc_score(y_valid, y_valid_LR_pred_proba),3))

print('Основные метрики на тестовом наборе:')
print(metrics.classification_report(y_valid,y_valid_LR_pred))

# time: 3m 5s

##### <span style="color:MediumSlateBlue">Анализ влияния scaler преобразования на качество и скорость схождения модели</span>

In [ ]:
# формируем список из необходимых scaler
scalers = [MinMaxScaler(),RobustScaler() ,StandardScaler()]

# сформируем списко для заполнения данными результатов анализа
scalers_data = []

# установим ограничение на время обучения модели в 600 секунд (10 минут)
time_out = 600

for scaler in scalers:
        # для того чтобы следить за выполнением цикла введем индикатор
        print('current scaler: ', scaler)

         # сформируем список для заполнения данными текушего scaler
        current_scaler_data = [scaler]
        
        # выполним scaler преобразование
        X_train_s_balanced = scaler.fit_transform(X_train_balanced[list_ncorr_features])
        X_train_s = scaler.transform(X_train[list_ncorr_features])
        X_valid_s = scaler.transform(X_valid[list_ncorr_features])

        try:
                # для модуля func_time_out упакуем обучение модели в функцию try_func
                def try_func():
                        logistic_regression = linear_model.LogisticRegression(random_state=42, max_iter=10000)
                        return logistic_regression.fit(X_train_s_balanced,y_train_balanced)
                    
                # фиксируем время начала
                start_time = time.time()
                
                # запускаем обучение модели с ограничением по времени
                logistic_regression = func_timeout.func_timeout(time_out, try_func)
                
                # фиксируем время окончания обучения
                end_time = time.time()
                
                # запишем время обучения модели
                current_scaler_data.append(round(end_time-start_time))

                # Делаем предсказание для обучающей и валидационной выборки
                y_valid_lr_pred = logistic_regression.predict(X_valid_s)
                
                # для метрик ROC AUC делаем предсказание модели для обучающей и валидационной выборки  
                # в виде вероятности принадлежности к классу 1 
                y_train_lr_pred_proba = logistic_regression.predict_proba(X_train_s)[:,1]
                y_valid_lr_pred_proba = logistic_regression.predict_proba(X_valid_s)[:,1]

                #Считаем метрики ROC AUC yна обуающем и валидационном наборе и добавляем их в спискок
                current_scaler_data.append(round(metrics.roc_auc_score(y_train, y_train_lr_pred_proba),3))
                current_scaler_data.append(round(metrics.roc_auc_score(y_valid, y_valid_lr_pred_proba),3))

                #Считаем метрики для класса 0 на валидационном наборе и добавляем их в список
                current_scaler_data.append(round(metrics.recall_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))
                current_scaler_data.append(round(metrics.precision_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))


                #Считаем метрики для класса 1 на валидационном набопе и добавляем их в список
                current_scaler_data.append(round(metrics.recall_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))
                current_scaler_data.append(round(metrics.precision_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))

                # добавляем полученные данные в список scalers_data
                scalers_data.append(current_scaler_data)

                # чтобы подстраховаться на случай ошибки : "The Kernel crashed while executing code"
                # сохраним в локальную папку удачную итерацию
                # для этого из полученных данных сформируем dataframe
                # из полученных данных сформируем dataframe
                scaler_pd =pd.DataFrame(
                        columns=['scaler','time_fit',
                                 'roc_train','roc_valid', 
                                 'recall_0','precision_0',
                                 'recall_1','precision_1'],
                        data = scalers_data)
                # сохраняем dataframe
                scaler_pd.to_csv('data_recovery/scaler_pd.csv',index=False)
               
                # очистим output от прошедшей итерации
                clear_output()

        except func_timeout.FunctionTimedOut:
                # если время обучения привышает лимит
                # запишем в current_scaler_data соотвествующую данные
                current_scaler_data = [scaler,'>30m',0,0,0,0,0,0]
                # добавляем полученные данные в список scalers_data
                scalers_data.append(current_scaler_data)

                # чтобы подстраховаться на случай ошибки : "The Kernel crashed while executing code"
                # сохраним в локальную папку удачную итерацию
                # для этого из полученных данных сформируем dataframe
                # из полученных данных сформируем dataframe
                scaler_pd =pd.DataFrame(
                        columns=['scaler','time_fit',
                                 'roc_train','roc_valid', 
                                 'recall_0','precision_0',
                                 'recall_1','precision_1'],
                        data = scalers_data)
                # сохраняем dataframe
                scaler_pd.to_csv('data_recovery/scaler_pd.csv',index=False)
                
                # очистим output от прошедшей итерации
                clear_output()
                # переходим к следующему scaler
                pass
# очистим output
clear_output()

# сформируем данные соотвествующие лучщему ROC AUC на валидацинном наборе
best_roc_valid_data = scaler_pd[scaler_pd['roc_valid']==scaler_pd['roc_valid'].max()]

# выведем лучший результат на валидационном наборе
print('result:')
print('best scaler on valid:',best_roc_valid_data.iloc[0]['scaler'])
print('ROC AUC on train:', best_roc_valid_data.iloc[0]['roc_train'])
print('ROC AUC on valid:', best_roc_valid_data.iloc[0]['roc_valid'])
print('Time fit for best scaler:', best_roc_valid_data.iloc[0]['time_fit'],' seconds')

# выведем полученный dataframe
scaler_pd

# time: 

##### <span style="color:MediumSlateBlue"> Анализ влияния solver оптимизатора на качество и скорость обучения модели</span>

Проверим качество и скорость сходимости модели при разных solver  
Проверку осуществим через цикл  
Чтобы модель не сходилась бесконечно с помощью модуля func_timeout  
поставим ограничение времени схождения в 10 минут

In [ ]:
# подготовим данные для обучения модели

# обьявим scaler
scaler = StandardScaler()
# выполним scaler преобразование
X_train_s_balanced = scaler.fit_transform(X_train_balanced[list_ncorr_features])
X_train_s = scaler.transform(X_train[list_ncorr_features])
X_valid_s = scaler.transform(X_valid[list_ncorr_features])

# time: 10s

In [ ]:
# Зададим список solvers
solvers_list = ['lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga']

# сформируем список для заполнения
solvers_data = []

# установим ограничение на время обучения модели в 180 секунд (3 минут)
time_out = 180

for solver in solvers_list:
        # для того чтобы следить за выполнением цикла введем индикатор
        print('current solver: ', solver)

         # сформируем список для заполнения данными текушего solver
        current_solver_data = [solver]
        
        try:
                # для модуля func_time_out упакуем обучение модели в функцию try_func
                def try_func():
                        logistic_regression = linear_model.LogisticRegression(solver= solver,
                                                                  random_state=42, 
                                                                  max_iter=10000)
                        return logistic_regression.fit(X_train_s_balanced,y_train_balanced)
                    
                # фиксируем время начала
                start_time = time.time()
                
                # запускаем обучение модели с ограничением по времени
                logistic_regression = func_timeout.func_timeout(time_out, try_func)
                
                # фиксируем время окончания обучения
                end_time = time.time()
                
                # запишем время обучения модели
                current_solver_data.append(round(end_time-start_time))

                # Делаем предсказание для обучающей и валидационной выборки
                y_valid_lr_pred = logistic_regression.predict(X_valid_s)
                
                # для метрик ROC AUC делаем предсказание модели для обучающей и валидационной выборки  
                # в виде вероятности принадлежности к классу 1 
                y_train_lr_pred_proba = logistic_regression.predict_proba(X_train_s)[:,1]
                y_valid_lr_pred_proba = logistic_regression.predict_proba(X_valid_s)[:,1]

                #Считаем метрики ROC AUC yна обуающем и валидационном наборе и добавляем их в спискок
                current_solver_data.append(round(metrics.roc_auc_score(y_train, y_train_lr_pred_proba),3))
                current_solver_data.append(round(metrics.roc_auc_score(y_valid, y_valid_lr_pred_proba),3))

                #Считаем метрики для класса 0 на валидационном наборе и добавляем их в список
                current_solver_data.append(round(metrics.recall_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))
                current_solver_data.append(round(metrics.precision_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))


                #Считаем метрики для класса 1 на валидационном набопе и добавляем их в список
                current_solver_data.append(round(metrics.recall_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))
                current_solver_data.append(round(metrics.precision_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))

                # запишем данные полученные на одной итерации
                solvers_data.append(current_solver_data)

                # чтобы подстраховаться на случай ошибки : "The Kernel crashed while executing code"
                # сохраним в локальную папку удачную итерацию
                # для этого из полученных данных сформируем dataframe
                # из полученных данных сформируем dataframe
                solvers_pd =pd.DataFrame(
                        columns=['solver','time_fit',
                                 'roc_train','roc_valid', 
                                 'recall_0','precision_0',
                                 'recall_1','precision_1'],
                        data = solvers_data)
                # сохраняем dataframe
                solvers_pd.to_csv('data_recovery/solvers_pd.csv',index=False)
               
                # очистим output от прошедшей итерации
                clear_output()

        except func_timeout.FunctionTimedOut:
                # если время обучения привышает лимит
                # запишем в current_solver_data соотвествующую данные
                current_solver_data = [solver,'>3m',0,0,0,0,0,0]
                # добавляем полученные данные в список solvers_data
                solvers_data.append(current_solver_data)

                # чтобы подстраховаться на случай ошибки : "The Kernel crashed while executing code"
                # сохраним в локальную папку удачную итерацию
                # для этого из полученных данных сформируем dataframe
                # из полученных данных сформируем dataframe
                solvers_pd =pd.DataFrame(
                        columns=['solver','time_fit',
                                 'roc_train','roc_valid', 
                                 'recall_0','precision_0',
                                 'recall_1','precision_1'],
                        data = solvers_data)
                # сохраняем dataframe
                solvers_pd.to_csv('data_recovery/solvers_pd.csv',index=False)
                
                # очистим output от прошедшей итерации
                clear_output()
                # переходим к следующему solver
                pass
# очистим output
clear_output()

# сформируем данные соотвествующие лучщему ROC AUC на валидацинном наборе
best_roc_valid_data = solvers_pd[solvers_pd['roc_valid']==solvers_pd['roc_valid'].max()]

# выведем лучший результат на валидационном наборе
print('result:')
print('best solver on valid:',best_roc_valid_data.iloc[0]['solver'])
print('ROC AUC on train:', best_roc_valid_data.iloc[0]['roc_train'])
print('ROC AUC on valid:', best_roc_valid_data.iloc[0]['roc_valid'])
print('Time fit for best solver:', best_roc_valid_data.iloc[0]['time_fit'],' seconds')

# выведем полученный dataframe
solvers_pd

# time: 2m 35s

##### <span style="color:MediumSlateBlue">Подбор гиперпараметров модели</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'service'
count_features = dict_spaces[feature_space]

# для работы функции corr_transform_to_force
# подгрузим DataFrame с матрицей корреляции признаков
corr_matrix = fp.ParquetFile('base_models/data/stat/corr_matrix_'+feature_space).to_pandas()

# для выбора sceler преобразвания нам понадобится словарь
dict_scalers ={
    'MinMaxScaler': MinMaxScaler(),
    'RobustScaler':RobustScaler(),
    'StandardScaler':StandardScaler()}

# настроим оптимизацию гипер параметров
def optuna_lg(trial):
  # задаем пространства поиска гиперпараметров
  solver = trial.suggest_categorical("solver", ['lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky'])
  C = trial.suggest_float('C',0.01,1)
  class_0_weight = trial.suggest_float('class_0_weight',0.2,0.6)
  class_1_weight = trial.suggest_float('class_1_weight',0.2,0.6)
  threshold = trial.suggest_float('threshold',0.6,1)
  class_1_percent = trial.suggest_float('class_1_percent',0.01,0.3)
  scaler = trial.suggest_categorical("scaler", ['MinMaxScaler','RobustScaler','StandardScaler'])
  random_state = trial.suggest_int('random_state', 1, 1000000)


  # создаем модель
  optuna_log_reg = linear_model.LogisticRegression(
      solver=solver,
      C=C,
      class_weight={0:class_0_weight,1:class_1_weight},
      random_state=random_state,
      max_iter=10000)

  # с помощью функции corr_transform_to_force получим
  # список не коррелируемых признаков
  list_ncorr_features = corr_transform_to_force(corr_matrix,threshold=threshold)[1]

  # обьявим scaler
  scaler = dict_scalers[scaler]

  # сформируем данные для анализа сбалансированности обучающих данных
  X_train_pd = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
  y_train_pd = pd.read_csv('target/target_train.csv')

  # поготовим данные y_train_pd к работе с функцией class_1_percent_samples
  y_train_pd.set_index('id',drop=False,inplace=True)

  # подготовим данные для обучения модели
  # с помощью функции class_1_percent_samples зададим долю 
  # класса 1
  list_c1_percent_id = class_1_percent_samples(y_train_pd,class_1_percent,random_state=random_state)[:1000000]

  # подготовим данные для обучения
  X_train_s_balanced = scaler.fit_transform(X_train_pd.loc[list_c1_percent_id])
  y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

  # освободим память от "тяжелых" и ненужных файлов
  del X_train_pd, y_train_pd
  gc.collect()

  # я не воспользовался параметром timeout метода optimize потому, что  
  # optimize отставливает поиск параметров, если время обучения 
  # превышает timeout. Мне нужно чтобы поиск параметров продолжился.
  try:
    # для модуля func_time_out упакуем обучение модели в функцию try_func
    def try_func():
            return optuna_log_reg.fit(X_train_s_balanced, y_train_balanced)
    # обучаем модель с ограничением по времени 10 минут (600 секунд)
    optuna_log_reg = func_timeout.func_timeout(600, try_func)

    # удаляем крупные файлы чтобы высвободить память 
    del X_train_s_balanced, y_train_balanced
    gc.collect()

    # загружаем тестовые наборы
    # сформируем данные для анализа сбалансированности обучающих данных
    X_train = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
    X_valid = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_valid').to_pandas()[list_ncorr_features]
    y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
    y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()

    # подготовим данные для проверки модели
    X_train_s = scaler.transform(X_train)
    X_valid_s = scaler.transform(X_valid)

    # удаляем крупные файлы чтобы высвободить память 
    del X_train, X_valid
    gc.collect()

    # делаем предсказание на обучающем и валидационном наборе и считаем метрики
    roc_train = metrics.roc_auc_score(y_train, optuna_log_reg.predict_proba(X_train_s)[:,1])
    roc_valid = metrics.roc_auc_score(y_valid, optuna_log_reg.predict_proba(X_valid_s)[:,1])
    f1_score = metrics.f1_score(y_valid, optuna_log_reg.predict(X_valid_s))
    recall_1 = metrics.recall_score(y_valid, optuna_log_reg.predict(X_valid_s),average=None,zero_division=0)[1]
    precision_1 = metrics.precision_score(y_valid, optuna_log_reg.predict(X_valid_s),average=None,zero_division=0)[1]

    # удаляем крупные файлы чтобы высвободить память 
    del X_train_s, X_valid_s, y_train, y_valid
    gc.collect()

  except func_timeout.FunctionTimedOut:
    # удаляем крупные файлы чтобы высвободить память 
    del X_train_s_balanced, y_train_balanced
    gc.collect()
    
    # фиксируем пустые значения метрик
    roc_train = 0
    roc_valid = 0
    f1_score = 0
    recall_1 = 0
    precision_1 = 0
    pass

  return roc_train, roc_valid, f1_score, recall_1, precision_1

In [ ]:
# cоздаем объект исследования
# чтобы модель не переобучалась минимизируем roc_train 
# и максимизируем roc_valid
optuna_study_lg = optuna.create_study(study_name='LogisticRegression_'+feature_space+'_stat', 
                               directions=['maximize','maximize','maximize','maximize','maximize'], 
                               sampler=optuna.samplers.TPESampler(),
                               pruner='Hyperband',
                               storage='sqlite:///optuna_studies.db',
                               load_if_exists=True)
# optuna.delete_study(study_name='LogisticRegression_'+feature_space+'_stat', storage='sqlite:///optuna_studies.db')

In [ ]:
# ищем лучшую комбинацию гиперпараметров n_trials раз
optuna_study_lg.optimize(optuna_lg, n_trials=300)

In [ ]:
# из полученного результат соврмируем Data Frame
optuna_study_lg_pd = optuna_study_lg.trials_dataframe().dropna()
# переименуем столбы
optuna_study_lg_pd.rename(columns={
    'values_0': 'roc_train',
    'values_1': 'roc_valid',
    'values_2': 'f1_score',
    'values_3': 'recall_1',
    'values_4': 'precision_1'
},inplace=True)
# Выделим время потраченное на обучение модели
optuna_study_lg_pd.tail()

In [ ]:
# покаждем статистику обучения
print('Максимальное значение метрики ROC AUC на валидационном наборе:',round(optuna_study_lg_pd['roc_valid'].max(),3))
print('Среднее значение метрики ROC AUC на валидационном наборе:',round(optuna_study_lg_pd['roc_valid'].mean(),3))
print('Максимальное значение метрики f1_score на валидационном наборе:',round(optuna_study_lg_pd['f1_score'].max(),3))
print('Среднее значение метрики f1_score на валидационном наборе:',round(optuna_study_lg_pd['f1_score'].mean(),3))
print('Максимальное значение метрики recall_1 на валидационном наборе:',round(optuna_study_lg_pd['recall_1'].max(),3))
print('Среднее значение метрики recall_1 на валидационном наборе:',round(optuna_study_lg_pd['recall_1'].mean(),3))
print('Максимальное значение метрики precision_1 на валидационном наборе:',round(optuna_study_lg_pd['precision_1'].max(),3))
print('Среднее значение метрики precision_1 на валидационном наборе:',round(optuna_study_lg_pd['precision_1'].mean(),3))

In [ ]:
# построим график важности гиперпарметров
optuna.visualization.plot_param_importances(optuna_study_lg, target_name='Score')

In [ ]:
# Построим зависимость метрики precision_1
fig = px.scatter(
    data_frame=optuna_study_lg_pd,
    x='precision_1', #ось абсцисс
    y=['f1_score', 'recall_1'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision от recall на классе 1', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision',
    yaxis_title='Величина метрик recall и f1-score')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики precision_1 от гипер параметров

fig = px.scatter(
    data_frame=optuna_study_lg_pd,
    x='precision_1', #ось абсцисс
    y=['params_C', 'params_class_0_weight', 
       'params_class_1_weight','params_class_1_percent',
       'params_threshold'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики f1_score от гипер параметров

fig = px.scatter(
    data_frame=optuna_study_lg_pd,
    x='f1_score', #ось абсцисс
    y=['params_C', 'params_class_0_weight', 
       'params_class_1_weight','params_class_1_percent',
       'params_threshold'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость f1-score от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики f1-score',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики roc_valid от гипер параметров

fig = px.scatter(
    data_frame=optuna_study_lg_pd,
    x='roc_valid', #ось абсцисс
    y=['params_C', 'params_class_0_weight', 
       'params_class_1_weight','params_class_1_percent',
       'params_threshold'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость ROC AUC на валидационном наборе от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики ROC AUC',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Визуализируем метрики полученные при optuna оптимизации
fig = px.scatter(
    data_frame=optuna_study_lg_pd[(optuna_study_lg_pd['precision_1']>0.08)
                                  & (optuna_study_lg_pd['recall_1']>0.08) 
                                  & (optuna_study_lg_pd['roc_valid']>0.8*optuna_study_lg_pd['roc_valid'].max())],
    x='number', #ось абсцисс
    y=['roc_train','roc_valid','recall_1','precision_1'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость качества модели от trials optuna', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='trials optuna',
    yaxis_title='Величина метрики')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

<span style="color:Blue">

Вывод:  

Их всех выбираем точку $number =234$.   
В этой точке самые оптимальные значения $recall$ и $precision$.  
Посмотрим каким значениям гиперпараметров соотвествует выбранная точка.

In [ ]:
# определим номер лучшге варианта
best_optuna_number = 234

# сформируем словарь лучших гипирпарметров RandomForestClassifier
best_param_lr = {
    'solver' : optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_solver'].iloc[0],
    'C' : round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_C'].iloc[0],3),
    'class_weight' : {0:round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_class_0_weight'].iloc[0],3),
                      1:round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_class_1_weight'].iloc[0],3)},
    }

# создадим перменные
best_threshold = optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_threshold'].iloc[0]
best_scaler = optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_scaler'].iloc[0]
best_class_1_percent = optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_class_1_percent'].iloc[0]
best_random_state = optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_random_state'].iloc[0]

# Выведем принятые наилучшие праметры
print('best solver:',best_param_lr['solver'])
print('best C:',best_param_lr['C'])
print('best class 0 weight:',best_param_lr['class_weight'][0])
print('best class 1 weight:',best_param_lr['class_weight'][1])

print('best class 1 percent:',round(best_class_1_percent,3))
print('best scaler:',best_scaler)
print('best threshold:',round(best_threshold,3))
print('best best random state:',best_random_state)
print('time for best train:',round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['duration'].iloc[0].seconds/60),'minutes')

print()
print('ROC AUC на обучающем наборе:', round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['roc_train'].iloc[0],3))
print('ROC AUC на валидационном наборе:', round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['roc_valid'].iloc[0],3))
print('precision класса 1:', round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['precision_1'].iloc[0],3))
print('recall класса 1:', round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['recall_1'].iloc[0],3))


##### <span style="color:MediumSlateBlue">Обучение модели с лучшими параметрами</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'service'
count_features = dict_spaces[feature_space]

# для работы функции corr_transform_to_force
# подгрузим DataFrame с матрицей корреляции признаков
corr_matrix = fp.ParquetFile('base_models/data/stat/corr_matrix_'+feature_space).to_pandas()

# подготовим данные
# для выбора sceler преобразвания нам понадобится словарь
dict_scalers ={
    'MinMaxScaler': MinMaxScaler(),
    'RobustScaler':RobustScaler(),
    'StandardScaler':StandardScaler(),
}

# с помощью функции corr_transform_to_force получим
# список не коррелируемых признаков
list_ncorr_features = corr_transform_to_force(corr_matrix,threshold=best_threshold)[1]

# обьявим scaler
scaler = dict_scalers[best_scaler]

# сформируем данные для анализа сбалансированности обучающих данных
X_train_pd = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
y_train_pd = pd.read_csv('target/target_train.csv')

# поготовим данные y_train_pd к работе с функцией class_1_percent_samples
y_train_pd.set_index('id',drop=False,inplace=True)

# подготовим данные для обучения модели
# с помощью функции class_1_percent_samples зададим долю 
# класса 1
list_c1_percent_id = class_1_percent_samples(y_train_pd,best_class_1_percent,random_state=best_random_state)[:1000000]

# подготовим данные для обучения
X_train_s_balanced = scaler.fit_transform(X_train_pd.loc[list_c1_percent_id])
y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

# освободим память от "тяжелых" и ненужных файлов
del X_train_pd, y_train_pd
gc.collect()


# обучаем модель LogisticRegression с наилучшеми параметрами
logistic_regression = linear_model.LogisticRegression(
        **best_param_lr,
        random_state=42,
        max_iter=10000)
logistic_regression.fit(X_train_s_balanced, y_train_balanced)

# удаляем крупные файлы чтобы высвободить память 
del X_train_s_balanced, y_train_balanced
gc.collect()

# загружаем тестовые наборы
# сформируем данные для анализа сбалансированности обучающих данных
X_train = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
X_valid = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_valid').to_pandas()[list_ncorr_features]
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()

# подготовим данные для проверки модели
X_train_s = scaler.transform(X_train)
X_valid_s = scaler.transform(X_valid)

# удаляем крупные файлы чтобы высвободить память 
del X_train, X_valid
gc.collect()

# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_lr_pred_proba = logistic_regression.predict_proba(X_train_s)[:,1]
y_valid_lr_pred_proba = logistic_regression.predict_proba(X_valid_s)[:,1]

# Делаем предсказание для валидационной выборки
y_valid_lr_pred = logistic_regression.predict(X_valid_s)

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_lr_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_lr_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_valid,y_valid_lr_pred,zero_division=0))

# time: 2m 30s

##### <span style="color:MediumSlateBlue">Анализ влияния порога классификации на качество модели</span>

In [ ]:
# чтобы проше обращаться к каждому обьекту в данных
# преобразуем y_valid_LR_pred к Series типу
y_valid_lr_pred_proba = pd.Series(y_valid_lr_pred_proba)

# формируем список из необходимых значений thresholds
thresholds = np.arange(0.01, 1, 0.01)

# сформируем списко для заполнения данными результатов анализа
threshold_data = []


for threshold in thresholds:
        # сформируем список для заполнения данными текушего threshold
        threshold_current_data = [threshold]

        #клиентов, для которых вероятность дефолта > threshold, относим к классу 1
        #в противном случае — к классу 0
        y_valid_lr_pred = y_valid_lr_pred_proba.apply(lambda x: 1 if x>threshold else 0)


        #Считаем метрики для класса 0 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.precision_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.f1_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[0],3))

        #Считаем метрики для класса 1 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.precision_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.f1_score(y_valid, y_valid_lr_pred,average=None,zero_division=0)[1],3))

        # добавляем полученные данные в список threshold_data
        threshold_data.append(threshold_current_data)

        # из полученных данных сформируем dataframe
        thresholds_pd =pd.DataFrame(
        columns=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'],
        data = threshold_data)

# time: 17s

In [ ]:
# Визуализируем метрики на валидационном наборе при различных threshold
fig_threshold = px.line(
    data_frame=thresholds_pd,
    x='threshold', #ось абсцисс
    y=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'], #ось ординат
)
fig_threshold.update_layout(
    title ={
        'text':'Зависимость качества модели от threshold', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Порог классификации threshold',
    yaxis_title='Величина метрики')
fig_threshold.update_xaxes(showspikes=True)
fig_threshold.update_yaxes(showspikes=True)

fig_threshold.show()

Метрика $precision$ показывает на сколько сильно ошиблась модель при определиние класса.  
Чем выше метрика, тем меньше ошиблась модель. 

$$precision = \frac{TP}{TP+FP}$$

Метрика $recall$ показывает способность модели найти класс.  
Чем выше метрика, тем выше способность модели находить класс.  

$$recall = \frac{TP}{TP+FN}$$

Метрика $f1-score$ показывает баланс между $precision$ и $recall$ 

$threshold$- порог класификации модели.  
Все объекты для которых $у_{\text{proba pred}} < threshold$, модель относит к класс 0.  
Соотвественно, если $у_{\text{proba pred}}  > threshold$ модель относит обьект к классу 1.  
Для идельаной модели $threshold$, является границей между областями класса 0 и класса 1.  

<span style="color:Blue">

Выводы по результам анализа:  

Для улучшения качества модели по метрике $ROC AUC$, нет смысла сдвигать порог   
классификации. Но зависимость метрик качества модели от $threshold$, может рассказать о  
спосбоности модели искать класс 1 и класс 0.

При фокусировке модели на классе 1 (более низкий $threshold$):  
- на классе 1 метрика $precision$ уменьшается, метрика $recall$ растет;  
- на классе 0 метрика $precision$ растет, метрика $recall$ уменьшается.  

При фокусировке модели на классе 0 (более высокий $threshold$):  
- на классе 1 метрика $precision$ увеличивается, метрика $recall$ уменьшается;  
- на классе 0 метрика $precision$ уменьшается, метрика $recall$ растет.  

Обьсняется тем, что в условия дисбаланса модель научилась хорошо определять класс 0 и плохо класс 1.  

При фокусировке модели на классе 1 (более низкий $threshold$), все больше обьектов отнесены к классу 1.   
Так как $precision$ класса 1 уменьшается, значит в области класса 1 стало больше обектов класса 0,  
что явлется закономерным для "идеальной" модели.  
Однако при этом, $recall$ класса 1 растет, значит в добавленных обьектах также присутвует класс 1.  
Это как раз указывает на то, что модель не научилась находить среди класса 0 класс 1. В области с     
высокой вероятностью класса 0 (более низкий $threshold$), находятся обьекты класса 1.  

Фокусировка на классе 0 (более высокий $threshold$), заставляет модель все больше обьектов из области класса 1  
относить к классу 0. При этом увеличение метрики $precision$ класса 1 говорит о том, что в области класса 1     
становится меньше 0, а значит модель умеет среди класса 1, находить класс 0.  
После  $threshold=0.64$ модель впринципе теряет способность отличать класс 1. У модели нет обьектов класса 1   
в которых она "уверена" на 80+%.

Обратим внимание на $threshold=0.52$, в этой точке находится баланс межу метриками качества модели:
$$precision_1 = recall_1 = \text{f1-score}_1 = 0.082$$
$$precision_0 = recall_0 = \text{f1-score}_0 = 0.965$$

Для модели с лучшим качеством эти две точки должны находится максимально близко к друг другу и иметь  
высокое значение метрик $precision_1$ и $recall_1$.
Для класса 0 это условие выполнется, что также говорит о том, что модель научилась искать этот класс.  
Для класса 1 это условие не выполнется, что также говорит о том, что модель плохо ищет класс 1.

##### <span style="color:MediumSlateBlue">Построение ROC-кривой выбранной модели на тестовом наборе</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'service'
count_features = dict_spaces[feature_space]

In [ ]:
# сформируем данные для проверки модели
X_train = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
X_valid = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_valid').to_pandas()[list_ncorr_features]
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()
X_test = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_test').to_pandas()[list_ncorr_features]
y_test = pd.read_csv('target/target_test.csv')['flag'].to_numpy()

In [ ]:
# выполним scaler преобразование
X_train_s = scaler.transform(X_train)
X_valid_s = scaler.transform(X_valid)
X_test_s = scaler.transform(X_test)

print("Размеры обучающего набора",X_train_s.shape)
print("Размеры валидационного набора",X_valid_s.shape)
print("Размеры тестового набора",X_test_s.shape)
# time: 1m 43s

In [ ]:
# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_lr_pred_proba = logistic_regression.predict_proba(X_train_s)[:,1]
y_valid_lr_pred_proba = logistic_regression.predict_proba(X_valid_s)[:,1]
y_test_lr_pred_proba = logistic_regression.predict_proba(X_test_s)[:,1]

# Делаем предсказание для валидационной выборки
y_test_lr_pred = logistic_regression.predict(X_test_s)

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_lr_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_lr_pred_proba),3))
print('ROC AUC на тестовом наборе', round(metrics.roc_auc_score(y_test, y_test_lr_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_test,y_test_lr_pred,zero_division=0))

In [ ]:
# Для расчета значений TPR и FPR используем функцию roc_curve
fpr_train, tpr_train, _ = metrics.roc_curve(y_train, y_train_lr_pred_proba)
fpr_valid, tpr_valid, _ = metrics.roc_curve(y_valid, y_valid_lr_pred_proba)
fpr_test, tpr_test, _ = metrics.roc_curve(y_test, y_test_lr_pred_proba)

# рассчитаем площадь под кривой
auc_train = round(metrics.roc_auc_score(y_train, y_train_lr_pred_proba),3)
auc_valid = round(metrics.roc_auc_score(y_valid, y_valid_lr_pred_proba),3)
auc_test = round(metrics.roc_auc_score(y_test, y_test_lr_pred_proba),3)

trace_train = go.Scatter(x=fpr_train, y=tpr_train, mode='lines', name='AUC_train = %0.2f' % auc_train,
                   line=dict(color='red', width=2),
                   hovertemplate='train<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_train])

trace_valid = go.Scatter(x=fpr_valid, y=tpr_valid, mode='lines', name='AUC_valid = %0.2f' % auc_valid,
                   line=dict(color='green', width=2),
                   hovertemplate='valid<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_valid])

trace_test = go.Scatter(x=fpr_test, y=tpr_test, mode='lines', name='AUC_test = %0.2f' % auc_test,
                   line=dict(color='darkorange', width=2),
                   hovertemplate='test<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_test])


reference_line = go.Scatter(x=[0,1], y=[0,1], mode='lines', name='Reference Line',
                            line=dict(color='navy', width=2, dash='dash'))

fig_roc = go.Figure(data=[trace_train,trace_valid,trace_test,reference_line])

fig_roc.update_layout(
    title ={
        'text':'ROC-кривая', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 1350,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate')

fig_roc.update_xaxes(showspikes=True)
fig_roc.update_yaxes(showspikes=True)


fig_roc.show()

<span style="color:Blue">

Вывод по ROC-кривой:
1. Модель выдает сталибильное качество на всех наборах, а значит  
не переобучена - разброс ($variance$) не большой.  
2. Смещение ($bias$) модели относительно большое.  
На тестовом наборе полученно качество $ROC AUC = 0.6$.

#### <span style="color:MediumBlue">Построение модели Random Forest Classifier</span>

##### <span style="color:MediumSlateBlue">Baseline обучение модели RandomForestClassifier</span>

In [ ]:
# обучаем модель логистической регрессии
# чтобы модель пыталась найти связь во всех признаках
random_forest = RandomForestClassifier(random_state=42, n_jobs=-1)
random_forest.fit(X_train_balanced[list_ncorr_features],y_train_balanced)

# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_rf__pred_proba = random_forest.predict_proba(X_train[list_ncorr_features])[:,1]
y_valid_rf_pred_proba = random_forest.predict_proba(X_valid[list_ncorr_features])[:,1]

# Делаем предсказание для валидационной выборки
y_valid_rf_pred = random_forest.predict(X_valid[list_ncorr_features])

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_rf__pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_rf_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_valid,y_valid_rf_pred))

# time: 17s

##### <span style="color:MediumSlateBlue">Подбор гиперпараметров модели</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'service'
count_features = dict_spaces[feature_space]

# для работы функции corr_transform_to_force
# подгрузим DataFrame с матрицей корреляции признаков
corr_matrix = fp.ParquetFile('base_models/data/stat/corr_matrix_'+feature_space).to_pandas()

# настроим оптимизацию гипер параметров
def optuna_rf(trial):
  # задаем пространства поиска гиперпараметров
  # задаем пространства поиска гиперпараметров
  n_estimators = trial.suggest_int('n_estimators', 40, 100,step = 5)
  criterion = trial.suggest_categorical('criterion',['gini', 'entropy', 'log_loss'])
  max_depth = trial.suggest_int('max_depth', 10, 30,step = 1)
  min_samples_split = trial.suggest_int('min_samples_split', 5, 15,step = 1)
  min_samples_leaf = trial.suggest_int('min_samples_leaf', 5, 15,step = 1)
  max_features = trial.suggest_float('max_features',0.4,0.8)
  max_samples = trial.suggest_float('max_samples',0.4,0.8)
  class_0_weight = trial.suggest_float('class_0_weight',0.1,1)
  class_1_weight = trial.suggest_float('class_1_weight',0.1,1)
  threshold = trial.suggest_float('threshold',0.4,0.8)
  class_1_percent = trial.suggest_float('class_1_percent',0.01,0.5)
  random_state = trial.suggest_int('random_state', 1, 1000000)


  # создаем модель
  optuna_random_forest = RandomForestClassifier(
      n_estimators = n_estimators, 
      criterion = criterion,
      max_depth = max_depth,
      min_samples_split = min_samples_split,  
      min_samples_leaf = min_samples_leaf,
      max_features=max_features,
      max_samples=max_samples,
      class_weight={0:class_0_weight,1:class_1_weight},
      n_jobs=-1,
      random_state=42)

  # с помощью функции corr_transform_to_force получим
  # список не коррелируемых признаков
  list_ncorr_features = corr_transform_to_force(corr_matrix,threshold=threshold)[1]

  # сформируем данные для анализа сбалансированности обучающих данных
  X_train_pd = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
  y_train_pd = pd.read_csv('target/target_train.csv')

  # поготовим данные y_train_pd к работе с функцией class_1_percent_samples
  y_train_pd.set_index('id',drop=False,inplace=True)

  # подготовим данные для обучения модели
  # с помощью функции class_1_percent_samples зададим долю 
  # класса 1
  list_c1_percent_id = class_1_percent_samples(y_train_pd,class_1_percent,random_state=random_state)[:1000000]

  # подготовим данные для обучения
  X_train_balanced = X_train_pd.loc[list_c1_percent_id]
  y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag']

  # освободим память от "тяжелых" и ненужных файлов
  del X_train_pd, y_train_pd
  gc.collect()

  # я не воспользовался параметром timeout метода optimize потому, что  
  # optimize отставливает поиск параметров, если время обучения 
  # превышает timeout. Мне нужно чтобы поиск параметров продолжился.
  try:
    # для модуля func_time_out упакуем обучение модели в функцию try_func
    def try_func():
            return optuna_random_forest.fit(X_train_balanced, y_train_balanced)
    # обучаем модель с ограничением по времени 10 минут (600 секунд)
    optuna_random_forest = func_timeout.func_timeout(600, try_func)

    # удаляем крупные файлы чтобы высвободить память 
    del X_train_balanced, y_train_balanced
    gc.collect()

    # загружаем тестовые наборы
    # сформируем данные для анализа сбалансированности обучающих данных
    X_train = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
    X_valid = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_valid').to_pandas()[list_ncorr_features]
    y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
    y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()
    
    # делаем предсказание на обучающем и валидационном наборе
    #Считаем метрики для класса 1 добавляем их в список
    roc_train = metrics.roc_auc_score(y_train, optuna_random_forest.predict_proba(X_train)[:,1])
    roc_valid = metrics.roc_auc_score(y_valid, optuna_random_forest.predict_proba(X_valid)[:,1])
    f1_score = metrics.f1_score(y_valid, optuna_random_forest.predict(X_valid))
    recall_1 = metrics.recall_score(y_valid, optuna_random_forest.predict(X_valid),average=None,zero_division=0)[1]
    precision_1 = metrics.precision_score(y_valid, optuna_random_forest.predict(X_valid),average=None,zero_division=0)[1]

    # удаляем крупные файлы чтобы высвободить память 
    del X_train, X_valid, y_train, y_valid
    gc.collect()

  except func_timeout.FunctionTimedOut:
    # удаляем крупные файлы чтобы высвободить память 
    del X_train_balanced, y_train_balanced
    gc.collect()
    
    # фиксируем пустые значения метрик
    roc_train = 0
    roc_valid = 0
    f1_score = 0
    recall_1 = 0
    precision_1 = 0
    pass

  return roc_train, roc_valid, f1_score, recall_1, precision_1

In [ ]:
# cоздаем объект исследования
optuna_study_rf = optuna.create_study(study_name= 'RandomForestClassifier_'+feature_space+'_stat', 
                               directions=['maximize','maximize','maximize','maximize','maximize'], 
                               sampler=optuna.samplers.TPESampler(),
                               pruner='Hyperband',
                               storage='sqlite:///optuna_studies.db',
                               load_if_exists=True)
# на случай удаления 
# optuna.delete_study(study_name='RandomForestClassifier_'+feature_space+'_stat', storage='sqlite:///optuna_studies.db')

In [ ]:
# ищем лучшую комбинацию гиперпараметров n_trials раз
optuna_study_rf.optimize(optuna_rf, n_trials=300)

In [ ]:
# из полученного результат соврмируем Data Frame
optuna_study_rf_pd = optuna_study_rf.trials_dataframe().dropna()
# переименуем столбы
optuna_study_rf_pd.rename(columns={
    'values_0': 'roc_train',
    'values_1': 'roc_valid',
    'values_2': 'f1_score',
    'values_3': 'recall_1',
    'values_4': 'precision_1'
},inplace=True)
optuna_study_rf_pd.tail()

In [ ]:
# покаждем статистику обучения
print('Максимальное значение метрики ROC AUC на валидационном наборе:',round(optuna_study_rf_pd['roc_valid'].max(),3))
print('Среднее значение метрики ROC AUC на валидационном наборе:',round(optuna_study_rf_pd['roc_valid'].mean(),3))
print('Максимальное значение метрики f1_score на валидационном наборе:',round(optuna_study_rf_pd['f1_score'].max(),3))
print('Среднее значение метрики f1_score на валидационном наборе:',round(optuna_study_rf_pd['f1_score'].mean(),3))
print('Максимальное значение метрики recall_1 на валидационном наборе:',round(optuna_study_rf_pd['recall_1'].max(),3))
print('Среднее значение метрики recall_1 на валидационном наборе:',round(optuna_study_rf_pd['recall_1'].mean(),3))
print('Максимальное значение метрики precision_1 на валидационном наборе:',round(optuna_study_rf_pd['precision_1'].max(),3))
print('Среднее значение метрики precision_1 на валидационном наборе:',round(optuna_study_rf_pd['precision_1'].mean(),3))

In [ ]:
# Построим зависимость метрики precision_1

fig = px.scatter(
    data_frame=optuna_study_rf_pd,
    x='precision_1', #ось абсцисс
    y=['recall_1','f1_score'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision от recall на классе 1', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision на классе 1',
    yaxis_title='Величина метрик recall и f1-score')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики precision_1 от гиперпараметров

fig = px.scatter(
    data_frame=optuna_study_rf_pd,
    x='precision_1', #ось абсцисс
    y=['params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight', 'params_max_depth',
       'params_max_features', 'params_max_samples', 'params_min_samples_leaf',
       'params_min_samples_split', 'params_n_estimators', 'params_threshold'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision класса 1 от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision на классе 1',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики f1_score от гиперпараметров

fig = px.scatter(
    data_frame=optuna_study_rf_pd,
    x='f1_score', #ось абсцисс
    y=['params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight', 'params_max_depth',
       'params_max_features', 'params_max_samples', 'params_min_samples_leaf',
       'params_min_samples_split', 'params_n_estimators', 'params_threshold'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость f1-score класса 1 от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики f1-score на классе 1',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики roc_valid от гиперпараметров

fig = px.scatter(
    data_frame=optuna_study_rf_pd,
    x='roc_valid', #ось абсцисс
    y=['params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight', 'params_max_depth',
       'params_max_features', 'params_max_samples', 'params_min_samples_leaf',
       'params_min_samples_split', 'params_n_estimators', 'params_threshold'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость ROC AUC на валидационном наборе от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики ROC AUV на валидационном наборе',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрик качества модели от number

fig = px.scatter(
    data_frame=optuna_study_rf_pd[(optuna_study_rf_pd['recall_1']>=0.08) & (optuna_study_rf_pd['precision_1']>0.08)],
    x='number', #ось абсцисс
    y=['roc_valid','roc_train','precision_1','recall_1'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость качества модели от trials optuna', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='trials optuna',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

<span style="color:Blue">

Вывод:  

Их всех выбираем точку $number =188$.   
Не смотря на, то что в этой точке модель подает признаки переобучеености,  
в этой точке оптимальные значения метрик $recall_1$ и $precision_1$, а  
также относительно высокая метрика $ROC AUC$.   
Посмотрим каким значениям гиперпараметров соотвествует выбраннаяточка.

In [ ]:
# определим номер лучшге варианта
best_optuna_number = 188

# сформируем словарь лучших гипирпарметров RandomForestClassifier
best_param_rf = {
    'n_estimators' : int(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_n_estimators'].iloc[0]),
    'criterion': optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_criterion'].iloc[0],
    'max_depth' : int(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_max_depth'].iloc[0]),
    'min_samples_split': int(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_min_samples_split'].iloc[0]),
    'min_samples_leaf' : int(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_min_samples_leaf'].iloc[0]),
    'max_features': optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_max_features'].iloc[0],
    'max_samples' : optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_max_samples'].iloc[0],
    'class_weight' : {0:optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_class_0_weight'].iloc[0],
                      1:optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_class_1_weight'].iloc[0]}
    }

# определим перменные для лучших значений параметров
best_threshold = optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_threshold'].iloc[0]
best_class_1_percent = optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_class_1_percent'].iloc[0]
best_random_state = optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['params_random_state'].iloc[0]


# Выведем принятые наилучшие праметры
print('best n estimators:',best_param_rf['n_estimators'])
print('best criterion:',best_param_rf['criterion'])
print('best max depth:',best_param_rf['max_depth'])
print('best min samples split:',best_param_rf['min_samples_split'])
print('best min samples leaf:',best_param_rf['min_samples_leaf'])
print('best max features(%):',round(best_param_rf['max_features'],3))
print('best max samples(%):',round(best_param_rf['max_samples'],3))
print('best class 0 weight:',round(best_param_rf['class_weight'][0],3))
print('best class 1 weight:',round(best_param_rf['class_weight'][1],3))

print('best class 1 percent:',round(best_class_1_percent,3))
print('best threshold:',round(best_threshold,3))
print('best random state:', best_random_state)
print('time for best train:',round(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['duration'].iloc[0].seconds/60),'minutes')

print()
print('ROC AUC на обучающем наборе:', round(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['roc_train'].iloc[0],3))
print('ROC AUC на валидационном наборе:', round(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['roc_valid'].iloc[0],3))
print('precision класса 1:', round(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['precision_1'].iloc[0],3))
print('recall класса 1:', round(optuna_study_rf_pd[optuna_study_rf_pd['number']==best_optuna_number]['recall_1'].iloc[0],3))

##### <span style="color:MediumSlateBlue">Обучение модели с лучшими параметрами</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'service'
count_features = dict_spaces[feature_space]

# для работы функции corr_transform_to_force
# подгрузим DataFrame с матрицей корреляции признаков
corr_matrix = fp.ParquetFile('base_models/data/stat/corr_matrix_'+feature_space).to_pandas()

# с помощью функции corr_transform_to_force получим
# список не коррелируемых признаков
list_ncorr_features = corr_transform_to_force(corr_matrix,threshold=best_threshold)[1]

# сформируем данные для анализа сбалансированности обучающих данных
X_train_pd = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
y_train_pd = pd.read_csv('target/target_train.csv')

# поготовим данные y_train_pd к работе с функцией class_1_percent_samples
y_train_pd.set_index('id',drop=False,inplace=True)

# подготовим данные для обучения модели
# с помощью функции class_1_percent_samples зададим долю 
# класса 1
list_c1_percent_id = class_1_percent_samples(y_train_pd,best_class_1_percent,random_state=best_random_state)[:1000000]

# подготовим данные для обучения
X_train_balanced = X_train_pd.loc[list_c1_percent_id]
y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag']

# освободим память от "тяжелых" и ненужных файлов
del X_train_pd, y_train_pd
gc.collect()


# обучаем модель RandomForestClassifier с наилучшеми параметрами
random_forest = RandomForestClassifier(
        **best_param_rf,
        n_jobs=-1,
        random_state=42)
random_forest.fit(X_train_balanced, y_train_balanced)

# удаляем крупные файлы чтобы высвободить память 
del X_train_balanced, y_train_balanced
gc.collect()

# загружаем тестовые наборы
# сформируем данные для анализа сбалансированности обучающих данных
X_train = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
X_valid = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_valid').to_pandas()[list_ncorr_features]
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()

# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_rf_pred_proba = random_forest.predict_proba(X_train)[:,1]
y_valid_rf_pred_proba = random_forest.predict_proba(X_valid)[:,1]

# Делаем предсказание для валидационной выборки
y_valid_rf_pred = random_forest.predict(X_valid)

# удаляем крупные файлы чтобы высвободить память 
del X_train, X_valid
gc.collect()

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_rf_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_rf_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_valid,y_valid_rf_pred,zero_division=0))

# time: 5m 30s

##### <span style="color:MediumSlateBlue">Анализ влияния порога классификации на качество модели</span>

In [ ]:
# чтобы проше обращаться к каждому обьекту в данных
# преобразуем y_valid_LR_pred к Series типу
y_valid_rf_pred_proba = pd.Series(y_valid_rf_pred_proba)

# формируем список из необходимых значений thresholds
thresholds = np.arange(0.01, 1, 0.01)

# сформируем списко для заполнения данными результатов анализа
threshold_data = []


for threshold in thresholds:
        # сформируем список для заполнения данными текушего threshold
        threshold_current_data = [threshold]

        #клиентов, для которых вероятность дефолта > threshold, относим к классу 1
        #в противном случае — к классу 0
        y_valid_rf_pred = y_valid_rf_pred_proba.apply(lambda x: 1 if x>threshold else 0)


        #Считаем метрики для класса 0 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(y_valid, y_valid_rf_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.precision_score(y_valid, y_valid_rf_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.f1_score(y_valid, y_valid_rf_pred,average=None,zero_division=0)[0],3))

        #Считаем метрики для класса 1 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(y_valid, y_valid_rf_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.precision_score(y_valid, y_valid_rf_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.f1_score(y_valid, y_valid_rf_pred,average=None,zero_division=0)[1],3))

        # добавляем полученные данные в список threshold_data
        threshold_data.append(threshold_current_data)

        # из полученных данных сформируем dataframe
        thresholds_pd =pd.DataFrame(
        columns=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'],
        data = threshold_data)

# time: 17s

In [ ]:
# Визуализируем метрики на валидационном наборе при различных threshold
fig_threshold = px.line(
    data_frame=thresholds_pd,
    x='threshold', #ось абсцисс
    y=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'], #ось ординат
)
fig_threshold.update_layout(
    title ={
        'text':'Зависимость качества модели от threshold', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Порог классификации threshold',
    yaxis_title='Величина метрики')
fig_threshold.update_xaxes(showspikes=True)
fig_threshold.update_yaxes(showspikes=True)

fig_threshold.show()

Метрика $precision$ показывает на сколько сильно ошиблась модель при определиние класса.  
Чем выше метрика, тем меньше ошиблась модель. 

$$precision = \frac{TP}{TP+FP}$$

Метрика $recall$ показывает способность модели найти класс.  
Чем выше метрика, тем выше способность модели находить класс.  

$$recall = \frac{TP}{TP+FN}$$

Метрика $f1-score$ показывает баланс между $precision$ и $recall$ 

$threshold$- порог класификации модели.  
Все объекты для которых $у_{\text{proba pred}} < threshold$, модель относит к класс 0.  
Соотвественно, если $у_{\text{proba pred}}  > threshold$ модель относит обьект к классу 1.  
Для идельаной модели $threshold$, является границей между областями класса 0 и класса 1.  

<span style="color:Blue">

Выводы по результам анализа:  

Для улучшения качества модели по метрике $ROC AUC$, нет смысла сдвигать порог   
классификации. Но зависимость метрик качества модели от $threshold$, может рассказать о  
способности модели искать класс 1 и класс 0.

При фокусировке модели на классе 1 (более низкий $threshold$):  
- на классе 1 метрика $precision$ уменьшается, метрика $recall$ растет;  
- на классе 0 метрика $precision$ растет, метрика $recall$ уменьшается.  

При фокусировке модели на классе 0 (более высокий $threshold$):  
- на классе 1 метрика $precision$ увеличивается, метрика $recall$ уменьшается;  
- на классе 0 метрика $precision$ уменьшается, метрика $recall$ растет.  

Обьсняется тем, что в условия дисбаланса модель научилась хорошо определять класс 0 и плохо класс 1.  

При фокусировке модели на классе 1 (более низкий $threshold$), все больше обьектов отнесены к классу 1.   
Так как $precision$ класса 1 уменьшается, значит в области класса 1 стало больше обектов класса 0,  
что явлется закономерным для "идеальной" модели.  
Однако при этом, $recall$ класса 1 растет, значит в добавленных обьектах также присутвует класс 1.  
Это как раз указывает на то, что модель не нучилась находить среди класса 0 класс 1. В области с     
высокой вероятностью класса 0 (более низкий $threshold$), находятся обьекты класса 1.  

Фокусировка на классе 0 (более высокий $threshold$), заставляет модель все больше обьектов из области класса 1  
относить к классу 0. При этом увеличение метрики $precision$ класса 1 говорит о том, что в области класса 1     
становится меньше 0, а значит модель умеет среди класса 1, находить класс 0.  
После  $threshold=0.77$ модель впринципе теряет способность отличать класс 1. У модели нет обьектов класса 1   
в которых она "уверена" на 80+%.

Обратим внимание на $threshold=0.49$, в этой точке находится баланс межу метриками качества модели:
$$precision_1 = recall_1 = \text{f1-score}_1 = 0.09$$
$$precision_0 = recall_0 = \text{f1-score}_0 = 0.968$$

Для модели с лучшим качеством эти две точки должны находится максимально близко к друг другу и иметь   
высокое значение метрик $precision_1$ и $recall_1$.  
Для класса 0 это условие выполнется, что также говорит о том, что модель научилась искать этот класс.    
Для класса 1 это условие не выполнется, что также говорит о том, что модель плохо ищет класс 1.

##### <span style="color:MediumSlateBlue">Построение ROC-кривой выбранной модели на тестовом наборе</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'service'
count_features = dict_spaces[feature_space]

In [ ]:
# сформируем данные для проверки модели
X_train = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
X_valid = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_valid').to_pandas()[list_ncorr_features]
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()
X_test = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_test').to_pandas()[list_ncorr_features]
y_test = pd.read_csv('target/target_test.csv')['flag'].to_numpy()

In [ ]:
print("Размеры обучающего набора",X_train.shape,y_train.shape)
print("Размеры валидационного набора",X_valid.shape,y_valid.shape)
print("Размеры тестового набора",X_test.shape,y_test.shape)
# time: 1m 43s

In [ ]:
# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_rf_pred_proba = random_forest.predict_proba(X_train)[:,1]
y_valid_rf_pred_proba = random_forest.predict_proba(X_valid)[:,1]
y_test_rf_pred_proba = random_forest.predict_proba(X_test)[:,1]

# Делаем предсказание для валидационной выборки
y_test_lr_pred = random_forest.predict(X_test)

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_rf_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_rf_pred_proba),3))
print('ROC AUC на тестовом наборе', round(metrics.roc_auc_score(y_test, y_test_rf_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_test,y_test_lr_pred,zero_division=0))

In [ ]:
# Для расчета значений TPR и FPR используем функцию roc_curve
fpr_train, tpr_train, _ = metrics.roc_curve(y_train, y_train_rf_pred_proba)
fpr_valid, tpr_valid, _ = metrics.roc_curve(y_valid, y_valid_rf_pred_proba)
fpr_test, tpr_test, _ = metrics.roc_curve(y_test, y_test_rf_pred_proba)

# рассчитаем площадь под кривой
auc_train = round(metrics.roc_auc_score(y_train, y_train_rf_pred_proba),3)
auc_valid = round(metrics.roc_auc_score(y_valid, y_valid_rf_pred_proba),3)
auc_test = round(metrics.roc_auc_score(y_test, y_test_rf_pred_proba),3)

trace_train = go.Scatter(x=fpr_train, y=tpr_train, mode='lines', name='AUC_train = %0.2f' % auc_train,
                   line=dict(color='red', width=2),
                   hovertemplate='train<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_train])

trace_valid = go.Scatter(x=fpr_valid, y=tpr_valid, mode='lines', name='AUC_valid = %0.2f' % auc_valid,
                   line=dict(color='green', width=2),
                   hovertemplate='valid<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_valid])

trace_test = go.Scatter(x=fpr_test, y=tpr_test, mode='lines', name='AUC_test = %0.2f' % auc_test,
                   line=dict(color='darkorange', width=2),
                   hovertemplate='test<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_test])


reference_line = go.Scatter(x=[0,1], y=[0,1], mode='lines', name='Reference Line',
                            line=dict(color='navy', width=2, dash='dash'))

fig_roc = go.Figure(data=[trace_train,trace_valid,trace_test,reference_line])

fig_roc.update_layout(
    title ={
        'text':'ROC-кривая', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 1350,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate')

fig_roc.update_xaxes(showspikes=True)
fig_roc.update_yaxes(showspikes=True)


fig_roc.show()

<span style="color:Blue">

Вывод по ROC-кривой:
1. Не смотря на то, что модель показывает, очень хорошую $ROC$ кривую на обучающем наборе,  
модель, так же выдает сталибильное качество на валидационном и тестовом наборах, а значит  
не переобучена - разброс ($variance$) не большой.
2. Смещение ($bias$) модели такое же как у модели $Logistic$ $Regression$ $Classifier$,  
обученной  на тех же данных.  
На тестовом наборе полученно качество $ROC AUC = 0.61$.

#### <span style="color:MediumBlue">Построение модели Gradient Boosting Classifier</span>

##### <span style="color:MediumSlateBlue">Baseline обучение модели Hist Gradient Boosting Classifier</span>

In [ ]:
# обучаем модель Gradient Boosting Classifier
gradient_boosting = HistGradientBoostingClassifier(random_state=42)
gradient_boosting.fit(X_train_balanced[list_ncorr_features],y_train_balanced)

# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_rf__pred_proba = gradient_boosting.predict_proba(X_train[list_ncorr_features])[:,1]
y_valid_rf_pred_proba = gradient_boosting.predict_proba(X_valid[list_ncorr_features])[:,1]

# Делаем предсказание для валидационной выборки
y_valid_rf_pred = gradient_boosting.predict(X_valid[list_ncorr_features])

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_rf__pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_rf_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_valid,y_valid_rf_pred))

# time: 30s

##### <span style="color:MediumSlateBlue">Подбор гиперпараметров модели</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'service'
count_features = dict_spaces[feature_space]

# для работы функции corr_transform_to_force
# подгрузим DataFrame с матрицей корреляции признаков
corr_matrix = fp.ParquetFile('base_models/data/stat/corr_matrix_'+feature_space).to_pandas()

# настроим оптимизацию гипер параметров
def optuna_hgbc(trial):
  # задаем пространства поиска гиперпараметров
  learning_rate = trial.suggest_float('learning_rate',0.01,0.1)
  max_iter = trial.suggest_int('max_iter', 150, 250,step = 1)
  max_leaf_nodes = trial.suggest_int('max_leaf_nodes', 2, 60,step = 1)
  max_depth = trial.suggest_int('max_depth', 1, 10,step = 1)
  min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 60,step = 1)
  max_features = trial.suggest_float('max_features',0.6,1)
  l2_regularization = trial.suggest_float('l2_regularization',0.01,1)
  class_0_weight = trial.suggest_float('class_0_weight',0.01,1)
  class_1_weight = trial.suggest_float('class_1_weight',0.5,1)
  threshold = trial.suggest_float('threshold',0.7,1)
  class_1_percent = trial.suggest_float('class_1_percent',0.01,0.25)
  random_state = trial.suggest_int('random_state', 1, 1000000)

  # создаем модель
  optuna_gradient_boosting = HistGradientBoostingClassifier(
      learning_rate= learning_rate,
      max_iter = max_iter,
      max_leaf_nodes =max_leaf_nodes,
      max_depth = max_depth,
      min_samples_leaf = min_samples_leaf,
      max_features=max_features,
      l2_regularization = l2_regularization,
      class_weight={0:class_0_weight,1:class_1_weight},
      random_state=42)

  # с помощью функции corr_transform_to_force получим
  # список не коррелируемых признаков
  list_ncorr_features = corr_transform_to_force(corr_matrix,threshold=threshold)[1]

  # сформируем данные для анализа сбалансированности обучающих данных
  X_train_pd = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
  y_train_pd = pd.read_csv('target/target_train.csv')

  # поготовим данные y_train_pd к работе с функцией class_1_percent_samples
  y_train_pd.set_index('id',drop=False,inplace=True)

  # подготовим данные для обучения модели
  # с помощью функции class_1_percent_samples зададим долю 
  # класса 1
  list_c1_percent_id = class_1_percent_samples(y_train_pd,class_1_percent,random_state=random_state)[:1000000]

  # подготовим данные для обучения
  X_train_balanced = X_train_pd.loc[list_c1_percent_id]
  y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag']

  # освободим память от "тяжелых" и ненужных файлов
  del X_train_pd, y_train_pd
  gc.collect()

  # я не воспользовался параметром timeout метода optimize потому, что  
  # optimize отставливает поиск параметров, если время обучения 
  # превышает timeout. Мне нужно чтобы поиск параметров продолжился.
  try:
    # для модуля func_time_out упакуем обучение модели в функцию try_func
    def try_func():
            return optuna_gradient_boosting.fit(X_train_balanced,y_train_balanced)
    # обучаем модель с ограничением по времени 10 минут (600 секунд)
    optuna_gradient_boosting = func_timeout.func_timeout(600, try_func)

    # удаляем крупные файлы чтобы высвободить память 
    del X_train_balanced, y_train_balanced
    gc.collect()

        # загружаем тестовые наборы
    # сформируем данные для анализа сбалансированности обучающих данных
    X_train = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
    X_valid = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_valid').to_pandas()[list_ncorr_features]
    y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
    y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()
    
    # делаем предсказание на обучающем и валидационном наборе и считаем метрики
    roc_train = metrics.roc_auc_score(y_train, optuna_gradient_boosting.predict_proba(X_train)[:,1])
    roc_valid = metrics.roc_auc_score(y_valid, optuna_gradient_boosting.predict_proba(X_valid)[:,1])
    f1_score = metrics.f1_score(y_valid, optuna_gradient_boosting.predict(X_valid))
    recall_1 = metrics.recall_score(y_valid, optuna_gradient_boosting.predict(X_valid),average=None,zero_division=0)[1]
    precision_1 = metrics.precision_score(y_valid, optuna_gradient_boosting.predict(X_valid),average=None,zero_division=0)[1]


    # удаляем крупные файлы чтобы высвободить память 
    del X_train, X_valid, y_train, y_valid
    gc.collect()

  except func_timeout.FunctionTimedOut:
    # удаляем крупные файлы чтобы высвободить память 
    del X_train_balanced, y_train_balanced
    gc.collect()
    
    # фиксируем пустые значения метрик
    roc_train = 0
    roc_valid = 0
    f1_score = 0
    recall_1 = 0
    precision_1 = 0
    pass

  return roc_train, roc_valid, f1_score, recall_1, precision_1

In [ ]:
# cоздаем объект исследования
# чтобы модель не переобучалась минимизируем roc_train 
# и максимизируем roc_valid
optuna_study_hgbc = optuna.create_study(study_name='HistGradientBoostingClassifier_'+feature_space+'_stat', 
                               directions=['maximize','maximize','maximize','maximize','maximize'], 
                               sampler=optuna.samplers.TPESampler(),
                               pruner='Hyperband',
                               storage='sqlite:///optuna_studies.db',
                               load_if_exists=True)

# ну случай удаления обучения
# optuna.delete_study(study_name='HistGradientBoostingClassifier'+feature_space+'_stat', storage='sqlite:///optuna_studies.db')

In [ ]:
# ищем лучшую комбинацию гиперпараметров n_trials раз
optuna_study_hgbc.optimize(optuna_hgbc, n_trials=300)

In [ ]:
# из полученного результат соврмируем Data Frame
optuna_study_hgbc_pd = optuna_study_hgbc.trials_dataframe()
# переименуем столбы
optuna_study_hgbc_pd.rename(columns={
    'values_0': 'roc_train',
    'values_1': 'roc_valid',
    'values_2': 'f1_score',
    'values_3': 'recall_1',
    'values_4': 'precision_1'
},inplace=True)
optuna_study_hgbc_pd.tail(5)

In [ ]:
# покаждем статистику обучения
print('Максимальное значение метрики ROC AUC на валидационном наборе:',round(optuna_study_hgbc_pd['roc_valid'].max(),3))
print('Среднее значение метрики ROC AUC на валидационном наборе:',round(optuna_study_hgbc_pd['roc_valid'].mean(),3))
print('Максимальное значение метрики f1_score на валидационном наборе:',round(optuna_study_hgbc_pd['f1_score'].max(),3))
print('Среднее значение метрики f1_score на валидационном наборе:',round(optuna_study_hgbc_pd['f1_score'].mean(),3))
print('Максимальное значение метрики recall_1 на валидационном наборе:',round(optuna_study_hgbc_pd['recall_1'].max(),3))
print('Среднее значение метрики recall_1 на валидационном наборе:',round(optuna_study_hgbc_pd['recall_1'].mean(),3))
print('Максимальное значение метрики precision_1 на валидационном наборе:',round(optuna_study_hgbc_pd['precision_1'].max(),3))
print('Среднее значение метрики precision_1 на валидационном наборе:',round(optuna_study_hgbc_pd['precision_1'].mean(),3))

In [ ]:
# построим график важности гиперпарметров
optuna.visualization.plot_param_importances(optuna_study_hgbc, target_name='Score')

In [ ]:
# Построим зависимость метрики precision_1
fig = px.scatter(
    data_frame=optuna_study_hgbc_pd,
    x='precision_1', #ось абсцисс
    y=['f1_score','recall_1'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision от recall на классе 1', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision класса 1',
    yaxis_title='Величина метрик recall и f1-score класса 1')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики precision_1 от гиперпараметров
fig = px.scatter(
    data_frame=optuna_study_hgbc_pd,
    x='precision_1', #ось абсцисс
    y=['params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight', 'params_l2_regularization',
       'params_learning_rate', 'params_max_depth', 'params_max_features',
       'params_max_iter', 'params_max_leaf_nodes', 'params_min_samples_leaf',
       'params_threshold'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision класса 1 от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision класса 1',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики f1_score от гиперпараметров
fig = px.scatter(
    data_frame=optuna_study_hgbc_pd,
    x='f1_score', #ось абсцисс
    y=['params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight', 'params_l2_regularization',
       'params_learning_rate', 'params_max_depth', 'params_max_features',
       'params_max_iter', 'params_max_leaf_nodes', 'params_min_samples_leaf',
       'params_threshold'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость f1-score класса 1 от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики f1-score класса 1',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрики roc_valid от гиперпараметров
fig = px.scatter(
    data_frame=optuna_study_hgbc_pd,
    x='roc_valid', #ось абсцисс
    y=['params_class_0_weight', 'params_class_1_percent',
       'params_class_1_weight', 'params_l2_regularization',
       'params_learning_rate', 'params_max_depth', 'params_max_features',
       'params_max_iter', 'params_max_leaf_nodes', 'params_min_samples_leaf',
       'params_threshold'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость ROC AUC на валидационном наборе от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики ROC AUC на валидационном наборе',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрик качества модели от number

fig = px.scatter(
    data_frame=optuna_study_hgbc_pd[(optuna_study_hgbc_pd['recall_1']>0.1)&(optuna_study_hgbc_pd['precision_1']>0.1)],
    x='number', #ось абсцисс
    y=['roc_train','roc_valid','recall_1','precision_1'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость качества модели от trials optuna', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='trials optuna',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

<span style="color:Blue">

Вывод:  

Их всех выбираем точку $number =127$.   
В этой точке оптимальные значения метрик $recall_1$ и $precision_1$, а  
также относительно высокая метрика $ROC AUC$.   
Посмотрим каким значениям гиперпараметров соотвествует выбранная точка.

In [ ]:
# определим номер лучшге варианта
best_optuna_number = 127

# сформируем словарь лучших гипирпарметров HistGradientBoostingClassifier
best_param_hgbc = {
    'learning_rate' : optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['params_learning_rate'].iloc[0],
    'max_iter' : optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['params_max_iter'].iloc[0],
    'max_leaf_nodes' : optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['params_max_leaf_nodes'].iloc[0],
    'max_depth' : optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['params_max_depth'].iloc[0],
    'min_samples_leaf' : optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['params_min_samples_leaf'].iloc[0],
    'max_features' : optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['params_max_features'].iloc[0],
    'l2_regularization' : optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['params_l2_regularization'].iloc[0],
    'class_weight' : {0: optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['params_class_0_weight'].iloc[0],
                      1: optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['params_class_1_weight'].iloc[0],}
    }

# определим перменные для лучших значений параметров
best_threshold = optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['params_threshold'].iloc[0]
best_class_1_percent = optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['params_class_1_percent'].iloc[0]
best_random_state = optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['params_random_state'].iloc[0]


# Выведем принятые наилучшие параметры
print('best learning rate:',round(best_param_hgbc['learning_rate'],3))
print('best max iter:',best_param_hgbc['max_iter'])
print('best max leaf nodes:',best_param_hgbc['max_leaf_nodes'])
print('best max depth:',best_param_hgbc['max_depth'])
print('best min samples leaf:',best_param_hgbc['min_samples_leaf'])
print('best max features:',round(best_param_hgbc['max_features'],3))
print('best l2 regularization:',round(best_param_hgbc['l2_regularization'],3))
print('best class 0 weight:',round(best_param_hgbc['class_weight'][0],3))
print('best class 1 weight:',round(best_param_hgbc['class_weight'][1],3))

print('best class 1 percent:',round(best_class_1_percent,3))
print('best threshold:',round(best_threshold,3))
print('time for best train:',round(optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['duration'].iloc[0].seconds/60),'minutes')

print()
print('ROC AUC на обучающем наборе:', round(optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['roc_train'].iloc[0],3))
print('ROC AUC на валидационном наборе:', round(optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['roc_valid'].iloc[0],3))
print('precision класса 1:', round(optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['precision_1'].iloc[0],3))
print('recall класса 1:', round(optuna_study_hgbc_pd[optuna_study_hgbc_pd['number']==best_optuna_number]['recall_1'].iloc[0],3))

##### <span style="color:MediumSlateBlue">Обучение модели с лучшими параметрами</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'service'
count_features = dict_spaces[feature_space]

# для работы функции corr_transform_to_force
# подгрузим DataFrame с матрицей корреляции признаков
corr_matrix = fp.ParquetFile('base_models/data/stat/corr_matrix_'+feature_space).to_pandas()

# с помощью функции corr_transform_to_force получим
# список не коррелируемых признаков
list_ncorr_features = corr_transform_to_force(corr_matrix,threshold=best_threshold)[1]

# сформируем данные для анализа сбалансированности обучающих данных
X_train_pd = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
y_train_pd = pd.read_csv('target/target_train.csv')

# поготовим данные y_train_pd к работе с функцией class_1_percent_samples
y_train_pd.set_index('id',drop=False,inplace=True)

# подготовим данные для обучения модели
# с помощью функции class_1_percent_samples зададим долю 
# класса 1
list_c1_percent_id = class_1_percent_samples(y_train_pd,best_class_1_percent,random_state=best_random_state)[:1000000]

# подготовим данные для обучения
X_train_balanced = X_train_pd.loc[list_c1_percent_id]
y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag']

# освободим память от "тяжелых" и ненужных файлов
del X_train_pd, y_train_pd
gc.collect()

# обучаем модель HistGradientBoostingClassifier с наилучшеми параметрами
gradient_boosting = HistGradientBoostingClassifier(
        **best_param_hgbc,
        random_state=42)
gradient_boosting.fit(X_train_balanced,y_train_balanced)

# удаляем крупные файлы чтобы высвободить память 
del X_train_balanced, y_train_balanced
gc.collect()

# сформируем данные для анализа сбалансированности обучающих данных
X_train = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
X_valid = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_valid').to_pandas()[list_ncorr_features]
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()

# делаем предсказание на обучающем и валидационном наборе и считаем метрики
y_train_gb_pred_proba = gradient_boosting.predict_proba(X_train)[:,1]
y_valid_gb_pred_proba = gradient_boosting.predict_proba(X_valid)[:,1]
y_valid_gb_pred = gradient_boosting.predict(X_valid)

# удаляем крупные файлы чтобы высвободить память 
del X_train, X_valid
gc.collect()

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_gb_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_gb_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_valid,y_valid_gb_pred,zero_division=0))

# time: 1m 40s

##### <span style="color:MediumSlateBlue">Анализ влияния порога классификации на качество модели</span>

In [ ]:
# чтобы проше обращаться к каждому обьекту в данных
# преобразуем y_valid_LR_pred к Series типу
y_valid_gb_pred_proba = pd.Series(y_valid_gb_pred_proba)

# формируем список из необходимых значений thresholds
thresholds = np.arange(0.01, 1, 0.01)

# сформируем списко для заполнения данными результатов анализа
threshold_data = []


for threshold in thresholds:
        # сформируем список для заполнения данными текушего threshold
        threshold_current_data = [threshold]

        #клиентов, для которых вероятность дефолта > threshold, относим к классу 1
        #в противном случае — к классу 0
        y_valid_gb_pred = y_valid_gb_pred_proba.apply(lambda x: 1 if x>threshold else 0)


        #Считаем метрики для класса 0 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(y_valid, y_valid_gb_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.precision_score(y_valid, y_valid_gb_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.f1_score(y_valid, y_valid_gb_pred,average=None,zero_division=0)[0],3))

        #Считаем метрики для класса 1 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(y_valid, y_valid_gb_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.precision_score(y_valid, y_valid_gb_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.f1_score(y_valid, y_valid_gb_pred,average=None,zero_division=0)[1],3))

        # добавляем полученные данные в список threshold_data
        threshold_data.append(threshold_current_data)

        # из полученных данных сформируем dataframe
        thresholds_pd =pd.DataFrame(
        columns=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'],
        data = threshold_data)

# time: 17s

In [ ]:
# Визуализируем метрики на валидационном наборе при различных threshold
fig_threshold = px.line(
    data_frame=thresholds_pd,
    x='threshold', #ось абсцисс
    y=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'], #ось ординат
)
fig_threshold.update_layout(
    title ={
        'text':'Зависимость качества модели от threshold', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Порог классификации threshold',
    yaxis_title='Величина метрики')
fig_threshold.update_xaxes(showspikes=True)
fig_threshold.update_yaxes(showspikes=True)

fig_threshold.show()

Метрика $precision$ показывает на сколько сильно ошиблась модель при определиние класса.  
Чем выше метрика, тем меньше ошиблась модель. 

$$precision = \frac{TP}{TP+FP}$$

Метрика $recall$ показывает способность модели найти класс.  
Чем выше метрика, тем выше способность модели находить класс.  

$$recall = \frac{TP}{TP+FN}$$

Метрика $f1-score$ показывает баланс между $precision$ и $recall$ 

$threshold$- порог класификации модели.  
Все объекты для которых $у_{\text{proba pred}} < threshold$, модель относит к класс 0.  
Соотвественно, если $у_{\text{proba pred}}  > threshold$ модель относит обьект к классу 1.  
Для идельаной модели $threshold$, является границей между областями класса 0 и класса 1.  

<span style="color:Blue">

Выводы по результам анализа:  

Для улучшения качества модели по метрике $ROC AUC$, нет смысла сдвигать порог   
классификации. Но зависимость метрик качества модели от $threshold$, может рассказать о  
способности модели искать класс 1 и класс 0.

При фокусировке модели на классе 1 (более низкий $threshold$):  
- на классе 1 метрика $precision$ уменьшается, метрика $recall$ растет;  
- на классе 0 метрика $precision$ растет, метрика $recall$ уменьшается.  

При фокусировке модели на классе 0 (более высокий $threshold$):  
- на классе 1 метрика $precision$ увеличивается, метрика $recall$ уменьшается;  
- на классе 0 метрика $precision$ уменьшается, метрика $recall$ растет.  

Обьсняется тем, что в условия дисбаланса модель научилась хорошо определять класс 0 и плохо класс 1.  

При фокусировке модели на классе 1 (более низкий $threshold$), все больше обьектов отнесены к классу 1.   
Так как $precision$ класса 1 уменьшается, значит в области класса 1 стало больше обектов класса 0,  
что явлется закономерным для "идеальной" модели.  
Однако при этом, $recall$ класса 1 растет, значит в добавленных обьектах также присутвует класс 1.  
Это как раз указывает на то, что модель не нучилась находить среди класса 0 класс 1. В области с     
высокой вероятностью класса 0 (более низкий $threshold$), находятся обьекты класса 1.  

Фокусировка на классе 0 (более высокий $threshold$), заставляет модель все больше обьектов из области класса 1  
относить к классу 0. При этом увеличение метрики $precision$ класса 1 говорит о том, что в области класса 1     
становится меньше 0, а значит модель умеет среди класса 1, находить класс 0.  
После  $threshold=0.88$ модель впринципе теряет способность отличать класс 1. У модели нет обьектов класса 1   
в которых она "уверена" на 90+%.


Обратим внимание на $threshold=0.51$, в этой точке находится баланс межу метриками качества модели:
$$precision_1 = recall_1 = \text{f1-score}_1 = 0.103$$
$$precision_0 = recall_0 = \text{f1-score}_0 = 0.97$$

Для модели с лучшим качеством эти две точки должны находится максимально близко к друг другу и иметь   
высокое значение метрик $precision_1$ и $recall_1$.  
Для класса 0 это условие выполнется, что также говорит о том, что модель научилась искать этот класс.    
Для класса 1 это условие не выполнется, что также говорит о том, что модель плохо ищет класс 1.

##### <span style="color:MediumSlateBlue">Построение ROC-кривой выбранной модели на тестовом наборе</span>

In [ ]:
# определим прострастравсо признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}
feature_space = 'service'
count_features = dict_spaces[feature_space]

In [ ]:
# сформируем данные для проверки модели
X_train = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
y_train = pd.read_csv('target/target_train.csv')['flag'].to_numpy()
X_valid = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_valid').to_pandas()[list_ncorr_features]
y_valid = pd.read_csv('target/target_valid.csv')['flag'].to_numpy()
X_test = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_test').to_pandas()[list_ncorr_features]
y_test = pd.read_csv('target/target_test.csv')['flag'].to_numpy()

In [ ]:
print("Размеры обучающего набора",X_train.shape,y_train.shape)
print("Размеры валидационного набора",X_valid.shape,y_valid.shape)
print("Размеры тестового набора",X_test.shape,y_test.shape)
# time: 1m 43s

In [ ]:
# для метрик ROC AUC делаем предсказание модели в виде вероятности
y_train_gb_pred_proba = gradient_boosting.predict_proba(X_train)[:,1]
y_valid_gb_pred_proba = gradient_boosting.predict_proba(X_valid)[:,1]
y_test_gb_pred_proba = gradient_boosting.predict_proba(X_test)[:,1]

# Делаем предсказание для валидационной выборки
y_test_gb_pred = gradient_boosting.predict(X_test)

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(y_train, y_train_gb_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(y_valid, y_valid_gb_pred_proba),3))
print('ROC AUC на тестовом наборе', round(metrics.roc_auc_score(y_test, y_test_gb_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(y_test,y_test_gb_pred,zero_division=0))

In [ ]:
# Для расчета значений TPR и FPR используем функцию roc_curve
fpr_train, tpr_train, _ = metrics.roc_curve(y_train, y_train_gb_pred_proba)
fpr_valid, tpr_valid, _ = metrics.roc_curve(y_valid, y_valid_gb_pred_proba)
fpr_test, tpr_test, _ = metrics.roc_curve(y_test, y_test_gb_pred_proba)

# рассчитаем площадь под кривой
auc_train = round(metrics.roc_auc_score(y_train, y_train_gb_pred_proba),3)
auc_valid = round(metrics.roc_auc_score(y_valid, y_valid_gb_pred_proba),3)
auc_test = round(metrics.roc_auc_score(y_test, y_test_gb_pred_proba),3)

trace_train = go.Scatter(x=fpr_train, y=tpr_train, mode='lines', name='AUC_train = %0.2f' % auc_train,
                   line=dict(color='red', width=2),
                   hovertemplate='train<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_train])

trace_valid = go.Scatter(x=fpr_valid, y=tpr_valid, mode='lines', name='AUC_valid = %0.2f' % auc_valid,
                   line=dict(color='green', width=2),
                   hovertemplate='valid<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_valid])

trace_test = go.Scatter(x=fpr_test, y=tpr_test, mode='lines', name='AUC_test = %0.2f' % auc_test,
                   line=dict(color='darkorange', width=2),
                   hovertemplate='test<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_test])


reference_line = go.Scatter(x=[0,1], y=[0,1], mode='lines', name='Reference Line',
                            line=dict(color='navy', width=2, dash='dash'))

fig_roc = go.Figure(data=[trace_train,trace_valid,trace_test,reference_line])

fig_roc.update_layout(
    title ={
        'text':'ROC-кривая', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 1350,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate')

fig_roc.update_xaxes(showspikes=True)
fig_roc.update_yaxes(showspikes=True)


fig_roc.show()

<span style="color:Blue">

Вывод по ROC-кривой:
1. Модель выдает сталибильное качество на валидационном и тестовом наборах, а значит  
не переобучена - разброс ($variance$) не большой. Хотя, качество модели на обучающем наборе,  
заметно лучше.
2. Смещение ($bias$) модели чуть лучше, чем у модели $Random$ $Forest$ $Classifier$,  
обученной на тех же данных.    
На тестовом наборе полученно качество $ROC AUC = 0.63$, против $ROC AUC = 0.61$.

## <span style="color:DodgerBlue">Второй этап построения блендинга моделей</span>

### <span style="color:RoyalBlue">Формирование данных для обучения first meta models</span>

#### <span style="color:MediumBlue">Формирование метапризнаков из моделей обученных на transform data torow данных</span>

Попробуем улучшить качество модели за счет стекинга.  
Сформируем стегинг из рассмотренных базовых модей:
- $Logistic$ $Regression$;
- $Random$ $Forest$ $Classifier$;
- $Hist$ $Gradient$ $Boosting$ $Classifier$,  
обученных на данных $transform$ $data$ $torow$ и $transform$ $data$ $stat$.

Для блендинга выберем следующие модели:
- модели с высоким значением метрики $ROC AUC$ на валидационном наборе;
- модели с высоким значением метрики $precision_1$ на валидационном наборе;
- модели с высоким значением метрики $f1_{score}$ на валидационном наборе 
($precision_1 \approx recall_1$);

In [ ]:
# сформируем заготовку под метаданные для обучения и проверики метамодели 
features_first_meta_torow = pd.read_csv('target/target_test.csv')

In [ ]:
# для выбора sceler преобразвания понадобится словарь
dict_scalers ={
    'MinMaxScaler':MinMaxScaler(),
    'RobustScaler':RobustScaler(),
    'StandardScaler':StandardScaler(),
}
# определим пространство признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}

# задаим количество отбираемых "лучших" моделей
best_models_number = 10

##### <span style="color:MediumSlateBlue">Формирование метапризнаков от модели *Logistic Regression* обученной сбалансированными *transform data torow* данными</span>

In [ ]:
# сформируем метапризнаки для каждого подпространства признаков
for feature_space in dict_spaces.keys():
    # обьявим количество признаков подпространства
    count_features = dict_spaces[feature_space]

    # загрузим обьект исcледования optuna
    optuna_study_lr_torow = optuna.load_study(study_name='LogisticRegression_'+feature_space+'_torow',
                               storage='sqlite:///optuna_studies.db')
    
    # из полученного обьекта сформируем Data Frame
    optuna_study_lr_torow_pd = optuna_study_lr_torow.trials_dataframe().dropna()

    # переименуем столбы
    optuna_study_lr_torow_pd.rename(columns={
        'values_0': 'roc_train', 
        'values_1': 'roc_valid', 
        'values_2': 'f1_score', 
        'values_3': 'recall_1',
        'values_4': 'precision_1'
    },inplace=True)

    # удалим ненужные столбцы
    optuna_study_lr_torow_pd.drop(['datetime_start','datetime_complete','state','duration'],axis=1, inplace=True)

    # выберем точки с лучшими значениями метрики roc auc на валидационном наборе
    best_roc_lr_torow = optuna_study_lr_torow_pd.sort_values(by='roc_valid',ascending=False)[:best_models_number] 

    # выберем точки с лучшими значениями метрики precion класса 1 на валидационном наборе
    best_precion_lr_torow = optuna_study_lr_torow_pd.sort_values(by='precision_1',ascending=False)[:best_models_number]   

    # выберем точки в которых максимальна метрика f1-score класса 1 (precision_1=recall_1)
    best_f1_lr_torow = optuna_study_lr_torow_pd.sort_values(by='f1_score',ascending=False)[:best_models_number]

    # сформируем набор признаков для лучших моделей
    best_models_lr_torow = pd.concat([best_roc_lr_torow,best_precion_lr_torow,best_f1_lr_torow]).drop_duplicates().reset_index(drop=True)

    # сформируем список индексов best_models_lr_torow
    list_index = best_models_lr_torow.index

    # сформируем  метапризнаки для обучения метамодели
    # для этого необходиом обучить модель LogisticRegression с выбранными гиперпараметрами
    for index in list_index:

        # для того чтобы следить за выполнением цикла введем индикатор
        print('Current space: ', feature_space)
        print('Number of indexes: ', max(list_index))
        print('Current index: ', index)
        
        # сформируем необходимые параметры
        C = best_models_lr_torow.loc[index,'params_C']
        solver = best_models_lr_torow.loc[index,'params_solver']
        class_weight = {0:best_models_lr_torow.loc[index,'params_class_0_weight'],
                        1:best_models_lr_torow.loc[index,'params_class_1_weight']}
        n_last = int(best_models_lr_torow.loc[index,'params_n_last'])
        scaler = best_models_lr_torow.loc[index,'params_scaler']
        class_1_percent = best_models_lr_torow.loc[index,'params_class_1_percent']
        random_state = int(best_models_lr_torow.loc[index,'params_random_state'])

        # с помощью функции features_from_transform_data_torow извлечем  
        # из данных transform_data_torow
        # признаки сооствествующие n_last последним клиенским операциям
        list_n_last_features = features_from_transform_data_torow(n_last,count_features,25)

        # сохраним list_n_last_features для дальнейшего воспроизведения
        dump(list_n_last_features, 'base_models/list_n_last_features/'+'list_n_last_features_LRTR_'+feature_space+'_'+str('0'+str(index))[-2:]+'.joblib')

        # обьявим scaler
        scaler = dict_scalers[scaler]

        # загружаем обучующие наборы
        print('Loading train data')
        X_train_pd = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas()
        y_train_pd = pd.read_csv('target/target_train.csv')

        # поготовим данные y_train_pd к работе с функцией class_1_percent_samples
        y_train_pd.set_index('id',drop=False,inplace=True)

        # подготовим данные для обучения модели
        # с помощью функции class_1_percent_samples зададим долю 
        # класса 1
        list_c1_percent_id = class_1_percent_samples(y_train_pd,class_1_percent,random_state=random_state)[:1000000]

        # сформируем данные для обучения базовой модели
        X_train_s_balanced = scaler.fit_transform(np.array(X_train_pd.loc[list_c1_percent_id])[:,list_n_last_features])
        y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

        # сохраним scaler для дальнейшего воспроизведения
        dump(scaler, 'base_models/scalers/'+'scaler_torow_'+feature_space+'_'+str('0'+str(index))[-2:]+'.joblib')

        # освободим память от "тяжелых" и ненужных файлов
        del X_train_pd, y_train_pd
        gc.collect()

        # для того чтобы следить за выполнением цикла введем индикатор
        print('start train')
        # обучаем модель LogisticRegression с наилучшеми параметрами
        logistic_regression = linear_model.LogisticRegression(
                C=C,
                solver = solver,
                class_weight=class_weight,
                random_state=random_state,
                max_iter=10000)
        logistic_regression.fit(X_train_s_balanced,y_train_balanced)
        # для того чтобы следить за выполнением цикла введем индикатор
        print('finished train')

        # сохраним модель для дальнейшего воспроизведения
        dump(logistic_regression, 'base_models/models/'+'LRTR_'+feature_space+'_'+str('0'+str(index))[-2:]+'.joblib')

        # удаляем крупные файлы чтобы высвободить память 
        del X_train_s_balanced,y_train_balanced
        gc.collect()

        # для того чтобы следить за выполнением цикла введем индикатор
        print('Loading test data')
        # подгружаем данные для тестирования
        X_test = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_test').to_pandas().to_numpy()[:,list_n_last_features]
        # сформируем данные для проверки модели
        X_test_s = scaler.transform(X_test)
        # для метрик ROC AUC делаем предсказание модели в виде вероятности и записываем в методанные
        features_first_meta_torow['LRTR_'+feature_space+'_'+str('0'+str(index))[-2:]] = logistic_regression.predict_proba(X_test_s)[:,1]
        # удаляем крупные файлы чтобы высвободить память 
        del X_test_s, X_test
        gc.collect()
        clear_output()

# сохраним полученные метапризнаки
fp.write('first_metamodel/features/features_first_meta_torow',features_first_meta_torow)

##### <span style="color:MediumSlateBlue">Формирование метапризнаков от модели *Random Forest Classifier* обученной сбалансированными *transform data torow* данными</span>

In [ ]:
# сформируем метапризнаки для каждого подпространства признаков
for feature_space in dict_spaces.keys():
    # обьявим количество признаков подпространства
    count_features = dict_spaces[feature_space]

    # загрузим обьект исcледования optuna
    optuna_study_rf_torow = optuna.load_study(study_name='RandomForestClassifier_'+feature_space+'_torow',
                               storage='sqlite:///optuna_studies.db')
    
    # из полученного обьекта сформируем Data Frame
    optuna_study_rf_torow_pd = optuna_study_rf_torow.trials_dataframe().dropna()

    # переименуем столбы
    optuna_study_rf_torow_pd.rename(columns={
        'values_0': 'roc_train', 
        'values_1': 'roc_valid', 
        'values_2': 'f1_score', 
        'values_3': 'recall_1',
        'values_4': 'precision_1'
    },inplace=True)

    # удалим ненужные столбцы
    optuna_study_rf_torow_pd.drop(['datetime_start','datetime_complete','state','duration'],axis=1, inplace=True)

    # выберем точки с лучшими значениями метрики roc auc на валидационном наборе
    best_roc_rf_torow = optuna_study_rf_torow_pd.sort_values(by='roc_valid',ascending=False)[:best_models_number] 

    # выберем точки с лучшими значениями метрики precion класса 1 на валидационном наборе
    best_precion_rf_torow = optuna_study_rf_torow_pd.sort_values(by='precision_1',ascending=False)[:best_models_number]   

    # выберем точки в которых максимальна метрика f1-score класса 1 (precision_1=recall_1)
    best_f1_rf_torow = optuna_study_rf_torow_pd.sort_values(by='f1_score',ascending=False)[:best_models_number]

    # сформируем набор признаков для лучших моделей
    best_models_rf_torow = pd.concat([best_roc_rf_torow,best_precion_rf_torow,best_f1_rf_torow]).drop_duplicates().reset_index(drop=True)

    # сформируем список индексов best_models_rf_torow
    list_index = best_models_rf_torow.index

    # сформируем  метапризнаки для обучения метамодели
    # для этого необходиом обучить модель RandomForestClassifier с выбранными гиперпараметрами
    for index in list_index:

        # для того чтобы следить за выполнением цикла введем индикатор
        print('Current space: ', feature_space)
        print('Number of indexes: ', max(list_index))
        print('Current index: ', index)
        
        # сформируем необходимые параметры
        n_estimators = best_models_rf_torow.loc[index,'params_n_estimators']
        criterion = best_models_rf_torow.loc[index,'params_criterion']
        max_depth = best_models_rf_torow.loc[index,'params_max_depth']
        min_samples_split = best_models_rf_torow.loc[index,'params_min_samples_split']
        min_samples_leaf = best_models_rf_torow.loc[index,'params_min_samples_leaf']
        max_features = best_models_rf_torow.loc[index,'params_max_features']
        class_weight = {0:best_models_rf_torow.loc[index,'params_class_0_weight'],
                        1:best_models_rf_torow.loc[index,'params_class_1_weight']}
        max_samples = best_models_rf_torow.loc[index,'params_max_samples']
        n_last = best_models_rf_torow.loc[index,'params_n_last']
        class_1_percent = best_models_rf_torow.loc[index,'params_class_1_percent']
        random_state = int(best_models_rf_torow.loc[index,'params_random_state'])

        # с помощью функции features_from_transform_data_torow извлечем  
        # из данных transform_data_torow
        # признаки сооствествующие n_last последним клиенским операциям
        list_n_last_features = features_from_transform_data_torow(n_last,count_features,25)

        # сохраним list_n_last_features для дальнейшего воспроизведения
        dump(list_n_last_features, 'base_models/list_n_last_features/'+'list_n_last_features_RFTR_'+feature_space+'_'+str('0'+str(index))[-2:]+'.joblib')

        # загружаем обучующие наборы
        print('Loading train data')
        X_train_pd = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas()
        y_train_pd = pd.read_csv('target/target_train.csv')

        # поготовим данные y_train_pd к работе с функцией class_1_percent_samples
        y_train_pd.set_index('id',drop=False,inplace=True)

        # подготовим данные для обучения модели
        # с помощью функции class_1_percent_samples зададим долю 
        # класса 1
        list_c1_percent_id = class_1_percent_samples(y_train_pd,class_1_percent,random_state=random_state)[:1000000]

        # сформируем данные для обучения базовой модели
        X_train_balanced = np.array(X_train_pd.loc[list_c1_percent_id])[:,list_n_last_features]
        y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

        # освободим память от "тяжелых" и ненужных файлов
        del X_train_pd, y_train_pd
        gc.collect()

        # для того чтобы следить за выполнением цикла введем индикатор
        print('start train')
        # обучаем модель RandomForestClassifier с наилучшеми параметрами
        random_forest = RandomForestClassifier(
                n_estimators = n_estimators,
                criterion = criterion,
                max_depth = max_depth,
                min_samples_split = min_samples_split,
                min_samples_leaf = min_samples_leaf,
                max_features = max_features,
                class_weight = class_weight,
                max_samples = max_samples,
                n_jobs=-1,
                random_state=random_state)
        random_forest.fit(X_train_balanced, y_train_balanced)
        # для того чтобы следить за выполнением цикла введем индикатор
        print('finished train')

        # сохраним модель для дальнейшего воспроизведения
        dump(random_forest, 'base_models/models/'+'RFTR_'+feature_space+'_'+str('0'+str(index))[-2:]+'.joblib')

        # удаляем крупные файлы чтобы высвободить память 
        del X_train_balanced,y_train_balanced
        gc.collect()

        # для того чтобы следить за выполнением цикла введем индикатор
        print('Loading test data')
        # подгружаем данные для тестирования
        X_test = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_test').to_pandas().to_numpy()[:,list_n_last_features]
        # для метрик ROC AUC делаем предсказание модели в виде вероятности и записываем в методанные
        features_first_meta_torow['RFTR_'+feature_space+'_'+str('0'+str(index))[-2:]] = random_forest.predict_proba(X_test)[:,1]
        # удаляем крупные файлы чтобы высвободить память 
        del X_test
        gc.collect()
        clear_output()

# сохраним полученные метапризнаки
fp.write('first_metamodel/features/features_first_meta_torow',features_first_meta_torow)

##### <span style="color:MediumSlateBlue">Формирование метапризнаков от модели *Gradient Boosting Classifier* обученной сбалансированными *transform data torow* данными</span>

In [ ]:
# сформируем метапризнаки для каждого подпространства признаков
for feature_space in dict_spaces.keys():
    # обьявим количество признаков подпространства
    count_features = dict_spaces[feature_space]

    # загрузим обьект исcледования optuna
    optuna_study_gb_torow = optuna.load_study(study_name='HistGradientBoostingClassifier_'+feature_space+'_torow',
                               storage='sqlite:///optuna_studies.db')
    
    # из полученного обьекта сформируем Data Frame
    optuna_study_gb_torow_pd = optuna_study_gb_torow.trials_dataframe().dropna()

    # переименуем столбы
    optuna_study_gb_torow_pd.rename(columns={
        'values_0': 'roc_train', 
        'values_1': 'roc_valid', 
        'values_2': 'f1_score', 
        'values_3': 'recall_1',
        'values_4': 'precision_1'
    },inplace=True)

    # удалим ненужные столбцы
    optuna_study_gb_torow_pd.drop(['datetime_start','datetime_complete','state','duration'],axis=1, inplace=True)

    # выберем точки с лучшими значениями метрики roc auc на валидационном наборе
    best_roc_gb_torow = optuna_study_gb_torow_pd.sort_values(by='roc_valid',ascending=False)[:best_models_number] 

    # выберем точки с лучшими значениями метрики precion класса 1 на валидационном наборе
    best_precion_gb_torow = optuna_study_gb_torow_pd.sort_values(by='precision_1',ascending=False)[:best_models_number]   

    # выберем точки в которых максимальна метрика f1-score класса 1 (precision_1=recall_1)
    best_f1_gb_torow = optuna_study_gb_torow_pd.sort_values(by='f1_score',ascending=False)[:best_models_number]

    # сформируем набор признаков для лучших моделей
    best_models_gb_torow = pd.concat([best_roc_gb_torow,best_precion_gb_torow,best_f1_gb_torow]).drop_duplicates().reset_index(drop=True)

    # сформируем список индексов best_models_gb_torow
    list_index = best_models_gb_torow.index

    # сформируем  метапризнаки для обучения метамодели
    # для этого необходиом обучить модель HistGradientBoostingClassifier с выбранными гиперпараметрами
    for index in list_index:

        # для того чтобы следить за выполнением цикла введем индикатор
        print('Current space: ', feature_space)
        print('Number of indexes: ', max(list_index))
        print('Current index: ', index)
        
        # сформируем необходимые параметры
        learning_rate = best_models_gb_torow.loc[index,'params_learning_rate']
        max_iter = best_models_gb_torow.loc[index,'params_max_iter']
        max_leaf_nodes = best_models_gb_torow.loc[index,'params_max_leaf_nodes']
        max_depth = best_models_gb_torow.loc[index,'params_max_depth']
        min_samples_leaf = best_models_gb_torow.loc[index,'params_min_samples_leaf']
        l2_regularization = best_models_gb_torow.loc[index,'params_l2_regularization']
        max_features = best_models_gb_torow.loc[index,'params_max_features']
        class_weight = {0:best_models_gb_torow.loc[index,'params_class_0_weight'],
                        1:best_models_gb_torow.loc[index,'params_class_1_weight']}
        n_last = best_models_gb_torow.loc[index,'params_n_last']
        class_1_percent = best_models_gb_torow.loc[index,'params_class_1_percent']
        random_state = best_models_gb_torow.loc[index,'params_random_state']

        # с помощью функции features_from_transform_data_torow извлечем  
        # из данных transform_data_torow
        # признаки сооствествующие n_last последним клиенским операциям
        list_n_last_features = features_from_transform_data_torow(n_last,count_features,25)

            # сохраним list_n_last_features для дальнейшего воспроизведения
        dump(list_n_last_features, 'base_models/list_n_last_features/'+'list_n_last_features_GBTR_'+feature_space+'_'+str('0'+str(index))[-2:]+'.joblib')

        # загружаем обучующие наборы
        print('Loading train data')
        X_train_pd = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_train').to_pandas()
        y_train_pd = pd.read_csv('target/target_train.csv')

        # поготовим данные y_train_pd к работе с функцией class_1_percent_samples
        y_train_pd.set_index('id',drop=False,inplace=True)

        # подготовим данные для обучения модели
        # с помощью функции class_1_percent_samples зададим долю класса 1
        list_c1_percent_id = class_1_percent_samples(y_train_pd,class_1_percent,random_state=random_state)[:1000000]

        # сформируем данные для обучения модели
        X_train_balanced = np.array(X_train_pd.loc[list_c1_percent_id])[:,list_n_last_features]
        y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

        # освободим память от "тяжелых" и ненужных файлов
        del X_train_pd, y_train_pd
        gc.collect()

        # для того чтобы следить за выполнением цикла введем индикатор
        print('start train')
        # обучаем модель HistGradientBoostingClassifier с наилучшеми параметрами
        gradient_boosting = HistGradientBoostingClassifier(
                learning_rate = learning_rate,
                max_iter = max_iter,
                max_leaf_nodes = max_leaf_nodes,
                max_depth = max_depth,
                min_samples_leaf = min_samples_leaf,
                l2_regularization = l2_regularization,
                max_features = max_features,
                class_weight = class_weight,
                random_state=42)
        gradient_boosting.fit(X_train_balanced,y_train_balanced)
        # для того чтобы следить за выполнением цикла введем индикатор
        print('finished train')

        # сохраним модель для дальнейшего воспроизведения
        dump(gradient_boosting, 'base_models/models/'+'GBTR_'+feature_space+'_'+str('0'+str(index))[-2:]+'.joblib')

        # удаляем крупные файлы чтобы высвободить память 
        del X_train_balanced,y_train_balanced
        gc.collect()

        # для того чтобы следить за выполнением цикла введем индикатор
        print('Loading test data')
        # подгружаем данные для тестирования
        X_test = fp.ParquetFile('base_models/data/torow/'+feature_space+'_torow_test').to_pandas().to_numpy()[:,list_n_last_features]
        # для метрик ROC AUC делаем предсказание модели в виде вероятности и записываем в методанные
        features_first_meta_torow['GBTR_'+feature_space+'_'+str('0'+str(index))[-2:]] = gradient_boosting.predict_proba(X_test)[:,1]
        # удаляем крупные файлы чтобы высвободить память 
        del X_test
        gc.collect()
        clear_output()

# сохраним полученные метапризнаки
fp.write('first_metamodel/features/features_first_meta_torow',features_first_meta_torow)

#### <span style="color:MediumBlue">Формирование метапризнаков из моделей обученных на transform data stat данных</span>

Попробуем улучшить качество модели за счет стекинга.  
Сформируем стегинг из рассмотренных базовых модей:
- $Logistic$ $Regression$;
- $Random$ $Forest$ $Classifier$;
- $Hist$ $Gradient$ $Boosting$ $Classifier$,  
обученных на данных $transform$ $data$ $torow$ и $transform$ $data$ $stat$.

Для стекинга выберем следующие модели:
- модели с высоким значением метрики $ROC AUC$ на валидационном наборе;
- модели с высоким значением метрики $precision_1$ на валидационном наборе;
- модели с высоким значением метрики $f1_{score}$ на валидационном наборе 
($precision_1 \approx recall_1$);

In [ ]:
# сформируем заготовки под метаданные для обучения и проверики метамодели 
features_first_meta_stat = pd.read_csv('target/target_test.csv')

In [ ]:
# для выбора sceler преобразвания понадобится словарь
dict_scalers ={
    'MinMaxScaler':MinMaxScaler(),
    'RobustScaler':RobustScaler(),
    'StandardScaler':StandardScaler(),
}
# определим пространство признаков
dict_spaces = {
    'date' : 8,
    'late': 12,
    'credit': 4,
    'relative' : 6,
    'payments': 25,
    'service': 4}

# задаим количество отбираемых "лучших" моделей
best_models_number = 10

##### <span style="color:MediumSlateBlue">Формирование метапризнаков от модели *Logistic Regression* обученной сбалансированными *transform data stat* данными</span>

In [ ]:
# сформируем метапризнаки для каждого подпространства признаков
for feature_space in dict_spaces.keys():
    # обьявим количество признаков подпространства
    count_features = dict_spaces[feature_space]

    # для работы функции corr_transform_to_force
    # подгрузим DataFrame с матрицей корреляции признаков
    corr_matrix = fp.ParquetFile('base_models/data/stat/corr_matrix_'+feature_space).to_pandas()

    # загрузим обьект исcледования optuna
    optuna_study_lr_stat = optuna.load_study(study_name='LogisticRegression_'+feature_space+'_stat',
                               storage='sqlite:///optuna_studies.db')
    
    # из полученного обьекта сформируем Data Frame
    optuna_study_lr_stat_pd = optuna_study_lr_stat.trials_dataframe().dropna()

    # переименуем столбы
    optuna_study_lr_stat_pd.rename(columns={
        'values_0': 'roc_train', 
        'values_1': 'roc_valid', 
        'values_2': 'f1_score', 
        'values_3': 'recall_1',
        'values_4': 'precision_1'
    },inplace=True)

    # удалим ненужные столбцы
    optuna_study_lr_stat_pd.drop(['datetime_start','datetime_complete','state','duration'],axis=1, inplace=True)

    # выберем точки с лучшими значениями метрики roc auc на валидационном наборе
    best_roc_lr_stat = optuna_study_lr_stat_pd.sort_values(by='roc_valid',ascending=False)[:best_models_number] 

    # выберем точки с лучшими значениями метрики precion класса 1 на валидационном наборе
    best_precion_lr_stat = optuna_study_lr_stat_pd.sort_values(by='precision_1',ascending=False)[:best_models_number]   

    # выберем точки в которых максимальна метрика f1-score класса 1 (precision_1=recall_1)
    best_f1_lr_stat = optuna_study_lr_stat_pd.sort_values(by='f1_score',ascending=False)[:best_models_number]

    # сформируем набор признаков для лучших моделей
    best_models_lr_stat = pd.concat([best_roc_lr_stat,best_precion_lr_stat,best_f1_lr_stat]).drop_duplicates().reset_index(drop=True)

    # сформируем список индексов best_models_lr_stat
    list_index = best_models_lr_stat.index

    # сформируем  метапризнаки для обучения метамодели
    # для этого необходиом обучить модель LogisticRegression с выбранными гиперпараметрами
    for index in list_index:

        # для того чтобы следить за выполнением цикла введем индикатор
        print('Current space: ', feature_space)
        print('Number of indexes: ', max(list_index))
        print('Current index: ', index)
        
        # сформируем необходимые параметры
        C = best_models_lr_stat.loc[index,'params_C']
        solver = best_models_lr_stat.loc[index,'params_solver']
        class_weight = {0:best_models_lr_stat.loc[index,'params_class_0_weight'],
                        1:best_models_lr_stat.loc[index,'params_class_1_weight']}
        threshold = best_models_lr_stat.loc[index,'params_threshold']
        scaler = best_models_lr_stat.loc[index,'params_scaler']
        class_1_percent = best_models_lr_stat.loc[index,'params_class_1_percent']
        random_state = int(best_models_lr_stat.loc[index,'params_random_state'])

        # с помощью функции corr_transform_to_force получим
        # список не коррелируемых признаков
        list_ncorr_features = corr_transform_to_force(corr_matrix,threshold=threshold)[1]

        # сохраним list_ncorr_features для дальнейшего воспроизведения
        dump(list_ncorr_features, 'base_models/list_ncorr_features/'+'list_ncorr_features_LRST_'+feature_space+'_'+str('0'+str(index))[-2:]+'.joblib')

        # обьявим scaler
        scaler = dict_scalers[scaler]

        # сформируем данные для анализа сбалансированности обучающих данных
        print('Loading train data')
        X_train_pd = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
        y_train_pd = pd.read_csv('target/target_train.csv')

        # поготовим данные y_train_pd к работе с функцией class_1_percent_samples
        y_train_pd.set_index('id',drop=False,inplace=True)

        # подготовим данные для обучения модели
        # с помощью функции class_1_percent_samples зададим долю 
        # класса 1
        list_c1_percent_id = class_1_percent_samples(y_train_pd,class_1_percent,random_state=random_state)[:1000000]

        # подготовим данные для обучения
        X_train_s_balanced = scaler.fit_transform(X_train_pd.loc[list_c1_percent_id])
        y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

        # сохраним scaler для дальнейшего воспроизведения
        dump(scaler, 'base_models/scalers/'+'scaler_stat_'+feature_space+'_'+str('0'+str(index))[-2:]+'.joblib')

        # освободим память от "тяжелых" и ненужных файлов
        del X_train_pd, y_train_pd
        gc.collect()

        # для того чтобы следить за выполнением цикла введем индикатор
        print('start train')
        # обучаем модель LogisticRegression с наилучшеми параметрами
        logistic_regression = linear_model.LogisticRegression(
                C=C,
                solver = solver,
                class_weight=class_weight,
                random_state=random_state,
                max_iter=10000)
        logistic_regression.fit(X_train_s_balanced,y_train_balanced)
        # для того чтобы следить за выполнением цикла введем индикатор
        print('finished train')

        # сохраним модель для дальнейшего воспроизведения
        dump(logistic_regression, 'base_models/models/'+'LRST_'+feature_space+'_'+str('0'+str(index))[-2:]+'.joblib')

        # удаляем крупные файлы чтобы высвободить память 
        del X_train_s_balanced,y_train_balanced
        gc.collect()

        # для того чтобы следить за выполнением цикла введем индикатор
        print('Loading test data')
        # подгружаем данные для тестирования
        X_test = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_test').to_pandas()[list_ncorr_features]
        # сформируем данные для проверки модели
        X_test_s = scaler.transform(X_test)
        # для метрик ROC AUC делаем предсказание модели в виде вероятности и записываем в методанные
        features_first_meta_stat['LRST_'+feature_space+'_'+str('0'+str(index))[-2:]] = logistic_regression.predict_proba(X_test_s)[:,1]
        # удаляем крупные файлы чтобы высвободить память 
        del X_test_s, X_test
        gc.collect()
        clear_output()

# сохраним полученные метапризнаки
fp.write('first_metamodel/features/features_first_meta_stat',features_first_meta_stat)

##### <span style="color:MediumSlateBlue">Формирование метапризнаков от модели *Random Forest Classifier* обученной сбалансированными *transform data stat* данными</span>

In [ ]:
# сформируем метапризнаки для каждого подпространства признаков
for feature_space in dict_spaces.keys():
    # обьявим количество признаков подпространства
    count_features = dict_spaces[feature_space]

    # для работы функции corr_transform_to_force
    # подгрузим DataFrame с матрицей корреляции признаков
    corr_matrix = fp.ParquetFile('base_models/data/stat/corr_matrix_'+feature_space).to_pandas()

    # загрузим обьект исcледования optuna
    optuna_study_rf_stat = optuna.load_study(study_name='RandomForestClassifier_'+feature_space+'_stat',
                               storage='sqlite:///optuna_studies.db')
    
    # из полученного обьекта сформируем Data Frame
    optuna_study_rf_stat_pd = optuna_study_rf_stat.trials_dataframe().dropna()

    # переименуем столбы
    optuna_study_rf_stat_pd.rename(columns={
        'values_0': 'roc_train', 
        'values_1': 'roc_valid', 
        'values_2': 'f1_score', 
        'values_3': 'recall_1',
        'values_4': 'precision_1'
    },inplace=True)

    # удалим ненужные столбцы
    optuna_study_rf_stat_pd.drop(['datetime_start','datetime_complete','state','duration'],axis=1, inplace=True)

    # выберем точки с лучшими значениями метрики roc auc на валидационном наборе
    best_roc_rf_stat = optuna_study_rf_stat_pd.sort_values(by='roc_valid',ascending=False)[:best_models_number] 

    # выберем точки с лучшими значениями метрики precion класса 1 на валидационном наборе
    best_precion_rf_stat = optuna_study_rf_stat_pd.sort_values(by='precision_1',ascending=False)[:best_models_number]   

    # выберем точки в которых максимальна метрика f1-score класса 1 (precision_1=recall_1)
    best_f1_rf_stat = optuna_study_rf_stat_pd.sort_values(by='f1_score',ascending=False)[:best_models_number]

    # сформируем набор признаков для лучших моделей
    best_models_rf_stat = pd.concat([best_roc_rf_stat,best_precion_rf_stat,best_f1_rf_stat]).drop_duplicates().reset_index(drop=True)

    # сформируем список индексов best_models_rf_stat
    list_index = best_models_rf_stat.index

    # сформируем  метапризнаки для обучения метамодели
    # для этого необходиом обучить модель RandomForestClassifier с выбранными гиперпараметрами
    for index in list_index:

        # для того чтобы следить за выполнением цикла введем индикатор
        print('Current space: ', feature_space)
        print('Number of indexes: ', max(list_index))
        print('Current index: ', index)
        
        # сформируем необходимые параметры
        n_estimators = best_models_rf_stat.loc[index,'params_n_estimators']
        criterion = best_models_rf_stat.loc[index,'params_criterion']
        max_depth = best_models_rf_stat.loc[index,'params_max_depth']
        min_samples_split = best_models_rf_stat.loc[index,'params_min_samples_split']
        min_samples_leaf = best_models_rf_stat.loc[index,'params_min_samples_leaf']
        max_features = best_models_rf_stat.loc[index,'params_max_features']
        class_weight = {0:best_models_rf_stat.loc[index,'params_class_0_weight'],
                        1:best_models_rf_stat.loc[index,'params_class_1_weight']}
        max_samples = best_models_rf_stat.loc[index,'params_max_samples']
        threshold = best_models_rf_stat.loc[index,'params_threshold']
        class_1_percent = best_models_rf_stat.loc[index,'params_class_1_percent']
        random_state = best_models_rf_stat.loc[index,'params_random_state']

        # с помощью функции corr_transform_to_force получим
        # список не коррелируемых признаков
        list_ncorr_features = corr_transform_to_force(corr_matrix,threshold=threshold)[1]

        # сохраним list_ncorr_features для дальнейшего воспроизведения
        dump(list_ncorr_features, 'base_models/list_ncorr_features/'+'list_ncorr_features_RFST_'+feature_space+'_'+str('0'+str(index))[-2:]+'.joblib')

        # сформируем данные для анализа сбалансированности обучающих данных
        print('Loading train data')
        X_train_pd = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
        y_train_pd = pd.read_csv('target/target_train.csv')

        # поготовим данные y_train_pd к работе с функцией class_1_percent_samples
        y_train_pd.set_index('id',drop=False,inplace=True)

        # подготовим данные для обучения модели
        # с помощью функции class_1_percent_samples зададим долю 
        # класса 1
        list_c1_percent_id = class_1_percent_samples(y_train_pd,class_1_percent,random_state=random_state)[:1000000]

        # подготовим данные для обучения
        X_train_balanced = X_train_pd.loc[list_c1_percent_id]
        y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag']

        # освободим память от "тяжелых" и ненужных файлов
        del X_train_pd, y_train_pd
        gc.collect()

        # для того чтобы следить за выполнением цикла введем индикатор
        print('start train')
        # обучаем модель RandomForestClassifier с наилучшеми параметрами
        random_forest = RandomForestClassifier(
                n_estimators = n_estimators,
                criterion = criterion,
                max_depth = max_depth,
                min_samples_split = min_samples_split,
                min_samples_leaf = min_samples_leaf,
                max_features = max_features,
                class_weight = class_weight,
                max_samples = max_samples,
                n_jobs=-1,
                random_state=42)
        random_forest.fit(X_train_balanced, y_train_balanced)
        # для того чтобы следить за выполнением цикла введем индикатор
        print('finished train')

        # сохраним модель для дальнейшего воспроизведения
        dump(random_forest, 'base_models/models/'+'RFST_'+feature_space+'_'+str('0'+str(index))[-2:]+'.joblib')

        # удаляем крупные файлы чтобы высвободить память 
        del X_train_balanced,y_train_balanced
        gc.collect()

        # для того чтобы следить за выполнением цикла введем индикатор
        print('Loading test data')
        # подгружаем данные для тестирования
        X_test = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_test').to_pandas()[list_ncorr_features]
        # для метрик ROC AUC делаем предсказание модели в виде вероятности и записываем в методанные
        features_first_meta_stat['RFST_'+feature_space+'_'+str('0'+str(index))[-2:]] = random_forest.predict_proba(X_test)[:,1]
        # удаляем крупные файлы чтобы высвободить память 
        del X_test
        gc.collect()
        clear_output()

# сохраним полученные метапризнаки
fp.write('first_metamodel/features/features_first_meta_stat',features_first_meta_stat)

##### <span style="color:MediumSlateBlue">Формирование метапризнаков от модели *Gradient Boosting Classifier* обученной сбалансированными *transform data stat* данными</span>

In [ ]:
# сформируем метапризнаки для каждого подпространства признаков
for feature_space in dict_spaces.keys():
    # обьявим количество признаков подпространства
    count_features = dict_spaces[feature_space]

    # для работы функции corr_transform_to_force
    # подгрузим DataFrame с матрицей корреляции признаков
    corr_matrix = fp.ParquetFile('base_models/data/stat/corr_matrix_'+feature_space).to_pandas()

    # загрузим обьект исcледования optuna
    optuna_study_gb_stat = optuna.load_study(study_name='HistGradientBoostingClassifier_'+feature_space+'_stat',
                               storage='sqlite:///optuna_studies.db')
    
    # из полученного обьекта сформируем Data Frame
    optuna_study_gb_stat_pd = optuna_study_gb_stat.trials_dataframe().dropna()

    # переименуем столбы
    optuna_study_gb_stat_pd.rename(columns={
        'values_0': 'roc_train', 
        'values_1': 'roc_valid', 
        'values_2': 'f1_score', 
        'values_3': 'recall_1',
        'values_4': 'precision_1'
    },inplace=True)

    # удалим ненужные столбцы
    optuna_study_gb_stat_pd.drop(['datetime_start','datetime_complete','state','duration'],axis=1, inplace=True)

    # выберем точки с лучшими значениями метрики roc auc на валидационном наборе
    best_roc_gb_stat = optuna_study_gb_stat_pd.sort_values(by='roc_valid',ascending=False)[:best_models_number] 

    # выберем точки с лучшими значениями метрики precion класса 1 на валидационном наборе
    best_precion_gb_stat = optuna_study_gb_stat_pd.sort_values(by='precision_1',ascending=False)[:best_models_number]   

    # выберем точки в которых максимальна метрика f1-score класса 1 (precision_1=recall_1)
    best_f1_gb_stat = optuna_study_gb_stat_pd.sort_values(by='f1_score',ascending=False)[:best_models_number]

    # сформируем набор признаков для лучших моделей
    best_models_gb_stat = pd.concat([best_roc_gb_stat,best_precion_gb_stat,best_f1_gb_stat]).drop_duplicates().reset_index(drop=True)

    # сформируем список индексов best_models_gb_stat
    list_index = best_models_gb_stat.index

    # сформируем  метапризнаки для обучения метамодели
    # для этого необходиом обучить модель HistGradientBoostingClassifier с выбранными гиперпараметрами
    for index in list_index:

        # для того чтобы следить за выполнением цикла введем индикатор
        print('Current space: ', feature_space)
        print('Number of indexes: ', max(list_index))
        print('Current index: ', index)
        
        # сформируем необходимые параметры
        learning_rate = best_models_gb_stat.loc[index,'params_learning_rate']
        max_iter = best_models_gb_stat.loc[index,'params_max_iter']
        max_leaf_nodes = best_models_gb_stat.loc[index,'params_max_leaf_nodes']
        max_depth = best_models_gb_stat.loc[index,'params_max_depth']
        min_samples_leaf = best_models_gb_stat.loc[index,'params_min_samples_leaf']
        l2_regularization = best_models_gb_stat.loc[index,'params_l2_regularization']
        max_features = best_models_gb_stat.loc[index,'params_max_features']
        class_weight = {0:best_models_gb_stat.loc[index,'params_class_0_weight'],
                        1:best_models_gb_stat.loc[index,'params_class_1_weight']}
        threshold = best_models_gb_stat.loc[index,'params_threshold']
        class_1_percent = best_models_gb_stat.loc[index,'params_class_1_percent']
        random_state = best_models_gb_stat.loc[index,'params_random_state']

        # с помощью функции corr_transform_to_force получим
        # список не коррелируемых признаков
        list_ncorr_features = corr_transform_to_force(corr_matrix,threshold=threshold)[1]

        # сохраним list_ncorr_features для дальнейшего воспроизведения
        dump(list_ncorr_features, 'base_models/list_ncorr_features/'+'list_ncorr_features_GBST_'+feature_space+'_'+str('0'+str(index))[-2:]+'.joblib')

        # сформируем данные для анализа сбалансированности обучающих данных
        print('Loading train data')
        X_train_pd = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_train').to_pandas()[list_ncorr_features]
        y_train_pd = pd.read_csv('target/target_train.csv')

        # поготовим данные y_train_pd к работе с функцией class_1_percent_samples
        y_train_pd.set_index('id',drop=False,inplace=True)

        # подготовим данные для обучения модели
        # с помощью функции class_1_percent_samples зададим долю 
        # класса 1
        list_c1_percent_id = class_1_percent_samples(y_train_pd,class_1_percent,random_state=random_state)[:1000000]

        # подготовим данные для обучения
        X_train_balanced = X_train_pd.loc[list_c1_percent_id]
        y_train_balanced = y_train_pd.loc[list_c1_percent_id]['flag']

        # освободим память от "тяжелых" и ненужных файлов
        del X_train_pd, y_train_pd
        gc.collect()

        # для того чтобы следить за выполнением цикла введем индикатор
        print('start train')
        # обучаем модель HistGradientBoostingClassifier с наилучшеми параметрами
        gradient_boosting = HistGradientBoostingClassifier(
                learning_rate = learning_rate,
                max_iter = max_iter,
                max_leaf_nodes = max_leaf_nodes,
                max_depth = max_depth,
                min_samples_leaf = min_samples_leaf,
                l2_regularization = l2_regularization,
                max_features = max_features,
                class_weight = class_weight,
                random_state=42)
        gradient_boosting.fit(X_train_balanced,y_train_balanced)
        # для того чтобы следить за выполнением цикла введем индикатор
        print('finished train')

        # сохраним модель для дальнейшего воспроизведения
        dump(gradient_boosting, 'base_models/models/'+'GBST_'+feature_space+'_'+str('0'+str(index))[-2:]+'.joblib')

        # удаляем крупные файлы чтобы высвободить память 
        del X_train_balanced,y_train_balanced
        gc.collect()

        # для того чтобы следить за выполнением цикла введем индикатор
        print('Loading test data')
        # подгружаем данные для тестирования
        X_test = fp.ParquetFile('base_models/data/stat/'+feature_space+'_stat_test').to_pandas()[list_ncorr_features]
        # для метрик ROC AUC делаем предсказание модели в виде вероятности и записываем в методанные
        features_first_meta_stat['GBST_'+feature_space+'_'+str('0'+str(index))[-2:]] = gradient_boosting.predict_proba(X_test)[:,1]
        # удаляем крупные файлы чтобы высвободить память 
        del X_test
        gc.collect()
        clear_output()

# сохраним полученные метапризнаки
fp.write('first_metamodel/features/features_first_meta_stat',features_first_meta_stat)

In [ ]:
features_first_meta_stat = fp.ParquetFile('first_metamodel/features/features_first_meta_stat').to_pandas()
features_first_meta_stat

#### <span style="color:MediumBlue">Формирование тренировочного, валидационного и тестового набора данных</span>

Поставленная задача сформировать блендинг моделей  
блендинг будет 3 слойный
сформируем для каждого слоя свой обучающий набор данных.

In [ ]:
features_first_meta_stat = fp.ParquetFile('first_metamodel/features/features_first_meta_stat').to_pandas().iloc[:,:10]

In [ ]:
# Выделим из features_first_meta_stat тренировочную часть
MX_train, my_train, MX_test, my_test = my_train_test_split(features_first_meta_stat.set_index('id').drop(['flag'],axis=1),
                                                           features_first_meta_stat.set_index('id')['flag'], train_size=0.35)

In [ ]:
# Раздедлим обучающий набор на тренировочную и валидационную части
MX_train, my_train, MX_valid, my_valid = my_train_test_split(MX_train,my_train, train_size=0.8)

Для воспроизводимости кода и корректного сравнения   
различных моделей завиксируем id для тренировочногоб валидационного   
и тестосого наьоров данных

In [ ]:
# Спрячем данные в DataDfame для последующего сохраниения в csv
mf_train_id = pd.DataFrame(my_train.index)
mf_valid_id = pd.DataFrame(my_valid.index)
mf_test_id = pd.DataFrame(my_test.index)

In [ ]:
# Сохраняем полученные DataFrame
mf_train_id.to_csv('first_metamodel/data/mf_train_id.csv')
mf_valid_id.to_csv('first_metamodel/data/mf_valid_id.csv')
mf_test_id.to_csv('first_metamodel/data/mf_test_id.csv')

In [ ]:
# подгрузим DataFrame с id для тренировочного/валидационного/тестового набора данных
mf_train_id = pd.read_csv('first_metamodel/data/mf_train_id.csv')['id'].to_list()
mf_valid_id = pd.read_csv('first_metamodel/data/mf_valid_id.csv')['id'].to_list()
mf_test_id = pd.read_csv('first_metamodel/data/mf_test_id.csv')['id'].to_list()

print(len(mf_train_id))
print(len(mf_valid_id))
print(len(mf_test_id))


In [ ]:
# сохраним тренировочный валидационный и тестовый целевой переменной
# чтобы иметь возможность к ними обратиться
my_train.to_csv('first_metamodel/data/mfy_train.csv')
my_valid.to_csv('first_metamodel/data/mfy_valid.csv')
my_test.to_csv('first_metamodel/data/mfy_test.csv')

#### <span style="color:MediumBlue">Формирование тренировочного, валидационного и тестового набора на features_first_meta_torow данных</span>

In [ ]:
features_first_meta_torow = fp.ParquetFile('first_metamodel/features/features_first_meta_torow').to_pandas().set_index('id').drop(['flag'],axis=1)
features_first_meta_torow.head(5)

In [ ]:
# сохраним тренировочный валидационный и тестовый наборы данных
# чтобы иметь возможность к ними обратиться
fp.write('first_metamodel/features/features_first_meta_torow_train',features_first_meta_torow.loc[mf_train_id])
fp.write('first_metamodel/features/features_first_meta_torow_valid',features_first_meta_torow.loc[mf_valid_id])
fp.write('first_metamodel/features/features_first_meta_torow_test',features_first_meta_torow.loc[mf_test_id])

#### <span style="color:MediumBlue">Формирование тренировочного, валидационного и тестового набора на features_first_meta_stat данных</span>

In [ ]:
features_first_meta_stat = fp.ParquetFile('first_metamodel/features/features_first_meta_stat').to_pandas().set_index('id').drop(['flag'],axis=1)
features_first_meta_stat.head(5)

In [ ]:
# сохраним тренировочный валидационный и тестовый наборы данных
# чтобы иметь возможность к ними обратиться
fp.write('first_metamodel/features/features_first_meta_stat_train',features_first_meta_stat.loc[mf_train_id])
fp.write('first_metamodel/features/features_first_meta_stat_valid',features_first_meta_stat.loc[mf_valid_id])
fp.write('first_metamodel/features/features_first_meta_stat_test',features_first_meta_stat.loc[mf_test_id])

### <span style="color:RoyalBlue">Построение блендинга из базовых моделей обученных на torow данных</span>

#### <span style="color:MediumBlue">Построение метамодели Logistic Regression</span>

##### <span style="color:MediumSlateBlue">Baseline обучение метамодели LogisticRegression</span>

In [ ]:
MX_train = fp.ParquetFile('first_metamodel/features/features_first_meta_torow_train').to_pandas()
my_train = pd.read_csv('first_metamodel/data/mfy_train.csv').set_index('id')['flag']
MX_valid = fp.ParquetFile('first_metamodel/features/features_first_meta_torow_valid').to_pandas()
my_valid = pd.read_csv('first_metamodel/data/mfy_valid.csv').set_index('id')['flag']

In [ ]:
# обучаем модель логистической регрессии
logistic_regression = linear_model.LogisticRegression(random_state=42, max_iter=10000)
logistic_regression.fit(MX_train,my_train)

# для метрик ROC AUC делаем предсказание модели в виде вероятности
my_train_LR_pred_proba = logistic_regression.predict_proba(MX_train)[:,1]
my_valid_LR_pred_proba = logistic_regression.predict_proba(MX_valid)[:,1]

# Делаем предсказание для валидационной выборки
my_valid_LR_pred = logistic_regression.predict(MX_valid)

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(my_train, my_train_LR_pred_proba),3))
print('ROC AUC на тестовом наборе', round(metrics.roc_auc_score(my_valid, my_valid_LR_pred_proba),3))

print('Основные метрики на тестовом наборе:')
print(metrics.classification_report(my_valid,my_valid_LR_pred,zero_division=0))

# time: 3m 5s

##### <span style="color:MediumSlateBlue">Подбор гиперпараметров модели</span>

In [ ]:
# настроим оптимизацию гипер параметров
def optuna_lg_meta(trial):
  # задаем пространства поиска гиперпараметров
  C = trial.suggest_float('C',0.01,1)
  solver = trial.suggest_categorical('solver', ['lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky'])
  class_0_weight = trial.suggest_float('class_0_weight',0.01,1)
  class_1_weight = trial.suggest_float('class_1_weight',0.01,1)
  k_features = trial.suggest_int('k_features', 20, 506,step = 1)
  class_1_percent = trial.suggest_float('class_1_percent',0.1,1)
  random_state = trial.suggest_int('random_state', 1, 1000000)

  # создаем модель
  optuna_log_reg_meta = linear_model.LogisticRegression(
      C=C,
      solver = solver,
      class_weight={0:class_0_weight,1:class_1_weight},
      random_state=random_state,
      max_iter=10000)

  # Загружаем обучающие наборы
  MX_train_pd = fp.ParquetFile('first_metamodel/features/features_first_meta_torow_train').to_pandas()
  my_train_pd = pd.read_csv('first_metamodel/data/mfy_train.csv')

  # поготовим данные my_train_pd к работе с функцией class_1_percent_samples
  my_train_pd.set_index('id',drop=False,inplace=True)

  # с помощью функции class_1_percent_samples зададим долю класса 1
  list_c1_percent_id = class_1_percent_samples(my_train_pd,class_1_percent,random_state=random_state)[:1000000]

  # подготовим данные для обучения
  MX_train_balanced = MX_train_pd.loc[list_c1_percent_id]
  my_train_balanced = my_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

  # освободим память от "тяжелых" и ненужных файлов
  del MX_train_pd, my_train_pd
  gc.collect()
  
  # с помощью класса SelectKBest получим список лучших признаков
  selector = SelectKBest(f_classif, k=k_features)
  selector.fit(MX_train_balanced, my_train_balanced)
  list_best_features = selector.get_feature_names_out()

  MX_train_balanced = MX_train_balanced[list_best_features].to_numpy()

  # я не воспользовался параметром timeout метода optimize потому, что  
  # optimize отставливает поиск параметров, если время обучения 
  # превышает timeout. Мне нужно чтобы поиск параметров продолжился.
  try:
    # для модуля func_time_out упакуем обучение модели в функцию try_func
    def try_func():
            return optuna_log_reg_meta.fit(MX_train_balanced, my_train_balanced)
    # обучаем модель с ограничением по времени 10 минут (600 секунд)
    optuna_log_reg_meta = func_timeout.func_timeout(600, try_func)

    # удаляем крупные файлы чтобы высвободить память 
    del MX_train_balanced, my_train_balanced
    gc.collect()

    # загружаем тестовые наборы
    MX_train = fp.ParquetFile('first_metamodel/features/features_first_meta_torow_train').to_pandas()[list_best_features].to_numpy()
    MX_valid = fp.ParquetFile('first_metamodel/features/features_first_meta_torow_valid').to_pandas()[list_best_features].to_numpy()
    my_train = pd.read_csv('first_metamodel/data/mfy_train.csv')['flag'].to_numpy()
    my_valid = pd.read_csv('first_metamodel/data/mfy_valid.csv')['flag'].to_numpy()
    
    # делаем предсказание на обучающем и валидационном наборе и считаем метрики
    roc_train = metrics.roc_auc_score(my_train, optuna_log_reg_meta.predict_proba(MX_train)[:,1])
    roc_valid = metrics.roc_auc_score(my_valid, optuna_log_reg_meta.predict_proba(MX_valid)[:,1])
    f1_score = metrics.f1_score(my_valid, optuna_log_reg_meta.predict(MX_valid))
    recall_1 = metrics.recall_score(my_valid, optuna_log_reg_meta.predict(MX_valid),average=None,zero_division=0)[1]
    precision_1 = metrics.precision_score(my_valid, optuna_log_reg_meta.predict(MX_valid),average=None,zero_division=0)[1]

    # удаляем крупные файлы чтобы высвободить память 
    del MX_train, MX_valid, my_train, my_valid
    gc.collect()

  except func_timeout.FunctionTimedOut:
    # удаляем крупные файлы чтобы высвободить память 
    del MX_train_balanced, my_train_balanced
    gc.collect()
    
    # фиксируем пустые значения метрик
    roc_train = 0
    roc_valid = 0
    f1_score = 0
    recall_1 = 0
    precision_1 = 0
    pass

  return roc_train, roc_valid, f1_score, recall_1, precision_1

In [ ]:
# cоздаем объект исследования
# чтобы модель не переобучалась минимизируем roc_train 
# и максимизируем roc_valid
optuna_study_lg_meta = optuna.create_study(study_name="LogisticRegression_meta_first_torow", 
                               directions=['maximize','maximize','maximize','maximize','maximize'], 
                               sampler=optuna.samplers.TPESampler(),
                               pruner='Hyperband',
                               storage='sqlite:///optuna_studies.db',
                               load_if_exists=True)
# optuna.delete_study(study_name="LogisticRegression_meta_first_torow", storage='sqlite:///optuna_studies.db')

In [ ]:
# ищем лучшую комбинацию гиперпараметров n_trials раз
optuna_study_lg_meta.optimize(optuna_lg_meta, n_trials=300)

##### <span style="color:MediumSlateBlue">Анализ гиперпараметров</span>

In [ ]:
# из полученного результат соврмируем Data Frame
optuna_study_lg_meta_pd = optuna_study_lg_meta.trials_dataframe().dropna()
# переименуем столбы
optuna_study_lg_meta_pd.rename(columns={
    'values_0': 'roc_train',
    'values_1': 'roc_valid',
    'values_2': 'f1_score',
    'values_3': 'recall_1',
    'values_4': 'precision_1'
},inplace=True)
# Выделим время потраченное на обучение модели
optuna_study_lg_meta_pd.tail()

In [ ]:
# покаждем статистику обучения
print('Максимальное значение метрики ROC AUC на валидационном наборе:',round(optuna_study_lg_meta_pd['roc_valid'].max(),3))
print('Среднее значение метрики ROC AUC на валидационном наборе:',round(optuna_study_lg_meta_pd['roc_valid'].mean(),3))
print('Максимальное значение метрики f1_score на валидационном наборе:',round(optuna_study_lg_meta_pd['f1_score'].max(),3))
print('Среднее значение метрики f1_score на валидационном наборе:',round(optuna_study_lg_meta_pd['f1_score'].mean(),3))
print('Максимальное значение метрики recall_1 на валидационном наборе:',round(optuna_study_lg_meta_pd['recall_1'].max(),3))
print('Среднее значение метрики recall_1 на валидационном наборе:',round(optuna_study_lg_meta_pd['recall_1'].mean(),3))
print('Максимальное значение метрики precision_1 на валидационном наборе:',round(optuna_study_lg_meta_pd['precision_1'].max(),3))
print('Среднее значение метрики precision_1 на валидационном наборе:',round(optuna_study_lg_meta_pd['precision_1'].mean(),3))

In [ ]:
# построим график важности гиперпарметров
optuna.visualization.plot_param_importances(optuna_study_lg_meta, target_name='Score')

In [ ]:
# Построим зависимость метрики precision_1 от гипер параметров

fig = px.scatter(
    data_frame=optuna_study_lg_meta_pd,
    x='precision_1', #ось абсцисс
    y=['f1_score','roc_train', 'roc_valid', 'recall_1'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision_1',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Визуализируем метрики полученные при optuna оптимизации
fig = px.scatter(
    data_frame=optuna_study_lg_meta_pd[(optuna_study_lg_meta_pd['precision_1']>0.15)&
                                  (optuna_study_lg_meta_pd['recall_1']>0.1)],
    x='number', #ось абсцисс
    y=['roc_train','roc_valid','recall_1','precision_1'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость качества модели от trials optuna', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='trials optuna',
    yaxis_title='Величина метрики')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

<span style="color:Blue">

Вывод:  

Их всех выбираем точку $number =248$.   
В этой точке самые оптимальные значения $recall$ и $precision$.  
Посмотрим каким значениям гиперпараметров соотвествует выбранная точка.

In [ ]:
# определим номер лучшге варианта
best_optuna_number = 248

# сформируем словарь лучших гипирпарметров RandomForestClassifier
best_param_lr = {
    'solver' : optuna_study_lg_meta_pd[optuna_study_lg_meta_pd['number']==best_optuna_number]['params_solver'].iloc[0],
    'C' : round(optuna_study_lg_meta_pd[optuna_study_lg_meta_pd['number']==best_optuna_number]['params_C'].iloc[0],3),
    'class_weight' : {0:round(optuna_study_lg_meta_pd[optuna_study_lg_meta_pd['number']==best_optuna_number]['params_class_0_weight'].iloc[0],3),
                      1:round(optuna_study_lg_meta_pd[optuna_study_lg_meta_pd['number']==best_optuna_number]['params_class_1_weight'].iloc[0],3)},
    }

# создадим перменные
best_k_features = optuna_study_lg_meta_pd[optuna_study_lg_meta_pd['number']==best_optuna_number]['params_k_features'].iloc[0]
best_random_state = optuna_study_lg_meta_pd[optuna_study_lg_meta_pd['number']==best_optuna_number]['params_random_state'].iloc[0]
best_class_1_percent = optuna_study_lg_meta_pd[optuna_study_lg_meta_pd['number']==best_optuna_number]['params_class_1_percent'].iloc[0]

# Выведем принятые наилучшие праметры
print('best solver:',best_param_lr['solver'])
print('best C:',best_param_lr['C'])
print('best class 0 weight:',best_param_lr['class_weight'][0])
print('best class 1 weight:',best_param_lr['class_weight'][1])

print('best class 1 percent:',round(best_class_1_percent,3))
print('best k features:',best_k_features)
print('best random state:',best_random_state)
print('time for best train:',round(optuna_study_lg_meta_pd[optuna_study_lg_meta_pd['number']==best_optuna_number]['duration'].iloc[0].seconds/60),'minutes')

print()
print('ROC AUC на обучающем наборе:', round(optuna_study_lg_meta_pd[optuna_study_lg_meta_pd['number']==best_optuna_number]['roc_train'].iloc[0],3))
print('ROC AUC на валидационном наборе:', round(optuna_study_lg_meta_pd[optuna_study_lg_meta_pd['number']==best_optuna_number]['roc_valid'].iloc[0],3))
print('precision класса 1:', round(optuna_study_lg_meta_pd[optuna_study_lg_meta_pd['number']==best_optuna_number]['precision_1'].iloc[0],3))
print('recall класса 1:', round(optuna_study_lg_meta_pd[optuna_study_lg_meta_pd['number']==best_optuna_number]['recall_1'].iloc[0],3))


##### <span style="color:MediumSlateBlue">Обучение модели с лучшими параметрами</span>

In [ ]:
# Загружаем обучающие наборы
MX_train_pd = fp.ParquetFile('first_metamodel/features/features_first_meta_torow_train').to_pandas()
my_train_pd = pd.read_csv('first_metamodel/data/mfy_train.csv')

# поготовим данные my_train_pd к работе с функцией class_1_percent_samples
my_train_pd.set_index('id',drop=False,inplace=True)

# с помощью функции class_1_percent_samples зададим долю класса 1
list_c1_percent_id = class_1_percent_samples(my_train_pd,best_class_1_percent,random_state=best_random_state)[:1000000]

# подготовим данные для обучения
MX_train_balanced = MX_train_pd.loc[list_c1_percent_id]
my_train_balanced = my_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

# освободим память от "тяжелых" и ненужных файлов
del MX_train_pd, my_train_pd
gc.collect()

# с помощью класса SelectKBest получим список лучших признаков
selector = SelectKBest(f_classif, k=best_k_features)
selector.fit(MX_train_balanced, my_train_balanced)
list_best_features = selector.get_feature_names_out()

MX_train_balanced = MX_train_balanced[list_best_features].to_numpy()


# обучаем модель LogisticRegression с наилучшеми параметрами
logistic_regression = linear_model.LogisticRegression(
        **best_param_lr,
        random_state=best_random_state,
        max_iter=10000)
logistic_regression.fit(MX_train_balanced, my_train_balanced)

# удаляем крупные файлы чтобы высвободить память 
del MX_train_balanced, my_train_balanced
gc.collect()

# загружаем тестовые наборы
MX_train = fp.ParquetFile('first_metamodel/features/features_first_meta_torow_train').to_pandas()[list_best_features].to_numpy()
MX_valid = fp.ParquetFile('first_metamodel/features/features_first_meta_torow_valid').to_pandas()[list_best_features].to_numpy()
my_train = pd.read_csv('first_metamodel/data/mfy_train.csv')['flag'].to_numpy()
my_valid = pd.read_csv('first_metamodel/data/mfy_valid.csv')['flag'].to_numpy()

# для метрик ROC AUC делаем предсказание модели в виде вероятности
my_train_lr_pred_proba = logistic_regression.predict_proba(MX_train)[:,1]
my_valid_lr_pred_proba = logistic_regression.predict_proba(MX_valid)[:,1]

# Делаем предсказание для валидационной выборки
my_valid_lr_pred = logistic_regression.predict(MX_valid)

# удаляем крупные файлы чтобы высвободить память 
del MX_train, MX_valid
gc.collect()

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(my_train, my_train_lr_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(my_valid, my_valid_lr_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(my_valid,my_valid_lr_pred,zero_division=0))

# time: 2m 30s

<span style="color:Blue">
Вывод:  

Модель $LogisticRegression$, обученная на сбалансированных данных   
$transform$ $data$ $torow$ имеет такое же качество как и модель  
модель $LogisticRegression$, обученная на не тех же не сбалансированных  
данных.

Подбор гиперпараметров модели показал, что модель $LogisticRegression$   
имеет низкую пресказательную способность на данных $transform$ $data$ $torow$.

##### <span style="color:MediumSlateBlue">Анализ влияния порога классификации на качество модели</span>

In [ ]:
# чтобы проше обращаться к каждому обьекту в данных
# преобразуем y_valid_LR_pred к Series типу
my_valid_lr_pred_proba = pd.Series(my_valid_lr_pred_proba)

# формируем список из необходимых значений thresholds
thresholds = np.arange(0.01, 1, 0.01)

# сформируем списко для заполнения данными результатов анализа
threshold_data = []


for threshold in thresholds:
        # сформируем список для заполнения данными текушего threshold
        threshold_current_data = [threshold]

        #клиентов, для которых вероятность дефолта > threshold, относим к классу 1
        #в противном случае — к классу 0
        my_valid_lr_pred = my_valid_lr_pred_proba.apply(lambda x: 1 if x>threshold else 0)


        #Считаем метрики для класса 0 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(my_valid, my_valid_lr_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.precision_score(my_valid, my_valid_lr_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.f1_score(my_valid, my_valid_lr_pred,average=None,zero_division=0)[0],3))

        #Считаем метрики для класса 1 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(my_valid, my_valid_lr_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.precision_score(my_valid, my_valid_lr_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.f1_score(my_valid, my_valid_lr_pred,average=None,zero_division=0)[1],3))

        # добавляем полученные данные в список threshold_data
        threshold_data.append(threshold_current_data)

        # из полученных данных сформируем dataframe
        thresholds_pd =pd.DataFrame(
        columns=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'],
        data = threshold_data)

# time: 17s

In [ ]:
# Визуализируем метрики на валидационном наборе при различных threshold
fig_threshold = px.line(
    data_frame=thresholds_pd,
    x='threshold', #ось абсцисс
    y=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'], #ось ординат
)
fig_threshold.update_layout(
    title ={
        'text':'Зависимость качества модели от threshold', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Порог классификации threshold',
    yaxis_title='Величина метрики')
fig_threshold.update_xaxes(showspikes=True)
fig_threshold.update_yaxes(showspikes=True)

fig_threshold.show()

Метрика $precision$ показывает на сколько сильно ошиблась модель при определиние класса.  
Чем выше метрика, тем меньше ошиблась модель. 

$$precision = \frac{TP}{TP+FP}$$

Метрика $recall$ показывает способность модели найти класс.  
Чем выше метрика, тем выше способность модели находить класс.  

$$recall = \frac{TP}{TP+FN}$$

Метрика $f1-score$ показывает баланс между $precision$ и $recall$ 

$threshold$- порог класификации модели.  
Все объекты для которых $у_{\text{proba pred}} < threshold$, модель относит к класс 0.  
Соотвественно, если $у_{\text{proba pred}}  > threshold$ модель относит обьект к классу 1.  
Для идельаной модели $threshold$, является границей между областями класса 0 и класса 1.  

<span style="color:Blue">

Выводы по результам анализа:  

Для улучшения качества модели по метрике $ROC AUC$, нет смысла сдвигать порог   
классификации. Но зависимость метрик качества модели от $threshold$, может рассказать о  
спосбоности модели искать класс 1 и класс 0.

При фокусировке модели на классе 1 (более низкий $threshold$):  
- на классе 1 метрика $precision$ уменьшается, метрика $recall$ растет;  
- на классе 0 метрика $precision$ растет, метрика $recall$ уменьшается.  

При фокусировке модели на классе 0 (более высокий $threshold$):  
- на классе 1 метрика $precision$ увеличивается, метрика $recall$ уменьшается;  
- на классе 0 метрика $precision$ уменьшается, метрика $recall$ растет.  

Обьсняется тем, что в условия дисбаланса модель научилась хорошо определять класс 0 и плохо класс 1.  

При фокусировке модели на классе 1 (более низкий $threshold$), все больше обьектов отнесены к классу 1.   
Так как $precision$ класса 1 уменьшается, значит в области класса 1 стало больше обектов класса 0,  
что явлется закономерным для "идеальной" модели.  
Однако при этом, $recall$ класса 1 растет, значит в добавленных обьектах также присутвует класс 1.  
Это как раз указывает на то, что модель не научилась находить среди класса 0 класс 1. В области с     
высокой вероятностью класса 0 (более низкий $threshold$), находятся обьекты класса 1.  

Фокусировка на классе 0 (более высокий $threshold$), заставляет модель все больше обьектов из области класса 1  
относить к классу 0. При этом увеличение метрики $precision$ класса 1 говорит о том, что в области класса 1     
становится меньше 0, а значит модель умеет среди класса 1, находить класс 0.  
После  $threshold=0.92$ модель впринципе теряет способность отличать класс 1. У модели нет обьектов класса 1   
в которых она "уверена" на 92+%.

Обратим внимание на $threshold=0.52$, в этой точке находится баланс межу метриками качества модели:
$$precision_1 = recall_1 = \text{f1-score}_1 = 0.155$$
$$precision_0 = recall_0 = \text{f1-score}_0 = 0.968$$

Для модели с лучшим качеством эти две точки должны находится максимально близко к друг другу и иметь  
высокое значение метрик $precision_1$ и $recall_1$.
Для класса 0 это условие выполнется, что также говорит о том, что модель научилась искать этот класс.  
Для класса 1 это условие не выполнется, что также говорит о том, что модель плохо ищет класс 1.

##### <span style="color:MediumSlateBlue">Построение ROC-кривой выбранной модели на тестовом наборе</span>

In [ ]:
# загружаем тестовые наборы
MX_train = fp.ParquetFile('first_metamodel/features/features_first_meta_torow_train').to_pandas()[list_best_features].to_numpy()
MX_valid = fp.ParquetFile('first_metamodel/features/features_first_meta_torow_valid').to_pandas()[list_best_features].to_numpy()
MX_test = fp.ParquetFile('first_metamodel/features/features_first_meta_torow_test').to_pandas()[list_best_features].to_numpy()
my_train = pd.read_csv('first_metamodel/data/mfy_train.csv')['flag'].to_numpy()
my_valid = pd.read_csv('first_metamodel/data/mfy_valid.csv')['flag'].to_numpy()
my_test = pd.read_csv('first_metamodel/data/mfy_test.csv')['flag'].to_numpy()

In [ ]:
print("Размеры обучающего набора",MX_train.shape)
print("Размеры валидационного набора",MX_valid.shape)
print("Размеры тестового набора",MX_test.shape)
# time: 1m 43s

In [ ]:
# для метрик ROC AUC делаем предсказание модели в виде вероятности
my_train_lr_pred_proba = logistic_regression.predict_proba(MX_train)[:,1]
my_valid_lr_pred_proba = logistic_regression.predict_proba(MX_valid)[:,1]
my_test_lr_pred_proba = logistic_regression.predict_proba(MX_test)[:,1]

# Делаем предсказание для валидационной выборки
my_test_lr_pred = logistic_regression.predict(MX_test)

del MX_train, MX_valid, MX_test
gc.collect()

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(my_train, my_train_lr_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(my_valid, my_valid_lr_pred_proba),3))
print('ROC AUC на тестовом наборе', round(metrics.roc_auc_score(my_test, my_test_lr_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(my_test,my_test_lr_pred,zero_division=0))

In [ ]:
# Для расчета значений TPR и FPR используем функцию roc_curve
fpr_train, tpr_train, _ = metrics.roc_curve(my_train, my_train_lr_pred_proba)
fpr_valid, tpr_valid, _ = metrics.roc_curve(my_valid, my_valid_lr_pred_proba)
fpr_test, tpr_test, _ = metrics.roc_curve(my_test, my_test_lr_pred_proba)

# рассчитаем площадь под кривой
auc_train = round(metrics.roc_auc_score(my_train, my_train_lr_pred_proba),3)
auc_valid = round(metrics.roc_auc_score(my_valid, my_valid_lr_pred_proba),3)
auc_test = round(metrics.roc_auc_score(my_test, my_test_lr_pred_proba),3)

trace_train = go.Scatter(x=fpr_train, y=tpr_train, mode='lines', name='AUC_train = %0.2f' % auc_train,
                   line=dict(color='red', width=2),
                   hovertemplate='train<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_train])

trace_valid = go.Scatter(x=fpr_valid, y=tpr_valid, mode='lines', name='AUC_valid = %0.2f' % auc_valid,
                   line=dict(color='green', width=2),
                   hovertemplate='valid<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_valid])

trace_test = go.Scatter(x=fpr_test, y=tpr_test, mode='lines', name='AUC_test = %0.2f' % auc_test,
                   line=dict(color='darkorange', width=2),
                   hovertemplate='test<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_test])


reference_line = go.Scatter(x=[0,1], y=[0,1], mode='lines', name='Reference Line',
                            line=dict(color='navy', width=2, dash='dash'))

fig_roc = go.Figure(data=[trace_train,trace_valid,trace_test,reference_line])

fig_roc.update_layout(
    title ={
        'text':'ROC-кривая', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 1350,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate')

fig_roc.update_xaxes(showspikes=True)
fig_roc.update_yaxes(showspikes=True)


fig_roc.show()

<span style="color:Blue">

Вывод по ROC-кривой:
1. Модель выдает сталибильное качество на всех наборах, а значит  
не переобучена - разброс ($variance$) не большой.  
2. На тестовом наборе полученно качество $ROC AUC = 0.75$.

#### <span style="color:MediumBlue">Построение метамодели Random Forest Classifier</span>

##### <span style="color:MediumSlateBlue">Baseline обучение метамодели RandomForestClassifier</span>

In [ ]:
MX_train = fp.ParquetFile('first_metamodel/features/features_first_meta_torow_train').to_pandas()
my_train = pd.read_csv('first_metamodel/data/mfy_train.csv').set_index('id')['flag']
MX_valid = fp.ParquetFile('first_metamodel/features/features_first_meta_torow_valid').to_pandas()
my_valid = pd.read_csv('first_metamodel/data/mfy_valid.csv').set_index('id')['flag']

In [ ]:
# обучаем модель логистической регрессии
# чтобы модель пыталась найти связь во всех признаках
random_forest = RandomForestClassifier(random_state=42, n_jobs=-1)
random_forest.fit(MX_train,my_train)
# для метрик ROC AUC делаем предсказание модели в виде вероятности
my_train_rf__pred_proba = random_forest.predict_proba(MX_train)[:,1]
my_valid_rf_pred_proba = random_forest.predict_proba(MX_valid)[:,1]

# Делаем предсказание для валидационной выборки
my_valid_rf_pred = random_forest.predict(MX_valid)

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(my_train, my_train_rf__pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(my_valid, my_valid_rf_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(my_valid,my_valid_rf_pred))

# time: 17s

<span style="color:Blue">

Выводы:

В сравнение с моделью $RandomForestClassifier$ построенной   
на несбалансированных данных $transform$ $data$ $stat$:
1. Качество модели по метрике $ROC AUC$ значительно улучшилось.
2. Качество модели по метрике $recall_1$ значительно  
улучшилось. Правда, $recall_0$ уменьшилось.
3. Качество модели по метрике $precision_1$ уменьшилось.   
4. На порядок уменьшилось время обучения модели. Это логично,  
велична сбалансированной выборки всего $107000$ $samples$. 

##### <span style="color:MediumSlateBlue">Подбор гиперпараметров модели</span>

In [ ]:
# настроим оптимизацию гипер параметров
def optuna_rf_meta(trial):
  # задаем пространства поиска гиперпараметров
  # задаем пространства поиска гиперпараметров
  n_estimators = trial.suggest_int('n_estimators', 40, 100,step = 5)
  criterion = trial.suggest_categorical('criterion',['gini', 'entropy', 'log_loss'])
  max_depth = trial.suggest_int('max_depth', 10, 30,step = 1)
  min_samples_split = trial.suggest_int('min_samples_split', 5, 15,step = 1)
  min_samples_leaf = trial.suggest_int('min_samples_leaf', 5, 15,step = 1)
  max_features = trial.suggest_float('max_features',0.4,0.8)
  max_samples = trial.suggest_float('max_samples',0.4,0.8)
  class_0_weight = trial.suggest_float('class_0_weight',0.1,1)
  class_1_weight = trial.suggest_float('class_1_weight',0.1,1)
  class_1_percent = trial.suggest_float('class_1_percent',0.01,1)
  k_features = trial.suggest_int('k_features', 20, 506,step = 1)
  random_state = trial.suggest_int('random_state', 1, 1000000)


  # создаем модель
  optuna_random_forest_meta = RandomForestClassifier(
      n_estimators = n_estimators, 
      criterion = criterion,
      max_depth = max_depth,
      min_samples_split = min_samples_split,  
      min_samples_leaf = min_samples_leaf,
      max_features=max_features,
      max_samples=max_samples,
      class_weight={0:class_0_weight,1:class_1_weight},
      n_jobs=-1,
      random_state=random_state)
  
  # Загружаем обучающие наборы
  MX_train_pd = fp.ParquetFile('first_metamodel/features/features_first_meta_torow_train').to_pandas()
  my_train_pd = pd.read_csv('first_metamodel/data/mfy_train.csv')

  # поготовим данные my_train_pd к работе с функцией class_1_percent_samples
  my_train_pd.set_index('id',drop=False,inplace=True)

  # с помощью функции class_1_percent_samples зададим долю класса 1
  list_c1_percent_id = class_1_percent_samples(my_train_pd,class_1_percent,random_state=random_state)[:1000000]

  # подготовим данные для обучения
  MX_train_balanced = MX_train_pd.loc[list_c1_percent_id]
  my_train_balanced = my_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

  # освободим память от "тяжелых" и ненужных файлов
  del MX_train_pd, my_train_pd
  gc.collect()
  
  # с помощью класса SelectKBest получим список лучших признаков
  selector = SelectKBest(f_classif, k=k_features)
  selector.fit(MX_train_balanced, my_train_balanced)
  list_best_features = selector.get_feature_names_out()

  MX_train_balanced = MX_train_balanced[list_best_features].to_numpy()
  
  # я не воспользовался параметром timeout метода optimize потому, что  
  # optimize отставливает поиск параметров, если время обучения 
  # превышает timeout. Мне нужно чтобы поиск параметров продолжился.
  try:
    # для модуля func_time_out упакуем обучение модели в функцию try_func
    def try_func():
            return optuna_random_forest_meta.fit(MX_train_balanced, my_train_balanced)
    # обучаем модель с ограничением по времени 15 минут (900 секунд)
    optuna_random_forest_meta = func_timeout.func_timeout(900, try_func)

    # удаляем крупные файлы чтобы высвободить память 
    del MX_train_balanced, my_train_balanced
    gc.collect()

    # загружаем тестовые наборы
    MX_train = fp.ParquetFile('first_metamodel/features/features_first_meta_torow_train').to_pandas()[list_best_features].to_numpy()
    MX_valid = fp.ParquetFile('first_metamodel/features/features_first_meta_torow_valid').to_pandas()[list_best_features].to_numpy()
    my_train = pd.read_csv('first_metamodel/data/mfy_train.csv')['flag'].to_numpy()
    my_valid = pd.read_csv('first_metamodel/data/mfy_valid.csv')['flag'].to_numpy()

    # делаем предсказание на обучающем и валидационном наборе
    #Считаем метрики для класса 1 добавляем их в список
    roc_train = metrics.roc_auc_score(my_train, optuna_random_forest_meta.predict_proba(MX_train)[:,1])
    roc_valid = metrics.roc_auc_score(my_valid, optuna_random_forest_meta.predict_proba(MX_valid)[:,1])
    f1_score = metrics.f1_score(my_valid, optuna_random_forest_meta.predict(MX_valid))
    recall_1 = metrics.recall_score(my_valid, optuna_random_forest_meta.predict(MX_valid),average=None,zero_division=0)[1]
    precision_1 = metrics.precision_score(my_valid, optuna_random_forest_meta.predict(MX_valid),average=None,zero_division=0)[1]

    # удаляем крупные файлы чтобы высвободить память 
    del MX_train, MX_valid, my_train, my_valid
    gc.collect()

  except func_timeout.FunctionTimedOut:
    # удаляем крупные файлы чтобы высвободить память 
    del MX_train_balanced, my_train_balanced
    gc.collect()
    
    # фиксируем пустые значения метрик
    roc_train = 0
    roc_valid = 0
    f1_score = 0
    recall_1 = 0
    precision_1 = 0
    pass

  return roc_train, roc_valid, f1_score, recall_1, precision_1

In [ ]:
# cоздаем объект исследования
optuna_study_rf_meta = optuna.create_study(study_name="RandomForestClassifier_meta_first_torow", 
                               directions=['maximize','maximize','maximize','maximize','maximize'], 
                               sampler=optuna.samplers.TPESampler(),
                               pruner='Hyperband',
                               storage='sqlite:///optuna_studies.db',
                               load_if_exists=True)
# на случай удаления 
# optuna.delete_study(study_name="RandomForestClassifier_meta_first_torow", storage='sqlite:///optuna_studies.db')

In [ ]:
# ищем лучшую комбинацию гиперпараметров n_trials раз
optuna_study_rf_meta.optimize(optuna_rf_meta, n_trials=100)

##### <span style="color:MediumSlateBlue">Анализ гиперпараметров</span>

In [ ]:
# из полученного результат соврмируем Data Frame
optuna_study_rf_meta_pd = optuna_study_rf_meta.trials_dataframe().dropna()
# переименуем столбы
optuna_study_rf_meta_pd.rename(columns={
    'values_0': 'roc_train',
    'values_1': 'roc_valid',
    'values_2': 'f1_score',
    'values_3': 'recall_1',
    'values_4': 'precision_1'
},inplace=True)
optuna_study_rf_meta_pd.tail()

In [ ]:
# покаждем статистику обучения
print('Максимальное значение метрики ROC AUC на валидационном наборе:',round(optuna_study_rf_meta_pd['roc_valid'].max(),3))
print('Среднее значение метрики ROC AUC на валидационном наборе:',round(optuna_study_rf_meta_pd['roc_valid'].mean(),3))
print('Максимальное значение метрики f1_score на валидационном наборе:',round(optuna_study_rf_meta_pd['f1_score'].max(),3))
print('Среднее значение метрики f1_score на валидационном наборе:',round(optuna_study_rf_meta_pd['f1_score'].mean(),3))
print('Максимальное значение метрики recall_1 на валидационном наборе:',round(optuna_study_rf_meta_pd['recall_1'].max(),3))
print('Среднее значение метрики recall_1 на валидационном наборе:',round(optuna_study_rf_meta_pd['recall_1'].mean(),3))
print('Максимальное значение метрики precision_1 на валидационном наборе:',round(optuna_study_rf_meta_pd['precision_1'].max(),3))
print('Среднее значение метрики precision_1 на валидационном наборе:',round(optuna_study_rf_meta_pd['precision_1'].mean(),3))

In [ ]:
# Построим зависимость метрики precision_1 от гиперпараметров

fig = px.scatter(
    data_frame=optuna_study_rf_meta_pd,
    x='precision_1', #ось абсцисс
    y=['roc_train', 'roc_valid', 'recall_1','f1_score'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision класса 1 от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision на классе 1',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрик качества модели от number

fig = px.scatter(
    data_frame=optuna_study_rf_meta_pd[(optuna_study_rf_meta_pd['recall_1']>=0.135) & (optuna_study_rf_meta_pd['precision_1']>0.15)],
    x='number', #ось абсцисс
    y=['roc_valid','roc_train','precision_1','recall_1'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость метрик качества модели от number', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики ROC Valid',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

<span style="color:Blue">

Вывод:  

Их всех выбираем точку $number = 226$.   
Не смотря на, то что в этой точке модель подает признаки переобучеености,  
в этой точке оптимальные значения метрик $recall_1$ и $precision_1$, а  
также относительно высокая метрика $ROC AUC$.   
Посмотрим каким значениям гиперпараметров соотвествует выбраннаяточка.

In [ ]:
# определим номер лучшге варианта
best_optuna_number = 226

# сформируем словарь лучших гипирпарметров RandomForestClassifier
best_param_rf = {
    'n_estimators' : int(optuna_study_rf_meta_pd[optuna_study_rf_meta_pd['number']==best_optuna_number]['params_n_estimators'].iloc[0]),
    'criterion': optuna_study_rf_meta_pd[optuna_study_rf_meta_pd['number']==best_optuna_number]['params_criterion'].iloc[0],
    'max_depth' : int(optuna_study_rf_meta_pd[optuna_study_rf_meta_pd['number']==best_optuna_number]['params_max_depth'].iloc[0]),
    'min_samples_split': int(optuna_study_rf_meta_pd[optuna_study_rf_meta_pd['number']==best_optuna_number]['params_min_samples_split'].iloc[0]),
    'min_samples_leaf' : int(optuna_study_rf_meta_pd[optuna_study_rf_meta_pd['number']==best_optuna_number]['params_min_samples_leaf'].iloc[0]),
    'max_features': optuna_study_rf_meta_pd[optuna_study_rf_meta_pd['number']==best_optuna_number]['params_max_features'].iloc[0],
    'max_samples' : optuna_study_rf_meta_pd[optuna_study_rf_meta_pd['number']==best_optuna_number]['params_max_samples'].iloc[0],
    'class_weight' : {0:optuna_study_rf_meta_pd[optuna_study_rf_meta_pd['number']==best_optuna_number]['params_class_0_weight'].iloc[0],
                      1:optuna_study_rf_meta_pd[optuna_study_rf_meta_pd['number']==best_optuna_number]['params_class_1_weight'].iloc[0]}
    }

# определим перменные для лучших значений параметров
best_k_features = optuna_study_rf_meta_pd[optuna_study_rf_meta_pd['number']==best_optuna_number]['params_k_features'].iloc[0]
best_random_state = optuna_study_rf_meta_pd[optuna_study_rf_meta_pd['number']==best_optuna_number]['params_random_state'].iloc[0]
best_class_1_percent = optuna_study_rf_meta_pd[optuna_study_rf_meta_pd['number']==best_optuna_number]['params_class_1_percent'].iloc[0]


# Выведем принятые наилучшие праметры
print('best n estimators:',best_param_rf['n_estimators'])
print('best criterion:',best_param_rf['criterion'])
print('best max depth:',best_param_rf['max_depth'])
print('best min samples split:',best_param_rf['min_samples_split'])
print('best min samples leaf:',best_param_rf['min_samples_leaf'])
print('best max features(%):',round(best_param_rf['max_features'],3))
print('best max samples(%):',round(best_param_rf['max_samples'],3))
print('best class 0 weight:',round(best_param_rf['class_weight'][0],3))
print('best class 1 weight:',round(best_param_rf['class_weight'][1],3))

print('best class 1 percent:',round(best_class_1_percent,3))
print('best k features:',best_k_features)
print('best random state:',best_random_state)
print('time for best train:',round(optuna_study_rf_meta_pd[optuna_study_rf_meta_pd['number']==best_optuna_number]['duration'].iloc[0].seconds/60),'minutes')

print()
print('ROC AUC на обучающем наборе:', round(optuna_study_rf_meta_pd[optuna_study_rf_meta_pd['number']==best_optuna_number]['roc_train'].iloc[0],3))
print('ROC AUC на валидационном наборе:', round(optuna_study_rf_meta_pd[optuna_study_rf_meta_pd['number']==best_optuna_number]['roc_valid'].iloc[0],3))
print('precision класса 1:', round(optuna_study_rf_meta_pd[optuna_study_rf_meta_pd['number']==best_optuna_number]['precision_1'].iloc[0],3))
print('recall класса 1:', round(optuna_study_rf_meta_pd[optuna_study_rf_meta_pd['number']==best_optuna_number]['recall_1'].iloc[0],3))

##### <span style="color:MediumSlateBlue">Обучение модели с лучшими параметрами</span>

In [ ]:
# Загружаем обучающие наборы
MX_train_pd = fp.ParquetFile('first_metamodel/features/features_first_meta_torow_train').to_pandas()
my_train_pd = pd.read_csv('first_metamodel/data/mfy_train.csv')

# поготовим данные my_train_pd к работе с функцией class_1_percent_samples
my_train_pd.set_index('id',drop=False,inplace=True)

# с помощью функции class_1_percent_samples зададим долю класса 1
list_c1_percent_id = class_1_percent_samples(my_train_pd,best_class_1_percent,random_state=best_random_state)[:1000000]

# подготовим данные для обучения
MX_train_balanced = MX_train_pd.loc[list_c1_percent_id]
my_train_balanced = my_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

# освободим память от "тяжелых" и ненужных файлов
del MX_train_pd, my_train_pd
gc.collect()

# с помощью класса SelectKBest получим список лучших признаков
selector = SelectKBest(f_classif, k=best_k_features)
selector.fit(MX_train_balanced, my_train_balanced)
list_best_features = selector.get_feature_names_out()

MX_train_balanced = MX_train_balanced[list_best_features].to_numpy()


# обучаем модель RandomForestClassifier с наилучшеми параметрами
random_forest = RandomForestClassifier(
        **best_param_rf,
        n_jobs=-1,
        random_state=best_random_state)
random_forest.fit(MX_train_balanced, my_train_balanced)

# удаляем крупные файлы чтобы высвободить память 
del MX_train_balanced, my_train_balanced
gc.collect()

# загружаем тестовые наборы
MX_train = fp.ParquetFile('first_metamodel/features/features_first_meta_torow_train').to_pandas()[list_best_features].to_numpy()
MX_valid = fp.ParquetFile('first_metamodel/features/features_first_meta_torow_valid').to_pandas()[list_best_features].to_numpy()
my_train = pd.read_csv('first_metamodel/data/mfy_train.csv')['flag'].to_numpy()
my_valid = pd.read_csv('first_metamodel/data/mfy_valid.csv')['flag'].to_numpy()

# для метрик ROC AUC делаем предсказание модели в виде вероятности
my_train_rf_pred_proba = random_forest.predict_proba(MX_train)[:,1]
my_valid_rf_pred_proba = random_forest.predict_proba(MX_valid)[:,1]

# Делаем предсказание для валидационной выборки
my_valid_rf_pred = random_forest.predict(MX_valid)

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(my_train, my_train_rf_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(my_valid, my_valid_rf_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(my_valid,my_valid_rf_pred,zero_division=0))

# time: 5m 30s

##### <span style="color:MediumSlateBlue">Анализ влияния порога классификации на качество модели</span>

In [ ]:
# чтобы проше обращаться к каждому обьекту в данных
# преобразуем y_valid_LR_pred к Series типу
my_valid_rf_pred_proba = pd.Series(my_valid_rf_pred_proba)

# формируем список из необходимых значений thresholds
thresholds = np.arange(0.01, 1, 0.01)

# сформируем списко для заполнения данными результатов анализа
threshold_data = []


for threshold in thresholds:
        # сформируем список для заполнения данными текушего threshold
        threshold_current_data = [threshold]

        #клиентов, для которых вероятность дефолта > threshold, относим к классу 1
        #в противном случае — к классу 0
        my_valid_rf_pred = my_valid_rf_pred_proba.apply(lambda x: 1 if x>threshold else 0)


        #Считаем метрики для класса 0 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(my_valid, my_valid_rf_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.precision_score(my_valid, my_valid_rf_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.f1_score(my_valid, my_valid_rf_pred,average=None,zero_division=0)[0],3))

        #Считаем метрики для класса 1 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(my_valid, my_valid_rf_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.precision_score(my_valid, my_valid_rf_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.f1_score(my_valid, my_valid_rf_pred,average=None,zero_division=0)[1],3))

        # добавляем полученные данные в список threshold_data
        threshold_data.append(threshold_current_data)

        # из полученных данных сформируем dataframe
        thresholds_pd =pd.DataFrame(
        columns=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'],
        data = threshold_data)

# time: 17s

In [ ]:
# Визуализируем метрики на валидационном наборе при различных threshold
fig_threshold = px.line(
    data_frame=thresholds_pd,
    x='threshold', #ось абсцисс
    y=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'], #ось ординат
)
fig_threshold.update_layout(
    title ={
        'text':'Зависимость качества модели от threshold', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Порог классификации threshold',
    yaxis_title='Величина метрики')
fig_threshold.update_xaxes(showspikes=True)
fig_threshold.update_yaxes(showspikes=True)

fig_threshold.show()

Метрика $precision$ показывает на сколько сильно ошиблась модель при определиние класса.  
Чем выше метрика, тем меньше ошиблась модель. 

$$precision = \frac{TP}{TP+FP}$$

Метрика $recall$ показывает способность модели найти класс.  
Чем выше метрика, тем выше способность модели находить класс.  

$$recall = \frac{TP}{TP+FN}$$

Метрика $f1-score$ показывает баланс между $precision$ и $recall$ 

$threshold$- порог класификации модели.  
Все объекты для которых $у_{\text{proba pred}} < threshold$, модель относит к класс 0.  
Соотвественно, если $у_{\text{proba pred}}  > threshold$ модель относит обьект к классу 1.  
Для идельаной модели $threshold$, является границей между областями класса 0 и класса 1.  

<span style="color:Blue">

Выводы по результам анализа:  

Для улучшения качества модели по метрике $ROC AUC$, нет смысла сдвигать порог   
классификации. Но зависимость метрик качества модели от $threshold$, может рассказать о  
способности модели искать класс 1 и класс 0.

При фокусировке модели на классе 1 (более низкий $threshold$):  
- на классе 1 метрика $precision$ уменьшается, метрика $recall$ растет;  
- на классе 0 метрика $precision$ растет, метрика $recall$ уменьшается.  

При фокусировке модели на классе 0 (более высокий $threshold$):  
- на классе 1 метрика $precision$ увеличивается, метрика $recall$ уменьшается;  
- на классе 0 метрика $precision$ уменьшается, метрика $recall$ растет.  

Обьсняется тем, что в условия дисбаланса модель научилась хорошо определять класс 0 и плохо класс 1.  

При фокусировке модели на классе 1 (более низкий $threshold$), все больше обьектов отнесены к классу 1.   
Так как $precision$ класса 1 уменьшается, значит в области класса 1 стало больше обектов класса 0,  
что явлется закономерным для "идеальной" модели.  
Однако при этом, $recall$ класса 1 растет, значит в добавленных обьектах также присутвует класс 1.  
Это как раз указывает на то, что модель не нучилась находить среди класса 0 класс 1. В области с     
высокой вероятностью класса 0 (более низкий $threshold$), находятся обьекты класса 1.  

Фокусировка на классе 0 (более высокий $threshold$), заставляет модель все больше обьектов из области класса 1  
относить к классу 0. При этом увеличение метрики $precision$ класса 1 говорит о том, что в области класса 1     
становится меньше 0, а значит модель умеет среди класса 1, находить класс 0.  
После  $threshold=0.92$ модель впринципе теряет способность отличать класс 1. У модели нет обьектов класса 1   
в которых она "уверена" на 95+%.

Обратим внимание на $threshold=0.51$, в этой точке находится баланс межу метриками качества модели:
$$precision_1 = recall_1 = \text{f1-score}_1 = 0.154$$
$$precision_0 = recall_0 = \text{f1-score}_0 = 0.968$$

Для модели с лучшим качеством эти две точки должны находится максимально близко к друг другу и иметь   
высокое значение метрик $precision_1$ и $recall_1$.  
Для класса 0 это условие выполнется, что также говорит о том, что модель научилась искать этот класс.    
Для класса 1 это условие не выполнется, что также говорит о том, что модель плохо ищет класс 1.

##### <span style="color:MediumSlateBlue">Построение ROC-кривой выбранной модели на тестовом наборе</span>

In [ ]:
# загружаем тестовые наборы
MX_train = fp.ParquetFile('first_metamodel/features/features_first_meta_torow_train').to_pandas()[list_best_features].to_numpy()
MX_valid = fp.ParquetFile('first_metamodel/features/features_first_meta_torow_valid').to_pandas()[list_best_features].to_numpy()
MX_test = fp.ParquetFile('first_metamodel/features/features_first_meta_torow_test').to_pandas()[list_best_features].to_numpy()
my_train = pd.read_csv('first_metamodel/data/mfy_train.csv')['flag'].to_numpy()
my_valid = pd.read_csv('first_metamodel/data/mfy_valid.csv')['flag'].to_numpy()
my_test = pd.read_csv('first_metamodel/data/mfy_test.csv')['flag'].to_numpy()

In [ ]:
print("Размеры обучающего набора",MX_train.shape,my_train.shape)
print("Размеры валидационного набора",MX_valid.shape,my_valid.shape)
print("Размеры тестового набора",MX_test.shape,my_test.shape)
# time: 1m 43s

In [ ]:
# для метрик ROC AUC делаем предсказание модели в виде вероятности
my_train_rf_pred_proba = random_forest.predict_proba(MX_train)[:,1]
my_valid_rf_pred_proba = random_forest.predict_proba(MX_valid)[:,1]
my_test_rf_pred_proba = random_forest.predict_proba(MX_test)[:,1]

# Делаем предсказание для валидационной выборки
my_test_lr_pred = random_forest.predict(MX_test)

del MX_train, MX_valid, MX_test

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(my_train, my_train_rf_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(my_valid, my_valid_rf_pred_proba),3))
print('ROC AUC на тестовом наборе', round(metrics.roc_auc_score(my_test, my_test_rf_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(my_test,my_test_lr_pred,zero_division=0))

In [ ]:
# Для расчета значений TPR и FPR используем функцию roc_curve
fpr_train, tpr_train, _ = metrics.roc_curve(my_train, my_train_rf_pred_proba)
fpr_valid, tpr_valid, _ = metrics.roc_curve(my_valid, my_valid_rf_pred_proba)
fpr_test, tpr_test, _ = metrics.roc_curve(my_test, my_test_rf_pred_proba)

# рассчитаем площадь под кривой
auc_train = round(metrics.roc_auc_score(my_train, my_train_rf_pred_proba),3)
auc_valid = round(metrics.roc_auc_score(my_valid, my_valid_rf_pred_proba),3)
auc_test = round(metrics.roc_auc_score(my_test, my_test_rf_pred_proba),3)

trace_train = go.Scatter(x=fpr_train, y=tpr_train, mode='lines', name='AUC_train = %0.2f' % auc_train,
                   line=dict(color='red', width=2),
                   hovertemplate='train<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_train])

trace_valid = go.Scatter(x=fpr_valid, y=tpr_valid, mode='lines', name='AUC_valid = %0.2f' % auc_valid,
                   line=dict(color='green', width=2),
                   hovertemplate='valid<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_valid])

trace_test = go.Scatter(x=fpr_test, y=tpr_test, mode='lines', name='AUC_test = %0.2f' % auc_test,
                   line=dict(color='darkorange', width=2),
                   hovertemplate='test<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_test])


reference_line = go.Scatter(x=[0,1], y=[0,1], mode='lines', name='Reference Line',
                            line=dict(color='navy', width=2, dash='dash'))

fig_roc = go.Figure(data=[trace_train,trace_valid,trace_test,reference_line])

fig_roc.update_layout(
    title ={
        'text':'ROC-кривая', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 1350,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate')

fig_roc.update_xaxes(showspikes=True)
fig_roc.update_yaxes(showspikes=True)


fig_roc.show()

<span style="color:Blue">

Вывод по ROC-кривой:
1. Не смотря на то, что модель показывает, очень хорошую $ROC$ кривую на обучающем наборе,  
модель, так же выдает сталибильное качество на валидационном и тестовом наборах, а значит  
не переобучена - разброс ($variance$) не большой.
2. На тестовом наборе полученно качество $ROC AUC = 0.75$.

#### <span style="color:MediumBlue">Построение модели Gradient Boosting Classifier</span>

##### <span style="color:MediumSlateBlue">Baseline обучение модели Hist Gradient Boosting Classifier</span>

In [ ]:
MX_train = fp.ParquetFile('first_metamodel/features/features_first_meta_torow_train').to_pandas()
my_train = pd.read_csv('first_metamodel/data/mfy_train.csv').set_index('id')['flag']
MX_valid = fp.ParquetFile('first_metamodel/features/features_first_meta_torow_valid').to_pandas()
my_valid = pd.read_csv('first_metamodel/data/mfy_valid.csv').set_index('id')['flag']

In [ ]:
# обучаем модель Gradient Boosting Classifier
gradient_boosting = HistGradientBoostingClassifier(random_state=42)
gradient_boosting.fit(MX_train,my_train)

# для метрик ROC AUC делаем предсказание модели в виде вероятности
my_train_rf__pred_proba = gradient_boosting.predict_proba(MX_train)[:,1]
my_valid_rf_pred_proba = gradient_boosting.predict_proba(MX_valid)[:,1]

# Делаем предсказание для валидационной выборки
my_valid_rf_pred = gradient_boosting.predict(MX_valid)

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(my_train, my_train_rf__pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(my_valid, my_valid_rf_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(my_valid,my_valid_rf_pred))

# time: 30s

<span style="color:Blue">

Выводы:

В сравнение с моделью $HistGradientBoostingClassifier$ построенной   
на несбалансированных данных $transform$ $data$ $stat$:
1. Качество модели по метрике $ROC AUC$ не изменилась.
2. Качество модели по метрике $recall_1$ значительно  
улучшилось. Правда, $recall_0$ уменьшилось.
3. Качество модели по метрике $precision_1$ уменьшилось.   
4. На порядок уменьшилось время обучения модели. Это логично,  
велична сбалансированной выборки всего $107000$ $samples$. 

##### <span style="color:MediumSlateBlue">Подбор гиперпараметров модели</span>

In [ ]:
# настроим оптимизацию гипер параметров
def optuna_gb_meta(trial):
  # задаем пространства поиска гиперпараметров
  learning_rate = trial.suggest_float('learning_rate',0.01,0.1)
  max_iter = trial.suggest_int('max_iter', 150, 250,step = 1)
  max_leaf_nodes = trial.suggest_int('max_leaf_nodes', 2, 60,step = 1)
  max_depth = trial.suggest_int('max_depth', 1, 10,step = 1)
  min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 60,step = 1)
  max_features = trial.suggest_float('max_features',0.1,1)
  l2_regularization = trial.suggest_float('l2_regularization',0.01,1)
  class_0_weight = trial.suggest_float('class_0_weight',0.01,1)
  class_1_weight = trial.suggest_float('class_1_weight',0.01,1)
  k_features = trial.suggest_int('k_features', 20, 506,step = 1)
  class_1_percent = trial.suggest_float('class_1_percent',0.01,1)
  random_state = trial.suggest_int('random_state', 1, 1000000)

  # создаем модель
  optuna_gradient_boosting_meta = HistGradientBoostingClassifier(
      learning_rate= learning_rate,
      max_iter = max_iter,
      max_leaf_nodes =max_leaf_nodes,
      max_depth = max_depth,
      min_samples_leaf = min_samples_leaf,
      max_features=max_features,
      l2_regularization = l2_regularization,
      class_weight={0:class_0_weight,1:class_1_weight},
      random_state=random_state)

  # Загружаем обучающие наборы
  MX_train_pd = fp.ParquetFile('first_metamodel/features/features_first_meta_torow_train').to_pandas()
  my_train_pd = pd.read_csv('first_metamodel/data/mfy_train.csv')

  # поготовим данные my_train_pd к работе с функцией class_1_percent_samples
  my_train_pd.set_index('id',drop=False,inplace=True)

  # с помощью функции class_1_percent_samples зададим долю класса 1
  list_c1_percent_id = class_1_percent_samples(my_train_pd,class_1_percent,random_state=random_state)[:1000000]

  # подготовим данные для обучения
  MX_train_balanced = MX_train_pd.loc[list_c1_percent_id]
  my_train_balanced = my_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

  # освободим память от "тяжелых" и ненужных файлов
  del MX_train_pd, my_train_pd
  gc.collect()
  
  # с помощью класса SelectKBest получим список лучших признаков
  selector = SelectKBest(f_classif, k=k_features)
  selector.fit(MX_train_balanced, my_train_balanced)
  list_best_features = selector.get_feature_names_out()

  MX_train_balanced = MX_train_balanced[list_best_features].to_numpy()

  # я не воспользовался параметром timeout метода optimize потому, что  
  # optimize отставливает поиск параметров, если время обучения 
  # превышает timeout. Мне нужно чтобы поиск параметров продолжился.
  try:
    # для модуля func_time_out упакуем обучение модели в функцию try_func
    def try_func():
            return optuna_gradient_boosting_meta.fit(MX_train_balanced,my_train_balanced)
    # обучаем модель с ограничением по времени 10 минут (600 секунд)
    optuna_gradient_boosting_meta = func_timeout.func_timeout(600, try_func)

    # удаляем крупные файлы чтобы высвободить память 
    del MX_train_balanced, my_train_balanced
    gc.collect()

    # загружаем тестовые наборы
    MX_train = fp.ParquetFile('first_metamodel/features/features_first_meta_torow_train').to_pandas()[list_best_features].to_numpy()
    MX_valid = fp.ParquetFile('first_metamodel/features/features_first_meta_torow_valid').to_pandas()[list_best_features].to_numpy()
    my_train = pd.read_csv('first_metamodel/data/mfy_train.csv')['flag'].to_numpy()
    my_valid = pd.read_csv('first_metamodel/data/mfy_valid.csv')['flag'].to_numpy()

    # делаем предсказание на обучающем и валидационном наборе
    #Считаем метрики для класса 1 добавляем их в список
    roc_train = metrics.roc_auc_score(my_train, optuna_gradient_boosting_meta.predict_proba(MX_train)[:,1])
    roc_valid = metrics.roc_auc_score(my_valid, optuna_gradient_boosting_meta.predict_proba(MX_valid)[:,1])
    f1_score = metrics.f1_score(my_valid, optuna_gradient_boosting_meta.predict(MX_valid))
    recall_1 = metrics.recall_score(my_valid, optuna_gradient_boosting_meta.predict(MX_valid),average=None,zero_division=0)[1]
    precision_1 = metrics.precision_score(my_valid, optuna_gradient_boosting_meta.predict(MX_valid),average=None,zero_division=0)[1]

    # удаляем крупные файлы чтобы высвободить память 
    del MX_train, MX_valid, my_train, my_valid
    gc.collect()

  except func_timeout.FunctionTimedOut:
    # удаляем крупные файлы чтобы высвободить память 
    del MX_train_balanced, my_train_balanced
    gc.collect()
    
    # фиксируем пустые значения метрик
    roc_train = 0
    roc_valid = 0
    f1_score = 0
    recall_1 = 0
    precision_1 = 0
    pass

  return roc_train, roc_valid, f1_score, recall_1, precision_1

In [ ]:
# cоздаем объект исследования
# чтобы модель не переобучалась минимизируем roc_train 
# и максимизируем roc_valid
optuna_study_gb_meta = optuna.create_study(study_name="HistGradientBoostingClassifier_meta_first_torow", 
                               directions=['maximize','maximize','maximize','maximize','maximize'], 
                               sampler=optuna.samplers.TPESampler(),
                               pruner='Hyperband',
                               storage='sqlite:///optuna_studies.db',
                               load_if_exists=True)

# ну случай удаления обучения
# optuna.delete_study(study_name="HistGradientBoostingClassifier_meta_first_torow", storage='sqlite:///optuna_studies.db')

In [ ]:
# ищем лучшую комбинацию гиперпараметров n_trials раз
optuna_study_gb_meta.optimize(optuna_gb_meta, n_trials=300)

##### <span style="color:MediumSlateBlue">Анализ гиперпараметров</span>

In [ ]:
# из полученного результат соврмируем Data Frame
optuna_study_gb_meta_pd = optuna_study_gb_meta.trials_dataframe()
# переименуем столбы
optuna_study_gb_meta_pd.rename(columns={
    'values_0': 'roc_train',
    'values_1': 'roc_valid',
    'values_2': 'f1_score',
    'values_3': 'recall_1',
    'values_4': 'precision_1'
},inplace=True)
optuna_study_gb_meta_pd.tail(5)

In [ ]:
# покаждем статистику обучения
print('Максимальное значение метрики ROC AUC на валидационном наборе:',round(optuna_study_gb_meta_pd['roc_valid'].max(),3))
print('Среднее значение метрики ROC AUC на валидационном наборе:',round(optuna_study_gb_meta_pd['roc_valid'].mean(),3))
print('Максимальное значение метрики f1_score на валидационном наборе:',round(optuna_study_gb_meta_pd['f1_score'].max(),3))
print('Среднее значение метрики f1_score на валидационном наборе:',round(optuna_study_gb_meta_pd['f1_score'].mean(),3))
print('Максимальное значение метрики recall_1 на валидационном наборе:',round(optuna_study_gb_meta_pd['recall_1'].max(),3))
print('Среднее значение метрики recall_1 на валидационном наборе:',round(optuna_study_gb_meta_pd['recall_1'].mean(),3))
print('Максимальное значение метрики precision_1 на валидационном наборе:',round(optuna_study_gb_meta_pd['precision_1'].max(),3))
print('Среднее значение метрики precision_1 на валидационном наборе:',round(optuna_study_gb_meta_pd['precision_1'].mean(),3))

In [ ]:
# построим график важности гиперпарметров
optuna.visualization.plot_param_importances(optuna_study_gb_meta, target_name='Score')

In [ ]:
# Построим зависимость метрики precision_1 от гиперпараметров
fig = px.scatter(
    data_frame=optuna_study_gb_meta_pd,
    x='precision_1', #ось абсцисс
    y=['roc_train', 'roc_valid', 'recall_1','f1_score'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision класса 1 от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision класса 1',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрик качества модели от number

fig = px.scatter(
    data_frame=optuna_study_gb_meta_pd[(optuna_study_gb_meta_pd['recall_1']>0.14)&(optuna_study_gb_meta_pd['precision_1']>0.14)],
    x='number', #ось абсцисс
    y=['roc_train','roc_valid','recall_1','precision_1'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision_1',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

<span style="color:Blue">

Вывод:  

Их всех выбираем точку $number =61$.   
В этой точке оптимальные значения метрик $recall_1$ и $precision_1$, а  
также относительно высокая метрика $ROC AUC$.   
Посмотрим каким значениям гиперпараметров соотвествует выбранная точка.

In [ ]:
# определим номер лучшге варианта
best_optuna_number = 61

# сформируем словарь лучших гипирпарметров HistGradientBoostingClassifier
best_param_hgbc = {
    'learning_rate' : optuna_study_gb_meta_pd[optuna_study_gb_meta_pd['number']==best_optuna_number]['params_learning_rate'].iloc[0],
    'max_iter' : optuna_study_gb_meta_pd[optuna_study_gb_meta_pd['number']==best_optuna_number]['params_max_iter'].iloc[0],
    'max_leaf_nodes' : optuna_study_gb_meta_pd[optuna_study_gb_meta_pd['number']==best_optuna_number]['params_max_leaf_nodes'].iloc[0],
    'max_depth' : optuna_study_gb_meta_pd[optuna_study_gb_meta_pd['number']==best_optuna_number]['params_max_depth'].iloc[0],
    'min_samples_leaf' : optuna_study_gb_meta_pd[optuna_study_gb_meta_pd['number']==best_optuna_number]['params_min_samples_leaf'].iloc[0],
    'max_features' : optuna_study_gb_meta_pd[optuna_study_gb_meta_pd['number']==best_optuna_number]['params_max_features'].iloc[0],
    'l2_regularization' : optuna_study_gb_meta_pd[optuna_study_gb_meta_pd['number']==best_optuna_number]['params_l2_regularization'].iloc[0],
    'class_weight' : {0: optuna_study_gb_meta_pd[optuna_study_gb_meta_pd['number']==best_optuna_number]['params_class_0_weight'].iloc[0],
                      1: optuna_study_gb_meta_pd[optuna_study_gb_meta_pd['number']==best_optuna_number]['params_class_1_weight'].iloc[0],}
    }

# определим перменные для лучших значений параметров
best_k_features = optuna_study_gb_meta_pd[optuna_study_gb_meta_pd['number']==best_optuna_number]['params_k_features'].iloc[0]
best_random_state = optuna_study_gb_meta_pd[optuna_study_gb_meta_pd['number']==best_optuna_number]['params_random_state'].iloc[0]
best_class_1_percent = optuna_study_gb_meta_pd[optuna_study_gb_meta_pd['number']==best_optuna_number]['params_class_1_percent'].iloc[0]


# Выведем принятые наилучшие параметры
print('best learning rate:',round(best_param_hgbc['learning_rate'],3))
print('best max iter:',best_param_hgbc['max_iter'])
print('best max leaf nodes:',best_param_hgbc['max_leaf_nodes'])
print('best max depth:',best_param_hgbc['max_depth'])
print('best min samples leaf:',best_param_hgbc['min_samples_leaf'])
print('best max features:',round(best_param_hgbc['max_features'],3))
print('best l2 regularization:',round(best_param_hgbc['l2_regularization'],3))
print('best class 0 weight:',round(best_param_hgbc['class_weight'][0],3))
print('best class 1 weight:',round(best_param_hgbc['class_weight'][1],3))

print('best class 1 percent:',round(best_class_1_percent,3))
print('best k features:',best_k_features)
print('best random state:',best_random_state)
print('time for best train:',round(optuna_study_gb_meta_pd[optuna_study_gb_meta_pd['number']==best_optuna_number]['duration'].iloc[0].seconds/60),'minutes')

print()
print('ROC AUC на обучающем наборе:', round(optuna_study_gb_meta_pd[optuna_study_gb_meta_pd['number']==best_optuna_number]['roc_train'].iloc[0],3))
print('ROC AUC на валидационном наборе:', round(optuna_study_gb_meta_pd[optuna_study_gb_meta_pd['number']==best_optuna_number]['roc_valid'].iloc[0],3))
print('precision класса 1:', round(optuna_study_gb_meta_pd[optuna_study_gb_meta_pd['number']==best_optuna_number]['precision_1'].iloc[0],3))
print('recall класса 1:', round(optuna_study_gb_meta_pd[optuna_study_gb_meta_pd['number']==best_optuna_number]['recall_1'].iloc[0],3))

##### <span style="color:MediumSlateBlue">Обучение модели с лучшими параметрами</span>

In [ ]:
# Загружаем обучающие наборы
MX_train_pd = fp.ParquetFile('first_metamodel/features/features_first_meta_torow_train').to_pandas()
my_train_pd = pd.read_csv('first_metamodel/data/mfy_train.csv')

# поготовим данные my_train_pd к работе с функцией class_1_percent_samples
my_train_pd.set_index('id',drop=False,inplace=True)

# с помощью функции class_1_percent_samples зададим долю класса 1
list_c1_percent_id = class_1_percent_samples(my_train_pd,best_class_1_percent,random_state=best_random_state)[:1000000]

# подготовим данные для обучения
MX_train_balanced = MX_train_pd.loc[list_c1_percent_id]
my_train_balanced = my_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

# освободим память от "тяжелых" и ненужных файлов
del MX_train_pd, my_train_pd
gc.collect()

# с помощью класса SelectKBest получим список лучших признаков
selector = SelectKBest(f_classif, k=best_k_features)
selector.fit(MX_train_balanced, my_train_balanced)
list_best_features = selector.get_feature_names_out()

MX_train_balanced = MX_train_balanced[list_best_features].to_numpy()

# обучаем модель HistGradientBoostingClassifier с наилучшеми параметрами
gradient_boosting = HistGradientBoostingClassifier(
        **best_param_hgbc,
        random_state=best_random_state)
gradient_boosting.fit(MX_train_balanced,my_train_balanced)

# удаляем крупные файлы чтобы высвободить память 
del MX_train_balanced, my_train_balanced
gc.collect()

# загружаем тестовые наборы
MX_train = fp.ParquetFile('first_metamodel/features/features_first_meta_torow_train').to_pandas()[list_best_features].to_numpy()
MX_valid = fp.ParquetFile('first_metamodel/features/features_first_meta_torow_valid').to_pandas()[list_best_features].to_numpy()
my_train = pd.read_csv('first_metamodel/data/mfy_train.csv')['flag'].to_numpy()
my_valid = pd.read_csv('first_metamodel/data/mfy_valid.csv')['flag'].to_numpy()

# для метрик ROC AUC делаем предсказание модели в виде вероятности
my_train_gb_pred_proba = gradient_boosting.predict_proba(MX_train)[:,1]
my_valid_gb_pred_proba = gradient_boosting.predict_proba(MX_valid)[:,1]

# Делаем предсказание для валидационной выборки
my_valid_gb_pred = gradient_boosting.predict(MX_valid)

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(my_train, my_train_gb_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(my_valid, my_valid_gb_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(my_valid,my_valid_gb_pred,zero_division=0))

# time: 1m 40s

##### <span style="color:MediumSlateBlue">Анализ влияния порога классификации на качество модели</span>

In [ ]:
# чтобы проше обращаться к каждому обьекту в данных
# преобразуем y_valid_LR_pred к Series типу
my_valid_gb_pred_proba = pd.Series(my_valid_gb_pred_proba)

# формируем список из необходимых значений thresholds
thresholds = np.arange(0.01, 1, 0.01)

# сформируем списко для заполнения данными результатов анализа
threshold_data = []


for threshold in thresholds:
        # сформируем список для заполнения данными текушего threshold
        threshold_current_data = [threshold]

        #клиентов, для которых вероятность дефолта > threshold, относим к классу 1
        #в противном случае — к классу 0
        my_valid_gb_pred = my_valid_gb_pred_proba.apply(lambda x: 1 if x>threshold else 0)


        #Считаем метрики для класса 0 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(my_valid, my_valid_gb_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.precision_score(my_valid, my_valid_gb_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.f1_score(my_valid, my_valid_gb_pred,average=None,zero_division=0)[0],3))

        #Считаем метрики для класса 1 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(my_valid, my_valid_gb_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.precision_score(my_valid, my_valid_gb_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.f1_score(my_valid, my_valid_gb_pred,average=None,zero_division=0)[1],3))

        # добавляем полученные данные в список threshold_data
        threshold_data.append(threshold_current_data)

        # из полученных данных сформируем dataframe
        thresholds_pd =pd.DataFrame(
        columns=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'],
        data = threshold_data)

# time: 17s

In [ ]:
# Визуализируем метрики на валидационном наборе при различных threshold
fig_threshold = px.line(
    data_frame=thresholds_pd,
    x='threshold', #ось абсцисс
    y=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'], #ось ординат
)
fig_threshold.update_layout(
    title ={
        'text':'Зависимость качества модели от threshold', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Порог классификации threshold',
    yaxis_title='Величина метрики')
fig_threshold.update_xaxes(showspikes=True)
fig_threshold.update_yaxes(showspikes=True)

fig_threshold.show()

Метрика $precision$ показывает на сколько сильно ошиблась модель при определиние класса.  
Чем выше метрика, тем меньше ошиблась модель. 

$$precision = \frac{TP}{TP+FP}$$

Метрика $recall$ показывает способность модели найти класс.  
Чем выше метрика, тем выше способность модели находить класс.  

$$recall = \frac{TP}{TP+FN}$$

Метрика $f1-score$ показывает баланс между $precision$ и $recall$ 

$threshold$- порог класификации модели.  
Все объекты для которых $у_{\text{proba pred}} < threshold$, модель относит к класс 0.  
Соотвественно, если $у_{\text{proba pred}}  > threshold$ модель относит обьект к классу 1.  
Для идельаной модели $threshold$, является границей между областями класса 0 и класса 1.  

<span style="color:Blue">

Выводы по результам анализа:  

Для улучшения качества модели по метрике $ROC AUC$, нет смысла сдвигать порог   
классификации. Но зависимость метрик качества модели от $threshold$, может рассказать о  
способности модели искать класс 1 и класс 0.

При фокусировке модели на классе 1 (более низкий $threshold$):  
- на классе 1 метрика $precision$ уменьшается, метрика $recall$ растет;  
- на классе 0 метрика $precision$ растет, метрика $recall$ уменьшается.  

При фокусировке модели на классе 0 (более высокий $threshold$):  
- на классе 1 метрика $precision$ увеличивается, метрика $recall$ уменьшается;  
- на классе 0 метрика $precision$ уменьшается, метрика $recall$ растет.  

Обьсняется тем, что в условия дисбаланса модель научилась хорошо определять класс 0 и плохо класс 1.  

При фокусировке модели на классе 1 (более низкий $threshold$), все больше обьектов отнесены к классу 1.   
Так как $precision$ класса 1 уменьшается, значит в области класса 1 стало больше обектов класса 0,  
что явлется закономерным для "идеальной" модели.  
Однако при этом, $recall$ класса 1 растет, значит в добавленных обьектах также присутвует класс 1.  
Это как раз указывает на то, что модель не нучилась находить среди класса 0 класс 1. В области с     
высокой вероятностью класса 0 (более низкий $threshold$), находятся обьекты класса 1.  

Фокусировка на классе 0 (более высокий $threshold$), заставляет модель все больше обьектов из области класса 1  
относить к классу 0. При этом увеличение метрики $precision$ класса 1 говорит о том, что в области класса 1     
становится меньше 0, а значит модель умеет среди класса 1, находить класс 0.  
После  $threshold=0.88$ модель впринципе теряет способность отличать класс 1. У модели нет обьектов класса 1   
в которых она "уверена" на 90+%.


Обратим внимание на $threshold=0.52$, в этой точке находится баланс межу метриками качества модели:
$$precision_1 = recall_1 = \text{f1-score}_1 = 0.153$$
$$precision_0 = recall_0 = \text{f1-score}_0 = 0.97$$

Для модели с лучшим качеством эти две точки должны находится максимально близко к друг другу и иметь   
высокое значение метрик $precision_1$ и $recall_1$.  
Для класса 0 это условие выполнется, что также говорит о том, что модель научилась искать этот класс.    
Для класса 1 это условие не выполнется, что также говорит о том, что модель плохо ищет класс 1.

##### <span style="color:MediumSlateBlue">Построение ROC-кривой выбранной модели на тестовом наборе</span>

In [ ]:
# загружаем тестовые наборы
MX_train = fp.ParquetFile('first_metamodel/features/features_first_meta_torow_train').to_pandas()[list_best_features].to_numpy()
MX_valid = fp.ParquetFile('first_metamodel/features/features_first_meta_torow_valid').to_pandas()[list_best_features].to_numpy()
MX_test = fp.ParquetFile('first_metamodel/features/features_first_meta_torow_test').to_pandas()[list_best_features].to_numpy()
my_train = pd.read_csv('first_metamodel/data/mfy_train.csv')['flag'].to_numpy()
my_valid = pd.read_csv('first_metamodel/data/mfy_valid.csv')['flag'].to_numpy()
my_test = pd.read_csv('first_metamodel/data/mfy_test.csv')['flag'].to_numpy()

In [ ]:
print("Размеры обучающего набора",MX_train.shape,my_train.shape)
print("Размеры валидационного набора",MX_valid.shape,my_valid.shape)
print("Размеры тестового набора",MX_test.shape,my_test.shape)
# time: 1m 43s

In [ ]:
# для метрик ROC AUC делаем предсказание модели в виде вероятности
my_train_gb_pred_proba = gradient_boosting.predict_proba(MX_train)[:,1]
my_valid_gb_pred_proba = gradient_boosting.predict_proba(MX_valid)[:,1]
my_test_gb_pred_proba = gradient_boosting.predict_proba(MX_test)[:,1]

# Делаем предсказание для валидационной выборки
my_test_gb_pred = gradient_boosting.predict(MX_test)

del MX_train, MX_valid, MX_test
gc.collect()

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(my_train, my_train_gb_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(my_valid, my_valid_gb_pred_proba),3))
print('ROC AUC на тестовом наборе', round(metrics.roc_auc_score(my_test, my_test_gb_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(my_test,my_test_gb_pred,zero_division=0))

In [ ]:
# Для расчета значений TPR и FPR используем функцию roc_curve
fpr_train, tpr_train, _ = metrics.roc_curve(my_train, my_train_gb_pred_proba)
fpr_valid, tpr_valid, _ = metrics.roc_curve(my_valid, my_valid_gb_pred_proba)
fpr_test, tpr_test, _ = metrics.roc_curve(my_test, my_test_gb_pred_proba)

# рассчитаем площадь под кривой
auc_train = round(metrics.roc_auc_score(my_train, my_train_gb_pred_proba),3)
auc_valid = round(metrics.roc_auc_score(my_valid, my_valid_gb_pred_proba),3)
auc_test = round(metrics.roc_auc_score(my_test, my_test_gb_pred_proba),3)

trace_train = go.Scatter(x=fpr_train, y=tpr_train, mode='lines', name='AUC_train = %0.2f' % auc_train,
                   line=dict(color='red', width=2),
                   hovertemplate='train<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_train])

trace_valid = go.Scatter(x=fpr_valid, y=tpr_valid, mode='lines', name='AUC_valid = %0.2f' % auc_valid,
                   line=dict(color='green', width=2),
                   hovertemplate='valid<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_valid])

trace_test = go.Scatter(x=fpr_test, y=tpr_test, mode='lines', name='AUC_test = %0.2f' % auc_test,
                   line=dict(color='darkorange', width=2),
                   hovertemplate='test<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_test])


reference_line = go.Scatter(x=[0,1], y=[0,1], mode='lines', name='Reference Line',
                            line=dict(color='navy', width=2, dash='dash'))

fig_roc = go.Figure(data=[trace_train,trace_valid,trace_test,reference_line])

fig_roc.update_layout(
    title ={
        'text':'ROC-кривая', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 1350,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate')

fig_roc.update_xaxes(showspikes=True)
fig_roc.update_yaxes(showspikes=True)


fig_roc.show()

<span style="color:Blue">

Вывод по ROC-кривой:
1. Модель выдает сталибильное качество на тренировочном, валидационном и тестовом наборах, а значит  
не переобучена - разброс ($variance$) не большой.
2. На тестовом наборе полученно качество $ROC AUC = 0.75$.

### <span style="color:RoyalBlue">Построение блендинга из базовых моделей обученных на transform data stat данных</span>

#### <span style="color:MediumBlue">Построение метамодели Logistic Regression</span>

##### <span style="color:MediumSlateBlue">Baseline обучение метамодели LogisticRegression</span>

In [ ]:
MX_train = fp.ParquetFile('first_metamodel/features/features_first_meta_stat_train').to_pandas()
my_train = pd.read_csv('first_metamodel/data/mfy_train.csv').set_index('id')['flag']
MX_valid = fp.ParquetFile('first_metamodel/features/features_first_meta_stat_valid').to_pandas()
my_valid = pd.read_csv('first_metamodel/data/mfy_valid.csv').set_index('id')['flag']

In [ ]:
# обучаем модель логистической регрессии
logistic_regression = linear_model.LogisticRegression(random_state=42, max_iter=10000)
logistic_regression.fit(MX_train,my_train)

# для метрик ROC AUC делаем предсказание модели в виде вероятности
my_train_LR_pred_proba = logistic_regression.predict_proba(MX_train)[:,1]
my_valid_LR_pred_proba = logistic_regression.predict_proba(MX_valid)[:,1]

# Делаем предсказание для валидационной выборки
my_valid_LR_pred = logistic_regression.predict(MX_valid)

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(my_train, my_train_LR_pred_proba),3))
print('ROC AUC на тестовом наборе', round(metrics.roc_auc_score(my_valid, my_valid_LR_pred_proba),3))

print('Основные метрики на тестовом наборе:')
print(metrics.classification_report(my_valid,my_valid_LR_pred,zero_division=0))

# time: 3m 5s

##### <span style="color:MediumSlateBlue">Подбор гиперпараметров модели</span>

In [ ]:
# настроим оптимизацию гипер параметров
def optuna_lg_meta(trial):
  # задаем пространства поиска гиперпараметров
  C = trial.suggest_float('C',0.01,1)
  class_0_weight = trial.suggest_float('class_0_weight',0.01,1)
  class_1_weight = trial.suggest_float('class_1_weight',0.01,1)
  k_features = trial.suggest_int('k_features', 20, 507,step = 1)
  class_1_percent = trial.suggest_float('class_1_percent',0.1,1)
  random_state = trial.suggest_int('random_state', 1, 1000000)


  # создаем модель
  optuna_log_reg_meta = linear_model.LogisticRegression(
      C=C,
      class_weight={0:class_0_weight,1:class_1_weight},
      random_state=random_state,
      max_iter=10000)

  # Загружаем обучающие наборы
  MX_train_pd = fp.ParquetFile('first_metamodel/features/features_first_meta_stat_train').to_pandas()
  my_train_pd = pd.read_csv('first_metamodel/data/mfy_train.csv')

  # поготовим данные my_train_pd к работе с функцией class_1_percent_samples
  my_train_pd.set_index('id',drop=False,inplace=True)

  # с помощью функции class_1_percent_samples зададим долю класса 1
  list_c1_percent_id = class_1_percent_samples(my_train_pd,class_1_percent,random_state=random_state)[:1000000]

  # подготовим данные для обучения
  MX_train_balanced = MX_train_pd.loc[list_c1_percent_id]
  my_train_balanced = my_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

  # освободим память от "тяжелых" и ненужных файлов
  del MX_train_pd, my_train_pd
  gc.collect()
  
  # с помощью класса SelectKBest получим список лучших признаков
  selector = SelectKBest(f_classif, k=k_features)
  selector.fit(MX_train_balanced, my_train_balanced)
  list_best_features = selector.get_feature_names_out()

  MX_train_balanced = MX_train_balanced[list_best_features].to_numpy()

  # я не воспользовался параметром timeout метода optimize потому, что  
  # optimize отставливает поиск параметров, если время обучения 
  # превышает timeout. Мне нужно чтобы поиск параметров продолжился.
  try:
    # для модуля func_time_out упакуем обучение модели в функцию try_func
    def try_func():
            return optuna_log_reg_meta.fit(MX_train_balanced, my_train_balanced)
    # обучаем модель с ограничением по времени 10 минут (600 секунд)
    optuna_log_reg_meta = func_timeout.func_timeout(600, try_func)

    # удаляем крупные файлы чтобы высвободить память 
    del MX_train_balanced, my_train_balanced
    gc.collect()

    # загружаем тестовые наборы
    MX_train = fp.ParquetFile('first_metamodel/features/features_first_meta_stat_train').to_pandas()[list_best_features].to_numpy()
    MX_valid = fp.ParquetFile('first_metamodel/features/features_first_meta_stat_valid').to_pandas()[list_best_features].to_numpy()
    my_train = pd.read_csv('first_metamodel/data/mfy_train.csv')['flag'].to_numpy()
    my_valid = pd.read_csv('first_metamodel/data/mfy_valid.csv')['flag'].to_numpy()
    
    # делаем предсказание на обучающем и валидационном наборе и считаем метрики
    roc_train = metrics.roc_auc_score(my_train, optuna_log_reg_meta.predict_proba(MX_train)[:,1])
    roc_valid = metrics.roc_auc_score(my_valid, optuna_log_reg_meta.predict_proba(MX_valid)[:,1])
    f1_score = metrics.f1_score(my_valid, optuna_log_reg_meta.predict(MX_valid))
    recall_1 = metrics.recall_score(my_valid, optuna_log_reg_meta.predict(MX_valid),average=None,zero_division=0)[1]
    precision_1 = metrics.precision_score(my_valid, optuna_log_reg_meta.predict(MX_valid),average=None,zero_division=0)[1]

    # удаляем крупные файлы чтобы высвободить память 
    del MX_train, MX_valid, my_train, my_valid
    gc.collect()

  except func_timeout.FunctionTimedOut:
    # удаляем крупные файлы чтобы высвободить память 
    del MX_train_balanced, my_train_balanced
    gc.collect()
    
    # фиксируем пустые значения метрик
    roc_train = 0
    roc_valid = 0
    f1_score = 0
    recall_1 = 0
    precision_1 = 0
    pass

  return roc_train, roc_valid, f1_score, recall_1, precision_1

In [ ]:
# cоздаем объект исследования
# чтобы модель не переобучалась минимизируем roc_train 
# и максимизируем roc_valid
optuna_study_lg_meta = optuna.create_study(study_name="LogisticRegression_meta_first_stat", 
                               directions=['maximize','maximize','maximize','maximize','maximize'], 
                               sampler=optuna.samplers.TPESampler(),
                               pruner='Hyperband',
                               storage='sqlite:///optuna_studies.db',
                               load_if_exists=True)
# optuna.delete_study(study_name="LogisticRegression_meta_first_stat", storage='sqlite:///optuna_studies.db')

In [ ]:
# ищем лучшую комбинацию гиперпараметров n_trials раз
optuna_study_lg_meta.optimize(optuna_lg_meta, n_trials=300)

##### <span style="color:MediumSlateBlue">Анализ гиперпараметров</span>

In [ ]:
# из полученного результат соврмируем Data Frame
optuna_study_lg_pd = optuna_study_lg_meta.trials_dataframe().dropna()
# переименуем столбы
optuna_study_lg_pd.rename(columns={
    'values_0': 'roc_train',
    'values_1': 'roc_valid',
    'values_2': 'f1_score',
    'values_3': 'recall_1',
    'values_4': 'precision_1'
},inplace=True)
# Выделим время потраченное на обучение модели
optuna_study_lg_pd.tail()

In [ ]:
# покаждем статистику обучения
print('Максимальное значение метрики ROC AUC на валидационном наборе:',round(optuna_study_lg_pd['roc_valid'].max(),3))
print('Среднее значение метрики ROC AUC на валидационном наборе:',round(optuna_study_lg_pd['roc_valid'].mean(),3))
print('Максимальное значение метрики recall_1 на валидационном наборе:',round(optuna_study_lg_pd['recall_1'].max(),3))
print('Среднее значение метрики recall_1 на валидационном наборе:',round(optuna_study_lg_pd['recall_1'].mean(),3))
print('Максимальное значение метрики precision_1 на валидационном наборе:',round(optuna_study_lg_pd['precision_1'].max(),3))
print('Среднее значение метрики precision_1 на валидационном наборе:',round(optuna_study_lg_pd['precision_1'].mean(),3))

In [ ]:
# построим график важности гиперпарметров
optuna.visualization.plot_param_importances(optuna_study_lg_meta, target_name='Score')

In [ ]:
optuna_study_lg_pd

In [ ]:
# Построим зависимость метрики precision_1 от гипер параметров
fig = px.scatter(
    data_frame=optuna_study_lg_pd,
    x='precision_1', #ось абсцисс
    y=['roc_train', 'roc_valid', 'recall_1','f1_score'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision_1',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Визуализируем метрики полученные при optuna оптимизации
fig = px.scatter(
    data_frame=optuna_study_lg_pd[(optuna_study_lg_pd['precision_1']>0.2*optuna_study_lg_pd['precision_1'].max())&
                                  (optuna_study_lg_pd['recall_1']>0.2*optuna_study_lg_pd['precision_1'].max())],
    x='number', #ось абсцисс
    y=['roc_train','roc_valid','recall_1','precision_1'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость качества модели от trials optuna', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='trials optuna',
    yaxis_title='Величина метрики')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

<span style="color:Blue">

Вывод:  

Их всех выбираем точку $number =165$.   
В этой точке самые оптимальные значения $recall$ и $precision$.  
Посмотрим каким значениям гиперпараметров соотвествует выбранная точка.

In [ ]:
# определим номер лучшге варианта
best_optuna_number = 165

# сформируем словарь лучших гипирпарметров RandomForestClassifier
best_param_lr = {
    'C' : round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_C'].iloc[0],3),
    'class_weight' : {0:round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_class_0_weight'].iloc[0],3),
                      1:round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_class_1_weight'].iloc[0],3)},
    }

# создадим перменные
best_class_1_percent = optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_class_1_percent'].iloc[0]
best_k_features = optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_k_features'].iloc[0]
best_random_state = optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['params_random_state'].iloc[0]
# Выведем принятые наилучшие праметры
print('best C:',best_param_lr['C'])
print('best class 0 weight:',best_param_lr['class_weight'][0])
print('best class 1 weight:',best_param_lr['class_weight'][1])
print('best class 1 percent:',round(best_class_1_percent,3))
print('best k features:',best_k_features)
print('best random state:',best_random_state)
print('time for best train:',round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['duration'].iloc[0].seconds/60),'minutes')

print()
print('ROC AUC на обучающем наборе:', round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['roc_train'].iloc[0],3))
print('ROC AUC на валидационном наборе:', round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['roc_valid'].iloc[0],3))
print('precision класса 1:', round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['precision_1'].iloc[0],3))
print('recall класса 1:', round(optuna_study_lg_pd[optuna_study_lg_pd['number']==best_optuna_number]['recall_1'].iloc[0],3))


##### <span style="color:MediumSlateBlue">Обучение модели с лучшими параметрами</span>

In [ ]:
# Загружаем обучающие наборы
MX_train_pd = fp.ParquetFile('first_metamodel/features/features_first_meta_stat_train').to_pandas()
my_train_pd = pd.read_csv('first_metamodel/data/mfy_train.csv')

# поготовим данные my_train_pd к работе с функцией class_1_percent_samples
my_train_pd.set_index('id',drop=False,inplace=True)

# с помощью функции class_1_percent_samples зададим долю класса 1
list_c1_percent_id = class_1_percent_samples(my_train_pd,best_class_1_percent,random_state=best_random_state)[:1000000]

# подготовим данные для обучения
MX_train_balanced = MX_train_pd.loc[list_c1_percent_id]
my_train_balanced = my_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

# освободим память от "тяжелых" и ненужных файлов
del MX_train_pd, my_train_pd
gc.collect()

# с помощью класса SelectKBest получим список лучших признаков
selector = SelectKBest(f_classif, k=best_k_features)
selector.fit(MX_train_balanced, my_train_balanced)
list_best_features = selector.get_feature_names_out()

MX_train_balanced = MX_train_balanced[list_best_features].to_numpy()


# обучаем модель LogisticRegression с наилучшеми параметрами
logistic_regression = linear_model.LogisticRegression(
        **best_param_lr,
        random_state=42,
        max_iter=10000)
logistic_regression.fit(MX_train_balanced, my_train_balanced)

# удаляем крупные файлы чтобы высвободить память 
del MX_train_balanced, my_train_balanced
gc.collect()

# загружаем тестовые наборы
MX_train = fp.ParquetFile('first_metamodel/features/features_first_meta_stat_train').to_pandas()[list_best_features].to_numpy()
MX_valid = fp.ParquetFile('first_metamodel/features/features_first_meta_stat_valid').to_pandas()[list_best_features].to_numpy()
my_train = pd.read_csv('first_metamodel/data/mfy_train.csv')['flag'].to_numpy()
my_valid = pd.read_csv('first_metamodel/data/mfy_valid.csv')['flag'].to_numpy()

# для метрик ROC AUC делаем предсказание модели в виде вероятности
my_train_lr__pred_proba = logistic_regression.predict_proba(MX_train)[:,1]
my_valid_lr_pred_proba = logistic_regression.predict_proba(MX_valid)[:,1]

# Делаем предсказание для валидационной выборки
my_valid_lr_pred = logistic_regression.predict(MX_valid)

# удаляем крупные файлы чтобы высвободить память 
del MX_train, MX_valid
gc.collect()

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(my_train, my_train_lr__pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(my_valid, my_valid_lr_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(my_valid,my_valid_lr_pred,zero_division=0))

# time: 2m 30s

##### <span style="color:MediumSlateBlue">Анализ влияния порога классификации на качество модели</span>

In [ ]:
# чтобы проше обращаться к каждому обьекту в данных
# преобразуем my_valid_LR_pred к Series типу
my_valid_lr_pred_proba = pd.Series(my_valid_lr_pred_proba)

# формируем список из необходимых значений thresholds
thresholds = np.arange(0.01, 1, 0.01)

# сформируем списко для заполнения данными результатов анализа
threshold_data = []


for threshold in thresholds:
        # сформируем список для заполнения данными текушего threshold
        threshold_current_data = [threshold]

        #клиентов, для которых вероятность дефолта > threshold, относим к классу 1
        #в противном случае — к классу 0
        my_valid_lr_pred = my_valid_lr_pred_proba.apply(lambda x: 1 if x>threshold else 0)


        #Считаем метрики для класса 0 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(my_valid, my_valid_lr_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.precision_score(my_valid, my_valid_lr_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.f1_score(my_valid, my_valid_lr_pred,average=None,zero_division=0)[0],3))

        #Считаем метрики для класса 1 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(my_valid, my_valid_lr_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.precision_score(my_valid, my_valid_lr_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.f1_score(my_valid, my_valid_lr_pred,average=None,zero_division=0)[1],3))

        # добавляем полученные данные в список threshold_data
        threshold_data.append(threshold_current_data)

        # из полученных данных сформируем dataframe
        thresholds_pd =pd.DataFrame(
        columns=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'],
        data = threshold_data)

# time: 17s

In [ ]:
# Визуализируем метрики на валидационном наборе при различных threshold
fig_threshold = px.line(
    data_frame=thresholds_pd,
    x='threshold', #ось абсцисс
    y=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'], #ось ординат
)
fig_threshold.update_layout(
    title ={
        'text':'Зависимость качества модели от threshold', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Порог классификации threshold',
    yaxis_title='Величина метрики')
fig_threshold.update_xaxes(showspikes=True)
fig_threshold.update_yaxes(showspikes=True)

fig_threshold.show()

Метрика $precision$ показывает на сколько сильно ошиблась модель при определиние класса.  
Чем выше метрика, тем меньше ошиблась модель. 

$$precision = \frac{TP}{TP+FP}$$

Метрика $recall$ показывает способность модели найти класс.  
Чем выше метрика, тем выше способность модели находить класс.  

$$recall = \frac{TP}{TP+FN}$$

Метрика $f1-score$ показывает баланс между $precision$ и $recall$ 

$threshold$- порог класификации модели.  
Все объекты для которых $у_{\text{proba pred}} < threshold$, модель относит к класс 0.  
Соотвественно, если $у_{\text{proba pred}}  > threshold$ модель относит обьект к классу 1.  
Для идельаной модели $threshold$, является границей между областями класса 0 и класса 1.  

<span style="color:Blue">

Выводы по результам анализа:  

Для улучшения качества модели по метрике $ROC AUC$, нет смысла сдвигать порог   
классификации. Но зависимость метрик качества модели от $threshold$, может рассказать о  
спосбоности модели искать класс 1 и класс 0.

При фокусировке модели на классе 1 (более низкий $threshold$):  
- на классе 1 метрика $precision$ уменьшается, метрика $recall$ растет;  
- на классе 0 метрика $precision$ растет, метрика $recall$ уменьшается.  

При фокусировке модели на классе 0 (более высокий $threshold$):  
- на классе 1 метрика $precision$ увеличивается, метрика $recall$ уменьшается;  
- на классе 0 метрика $precision$ уменьшается, метрика $recall$ растет.  

Обьсняется тем, что в условия дисбаланса модель научилась хорошо определять класс 0 и плохо класс 1.  

При фокусировке модели на классе 1 (более низкий $threshold$), все больше обьектов отнесены к классу 1.   
Так как $precision$ класса 1 уменьшается, значит в области класса 1 стало больше обектов класса 0,  
что явлется закономерным для "идеальной" модели.  
Однако при этом, $recall$ класса 1 растет, значит в добавленных обьектах также присутвует класс 1.  
Это как раз указывает на то, что модель не научилась находить среди класса 0 класс 1. В области с     
высокой вероятностью класса 0 (более низкий $threshold$), находятся обьекты класса 1.  

Фокусировка на классе 0 (более высокий $threshold$), заставляет модель все больше обьектов из области класса 1  
относить к классу 0. При этом увеличение метрики $precision$ класса 1 говорит о том, что в области класса 1     
становится меньше 0, а значит модель умеет среди класса 1, находить класс 0.  
После  $threshold=0.92$ модель впринципе теряет способность отличать класс 1. У модели нет обьектов класса 1   
в которых она "уверена" на 92+%.

Обратим внимание на $threshold=0.52$, в этой точке находится баланс межу метриками качества модели:
$$precision_1 = recall_1 = \text{f1-score}_1 = 0.145$$
$$precision_0 = recall_0 = \text{f1-score}_0 = 0.968$$

Для модели с лучшим качеством эти две точки должны находится максимально близко к друг другу и иметь  
высокое значение метрик $precision_1$ и $recall_1$.
Для класса 0 это условие выполнется, что также говорит о том, что модель научилась искать этот класс.  
Для класса 1 это условие не выполнется, что также говорит о том, что модель плохо ищет класс 1.

##### <span style="color:MediumSlateBlue">Построение ROC-кривой выбранной модели на тестовом наборе</span>

In [ ]:
# сформируем данные для обучения и проверики модели
# для подбора гиперпараметров модели ограничем размер обучающей выборки
MX_train = fp.ParquetFile('first_metamodel/features/features_first_meta_stat_train').to_pandas()[list_best_features].to_numpy()
my_train = pd.read_csv('first_metamodel/data/mfy_train.csv')['flag'].to_numpy()
MX_valid = fp.ParquetFile('first_metamodel/features/features_first_meta_stat_valid').to_pandas()[list_best_features].to_numpy()
my_valid = pd.read_csv('first_metamodel/data/mfy_valid.csv')['flag'].to_numpy()
MX_test = fp.ParquetFile('first_metamodel/features/features_first_meta_stat_test').to_pandas()[list_best_features].to_numpy()
my_test = pd.read_csv('first_metamodel/data/mfy_test.csv')['flag'].to_numpy()

In [ ]:
print("Размеры обучающего набора",MX_train.shape)
print("Размеры валидационного набора",MX_valid.shape)
print("Размеры тестового набора",MX_test.shape)
# time: 1m 43s

In [ ]:
# для метрик ROC AUC делаем предсказание модели в виде вероятности
my_train_lr_pred_proba = logistic_regression.predict_proba(MX_train)[:,1]
my_valid_lr_pred_proba = logistic_regression.predict_proba(MX_valid)[:,1]
my_test_lr_pred_proba = logistic_regression.predict_proba(MX_test)[:,1]

# Делаем предсказание для тестовой выборки
my_test_lr_pred = logistic_regression.predict(MX_test)

del MX_train, MX_valid, MX_test
gc.collect()

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(my_train, my_train_lr__pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(my_valid, my_valid_lr_pred_proba),3))
print('ROC AUC на тестовом наборе', round(metrics.roc_auc_score(my_test, my_test_lr_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(my_test,my_test_lr_pred,zero_division=0))

In [ ]:
# Для расчета значений TPR и FPR используем функцию roc_curve
fpr_train, tpr_train, _ = metrics.roc_curve(my_train, my_train_lr_pred_proba)
fpr_valid, tpr_valid, _ = metrics.roc_curve(my_valid, my_valid_lr_pred_proba)
fpr_test, tpr_test, _ = metrics.roc_curve(my_test, my_test_lr_pred_proba)

# рассчитаем площадь под кривой
auc_train = round(metrics.roc_auc_score(my_train, my_train_lr_pred_proba),3)
auc_valid = round(metrics.roc_auc_score(my_valid, my_valid_lr_pred_proba),3)
auc_test = round(metrics.roc_auc_score(my_test, my_test_lr_pred_proba),3)

trace_train = go.Scatter(x=fpr_train, y=tpr_train, mode='lines', name='AUC_train = %0.2f' % auc_train,
                   line=dict(color='red', width=2),
                   hovertemplate='train<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_train])

trace_valid = go.Scatter(x=fpr_valid, y=tpr_valid, mode='lines', name='AUC_valid = %0.2f' % auc_valid,
                   line=dict(color='green', width=2),
                   hovertemplate='valid<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_valid])

trace_test = go.Scatter(x=fpr_test, y=tpr_test, mode='lines', name='AUC_test = %0.2f' % auc_test,
                   line=dict(color='darkorange', width=2),
                   hovertemplate='test<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_test])


reference_line = go.Scatter(x=[0,1], y=[0,1], mode='lines', name='Reference Line',
                            line=dict(color='navy', width=2, dash='dash'))

fig_roc = go.Figure(data=[trace_train,trace_valid,trace_test,reference_line])

fig_roc.update_layout(
    title ={
        'text':'ROC-кривая', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 1350,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate')

fig_roc.update_xaxes(showspikes=True)
fig_roc.update_yaxes(showspikes=True)


fig_roc.show()

<span style="color:Blue">

Вывод по ROC-кривой:
1. Модель выдает сталибильное качество на всех наборах, а значит  
не переобучена - разброс ($variance$) не большой.  
2. На тестовом наборе полученно качество $ROC AUC = 0.74$.

#### <span style="color:MediumBlue">Построение метамодели Random Forest Classifier</span>

##### <span style="color:MediumSlateBlue">Baseline обучение метамодели RandomForestClassifier</span>

In [ ]:
MX_train = fp.ParquetFile('first_metamodel/features/features_first_meta_stat_train').to_pandas()
my_train = pd.read_csv('first_metamodel/data/mfy_train.csv').set_index('id')['flag']
MX_valid = fp.ParquetFile('first_metamodel/features/features_first_meta_stat_valid').to_pandas()
my_valid = pd.read_csv('first_metamodel/data/mfy_valid.csv').set_index('id')['flag']

In [ ]:
# обучаем модель логистической регрессии
# чтобы модель пыталась найти связь во всех признаках
random_forest = RandomForestClassifier(random_state=42, n_jobs=-1)
random_forest.fit(MX_train,my_train)
# для метрик ROC AUC делаем предсказание модели в виде вероятности
my_train_rf__pred_proba = random_forest.predict_proba(MX_train)[:,1]
my_valid_rf_pred_proba = random_forest.predict_proba(MX_valid)[:,1]

# Делаем предсказание для валидационной выборки
my_valid_rf_pred = random_forest.predict(MX_valid)

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(my_train, my_train_rf__pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(my_valid, my_valid_rf_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(my_valid,my_valid_rf_pred))

# time: 17s

<span style="color:Blue">

Выводы:

В сравнение с моделью $RandomForestClassifier$ построенной   
на несбалансированных данных $transform$ $data$ $stat$:
1. Качество модели по метрике $ROC AUC$ значительно улучшилось.
2. Качество модели по метрике $recall_1$ значительно  
улучшилось. Правда, $recall_0$ уменьшилось.
3. Качество модели по метрике $precision_1$ уменьшилось.   
4. На порядок уменьшилось время обучения модели. Это логично,  
велична сбалансированной выборки всего $107000$ $samples$. 

##### <span style="color:MediumSlateBlue">Подбор гиперпараметров модели</span>

In [ ]:
# настроим оптимизацию гипер параметров
def optuna_rf_meta(trial):
  # задаем пространства поиска гиперпараметров
  # задаем пространства поиска гиперпараметров
  n_estimators = trial.suggest_int('n_estimators', 40, 100,step = 5)
  criterion = trial.suggest_categorical('criterion',['gini', 'entropy', 'log_loss'])
  max_depth = trial.suggest_int('max_depth', 10, 30,step = 1)
  min_samples_split = trial.suggest_int('min_samples_split', 5, 15,step = 1)
  min_samples_leaf = trial.suggest_int('min_samples_leaf', 5, 15,step = 1)
  max_features = trial.suggest_float('max_features',0.4,0.8)
  max_samples = trial.suggest_float('max_samples',0.4,0.8)
  class_0_weight = trial.suggest_float('class_0_weight',0.1,1)
  class_1_weight = trial.suggest_float('class_1_weight',0.1,1)
  class_1_percent = trial.suggest_float('class_1_percent',0.01,1)
  k_features = trial.suggest_int('k_features', 20, 507,step = 1)
  random_state = trial.suggest_int('random_state', 1, 1000000)


  # создаем модель
  optuna_random_forest_meta = RandomForestClassifier(
      n_estimators = n_estimators, 
      criterion = criterion,
      max_depth = max_depth,
      min_samples_split = min_samples_split,  
      min_samples_leaf = min_samples_leaf,
      max_features=max_features,
      max_samples=max_samples,
      class_weight={0:class_0_weight,1:class_1_weight},
      n_jobs=-1,
      random_state=random_state)

  # Загружаем обучающие наборы
  MX_train_pd = fp.ParquetFile('first_metamodel/features/features_first_meta_stat_train').to_pandas()
  my_train_pd = pd.read_csv('first_metamodel/data/mfy_train.csv')

  # поготовим данные my_train_pd к работе с функцией class_1_percent_samples
  my_train_pd.set_index('id',drop=False,inplace=True)

  # с помощью функции class_1_percent_samples зададим долю класса 1
  list_c1_percent_id = class_1_percent_samples(my_train_pd,class_1_percent,random_state=random_state)[:1000000]

  # подготовим данные для обучения
  MX_train_balanced = MX_train_pd.loc[list_c1_percent_id]
  my_train_balanced = my_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

  # освободим память от "тяжелых" и ненужных файлов
  del MX_train_pd, my_train_pd
  gc.collect()
  
  # с помощью класса SelectKBest получим список лучших признаков
  selector = SelectKBest(f_classif, k=k_features)
  selector.fit(MX_train_balanced, my_train_balanced)
  list_best_features = selector.get_feature_names_out()

  MX_train_balanced = MX_train_balanced[list_best_features].to_numpy()
  
  # я не воспользовался параметром timeout метода optimize потому, что  
  # optimize отставливает поиск параметров, если время обучения 
  # превышает timeout. Мне нужно чтобы поиск параметров продолжился.
  try:
    # для модуля func_time_out упакуем обучение модели в функцию try_func
    def try_func():
            return optuna_random_forest_meta.fit(MX_train_balanced, my_train_balanced)
    # обучаем модель с ограничением по времени 10 минут (600 секунд)
    optuna_random_forest_meta = func_timeout.func_timeout(600, try_func)

    # удаляем крупные файлы чтобы высвободить память 
    del MX_train_balanced, my_train_balanced
    gc.collect()

    # загружаем тестовые наборы
    MX_train = fp.ParquetFile('first_metamodel/features/features_first_meta_stat_train').to_pandas()[list_best_features].to_numpy()
    MX_valid = fp.ParquetFile('first_metamodel/features/features_first_meta_stat_valid').to_pandas()[list_best_features].to_numpy()
    my_train = pd.read_csv('first_metamodel/data/mfy_train.csv')['flag'].to_numpy()
    my_valid = pd.read_csv('first_metamodel/data/mfy_valid.csv')['flag'].to_numpy()

    
    # делаем предсказание на обучающем и валидационном наборе
    #Считаем метрики для класса 1 добавляем их в список
    roc_train = metrics.roc_auc_score(my_train, optuna_random_forest_meta.predict_proba(MX_train)[:,1])
    roc_valid = metrics.roc_auc_score(my_valid, optuna_random_forest_meta.predict_proba(MX_valid)[:,1])
    f1_score = metrics.f1_score(my_valid, optuna_random_forest_meta.predict(MX_valid))
    recall_1 = metrics.recall_score(my_valid, optuna_random_forest_meta.predict(MX_valid),average=None,zero_division=0)[1]
    precision_1 = metrics.precision_score(my_valid, optuna_random_forest_meta.predict(MX_valid),average=None,zero_division=0)[1]

    # удаляем крупные файлы чтобы высвободить память 
    del MX_train, MX_valid, my_train, my_valid
    gc.collect()

  except func_timeout.FunctionTimedOut:
    # удаляем крупные файлы чтобы высвободить память 
    del MX_train_balanced, my_train_balanced
    gc.collect()
    
    # фиксируем пустые значения метрик
    roc_train = 0
    roc_valid = 0
    f1_score = 0
    recall_1 = 0
    precision_1 = 0
    pass

  return roc_train, roc_valid, f1_score, recall_1, precision_1

In [ ]:
# cоздаем объект исследования
optuna_study_rf_meta = optuna.create_study(study_name="RandomForestClassifier_meta_first_stat", 
                               directions=['maximize','maximize','maximize','maximize','maximize'], 
                               sampler=optuna.samplers.TPESampler(),
                               pruner='Hyperband',
                               storage='sqlite:///optuna_studies.db',
                               load_if_exists=True)
# на случай удаления 
# optuna.delete_study(study_name="RandomForestClassifier_meta_first_stat", storage='sqlite:///optuna_studies.db')

In [ ]:
# ищем лучшую комбинацию гиперпараметров n_trials раз
optuna_study_rf_meta.optimize(optuna_rf_meta, n_trials=100)

##### <span style="color:MediumSlateBlue">Анализ гиперпараметров</span>

In [ ]:
# из полученного результат соврмируем Data Frame
optuna_study_rf_meta_pd = optuna_study_rf_meta.trials_dataframe().dropna()
# переименуем столбы
optuna_study_rf_meta_pd.rename(columns={
    'values_0': 'roc_train',
    'values_1': 'roc_valid',
    'values_2': 'f1_score',
    'values_3': 'recall_1',
    'values_4': 'precision_1'
},inplace=True)
optuna_study_rf_meta_pd.tail()

In [ ]:
# покаждем статистику обучения
print('Максимальное значение метрики ROC AUC на валидационном наборе:',round(optuna_study_rf_meta_pd['roc_valid'].max(),3))
print('Среднее значение метрики ROC AUC на валидационном наборе:',round(optuna_study_rf_meta_pd['roc_valid'].mean(),3))
print('Максимальное значение метрики f1_score на валидационном наборе:',round(optuna_study_rf_meta_pd['f1_score'].max(),3))
print('Среднее значение метрики f1_score на валидационном наборе:',round(optuna_study_rf_meta_pd['f1_score'].mean(),3))
print('Максимальное значение метрики recall_1 на валидационном наборе:',round(optuna_study_rf_meta_pd['recall_1'].max(),3))
print('Среднее значение метрики recall_1 на валидационном наборе:',round(optuna_study_rf_meta_pd['recall_1'].mean(),3))
print('Максимальное значение метрики precision_1 на валидационном наборе:',round(optuna_study_rf_meta_pd['precision_1'].max(),3))
print('Среднее значение метрики precision_1 на валидационном наборе:',round(optuna_study_rf_meta_pd['precision_1'].mean(),3))

In [ ]:
# Построим зависимость метрики precision_1 от гиперпараметров

fig = px.scatter(
    data_frame=optuna_study_rf_meta_pd,
    x='precision_1', #ось абсцисс
    y=['roc_train', 'roc_valid', 'recall_1','f1_score'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision класса 1 от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision на классе 1',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрик качества модели от number

fig = px.scatter(
    data_frame=optuna_study_rf_meta_pd[(optuna_study_rf_meta_pd['recall_1']>=0.135) & (optuna_study_rf_meta_pd['precision_1']>0.135)],
    x='number', #ось абсцисс
    y=['roc_valid','roc_train','precision_1','recall_1'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость метрик качества модели от number', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики ROC Valid',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

<span style="color:Blue">

Вывод:  

Их всех выбираем точку $number =268$.   
Не смотря на, то что в этой точке модель подает признаки переобучеености,  
в этой точке оптимальные значения метрик $recall_1$ и $precision_1$, а  
также относительно высокая метрика $ROC AUC$.   
Посмотрим каким значениям гиперпараметров соотвествует выбраннаяточка.

In [ ]:
# определим номер лучшге варианта
best_optuna_number = 268

# сформируем словарь лучших гипирпарметров RandomForestClassifier
best_param_rf = {
    'n_estimators' : int(optuna_study_rf_meta_pd[optuna_study_rf_meta_pd['number']==best_optuna_number]['params_n_estimators'].iloc[0]),
    'criterion': optuna_study_rf_meta_pd[optuna_study_rf_meta_pd['number']==best_optuna_number]['params_criterion'].iloc[0],
    'max_depth' : int(optuna_study_rf_meta_pd[optuna_study_rf_meta_pd['number']==best_optuna_number]['params_max_depth'].iloc[0]),
    'min_samples_split': int(optuna_study_rf_meta_pd[optuna_study_rf_meta_pd['number']==best_optuna_number]['params_min_samples_split'].iloc[0]),
    'min_samples_leaf' : int(optuna_study_rf_meta_pd[optuna_study_rf_meta_pd['number']==best_optuna_number]['params_min_samples_leaf'].iloc[0]),
    'max_features': optuna_study_rf_meta_pd[optuna_study_rf_meta_pd['number']==best_optuna_number]['params_max_features'].iloc[0],
    'max_samples' : optuna_study_rf_meta_pd[optuna_study_rf_meta_pd['number']==best_optuna_number]['params_max_samples'].iloc[0],
    'class_weight' : {0:optuna_study_rf_meta_pd[optuna_study_rf_meta_pd['number']==best_optuna_number]['params_class_0_weight'].iloc[0],
                      1:optuna_study_rf_meta_pd[optuna_study_rf_meta_pd['number']==best_optuna_number]['params_class_1_weight'].iloc[0]}
    }

# определим перменные для лучших значений параметров
best_k_features = optuna_study_rf_meta_pd[optuna_study_rf_meta_pd['number']==best_optuna_number]['params_k_features'].iloc[0]
best_random_state = optuna_study_rf_meta_pd[optuna_study_rf_meta_pd['number']==best_optuna_number]['params_random_state'].iloc[0]
best_class_1_percent = optuna_study_rf_meta_pd[optuna_study_rf_meta_pd['number']==best_optuna_number]['params_class_1_percent'].iloc[0]


# Выведем принятые наилучшие праметры
print('best n estimators:',best_param_rf['n_estimators'])
print('best criterion:',best_param_rf['criterion'])
print('best max depth:',best_param_rf['max_depth'])
print('best min samples split:',best_param_rf['min_samples_split'])
print('best min samples leaf:',best_param_rf['min_samples_leaf'])
print('best max features(%):',round(best_param_rf['max_features'],3))
print('best max samples(%):',round(best_param_rf['max_samples'],3))
print('best class 0 weight:',round(best_param_rf['class_weight'][0],3))
print('best class 1 weight:',round(best_param_rf['class_weight'][1],3))

print('best class 1 percent:',round(best_class_1_percent,3))
print('best k features:',best_k_features)
print('best random state:',best_random_state)
print('time for best train:',round(optuna_study_rf_meta_pd[optuna_study_rf_meta_pd['number']==best_optuna_number]['duration'].iloc[0].seconds/60),'minutes')

print()
print('ROC AUC на обучающем наборе:', round(optuna_study_rf_meta_pd[optuna_study_rf_meta_pd['number']==best_optuna_number]['roc_train'].iloc[0],3))
print('ROC AUC на валидационном наборе:', round(optuna_study_rf_meta_pd[optuna_study_rf_meta_pd['number']==best_optuna_number]['roc_valid'].iloc[0],3))
print('precision класса 1:', round(optuna_study_rf_meta_pd[optuna_study_rf_meta_pd['number']==best_optuna_number]['precision_1'].iloc[0],3))
print('recall класса 1:', round(optuna_study_rf_meta_pd[optuna_study_rf_meta_pd['number']==best_optuna_number]['recall_1'].iloc[0],3))

##### <span style="color:MediumSlateBlue">Обучение модели с лучшими параметрами</span>

In [ ]:
# Загружаем обучающие наборы
MX_train_pd = fp.ParquetFile('first_metamodel/features/features_first_meta_stat_train').to_pandas()
my_train_pd = pd.read_csv('first_metamodel/data/mfy_train.csv')

# поготовим данные my_train_pd к работе с функцией class_1_percent_samples
my_train_pd.set_index('id',drop=False,inplace=True)

# с помощью функции class_1_percent_samples зададим долю класса 1
list_c1_percent_id = class_1_percent_samples(my_train_pd,best_class_1_percent,random_state=best_random_state)[:1000000]

# подготовим данные для обучения
MX_train_balanced = MX_train_pd.loc[list_c1_percent_id]
my_train_balanced = my_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

# освободим память от "тяжелых" и ненужных файлов
del MX_train_pd, my_train_pd
gc.collect()

# с помощью класса SelectKBest получим список лучших признаков
selector = SelectKBest(f_classif, k=best_k_features)
selector.fit(MX_train_balanced, my_train_balanced)
list_best_features = selector.get_feature_names_out()

MX_train_balanced = MX_train_balanced[list_best_features].to_numpy()


# обучаем модель RandomForestClassifier с наилучшеми параметрами
random_forest = RandomForestClassifier(
        **best_param_rf,
        n_jobs=-1,
        random_state=42)
random_forest.fit(MX_train_balanced, my_train_balanced)

# удаляем крупные файлы чтобы высвободить память 
del MX_train_balanced, my_train_balanced
gc.collect()

# загружаем тестовые наборы
MX_train = fp.ParquetFile('first_metamodel/features/features_first_meta_stat_train').to_pandas()[list_best_features].to_numpy()
MX_valid = fp.ParquetFile('first_metamodel/features/features_first_meta_stat_valid').to_pandas()[list_best_features].to_numpy()
my_train = pd.read_csv('first_metamodel/data/mfy_train.csv')['flag'].to_numpy()
my_valid = pd.read_csv('first_metamodel/data/mfy_valid.csv')['flag'].to_numpy()

# для метрик ROC AUC делаем предсказание модели в виде вероятности
my_train_rf_pred_proba = random_forest.predict_proba(MX_train)[:,1]
my_valid_rf_pred_proba = random_forest.predict_proba(MX_valid)[:,1]

# Делаем предсказание для валидационной выборки
my_valid_rf_pred = random_forest.predict(MX_valid)

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(my_train, my_train_rf_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(my_valid, my_valid_rf_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(my_valid,my_valid_rf_pred,zero_division=0))

# time: 5m 30s

##### <span style="color:MediumSlateBlue">Анализ влияния порога классификации на качество модели</span>

In [ ]:
# чтобы проше обращаться к каждому обьекту в данных
# преобразуем y_valid_LR_pred к Series типу
my_valid_rf_pred_proba = pd.Series(my_valid_rf_pred_proba)

# формируем список из необходимых значений thresholds
thresholds = np.arange(0.01, 1, 0.01)

# сформируем списко для заполнения данными результатов анализа
threshold_data = []


for threshold in thresholds:
        # сформируем список для заполнения данными текушего threshold
        threshold_current_data = [threshold]

        #клиентов, для которых вероятность дефолта > threshold, относим к классу 1
        #в противном случае — к классу 0
        my_valid_rf_pred = my_valid_rf_pred_proba.apply(lambda x: 1 if x>threshold else 0)


        #Считаем метрики для класса 0 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(my_valid, my_valid_rf_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.precision_score(my_valid, my_valid_rf_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.f1_score(my_valid, my_valid_rf_pred,average=None,zero_division=0)[0],3))

        #Считаем метрики для класса 1 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(my_valid, my_valid_rf_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.precision_score(my_valid, my_valid_rf_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.f1_score(my_valid, my_valid_rf_pred,average=None,zero_division=0)[1],3))

        # добавляем полученные данные в список threshold_data
        threshold_data.append(threshold_current_data)

        # из полученных данных сформируем dataframe
        thresholds_pd =pd.DataFrame(
        columns=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'],
        data = threshold_data)

# time: 17s

In [ ]:
# Визуализируем метрики на валидационном наборе при различных threshold
fig_threshold = px.line(
    data_frame=thresholds_pd,
    x='threshold', #ось абсцисс
    y=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'], #ось ординат
)
fig_threshold.update_layout(
    title ={
        'text':'Зависимость качества модели от threshold', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Порог классификации threshold',
    yaxis_title='Величина метрики')
fig_threshold.update_xaxes(showspikes=True)
fig_threshold.update_yaxes(showspikes=True)

fig_threshold.show()

Метрика $precision$ показывает на сколько сильно ошиблась модель при определиние класса.  
Чем выше метрика, тем меньше ошиблась модель. 

$$precision = \frac{TP}{TP+FP}$$

Метрика $recall$ показывает способность модели найти класс.  
Чем выше метрика, тем выше способность модели находить класс.  

$$recall = \frac{TP}{TP+FN}$$

Метрика $f1-score$ показывает баланс между $precision$ и $recall$ 

$threshold$- порог класификации модели.  
Все объекты для которых $у_{\text{proba pred}} < threshold$, модель относит к класс 0.  
Соотвественно, если $у_{\text{proba pred}}  > threshold$ модель относит обьект к классу 1.  
Для идельаной модели $threshold$, является границей между областями класса 0 и класса 1.  

<span style="color:Blue">

Выводы по результам анализа:  

Для улучшения качества модели по метрике $ROC AUC$, нет смысла сдвигать порог   
классификации. Но зависимость метрик качества модели от $threshold$, может рассказать о  
способности модели искать класс 1 и класс 0.

При фокусировке модели на классе 1 (более низкий $threshold$):  
- на классе 1 метрика $precision$ уменьшается, метрика $recall$ растет;  
- на классе 0 метрика $precision$ растет, метрика $recall$ уменьшается.  

При фокусировке модели на классе 0 (более высокий $threshold$):  
- на классе 1 метрика $precision$ увеличивается, метрика $recall$ уменьшается;  
- на классе 0 метрика $precision$ уменьшается, метрика $recall$ растет.  

Обьсняется тем, что в условия дисбаланса модель научилась хорошо определять класс 0 и плохо класс 1.  

При фокусировке модели на классе 1 (более низкий $threshold$), все больше обьектов отнесены к классу 1.   
Так как $precision$ класса 1 уменьшается, значит в области класса 1 стало больше обектов класса 0,  
что явлется закономерным для "идеальной" модели.  
Однако при этом, $recall$ класса 1 растет, значит в добавленных обьектах также присутвует класс 1.  
Это как раз указывает на то, что модель не нучилась находить среди класса 0 класс 1. В области с     
высокой вероятностью класса 0 (более низкий $threshold$), находятся обьекты класса 1.  

Фокусировка на классе 0 (более высокий $threshold$), заставляет модель все больше обьектов из области класса 1  
относить к классу 0. При этом увеличение метрики $precision$ класса 1 говорит о том, что в области класса 1     
становится меньше 0, а значит модель умеет среди класса 1, находить класс 0.  
После  $threshold=0.92$ модель впринципе теряет способность отличать класс 1. У модели нет обьектов класса 1   
в которых она "уверена" на 95+%.

Обратим внимание на $threshold=0.51$, в этой точке находится баланс межу метриками качества модели:
$$precision_1 = recall_1 = \text{f1-score}_1 = 0.147$$
$$precision_0 = recall_0 = \text{f1-score}_0 = 0.968$$

Для модели с лучшим качеством эти две точки должны находится максимально близко к друг другу и иметь   
высокое значение метрик $precision_1$ и $recall_1$.  
Для класса 0 это условие выполнется, что также говорит о том, что модель научилась искать этот класс.    
Для класса 1 это условие не выполнется, что также говорит о том, что модель плохо ищет класс 1.

##### <span style="color:MediumSlateBlue">Построение ROC-кривой выбранной модели на тестовом наборе</span>

In [ ]:
# сформируем данные для обучения и проверики модели
# для подбора гиперпараметров модели ограничем размер обучающей выборки
MX_train = fp.ParquetFile('first_metamodel/features/features_first_meta_stat_train').to_pandas()[list_best_features].to_numpy()
my_train = pd.read_csv('first_metamodel/data/mfy_train.csv')['flag'].to_numpy()
MX_valid = fp.ParquetFile('first_metamodel/features/features_first_meta_stat_valid').to_pandas()[list_best_features].to_numpy()
my_valid = pd.read_csv('first_metamodel/data/mfy_valid.csv')['flag'].to_numpy()
MX_test = fp.ParquetFile('first_metamodel/features/features_first_meta_stat_test').to_pandas()[list_best_features].to_numpy()
my_test = pd.read_csv('first_metamodel/data/mfy_test.csv')['flag'].to_numpy()

In [ ]:
print("Размеры обучающего набора",MX_train.shape,my_train.shape)
print("Размеры валидационного набора",MX_valid.shape,my_valid.shape)
print("Размеры тестового набора",MX_test.shape,my_test.shape)
# time: 1m 43s

In [ ]:
# для метрик ROC AUC делаем предсказание модели в виде вероятности
my_train_rf_pred_proba = random_forest.predict_proba(MX_train)[:,1]
my_valid_rf_pred_proba = random_forest.predict_proba(MX_valid)[:,1]
my_test_rf_pred_proba = random_forest.predict_proba(MX_test)[:,1]

# Делаем предсказание для валидационной выборки
my_test_lr_pred = random_forest.predict(MX_test)

del MX_train, MX_valid, MX_test
gc.collect()

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(my_train, my_train_rf_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(my_valid, my_valid_rf_pred_proba),3))
print('ROC AUC на тестовом наборе', round(metrics.roc_auc_score(my_test, my_test_rf_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(my_test,my_test_lr_pred,zero_division=0))

In [ ]:
# Для расчета значений TPR и FPR используем функцию roc_curve
fpr_train, tpr_train, _ = metrics.roc_curve(my_train, my_train_rf_pred_proba)
fpr_valid, tpr_valid, _ = metrics.roc_curve(my_valid, my_valid_rf_pred_proba)
fpr_test, tpr_test, _ = metrics.roc_curve(my_test, my_test_rf_pred_proba)

# рассчитаем площадь под кривой
auc_train = round(metrics.roc_auc_score(my_train, my_train_rf_pred_proba),3)
auc_valid = round(metrics.roc_auc_score(my_valid, my_valid_rf_pred_proba),3)
auc_test = round(metrics.roc_auc_score(my_test, my_test_rf_pred_proba),3)

trace_train = go.Scatter(x=fpr_train, y=tpr_train, mode='lines', name='AUC_train = %0.2f' % auc_train,
                   line=dict(color='red', width=2),
                   hovertemplate='train<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_train])

trace_valid = go.Scatter(x=fpr_valid, y=tpr_valid, mode='lines', name='AUC_valid = %0.2f' % auc_valid,
                   line=dict(color='green', width=2),
                   hovertemplate='valid<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_valid])

trace_test = go.Scatter(x=fpr_test, y=tpr_test, mode='lines', name='AUC_test = %0.2f' % auc_test,
                   line=dict(color='darkorange', width=2),
                   hovertemplate='test<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_test])


reference_line = go.Scatter(x=[0,1], y=[0,1], mode='lines', name='Reference Line',
                            line=dict(color='navy', width=2, dash='dash'))

fig_roc = go.Figure(data=[trace_train,trace_valid,trace_test,reference_line])

fig_roc.update_layout(
    title ={
        'text':'ROC-кривая', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 1350,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate')

fig_roc.update_xaxes(showspikes=True)
fig_roc.update_yaxes(showspikes=True)


fig_roc.show()

<span style="color:Blue">

Вывод по ROC-кривой:
1. Не смотря на то, что модель показывает, очень хорошую $ROC$ кривую на обучающем наборе,  
модель, так же выдает сталибильное качество на валидационном и тестовом наборах, а значит  
не переобучена - разброс ($variance$) не большой.
2. На тестовом наборе полученно качество $ROC AUC = 0.74$.

#### <span style="color:MediumBlue">Построение модели Gradient Boosting Classifier</span>

##### <span style="color:MediumSlateBlue">Baseline обучение модели Hist Gradient Boosting Classifier</span>

In [ ]:
MX_train = fp.ParquetFile('first_metamodel/features/features_first_meta_stat_train').to_pandas()
my_train = pd.read_csv('first_metamodel/data/mfy_train.csv').set_index('id')['flag']
MX_valid = fp.ParquetFile('first_metamodel/features/features_first_meta_stat_valid').to_pandas()
my_valid = pd.read_csv('first_metamodel/data/mfy_valid.csv').set_index('id')['flag']

In [ ]:
# обучаем модель Gradient Boosting Classifier
gradient_boosting = HistGradientBoostingClassifier(random_state=42)
gradient_boosting.fit(MX_train,my_train)

# для метрик ROC AUC делаем предсказание модели в виде вероятности
my_train_rf__pred_proba = gradient_boosting.predict_proba(MX_train)[:,1]
my_valid_rf_pred_proba = gradient_boosting.predict_proba(MX_valid)[:,1]

# Делаем предсказание для валидационной выборки
my_valid_rf_pred = gradient_boosting.predict(MX_valid)

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(my_train, my_train_rf__pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(my_valid, my_valid_rf_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(my_valid,my_valid_rf_pred))

# time: 30s

<span style="color:Blue">

Выводы:

В сравнение с моделью $HistGradientBoostingClassifier$ построенной   
на несбалансированных данных $transform$ $data$ $stat$:
1. Качество модели по метрике $ROC AUC$ не изменилась.
2. Качество модели по метрике $recall_1$ значительно  
улучшилось. Правда, $recall_0$ уменьшилось.
3. Качество модели по метрике $precision_1$ уменьшилось.   
4. На порядок уменьшилось время обучения модели. Это логично,  
велична сбалансированной выборки всего $107000$ $samples$. 

##### <span style="color:MediumSlateBlue">Подбор гиперпараметров модели</span>

In [ ]:
# настроим оптимизацию гипер параметров
def optuna_gb_meta(trial):
  # задаем пространства поиска гиперпараметров
  learning_rate = trial.suggest_float('learning_rate',0.01,0.1)
  max_iter = trial.suggest_int('max_iter', 150, 250,step = 1)
  max_leaf_nodes = trial.suggest_int('max_leaf_nodes', 2, 60,step = 1)
  max_depth = trial.suggest_int('max_depth', 1, 10,step = 1)
  min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 60,step = 1)
  max_features = trial.suggest_float('max_features',0.1,1)
  l2_regularization = trial.suggest_float('l2_regularization',0.01,1)
  class_0_weight = trial.suggest_float('class_0_weight',0.01,1)
  class_1_weight = trial.suggest_float('class_1_weight',0.01,1)
  k_features = trial.suggest_int('k_features', 20, 507,step = 1)
  class_1_percent = trial.suggest_float('class_1_percent',0.01,1)
  random_state = trial.suggest_int('random_state', 1, 1000000)

  # создаем модель
  optuna_gradient_boosting_meta = HistGradientBoostingClassifier(
      learning_rate= learning_rate,
      max_iter = max_iter,
      max_leaf_nodes =max_leaf_nodes,
      max_depth = max_depth,
      min_samples_leaf = min_samples_leaf,
      max_features=max_features,
      l2_regularization = l2_regularization,
      class_weight={0:class_0_weight,1:class_1_weight},
      random_state=random_state)

  # Загружаем обучающие наборы
  MX_train_pd = fp.ParquetFile('first_metamodel/features/features_first_meta_stat_train').to_pandas()
  my_train_pd = pd.read_csv('first_metamodel/data/mfy_train.csv')

  # поготовим данные my_train_pd к работе с функцией class_1_percent_samples
  my_train_pd.set_index('id',drop=False,inplace=True)

  # с помощью функции class_1_percent_samples зададим долю класса 1
  list_c1_percent_id = class_1_percent_samples(my_train_pd,class_1_percent,random_state=random_state)[:1000000]

  # подготовим данные для обучения
  MX_train_balanced = MX_train_pd.loc[list_c1_percent_id]
  my_train_balanced = my_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

  # освободим память от "тяжелых" и ненужных файлов
  del MX_train_pd, my_train_pd
  gc.collect()
  
  # с помощью класса SelectKBest получим список лучших признаков
  selector = SelectKBest(f_classif, k=k_features)
  selector.fit(MX_train_balanced, my_train_balanced)
  list_best_features = selector.get_feature_names_out()

  MX_train_balanced = MX_train_balanced[list_best_features].to_numpy()

  # я не воспользовался параметром timeout метода optimize потому, что  
  # optimize отставливает поиск параметров, если время обучения 
  # превышает timeout. Мне нужно чтобы поиск параметров продолжился.
  try:
    # для модуля func_time_out упакуем обучение модели в функцию try_func
    def try_func():
            return optuna_gradient_boosting_meta.fit(MX_train_balanced,my_train_balanced)
    # обучаем модель с ограничением по времени 10 минут (600 секунд)
    optuna_gradient_boosting_meta = func_timeout.func_timeout(600, try_func)

    # удаляем крупные файлы чтобы высвободить память 
    del MX_train_balanced, my_train_balanced
    gc.collect()

    # загружаем тестовые наборы
    MX_train = fp.ParquetFile('first_metamodel/features/features_first_meta_stat_train').to_pandas()[list_best_features].to_numpy()
    MX_valid = fp.ParquetFile('first_metamodel/features/features_first_meta_stat_valid').to_pandas()[list_best_features].to_numpy()
    my_train = pd.read_csv('first_metamodel/data/mfy_train.csv')['flag'].to_numpy()
    my_valid = pd.read_csv('first_metamodel/data/mfy_valid.csv')['flag'].to_numpy()

    # делаем предсказание на обучающем и валидационном наборе
    # cчитаем метрики для класса 1 добавляем их в список
    roc_train = metrics.roc_auc_score(my_train, optuna_gradient_boosting_meta.predict_proba(MX_train)[:,1])
    roc_valid = metrics.roc_auc_score(my_valid, optuna_gradient_boosting_meta.predict_proba(MX_valid)[:,1])
    f1_score = metrics.f1_score(my_valid, optuna_gradient_boosting_meta.predict(MX_valid))
    recall_1 = metrics.recall_score(my_valid, optuna_gradient_boosting_meta.predict(MX_valid),average=None,zero_division=0)[1]
    precision_1 = metrics.precision_score(my_valid, optuna_gradient_boosting_meta.predict(MX_valid),average=None,zero_division=0)[1]

    # удаляем крупные файлы чтобы высвободить память 
    del MX_train, MX_valid, my_train, my_valid
    gc.collect()

  except func_timeout.FunctionTimedOut:
    # удаляем крупные файлы чтобы высвободить память 
    del MX_train_balanced, my_train_balanced
    gc.collect()
    
    # фиксируем пустые значения метрик
    roc_train = 0
    roc_valid = 0
    f1_score = 0
    recall_1 = 0
    precision_1 = 0
    pass

  return roc_train, roc_valid, f1_score, recall_1, precision_1

In [ ]:
# cоздаем объект исследования
# чтобы модель не переобучалась минимизируем roc_train 
# и максимизируем roc_valid
optuna_study_gb_meta = optuna.create_study(study_name="HistGradientBoostingClassifier_meta_first_stat", 
                               directions=['maximize','maximize','maximize','maximize','maximize'], 
                               sampler=optuna.samplers.TPESampler(),
                               pruner='Hyperband',
                               storage='sqlite:///optuna_studies.db',
                               load_if_exists=True)

# ну случай удаления обучения
# optuna.delete_study(study_name="HistGradientBoostingClassifier_meta_first_stat", storage='sqlite:///optuna_studies.db')

In [ ]:
# ищем лучшую комбинацию гиперпараметров n_trials раз
optuna_study_gb_meta.optimize(optuna_gb_meta, n_trials=300)

##### <span style="color:MediumSlateBlue">Анализ гиперпараметров</span>

In [ ]:
# из полученного результат соврмируем Data Frame
optuna_study_gb_meta_pd = optuna_study_gb_meta.trials_dataframe()
# переименуем столбы
optuna_study_gb_meta_pd.rename(columns={
    'values_0': 'roc_train',
    'values_1': 'roc_valid',
    'values_2': 'f1_score',
    'values_3': 'recall_1',
    'values_4': 'precision_1'
},inplace=True)
optuna_study_gb_meta_pd.tail(5)

In [ ]:
# покаждем статистику обучения
print('Максимальное значение метрики ROC AUC на валидационном наборе:',round(optuna_study_gb_meta_pd['roc_valid'].max(),3))
print('Среднее значение метрики ROC AUC на валидационном наборе:',round(optuna_study_gb_meta_pd['roc_valid'].mean(),3))
print('Максимальное значение метрики f1_score на валидационном наборе:',round(optuna_study_gb_meta_pd['f1_score'].max(),3))
print('Среднее значение метрики f1_score на валидационном наборе:',round(optuna_study_gb_meta_pd['f1_score'].mean(),3))
print('Максимальное значение метрики recall_1 на валидационном наборе:',round(optuna_study_gb_meta_pd['recall_1'].max(),3))
print('Среднее значение метрики recall_1 на валидационном наборе:',round(optuna_study_gb_meta_pd['recall_1'].mean(),3))
print('Максимальное значение метрики precision_1 на валидационном наборе:',round(optuna_study_gb_meta_pd['precision_1'].max(),3))
print('Среднее значение метрики precision_1 на валидационном наборе:',round(optuna_study_gb_meta_pd['precision_1'].mean(),3))

In [ ]:
# построим график важности гиперпарметров
optuna.visualization.plot_param_importances(optuna_study_gb_meta, target_name='Score')

In [ ]:
optuna_study_gb_meta_pd.head(1)

In [ ]:
# Построим зависимость метрики precision_1 от гиперпараметров
fig = px.scatter(
    data_frame=optuna_study_gb_meta_pd,
    x='precision_1', #ось абсцисс
    y=['recall_1','f1_score','roc_train','roc_valid'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision класса 1 от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision класса 1',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрик качества модели от number

fig = px.scatter(
    data_frame=optuna_study_gb_meta_pd[(optuna_study_gb_meta_pd['recall_1']>0.14)&(optuna_study_gb_meta_pd['precision_1']>0.14)],
    x='number', #ось абсцисс
    y=['roc_train','roc_valid','recall_1','precision_1'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision_1',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

<span style="color:Blue">

Вывод:  

Их всех выбираем точку $number =184$.   
В этой точке оптимальные значения метрик $recall_1$ и $precision_1$, а  
также относительно высокая метрика $ROC AUC$.   
Посмотрим каким значениям гиперпараметров соотвествует выбранная точка.

In [ ]:
# определим номер лучшге варианта
best_optuna_number = 184

# сформируем словарь лучших гипирпарметров HistGradientBoostingClassifier
best_param_hgbc = {
    'learning_rate' : optuna_study_gb_meta_pd[optuna_study_gb_meta_pd['number']==best_optuna_number]['params_learning_rate'].iloc[0],
    'max_iter' : optuna_study_gb_meta_pd[optuna_study_gb_meta_pd['number']==best_optuna_number]['params_max_iter'].iloc[0],
    'max_leaf_nodes' : optuna_study_gb_meta_pd[optuna_study_gb_meta_pd['number']==best_optuna_number]['params_max_leaf_nodes'].iloc[0],
    'max_depth' : optuna_study_gb_meta_pd[optuna_study_gb_meta_pd['number']==best_optuna_number]['params_max_depth'].iloc[0],
    'min_samples_leaf' : optuna_study_gb_meta_pd[optuna_study_gb_meta_pd['number']==best_optuna_number]['params_min_samples_leaf'].iloc[0],
    'max_features' : optuna_study_gb_meta_pd[optuna_study_gb_meta_pd['number']==best_optuna_number]['params_max_features'].iloc[0],
    'l2_regularization' : optuna_study_gb_meta_pd[optuna_study_gb_meta_pd['number']==best_optuna_number]['params_l2_regularization'].iloc[0],
    'class_weight' : {0: optuna_study_gb_meta_pd[optuna_study_gb_meta_pd['number']==best_optuna_number]['params_class_0_weight'].iloc[0],
                      1: optuna_study_gb_meta_pd[optuna_study_gb_meta_pd['number']==best_optuna_number]['params_class_1_weight'].iloc[0],}
    }

# определим перменные для лучших значений параметров
best_k_features = optuna_study_gb_meta_pd[optuna_study_gb_meta_pd['number']==best_optuna_number]['params_k_features'].iloc[0]
best_random_state = optuna_study_gb_meta_pd[optuna_study_gb_meta_pd['number']==best_optuna_number]['params_random_state'].iloc[0]
best_class_1_percent = optuna_study_gb_meta_pd[optuna_study_gb_meta_pd['number']==best_optuna_number]['params_class_1_percent'].iloc[0]


# Выведем принятые наилучшие параметры
print('best learning rate:',round(best_param_hgbc['learning_rate'],3))
print('best max iter:',best_param_hgbc['max_iter'])
print('best max leaf nodes:',best_param_hgbc['max_leaf_nodes'])
print('best max depth:',best_param_hgbc['max_depth'])
print('best min samples leaf:',best_param_hgbc['min_samples_leaf'])
print('best max features:',round(best_param_hgbc['max_features'],3))
print('best l2 regularization:',round(best_param_hgbc['l2_regularization'],3))
print('best class 0 weight:',round(best_param_hgbc['class_weight'][0],3))
print('best class 1 weight:',round(best_param_hgbc['class_weight'][1],3))

print('best class 1 percent:',round(best_class_1_percent,3))
print('best k features:',best_k_features)
print('best random state:',best_random_state)
print('time for best train:',round(optuna_study_gb_meta_pd[optuna_study_gb_meta_pd['number']==best_optuna_number]['duration'].iloc[0].seconds/60),'minutes')

print()
print('ROC AUC на обучающем наборе:', round(optuna_study_gb_meta_pd[optuna_study_gb_meta_pd['number']==best_optuna_number]['roc_train'].iloc[0],3))
print('ROC AUC на валидационном наборе:', round(optuna_study_gb_meta_pd[optuna_study_gb_meta_pd['number']==best_optuna_number]['roc_valid'].iloc[0],3))
print('precision класса 1:', round(optuna_study_gb_meta_pd[optuna_study_gb_meta_pd['number']==best_optuna_number]['precision_1'].iloc[0],3))
print('recall класса 1:', round(optuna_study_gb_meta_pd[optuna_study_gb_meta_pd['number']==best_optuna_number]['recall_1'].iloc[0],3))

##### <span style="color:MediumSlateBlue">Обучение модели с лучшими параметрами</span>

In [ ]:
# Загружаем обучающие наборы
MX_train_pd = fp.ParquetFile('first_metamodel/features/features_first_meta_stat_train').to_pandas()
my_train_pd = pd.read_csv('first_metamodel/data/mfy_train.csv')

# поготовим данные my_train_pd к работе с функцией class_1_percent_samples
my_train_pd.set_index('id',drop=False,inplace=True)

# с помощью функции class_1_percent_samples зададим долю класса 1
list_c1_percent_id = class_1_percent_samples(my_train_pd,best_class_1_percent,random_state=best_random_state)[:1000000]

# подготовим данные для обучения
MX_train_balanced = MX_train_pd.loc[list_c1_percent_id]
my_train_balanced = my_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

# освободим память от "тяжелых" и ненужных файлов
del MX_train_pd, my_train_pd
gc.collect()

# с помощью класса SelectKBest получим список лучших признаков
selector = SelectKBest(f_classif, k=best_k_features)
selector.fit(MX_train_balanced, my_train_balanced)
list_best_features = selector.get_feature_names_out()

MX_train_balanced = MX_train_balanced[list_best_features].to_numpy()

# обучаем модель HistGradientBoostingClassifier с наилучшеми параметрами
gradient_boosting = HistGradientBoostingClassifier(
        **best_param_hgbc,
        random_state=best_random_state)
gradient_boosting.fit(MX_train_balanced,my_train_balanced)

# удаляем крупные файлы чтобы высвободить память 
del MX_train_balanced, my_train_balanced
gc.collect()

# загружаем тестовые наборы
MX_train = fp.ParquetFile('first_metamodel/features/features_first_meta_stat_train').to_pandas()[list_best_features].to_numpy()
MX_valid = fp.ParquetFile('first_metamodel/features/features_first_meta_stat_valid').to_pandas()[list_best_features].to_numpy()
my_train = pd.read_csv('first_metamodel/data/mfy_train.csv')['flag'].to_numpy()
my_valid = pd.read_csv('first_metamodel/data/mfy_valid.csv')['flag'].to_numpy()

# для метрик ROC AUC делаем предсказание модели в виде вероятности
my_train_gb_pred_proba = gradient_boosting.predict_proba(MX_train)[:,1]
my_valid_gb_pred_proba = gradient_boosting.predict_proba(MX_valid)[:,1]

# Делаем предсказание для валидационной выборки
my_valid_gb_pred = gradient_boosting.predict(MX_valid)

# удаляем крупные файлы чтобы высвободить память 
del MX_train, MX_valid
gc.collect()

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(my_train, my_train_gb_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(my_valid, my_valid_gb_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(my_valid,my_valid_gb_pred,zero_division=0))

# time: 1m 40s

##### <span style="color:MediumSlateBlue">Анализ влияния порога классификации на качество модели</span>

In [ ]:
# чтобы проше обращаться к каждому обьекту в данных
# преобразуем y_valid_LR_pred к Series типу
my_valid_gb_pred_proba = pd.Series(my_valid_gb_pred_proba)

# формируем список из необходимых значений thresholds
thresholds = np.arange(0.01, 1, 0.01)

# сформируем списко для заполнения данными результатов анализа
threshold_data = []


for threshold in thresholds:
        # сформируем список для заполнения данными текушего threshold
        threshold_current_data = [threshold]

        #клиентов, для которых вероятность дефолта > threshold, относим к классу 1
        #в противном случае — к классу 0
        my_valid_gb_pred = my_valid_gb_pred_proba.apply(lambda x: 1 if x>threshold else 0)


        #Считаем метрики для класса 0 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(my_valid, my_valid_gb_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.precision_score(my_valid, my_valid_gb_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.f1_score(my_valid, my_valid_gb_pred,average=None,zero_division=0)[0],3))

        #Считаем метрики для класса 1 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(my_valid, my_valid_gb_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.precision_score(my_valid, my_valid_gb_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.f1_score(my_valid, my_valid_gb_pred,average=None,zero_division=0)[1],3))

        # добавляем полученные данные в список threshold_data
        threshold_data.append(threshold_current_data)

        # из полученных данных сформируем dataframe
        thresholds_pd =pd.DataFrame(
        columns=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'],
        data = threshold_data)

# time: 17s

In [ ]:
# Визуализируем метрики на валидационном наборе при различных threshold
fig_threshold = px.line(
    data_frame=thresholds_pd,
    x='threshold', #ось абсцисс
    y=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'], #ось ординат
)
fig_threshold.update_layout(
    title ={
        'text':'Зависимость качества модели от threshold', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Порог классификации threshold',
    yaxis_title='Величина метрики')
fig_threshold.update_xaxes(showspikes=True)
fig_threshold.update_yaxes(showspikes=True)

fig_threshold.show()

Метрика $precision$ показывает на сколько сильно ошиблась модель при определиние класса.  
Чем выше метрика, тем меньше ошиблась модель. 

$$precision = \frac{TP}{TP+FP}$$

Метрика $recall$ показывает способность модели найти класс.  
Чем выше метрика, тем выше способность модели находить класс.  

$$recall = \frac{TP}{TP+FN}$$

Метрика $f1-score$ показывает баланс между $precision$ и $recall$ 

$threshold$- порог класификации модели.  
Все объекты для которых $у_{\text{proba pred}} < threshold$, модель относит к класс 0.  
Соотвественно, если $у_{\text{proba pred}}  > threshold$ модель относит обьект к классу 1.  
Для идельаной модели $threshold$, является границей между областями класса 0 и класса 1.  

<span style="color:Blue">

Выводы по результам анализа:  

Для улучшения качества модели по метрике $ROC AUC$, нет смысла сдвигать порог   
классификации. Но зависимость метрик качества модели от $threshold$, может рассказать о  
способности модели искать класс 1 и класс 0.

При фокусировке модели на классе 1 (более низкий $threshold$):  
- на классе 1 метрика $precision$ уменьшается, метрика $recall$ растет;  
- на классе 0 метрика $precision$ растет, метрика $recall$ уменьшается.  

При фокусировке модели на классе 0 (более высокий $threshold$):  
- на классе 1 метрика $precision$ увеличивается, метрика $recall$ уменьшается;  
- на классе 0 метрика $precision$ уменьшается, метрика $recall$ растет.  

Обьсняется тем, что в условия дисбаланса модель научилась хорошо определять класс 0 и плохо класс 1.  

При фокусировке модели на классе 1 (более низкий $threshold$), все больше обьектов отнесены к классу 1.   
Так как $precision$ класса 1 уменьшается, значит в области класса 1 стало больше обектов класса 0,  
что явлется закономерным для "идеальной" модели.  
Однако при этом, $recall$ класса 1 растет, значит в добавленных обьектах также присутвует класс 1.  
Это как раз указывает на то, что модель не нучилась находить среди класса 0 класс 1. В области с     
высокой вероятностью класса 0 (более низкий $threshold$), находятся обьекты класса 1.  

Фокусировка на классе 0 (более высокий $threshold$), заставляет модель все больше обьектов из области класса 1  
относить к классу 0. При этом увеличение метрики $precision$ класса 1 говорит о том, что в области класса 1     
становится меньше 0, а значит модель умеет среди класса 1, находить класс 0.  
После  $threshold=0.88$ модель впринципе теряет способность отличать класс 1. У модели нет обьектов класса 1   
в которых она "уверена" на 90+%.


Обратим внимание на $threshold=0.52$, в этой точке находится баланс межу метриками качества модели:
$$precision_1 = recall_1 = \text{f1-score}_1 = 0.148$$
$$precision_0 = recall_0 = \text{f1-score}_0 = 0.97$$

Для модели с лучшим качеством эти две точки должны находится максимально близко к друг другу и иметь   
высокое значение метрик $precision_1$ и $recall_1$.  
Для класса 0 это условие выполнется, что также говорит о том, что модель научилась искать этот класс.    
Для класса 1 это условие не выполнется, что также говорит о том, что модель плохо ищет класс 1.

##### <span style="color:MediumSlateBlue">Построение ROC-кривой выбранной модели на тестовом наборе</span>

In [ ]:
# сформируем данные для обучения и проверики модели
# для подбора гиперпараметров модели ограничем размер обучающей выборки
MX_train = fp.ParquetFile('first_metamodel/features/features_first_meta_stat_train').to_pandas()[list_best_features].to_numpy()
my_train = pd.read_csv('first_metamodel/data/mfy_train.csv')['flag'].to_numpy()
MX_valid = fp.ParquetFile('first_metamodel/features/features_first_meta_stat_valid').to_pandas()[list_best_features].to_numpy()
my_valid = pd.read_csv('first_metamodel/data/mfy_valid.csv')['flag'].to_numpy()
MX_test = fp.ParquetFile('first_metamodel/features/features_first_meta_stat_test').to_pandas()[list_best_features].to_numpy()
my_test = pd.read_csv('first_metamodel/data/mfy_test.csv')['flag'].to_numpy()

In [ ]:
print("Размеры обучающего набора",MX_train.shape,my_train.shape)
print("Размеры валидационного набора",MX_valid.shape,my_valid.shape)
print("Размеры тестового набора",MX_test.shape,my_test.shape)
# time: 1m 43s

In [ ]:
# для метрик ROC AUC делаем предсказание модели в виде вероятности
my_train_gb_pred_proba = gradient_boosting.predict_proba(MX_train)[:,1]
my_valid_gb_pred_proba = gradient_boosting.predict_proba(MX_valid)[:,1]
my_test_gb_pred_proba = gradient_boosting.predict_proba(MX_test)[:,1]

# Делаем предсказание для валидационной выборки
my_test_gb_pred = gradient_boosting.predict(MX_test)

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(my_train, my_train_gb_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(my_valid, my_valid_gb_pred_proba),3))
print('ROC AUC на тестовом наборе', round(metrics.roc_auc_score(my_test, my_test_gb_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(my_test,my_test_gb_pred,zero_division=0))

In [ ]:
# Для расчета значений TPR и FPR используем функцию roc_curve
fpr_train, tpr_train, _ = metrics.roc_curve(my_train, my_train_gb_pred_proba)
fpr_valid, tpr_valid, _ = metrics.roc_curve(my_valid, my_valid_gb_pred_proba)
fpr_test, tpr_test, _ = metrics.roc_curve(my_test, my_test_gb_pred_proba)

# рассчитаем площадь под кривой
auc_train = round(metrics.roc_auc_score(my_train, my_train_gb_pred_proba),3)
auc_valid = round(metrics.roc_auc_score(my_valid, my_valid_gb_pred_proba),3)
auc_test = round(metrics.roc_auc_score(my_test, my_test_gb_pred_proba),3)

trace_train = go.Scatter(x=fpr_train, y=tpr_train, mode='lines', name='AUC_train = %0.2f' % auc_train,
                   line=dict(color='red', width=2),
                   hovertemplate='train<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_train])

trace_valid = go.Scatter(x=fpr_valid, y=tpr_valid, mode='lines', name='AUC_valid = %0.2f' % auc_valid,
                   line=dict(color='green', width=2),
                   hovertemplate='valid<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_valid])

trace_test = go.Scatter(x=fpr_test, y=tpr_test, mode='lines', name='AUC_test = %0.2f' % auc_test,
                   line=dict(color='darkorange', width=2),
                   hovertemplate='test<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_test])


reference_line = go.Scatter(x=[0,1], y=[0,1], mode='lines', name='Reference Line',
                            line=dict(color='navy', width=2, dash='dash'))

fig_roc = go.Figure(data=[trace_train,trace_valid,trace_test,reference_line])

fig_roc.update_layout(
    title ={
        'text':'ROC-кривая', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 1350,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate')

fig_roc.update_xaxes(showspikes=True)
fig_roc.update_yaxes(showspikes=True)


fig_roc.show()

<span style="color:Blue">

Вывод по ROC-кривой:
1. Модель выдает сталибильное качество на валидационном и тестовом наборах, а значит  
не переобучена - разброс ($variance$) не большой. Хотя, качество модели на обучающем наборе,  
заметно лучше.
2. На тестовом наборе полученно качество $ROC AUC = 0.75$.

## <span style="color:DodgerBlue">Третий этап построения блендинга моделей</span>

### <span style="color:RoyalBlue">Формирование данных для обучения second metamodels</span>

Попробуем улучшить качество модели за счет блендинга.  
Сформируем блендинг из рассмотренных мета модей:
- $Logistic$ $Regression$;
- $Random$ $Forest$ $Classifier$;
- $Hist$ $Gradient$ $Boosting$ $Classifier$,  
обученных на данных $transform$ $data$ $torow$ и $transform$ $data$ $stat$.

Для блендинга выберем следующие модели:
- модели с высоким значением метрики $ROC AUC$ на валидационном наборе;
- модели с высоким значением метрики $precision_1$ на валидационном наборе;
- модели с высоким значением метрики $f1_{score}$ на валидационном наборе 
($precision_1 \approx recall_1$);

In [ ]:
# сформируем заготовки под метаданные для обучения и проверики метамодели 
features_second_meta = pd.read_csv('first_metamodel/data/mfy_test.csv')

# задаим количество отбираемых "лучших" моделей
best_models_number = 5

#### <span style="color:MediumBlue">Формирование метапризнаков из моделей обученных на transform data torow данных</span>

##### <span style="color:MediumSlateBlue">Формирование метапризнаков от модели *Logistic Regression* обученной сбалансированными *transform data torow* данными</span>

In [ ]:
# загрузим обьект исcледования optuna
optuna_study_lr_torow_meta = optuna.load_study(study_name="LogisticRegression_meta_first_torow",
                               storage='sqlite:///optuna_studies.db')

In [ ]:
# из полученного обьекта сформируем Data Frame
optuna_study_lr_torow_meta_pd = optuna_study_lr_torow_meta.trials_dataframe().dropna()
# переименуем столбы
optuna_study_lr_torow_meta_pd.rename(columns={
    'values_0': 'roc_train',
    'values_1': 'roc_valid',
    'values_2': 'f1_score',
    'values_3': 'recall_1',
    'values_4': 'precision_1'
},inplace=True)
# удалим ненужные столбцы
optuna_study_lr_torow_meta_pd.drop(['datetime_start','datetime_complete','state'],axis=1, inplace=True)
# сфомируем столбец продолжительности времени обучения выраженное в минутах
optuna_study_lr_torow_meta_pd['duration'] = optuna_study_lr_torow_meta_pd['duration'].apply(lambda x: round(x.seconds/60))

# выберем точки с лучшими значениями метрики roc auc на валидационном наборе
best_roc_lr_torow_b = optuna_study_lr_torow_meta_pd.sort_values(by='roc_valid',ascending=False)[:best_models_number]

# выберем точки с лучшими значениями метрики precion класса 1 на валидационном наборе
best_precion_lr_torow_b = optuna_study_lr_torow_meta_pd.sort_values(by='precision_1',ascending=False)[:best_models_number]

# выберем точки в которых максимальна метрика f1-score класса 1 (precision_1=recall_1)
best_f1_lr_torow_b = optuna_study_lr_torow_meta_pd.sort_values(by='f1_score',ascending=False)[:best_models_number]

# сформируем набор признаков для лучших моделей
best_models_lr_torow_b = pd.concat([best_roc_lr_torow_b,best_precion_lr_torow_b,best_f1_lr_torow_b]).drop_duplicates().reset_index(drop=True)
print(best_models_lr_torow_b.shape)
best_models_lr_torow_b.head(5)

In [ ]:
# сформируем список индексов best_models_lr_torow_b
list_index = best_models_lr_torow_b.index

for index in list_index:
    
    # для того чтобы следить за выполнением цикла введем индикатор
    print('current index: ', index)

    # сформируем необходимые параметры
    solver = best_models_lr_torow_b.loc[index,'params_solver']
    C = best_models_lr_torow_b.loc[index,'params_C']
    class_weight = {0:best_models_lr_torow_b.loc[index,'params_class_0_weight'],
                    1:best_models_lr_torow_b.loc[index,'params_class_1_weight']}
    k_features = best_models_lr_torow_b.loc[index,'params_k_features']
    class_1_percent = best_models_lr_torow_b.loc[index,'params_class_1_percent']
    random_state = best_models_lr_torow_b.loc[index,'params_random_state']

    # для того чтобы следить за выполнением цикла введем индикатор
    print('Loading train data')
    # Загружаем обучающие наборы
    MX_train_pd = fp.ParquetFile('first_metamodel/features/features_first_meta_torow_train').to_pandas()
    my_train_pd = pd.read_csv('first_metamodel/data/mfy_train.csv')

    # поготовим данные my_train_pd к работе с функцией class_1_percent_samples
    my_train_pd.set_index('id',drop=False,inplace=True)

    # с помощью функции class_1_percent_samples зададим долю класса 1
    list_c1_percent_id = class_1_percent_samples(my_train_pd,class_1_percent,random_state=random_state)[:1000000]

    # подготовим данные для обучения
    MX_train_balanced = MX_train_pd.loc[list_c1_percent_id]
    my_train_balanced = my_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

    # освободим память от "тяжелых" и ненужных файлов
    del MX_train_pd, my_train_pd
    gc.collect()

    # с помощью класса SelectKBest получим список лучших признаков
    selector = SelectKBest(f_classif, k=k_features)
    selector.fit(MX_train_balanced, my_train_balanced)
    list_best_features = selector.get_feature_names_out()

    # сохраним list_best_features для дальнейшего воспроизведения
    dump(list_best_features, 'first_metamodel/list_best_features/'+'list_best_features_LRTR_'+str('0'+str(index))[-2:]+'.joblib')

    MX_train_balanced = MX_train_balanced[list_best_features].to_numpy()

    # для того чтобы следить за выполнением цикла введем индикатор
    print('start train')
    # обучаем модель LogisticRegression с наилучшеми параметрами
    logistic_regression = linear_model.LogisticRegression(
        solver = solver,
        C=C,
        class_weight=class_weight,
        random_state=random_state,
        max_iter=10000)
    logistic_regression.fit(MX_train_balanced,my_train_balanced)
    # для того чтобы следить за выполнением цикла введем индикатор
    print('finished train')

    # сохраним модель для дальнейшего воспроизведения
    dump(logistic_regression, 'first_metamodel/models/'+'LRTR'+str('0'+str(index))[-2:]+'.joblib')

    # удаляем крупные файлы чтобы высвободить память 
    del MX_train_balanced, my_train_balanced
    gc.collect()

    # для того чтобы следить за выполнением цикла введем индикатор
    print('Loading test data')
    MX_test = fp.ParquetFile('first_metamodel/features/features_first_meta_torow_test').to_pandas()[list_best_features].to_numpy()

    # для метрик ROC AUC делаем предсказание модели в виде вероятности и записываем в методанные
    features_second_meta['LRTR'+str('0'+str(index))[-2:]] = logistic_regression.predict_proba(MX_test)[:,1]
    # удаляем крупные файлы чтобы высвободить память 
    del MX_test
    gc.collect()

    clear_output()

In [ ]:
# Сохраняем полученные метаданные в DataFrame
fp.write('second_metamodels/features/features_second_meta',features_second_meta)

##### <span style="color:MediumSlateBlue">Формирование метапризнаков от модели *Random Forest Classifier* обученной сбалансированными *transform data torow* данными</span>

In [ ]:
# загрузим обьект исcледования optuna
optuna_study_fr_torow_b = optuna.load_study(study_name="RandomForestClassifier_meta_first_torow",
                                            storage='sqlite:///optuna_studies.db')

In [ ]:
# из полученного обьекта сформируем Data Frame
optuna_study_rf_torow_b_pd = optuna_study_fr_torow_b.trials_dataframe().dropna()
# переименуем столбы
optuna_study_rf_torow_b_pd.rename(columns={
    'values_0': 'roc_train',
    'values_1': 'roc_valid',
    'values_2': 'f1_score',
    'values_3': 'recall_1',
    'values_4': 'precision_1'
},inplace=True)
# удалим ненужные столбцы
optuna_study_rf_torow_b_pd.drop(['datetime_start','datetime_complete','state'],axis=1, inplace=True)
# сфомируем столбец продолжительности времени обучения выраженное в минутах
optuna_study_rf_torow_b_pd['duration'] = optuna_study_rf_torow_b_pd['duration'].apply(lambda x: round(x.seconds/60))

# выберем точки с лучшими значениями метрики roc auc на валидационном наборе
best_roc_rf_torow_b = optuna_study_rf_torow_b_pd.sort_values(by='roc_valid',ascending=False)[:best_models_number]

# выберем точки с лучшими значениями метрики precion класса 1 на валидационном наборе
best_precion_rf_torow_b = optuna_study_rf_torow_b_pd.sort_values(by='precision_1',ascending=False)[:best_models_number]

# выберем точки в которых максимальна метрика f1-score класса 1 (precision_1=recall_1)
best_f1_rf_torow_b = optuna_study_rf_torow_b_pd.sort_values(by='f1_score',ascending=False)[:best_models_number]

# сформируем список из лучших моделей
best_models_rf_torow_b = pd.concat([best_roc_rf_torow_b,best_precion_rf_torow_b,best_f1_rf_torow_b]).drop_duplicates().reset_index(drop=True)
print(best_models_rf_torow_b.shape)
best_models_rf_torow_b.head(4)

In [ ]:
# сформируем  метапризнаки для обучения метамодели
# для этого необходиом обучить модель RandomForestClassifier с выбранными параметрами

# сформируем список индексов best_models_rf_torow_b
list_index = best_models_rf_torow_b.index

for index in list_index:

  # для того чтобы следить за выполнением цикла введем индикатор
  print('current index: ', index)

  # сформируем необходимые параметры
  n_estimators = best_models_rf_torow_b.loc[index,'params_n_estimators']
  criterion = best_models_rf_torow_b.loc[index,'params_criterion']
  max_depth = best_models_rf_torow_b.loc[index,'params_max_depth']
  min_samples_split = best_models_rf_torow_b.loc[index,'params_min_samples_split']
  min_samples_leaf = best_models_rf_torow_b.loc[index,'params_min_samples_leaf']
  max_features = best_models_rf_torow_b.loc[index,'params_max_features']
  class_weight = {0:best_models_rf_torow_b.loc[index,'params_class_0_weight'],
                  1:best_models_rf_torow_b.loc[index,'params_class_1_weight']}
  max_samples = best_models_rf_torow_b.loc[index,'params_max_samples']
  k_features = best_models_rf_torow_b.loc[index,'params_k_features']
  class_1_percent = best_models_rf_torow_b.loc[index,'params_class_1_percent']
  random_state = best_models_rf_torow_b.loc[index,'params_random_state']

  # для того чтобы следить за выполнением цикла введем индикатор
  print('Loading train data')
  # Загружаем обучающие наборы
  MX_train_pd = fp.ParquetFile('first_metamodel/features/features_first_meta_torow_train').to_pandas()
  my_train_pd = pd.read_csv('first_metamodel/data/mfy_train.csv')

  # поготовим данные my_train_pd к работе с функцией class_1_percent_samples
  my_train_pd.set_index('id',drop=False,inplace=True)

  # с помощью функции class_1_percent_samples зададим долю класса 1
  list_c1_percent_id = class_1_percent_samples(my_train_pd,class_1_percent,random_state=random_state)[:1000000]

  # подготовим данные для обучения
  MX_train_balanced = MX_train_pd.loc[list_c1_percent_id]
  my_train_balanced = my_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

  # освободим память от "тяжелых" и ненужных файлов
  del MX_train_pd, my_train_pd
  gc.collect()

  # с помощью класса SelectKBest получим список лучших признаков
  selector = SelectKBest(f_classif, k=k_features)
  selector.fit(MX_train_balanced, my_train_balanced)
  list_best_features = selector.get_feature_names_out()

  # сохраним list_best_features для дальнейшего воспроизведения
  dump(list_best_features, 'first_metamodel/list_best_features/'+'list_best_features_RFTR_'+str('0'+str(index))[-2:]+'.joblib')

  MX_train_balanced = MX_train_balanced[list_best_features].to_numpy()

  # для того чтобы следить за выполнением цикла введем индикатор
  print('start train')
  # обучаем модель RandomForestClassifier с наилучшеми параметрами
  random_forest = RandomForestClassifier(
          n_estimators = n_estimators,
          criterion = criterion,
          max_depth = max_depth,
          min_samples_split = min_samples_split,
          min_samples_leaf = min_samples_leaf,
          max_features = max_features,
          class_weight = class_weight,
          max_samples = max_samples,
          n_jobs=-1,
          random_state=random_state)
  random_forest.fit(MX_train_balanced, my_train_balanced)
  # для того чтобы следить за выполнением цикла введем индикатор
  print('finished train')

  # сохраним модель для дальнейшего воспроизведения
  dump(random_forest, 'first_metamodel/models/'+'RFTR'+str('0'+str(index))[-2:]+'.joblib')

  # удаляем крупные файлы чтобы высвободить память 
  del MX_train_balanced, my_train_balanced
  gc.collect()

  # для того чтобы следить за выполнением цикла введем индикатор
  print('Loading test data')
  MX_test = fp.ParquetFile('first_metamodel/features/features_first_meta_torow_test').to_pandas()[list_best_features].to_numpy()

  # для метрик ROC AUC делаем предсказание модели в виде вероятности и записываем в методанные
  features_second_meta['RFTR'+str('0'+str(index))[-2:]] = random_forest.predict_proba(MX_test)[:,1]
  # удаляем крупные файлы чтобы высвободить память 
  del MX_test
  gc.collect()

  clear_output()

In [ ]:
# Сохраняем полученные метаданные в DataFrame
fp.write('second_metamodels/features/features_second_meta',features_second_meta)

##### <span style="color:MediumSlateBlue">Формирование метапризнаков от модели *Gradient Boosting Classifier* обученной сбалансированными *transform data torow* данными</span>

In [ ]:
# загрузим обьект исcледования optuna
optuna_study_gb_torow_b = optuna.load_study(study_name="HistGradientBoostingClassifier_meta_first_torow", 
                               storage='sqlite:///optuna_studies.db')

In [ ]:
# из полученного обьекта сформируем Data Frame
optuna_study_gb_torow_b_pd = optuna_study_gb_torow_b.trials_dataframe().dropna()
# переименуем столбы
optuna_study_gb_torow_b_pd.rename(columns={
    'values_0': 'roc_train',
    'values_1': 'roc_valid',
    'values_2': 'f1_score',
    'values_3': 'recall_1',
    'values_4': 'precision_1'
},inplace=True)
# удалим ненужные столбцы
optuna_study_gb_torow_b_pd.drop(['datetime_start','datetime_complete','state'],axis=1, inplace=True)
# сфомируем столбец продолжительности времени обучения выраженное в минутах
optuna_study_gb_torow_b_pd['duration'] = optuna_study_gb_torow_b_pd['duration'].apply(lambda x: round(x.seconds/60))

# выберем точки с лучшими значениями метрики roc auc на валидационном наборе
best_roc_gb_torow_b = optuna_study_gb_torow_b_pd.sort_values(by='roc_valid',ascending=False)[:best_models_number]

# выберем точки с лучшими значениями метрики precion класса 1 на валидационном наборе
best_precion_gb_torow_b = optuna_study_gb_torow_b_pd.sort_values(by='precision_1',ascending=False)[:best_models_number]

# выберем точки в которых максимальна метрика f1-score класса 1 (precision_1=recall_1)
best_f1_gb_torow_b = optuna_study_gb_torow_b_pd.sort_values(by='f1_score',ascending=False)[:best_models_number]

# сформируем список из лучших моделей
best_models_gb_torow_b = pd.concat([best_roc_gb_torow_b,best_precion_gb_torow_b,best_f1_gb_torow_b]).drop_duplicates().reset_index(drop=True)
print(best_models_gb_torow_b.shape)
best_models_gb_torow_b.head(5)

In [ ]:
# сформируем  метапризнаки для обучения метамодели
# для этого необходиом обучить модель HistGradientBoostingClassifier с выбранными параметрами

# сформируем список индексов best_models_gb_torow_b
list_index = best_models_gb_torow_b.index

for index in list_index:

    # для того чтобы следить за выполнением цикла введем индикатор
    print('current index:', index)

    # сформируем необходимые параметры
    learning_rate = best_models_gb_torow_b.loc[index,'params_learning_rate']
    max_iter = best_models_gb_torow_b.loc[index,'params_max_iter']
    max_leaf_nodes = best_models_gb_torow_b.loc[index,'params_max_leaf_nodes']
    max_depth = best_models_gb_torow_b.loc[index,'params_max_depth']
    min_samples_leaf = best_models_gb_torow_b.loc[index,'params_min_samples_leaf']
    l2_regularization = best_models_gb_torow_b.loc[index,'params_l2_regularization']
    max_features = best_models_gb_torow_b.loc[index,'params_max_features']
    class_weight = {0:best_models_gb_torow_b.loc[index,'params_class_0_weight'],
                    1:best_models_gb_torow_b.loc[index,'params_class_1_weight']}

    k_features = best_models_gb_torow_b.loc[index,'params_k_features']
    class_1_percent = best_models_gb_torow_b.loc[index,'params_class_1_percent']
    random_state = best_models_gb_torow_b.loc[index,'params_random_state']

    # для того чтобы следить за выполнением цикла введем индикатор
    print('Loading train data')
    # Загружаем обучающие наборы
    MX_train_pd = fp.ParquetFile('first_metamodel/features/features_first_meta_torow_train').to_pandas()
    my_train_pd = pd.read_csv('first_metamodel/data/mfy_train.csv')

    # поготовим данные my_train_pd к работе с функцией class_1_percent_samples
    my_train_pd.set_index('id',drop=False,inplace=True)

    # с помощью функции class_1_percent_samples зададим долю класса 1
    list_c1_percent_id = class_1_percent_samples(my_train_pd,class_1_percent,random_state=random_state)[:1000000]

    # подготовим данные для обучения
    MX_train_balanced = MX_train_pd.loc[list_c1_percent_id]
    my_train_balanced = my_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

    # освободим память от "тяжелых" и ненужных файлов
    del MX_train_pd, my_train_pd
    gc.collect()

    # с помощью класса SelectKBest получим список лучших признаков
    selector = SelectKBest(f_classif, k=k_features)
    selector.fit(MX_train_balanced, my_train_balanced)
    list_best_features = selector.get_feature_names_out()

    # сохраним list_best_features для дальнейшего воспроизведения
    dump(list_best_features, 'first_metamodel/list_best_features/'+'list_best_features_GBTR_'+str('0'+str(index))[-2:]+'.joblib')

    MX_train_balanced = MX_train_balanced[list_best_features].to_numpy()

    # для того чтобы следить за выполнением цикла введем индикатор
    print('start train')
    # обучаем модель HistGradientBoostingClassifier с наилучшеми параметрами
    gradient_boosting = HistGradientBoostingClassifier(
            learning_rate = learning_rate,
            max_iter = max_iter,
            max_leaf_nodes = max_leaf_nodes,
            max_depth = max_depth,
            min_samples_leaf = min_samples_leaf,
            l2_regularization = l2_regularization,
            max_features = max_features,
            class_weight = class_weight,
            random_state=random_state)
    gradient_boosting.fit(MX_train_balanced,my_train_balanced)
    # для того чтобы следить за выполнением цикла введем индикатор
    print('finished train')

    # сохраним модель для дальнейшего воспроизведения
    dump(gradient_boosting, 'first_metamodel/models/'+'GBTR'+str('0'+str(index))[-2:]+'.joblib')

    # удаляем крупные файлы чтобы высвободить память 
    del MX_train_balanced, my_train_balanced
    gc.collect()

    # для того чтобы следить за выполнением цикла введем индикатор
    print('Loading test data')
    MX_test = fp.ParquetFile('first_metamodel/features/features_first_meta_torow_test').to_pandas()[list_best_features].to_numpy()

    # для метрик ROC AUC делаем предсказание модели в виде вероятности и записываем в методанные
    features_second_meta['GBTR'+str('0'+str(index))[-2:]] = gradient_boosting.predict_proba(MX_test)[:,1]
    # удаляем крупные файлы чтобы высвободить память 
    del MX_test
    gc.collect()

    clear_output()

In [ ]:
# Сохраняем полученные метаданные в DataFrame
fp.write('second_metamodels/features/features_second_meta',features_second_meta)

#### <span style="color:MediumBlue">Формирование метапризнаков из моделей обученных на transform data stat данных</span>

##### <span style="color:MediumSlateBlue">Формирование метапризнаков от модели *Logistic Regression* обученной сбалансированными *transform data stat* данными</span>

In [ ]:
# загрузим обьект исcледования optuna
optuna_study_lr_stat_b = optuna.load_study(study_name="LogisticRegression_meta_first_stat",
                               storage='sqlite:///optuna_studies.db')

In [ ]:
# из полученного обьекта сформируем Data Frame
optuna_study_lr_stat_b_pd = optuna_study_lr_stat_b.trials_dataframe()
# переименуем столбы
optuna_study_lr_stat_b_pd.rename(columns={
    'values_0': 'roc_train',
    'values_1': 'roc_valid',
    'values_2': 'f1_score',
    'values_3': 'recall_1',
    'values_4': 'precision_1'
},inplace=True)
# удалим ненужные столбцы
optuna_study_lr_stat_b_pd.drop(['datetime_start','datetime_complete','state'],axis=1, inplace=True)
# сфомируем столбец продолжительности времени обучения выраженное в минутах
optuna_study_lr_stat_b_pd['duration'] = optuna_study_lr_stat_b_pd['duration'].apply(lambda x: round(x.seconds/60))

# выберем точки с лучшими значениями метрики roc auc на валидационном наборе
best_roc_lr_stat_b = optuna_study_lr_stat_b_pd.sort_values(by='roc_valid',ascending=False)[:best_models_number]

# выберем точки с лучшими значениями метрики precion класса 1 на валидационном наборе
best_precion_lr_stat_b = optuna_study_lr_stat_b_pd.sort_values(by='precision_1',ascending=False)[:best_models_number]

# выберем точки в которых максимальна метрика f1-score класса 1 (precision_1=recall_1)
best_f1_lr_stat_b = optuna_study_lr_stat_b_pd.sort_values(by='f1_score',ascending=False)[:best_models_number]

# сформируем список из лучших моделей
best_models_lr_stat_b = pd.concat([best_roc_lr_stat_b,best_precion_lr_stat_b,best_f1_lr_stat_b]).drop_duplicates().reset_index(drop=True)
print(best_models_lr_stat_b.shape)
best_models_lr_stat_b.head(4)

In [ ]:
# сформируем список индексов best_models_lr_stat_b
list_index = best_models_lr_stat_b.index

for index in list_index:

    # для того чтобы следить за выполнением цикла введем индикатор
    print('current index: ', index)
    
    # сформируем необходимые параметры
    C = best_models_lr_stat_b.loc[index,'params_C']
    class_weight = {0:best_models_lr_stat_b.loc[index,'params_class_0_weight'],
                    1:best_models_lr_stat_b.loc[index,'params_class_1_weight']}

    k_features = best_models_lr_stat_b.loc[index,'params_k_features']
    class_1_percent = best_models_lr_stat_b.loc[index,'params_class_1_percent']
    random_state = best_models_lr_stat_b.loc[index,'params_random_state']

    # для того чтобы следить за выполнением цикла введем индикатор
    print('Loading train data')
    # Загружаем обучающие наборы
    MX_train_pd = fp.ParquetFile('first_metamodel/features/features_first_meta_stat_train').to_pandas()
    my_train_pd = pd.read_csv('first_metamodel/data/mfy_train.csv')

    # поготовим данные my_train_pd к работе с функцией class_1_percent_samples
    my_train_pd.set_index('id',drop=False,inplace=True)

    # с помощью функции class_1_percent_samples зададим долю класса 1
    list_c1_percent_id = class_1_percent_samples(my_train_pd,class_1_percent,random_state=random_state)[:1000000]

    # подготовим данные для обучения
    MX_train_balanced = MX_train_pd.loc[list_c1_percent_id]
    my_train_balanced = my_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

    # освободим память от "тяжелых" и ненужных файлов
    del MX_train_pd, my_train_pd
    gc.collect()

    # с помощью класса SelectKBest получим список лучших признаков
    selector = SelectKBest(f_classif, k=k_features)
    selector.fit(MX_train_balanced, my_train_balanced)
    list_best_features = selector.get_feature_names_out()

    # сохраним list_best_features для дальнейшего воспроизведения
    dump(list_best_features, 'first_metamodel/list_best_features/'+'list_best_features_LRST_'+str('0'+str(index))[-2:]+'.joblib')

    MX_train_balanced = MX_train_balanced[list_best_features].to_numpy()

    # для того чтобы следить за выполнением цикла введем индикатор
    print('start train')
    # обучаем модель LogisticRegression с наилучшеми параметрами
    logistic_regression = linear_model.LogisticRegression(
            C=C,
            class_weight=class_weight,
            random_state=42,
            max_iter=10000)
    logistic_regression.fit(MX_train_balanced,my_train_balanced)
    # для того чтобы следить за выполнением цикла введем индикатор
    print('finished train')

    # сохраним модель для дальнейшего воспроизведения
    dump(logistic_regression, 'first_metamodel/models/'+'LRST_'+str('0'+str(index))[-2:]+'.joblib')

    # удаляем крупные файлы чтобы высвободить память 
    del MX_train_balanced, my_train_balanced
    gc.collect()

    # для того чтобы следить за выполнением цикла введем индикатор
    print('Loading test data')
    MX_test = fp.ParquetFile('first_metamodel/features/features_first_meta_stat_test').to_pandas()[list_best_features].to_numpy()
    # для метрик ROC AUC делаем предсказание модели в виде вероятности и записываем в методанные
    features_second_meta['LRST'+str('0'+str(index))[-2:]] = logistic_regression.predict_proba(MX_test)[:,1]
    # удаляем крупные файлы чтобы высвободить память 
    del MX_test
    gc.collect()
    
    clear_output()

In [ ]:
# Сохраняем полученные метаданные в DataFrame
fp.write('second_metamodels/features/features_second_meta',features_second_meta)

##### <span style="color:MediumSlateBlue">Формирование метапризнаков от модели *Random Forest Classifier* обученной сбалансированными *transform data stat* данными</span>

In [ ]:
# загрузим обьект исcледования optuna
optuna_study_fr_stat_b = optuna.load_study(study_name="RandomForestClassifier_meta_first_stat",
                               storage='sqlite:///optuna_studies.db')

In [ ]:
# из полученного обьекта сформируем Data Frame
optuna_study_rf_stat_b_pd = optuna_study_fr_stat_b.trials_dataframe().dropna()
# переименуем столбы
optuna_study_rf_stat_b_pd.rename(columns={
    'values_0': 'roc_train',
    'values_1': 'roc_valid',
    'values_2': 'f1_score',
    'values_3': 'recall_1',
    'values_4': 'precision_1'
},inplace=True)
# удалим ненужные столбцы
optuna_study_rf_stat_b_pd.drop(['datetime_start','datetime_complete','state'],axis=1, inplace=True)
# сфомируем столбец продолжительности времени обучения выраженное в минутах
optuna_study_rf_stat_b_pd['duration'] = optuna_study_rf_stat_b_pd['duration'].apply(lambda x: round(x.seconds/60))

# выберем точки с лучшими значениями метрики roc auc на валидационном наборе
best_roc_rf_stat_b = optuna_study_rf_stat_b_pd.sort_values(by='roc_valid',ascending=False)[:best_models_number]

# выберем точки с лучшими значениями метрики precion класса 1 на валидационном наборе
best_precion_rf_stat_b = optuna_study_rf_stat_b_pd.sort_values(by='precision_1',ascending=False)[:best_models_number]

# выберем точки в которых максимальна метрика f1-score класса 1 (precision_1=recall_1)
best_f1_rf_stat_b = optuna_study_rf_stat_b_pd.sort_values(by='f1_score',ascending=False)[:best_models_number]

# сформируем список из лучших моделей
best_models_rf_stat_b = pd.concat([best_roc_rf_stat_b,best_precion_rf_stat_b,best_f1_rf_stat_b]).drop_duplicates().reset_index(drop=True)
print(best_models_rf_stat_b.shape)
best_models_rf_stat_b.head(4)

In [ ]:
# сформируем список индексов best_models_rf_stat_b
list_index = best_models_rf_stat_b.index

for index in list_index:

  # для того чтобы следить за выполнением цикла введем индикатор
  print('current index: ', index)

  # сформируем необходимые параметры
  n_estimators = best_models_rf_stat_b.loc[index,'params_n_estimators']
  criterion = best_models_rf_stat_b.loc[index,'params_criterion']
  max_depth = best_models_rf_stat_b.loc[index,'params_max_depth']
  min_samples_split = best_models_rf_stat_b.loc[index,'params_min_samples_split']
  min_samples_leaf = best_models_rf_stat_b.loc[index,'params_min_samples_leaf']
  max_features = best_models_rf_stat_b.loc[index,'params_max_features']
  class_weight = {0:best_models_rf_stat_b.loc[index,'params_class_0_weight'],
                  1:best_models_rf_stat_b.loc[index,'params_class_1_weight']}
  max_samples = best_models_rf_stat_b.loc[index,'params_max_samples']
  k_features = best_models_rf_stat_b.loc[index,'params_k_features']
  class_1_percent = best_models_rf_stat_b.loc[index,'params_class_1_percent']
  random_state = best_models_rf_stat_b.loc[index,'params_random_state']

  # для того чтобы следить за выполнением цикла введем индикатор
  print('Loading train data')

  # Загружаем обучающие наборы
  MX_train_pd = fp.ParquetFile('first_metamodel/features/features_first_meta_stat_train').to_pandas()
  my_train_pd = pd.read_csv('first_metamodel/data/mfy_train.csv')

  # поготовим данные my_train_pd к работе с функцией class_1_percent_samples
  my_train_pd.set_index('id',drop=False,inplace=True)

  # с помощью функции class_1_percent_samples зададим долю класса 1
  list_c1_percent_id = class_1_percent_samples(my_train_pd,class_1_percent,random_state=random_state)[:1000000]

  # подготовим данные для обучения
  MX_train_balanced = MX_train_pd.loc[list_c1_percent_id]
  my_train_balanced = my_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

  # освободим память от "тяжелых" и ненужных файлов
  del MX_train_pd, my_train_pd
  gc.collect()

  # с помощью класса SelectKBest получим список лучших признаков
  selector = SelectKBest(f_classif, k=k_features)
  selector.fit(MX_train_balanced, my_train_balanced)
  list_best_features = selector.get_feature_names_out()

  # сохраним list_best_features для дальнейшего воспроизведения
  dump(list_best_features, 'first_metamodel/list_best_features/'+'list_best_features_RFST_'+str('0'+str(index))[-2:]+'.joblib')

  MX_train_balanced = MX_train_balanced[list_best_features].to_numpy()

  # для того чтобы следить за выполнением цикла введем индикатор
  print('start train')
  # обучаем модель RandomForestClassifier с наилучшеми параметрами
  random_forest = RandomForestClassifier(
          n_estimators = n_estimators,
          criterion = criterion,
          max_depth = max_depth,
          min_samples_split = min_samples_split,
          min_samples_leaf = min_samples_leaf,
          max_features = max_features,
          class_weight = class_weight,
          max_samples = max_samples,
          n_jobs=-1,
          random_state=random_state)
  random_forest.fit(MX_train_balanced, my_train_balanced)
  # для того чтобы следить за выполнением цикла введем индикатор
  print('finished train')

  # сохраним модель для дальнейшего воспроизведения
  dump(random_forest, 'first_metamodel/models/'+'RFST_'+str('0'+str(index))[-2:]+'.joblib')

  # удаляем крупные файлы чтобы высвободить память 
  del MX_train_balanced, my_train_balanced
  gc.collect()

  # для того чтобы следить за выполнением цикла введем индикатор
  print('Loading test data')
  MX_test= fp.ParquetFile('first_metamodel/features/features_first_meta_stat_test').to_pandas()[list_best_features].to_numpy()
  # для метрик ROC AUC делаем предсказание модели в виде вероятности и записываем в методанные
  features_second_meta['RFST'+str('0'+str(index))[-2:]] = random_forest.predict_proba(MX_test)[:,1]
  # удаляем крупные файлы чтобы высвободить память 
  del MX_test
  gc.collect()

  clear_output()

In [ ]:
# Сохраняем полученные метаданные в DataFrame
fp.write('second_metamodels/features/features_second_meta',features_second_meta)

##### <span style="color:MediumSlateBlue">Формирование метапризнаков от модели *Gradient Boosting Classifier* обученной сбалансированными *transform data stat* данными</span>

In [ ]:
# загрузим обьект исcледования optuna
optuna_study_gb_stat_b = optuna.load_study(study_name="HistGradientBoostingClassifier_meta_first_stat",
                               storage='sqlite:///optuna_studies.db')

In [ ]:
# из полученного обьекта сформируем Data Frame
optuna_study_gb_stat_b_pd = optuna_study_gb_stat_b.trials_dataframe().dropna()
# переименуем столбы
optuna_study_gb_stat_b_pd.rename(columns={
    'values_0': 'roc_train',
    'values_1': 'roc_valid',
    'values_2': 'f1_score',
    'values_3': 'recall_1',
    'values_4': 'precision_1'
},inplace=True)
# удалим ненужные столбцы
optuna_study_gb_stat_b_pd.drop(['datetime_start','datetime_complete','state'],axis=1, inplace=True)
# сфомируем столбец продолжительности времени обучения выраженное в минутах
optuna_study_gb_stat_b_pd['duration'] = optuna_study_gb_stat_b_pd['duration'].apply(lambda x: round(x.seconds/60))

# выберем точки с лучшими значениями метрики roc auc на валидационном наборе
best_roc_gb_stat_b = optuna_study_gb_stat_b_pd.sort_values(by='roc_valid',ascending=False)[:best_models_number]

# выберем точки с лучшими значениями метрики precion класса 1 на валидационном наборе
best_precion_gb_stat_b = optuna_study_gb_stat_b_pd.sort_values(by='precision_1',ascending=False)[:best_models_number]

# выберем точки в которых максимальна метрика f1-score класса 1 (precision_1=recall_1)
best_f1_gb_stat_b = optuna_study_gb_stat_b_pd.sort_values(by='f1_score',ascending=False)[:best_models_number]

# сформируем список из лучших моделей
best_models_gb_stat_b = pd.concat([best_roc_gb_stat_b,best_precion_gb_stat_b,best_f1_gb_stat_b]).drop_duplicates().reset_index(drop=True)
print(best_models_gb_stat_b.shape)
best_models_gb_stat_b.head(4)

In [ ]:
# сформируем список индексов best_models_gb_stat_b
list_index = best_models_gb_stat_b.index

for index in list_index:

  # для того чтобы следить за выполнением цикла введем индикатор
  print('current index:', index)

  # сформируем необходимые параметры
  learning_rate = best_models_gb_stat_b.loc[index,'params_learning_rate']
  max_iter = best_models_gb_stat_b.loc[index,'params_max_iter']
  max_leaf_nodes = best_models_gb_stat_b.loc[index,'params_max_leaf_nodes']
  max_depth = best_models_gb_stat_b.loc[index,'params_max_depth']
  min_samples_leaf = best_models_gb_stat_b.loc[index,'params_min_samples_leaf']
  l2_regularization = best_models_gb_stat_b.loc[index,'params_l2_regularization']
  max_features = best_models_gb_stat_b.loc[index,'params_max_features']
  class_weight = {0:best_models_gb_stat_b.loc[index,'params_class_0_weight'],
                  1:best_models_gb_stat_b.loc[index,'params_class_1_weight']}
  k_features = best_models_gb_stat_b.loc[index,'params_k_features']
  class_1_percent = best_models_gb_stat_b.loc[index,'params_class_1_percent']
  random_state = best_models_gb_stat_b.loc[index,'params_random_state']


  # для того чтобы следить за выполнением цикла введем индикатор
  print('Loading train data')
  # Загружаем обучающие наборы
  MX_train_pd = fp.ParquetFile('first_metamodel/features/features_first_meta_stat_train').to_pandas()
  my_train_pd = pd.read_csv('first_metamodel/data/mfy_train.csv')

  # поготовим данные my_train_pd к работе с функцией class_1_percent_samples
  my_train_pd.set_index('id',drop=False,inplace=True)

  # с помощью функции class_1_percent_samples зададим долю класса 1
  list_c1_percent_id = class_1_percent_samples(my_train_pd,class_1_percent,random_state=random_state)[:1000000]

  # подготовим данные для обучения
  MX_train_balanced = MX_train_pd.loc[list_c1_percent_id]
  my_train_balanced = my_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

  # освободим память от "тяжелых" и ненужных файлов
  del MX_train_pd, my_train_pd
  gc.collect()
  
  # с помощью класса SelectKBest получим список лучших признаков
  selector = SelectKBest(f_classif, k=k_features)
  selector.fit(MX_train_balanced, my_train_balanced)
  list_best_features = selector.get_feature_names_out()

  # сохраним list_best_features для дальнейшего воспроизведения
  dump(list_best_features, 'first_metamodel/list_best_features/'+'list_best_features_GBST_'+str('0'+str(index))[-2:]+'.joblib')

  MX_train_balanced = MX_train_balanced[list_best_features].to_numpy()

  # для того чтобы следить за выполнением цикла введем индикатор
  print('start train')
  # обучаем модель HistGradientBoostingClassifier с наилучшеми параметрами
  gradient_boosting = HistGradientBoostingClassifier(
          learning_rate = learning_rate,
          max_iter = max_iter,
          max_leaf_nodes = max_leaf_nodes,
          max_depth = max_depth,
          min_samples_leaf = min_samples_leaf,
          l2_regularization = l2_regularization,
          max_features = max_features,
          class_weight = class_weight,
          random_state=random_state)
  gradient_boosting.fit(MX_train_balanced,my_train_balanced)
  # для того чтобы следить за выполнением цикла введем индикатор
  print('finished train')

  # сохраним модель для дальнейшего воспроизведения
  dump(gradient_boosting, 'first_metamodel/models/'+'GBST_'+str('0'+str(index))[-2:]+'.joblib')

  del MX_train_balanced, my_train_balanced
  gc.collect()

  # для того чтобы следить за выполнением цикла введем индикатор
  print('Loading test data')
  MX_test = fp.ParquetFile('first_metamodel/features/features_first_meta_stat_test').to_pandas()[list_best_features].to_numpy()
  # для метрик ROC AUC делаем предсказание модели в виде вероятности и записываем в методанные
  features_second_meta['GBST'+str('0'+str(index))[-2:]] = gradient_boosting.predict_proba(MX_test)[:,1]
  # удаляем крупные файлы чтобы высвободить память 
  del MX_test
  gc.collect()

  clear_output()

In [ ]:
# Сохраняем полученные метаданные в DataFrame
fp.write('second_metamodels/features/features_second_meta',features_second_meta)

#### <span style="color:MediumBlue">Формирование тренировочного, валидационного и тестового набора данных</span>

Поставленная задача сформировать блендинг моделей  
блендинг будет 3 слойный
сформируем для каждого слоя свой обучающий набор данных.

In [ ]:
features_second_meta = fp.ParquetFile('second_metamodels/features/features_second_meta').to_pandas()
features_second_meta.head(1)

In [ ]:
# Выделим из features_second_meta тренировочную часть
MX_train, my_train, MX_test, my_test = my_train_test_split(features_second_meta.set_index('id').drop(['flag'],axis=1),
                                                           features_second_meta.set_index('id')['flag'], train_size=0.8, random_state=54)

In [ ]:
# Раздедлим обучающий набор на тренировочную и валидационную части
MX_train, my_train, MX_valid, my_valid = my_train_test_split(MX_train,my_train, train_size=0.8,random_state=48)

In [ ]:
print('Shape train part:',MX_train.shape)
print('Shape valid part:',MX_valid.shape)
print('Shape test part:',MX_test.shape)

In [ ]:
# сохраним тренировочный валидационный и тестовый наборы данных
# чтобы иметь возможность к ними обратиться
fp.write('second_metamodels/features/features_second_meta_train',MX_train)
fp.write('second_metamodels/features/features_second_meta_valid',MX_valid)
fp.write('second_metamodels/features/features_second_meta_test',MX_test)

In [ ]:
# сохраним тренировочный валидационный и тестовый целевой переменной
# чтобы иметь возможность к ними обратиться
my_train.to_csv('second_metamodels/data/msy_train.csv')
my_valid.to_csv('second_metamodels/data/msy_valid.csv')
my_test.to_csv('second_metamodels/data/msy_test.csv')

### <span style="color:RoyalBlue">Построение блендинга из метамоделей (second metamodels)</span>

#### <span style="color:MediumBlue">Построение метамодели Logistic Regression</span>

##### <span style="color:MediumSlateBlue">Baseline обучение метамодели LogisticRegression</span>

In [ ]:
MX_train = fp.ParquetFile('second_metamodels/features/features_second_meta_train').to_pandas()
my_train = pd.read_csv('second_metamodels/data/msy_train.csv').set_index('id')['flag']
MX_valid = fp.ParquetFile('second_metamodels/features/features_second_meta_valid').to_pandas()
my_valid = pd.read_csv('second_metamodels/data/msy_valid.csv').set_index('id')['flag']

In [ ]:
# обучаем модель логистической регрессии
logistic_regression = linear_model.LogisticRegression(random_state=42, max_iter=10000)
logistic_regression.fit(MX_train,my_train)

# для метрик ROC AUC делаем предсказание модели в виде вероятности
my_train_LR_pred_proba = logistic_regression.predict_proba(MX_train)[:,1]
my_valid_LR_pred_proba = logistic_regression.predict_proba(MX_valid)[:,1]

# Делаем предсказание для валидационной выборки
my_valid_LR_pred = logistic_regression.predict(MX_valid)

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(my_train, my_train_LR_pred_proba),3))
print('ROC AUC на тестовом наборе', round(metrics.roc_auc_score(my_valid, my_valid_LR_pred_proba),3))

print('Основные метрики на тестовом наборе:')
print(metrics.classification_report(my_valid,my_valid_LR_pred,zero_division=0))

# time: 3m 5s

<span style="color:Blue">

Выводы:

После двух итераций блендинга моделей, моделей $Logistic$ $Regression$,  
показывает отличное качество по метрике ROC AUC даже без подбора гиперпараметров.

##### <span style="color:MediumSlateBlue">Подбор гиперпараметров модели</span>

In [ ]:
# настроим оптимизацию гипер параметров
def optuna_lg_meta(trial):
  # задаем пространства поиска гиперпараметров
  C = trial.suggest_float('C',0.01,1)
  solver = trial.suggest_categorical('solver', ['lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky'])
  class_0_weight = trial.suggest_float('class_0_weight',0.01,1)
  class_1_weight = trial.suggest_float('class_1_weight',0.01,1)
  k_features = trial.suggest_int('k_features', 20, 88,step = 1)
  class_1_percent = trial.suggest_float('class_1_percent',0.1,1)
  random_state = trial.suggest_int('random_state', 1, 1000000)
  train_size = trial.suggest_float('train_size',0.6,0.9)

  # создаем модель
  optuna_log_reg_meta = linear_model.LogisticRegression(
      C=C,
      solver = solver,
      class_weight={0:class_0_weight,1:class_1_weight},
      random_state=random_state,
      max_iter=10000)

  features_second_meta = fp.ParquetFile('second_metamodels/features/features_second_meta').to_pandas()

  # Выделим из features_second_meta тренировочную часть
  MX_train_pd = my_train_test_split(features_second_meta.set_index('id').drop(['flag'],axis=1),
                                                           features_second_meta.set_index('id')['flag'], 
                                                           train_size=train_size, random_state=random_state)[0]
  my_train_pd = my_train_test_split(features_second_meta.set_index('id').drop(['flag'],axis=1),
                                                           features_second_meta.set_index('id')['flag'], 
                                                           train_size=train_size, random_state=random_state)[1].reset_index()
  
  # поготовим данные my_train_pd к работе с функцией class_1_percent_samples
  my_train_pd.set_index('id',drop=False,inplace=True)

  # с помощью функции class_1_percent_samples зададим долю класса 1
  list_c1_percent_id = class_1_percent_samples(my_train_pd,class_1_percent,random_state=random_state)[:1000000]

  # подготовим данные для обучения
  MX_train_balanced = MX_train_pd.loc[list_c1_percent_id]
  my_train_balanced = my_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

  # освободим память от "тяжелых" и ненужных файлов
  del MX_train_pd, my_train_pd
  gc.collect()
  
  # с помощью класса SelectKBest получим список лучших признаков
  selector = SelectKBest(f_classif, k=k_features)
  selector.fit(MX_train_balanced, my_train_balanced)
  list_best_features = selector.get_feature_names_out()

  MX_train_balanced = MX_train_balanced[list_best_features].to_numpy()

  # я не воспользовался параметром timeout метода optimize потому, что  
  # optimize отставливает поиск параметров, если время обучения 
  # превышает timeout. Мне нужно чтобы поиск параметров продолжился.
  try:
    # для модуля func_time_out упакуем обучение модели в функцию try_func
    def try_func():
            return optuna_log_reg_meta.fit(MX_train_balanced, my_train_balanced)
    # обучаем модель с ограничением по времени 10 минут (600 секунд)
    optuna_log_reg_meta = func_timeout.func_timeout(600, try_func)

    # удаляем крупные файлы чтобы высвободить память 
    del MX_train_balanced, my_train_balanced
    gc.collect()

    # загружаем тестовые наборы
    MX_train = my_train_test_split(features_second_meta.set_index('id').drop(['flag'],axis=1),
                                                           features_second_meta.set_index('id')['flag'], 
                                                           train_size=train_size, random_state=random_state)[0][list_best_features].to_numpy()
    my_train = my_train_test_split(features_second_meta.set_index('id').drop(['flag'],axis=1),
                                                           features_second_meta.set_index('id')['flag'], 
                                                           train_size=train_size, random_state=random_state)[1].to_numpy()
    
    MX_valid = my_train_test_split(features_second_meta.set_index('id').drop(['flag'],axis=1),
                                                           features_second_meta.set_index('id')['flag'], 
                                                           train_size=train_size, random_state=random_state)[2][list_best_features].to_numpy()
    my_valid = my_train_test_split(features_second_meta.set_index('id').drop(['flag'],axis=1),
                                                           features_second_meta.set_index('id')['flag'], 
                                                           train_size=train_size, random_state=random_state)[3].to_numpy()

    # делаем предсказание на обучающем и валидационном наборе и считаем метрики
    roc_train = metrics.roc_auc_score(my_train, optuna_log_reg_meta.predict_proba(MX_train)[:,1])
    roc_valid = metrics.roc_auc_score(my_valid, optuna_log_reg_meta.predict_proba(MX_valid)[:,1])
    f1_score = metrics.f1_score(my_valid, optuna_log_reg_meta.predict(MX_valid))
    recall_1 = metrics.recall_score(my_valid, optuna_log_reg_meta.predict(MX_valid),average=None,zero_division=0)[1]
    precision_1 = metrics.precision_score(my_valid, optuna_log_reg_meta.predict(MX_valid),average=None,zero_division=0)[1]

    # удаляем крупные файлы чтобы высвободить память 
    del MX_train, MX_valid, my_train, my_valid, features_second_meta
    gc.collect()

  except func_timeout.FunctionTimedOut:
    # удаляем крупные файлы чтобы высвободить память 
    del MX_train_balanced, my_train_balanced
    gc.collect()
    
    # фиксируем пустые значения метрик
    roc_train = 0
    roc_valid = 0
    f1_score = 0
    recall_1 = 0
    precision_1 = 0
    pass

  return roc_train, roc_valid, f1_score, recall_1, precision_1

In [ ]:
# cоздаем объект исследования
# чтобы модель не переобучалась минимизируем roc_train 
# и максимизируем roc_valid
optuna_study_lg_meta = optuna.create_study(study_name="LogisticRegression_meta_second", 
                               directions=['maximize','maximize','maximize','maximize','maximize'], 
                               sampler=optuna.samplers.TPESampler(),
                               pruner='Hyperband',
                               storage='sqlite:///optuna_studies.db',
                               load_if_exists=True)
# optuna.delete_study(study_name="LogisticRegression_meta_second", storage='sqlite:///optuna_studies.db')

In [ ]:
# ищем лучшую комбинацию гиперпараметров n_trials раз
optuna_study_lg_meta.optimize(optuna_lg_meta, n_trials=500)

##### <span style="color:MediumSlateBlue">Анализ гиперпараметров</span>

In [ ]:
# из полученного результат соврмируем Data Frame
optuna_study_lg_meta_pd = optuna_study_lg_meta.trials_dataframe().dropna()
# переименуем столбы
optuna_study_lg_meta_pd.rename(columns={
    'values_0': 'roc_train',
    'values_1': 'roc_valid',
    'values_2': 'f1_score',
    'values_3': 'recall_1',
    'values_4': 'precision_1'
},inplace=True)
# Выделим время потраченное на обучение модели
optuna_study_lg_meta_pd.tail()

In [ ]:
# покаждем статистику обучения
print('Максимальное значение метрики ROC AUC на валидационном наборе:',round(optuna_study_lg_meta_pd['roc_valid'].max(),3))
print('Среднее значение метрики ROC AUC на валидационном наборе:',round(optuna_study_lg_meta_pd['roc_valid'].mean(),3))
print('Максимальное значение метрики f1_score на валидационном наборе:',round(optuna_study_lg_meta_pd['f1_score'].max(),3))
print('Среднее значение метрики f1_score на валидационном наборе:',round(optuna_study_lg_meta_pd['f1_score'].mean(),3))
print('Максимальное значение метрики recall_1 на валидационном наборе:',round(optuna_study_lg_meta_pd['recall_1'].max(),3))
print('Среднее значение метрики recall_1 на валидационном наборе:',round(optuna_study_lg_meta_pd['recall_1'].mean(),3))
print('Максимальное значение метрики precision_1 на валидационном наборе:',round(optuna_study_lg_meta_pd['precision_1'].max(),3))
print('Среднее значение метрики precision_1 на валидационном наборе:',round(optuna_study_lg_meta_pd['precision_1'].mean(),3))

In [ ]:
# построим график важности гиперпарметров
optuna.visualization.plot_param_importances(optuna_study_lg_meta, target_name='Score')

In [ ]:
# Построим зависимость метрики precision_1 от гипер параметров

fig = px.scatter(
    data_frame=optuna_study_lg_meta_pd,
    x='precision_1', #ось абсцисс
    y=['f1_score','roc_train', 'roc_valid', 'recall_1'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision_1',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Визуализируем метрики полученные при optuna оптимизации
fig = px.scatter(
    data_frame=optuna_study_lg_meta_pd[(optuna_study_lg_meta_pd['roc_valid']>0.765)],
    x='number', #ось абсцисс
    y=['roc_train','roc_valid','recall_1','precision_1'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость качества модели от trials optuna', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='trials optuna',
    yaxis_title='Величина метрики')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

<span style="color:Blue">

Вывод:  

Их всех выбираем точку $number =833$.   
В этой точке самые оптимальные значения $recall$ и $precision$.  
Посмотрим каким значениям гиперпараметров соотвествует выбранная точка.

In [ ]:
# определим номер лучшге варианта
best_optuna_number = 833

# сформируем словарь лучших гипирпарметров RandomForestClassifier
best_param_lr = {
    'solver' : optuna_study_lg_meta_pd[optuna_study_lg_meta_pd['number']==best_optuna_number]['params_solver'].iloc[0],
    'C' : round(optuna_study_lg_meta_pd[optuna_study_lg_meta_pd['number']==best_optuna_number]['params_C'].iloc[0],3),
    'class_weight' : {0:round(optuna_study_lg_meta_pd[optuna_study_lg_meta_pd['number']==best_optuna_number]['params_class_0_weight'].iloc[0],3),
                      1:round(optuna_study_lg_meta_pd[optuna_study_lg_meta_pd['number']==best_optuna_number]['params_class_1_weight'].iloc[0],3)},
    }

# создадим перменные
best_k_features = optuna_study_lg_meta_pd[optuna_study_lg_meta_pd['number']==best_optuna_number]['params_k_features'].iloc[0]
best_random_state = optuna_study_lg_meta_pd[optuna_study_lg_meta_pd['number']==best_optuna_number]['params_random_state'].iloc[0]
best_class_1_percent = optuna_study_lg_meta_pd[optuna_study_lg_meta_pd['number']==best_optuna_number]['params_class_1_percent'].iloc[0]
best_train_size = optuna_study_lg_meta_pd[optuna_study_lg_meta_pd['number']==best_optuna_number]['params_train_size'].iloc[0]

# Выведем принятые наилучшие праметры
print('best solver:',best_param_lr['solver'])
print('best C:',best_param_lr['C'])
print('best class 0 weight:',best_param_lr['class_weight'][0])
print('best class 1 weight:',best_param_lr['class_weight'][1])

print('best class 1 percent:',round(best_class_1_percent,3))
print('best k features:',best_k_features)
print('best random state:',best_random_state)
print('best train size:',best_train_size)
print('time for best train:',round(optuna_study_lg_meta_pd[optuna_study_lg_meta_pd['number']==best_optuna_number]['duration'].iloc[0].seconds/60),'minutes')

print()
print('ROC AUC на обучающем наборе:', round(optuna_study_lg_meta_pd[optuna_study_lg_meta_pd['number']==best_optuna_number]['roc_train'].iloc[0],3))
print('ROC AUC на валидационном наборе:', round(optuna_study_lg_meta_pd[optuna_study_lg_meta_pd['number']==best_optuna_number]['roc_valid'].iloc[0],3))
print('precision класса 1:', round(optuna_study_lg_meta_pd[optuna_study_lg_meta_pd['number']==best_optuna_number]['precision_1'].iloc[0],3))
print('recall класса 1:', round(optuna_study_lg_meta_pd[optuna_study_lg_meta_pd['number']==best_optuna_number]['recall_1'].iloc[0],3))


##### <span style="color:MediumSlateBlue">Обучение модели с лучшими параметрами</span>

In [ ]:
features_second_meta = fp.ParquetFile('second_metamodels/features/features_second_meta').to_pandas()

# Выделим из features_second_meta тренировочную часть
MX_train_pd = my_train_test_split(features_second_meta.set_index('id').drop(['flag'],axis=1),
                                                        features_second_meta.set_index('id')['flag'], 
                                                        train_size=best_train_size, random_state=best_random_state)[0]
my_train_pd = my_train_test_split(features_second_meta.set_index('id').drop(['flag'],axis=1),
                                                        features_second_meta.set_index('id')['flag'], 
                                                        train_size=best_train_size, random_state=best_random_state)[1].reset_index()

# поготовим данные my_train_pd к работе с функцией class_1_percent_samples
my_train_pd.set_index('id',drop=False,inplace=True)

# с помощью функции class_1_percent_samples зададим долю класса 1
list_c1_percent_id = class_1_percent_samples(my_train_pd,best_class_1_percent,random_state=best_random_state)[:1000000]

# подготовим данные для обучения
MX_train_balanced = MX_train_pd.loc[list_c1_percent_id]
my_train_balanced = my_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

# освободим память от "тяжелых" и ненужных файлов
del MX_train_pd, my_train_pd
gc.collect()

# с помощью класса SelectKBest получим список лучших признаков
selector = SelectKBest(f_classif, k=best_k_features)
selector.fit(MX_train_balanced, my_train_balanced)
list_best_features = selector.get_feature_names_out()

MX_train_balanced = MX_train_balanced[list_best_features].to_numpy()


# обучаем модель LogisticRegression с наилучшеми параметрами
logistic_regression = linear_model.LogisticRegression(
        **best_param_lr,
        random_state=best_random_state,
        max_iter=10000)
logistic_regression.fit(MX_train_balanced, my_train_balanced)

# удаляем крупные файлы чтобы высвободить память 
del MX_train_balanced, my_train_balanced
gc.collect()

# загружаем тестовые наборы
MX_train = my_train_test_split(features_second_meta.set_index('id').drop(['flag'],axis=1),
                                                        features_second_meta.set_index('id')['flag'], 
                                                        train_size=best_train_size, random_state=best_random_state)[0][list_best_features].to_numpy()
my_train = my_train_test_split(features_second_meta.set_index('id').drop(['flag'],axis=1),
                                                        features_second_meta.set_index('id')['flag'], 
                                                        train_size=best_train_size, random_state=best_random_state)[1].to_numpy()

MX_valid = my_train_test_split(features_second_meta.set_index('id').drop(['flag'],axis=1),
                                                        features_second_meta.set_index('id')['flag'], 
                                                        train_size=best_train_size, random_state=best_random_state)[2][list_best_features].to_numpy()
my_valid = my_train_test_split(features_second_meta.set_index('id').drop(['flag'],axis=1),
                                                        features_second_meta.set_index('id')['flag'], 
                                                        train_size=best_train_size, random_state=best_random_state)[3].to_numpy()

# для метрик ROC AUC делаем предсказание модели в виде вероятности
my_train_lr_pred_proba = logistic_regression.predict_proba(MX_train)[:,1]
my_valid_lr_pred_proba = logistic_regression.predict_proba(MX_valid)[:,1]

# Делаем предсказание для валидационной выборки
my_valid_lr_pred = logistic_regression.predict(MX_valid)

# удаляем крупные файлы чтобы высвободить память 
del MX_train, MX_valid
gc.collect()

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(my_train, my_train_lr_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(my_valid, my_valid_lr_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(my_valid,my_valid_lr_pred,zero_division=0))

# time: 2m 30s

In [ ]:
# сохраним модель для дальнейшего воспроизведения
dump(logistic_regression, 'second_metamodels/models/best_model.joblib')
# сохраним SelectKBest для дальнейшего воспроизведения
dump(selector, 'second_metamodels/selector/best_selector.joblib')

##### <span style="color:MediumSlateBlue">Анализ влияния порога классификации на качество модели</span>

In [ ]:
# чтобы проше обращаться к каждому обьекту в данных
# преобразуем y_valid_LR_pred к Series типу
my_valid_lr_pred_proba = pd.Series(my_valid_lr_pred_proba)

# формируем список из необходимых значений thresholds
thresholds = np.arange(0.01, 1, 0.01)

# сформируем списко для заполнения данными результатов анализа
threshold_data = []


for threshold in thresholds:
        # сформируем список для заполнения данными текушего threshold
        threshold_current_data = [threshold]

        #клиентов, для которых вероятность дефолта > threshold, относим к классу 1
        #в противном случае — к классу 0
        my_valid_lr_pred = my_valid_lr_pred_proba.apply(lambda x: 1 if x>threshold else 0)


        #Считаем метрики для класса 0 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(my_valid, my_valid_lr_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.precision_score(my_valid, my_valid_lr_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.f1_score(my_valid, my_valid_lr_pred,average=None,zero_division=0)[0],3))

        #Считаем метрики для класса 1 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(my_valid, my_valid_lr_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.precision_score(my_valid, my_valid_lr_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.f1_score(my_valid, my_valid_lr_pred,average=None,zero_division=0)[1],3))

        # добавляем полученные данные в список threshold_data
        threshold_data.append(threshold_current_data)

        # из полученных данных сформируем dataframe
        thresholds_pd =pd.DataFrame(
        columns=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'],
        data = threshold_data)

# time: 17s

In [ ]:
# Визуализируем метрики на валидационном наборе при различных threshold
fig_threshold = px.line(
    data_frame=thresholds_pd,
    x='threshold', #ось абсцисс
    y=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'], #ось ординат
)
fig_threshold.update_layout(
    title ={
        'text':'Зависимость качества модели от threshold', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Порог классификации threshold',
    yaxis_title='Величина метрики')
fig_threshold.update_xaxes(showspikes=True)
fig_threshold.update_yaxes(showspikes=True)

fig_threshold.show()

Метрика $precision$ показывает на сколько сильно ошиблась модель при определиние класса.  
Чем выше метрика, тем меньше ошиблась модель. 

$$precision = \frac{TP}{TP+FP}$$

Метрика $recall$ показывает способность модели найти класс.  
Чем выше метрика, тем выше способность модели находить класс.  

$$recall = \frac{TP}{TP+FN}$$

Метрика $f1-score$ показывает баланс между $precision$ и $recall$ 

$threshold$- порог класификации модели.  
Все объекты для которых $у_{\text{proba pred}} < threshold$, модель относит к класс 0.  
Соотвественно, если $у_{\text{proba pred}}  > threshold$ модель относит обьект к классу 1.  
Для идельаной модели $threshold$, является границей между областями класса 0 и класса 1.  

<span style="color:Blue">

Выводы по результам анализа:  

Для улучшения качества модели по метрике $ROC AUC$, нет смысла сдвигать порог   
классификации. Но зависимость метрик качества модели от $threshold$, может рассказать о  
спосбоности модели искать класс 1 и класс 0.

При фокусировке модели на классе 1 (более низкий $threshold$):  
- на классе 1 метрика $precision$ уменьшается, метрика $recall$ растет;  
- на классе 0 метрика $precision$ растет, метрика $recall$ уменьшается.  

При фокусировке модели на классе 0 (более высокий $threshold$):  
- на классе 1 метрика $precision$ увеличивается, метрика $recall$ уменьшается;  
- на классе 0 метрика $precision$ уменьшается, метрика $recall$ растет.  

Обьсняется тем, что в условия дисбаланса модель научилась хорошо определять класс 0 и плохо класс 1.  

При фокусировке модели на классе 1 (более низкий $threshold$), все больше обьектов отнесены к классу 1.   
Так как $precision$ класса 1 уменьшается, значит в области класса 1 стало больше обектов класса 0,  
что явлется закономерным для "идеальной" модели.  
Однако при этом, $recall$ класса 1 растет, значит в добавленных обьектах также присутвует класс 1.  
Это как раз указывает на то, что модель не научилась находить среди класса 0 класс 1. В области с     
высокой вероятностью класса 0 (более низкий $threshold$), находятся обьекты класса 1.  

Фокусировка на классе 0 (более высокий $threshold$), заставляет модель все больше обьектов из области класса 1  
относить к классу 0. При этом увеличение метрики $precision$ класса 1 говорит о том, что в области класса 1     
становится меньше 0, а значит модель умеет среди класса 1, находить класс 0.  
После  $threshold=0.88$ модель впринципе теряет способность отличать класс 1. У модели нет обьектов класса 1   
в которых она "уверена" на 92+%.

Обратим внимание на $threshold=0.52$, в этой точке находится баланс межу метриками качества модели:
$$precision_1 = recall_1 = \text{f1-score}_1 = 0.163$$
$$precision_0 = recall_0 = \text{f1-score}_0 = 0.968$$

Для модели с лучшим качеством эти две точки должны находится максимально близко к друг другу и иметь  
высокое значение метрик $precision_1$ и $recall_1$.
Для класса 0 это условие выполнется, что также говорит о том, что модель научилась искать этот класс.  
Для класса 1 это условие не выполнется, что также говорит о том, что модель плохо ищет класс 1.

##### <span style="color:MediumSlateBlue">Построение ROC-кривой выбранной модели на тестовом наборе</span>

In [ ]:
# загружаем тестовые наборы
MX_train = fp.ParquetFile('second_metamodels/features/features_second_meta_train').to_pandas()[list_best_features].to_numpy()
my_train = pd.read_csv('second_metamodels/data/msy_train.csv').set_index('id')['flag'].to_numpy()
MX_valid = fp.ParquetFile('second_metamodels/features/features_second_meta_valid').to_pandas()[list_best_features].to_numpy()
my_valid = pd.read_csv('second_metamodels/data/msy_valid.csv').set_index('id')['flag'].to_numpy()
MX_test = fp.ParquetFile('second_metamodels/features/features_second_meta_test').to_pandas()[list_best_features].to_numpy()
my_test = pd.read_csv('second_metamodels/data/msy_test.csv').set_index('id')['flag'].to_numpy()

In [ ]:
print("Размеры обучающего набора",MX_train.shape)
print("Размеры валидационного набора",MX_valid.shape)
print("Размеры тестового набора",MX_test.shape)
# time: 1m 43s

In [ ]:
# для метрик ROC AUC делаем предсказание модели в виде вероятности
my_train_lr_pred_proba = logistic_regression.predict_proba(MX_train)[:,1]
my_valid_lr_pred_proba = logistic_regression.predict_proba(MX_valid)[:,1]
my_test_lr_pred_proba = logistic_regression.predict_proba(MX_test)[:,1]

# Делаем предсказание для валидационной выборки
my_test_lr_pred = logistic_regression.predict(MX_test)

del MX_train, MX_valid, MX_test
gc.collect()

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(my_train, my_train_lr_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(my_valid, my_valid_lr_pred_proba),3))
print('ROC AUC на тестовом наборе', round(metrics.roc_auc_score(my_test, my_test_lr_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(my_test,my_test_lr_pred,zero_division=0))

In [ ]:
# Для расчета значений TPR и FPR используем функцию roc_curve
fpr_train, tpr_train, _ = metrics.roc_curve(my_train, my_train_lr_pred_proba)
fpr_valid, tpr_valid, _ = metrics.roc_curve(my_valid, my_valid_lr_pred_proba)
fpr_test, tpr_test, _ = metrics.roc_curve(my_test, my_test_lr_pred_proba)

# рассчитаем площадь под кривой
auc_train = round(metrics.roc_auc_score(my_train, my_train_lr_pred_proba),3)
auc_valid = round(metrics.roc_auc_score(my_valid, my_valid_lr_pred_proba),3)
auc_test = round(metrics.roc_auc_score(my_test, my_test_lr_pred_proba),3)

trace_train = go.Scatter(x=fpr_train, y=tpr_train, mode='lines', name='AUC_train = %0.2f' % auc_train,
                   line=dict(color='red', width=2),
                   hovertemplate='train<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_train])

trace_valid = go.Scatter(x=fpr_valid, y=tpr_valid, mode='lines', name='AUC_valid = %0.2f' % auc_valid,
                   line=dict(color='green', width=2),
                   hovertemplate='valid<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_valid])

trace_test = go.Scatter(x=fpr_test, y=tpr_test, mode='lines', name='AUC_test = %0.2f' % auc_test,
                   line=dict(color='darkorange', width=2),
                   hovertemplate='test<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_test])


reference_line = go.Scatter(x=[0,1], y=[0,1], mode='lines', name='Reference Line',
                            line=dict(color='navy', width=2, dash='dash'))

fig_roc = go.Figure(data=[trace_train,trace_valid,trace_test,reference_line])

fig_roc.update_layout(
    title ={
        'text':'ROC-кривая', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 1350,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate')

fig_roc.update_xaxes(showspikes=True)
fig_roc.update_yaxes(showspikes=True)


fig_roc.show()

<span style="color:Blue">

Вывод по ROC-кривой:
1. Модель выдает сталибильное качество на всех наборах, а значит  
не переобучена - разброс ($variance$) не большой.  
2. На тестовом наборе полученно качество $ROC AUC = 0.76$.

#### <span style="color:MediumBlue">Построение метамодели Random Forest Classifier</span>

##### <span style="color:MediumSlateBlue">Baseline обучение метамодели RandomForestClassifier</span>

In [ ]:
MX_train = fp.ParquetFile('second_metamodels/features/features_second_meta_train').to_pandas()
my_train = pd.read_csv('second_metamodels/data/msy_train.csv').set_index('id')['flag']
MX_valid = fp.ParquetFile('second_metamodels/features/features_second_meta_valid').to_pandas()
my_valid = pd.read_csv('second_metamodels/data/msy_valid.csv').set_index('id')['flag']

In [ ]:
# обучаем модель логистической регрессии
# чтобы модель пыталась найти связь во всех признаках
random_forest = RandomForestClassifier(random_state=42, n_jobs=-1)
random_forest.fit(MX_train,my_train)
# для метрик ROC AUC делаем предсказание модели в виде вероятности
my_train_rf__pred_proba = random_forest.predict_proba(MX_train)[:,1]
my_valid_rf_pred_proba = random_forest.predict_proba(MX_valid)[:,1]

# Делаем предсказание для валидационной выборки
my_valid_rf_pred = random_forest.predict(MX_valid)

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(my_train, my_train_rf__pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(my_valid, my_valid_rf_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(my_valid,my_valid_rf_pred))

# time: 17s

<span style="color:Blue">

Выводы:

В сравнение с моделью $Logistic$ $Regression$ качество модели  
и время обучения заметно хуже.

##### <span style="color:MediumSlateBlue">Подбор гиперпараметров модели</span>

In [ ]:
# настроим оптимизацию гипер параметров
def optuna_rf_meta(trial):
  # задаем пространства поиска гиперпараметров
  # задаем пространства поиска гиперпараметров
  n_estimators = trial.suggest_int('n_estimators', 40, 100,step = 5)
  criterion = trial.suggest_categorical('criterion',['gini', 'entropy', 'log_loss'])
  max_depth = trial.suggest_int('max_depth', 10, 30,step = 1)
  min_samples_split = trial.suggest_int('min_samples_split', 5, 15,step = 1)
  min_samples_leaf = trial.suggest_int('min_samples_leaf', 5, 15,step = 1)
  max_features = trial.suggest_float('max_features',0.4,0.8)
  max_samples = trial.suggest_float('max_samples',0.4,0.8)
  class_0_weight = trial.suggest_float('class_0_weight',0.1,1)
  class_1_weight = trial.suggest_float('class_1_weight',0.1,1)
  class_1_percent = trial.suggest_float('class_1_percent',0.01,1)
  k_features = trial.suggest_int('k_features', 20, 88,step = 1)
  random_state = trial.suggest_int('random_state', 1, 1000000)
  train_size = trial.suggest_float('train_size',0.7,0.9)

  # создаем модель
  optuna_random_forest_meta = RandomForestClassifier(
      n_estimators = n_estimators, 
      criterion = criterion,
      max_depth = max_depth,
      min_samples_split = min_samples_split,  
      min_samples_leaf = min_samples_leaf,
      max_features=max_features,
      max_samples=max_samples,
      class_weight={0:class_0_weight,1:class_1_weight},
      n_jobs=-1,
      random_state=random_state)
  
  features_second_meta = fp.ParquetFile('second_metamodels/features/features_second_meta').to_pandas()

  # Выделим из features_second_meta тренировочную часть
  MX_train_pd = my_train_test_split(features_second_meta.set_index('id').drop(['flag'],axis=1),
                                                           features_second_meta.set_index('id')['flag'], 
                                                           train_size=train_size, random_state=random_state)[0]
  my_train_pd = my_train_test_split(features_second_meta.set_index('id').drop(['flag'],axis=1),
                                                           features_second_meta.set_index('id')['flag'], 
                                                           train_size=train_size, random_state=random_state)[1].reset_index()

  # поготовим данные my_train_pd к работе с функцией class_1_percent_samples
  my_train_pd.set_index('id',drop=False,inplace=True)

  # с помощью функции class_1_percent_samples зададим долю класса 1
  list_c1_percent_id = class_1_percent_samples(my_train_pd,class_1_percent,random_state=random_state)[:1000000]

  # подготовим данные для обучения
  MX_train_balanced = MX_train_pd.loc[list_c1_percent_id]
  my_train_balanced = my_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

  # освободим память от "тяжелых" и ненужных файлов
  del MX_train_pd, my_train_pd
  gc.collect()
  
  # с помощью класса SelectKBest получим список лучших признаков
  selector = SelectKBest(f_classif, k=k_features)
  selector.fit(MX_train_balanced, my_train_balanced)
  list_best_features = selector.get_feature_names_out()

  MX_train_balanced = MX_train_balanced[list_best_features].to_numpy()
  
  # я не воспользовался параметром timeout метода optimize потому, что  
  # optimize отставливает поиск параметров, если время обучения 
  # превышает timeout. Мне нужно чтобы поиск параметров продолжился.
  try:
    # для модуля func_time_out упакуем обучение модели в функцию try_func
    def try_func():
            return optuna_random_forest_meta.fit(MX_train_balanced, my_train_balanced)
    # обучаем модель с ограничением по времени 15 минут (900 секунд)
    optuna_random_forest_meta = func_timeout.func_timeout(900, try_func)

    # удаляем крупные файлы чтобы высвободить память 
    del MX_train_balanced, my_train_balanced
    gc.collect()

    # загружаем тестовые наборы
    MX_train = my_train_test_split(features_second_meta.set_index('id').drop(['flag'],axis=1),
                                                           features_second_meta.set_index('id')['flag'], 
                                                           train_size=train_size, random_state=random_state)[0][list_best_features].to_numpy()
    my_train = my_train_test_split(features_second_meta.set_index('id').drop(['flag'],axis=1),
                                                           features_second_meta.set_index('id')['flag'], 
                                                           train_size=train_size, random_state=random_state)[1].to_numpy()
    
    MX_valid = my_train_test_split(features_second_meta.set_index('id').drop(['flag'],axis=1),
                                                           features_second_meta.set_index('id')['flag'], 
                                                           train_size=train_size, random_state=random_state)[2][list_best_features].to_numpy()
    my_valid = my_train_test_split(features_second_meta.set_index('id').drop(['flag'],axis=1),
                                                           features_second_meta.set_index('id')['flag'], 
                                                           train_size=train_size, random_state=random_state)[3].to_numpy()

    # делаем предсказание на обучающем и валидационном наборе
    #Считаем метрики для класса 1 добавляем их в список
    roc_train = metrics.roc_auc_score(my_train, optuna_random_forest_meta.predict_proba(MX_train)[:,1])
    roc_valid = metrics.roc_auc_score(my_valid, optuna_random_forest_meta.predict_proba(MX_valid)[:,1])
    f1_score = metrics.f1_score(my_valid, optuna_random_forest_meta.predict(MX_valid))
    recall_1 = metrics.recall_score(my_valid, optuna_random_forest_meta.predict(MX_valid),average=None,zero_division=0)[1]
    precision_1 = metrics.precision_score(my_valid, optuna_random_forest_meta.predict(MX_valid),average=None,zero_division=0)[1]

    # удаляем крупные файлы чтобы высвободить память 
    del MX_train, MX_valid, my_train, my_valid
    gc.collect()

  except func_timeout.FunctionTimedOut:
    # удаляем крупные файлы чтобы высвободить память 
    del MX_train_balanced, my_train_balanced, features_second_meta
    gc.collect()
    
    # фиксируем пустые значения метрик
    roc_train = 0
    roc_valid = 0
    f1_score = 0
    recall_1 = 0
    precision_1 = 0
    pass

  return roc_train, roc_valid, f1_score, recall_1, precision_1

In [ ]:
# cоздаем объект исследования
optuna_study_rf_meta = optuna.create_study(study_name="RandomForestClassifier_meta_second", 
                               directions=['maximize','maximize','maximize','maximize','maximize'], 
                               sampler=optuna.samplers.TPESampler(),
                               pruner='Hyperband',
                               storage='sqlite:///optuna_studies.db',
                               load_if_exists=True)
# на случай удаления 
# optuna.delete_study(study_name="RandomForestClassifier_meta_second", storage='sqlite:///optuna_studies.db')

In [ ]:
# ищем лучшую комбинацию гиперпараметров n_trials раз
optuna_study_rf_meta.optimize(optuna_rf_meta, n_trials=50)

##### <span style="color:MediumSlateBlue">Анализ гиперпараметров</span>

In [ ]:
# из полученного результат соврмируем Data Frame
optuna_study_rf_meta_pd = optuna_study_rf_meta.trials_dataframe().dropna()
# переименуем столбы
optuna_study_rf_meta_pd.rename(columns={
    'values_0': 'roc_train',
    'values_1': 'roc_valid',
    'values_2': 'f1_score',
    'values_3': 'recall_1',
    'values_4': 'precision_1'
},inplace=True)
optuna_study_rf_meta_pd.tail()

In [ ]:
# покаждем статистику обучения
print('Максимальное значение метрики ROC AUC на валидационном наборе:',round(optuna_study_rf_meta_pd['roc_valid'].max(),3))
print('Среднее значение метрики ROC AUC на валидационном наборе:',round(optuna_study_rf_meta_pd['roc_valid'].mean(),3))
print('Максимальное значение метрики f1_score на валидационном наборе:',round(optuna_study_rf_meta_pd['f1_score'].max(),3))
print('Среднее значение метрики f1_score на валидационном наборе:',round(optuna_study_rf_meta_pd['f1_score'].mean(),3))
print('Максимальное значение метрики recall_1 на валидационном наборе:',round(optuna_study_rf_meta_pd['recall_1'].max(),3))
print('Среднее значение метрики recall_1 на валидационном наборе:',round(optuna_study_rf_meta_pd['recall_1'].mean(),3))
print('Максимальное значение метрики precision_1 на валидационном наборе:',round(optuna_study_rf_meta_pd['precision_1'].max(),3))
print('Среднее значение метрики precision_1 на валидационном наборе:',round(optuna_study_rf_meta_pd['precision_1'].mean(),3))

In [ ]:
# Построим зависимость метрики precision_1 от гиперпараметров

fig = px.scatter(
    data_frame=optuna_study_rf_meta_pd,
    x='precision_1', #ось абсцисс
    y=['roc_train', 'roc_valid', 'recall_1','f1_score'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision класса 1 от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision на классе 1',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрик качества модели от number

fig = px.scatter(
    data_frame=optuna_study_rf_meta_pd[(optuna_study_rf_meta_pd['recall_1']>=0.135) & (optuna_study_rf_meta_pd['precision_1']>0.15)],
    x='number', #ось абсцисс
    y=['roc_valid','roc_train','precision_1','recall_1'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость метрик качества модели от number', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики ROC Valid',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

<span style="color:Blue">

Вывод:  

Их всех выбираем точку $number = 93$.   
Не смотря на, то что в этой точке модель подает признаки переобучеености,  
в этой точке оптимальные значения метрик $recall_1$ и $precision_1$, а  
также относительно высокая метрика $ROC AUC$.   
Посмотрим каким значениям гиперпараметров соотвествует выбраннаяточка.

In [ ]:
# определим номер лучшге варианта
best_optuna_number = 93

# сформируем словарь лучших гипирпарметров RandomForestClassifier
best_param_rf = {
    'n_estimators' : int(optuna_study_rf_meta_pd[optuna_study_rf_meta_pd['number']==best_optuna_number]['params_n_estimators'].iloc[0]),
    'criterion': optuna_study_rf_meta_pd[optuna_study_rf_meta_pd['number']==best_optuna_number]['params_criterion'].iloc[0],
    'max_depth' : int(optuna_study_rf_meta_pd[optuna_study_rf_meta_pd['number']==best_optuna_number]['params_max_depth'].iloc[0]),
    'min_samples_split': int(optuna_study_rf_meta_pd[optuna_study_rf_meta_pd['number']==best_optuna_number]['params_min_samples_split'].iloc[0]),
    'min_samples_leaf' : int(optuna_study_rf_meta_pd[optuna_study_rf_meta_pd['number']==best_optuna_number]['params_min_samples_leaf'].iloc[0]),
    'max_features': optuna_study_rf_meta_pd[optuna_study_rf_meta_pd['number']==best_optuna_number]['params_max_features'].iloc[0],
    'max_samples' : optuna_study_rf_meta_pd[optuna_study_rf_meta_pd['number']==best_optuna_number]['params_max_samples'].iloc[0],
    'class_weight' : {0:optuna_study_rf_meta_pd[optuna_study_rf_meta_pd['number']==best_optuna_number]['params_class_0_weight'].iloc[0],
                      1:optuna_study_rf_meta_pd[optuna_study_rf_meta_pd['number']==best_optuna_number]['params_class_1_weight'].iloc[0]}
    }

# определим перменные для лучших значений параметров
best_k_features = optuna_study_rf_meta_pd[optuna_study_rf_meta_pd['number']==best_optuna_number]['params_k_features'].iloc[0]
best_random_state = optuna_study_rf_meta_pd[optuna_study_rf_meta_pd['number']==best_optuna_number]['params_random_state'].iloc[0]
best_class_1_percent = optuna_study_rf_meta_pd[optuna_study_rf_meta_pd['number']==best_optuna_number]['params_class_1_percent'].iloc[0]


# Выведем принятые наилучшие праметры
print('best n estimators:',best_param_rf['n_estimators'])
print('best criterion:',best_param_rf['criterion'])
print('best max depth:',best_param_rf['max_depth'])
print('best min samples split:',best_param_rf['min_samples_split'])
print('best min samples leaf:',best_param_rf['min_samples_leaf'])
print('best max features(%):',round(best_param_rf['max_features'],3))
print('best max samples(%):',round(best_param_rf['max_samples'],3))
print('best class 0 weight:',round(best_param_rf['class_weight'][0],3))
print('best class 1 weight:',round(best_param_rf['class_weight'][1],3))

print('best class 1 percent:',round(best_class_1_percent,3))
print('best k features:',best_k_features)
print('best random state:',best_random_state)
print('time for best train:',round(optuna_study_rf_meta_pd[optuna_study_rf_meta_pd['number']==best_optuna_number]['duration'].iloc[0].seconds/60),'minutes')

print()
print('ROC AUC на обучающем наборе:', round(optuna_study_rf_meta_pd[optuna_study_rf_meta_pd['number']==best_optuna_number]['roc_train'].iloc[0],3))
print('ROC AUC на валидационном наборе:', round(optuna_study_rf_meta_pd[optuna_study_rf_meta_pd['number']==best_optuna_number]['roc_valid'].iloc[0],3))
print('precision класса 1:', round(optuna_study_rf_meta_pd[optuna_study_rf_meta_pd['number']==best_optuna_number]['precision_1'].iloc[0],3))
print('recall класса 1:', round(optuna_study_rf_meta_pd[optuna_study_rf_meta_pd['number']==best_optuna_number]['recall_1'].iloc[0],3))

##### <span style="color:MediumSlateBlue">Обучение модели с лучшими параметрами</span>

In [ ]:
# Загружаем обучающие наборы
MX_train_pd = fp.ParquetFile('second_metamodels/features/features_second_meta_train').to_pandas()
my_train_pd = pd.read_csv('second_metamodels/data/msy_train.csv')

# поготовим данные my_train_pd к работе с функцией class_1_percent_samples
my_train_pd.set_index('id',drop=False,inplace=True)

# с помощью функции class_1_percent_samples зададим долю класса 1
list_c1_percent_id = class_1_percent_samples(my_train_pd,best_class_1_percent,random_state=best_random_state)[:1000000]

# подготовим данные для обучения
MX_train_balanced = MX_train_pd.loc[list_c1_percent_id]
my_train_balanced = my_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

# освободим память от "тяжелых" и ненужных файлов
del MX_train_pd, my_train_pd
gc.collect()

# с помощью класса SelectKBest получим список лучших признаков
selector = SelectKBest(f_classif, k=best_k_features)
selector.fit(MX_train_balanced, my_train_balanced)
list_best_features = selector.get_feature_names_out()

MX_train_balanced = MX_train_balanced[list_best_features].to_numpy()


# обучаем модель RandomForestClassifier с наилучшеми параметрами
random_forest = RandomForestClassifier(
        **best_param_rf,
        n_jobs=-1,
        random_state=best_random_state)
random_forest.fit(MX_train_balanced, my_train_balanced)

# удаляем крупные файлы чтобы высвободить память 
del MX_train_balanced, my_train_balanced
gc.collect()

# загружаем тестовые наборы
MX_train = fp.ParquetFile('second_metamodels/features/features_second_meta_train').to_pandas()[list_best_features].to_numpy()
my_train = pd.read_csv('second_metamodels/data/msy_train.csv').set_index('id')['flag'].to_numpy()
MX_valid = fp.ParquetFile('second_metamodels/features/features_second_meta_valid').to_pandas()[list_best_features].to_numpy()
my_valid = pd.read_csv('second_metamodels/data/msy_valid.csv').set_index('id')['flag'].to_numpy()

# для метрик ROC AUC делаем предсказание модели в виде вероятности
my_train_rf_pred_proba = random_forest.predict_proba(MX_train)[:,1]
my_valid_rf_pred_proba = random_forest.predict_proba(MX_valid)[:,1]

# Делаем предсказание для валидационной выборки
my_valid_rf_pred = random_forest.predict(MX_valid)

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(my_train, my_train_rf_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(my_valid, my_valid_rf_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(my_valid,my_valid_rf_pred,zero_division=0))

# time: 5m 30s

##### <span style="color:MediumSlateBlue">Анализ влияния порога классификации на качество модели</span>

In [ ]:
# чтобы проше обращаться к каждому обьекту в данных
# преобразуем y_valid_LR_pred к Series типу
my_valid_rf_pred_proba = pd.Series(my_valid_rf_pred_proba)

# формируем список из необходимых значений thresholds
thresholds = np.arange(0.01, 1, 0.01)

# сформируем списко для заполнения данными результатов анализа
threshold_data = []


for threshold in thresholds:
        # сформируем список для заполнения данными текушего threshold
        threshold_current_data = [threshold]

        #клиентов, для которых вероятность дефолта > threshold, относим к классу 1
        #в противном случае — к классу 0
        my_valid_rf_pred = my_valid_rf_pred_proba.apply(lambda x: 1 if x>threshold else 0)


        #Считаем метрики для класса 0 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(my_valid, my_valid_rf_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.precision_score(my_valid, my_valid_rf_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.f1_score(my_valid, my_valid_rf_pred,average=None,zero_division=0)[0],3))

        #Считаем метрики для класса 1 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(my_valid, my_valid_rf_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.precision_score(my_valid, my_valid_rf_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.f1_score(my_valid, my_valid_rf_pred,average=None,zero_division=0)[1],3))

        # добавляем полученные данные в список threshold_data
        threshold_data.append(threshold_current_data)

        # из полученных данных сформируем dataframe
        thresholds_pd =pd.DataFrame(
        columns=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'],
        data = threshold_data)

# time: 17s

In [ ]:
# Визуализируем метрики на валидационном наборе при различных threshold
fig_threshold = px.line(
    data_frame=thresholds_pd,
    x='threshold', #ось абсцисс
    y=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'], #ось ординат
)
fig_threshold.update_layout(
    title ={
        'text':'Зависимость качества модели от threshold', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Порог классификации threshold',
    yaxis_title='Величина метрики')
fig_threshold.update_xaxes(showspikes=True)
fig_threshold.update_yaxes(showspikes=True)

fig_threshold.show()

Метрика $precision$ показывает на сколько сильно ошиблась модель при определиние класса.  
Чем выше метрика, тем меньше ошиблась модель. 

$$precision = \frac{TP}{TP+FP}$$

Метрика $recall$ показывает способность модели найти класс.  
Чем выше метрика, тем выше способность модели находить класс.  

$$recall = \frac{TP}{TP+FN}$$

Метрика $f1-score$ показывает баланс между $precision$ и $recall$ 

$threshold$- порог класификации модели.  
Все объекты для которых $у_{\text{proba pred}} < threshold$, модель относит к класс 0.  
Соотвественно, если $у_{\text{proba pred}}  > threshold$ модель относит обьект к классу 1.  
Для идельаной модели $threshold$, является границей между областями класса 0 и класса 1.  

<span style="color:Blue">

Выводы по результам анализа:  

Для улучшения качества модели по метрике $ROC AUC$, нет смысла сдвигать порог   
классификации. Но зависимость метрик качества модели от $threshold$, может рассказать о  
способности модели искать класс 1 и класс 0.

При фокусировке модели на классе 1 (более низкий $threshold$):  
- на классе 1 метрика $precision$ уменьшается, метрика $recall$ растет;  
- на классе 0 метрика $precision$ растет, метрика $recall$ уменьшается.  

При фокусировке модели на классе 0 (более высокий $threshold$):  
- на классе 1 метрика $precision$ увеличивается, метрика $recall$ уменьшается;  
- на классе 0 метрика $precision$ уменьшается, метрика $recall$ растет.  

Обьсняется тем, что в условия дисбаланса модель научилась хорошо определять класс 0 и плохо класс 1.  

При фокусировке модели на классе 1 (более низкий $threshold$), все больше обьектов отнесены к классу 1.   
Так как $precision$ класса 1 уменьшается, значит в области класса 1 стало больше обектов класса 0,  
что явлется закономерным для "идеальной" модели.  
Однако при этом, $recall$ класса 1 растет, значит в добавленных обьектах также присутвует класс 1.  
Это как раз указывает на то, что модель не нучилась находить среди класса 0 класс 1. В области с     
высокой вероятностью класса 0 (более низкий $threshold$), находятся обьекты класса 1.  

Фокусировка на классе 0 (более высокий $threshold$), заставляет модель все больше обьектов из области класса 1  
относить к классу 0. При этом увеличение метрики $precision$ класса 1 говорит о том, что в области класса 1     
становится меньше 0, а значит модель умеет среди класса 1, находить класс 0.  
После  $threshold=0.92$ модель впринципе теряет способность отличать класс 1. У модели нет обьектов класса 1   
в которых она "уверена" на 95+%.

Обратим внимание на $threshold=0.52$, в этой точке находится баланс межу метриками качества модели:
$$precision_1 = recall_1 = \text{f1-score}_1 = 0.16$$
$$precision_0 = recall_0 = \text{f1-score}_0 = 0.968$$

Для модели с лучшим качеством эти две точки должны находится максимально близко к друг другу и иметь   
высокое значение метрик $precision_1$ и $recall_1$.  
Для класса 0 это условие выполнется, что также говорит о том, что модель научилась искать этот класс.    
Для класса 1 это условие не выполнется, что также говорит о том, что модель плохо ищет класс 1.

##### <span style="color:MediumSlateBlue">Построение ROC-кривой выбранной модели на тестовом наборе</span>

In [ ]:
# загружаем тестовые наборы
MX_train = fp.ParquetFile('second_metamodels/features/features_second_meta_train').to_pandas()[list_best_features].to_numpy()
my_train = pd.read_csv('second_metamodels/data/msy_train.csv').set_index('id')['flag'].to_numpy()
MX_valid = fp.ParquetFile('second_metamodels/features/features_second_meta_valid').to_pandas()[list_best_features].to_numpy()
my_valid = pd.read_csv('second_metamodels/data/msy_valid.csv').set_index('id')['flag'].to_numpy()
MX_test = fp.ParquetFile('second_metamodels/features/features_second_meta_test').to_pandas()[list_best_features].to_numpy()
my_test = pd.read_csv('second_metamodels/data/msy_test.csv').set_index('id')['flag'].to_numpy()

In [ ]:
print("Размеры обучающего набора",MX_train.shape,my_train.shape)
print("Размеры валидационного набора",MX_valid.shape,my_valid.shape)
print("Размеры тестового набора",MX_test.shape,my_test.shape)
# time: 1m 43s

In [ ]:
# для метрик ROC AUC делаем предсказание модели в виде вероятности
my_train_rf_pred_proba = random_forest.predict_proba(MX_train)[:,1]
my_valid_rf_pred_proba = random_forest.predict_proba(MX_valid)[:,1]
my_test_rf_pred_proba = random_forest.predict_proba(MX_test)[:,1]

# Делаем предсказание для валидационной выборки
my_test_lr_pred = random_forest.predict(MX_test)

del MX_train, MX_valid, MX_test

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(my_train, my_train_rf_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(my_valid, my_valid_rf_pred_proba),3))
print('ROC AUC на тестовом наборе', round(metrics.roc_auc_score(my_test, my_test_rf_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(my_test,my_test_lr_pred,zero_division=0))

In [ ]:
# Для расчета значений TPR и FPR используем функцию roc_curve
fpr_train, tpr_train, _ = metrics.roc_curve(my_train, my_train_rf_pred_proba)
fpr_valid, tpr_valid, _ = metrics.roc_curve(my_valid, my_valid_rf_pred_proba)
fpr_test, tpr_test, _ = metrics.roc_curve(my_test, my_test_rf_pred_proba)

# рассчитаем площадь под кривой
auc_train = round(metrics.roc_auc_score(my_train, my_train_rf_pred_proba),3)
auc_valid = round(metrics.roc_auc_score(my_valid, my_valid_rf_pred_proba),3)
auc_test = round(metrics.roc_auc_score(my_test, my_test_rf_pred_proba),3)

trace_train = go.Scatter(x=fpr_train, y=tpr_train, mode='lines', name='AUC_train = %0.2f' % auc_train,
                   line=dict(color='red', width=2),
                   hovertemplate='train<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_train])

trace_valid = go.Scatter(x=fpr_valid, y=tpr_valid, mode='lines', name='AUC_valid = %0.2f' % auc_valid,
                   line=dict(color='green', width=2),
                   hovertemplate='valid<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_valid])

trace_test = go.Scatter(x=fpr_test, y=tpr_test, mode='lines', name='AUC_test = %0.2f' % auc_test,
                   line=dict(color='darkorange', width=2),
                   hovertemplate='test<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_test])


reference_line = go.Scatter(x=[0,1], y=[0,1], mode='lines', name='Reference Line',
                            line=dict(color='navy', width=2, dash='dash'))

fig_roc = go.Figure(data=[trace_train,trace_valid,trace_test,reference_line])

fig_roc.update_layout(
    title ={
        'text':'ROC-кривая', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 1350,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate')

fig_roc.update_xaxes(showspikes=True)
fig_roc.update_yaxes(showspikes=True)


fig_roc.show()

<span style="color:Blue">

Вывод по ROC-кривой:
1. Не смотря на то, что модель показывает, очень хорошую $ROC$ кривую на обучающем наборе,  
модель, так же выдает сталибильное качество на валидационном и тестовом наборах, а значит  
не переобучена - разброс ($variance$) не большой.
2. На тестовом наборе полученно качество $ROC AUC = 0.755$.

#### <span style="color:MediumBlue">Построение модели Gradient Boosting Classifier</span>

##### <span style="color:MediumSlateBlue">Baseline обучение модели Hist Gradient Boosting Classifier</span>

In [ ]:
MX_train = fp.ParquetFile('second_metamodels/features/features_second_meta_train').to_pandas()
my_train = pd.read_csv('second_metamodels/data/msy_train.csv').set_index('id')['flag']
MX_valid = fp.ParquetFile('second_metamodels/features/features_second_meta_valid').to_pandas()
my_valid = pd.read_csv('second_metamodels/data/msy_valid.csv').set_index('id')['flag']

In [ ]:
# обучаем модель Gradient Boosting Classifier
gradient_boosting = HistGradientBoostingClassifier(random_state=42)
gradient_boosting.fit(MX_train,my_train)

# для метрик ROC AUC делаем предсказание модели в виде вероятности
my_train_rf__pred_proba = gradient_boosting.predict_proba(MX_train)[:,1]
my_valid_rf_pred_proba = gradient_boosting.predict_proba(MX_valid)[:,1]

# Делаем предсказание для валидационной выборки
my_valid_rf_pred = gradient_boosting.predict(MX_valid)

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(my_train, my_train_rf__pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(my_valid, my_valid_rf_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(my_valid,my_valid_rf_pred))

# time: 30s

<span style="color:Blue">

Выводы:

В сравнение с моделью $HistGradientBoostingClassifier$ построенной   
на несбалансированных данных $transform$ $data$ $stat$:
1. Качество модели по метрике $ROC AUC$ не изменилась.
2. Качество модели по метрике $recall_1$ значительно  
улучшилось. Правда, $recall_0$ уменьшилось.
3. Качество модели по метрике $precision_1$ уменьшилось.   
4. На порядок уменьшилось время обучения модели. Это логично,  
велична сбалансированной выборки всего $107000$ $samples$. 

##### <span style="color:MediumSlateBlue">Подбор гиперпараметров модели</span>

In [ ]:
# настроим оптимизацию гипер параметров
def optuna_gb_meta(trial):
  # задаем пространства поиска гиперпараметров
  learning_rate = trial.suggest_float('learning_rate',0.01,0.1)
  max_iter = trial.suggest_int('max_iter', 150, 250,step = 1)
  max_leaf_nodes = trial.suggest_int('max_leaf_nodes', 2, 60,step = 1)
  max_depth = trial.suggest_int('max_depth', 1, 10,step = 1)
  min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 60,step = 1)
  max_features = trial.suggest_float('max_features',0.1,1)
  l2_regularization = trial.suggest_float('l2_regularization',0.01,1)
  class_0_weight = trial.suggest_float('class_0_weight',0.01,1)
  class_1_weight = trial.suggest_float('class_1_weight',0.01,1)
  k_features = trial.suggest_int('k_features', 20, 88,step = 1)
  class_1_percent = trial.suggest_float('class_1_percent',0.01,1)
  random_state = trial.suggest_int('random_state', 1, 1000000)
  train_size = trial.suggest_float('train_size',0.7,0.9)

  # создаем модель
  optuna_gradient_boosting_meta = HistGradientBoostingClassifier(
      learning_rate= learning_rate,
      max_iter = max_iter,
      max_leaf_nodes =max_leaf_nodes,
      max_depth = max_depth,
      min_samples_leaf = min_samples_leaf,
      max_features=max_features,
      l2_regularization = l2_regularization,
      class_weight={0:class_0_weight,1:class_1_weight},
      random_state=random_state)

  features_second_meta = fp.ParquetFile('second_metamodels/features/features_second_meta').to_pandas()

  # Выделим из features_second_meta тренировочную часть
  MX_train_pd = my_train_test_split(features_second_meta.set_index('id').drop(['flag'],axis=1),
                                                           features_second_meta.set_index('id')['flag'], 
                                                           train_size=train_size, random_state=random_state)[0]
  my_train_pd = my_train_test_split(features_second_meta.set_index('id').drop(['flag'],axis=1),
                                                           features_second_meta.set_index('id')['flag'], 
                                                           train_size=train_size, random_state=random_state)[1].reset_index()

  # поготовим данные my_train_pd к работе с функцией class_1_percent_samples
  my_train_pd.set_index('id',drop=False,inplace=True)

  # с помощью функции class_1_percent_samples зададим долю класса 1
  list_c1_percent_id = class_1_percent_samples(my_train_pd,class_1_percent,random_state=random_state)[:1000000]

  # подготовим данные для обучения
  MX_train_balanced = MX_train_pd.loc[list_c1_percent_id]
  my_train_balanced = my_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

  # освободим память от "тяжелых" и ненужных файлов
  del MX_train_pd, my_train_pd
  gc.collect()
  
  # с помощью класса SelectKBest получим список лучших признаков
  selector = SelectKBest(f_classif, k=k_features)
  selector.fit(MX_train_balanced, my_train_balanced)
  list_best_features = selector.get_feature_names_out()

  MX_train_balanced = MX_train_balanced[list_best_features].to_numpy()

  # я не воспользовался параметром timeout метода optimize потому, что  
  # optimize отставливает поиск параметров, если время обучения 
  # превышает timeout. Мне нужно чтобы поиск параметров продолжился.
  try:
    # для модуля func_time_out упакуем обучение модели в функцию try_func
    def try_func():
            return optuna_gradient_boosting_meta.fit(MX_train_balanced,my_train_balanced)
    # обучаем модель с ограничением по времени 10 минут (600 секунд)
    optuna_gradient_boosting_meta = func_timeout.func_timeout(600, try_func)

    # удаляем крупные файлы чтобы высвободить память 
    del MX_train_balanced, my_train_balanced
    gc.collect()

    # загружаем тестовые наборы
    MX_train = my_train_test_split(features_second_meta.set_index('id').drop(['flag'],axis=1),
                                                           features_second_meta.set_index('id')['flag'], 
                                                           train_size=train_size, random_state=random_state)[0][list_best_features].to_numpy()
    my_train = my_train_test_split(features_second_meta.set_index('id').drop(['flag'],axis=1),
                                                           features_second_meta.set_index('id')['flag'], 
                                                           train_size=train_size, random_state=random_state)[1].to_numpy()
    
    MX_valid = my_train_test_split(features_second_meta.set_index('id').drop(['flag'],axis=1),
                                                           features_second_meta.set_index('id')['flag'], 
                                                           train_size=train_size, random_state=random_state)[2][list_best_features].to_numpy()
    my_valid = my_train_test_split(features_second_meta.set_index('id').drop(['flag'],axis=1),
                                                           features_second_meta.set_index('id')['flag'], 
                                                           train_size=train_size, random_state=random_state)[3].to_numpy()

    # делаем предсказание на обучающем и валидационном наборе
    #Считаем метрики для класса 1 добавляем их в список
    roc_train = metrics.roc_auc_score(my_train, optuna_gradient_boosting_meta.predict_proba(MX_train)[:,1])
    roc_valid = metrics.roc_auc_score(my_valid, optuna_gradient_boosting_meta.predict_proba(MX_valid)[:,1])
    f1_score = metrics.f1_score(my_valid, optuna_gradient_boosting_meta.predict(MX_valid))
    recall_1 = metrics.recall_score(my_valid, optuna_gradient_boosting_meta.predict(MX_valid),average=None,zero_division=0)[1]
    precision_1 = metrics.precision_score(my_valid, optuna_gradient_boosting_meta.predict(MX_valid),average=None,zero_division=0)[1]

    # удаляем крупные файлы чтобы высвободить память 
    del MX_train, MX_valid, my_train, my_valid, features_second_meta
    gc.collect()

  except func_timeout.FunctionTimedOut:
    # удаляем крупные файлы чтобы высвободить память 
    del MX_train_balanced, my_train_balanced
    gc.collect()
    
    # фиксируем пустые значения метрик
    roc_train = 0
    roc_valid = 0
    f1_score = 0
    recall_1 = 0
    precision_1 = 0
    pass

  return roc_train, roc_valid, f1_score, recall_1, precision_1

In [ ]:
# cоздаем объект исследования
# чтобы модель не переобучалась минимизируем roc_train 
# и максимизируем roc_valid
optuna_study_gb_meta = optuna.create_study(study_name="HistGradientBoostingClassifier_meta_second", 
                               directions=['maximize','maximize','maximize','maximize','maximize'], 
                               sampler=optuna.samplers.TPESampler(),
                               pruner='Hyperband',
                               storage='sqlite:///optuna_studies.db',
                               load_if_exists=True)

# ну случай удаления обучения
# optuna.delete_study(study_name="HistGradientBoostingClassifier_meta_second", storage='sqlite:///optuna_studies.db')

In [ ]:
# ищем лучшую комбинацию гиперпараметров n_trials раз
optuna_study_gb_meta.optimize(optuna_gb_meta, n_trials=500)

##### <span style="color:MediumSlateBlue">Анализ гиперпараметров</span>

In [ ]:
# из полученного результат соврмируем Data Frame
optuna_study_gb_meta_pd = optuna_study_gb_meta.trials_dataframe()
# переименуем столбы
optuna_study_gb_meta_pd.rename(columns={
    'values_0': 'roc_train',
    'values_1': 'roc_valid',
    'values_2': 'f1_score',
    'values_3': 'recall_1',
    'values_4': 'precision_1'
},inplace=True)
optuna_study_gb_meta_pd.tail(5)

In [ ]:
# покаждем статистику обучения
print('Максимальное значение метрики ROC AUC на валидационном наборе:',round(optuna_study_gb_meta_pd['roc_valid'].max(),3))
print('Среднее значение метрики ROC AUC на валидационном наборе:',round(optuna_study_gb_meta_pd['roc_valid'].mean(),3))
print('Максимальное значение метрики f1_score на валидационном наборе:',round(optuna_study_gb_meta_pd['f1_score'].max(),3))
print('Среднее значение метрики f1_score на валидационном наборе:',round(optuna_study_gb_meta_pd['f1_score'].mean(),3))
print('Максимальное значение метрики recall_1 на валидационном наборе:',round(optuna_study_gb_meta_pd['recall_1'].max(),3))
print('Среднее значение метрики recall_1 на валидационном наборе:',round(optuna_study_gb_meta_pd['recall_1'].mean(),3))
print('Максимальное значение метрики precision_1 на валидационном наборе:',round(optuna_study_gb_meta_pd['precision_1'].max(),3))
print('Среднее значение метрики precision_1 на валидационном наборе:',round(optuna_study_gb_meta_pd['precision_1'].mean(),3))

In [ ]:
# построим график важности гиперпарметров
optuna.visualization.plot_param_importances(optuna_study_gb_meta, target_name='Score')

In [ ]:
# Построим зависимость метрики precision_1 от гиперпараметров
fig = px.scatter(
    data_frame=optuna_study_gb_meta_pd,
    x='precision_1', #ось абсцисс
    y=['roc_train', 'roc_valid', 'recall_1','f1_score'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision класса 1 от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision класса 1',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

In [ ]:
# Построим зависимость метрик качества модели от number

fig = px.scatter(
    data_frame=optuna_study_gb_meta_pd[(optuna_study_gb_meta_pd['f1_score']>0.186)],
    x='number', #ось абсцисс
    y=['roc_train','roc_valid','recall_1','precision_1','f1_score'], #ось ординат
)
fig.update_layout(
    title ={
        'text':'Зависимость precision от гиперпараметров модели', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height =800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Величина метрики precision_1',
    yaxis_title='Величина параметра')
fig.update_xaxes(showspikes=True)
fig.update_yaxes(showspikes=True)

fig.show()

<span style="color:Blue">

Вывод:  

Их всех выбираем точку $number =564$.   
В этой точке оптимальные значения метрик $recall_1$ и $precision_1$, а  
также относительно высокая метрика $ROC AUC$.   
Посмотрим каким значениям гиперпараметров соотвествует выбранная точка.

In [ ]:
# определим номер лучшге варианта
best_optuna_number = 564

# сформируем словарь лучших гипирпарметров HistGradientBoostingClassifier
best_param_hgbc = {
    'learning_rate' : optuna_study_gb_meta_pd[optuna_study_gb_meta_pd['number']==best_optuna_number]['params_learning_rate'].iloc[0],
    'max_iter' : optuna_study_gb_meta_pd[optuna_study_gb_meta_pd['number']==best_optuna_number]['params_max_iter'].iloc[0],
    'max_leaf_nodes' : optuna_study_gb_meta_pd[optuna_study_gb_meta_pd['number']==best_optuna_number]['params_max_leaf_nodes'].iloc[0],
    'max_depth' : optuna_study_gb_meta_pd[optuna_study_gb_meta_pd['number']==best_optuna_number]['params_max_depth'].iloc[0],
    'min_samples_leaf' : optuna_study_gb_meta_pd[optuna_study_gb_meta_pd['number']==best_optuna_number]['params_min_samples_leaf'].iloc[0],
    'max_features' : optuna_study_gb_meta_pd[optuna_study_gb_meta_pd['number']==best_optuna_number]['params_max_features'].iloc[0],
    'l2_regularization' : optuna_study_gb_meta_pd[optuna_study_gb_meta_pd['number']==best_optuna_number]['params_l2_regularization'].iloc[0],
    'class_weight' : {0: optuna_study_gb_meta_pd[optuna_study_gb_meta_pd['number']==best_optuna_number]['params_class_0_weight'].iloc[0],
                      1: optuna_study_gb_meta_pd[optuna_study_gb_meta_pd['number']==best_optuna_number]['params_class_1_weight'].iloc[0],}
    }

# определим перменные для лучших значений параметров
best_k_features = optuna_study_gb_meta_pd[optuna_study_gb_meta_pd['number']==best_optuna_number]['params_k_features'].iloc[0]
best_random_state = optuna_study_gb_meta_pd[optuna_study_gb_meta_pd['number']==best_optuna_number]['params_random_state'].iloc[0]
best_class_1_percent = optuna_study_gb_meta_pd[optuna_study_gb_meta_pd['number']==best_optuna_number]['params_class_1_percent'].iloc[0]
best_train_size = optuna_study_gb_meta_pd[optuna_study_gb_meta_pd['number']==best_optuna_number]['params_train_size'].iloc[0]


# Выведем принятые наилучшие параметры
print('best learning rate:',round(best_param_hgbc['learning_rate'],3))
print('best max iter:',best_param_hgbc['max_iter'])
print('best max leaf nodes:',best_param_hgbc['max_leaf_nodes'])
print('best max depth:',best_param_hgbc['max_depth'])
print('best min samples leaf:',best_param_hgbc['min_samples_leaf'])
print('best max features:',round(best_param_hgbc['max_features'],3))
print('best l2 regularization:',round(best_param_hgbc['l2_regularization'],3))
print('best class 0 weight:',round(best_param_hgbc['class_weight'][0],3))
print('best class 1 weight:',round(best_param_hgbc['class_weight'][1],3))

print('best class 1 percent:',round(best_class_1_percent,3))
print('best k features:',best_k_features)
print('best random state:',best_random_state)
print('best train size:',best_train_size)
print('time for best train:',round(optuna_study_gb_meta_pd[optuna_study_gb_meta_pd['number']==best_optuna_number]['duration'].iloc[0].seconds/60),'minutes')

print()
print('ROC AUC на обучающем наборе:', round(optuna_study_gb_meta_pd[optuna_study_gb_meta_pd['number']==best_optuna_number]['roc_train'].iloc[0],3))
print('ROC AUC на валидационном наборе:', round(optuna_study_gb_meta_pd[optuna_study_gb_meta_pd['number']==best_optuna_number]['roc_valid'].iloc[0],3))
print('precision класса 1:', round(optuna_study_gb_meta_pd[optuna_study_gb_meta_pd['number']==best_optuna_number]['precision_1'].iloc[0],3))
print('recall класса 1:', round(optuna_study_gb_meta_pd[optuna_study_gb_meta_pd['number']==best_optuna_number]['recall_1'].iloc[0],3))

##### <span style="color:MediumSlateBlue">Обучение модели с лучшими параметрами</span>

In [ ]:
# Загружаем обучающие наборы
features_second_meta = fp.ParquetFile('second_metamodels/features/features_second_meta').to_pandas()

# Выделим из features_second_meta тренировочную часть
MX_train_pd = my_train_test_split(features_second_meta.set_index('id').drop(['flag'],axis=1),
                                                        features_second_meta.set_index('id')['flag'], 
                                                        train_size=best_train_size, random_state=best_random_state)[0]
my_train_pd = my_train_test_split(features_second_meta.set_index('id').drop(['flag'],axis=1),
                                                        features_second_meta.set_index('id')['flag'], 
                                                        train_size=best_train_size, random_state=best_random_state)[1].reset_index()

# поготовим данные my_train_pd к работе с функцией class_1_percent_samples
my_train_pd.set_index('id',drop=False,inplace=True)

# с помощью функции class_1_percent_samples зададим долю класса 1
list_c1_percent_id = class_1_percent_samples(my_train_pd,best_class_1_percent,random_state=best_random_state)[:1000000]

# подготовим данные для обучения
MX_train_balanced = MX_train_pd.loc[list_c1_percent_id]
my_train_balanced = my_train_pd.loc[list_c1_percent_id]['flag'].to_numpy()

# освободим память от "тяжелых" и ненужных файлов
del MX_train_pd, my_train_pd
gc.collect()

# с помощью класса SelectKBest получим список лучших признаков
selector = SelectKBest(f_classif, k=best_k_features)
selector.fit(MX_train_balanced, my_train_balanced)
list_best_features = selector.get_feature_names_out()

MX_train_balanced = MX_train_balanced[list_best_features].to_numpy()

# обучаем модель HistGradientBoostingClassifier с наилучшеми параметрами
gradient_boosting = HistGradientBoostingClassifier(
        **best_param_hgbc,
        random_state=best_random_state)
gradient_boosting.fit(MX_train_balanced,my_train_balanced)

# удаляем крупные файлы чтобы высвободить память 
del MX_train_balanced, my_train_balanced
gc.collect()

# загружаем тестовые наборы
MX_train = my_train_test_split(features_second_meta.set_index('id').drop(['flag'],axis=1),
                                                        features_second_meta.set_index('id')['flag'], 
                                                        train_size=best_train_size, random_state=best_random_state)[0][list_best_features].to_numpy()
my_train = my_train_test_split(features_second_meta.set_index('id').drop(['flag'],axis=1),
                                                        features_second_meta.set_index('id')['flag'], 
                                                        train_size=best_train_size, random_state=best_random_state)[1].to_numpy()

MX_valid = my_train_test_split(features_second_meta.set_index('id').drop(['flag'],axis=1),
                                                        features_second_meta.set_index('id')['flag'], 
                                                        train_size=best_train_size, random_state=best_random_state)[2][list_best_features].to_numpy()
my_valid = my_train_test_split(features_second_meta.set_index('id').drop(['flag'],axis=1),
                                                        features_second_meta.set_index('id')['flag'], 
                                                        train_size=best_train_size, random_state=best_random_state)[3].to_numpy()

# для метрик ROC AUC делаем предсказание модели в виде вероятности
my_train_gb_pred_proba = gradient_boosting.predict_proba(MX_train)[:,1]
my_valid_gb_pred_proba = gradient_boosting.predict_proba(MX_valid)[:,1]

# Делаем предсказание для валидационной выборки
my_valid_gb_pred = gradient_boosting.predict(MX_valid)

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(my_train, my_train_gb_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(my_valid, my_valid_gb_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(my_valid,my_valid_gb_pred,zero_division=0))

# time: 1m 40s

In [ ]:
# сохраним модель для дальнейшего воспроизведения
dump(gradient_boosting, 'second_metamodels/models/best_model.joblib')
# сохраним SelectKBest для дальнейшего воспроизведения
dump(selector, 'second_metamodels/selector/best_selector.joblib')

##### <span style="color:MediumSlateBlue">Анализ влияния порога классификации на качество модели</span>

In [ ]:
# чтобы проше обращаться к каждому обьекту в данных
# преобразуем y_valid_LR_pred к Series типу
my_valid_gb_pred_proba = pd.Series(my_valid_gb_pred_proba)

# формируем список из необходимых значений thresholds
thresholds = np.arange(0.01, 1, 0.01)

# сформируем списко для заполнения данными результатов анализа
threshold_data = []


for threshold in thresholds:
        # сформируем список для заполнения данными текушего threshold
        threshold_current_data = [threshold]

        #клиентов, для которых вероятность дефолта > threshold, относим к классу 1
        #в противном случае — к классу 0
        my_valid_gb_pred = my_valid_gb_pred_proba.apply(lambda x: 1 if x>threshold else 0)


        #Считаем метрики для класса 0 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(my_valid, my_valid_gb_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.precision_score(my_valid, my_valid_gb_pred,average=None,zero_division=0)[0],3))
        threshold_current_data.append(round(metrics.f1_score(my_valid, my_valid_gb_pred,average=None,zero_division=0)[0],3))

        #Считаем метрики для класса 1 добавляем их в список
        threshold_current_data.append(round(metrics.recall_score(my_valid, my_valid_gb_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.precision_score(my_valid, my_valid_gb_pred,average=None,zero_division=0)[1],3))
        threshold_current_data.append(round(metrics.f1_score(my_valid, my_valid_gb_pred,average=None,zero_division=0)[1],3))

        # добавляем полученные данные в список threshold_data
        threshold_data.append(threshold_current_data)

        # из полученных данных сформируем dataframe
        thresholds_pd =pd.DataFrame(
        columns=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'],
        data = threshold_data)

# time: 17s

In [ ]:
# Визуализируем метрики на валидационном наборе при различных threshold
fig_threshold = px.line(
    data_frame=thresholds_pd,
    x='threshold', #ось абсцисс
    y=['threshold','recall_0','precision_0','f1_0','recall_1','precision_1','f1_1'], #ось ординат
)
fig_threshold.update_layout(
    title ={
        'text':'Зависимость качества модели от threshold', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 800,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    bargap=0.2, # Добавил расстояния между столбами гистограммы
    xaxis_title='Порог классификации threshold',
    yaxis_title='Величина метрики')
fig_threshold.update_xaxes(showspikes=True)
fig_threshold.update_yaxes(showspikes=True)

fig_threshold.show()

Метрика $precision$ показывает на сколько сильно ошиблась модель при определиние класса.  
Чем выше метрика, тем меньше ошиблась модель. 

$$precision = \frac{TP}{TP+FP}$$

Метрика $recall$ показывает способность модели найти класс.  
Чем выше метрика, тем выше способность модели находить класс.  

$$recall = \frac{TP}{TP+FN}$$

Метрика $f1-score$ показывает баланс между $precision$ и $recall$ 

$threshold$- порог класификации модели.  
Все объекты для которых $у_{\text{proba pred}} < threshold$, модель относит к класс 0.  
Соотвественно, если $у_{\text{proba pred}}  > threshold$ модель относит обьект к классу 1.  
Для идельаной модели $threshold$, является границей между областями класса 0 и класса 1.  

<span style="color:Blue">

Выводы по результам анализа:  

Для улучшения качества модели по метрике $ROC AUC$, нет смысла сдвигать порог   
классификации. Но зависимость метрик качества модели от $threshold$, может рассказать о  
способности модели искать класс 1 и класс 0.

При фокусировке модели на классе 1 (более низкий $threshold$):  
- на классе 1 метрика $precision$ уменьшается, метрика $recall$ растет;  
- на классе 0 метрика $precision$ растет, метрика $recall$ уменьшается.  

При фокусировке модели на классе 0 (более высокий $threshold$):  
- на классе 1 метрика $precision$ увеличивается, метрика $recall$ уменьшается;  
- на классе 0 метрика $precision$ уменьшается, метрика $recall$ растет.  

Обьсняется тем, что в условия дисбаланса модель научилась хорошо определять класс 0 и плохо класс 1.  

При фокусировке модели на классе 1 (более низкий $threshold$), все больше обьектов отнесены к классу 1.   
Так как $precision$ класса 1 уменьшается, значит в области класса 1 стало больше обектов класса 0,  
что явлется закономерным для "идеальной" модели.  
Однако при этом, $recall$ класса 1 растет, значит в добавленных обьектах также присутвует класс 1.  
Это как раз указывает на то, что модель не нучилась находить среди класса 0 класс 1. В области с     
высокой вероятностью класса 0 (более низкий $threshold$), находятся обьекты класса 1.  

Фокусировка на классе 0 (более высокий $threshold$), заставляет модель все больше обьектов из области класса 1  
относить к классу 0. При этом увеличение метрики $precision$ класса 1 говорит о том, что в области класса 1     
становится меньше 0, а значит модель умеет среди класса 1, находить класс 0.  
После  $threshold=0.88$ модель впринципе теряет способность отличать класс 1. У модели нет обьектов класса 1   
в которых она "уверена" на 90+%.


Обратим внимание на $threshold=0.52$, в этой точке находится баланс межу метриками качества модели:
$$precision_1 = recall_1 = \text{f1-score}_1 = 0.163$$
$$precision_0 = recall_0 = \text{f1-score}_0 = 0.97$$

Для модели с лучшим качеством эти две точки должны находится максимально близко к друг другу и иметь   
высокое значение метрик $precision_1$ и $recall_1$.  
Для класса 0 это условие выполнется, что также говорит о том, что модель научилась искать этот класс.    
Для класса 1 это условие не выполнется, что также говорит о том, что модель плохо ищет класс 1.

##### <span style="color:MediumSlateBlue">Построение ROC-кривой выбранной модели на тестовом наборе</span>

In [ ]:
# загружаем тестовые наборы
MX_train = fp.ParquetFile('second_metamodels/features/features_second_meta_train').to_pandas()[list_best_features].to_numpy()
my_train = pd.read_csv('second_metamodels/data/msy_train.csv').set_index('id')['flag'].to_numpy()
MX_valid = fp.ParquetFile('second_metamodels/features/features_second_meta_valid').to_pandas()[list_best_features].to_numpy()
my_valid = pd.read_csv('second_metamodels/data/msy_valid.csv').set_index('id')['flag'].to_numpy()
MX_test = fp.ParquetFile('second_metamodels/features/features_second_meta_test').to_pandas()[list_best_features].to_numpy()
my_test = pd.read_csv('second_metamodels/data/msy_test.csv').set_index('id')['flag'].to_numpy()

In [ ]:
print("Размеры обучающего набора",MX_train.shape,my_train.shape)
print("Размеры валидационного набора",MX_valid.shape,my_valid.shape)
print("Размеры тестового набора",MX_test.shape,my_test.shape)
# time: 1m 43s

In [ ]:
# для метрик ROC AUC делаем предсказание модели в виде вероятности
my_train_gb_pred_proba = gradient_boosting.predict_proba(MX_train)[:,1]
my_valid_gb_pred_proba = gradient_boosting.predict_proba(MX_valid)[:,1]
my_test_gb_pred_proba = gradient_boosting.predict_proba(MX_test)[:,1]

# Делаем предсказание для валидационной выборки
my_test_gb_pred = gradient_boosting.predict(MX_test)

del MX_train, MX_valid, MX_test
gc.collect()

#Выводим значения метрик
print('ROC AUC на обучающем наборе', round(metrics.roc_auc_score(my_train, my_train_gb_pred_proba),3))
print('ROC AUC на валидационном наборе', round(metrics.roc_auc_score(my_valid, my_valid_gb_pred_proba),3))
print('ROC AUC на тестовом наборе', round(metrics.roc_auc_score(my_test, my_test_gb_pred_proba),3))

print('Основные метрики на валидационом наборе:')
print(metrics.classification_report(my_test,my_test_gb_pred,zero_division=0))

In [ ]:
# Для расчета значений TPR и FPR используем функцию roc_curve
fpr_train, tpr_train, _ = metrics.roc_curve(my_train, my_train_gb_pred_proba)
fpr_valid, tpr_valid, _ = metrics.roc_curve(my_valid, my_valid_gb_pred_proba)
fpr_test, tpr_test, _ = metrics.roc_curve(my_test, my_test_gb_pred_proba)

# рассчитаем площадь под кривой
auc_train = round(metrics.roc_auc_score(my_train, my_train_gb_pred_proba),3)
auc_valid = round(metrics.roc_auc_score(my_valid, my_valid_gb_pred_proba),3)
auc_test = round(metrics.roc_auc_score(my_test, my_test_gb_pred_proba),3)

trace_train = go.Scatter(x=fpr_train, y=tpr_train, mode='lines', name='AUC_train = %0.2f' % auc_train,
                   line=dict(color='red', width=2),
                   hovertemplate='train<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_train])

trace_valid = go.Scatter(x=fpr_valid, y=tpr_valid, mode='lines', name='AUC_valid = %0.2f' % auc_valid,
                   line=dict(color='green', width=2),
                   hovertemplate='valid<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_valid])

trace_test = go.Scatter(x=fpr_test, y=tpr_test, mode='lines', name='AUC_test = %0.2f' % auc_test,
                   line=dict(color='darkorange', width=2),
                   hovertemplate='test<br>'+
                                '%{text}<br>'+
                                'recall class 1: %{y:.3f}<br>',
                                text = ['recall class 0 {}'.format(round(1-i,3)) for i in fpr_test])


reference_line = go.Scatter(x=[0,1], y=[0,1], mode='lines', name='Reference Line',
                            line=dict(color='navy', width=2, dash='dash'))

fig_roc = go.Figure(data=[trace_train,trace_valid,trace_test,reference_line])

fig_roc.update_layout(
    title ={
        'text':'ROC-кривая', # Имя рабочей плоскости
        'font':{'size':35,'family':"Times New Roman"}, # размер и стиль написания имени рабочей плоскости
        'x':0.5, # Смешение имени по оси "x" на половину рабочей плоскости
        },
    height = 1350,# Высота рабочей плоскости
    width = 1350, # Ширина рабочей плоскости
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate')

fig_roc.update_xaxes(showspikes=True)
fig_roc.update_yaxes(showspikes=True)


fig_roc.show()

<span style="color:Blue">

Вывод по ROC-кривой:
1. Модель выдает сталибильное качество на тренировочном, валидационном и тестовом наборах, а значит  
не переобучена - разброс ($variance$) не большой.
2. На тестовом наборе полученно качество $ROC AUC = 0.76$.

## <span style="color:DodgerBlue">Обучение блендинга на отложенном наборе данных</span>

In [ ]:
# сформируем словарь из данных отложенных для тестирования модели
dict_datasets = {
  'date_torow': fp.ParquetFile('base_models/data/torow/date_torow_hold').to_pandas(),
  'late_torow': fp.ParquetFile('base_models/data/torow/late_torow_hold').to_pandas(),
  'credit_torow': fp.ParquetFile('base_models/data/torow/credit_torow_hold').to_pandas(),
  'relative_torow': fp.ParquetFile('base_models/data/torow/relative_torow_hold').to_pandas(),
  'payments_torow': fp.ParquetFile('base_models/data/torow/payments_torow_hold').to_pandas(),
  'service_torow': fp.ParquetFile('base_models/data/torow/service_torow_hold').to_pandas(),
  'date_stat': fp.ParquetFile('base_models/data/stat/date_stat_hold').to_pandas(),
  'late_stat': fp.ParquetFile('base_models/data/stat/late_stat_hold').to_pandas(),
  'credit_stat': fp.ParquetFile('base_models/data/stat/credit_stat_hold').to_pandas(),
  'relative_stat': fp.ParquetFile('base_models/data/stat/relative_stat_hold').to_pandas(),
  'payments_stat': fp.ParquetFile('base_models/data/stat/payments_stat_hold').to_pandas(),
  'service_stat': fp.ParquetFile('base_models/data/stat/service_stat_hold').to_pandas(),
  }

### <span style="color:RoyalBlue">Transform to first meta преобразование</span>

In [ ]:
def to_first_meta(dict_datasets):
  # формируем пространство для метапризнаков
  list_spaces = ['date','late','credit','relative','payments','service']
  torow_models = ['LRTR','RFTR','GBTR']
  stat_models = ['LRST','RFST','GBST']
  
  # фомируем заготовки для результитрующих данных
  features_first_meta_torow = pd.DataFrame(index=dict_datasets['credit_torow'].index)
  features_first_meta_stat = pd.DataFrame(index=dict_datasets['credit_torow'].index)

  # указываем путь к директории в которой находятся обученные base модели
  directory = "base_models/models/"
  # Получаем список моделей в директории
  list_models = os.listdir(directory)

  # формируем метапризнаки данных torow
  for space in list_spaces:
    for type_model in torow_models:
      # для текущего подбпространства формируем список предобученных моделей 
      list_base_models = sorted([x for x in list_models if (re.search(space, x))and(re.search(type_model, x))])
      # делаем предсказание на каждой предобученной модели
      for name_model in list_base_models:
        # добавим индекацию процесса
        clear_output()
        print('Current feature: torow')
        print('Current space: ', space)
        print('Current type_model: ', type_model)
        print('Current name_model: ', name_model)
        # загружаем предобученную модель
        model_classifier = joblib.load('base_models/models/'+name_model)
        # загружаем list_n_last_features
        list_n_last_features = joblib.load('base_models/list_n_last_features/'+'list_n_last_features_'+type_model+'_'+name_model[5:-7]+'.joblib')
        if type_model=='LRTR':
          # загужаемый предобученный scaler
          scaler = joblib.load('base_models/scalers/'+'scaler_torow_'+name_model[5:-7]+'.joblib')
          features_first_meta_torow[name_model[:-7]] = model_classifier.predict_proba(scaler.transform(dict_datasets[space+'_torow'].to_numpy()[:,list_n_last_features]))[:,1]
        else:
          features_first_meta_torow[name_model[:-7]] = model_classifier.predict_proba(dict_datasets[space+'_torow'].to_numpy()[:,list_n_last_features])[:,1]

    del dict_datasets[space+'_torow']
    gc.collect()

  # формируем метапризнаки данных stat
  for space in list_spaces:
    for type_model in stat_models:
      # для текущего подбпространства формируем список предобученных моделей 
      list_base_models = sorted([x for x in list_models if (re.search(space, x))and(re.search(type_model, x))])
      # делаем предсказание на каждой предобученной модели
      for name_model in list_base_models:
        # добавим индекацию процесса
        clear_output()
        print('Current feature: stat')
        print('Current space: ', space)
        print('Current type_model: ', type_model)
        print('Current name_model: ', name_model)
        # загружаем предобученную модель
        model_classifier = joblib.load('base_models/models/'+name_model)
        # загружаем list_ncorr_features
        list_ncorr_features = joblib.load('base_models/list_ncorr_features/'+'list_ncorr_features_'+type_model+'_'+name_model[5:-7]+'.joblib')
        # загружаем 
        if type_model=='LRST':
          # загужаемый предобученный scaler
          scaler = joblib.load('base_models/scalers/'+'scaler_stat_'+name_model[5:-7]+'.joblib')
          features_first_meta_stat[name_model[:-7]] = model_classifier.predict_proba(scaler.transform(dict_datasets[space+'_stat'][list_ncorr_features]))[:,1]
        else:
          features_first_meta_stat[name_model[:-7]] = model_classifier.predict_proba(dict_datasets[space+'_stat'][list_ncorr_features])[:,1]

    del dict_datasets[space+'_stat']
    gc.collect()

  dict_meta_first = {
    'features_first_meta_torow': features_first_meta_torow,
    'features_first_meta_stat': features_first_meta_stat
    }

  return dict_meta_first

# преобразуем функции в инструмент для преобразования данных (Transformer)
to_first_meta_transformer = FunctionTransformer(to_first_meta)

In [ ]:
# преобразуем отложенный набор данных до метапризнаков первой метамодели
dict_first_meta_datasets = to_first_meta_transformer.transform(dict_datasets)

### <span style="color:RoyalBlue">Transform to second meta преобразование</span>

In [ ]:
def to_second_meta(dict_datasets):
  # формируем пространство для метапризнаков
  space_models = ['LRTR','RFTR','GBTR','LRST','RFST','GBST']
  
  # фомируем заготовку для результитрующих данных
  features_second_meta = pd.DataFrame(index=dict_datasets['features_first_meta_stat'].index)

  # указываем путь к директории в которой находятся обученные base модели
  directory = "first_metamodel/models/"
  # Получаем список моделей в директории
  list_models = os.listdir(directory)

  # формируем метапризнаки данных torow

  for model in space_models:
    # для текущего подбпространства формируем список предобученных моделей 
    list_first_meta_models = sorted([x for x in list_models if re.search(model, x)])
    # выберем неободимый dataset
    choose_dataset_name = lambda torow, stat, model: torow if model[-2:] == 'TR' else stat
    dataset_name = choose_dataset_name('features_first_meta_torow','features_first_meta_stat',model)

    # делаем предсказание на каждой предобученной модели
    for name_model in list_first_meta_models:
      # добавим индекацию процесса
      clear_output()
      print('Current name_model: ', name_model)
      # загружаем предобученную модель
      model_classifier = joblib.load('first_metamodel/models/'+name_model)
      # загружаем list_best_features
      list_best_features = joblib.load('first_metamodel/list_best_features/'+'list_best_features_'+name_model[:-7]+'.joblib')
      # запоняем dataframe мета признаками 
      features_second_meta[name_model[:-7]] = model_classifier.predict_proba(dict_datasets[dataset_name][list_best_features].to_numpy())[:,1]

  return features_second_meta

# преобразуем функции в инструмент для преобразования данных (Transformer)
to_second_meta_transformer = FunctionTransformer(to_second_meta)

In [ ]:
# преобразуем отложенный набор данных до метапризнаков второй метамодели
hold_features_second_meta = to_second_meta_transformer.transform(dict_first_meta_datasets)

### <span style="color:RoyalBlue">Обучение результирующей модели, показавшей лучший результат</span>

In [ ]:
# загрузим загрузим модель показавшую наилучший результат
best_classifier = joblib.load('second_metamodels/models/best_model.joblib')
# загрузим selector для лучшей модели
best_selector = joblib.load('second_metamodels/selector/best_selector.joblib')
list_best_features = best_selector.get_feature_names_out()

In [ ]:
# подгрузим данные целевой переменной
target = pd.read_csv('data_source/train_data/train_target.csv')

In [ ]:
MX_hold = hold_features_second_meta[list_best_features].to_numpy()
my_hold = target.loc[hold_features_second_meta.index]['flag'].to_numpy()

In [ ]:
# для метрик ROC AUC делаем предсказание модели в виде вероятности
my_hold_pred_proba = best_classifier.predict_proba(MX_hold)[:,1]

# Делаем предсказание для валидационной выборки
my_hold_pred = best_classifier.predict(MX_hold)

#Выводим значения метрик
print('ROC AUC на отложенном наборе', round(metrics.roc_auc_score(my_hold, my_hold_pred_proba),3))

print('Основные метрики на отложенном наборе:')
print(metrics.classification_report(my_hold,my_hold_pred,zero_division=0))

In [25]:
# ROC AUC на отложенном наборе 0.756
# Основные метрики на отложенном наборе:
#               precision    recall  f1-score   support

#            0       0.97      0.93      0.95    144688
#            1       0.13      0.26      0.17      5312

#     accuracy                           0.91    150000
#    macro avg       0.55      0.60      0.56    150000
# weighted avg       0.94      0.91      0.93    150000

## <span style="color:DodgerBlue">Сбор результирующей модели в pipeline</span>

In [ ]:
# напишем функцию для преобразования данных от исходных  
# до признаков base models

def to_base_model(dataset):
    # сформируем списки признаков каждого подпространства
    date_features = ['id','rn','pre_since_opened','pre_since_confirmed','pre_pterm','pre_fterm',
                    'pre_till_pclose','pre_till_fclose','pclose_flag','fclose_flag']
    late_features = ['id','rn','pre_loans5','pre_loans530','pre_loans3060','pre_loans6090',
                    'pre_loans90','is_zero_loans5','is_zero_loans530','is_zero_loans3060',
                    'is_zero_loans6090','is_zero_loans90','pre_loans_total_overdue','pre_loans_max_overdue_sum']
    credit_features = ['id','rn','pre_loans_credit_limit','pre_loans_next_pay_summ','pre_loans_outstanding','pre_loans_credit_cost_rate']

    relative_features = ['id','rn','pre_util','pre_over2limit','pre_maxover2limit','is_zero_util',
                    'is_zero_over2limit','is_zero_maxover2limit']

    payments_features = ['id','rn','enc_paym_0','enc_paym_1','enc_paym_2','enc_paym_3','enc_paym_4','enc_paym_5','enc_paym_6',
                            'enc_paym_7','enc_paym_8','enc_paym_9','enc_paym_10','enc_paym_11','enc_paym_12','enc_paym_13',
                            'enc_paym_14', 'enc_paym_15', 'enc_paym_16', 'enc_paym_17','enc_paym_18','enc_paym_19','enc_paym_20',
                            'enc_paym_21','enc_paym_22', 'enc_paym_23','enc_paym_24']

    service_features = ['id','rn','enc_loans_account_holder_type','enc_loans_credit_status','enc_loans_credit_type','enc_loans_account_cur']

    # составим словарь небходимых данных и их признаков
    dict_torow_data = {
        'date_torow' : date_features,
        'late_torow' : late_features,
        'credit_torow' : credit_features,
        'relative_torow' : relative_features,
        'payments_torow': payments_features,
        'service_torow': service_features}
    # сформируем данные за последние 25 операций клиентов
    for space_features in dict_torow_data.keys():
        clear_output()
        # добавим индекацию процесса
        print('Current space features:',space_features)
        # сформируем данные для преобразования
        data_to_transform = dataset[dict_torow_data[space_features]]
        print('Start transform')
        data_torow = torow_transformer.transform({'data':data_to_transform,'n_last':25})[0]
        # сохраним преобразованные данные на диск для быстрого воспроизведения
        print('Start save')
        fp.write('sample_submission/pipeline/'+space_features,data_torow)
        # удалим использованные данные
        del data_to_transform
        gc.collect()

    # составим словарь небходимых данных и их признаков
    dict_stat_data = {
        'date_stat' : date_features,
        'late_stat' : late_features,
        'credit_stat' : credit_features,
        'relative_stat' : relative_features,
        'payments_stat': payments_features,
        'service_stat': service_features}
    # сформируем данные за последние 25 операций клиентов
    for space_features in dict_stat_data.keys():
        clear_output()
        # добавим индекацию процесса
        print('Current space features:',space_features)
        # сформируем данные для преобразования
        data_to_transform = dataset[dict_stat_data[space_features]]
        print('Start transform')
        data_stat = stat_transformer.transform(data_to_transform)[0]
        # сохраним преобразованные данные на диск для быстрого воспроизведения
        print('Start save')
        fp.write('sample_submission/pipeline/'+space_features,data_stat)
        # удалим использованные данные
        del data_to_transform, data_stat
        gc.collect()
    
    # сформируем словарь из данных отложенных для тестирования модели
    dict_datasets = {
    'date_torow': fp.ParquetFile('sample_submission/pipeline/date_torow').to_pandas().set_index('id'),
    'late_torow': fp.ParquetFile('sample_submission/pipeline/late_torow').to_pandas().set_index('id'),
    'credit_torow': fp.ParquetFile('sample_submission/pipeline/credit_torow').to_pandas().set_index('id'),
    'relative_torow': fp.ParquetFile('sample_submission/pipeline/relative_torow').to_pandas().set_index('id'),
    'payments_torow': fp.ParquetFile('sample_submission/pipeline/payments_torow').to_pandas().set_index('id'),
    'service_torow': fp.ParquetFile('sample_submission/pipeline/service_torow').to_pandas().set_index('id'),
    'date_stat': fp.ParquetFile('sample_submission/pipeline/date_stat').to_pandas().set_index('id'),
    'late_stat': fp.ParquetFile('sample_submission/pipeline/late_stat').to_pandas().set_index('id'),
    'credit_stat': fp.ParquetFile('sample_submission/pipeline/credit_stat').to_pandas().set_index('id'),
    'relative_stat': fp.ParquetFile('sample_submission/pipeline/relative_stat').to_pandas().set_index('id'),
    'payments_stat': fp.ParquetFile('sample_submission/pipeline/payments_stat').to_pandas().set_index('id'),
    'service_stat': fp.ParquetFile('sample_submission/pipeline/service_stat').to_pandas().set_index('id'),
    }
    return dict_datasets

# преобразуем функции в инструмент для преобразования данных (Transformer)
to_base_model_transformer = FunctionTransformer(to_base_model)

In [ ]:
# напишем функцию для преобразования данных от признаков base models
# до метапризнаков first metamodels

def to_first_meta(dict_datasets):
  # формируем пространство для метапризнаков
  list_spaces = ['date','late','credit','relative','payments','service']
  torow_models = ['LRTR','RFTR','GBTR']
  stat_models = ['LRST','RFST','GBST']
  
  # фомируем заготовки для результитрующих данных
  features_first_meta_torow = pd.DataFrame(index=dict_datasets['credit_torow'].index)
  features_first_meta_stat = pd.DataFrame(index=dict_datasets['credit_torow'].index)

  # указываем путь к директории в которой находятся обученные base модели
  directory = "base_models/models/"
  # Получаем список моделей в директории
  list_models = os.listdir(directory)

  # формируем метапризнаки данных torow
  for space in list_spaces:
    for type_model in torow_models:
      # для текущего подбпространства формируем список предобученных моделей 
      list_base_models = sorted([x for x in list_models if (re.search(space, x))and(re.search(type_model, x))])
      # делаем предсказание на каждой предобученной модели
      for name_model in list_base_models:
        # добавим индекацию процесса
        clear_output()
        print('Current feature: torow')
        print('Current space: ', space)
        print('Current type_model: ', type_model)
        print('Current name_model: ', name_model)
        # загружаем предобученную модель
        model_classifier = joblib.load('base_models/models/'+name_model)
        # загружаем list_n_last_features
        list_n_last_features = joblib.load('base_models/list_n_last_features/'+'list_n_last_features_'+type_model+'_'+name_model[5:-7]+'.joblib')
        if type_model=='LRTR':
          # загужаемый предобученный scaler
          scaler = joblib.load('base_models/scalers/'+'scaler_torow_'+name_model[5:-7]+'.joblib')
          features_first_meta_torow[name_model[:-7]] = model_classifier.predict_proba(scaler.transform(dict_datasets[space+'_torow'].to_numpy()[:,list_n_last_features]))[:,1]
        else:
          features_first_meta_torow[name_model[:-7]] = model_classifier.predict_proba(dict_datasets[space+'_torow'].to_numpy()[:,list_n_last_features])[:,1]

    del dict_datasets[space+'_torow']
    gc.collect()

  # формируем метапризнаки данных stat
  for space in list_spaces:
    for type_model in stat_models:
      # для текущего подбпространства формируем список предобученных моделей 
      list_base_models = sorted([x for x in list_models if (re.search(space, x))and(re.search(type_model, x))])
      # делаем предсказание на каждой предобученной модели
      for name_model in list_base_models:
        # добавим индекацию процесса
        clear_output()
        print('Current feature: stat')
        print('Current space: ', space)
        print('Current type_model: ', type_model)
        print('Current name_model: ', name_model)
        # загружаем предобученную модель
        model_classifier = joblib.load('base_models/models/'+name_model)
        # загружаем list_ncorr_features
        list_ncorr_features = joblib.load('base_models/list_ncorr_features/'+'list_ncorr_features_'+type_model+'_'+name_model[5:-7]+'.joblib')
        # загружаем 
        if type_model=='LRST':
          # загужаемый предобученный scaler
          scaler = joblib.load('base_models/scalers/'+'scaler_stat_'+name_model[5:-7]+'.joblib')
          features_first_meta_stat[name_model[:-7]] = model_classifier.predict_proba(scaler.transform(dict_datasets[space+'_stat'][list_ncorr_features]))[:,1]
        else:
          features_first_meta_stat[name_model[:-7]] = model_classifier.predict_proba(dict_datasets[space+'_stat'][list_ncorr_features])[:,1]

    del dict_datasets[space+'_stat']
    gc.collect()

  dict_meta_first = {
    'features_first_meta_torow': features_first_meta_torow,
    'features_first_meta_stat': features_first_meta_stat
    }

  return dict_meta_first

# преобразуем функции в инструмент для преобразования данных (Transformer)
to_first_meta_transformer = FunctionTransformer(to_first_meta)

In [ ]:
# напишем функцию для преобразования данных от метапризнаков first metamodels
# до метапризнаков second metamodels

def to_second_meta(dict_datasets):
  # формируем пространство для метапризнаков
  space_models = ['LRTR','RFTR','GBTR','LRST','RFST','GBST']
  
  # фомируем заготовку для результитрующих данных
  features_second_meta = pd.DataFrame(index=dict_datasets['features_first_meta_stat'].index)

  # указываем путь к директории в которой находятся обученные base модели
  directory = "first_metamodel/models/"
  # Получаем список моделей в директории
  list_models = os.listdir(directory)

  # формируем метапризнаки данных torow

  for model in space_models:
    # для текущего подбпространства формируем список предобученных моделей 
    list_first_meta_models = sorted([x for x in list_models if re.search(model, x)])
    # выберем неободимый dataset
    choose_dataset_name = lambda torow, stat, model: torow if model[-2:] == 'TR' else stat
    dataset_name = choose_dataset_name('features_first_meta_torow','features_first_meta_stat',model)

    # делаем предсказание на каждой предобученной модели
    for name_model in list_first_meta_models:
      # добавим индекацию процесса
      clear_output()
      print('Current name_model: ', name_model)
      # загружаем предобученную модель
      model_classifier = joblib.load('first_metamodel/models/'+name_model)
      # загружаем list_best_features
      list_best_features = joblib.load('first_metamodel/list_best_features/'+'list_best_features_'+name_model[:-7]+'.joblib')
      # запоняем dataframe мета признаками 
      features_second_meta[name_model[:-7]] = model_classifier.predict_proba(dict_datasets[dataset_name][list_best_features].to_numpy())[:,1]

  return features_second_meta

# преобразуем функции в инструмент для преобразования данных (Transformer)
to_second_meta_transformer = FunctionTransformer(to_second_meta)

In [ ]:
# напишем функцию для преобразования данных от метапризнаков second metamodels
# до предсказания результирующей метамодели

def best_model(data):
  # загрузим загрузим модель показавшую наилучший результат
  best_classifier = joblib.load('second_metamodels/models/best_model.joblib')
  # загрузим selector для лучшей модели
  best_selector = joblib.load('second_metamodels/selector/best_selector.joblib')
  list_best_features = best_selector.get_feature_names_out()

  # формируем данные для предсказания
  MX_data = data[list_best_features].to_numpy()

  # для метрик ROC AUC делаем предсказание модели в виде вероятности
  my_pred_proba = best_classifier.predict_proba(MX_data)[:,1]

  dataframe = pd.DataFrame(
    data=np.array([data.index,my_pred_proba]).transpose(),
    columns=['id','score'])
  dataframe['id'] = dataframe['id'].astype('int')

  dataframe.to_csv('sample_submission/sample_submission.csv',index=False)
  

  return dataframe

# преобразуем функции в инструмент для преобразования данных (Transformer)
sample_submission_transformer = FunctionTransformer(best_model)

In [ ]:
def pipeline_best_model(data):
  # выполним преобразования для предсказания на базовых моделях 
  base_model_data = to_base_model_transformer.transform(data)
  del data
  gc.collect()

  # выполним преобразования для предсказания на метамоделях первого порядка 
  first_meta_data = to_first_meta_transformer.transform(base_model_data)
  del base_model_data
  gc.collect()

  # выполним преобразования для предсказания на метамоделях второго порядка 
  second_meta_data = to_second_meta_transformer.transform(first_meta_data)
  del first_meta_data

  # выполним предсказание на результирующей метамодели 
  gc.collect()
  sample_submission = sample_submission_transformer.transform(second_meta_data)
  return sample_submission

In [ ]:
# pipeline_best_model = make_pipeline(to_base_model_transformer,to_first_meta_transformer,to_second_meta_transformer,sample_submission_transformer)

## <span style="color:DodgerBlue">Проверка модели на соревновательном dataset с kaggle</span>

In [ ]:
# Подгрузим данные о кредитной истории клиентов
kd0 = pd.read_parquet('data_source/test_data_for_Kaggle/test_data_0.pq')
kd1 = pd.read_parquet('data_source/test_data_for_Kaggle/test_data_1.pq')
# Обьединим данные в один датасет
kaggle_data = pd.concat([kd0,kd1],ignore_index=True)

In [ ]:
sample_submission = pipeline_best_model(kaggle_data)

In [ ]:
# подгрузим DataFrame с id для тренировочного/валидационного/тестового набора данных
sample_submission = pd.read_csv('sample_submission/sample_submission.csv')
sample_submission

Результат модели на соревновательном dataset:

<center> <img src = "Images/kaggle.png" alt="drawing" style="width:1400px;">

# <span style="color:DeepSkyBlue"> Выводы</span>

<span style="color:Blue">

Качество полученной модели бленндинга по метрике $ROC AUC$:
- на тренировочном наборе: $ROC AUC = 0.757$;
- на валидационном наборе: $ROC AUC = 0.765$;
- на отложенном наборе: $ROC AUC = 0.756$;
- на соревновательном (kaggle) наборе: $ROC AUC = 0.743$;

Мероприятия по возможному улучшению качества модели:

1. Проанализировать данные на наличие выбросов и нерепрезентотивных данных. Скорее всего   
есть клиенты, данные которых "вводят в заблуждение" модель.  

2. Проанализировать влияние выбранных статистических характеристик, формирующих dataset STAT.  
Возможно, использование некоторых характеристики наоборот ухудшает качество модели.   
Возможно, есть иные характеристики числовых рядов которые были не учтены.  

3. Проанализирровать влияние размера обучающего набора на каждом этапе на качество результирующей модели.   
Возможно имеет смысл большую часть данных "потратить" на обучение базовых моделей, возможно наоборот.  

4. Проанализировать влияние подпространств признаков на результаты first metamodels.  
Некоторые подпространства показали низкое качество на базовых моделеях, Возможно имеет смысл    
не использовать все подпространства признаков или разделить пространство признаков не на 6 выбранных     
подпространств, а с помощью методов класстеризации.   

5. Изучить вопрос выбранных базовых моделей. Возможно, имеет смысл сменить их или использовать не все.  
В проссе обучения на первом этапе, модель логистической регресии показала наихудшие результаты.

6. Проанализировать подбор гиперпараметров каждой модели. Сменить диапозоны параметров,  
выбрать дополнительные гиперпермараметры и т.д.

7. Сменить критерии "лучших" моделей. Возможно, имеет смысл использовать только один критерий для  
определения "лучших" моделей. В построенном блендинге на каждом этапе использовались все "лучшие"   
модели с предыдущего этапа. Возможно, есть смысл показывать метамодели результаты не от всех "лучших" моделей.

8. Увеличить или уменьшить глубину блендинга.  

9. Использовать не блендинг, а стекинг моделей.  

10. Использовать не блендинг, а нейронную сеть.  